# The AGN activation history of the three dust routes across the boxes

A clean restart of `noxray_analogues.ipynb`. That notebook grew a 2$\times$2 of *age* $\times$ *dust* and thirteen sub-parts on top of it; this one keeps a single, flat division and follows it through the AGN history.

**Molecular gas and dust only.** Every ISM axis that is *measured* here is $M_{\rm H_2}$ and $M_{\rm dust}$: $\log 1.36\,M_{\rm H_2}/M_\star$, $\log M_{\rm dust}/M_\star$, $\log M_{\rm dust}/M_{\rm H_2}$. The total gas $M_{\rm gas}$ survives in exactly two places, both of them structural rather than diagnostic — inside **SIMBA's own X-ray gate** $f_{\rm gas} = M_{\rm gas}/M_\star <$ `XRAY_FGAS_MAX` (a feedback criterion of the run, not a measurement of ours), and inside the **metal budget** $Z_{\rm gas}M_{\rm gas} + M_{\rm dust}$ of the dust-to-metal ratio, where the metals genuinely live in the whole gas reservoir.

**The sample, fixed once (Part 0).** Every quenched galaxy of the boxes' catalogue cache under the m25 rule, then three cuts declared in one place and inherited by **every** part of the notebook — $\log M_\star >$ `MASS_MIN` $= 10.25$, $z \leq$ `Z_MAX` $= 1$, age $\geq$ `AGE_MIN` $= 2$ Gyr — and then split by dust alone,

* **dusty QG** — $\log M_{\rm dust}/M_\star \geq$ `DUSTY_CUT`
* **dust-poor QG** — everything below it (zero dust sits at the `P9_FLOOR` spike)

Part 0 prints each cut's own toll, so the sample is auditable in one place and no part can quietly run on a different one. **The age cut is deliberate and not neutral**: the notebook originally took none, on the principle that age should be measured on a dust-selected pair rather than assumed, and the young under-dosed galaxies it removes are one of the three causes the *timing-explained* route is built from. `AGE_MIN = None` restores the original behaviour.

**The AGN history integrated in time, and the three routes (Part 2a).** One pass over every main-progenitor branch gives, per galaxy-anchor, the accretion integrals, the coupling integrated in time (gated at `P7_FEDD_ACT`, because $w_{\rm jet} = 1$ for *any* $f_{\rm Edd} \leq 0.02$ and a dead black hole must not score as maximally coupled), the **X-ray exposure** $E_x$ and the clock $t_{\rm quiet}$ since the channel last fired. Those three numbers plus the dust cut define the **three routes** that every later part is split by:

| route | what it is |
|---|---|
| **dusty, dosed & firing** | dusty, $E_x \geq$ `PF_EX`, $t_{\rm quiet} <$ `PF_TQ` — the channel had every chance and is firing *now*. **The AGN timing does not explain this one.** |
| **dusty, timing-explained** | dusty, but never armed / under-dosed / long since quiet |
| **dust-poor** | below `DUSTY_CUT` |

**The four AGN states (Part 2b).** For every snapshot of every branch the cache carries $M_\star$, ${\rm SFR}$, $M_{\rm gas}$, $M_{\rm BH}$, $f_{\rm Edd}$, $M_{\rm dust}$, $M_{\rm H_2}$, $Z_{\rm gas}$; from $f_{\rm gas}$, $\dot M_{\rm BH} = f_{\rm Edd} M_{\rm BH} / (\eta\,t_{\rm Sal})$ and $f_{\rm Edd}$ the snapshot is filed into **one of four exclusive AGN states**, which are SIMBA's own feedback branches read off the history:

| state | condition | what the run is doing |
|---|---|---|
| **radiative** | the BH accretes, no jets ($f_{\rm Edd} \geq 0.2$, or $M_{\rm BH}$ below the jet threshold) | quasar / wind mode |
| **X-ray choked** | jets ($M_{\rm BH} > 10^{7.5}$, $f_{\rm Edd} < 0.2$) but the X-ray condition fails | jets, no X-ray heating |
| **X-ray on** | full-velocity jets *and* $f_{\rm gas} <$ `XRAY_FGAS_MAX` | jets **+** X-ray heating |
| **dormant** | $\dot M_{\rm BH} = 0$ — the BH is not accreting (or there is none yet) | nothing |

**The tracks (Parts 3–4).** Every history is aligned on its own **SFT** (the start of the star-formation transition, `paper_ism_prediction_boxes` Part 6a), and per box (**m100, m50, m25**) the **median track** of each parameter is drawn for each of the **three routes**, with the **number of contributing sources as a histogram behind the tracks** so that a track is never read where the statistics have gone. Part 3 puts the parameters and the state occupancies on the $y$-axis; Part 4 replaces them by the **distance from the $0.2/t_{\rm H}$ quenching curve**, $\Delta_{\rm q} = \log\,{\rm sSFR} - \log\,(0.2/t_{\rm H})$ — the same tracks read as *how far from quenched* the route was, overall and while it sat in each AGN state. The panel that carries the physics is the **firing-dusty track against the dust-poor track**: two sets of galaxies dosed alike, differing only in whether the dust survived.

**Then the argument (Parts 8–13).** Part 8 reads the exposure as an *implication* (necessary, not sufficient); Part 9 is the figure — the four-way decomposition and the residual scored on every axis; Parts 11–12 put the routes' dust tracks on the SFT clock and then in the observable $t_{\rm sq}$ plane; Part 13 is the m25 close-up (histories, radial anatomy, K–S plane). **Parts 14–15 are ported from `noxray_analogues`** and re-cut on the routes — its Part 8, the critical-point sequences (ignition $\to$ jet $\to$ SFT $\to$ QT $\to$ end, core and outskirt), and its Part 10, the ISM grid with the **ALMA-C11 quenched galaxies on it** and a scorecard saying which route the observations actually sit on.

Everything here is a **pure read** of the `paper_ism_prediction_boxes.ipynb` products (the catalogue cache, the Part 6a classes and the per-box history caches) — except Part 13, which also reads the m25 notebook's own tables and gates on each of them. Products go to `output/box_resolution/agn_states/`.


In [ ]:
# ── Part 0 — configuration: the products read, the selection constants, the AGN-state definitions, the shared helpers ──
import os
import warnings
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from astropy.table import Table
from matplotlib.lines import Line2D
from scipy.stats import mannwhitneyu, spearmanr

IN  = os.path.join(os.getcwd(), "output", "box_resolution", "ism_prediction")    # the boxes notebook's products (this notebook only reads them)
OUT = os.path.join(os.getcwd(), "output", "box_resolution", "agn_states")
CACHE_FITS = os.path.join(IN, "ism_prediction_catalogue.fits")                   # its Part 1 cache: every galaxy with log M* > 9.5 per (box, anchor)
CLASS_CSV  = os.path.join(IN, "ism_prediction_agn_classes.csv")                  # its Part 6a: SFT / QT / tau_q per galaxy
HIST_FILE  = os.path.join(IN, "ism_prediction_histories_{box}.hdf5")             # its Part 6a: the main-branch chains (mstar, sfr, mgas, mbh, fedd, mdust, mh2, zgas)
os.makedirs(OUT, exist_ok=True)

BOXES = ["cis100", "cis50", "cis25"]        # sim100 / sim50 / sim25; append "cis50nox" to carry the channel-off run's WOULD-BE states as a fourth column
BOX_LABEL = {"cis100": "m100n1024 (sim100)", "cis50": "m50n512 s50 (sim50)", "cis25": "m25n512 (sim25)", "cis50nox": "m50n512 s50nox [X-ray off]"}
NOX_BOX = "cis50nox"

# ── the selection: the m25 quenched rule of paper_ism_prediction_boxes Part 0, verbatim, plus THE mass cut and THE dust split ──
MASS_FLOOR, PASSIVE_FACTOR, NGAS_MIN, NSTAR_MIN, DUST_TO_H2_MIN = 10.0, 0.2, 50, 500, 1e-15
MASS_MIN    = 10.25       # log M* strictly above this
# THE SAMPLE, declared ONCE and applied ONCE (right after the quenched rule below), so that every part of the notebook —
# the census, the tracks, the dose, the decomposition, the m25 close-ups and the observed grid — runs on the SAME galaxies.
# Before this existed, Parts 8-12 inherited a z <= 0.5 cut from the dose engine while Parts 13-15 used every anchor, and
# the m25 close-up silently saw 37 of its 178 galaxies. Set any of them to None to drop that cut.
Z_MAX       = 8         # anchors at or below this redshift (the ladder is z = 0.31, 0.49, 0.70, 0.85, 0.99, 1.15, ... so this takes the first five)
AGE_MIN     = 3         # Gyr, the mass-weighted stellar age at the anchor: at or above this
AGE_MAX     = None        # Gyr, an upper bound if wanted (None = none); the observed ALMA-C11 QGs run 2.95-6.84 Gyr
DUSTY_CUT   = -4.        # log M_dust/M*: at or above = dusty QG, below = pure QG (the split, not a sample cut)
P9_FLOOR    = -8.5        # zero dust is kept at this log (the end point of the removal, never dropped)
SSFR_FLOOR  = 1e-15
HE, R_PROJ_OVER_3D = 1.36, 1.
KEY = ["box", "snap", "gal_id"]

POPS = {"dusty": dict(color="#d7191c", lw=2.6, ls="-",  label=rf"dusty QG  ($\log M_{{\rm dust}}/M_\star \geq {DUSTY_CUT:g}$)"),
        "pure":  dict(color="#2c7bb6", lw=2.2, ls="-",  label=rf"pure QG  ($\log M_{{\rm dust}}/M_\star < {DUSTY_CUT:g}$)")}

# ── the AGN states: SIMBA's own feedback branches evaluated at every snapshot of the main branch ──
JET_LOGMBH, JET_FEDD = 7.5, 0.2       # SIMBA's jet criterion: M_BH above 10^7.5 AND f_Edd below 0.2 (the jet velocity then ramps as w_jet)
W_FULL        = 0.999                 # "full-velocity jet": w_jet = clip(log10(0.2 / f_Edd), 0, 1) at/above this, i.e. f_Edd <= 0.02
XRAY_FGAS_MAX = float(os.environ.get("XRAY_FGAS_MAX", "0.2"))   # SIMBA's X-ray condition, M_gas/M_* below this (Dave+19); noxray_analogues explored a 0.6 gate — set the env var to reproduce it
ETA_RAD       = 0.1                   # radiative efficiency behind Mdot_BH = f_Edd L_Edd / (eta c^2)
MDOT_PER_MBH  = 1.26e38 / (ETA_RAD * 8.98755e20) / 1.98892e33 * 3.15576e7        # Msun/yr of accretion per Msun of BH at f_Edd = 1 (= 1 / 4.5e7 yr)
FEDD_FLOOR, MDOT_FLOOR, FGAS_FLOOR = 1e-6, 1e-7, 1e-3                            # the log floors of the tracks (a dormant BH lands on them, it is not dropped)

STATES = [("radiative", "#9467bd", r"radiative — accreting, no jets"),
          ("choked",    "#2ca02c", rf"X-ray choked — jets, $f_{{\rm gas}} \geq {XRAY_FGAS_MAX:g}$"),
          ("xray",      "#1f77b4", rf"X-ray on — full jets $+$ $f_{{\rm gas}} < {XRAY_FGAS_MAX:g}$"),
          ("dormant",   "0.55",    r"dormant — $\dot M_{\rm BH} = 0$")]
SKEYS  = [s for s, _, _ in STATES]
SCOL   = {s: c for s, c, _ in STATES}
SLAB   = {s: l for s, _, l in STATES}

# ── the clock and the estimator settings ──
TG     = np.arange(-3.0, 4.001, 0.25)      # the SFT clock: t - t_SFT [Gyr], the grid every track is interpolated onto
NMIN   = 3                                 # an epoch drawn only where this many galaxies contribute (3, not 5: the tracks are now
                                           # split on the THREE ROUTES of Part 2a and the residual route is a handful of chains per box)
NBOOT  = int(os.environ.get("AGN_NBOOT", "500"))   # the bootstrap band; the runtime driver of Parts 3-4, drop it for a smoke test
PAPER_DPI = 300
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11, "legend.fontsize": 9, "xtick.labelsize": 9.5,
                     "ytick.labelsize": 9.5, "axes.titleweight": "bold", "pdf.fonttype": 42})


def need(path, made_by):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} is missing: {made_by}")
    return path


def paper_save(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUT, f"{stem}.{ext}"), dpi=PAPER_DPI, bbox_inches="tight")
    print("figure ->", os.path.join(OUT, f"{stem}.{{png,pdf}}"))


def mw_auc(x, m1):
    """x over two populations (m1 = population 1): AUC = P(x_1 > x_0), the two-sided Mann-Whitney p, the two finite counts."""
    x = np.asarray(x, float); m1 = np.asarray(m1, bool)
    a, b = x[m1 & np.isfinite(x)], x[~m1 & np.isfinite(x)]
    if len(a) < 3 or len(b) < 3 or np.nanstd(np.concatenate([a, b])) == 0:
        return np.nan, np.nan, len(a), len(b)
    u = mannwhitneyu(a, b, alternative="two-sided")
    return float(u.statistic) / (len(a) * len(b)), float(u.pvalue), len(a), len(b)


def med_band(A, stat="median", seed=11):
    """A (n_gal, n_t) stack -> the central track, its bootstrap 16-84 band and the per-epoch finite count; epochs under NMIN are blanked."""
    A = np.asarray(A, float)
    if A.ndim != 2 or not len(A):
        return (np.full(len(TG), np.nan),) * 3 + (np.zeros(len(TG), int),)
    n_t = np.isfinite(A).sum(axis=0)
    f = np.nanmedian if stat == "median" else np.nanmean
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        cent = f(A, axis=0)
        rng = np.random.default_rng(seed)
        bs = np.array([f(A[rng.integers(0, len(A), len(A))], axis=0) for _ in range(NBOOT)])
        lo, hi = np.nanpercentile(bs, 16, axis=0), np.nanpercentile(bs, 84, axis=0)
    ok = n_t >= NMIN
    return np.where(ok, cent, np.nan), np.where(ok, lo, np.nan), np.where(ok, hi, np.nan), n_t


def count_bg(ax, counts_by_pop, styles=None, headroom=2.8):
    """The source count per epoch drawn as a histogram BEHIND the tracks (one twin axis, every series on the same scale).
    `styles` is the {series: {"color": ...}} map — Part 2a's GRPS once it exists, POPS before that."""
    st = styles if styles is not None else globals().get("GRPS", POPS)
    axb = ax.twinx(); axb.set_zorder(ax.get_zorder() - 1); ax.patch.set_visible(False)
    top = max(1.0, max((np.nanmax(c) if len(c) else 0.0) for c in counts_by_pop.values()))
    for p, c in counts_by_pop.items():
        axb.bar(TG, c, width=0.92 * (TG[1] - TG[0]), color=st[p]["color"], alpha=0.13, lw=0, align="center", zorder=1)
    axb.set_ylim(0, top * headroom); axb.set_yticks([])
    ax.text(0.985, 0.965, f"N $\\leq$ {int(top)}", transform=ax.transAxes, ha="right", va="top", fontsize=7, color="0.45")
    return axb


def to_grid(trel, y):
    """One galaxy's track interpolated onto TG; nothing is extrapolated past the sampled range."""
    ok = np.isfinite(trel) & np.isfinite(y)
    return np.interp(TG, trel[ok], y[ok], left=np.nan, right=np.nan) if ok.sum() >= 3 else np.full(len(TG), np.nan)


def agn_states(mstar, mgas, mbh, fedd):
    """The four exclusive AGN states of an (n_t, n_gal) history block, plus w_jet, f_gas, Mdot_BH and the diagnostic sub-masks.
    A cell where the galaxy does not exist is NaN everywhere; a cell whose BH columns are absent from the catalogue counts as
    dormant (the boxes notebook's own convention: a missing BH is w_jet = 0 wherever the galaxy exists).
    NOTE f_gas = M_gas/M_* is SIMBA's OWN X-ray gate (Dave+19) and stays on the total gas — it is a feedback criterion, not an
    ISM diagnostic; the total gas appears nowhere else in this notebook except inside the metal budget of the dust-to-metal ratio."""
    with np.errstate(all="ignore"):
        exist = np.isfinite(mstar) & (mstar > 0)
        fgas  = np.where(exist, mgas / np.where(exist, mstar, np.nan), np.nan)
        bh    = np.isfinite(mbh) & (mbh > 0)
        acc   = exist & bh & np.isfinite(fedd) & (fedd > 0)                       # the BH is accreting: Mdot_BH > 0
        big   = bh & (mbh > 10 ** JET_LOGMBH)
        wjet  = np.where(acc & big, np.clip(np.log10(JET_FEDD / np.clip(fedd, 1e-12, None)), 0.0, 1.0), 0.0)
        mdot  = np.where(acc, MDOT_PER_MBH * np.nan_to_num(fedd) * np.nan_to_num(mbh), 0.0)
        jet   = acc & big & (fedd < JET_FEDD)                                     # SIMBA's jet criterion
        xray  = jet & (wjet >= W_FULL) & (fgas < XRAY_FGAS_MAX)                   # full velocity AND gas-poor: the X-ray channel fires
        S = {"xray": xray, "choked": jet & ~xray, "radiative": acc & ~jet, "dormant": exist & ~acc}
        D = {"nobh": exist & ~bh, "pjet": jet & ~xray & (wjet < W_FULL), "fjrich": jet & ~xray & (wjet >= W_FULL),
             "quasar": (acc & ~jet) & np.isfinite(fedd) & (fedd >= JET_FEDD), "lowmbh": (acc & ~jet) & ~big}
    f = lambda m: np.where(exist, m.astype(float), np.nan)
    return ({k: f(v) for k, v in S.items()}, {k: f(v) for k, v in D.items()},
            dict(exist=exist, fgas=fgas, wjet=np.where(exist, wjet, np.nan), mdot=np.where(exist, mdot, np.nan)))

# ── the catalogue cache -> the quenched sample (the boxes notebook's Part 1 rule) -> the dusty / pure split ──
CAT = Table.read(need(CACHE_FITS, "paper_ism_prediction_boxes Part 1")).to_pandas()
CAT["box"] = [(v.decode() if isinstance(v, bytes) else str(v)).strip() for v in CAT["box"]]
_ms, _sfr = CAT["mstar"].to_numpy(float), CAT["sfr"].to_numpy(float)
with np.errstate(divide="ignore", invalid="ignore"):
    _ssfr = np.where(_ms > 0, _sfr / _ms, np.nan)
    _lm   = np.log10(np.where(_ms > 0, _ms, np.nan))
_quenched = ((_lm > MASS_FLOOR) & (_ssfr < PASSIVE_FACTOR / (CAT["t_H_gyr"].to_numpy(float) * 1e9))
             & (CAT["ngas"].to_numpy(float) >= NGAS_MIN) & (CAT["nstar"].to_numpy(float) >= NSTAR_MIN)
             & (CAT["mdust"].to_numpy(float) >= DUST_TO_H2_MIN * CAT["mh2"].to_numpy(float)))
QM = CAT[_quenched].copy()
with np.errstate(divide="ignore", invalid="ignore"):
    ms = np.where(QM["mstar"].to_numpy(float) > 0, QM["mstar"].to_numpy(float), np.nan)
    QM["log_mstar"] = np.log10(ms)
    fd = QM["mdust"].to_numpy(float) / ms
    QM["lfd"]   = np.where(fd > 0, np.log10(np.where(fd > 0, fd, 1.0)), P9_FLOOR)       # zero dust at the floor, never dropped
    QM["lfgas"] = np.log10(np.where(QM["mgas"].to_numpy(float) > 0, QM["mgas"].to_numpy(float) / ms, np.nan))
    QM["lfh2"]  = np.log10(np.where(QM["mh2"].to_numpy(float) > 0, HE * QM["mh2"].to_numpy(float) / ms, np.nan))
    QM["lssfr"] = np.log10(np.clip(np.maximum(QM["sfr"].to_numpy(float), 0.0) / ms, SSFR_FLOOR, None))
    QM["re"]    = R_PROJ_OVER_3D * QM["r_half_star"].to_numpy(float)
    QM["lsig"]  = np.log10(np.where(QM["re"] > 0, 0.5 * ms / (np.pi * QM["re"] ** 2), np.nan))
    mbh = QM["mbh"].to_numpy(float)
    QM["log_mbh"] = np.where(mbh > 0, np.log10(np.where(mbh > 0, mbh, 1.0)), np.nan)

# THE mass, redshift and age cuts, stated once and applied once — every part downstream inherits them
_n_before = len(QM)
_cuts = [(f"log M* > {MASS_MIN:g}", QM["log_mstar"].to_numpy(float) > MASS_MIN)]
if Z_MAX is not None:
    _cuts.append((f"z <= {Z_MAX:g}", QM["z"].to_numpy(float) <= float(Z_MAX) + 1e-9))
if AGE_MIN is not None:
    _cuts.append((f"age >= {AGE_MIN:g} Gyr", QM["age"].to_numpy(float) >= float(AGE_MIN)))
if AGE_MAX is not None:
    _cuts.append((f"age <= {AGE_MAX:g} Gyr", QM["age"].to_numpy(float) <= float(AGE_MAX)))
_keep, _drop = np.ones(len(QM), bool), []
for _lab, _m in _cuts:                                    # each cut's own toll, in the order it is applied
    _drop.append((_lab, int((_keep & ~_m).sum())))
    _keep &= _m
QM = QM[_keep].reset_index(drop=True)
QM["pop"] = np.where(QM["lfd"].to_numpy(float) >= DUSTY_CUT, "dusty", "pure")

CLS = pd.read_csv(need(CLASS_CSV, "paper_ism_prediction_boxes Part 6a"))
CLS_COLS = [c for c in ["t_sft", "t_qt", "tau_q", "w_pre", "agn_class", "agn_class3", "agn_onset"] if c in CLS.columns]   # the timings Parts 2-4 need + the coupling / onset labels Part 5 rows by
QM = QM.merge(CLS[KEY + CLS_COLS], on=KEY, how="left")                                # left join: a box / anchor without histories keeps NaN and drops out of Parts 2-5

print("THE SAMPLE (declared in Part 0, inherited by every part): the m25 quenched rule on the catalogue cache, then "
      + ", ".join(l for l, _ in _cuts) + f"  ->  {_n_before} -> {len(QM)} rows"
      + ("  (" + ", ".join(f"{l}: -{n}" for l, n in _drop if n) + ")" if any(n for _, n in _drop) else ""))
print("  per box: " + ", ".join(f"{b} {int((QM['box'] == b).sum())}" for b in BOXES if (QM['box'] == b).any())
      + f";  anchors z = {QM['z'].min():.2f}-{QM['z'].max():.2f}, age = {QM['age'].min():.2f}-{QM['age'].max():.2f} Gyr")
print(f"  then the dust split at log M_dust/M* = {DUSTY_CUT:g}: dusty {int((QM['pop'] == 'dusty').sum())}, "
      f"pure {int((QM['pop'] == 'pure').sum())}.  NOTE age is now a CUT as well as a contrast — see the notes.")
print(f"the X-ray condition in force: f_gas < {XRAY_FGAS_MAX:g} (env XRAY_FGAS_MAX); full jet at w_jet >= {W_FULL:g}, i.e. f_Edd <= {JET_FEDD * 10 ** -W_FULL:.3f}")

# ── which boxes have a histories cache, and what that cache carries (Part 2a integrates it, Part 2b tracks it) ──
HAVE_MD, HAVE_MH2, BOXES_OK, CHAIN_COV = {}, {}, [], {}
for _b in BOXES:
    _p = HIST_FILE.format(box=_b)
    if not os.path.exists(_p):
        print(f"{_b}: {_p} missing — paper_ism_prediction_boxes Part 6a builds it (cluster); skipped"); continue
    with h5py.File(_p, "r") as _f:
        _g0 = _f[next(iter(_f))]
        HAVE_MD[_b], HAVE_MH2[_b] = "mdust" in _g0, "mh2" in _g0
        _sub, _nh = QM[QM["box"] == _b], 0                        # how much of THE SAMPLE the cache actually covers
        for _a, _Sa in _sub.groupby("snap"):
            _k = f"snap{int(_a):03d}"
            if _k in _f:
                _ids = set(int(_v) for _v in _f[_k]["gal_ids"][:])
                _nh += int(sum(int(_x) in _ids for _x in _Sa["gal_id"]))
        CHAIN_COV[_b] = (_nh, len(_sub))
    BOXES_OK.append(_b)
print("histories caches: " + ", ".join(f"{b} (mdust {'y' if HAVE_MD[b] else 'n'}, mh2 {'y' if HAVE_MH2[b] else 'n'})" for b in BOXES_OK))
print("THE EFFECTIVE SAMPLE of every part that reads histories (2a onwards): "
      + ", ".join(f"{b} {CHAIN_COV[b][0]}/{CHAIN_COV[b][1]} ({100 * CHAIN_COV[b][0] / max(1, CHAIN_COV[b][1]):.0f} %)" for b in BOXES_OK))
if any(CHAIN_COV[b][0] < 0.9 * CHAIN_COV[b][1] for b in BOXES_OK):
    print("  ^ THE DECLARED AND THE EFFECTIVE SAMPLE HAVE DRIFTED APART. The missing galaxies have no main branch: Part 6a of the boxes\n"
          "    notebook built the caches for ITS quenched pool, with ITS constants (PASSIVE_FACTOR 0.2, DUST_TO_H2_MIN 1e-4). Whichever\n"
          "    of the constants above is looser here decides WHICH galaxies are missing - so the gap is not random, it is along that axis\n"
          "    (DUST_TO_H2_MIN is the dust axis itself). Either match the boxes notebook's constants or rebuild its histories with these.")


## Part 1 — the selection: one sample, one dust cut

The census of what the cuts actually select, per box. **The sample is fixed once, in Part 0** — the m25 quenched rule, then $\log M_\star >$ `MASS_MIN`, $z \leq$ `Z_MAX` and age $\geq$ `AGE_MIN` — and every part of the notebook, from the tracks to the m25 close-ups to the observed grid, runs on exactly those galaxies. Part 0 prints each cut's own toll so the sample is auditable in one place.

Then the single split at $\log M_{\rm dust}/M_\star =$ `DUSTY_CUT`, and this cell reports what the two populations look like in the quantities the split does *not* see.

**On the age cut.** The notebook originally took no age cut at all, on the principle that if the dusty and the dust-poor population turn out to be an old and a young one, that should be a *measured* property of a dust-selected pair rather than a definition. `AGE_MIN` gives that up deliberately, in exchange for a sample that is uniform across every part. **It is not calibrated on the observed sample**, and cannot be: the ALMA-C11 ages (2.95–6.84 Gyr) are CIGALE `sfhdelayedbq` *onset* ages — the time since star formation began — while `age` here is caesar's **mass-weighted** age. For a delayed-$\tau$ SFH the mass-weighted age sits roughly $2\tau$ below the onset age, so the observed numbers are 0.6–1.8 Gyr high as a like-for-like reference. `AGE_MIN` is a cut on the simulation's own mass-weighted age and is internally consistent; just do not read it as "the observed floor". **It is not a neutral cut**: the young under-dosed galaxies it removes are one of the three causes the *timing-explained* route is built from, so that route shrinks — sharply in m25 — and the residual dose-response of Part 8 weakens with it. Set `AGE_MIN = None` in Part 0 to get the original behaviour back; age is still reported here as a contrast, now measured *within* the cut.

The figure is the check itself — the $\log M_{\rm dust}/M_\star$ distribution with the cut and the zero-dust spike at `P9_FLOOR`, and the $\log M_\star$ distribution with the mass cut drawn on it, per box. The last column of the printed table is the one that matters for Parts 2–4: **how many of each population carry an SFT**, i.e. a history to align on.


In [ ]:
# ── Part 1 — the census of the two populations and the visual check of the two cuts ──
Q1 = [("age", "age [Gyr]"), ("log_mstar", "log M*"), ("z", "redshift"), ("lfd", "log M_dust/M*"),
      ("lfh2", "log 1.36 M_H2/M*"), ("lssfr", "log sSFR"), ("lsig", "log Sigma_e"), ("kappa_gas", "kappa_rot gas"),
      ("kappa_star", "kappa_rot star"), ("log_mbh", "log M_BH"), ("tau_q", "tau_q [Gyr]")]

rows1 = []
print(f"the two populations per box (medians; AUC = P(dusty > pure), '*' = Mann-Whitney p < 0.01)\n")
for b in BOXES:
    g = QM[QM["box"] == b]
    if not len(g):
        print(f"  {b}: absent from the catalogue cache\n"); continue
    mD, mP = g["pop"] == "dusty", g["pop"] == "pure"
    print(f"  {'=' * 104}\n  {BOX_LABEL[b]}: {len(g)} quenched galaxies over {g['snap'].nunique()} anchors — "
          f"dusty {int(mD.sum())} ({100 * mD.mean():.0f} %), pure {int(mP.sum())}; with an SFT: dusty {int(np.isfinite(g.loc[mD, 't_sft']).sum())}, pure {int(np.isfinite(g.loc[mP, 't_sft']).sum())}; "
          f"satellites {100 * (1 - g['central'].mean()):.0f} %")
    print(f"  {'quantity':22s} {'dusty':>22s} {'pure':>23s} {'AUC':>7s} {'N d/p':>12s}")
    for q, lab in Q1:
        if q not in g.columns:
            continue
        vd, vp = g.loc[mD, q].to_numpy(float), g.loc[mP, q].to_numpy(float)
        auc, pv, n1, n0 = mw_auc(np.r_[vd, vp], np.r_[np.ones(len(vd), bool), np.zeros(len(vp), bool)])
        f3 = lambda v: (np.nan, np.nan, np.nan) if not np.isfinite(v).any() else tuple(np.nanpercentile(v, [50, 16, 84]))
        md, lo_d, hi_d = f3(vd); mp, lo_p, hi_p = f3(vp)
        print(f"  {lab:22s} {md:7.2f} [{lo_d:6.2f},{hi_d:6.2f}] {mp:8.2f} [{lo_p:6.2f},{hi_p:6.2f}] {auc:7.2f}{'*' if np.isfinite(pv) and pv < 0.01 else ' '} {n1:5d}/{n0:5d}")
        rows1.append(dict(box=b, quantity=q, med_dusty=md, lo_dusty=lo_d, hi_dusty=hi_d, med_pure=mp, lo_pure=lo_p, hi_pure=hi_p,
                          auc=auc, p=pv, n_dusty=n1, n_pure=n0))
    print()

C1 = pd.DataFrame(rows1)
C1.to_csv(os.path.join(OUT, "agn_states_census.csv"), index=False)
print(f"census -> {os.path.join(OUT, 'agn_states_census.csv')}")

# the two cuts drawn: the dust axis the split is made on, and the mass axis the sample is cut on (before the cut, so the cut is visible)
QALL = CAT[_quenched].copy()
with np.errstate(divide="ignore", invalid="ignore"):
    _msa = np.where(QALL["mstar"].to_numpy(float) > 0, QALL["mstar"].to_numpy(float), np.nan)
    QALL["log_mstar"] = np.log10(_msa)
    _fda = QALL["mdust"].to_numpy(float) / _msa
    QALL["lfd"] = np.where(_fda > 0, np.log10(np.where(_fda > 0, _fda, 1.0)), P9_FLOOR)

fig, axs = plt.subplots(2, len(BOXES), figsize=(4.2 * len(BOXES), 5.6), squeeze=False)
for ci, b in enumerate(BOXES):
    g, ga = QM[QM["box"] == b], QALL[QALL["box"] == b]
    ax = axs[0, ci]
    ax.hist(g["lfd"].to_numpy(float), bins=np.arange(P9_FLOOR - 0.25, -1.749, 0.25), color="0.75", edgecolor="0.35", lw=0.6)
    ax.axvline(DUSTY_CUT, color="k", lw=1.6, ls="--")
    for pn, m in (("dusty", g["pop"] == "dusty"), ("pure", g["pop"] == "pure")):
        ax.hist(g.loc[m, "lfd"].to_numpy(float), bins=np.arange(P9_FLOOR - 0.25, -1.749, 0.25), color=POPS[pn]["color"], alpha=0.55, lw=0)
    ax.set_title(BOX_LABEL[b], fontsize=10)
    ax.set_xlabel(r"$\log M_{\rm dust}/M_\star$"); ax.set_ylabel("galaxies" if ci == 0 else "")
    ax.text(0.03, 0.94, f"dusty {100 * (g['pop'] == 'dusty').mean():.0f} %\nof {len(g)}", transform=ax.transAxes, va="top", fontsize=8.5)
    ax.grid(alpha=0.25, lw=0.5)
    ax = axs[1, ci]
    bins_m = np.arange(9.9, 12.61, 0.1)
    ax.hist(ga["log_mstar"].to_numpy(float), bins=bins_m, color="0.85", edgecolor="0.6", lw=0.5, label="quenched, before the mass cut")
    for pn, m in (("dusty", g["pop"] == "dusty"), ("pure", g["pop"] == "pure")):
        ax.hist(g.loc[m, "log_mstar"].to_numpy(float), bins=bins_m, color=POPS[pn]["color"], alpha=0.6, lw=0)
    ax.axvline(MASS_MIN, color="k", lw=1.6, ls="--")
    ax.set_xlabel(r"$\log M_\star$ [$M_\odot$]"); ax.set_ylabel("galaxies" if ci == 0 else "")
    ax.grid(alpha=0.25, lw=0.5)
_h1 = [Line2D([], [], color=POPS[p]["color"], lw=6, alpha=0.6, label=POPS[p]["label"]) for p in POPS]
_h1 += [Line2D([], [], color="0.8", lw=6, label="quenched sample before the cut"), Line2D([], [], color="k", lw=1.6, ls="--", label="the cut")]
fig.legend(handles=_h1, loc="lower center", ncol=4, frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, -0.03))
fig.suptitle(rf"the selection: quenched, $\log M_\star > {MASS_MIN:g}$, split at $\log M_{{\rm dust}}/M_\star = {DUSTY_CUT:g}$ — no age division", y=0.99, fontsize=11)
fig.tight_layout(rect=(0, 0.05, 1, 0.965))
paper_save(fig, "agn_states_selection")
plt.show()

## Part 2 — the AGN history: the integrals and the three routes (2a), then the four states per snapshot (2b)

### Part 2a — what the black hole did, integrated in time, and the routes the rest of the notebook is split by

One pass over each box's main-branch chains. Per galaxy-anchor it records the accretion integrals ($\int \dot M_{\rm BH}\,{\rm d}t$, the same restricted to the X-ray snapshots, the coupling-weighted $\int w_{\rm jet}\dot M_{\rm BH}\,{\rm d}t$), the coupling integrated in time with and without the accretion floor `P7_FEDD_ACT` (w_jet $= 1$ for *any* $f_{\rm Edd} \leq 0.02$, so a dead black hole scores maximal coupling for free — the floor is what stops that counting), the **X-ray exposure** $E_x = \int_{\rm X\text{-}ray}{\rm d}t$, and the clocks $t_{\rm quiet} = t_{\rm anchor} - t_{\rm last\ X-ray}$ and $t_{\rm since\ acc}$. The gas metallicity at the anchor comes from the branch's last sampled snapshot (the catalogue cache carries none).

The integrals are built for **every** anchor, because Parts 2b–4 need a route for every tracked chain; `DOSE` — the frame Parts 8–13 read — is the $z \leq$ `P7_ZMAX` subset of that, exactly the sample those parts have always used.

From $E_x$, $t_{\rm quiet}$ and the dust cut come the **three routes**:

| route | what it is |
|---|---|
| **dusty, dosed & firing** | above `DUSTY_CUT`, $E_x \geq$ `PF_EX` **and** $t_{\rm quiet} <$ `PF_TQ` — fully dosed, being dosed *now*, and dusty anyway. The AGN timing does not explain it. |
| **dusty, timing-explained** | above `DUSTY_CUT` but never armed, under-dosed, or dosed and then off for `PF_TQ` or more |
| **dust-poor** | below `DUSTY_CUT` |

Part 9 draws the four-way version of the same rule (the timing-explained route split into its three causes); everything else in the notebook uses the three.

### Part 2b — the four exclusive AGN states per snapshot

For every anchor of every box the Part 6a cache holds the main-progenitor chain of each selected galaxy — $M_\star$, ${\rm SFR}$, $M_{\rm gas}$, $M_{\rm BH}$, $f_{\rm Edd}$, $M_{\rm dust}$, $M_{\rm H_2}$, $Z_{\rm gas}$ — at every snapshot down to the end of the tracked history. From those columns alone:

$$f_{\rm gas} = \frac{M_{\rm gas}}{M_\star}, \qquad \dot M_{\rm BH} = \frac{f_{\rm Edd}\,M_{\rm BH}}{4.5 \times 10^{7}\,{\rm yr}}\ \ (\eta = 0.1), \qquad w_{\rm jet} = {\rm clip}\Big[\log_{10}\frac{0.2}{f_{\rm Edd}},\,0,\,1\Big]\ \ \text{for } M_{\rm BH} > 10^{7.5}$$

and the snapshot is filed into exactly one state, in this order:

1. **dormant** — the galaxy exists but $\dot M_{\rm BH} = 0$: no BH yet, or a BH that has stopped accreting;
2. **X-ray on** — $M_{\rm BH} > 10^{7.5}$, the jet is at full velocity ($w_{\rm jet} \geq$ `W_FULL`, i.e. $f_{\rm Edd} \lesssim 0.02$) **and** $f_{\rm gas} <$ `XRAY_FGAS_MAX`: SIMBA injects the X-ray energy;
3. **X-ray choked** — the jet criterion holds but one of the two X-ray conditions fails (a partial-velocity jet, or a gas-rich host): jets, no X-ray heating;
4. **radiative** — the BH accretes but does not jet, either because $f_{\rm Edd} \geq 0.2$ (the quasar / wind mode) or because $M_{\rm BH}$ is still below the jet threshold.

$f_{\rm gas} = M_{\rm gas}/M_\star$ here is **SIMBA's own X-ray gate** (Davé+19), a feedback criterion — it is the one place the total gas is still used as a *condition*, and the only other place the total gas appears anywhere in the notebook is inside the metal budget of the dust-to-metal ratio. Every ISM axis that is *measured* is molecular: $M_{\rm H_2}$, $M_{\rm dust}/M_{\rm H_2}$.

The printed census splits **choked** into its two causes and **radiative** into its two, and reports what fraction of **dormant** is "no BH at all" rather than "a BH that stopped" — the four states stay four, but nothing is hidden inside them.

Each chain is then put on the **SFT clock** $t - t_{\rm SFT}$ and interpolated onto `TG`; a galaxy without an SFT (no history built, or no quenching event found) simply does not enter, and neither does one Part 2a could not integrate. Alongside the states the same interpolation carries $\log 1.36\,M_{\rm H_2}/M_\star$, $\log \dot M_{\rm BH}$, $\log f_{\rm Edd}$, $\log M_{\rm BH}$, $\log$ sSFR, $\log M_{\rm dust}/M_\star$ and the **quenching distance** $\Delta_{\rm q} = \log {\rm sSFR} - \log (0.2/t_{\rm H})$ that Part 4 reads — with $t_{\rm H}$ the cosmic time of that very snapshot, so the $0.2/t_{\rm H}$ curve is evaluated where the galaxy actually was. Every stack is keyed on the **three routes**, not on dusty/pure.


In [ ]:
# ── Part 2a — the AGN history integrated in time, and THE THREE ROUTES everything below is split by ──
# One pass over each box's main-branch chains: per galaxy-anchor, how much was accreted, how long the jet was coupled,
# how long the X-ray channel fired (E_x) and how long ago it stopped (t_quiet).  This used to be Part 7, drawn after the
# tracks; it sits here because the three routes are DEFINED from it and Parts 2b-4 are now split on them, not on dusty/pure.
P7_ZMAX     = None        # SUPERSEDED by Part 0's Z_MAX, which cuts the sample once for the whole notebook. Kept as an EXTRA
                          # restriction on the frame Parts 8-13 read (None = none): set it to tighten those parts alone, e.g. to
                          # compare exposure clocks over a narrower range. DOSE_ALL is always every routed anchor of Part 0's sample
P7_WINDOW   = None        # the integration window before the anchor: None = the WHOLE tracked branch; a float = that many Gyr back; "sft" = since t_SFT
P7_FEDD_ACT = float(os.environ.get("P7_FEDD_ACT", "1e-4"))   # THE caveat knob. w_jet = clip(log10(0.2/f_Edd), 0, 1) is 1 for ANY f_Edd <= 0.02, so a black
                                                             # hole that has all but stopped scores maximal coupling for free. Below this floor the BH counts
                                                             # as not accreting: the coupling integrals do not advance and the exposure is flagged as dead
                                                             # time — no accretion is NOT extreme coupling

# ── the three routes: what the AGN timing can and cannot account for.  Part 9 draws the four-way version of the same rule ──
PF_EX = 1.0               # 'fully dosed': the integrated X-ray exposure, in Gyr, above which the channel has had its chance
PF_TQ = 0.5               # 'still firing': t_quiet below this, in Gyr — the galaxy is being dosed AT the anchor, not recovering from an old episode
CATS = [("never armed",            "#7b3294", "the channel never fired"),
        ("under-dosed",            "#b07aa1", f"it fired for less than {PF_EX:g} Gyr"),
        ("dosed, then quiet",      "#4daf4a", f"dosed, then off for >= {PF_TQ:g} Gyr"),
        ("dosed and still firing", "#d7191c", "dosed, firing now, dusty anyway")]
CCOL = {c: k for c, k, _ in CATS}
ROUTES = [("dusty, dosed & firing",   "#d7191c", 2.7, "-",  "dusty — dosed and STILL FIRING (the unexplained residual)"),
          ("dusty, timing-explained", "#f0a860", 2.4, "-",  "dusty — never armed, under-dosed or gone quiet (the timing explains it)"),
          ("dust-poor",               "#2c7bb6", 2.2, "-",  rf"dust-poor  ($\log M_{{\rm dust}}/M_\star < {DUSTY_CUT:g}$)")]
GRPS = {g: dict(color=c, lw=lw, ls=ls, label=lab) for g, c, lw, ls, lab in ROUTES}
G3   = list(GRPS)
GSHORT = {G3[0]: "firing", G3[1]: "timing", G3[2]: "dust-poor"}      # the short tags the printed tables use
GPAIRS = [(G3[0], G3[2]), (G3[1], G3[2]), (G3[0], G3[1])]      # the per-epoch contrasts scored in Parts 2b-4


def route_cat(d):
    """The four-way AGN-timing category of a DOSE frame: never armed / under-dosed / dosed-then-quiet / dosed and still firing."""
    never = np.asarray(d["n_x"], int) == 0
    ex, tq = np.asarray(d["e_x"], float), np.asarray(d["t_quiet"], float)
    return np.where(never, CATS[0][0], np.where(ex < PF_EX, CATS[1][0], np.where(tq >= PF_TQ, CATS[2][0], CATS[3][0])))


def route_grp(d):
    """The three routes: the four categories collapsed onto the question the notebook actually asks — of the DUSTY galaxies,
    which ones the AGN timing accounts for (never armed / under-dosed / gone quiet) and which ones it does not."""
    dusty = np.asarray(d["lfd"], float) >= DUSTY_CUT
    return np.where(dusty & (np.asarray(d["cat"]) == CATS[3][0]), G3[0], np.where(dusty, G3[1], G3[2]))


p7log = lambda v: np.log10(np.where(np.asarray(v, float) > 0, np.asarray(v, float), np.nan))

# ── the integrals: one record per galaxy-anchor, EVERY anchor (the z cut is applied afterwards, to DOSE only) ──
P7S = QM[QM["box"].isin(BOXES_OK) & np.isfinite(QM["lfd"])].copy()      # QM is already Part 0's sample: mass, redshift and age cut
p7rows, p7_nochain = [], {b: 0 for b in BOXES_OK}
for b in BOXES_OK:
    sub = P7S[P7S["box"] == b]
    want = {(int(s), int(i)): r for (s, i), r in zip(zip(sub["snap"], sub["gal_id"]), sub.to_dict("records"))}
    seen = set()
    with h5py.File(HIST_FILE.format(box=b), "r") as f:
        for grpname in f:
            a = int(grpname.replace("snap", ""))
            if not any(k[0] == a for k in want):
                continue
            gh = f[grpname]
            t = gh["t_yr"][:] / 1e9; order = np.argsort(t); t = t[order]
            if len(t) < 2:
                continue
            ms7, mg7, mbh7, fedd7 = (gh[c][:][order] for c in ("mstar", "mgas", "mbh", "fedd"))
            zg7 = gh["zgas"][:][order] if "zgas" in gh else None           # the gas metallicity: the catalogue cache has none, the branch does
            S7, _, X7 = agn_states(ms7, mg7, mbh7, fedd7)                  # the Part 0 rule verbatim, along the branch
            dtw  = np.gradient(t)                                          # the sampled width of each snapshot [Gyr]
            xr   = np.nan_to_num(S7["xray"]) > 0.5
            jt   = xr | (np.nan_to_num(S7["choked"]) > 0.5)                # SIMBA's jet criterion, gate open or shut
            wj   = np.nan_to_num(X7["wjet"])                               # 0 wherever the BH is not accreting or sits under the jet mass threshold
            mdot = np.nan_to_num(X7["mdot"])                               # [Msun/yr], 0 where not accreting
            act  = X7["exist"] & np.isfinite(fedd7) & (fedd7 >= P7_FEDD_ACT)   # THE floor: the BH is genuinely accreting
            for j, gid in enumerate(gh["gal_ids"][:]):
                key = (a, int(gid))
                if key not in want:
                    continue
                r, ex = want[key], X7["exist"][:, j]
                if not ex.any():
                    continue
                t_anc = float(t[ex][-1])                                   # the branch's last sampled snapshot IS the anchor
                k_anc = int(np.flatnonzero(ex)[-1])                        # ... and its index, where the anchor-epoch extras are read
                if P7_WINDOW is None:
                    t0 = -np.inf
                elif isinstance(P7_WINDOW, str):                           # "sft": integrate only over the quenching itself
                    t0 = float(r.get("t_sft", np.nan))
                    t0 = t0 if np.isfinite(t0) else -np.inf
                else:
                    t0 = t_anc - float(P7_WINDOW)
                win = ex & (t >= t0)
                if not win.any():
                    continue
                xj, aj, jj = xr[:, j] & win, act[:, j] & win, jt[:, j] & win
                mdj, wjj = mdot[:, j], wj[:, j]

                def ig(m, w=None):                                         # the time integral over the window: sum w dt [Gyr, or Msun/yr x Gyr]
                    return float(np.sum(dtw[m] * (1.0 if w is None else w[m]))) if m.any() else 0.0

                e_x, e_x_act = ig(xj), ig(xj & aj)
                fed_x = fedd7[xj, j]; fed_x = fed_x[np.isfinite(fed_x)]
                seen.add(key)
                p7rows.append(dict(
                    box=b, snap=a, gal_id=int(gid), z=float(r["z"]), pop=r["pop"], lfd=float(r["lfd"]), lfh2=float(r["lfh2"]),
                    age=float(r["age"]), log_mstar=float(r["log_mstar"]), log_mbh=float(r["log_mbh"]), lfgas=float(r["lfgas"]),
                    # ── the accretion integrals: 1e9 turns Msun/yr x Gyr into Msun.  The total is the AGN energy budget, eta Mdot c^2 ──
                    m_acc=1e9 * ig(win, mdj), m_acc_x=1e9 * ig(xj, mdj), m_acc_jet=1e9 * ig(jj, mdj),
                    k_energy=1e9 * ig(win, wjj * mdj),                     # the coupling weighted by what was actually accreted through it
                    # ── the coupling integrated in time, with and without the accretion floor ──
                    k_time=ig(win, wjj), k_time_act=ig(aj, wjj),
                    e_x=e_x, e_x_act=e_x_act, f_dead_x=(e_x - e_x_act) / e_x if e_x > 0 else np.nan,
                    fedd_x_med=float(np.median(fed_x)) if len(fed_x) else np.nan,
                    # ── the clocks ──
                    t_anchor=t_anc, t_last_x=float(t[xj][-1]) if xj.any() else np.nan,
                    t_quiet=t_anc - float(t[xj][-1]) if xj.any() else np.nan,
                    t_since_acc=t_anc - float(t[aj][-1]) if aj.any() else np.nan,
                    n_x=int(xj.sum()), n_snap=int(win.sum()), f_x=float(xj.sum() / max(1, win.sum())),
                    t_span=t_anc - float(t[win][0]), dt_samp=float(np.median(np.diff(t))),
                    mbh_start=float(mbh7[win, j][0]) if np.isfinite(mbh7[win, j][0]) else np.nan,
                    zgas=float(zg7[k_anc, j]) if (zg7 is not None and np.isfinite(zg7[k_anc, j])) else np.nan))
    p7_nochain[b] = len(want) - len(seen)

DOSE_ALL = pd.DataFrame(p7rows)
if not len(DOSE_ALL):
    DOSE = DOSE_ALL
    GRP_OF, NGRP = {}, {}
    print("Part 2a: none of the selected anchors has a main-branch chain in the histories caches — nothing to integrate, "
          "and no routes: Parts 2b-13 will all skip.")
else:
    for src, dst in (("m_acc", "lm_acc"), ("m_acc_x", "lm_acc_x"), ("m_acc_jet", "lm_acc_jet"), ("k_energy", "lk_energy")):
        DOSE_ALL[dst] = p7log(DOSE_ALL[src])
    DOSE_ALL["dusty"] = DOSE_ALL["lfd"].to_numpy(float) >= DUSTY_CUT
    DOSE_ALL["never"] = DOSE_ALL["n_x"].to_numpy(int) == 0
    DOSE_ALL["cat"]   = route_cat(DOSE_ALL)
    DOSE_ALL["grp"]   = route_grp(DOSE_ALL)
    GRP_OF = {(b_, int(s_), int(g_)): v for b_, s_, g_, v in zip(DOSE_ALL["box"], DOSE_ALL["snap"], DOSE_ALL["gal_id"], DOSE_ALL["grp"])}
    # DOSE is what Parts 8-13 read: the low-z subset, exactly the sample those parts have always used
    DOSE = (DOSE_ALL if P7_ZMAX is None else DOSE_ALL[DOSE_ALL["z"] <= P7_ZMAX + 1e-9]).reset_index(drop=True)

    _w = ("the whole tracked branch" if P7_WINDOW is None else
          (f"since t_SFT (the galaxies without one integrate over the whole branch)" if isinstance(P7_WINDOW, str) else f"the last {float(P7_WINDOW):g} Gyr before the anchor"))
    print(f"Part 2a — the AGN history integrated over {_w}, the X-ray state read with XRAY_FGAS_MAX = {XRAY_FGAS_MAX:g}, "
          f"the accretion floor at f_Edd = {P7_FEDD_ACT:g}.")
    print(f"  {len(DOSE_ALL)} galaxy-anchors integrated over every anchor ("
          + ", ".join(f"{b} {int((DOSE_ALL['box'] == b).sum())}" for b in BOXES_OK)
          + (" ; without a chain: " + ", ".join(f"{b} {n}" for b, n in p7_nochain.items() if n) if any(p7_nochain.values()) else "")
          + f"); median tracked span {np.nanmedian(DOSE_ALL['t_span']):.1f} Gyr, snapshot spacing {np.nanmedian(DOSE_ALL['dt_samp']):.3f} Gyr.")
    print(f"  DOSE (what Parts 8-13 read) = "
          + (f"the {len(DOSE)} anchors at z <= {P7_ZMAX:g} — an EXTRA cut on top of Part 0's" if P7_ZMAX is not None
             else f"all {len(DOSE)} of them; Part 0's Z_MAX / AGE_MIN already fixed the sample for every part."))

    print(f"\nthe three routes (dust split at log M_dust/M* = {DUSTY_CUT:g}; 'fully dosed' = E_x >= {PF_EX:g} Gyr, "
          f"'still firing' = t_quiet < {PF_TQ:g} Gyr):")
    NGRP = {}
    for b in BOXES_OK:
        G_ = DOSE_ALL[DOSE_ALL["box"] == b]
        NGRP[b] = {g: int((G_["grp"] == g).sum()) for g in G3}
        print(f"  {BOX_LABEL[b]}: {len(G_)} anchors — " + ", ".join(f"{g} {NGRP[b][g]}" for g in G3)
              + "   [of the dusty ones: " + ", ".join(f"{c} {int(((G_['cat'] == c) & G_['dusty']).sum())}" for c, _, _ in CATS) + "]")
    print("\nwhy the floor: w_jet = 1 for any f_Edd <= "
          f"{JET_FEDD * 10 ** -W_FULL:.3f}, so a dead BH is 'maximally coupled' by construction. "
          + "; ".join(f"{b}: median dead share of E_x = {100 * np.nanmedian(DOSE_ALL[(DOSE_ALL['box'] == b) & (DOSE_ALL['e_x'] > 0)]['f_dead_x']):.0f} %"
                      for b in BOXES_OK if ((DOSE_ALL["box"] == b) & (DOSE_ALL["e_x"] > 0)).any())
          + " — that dead time counts in E_x and k_time, not in k_time_act, m_acc_x or k_energy.")
    DOSE_ALL.to_csv(os.path.join(OUT, "agn_states_dose_vs_dust_galaxies.csv"), index=False)
    print(f"\nper-galaxy integrals + routes -> {os.path.join(OUT, 'agn_states_dose_vs_dust_galaxies.csv')}")


In [ ]:
# ── Part 2b — the state engine: the four exclusive AGN states + the tracked parameters, per snapshot, aligned on SFT ──
# The stacks are keyed on Part 2a's THREE ROUTES, not on dusty/pure: the question is no longer "do dusty and pure QGs have
# different AGN histories" but "does the route the dust took show up in the history".
QUANTS = [("lfh2",  r"$\log 1.36\,M_{\rm H_2}/M_\star$",                 (-3.6, 0.4)),   # dropped automatically if the cache has no mh2
          ("lmdot", r"$\log \dot M_{\rm BH}$ [$M_\odot$ yr$^{-1}$]", (-7.3, 1.2)),
          ("lfedd", r"$\log f_{\rm Edd}$",                          (-6.3, 0.4)),
          ("lmbh",  r"$\log M_{\rm BH}$ [$M_\odot$]",               (6.2, 9.6)),
          ("lssfr", r"$\log$ sSFR [yr$^{-1}$]",                     (-13.4, -8.2)),
          ("lfd",   r"$\log M_{\rm dust}/M_\star$",                 (-6.7, -2.0))]   # dropped automatically if the cache has no mdust
DIAG = [("nobh",  "dormant", "no BH in the galaxy at all"),
        ("pjet",  "choked",  "partial-velocity jet (0.02 < f_Edd < 0.2)"),
        ("fjrich", "choked", "full jet but gas-rich (the gate is shut)"),
        ("quasar", "radiative", "f_Edd >= 0.2 (quasar / wind mode)"),
        ("lowmbh", "radiative", "M_BH below the jet threshold")]

QUANTS = [q for q in QUANTS if not (q[0] == "lfd" and not all(HAVE_MD[b] for b in BOXES_OK))
                           and not (q[0] == "lfh2" and not all(HAVE_MH2[b] for b in BOXES_OK))]
_absent = [k for k, h in (("mdust", HAVE_MD), ("mh2", HAVE_MH2)) if BOXES_OK and not all(h[b] for b in BOXES_OK)]
if _absent:
    print(f"NOTE: at least one histories cache carries no {', '.join(_absent)} — the matching row is dropped from the tracks "
          "(rm ism_prediction_histories_*.hdf5 and re-run paper_ism_prediction_boxes Part 6a to add it).\n")

TRK, OCC, DQ, STIDX, NG, TAUQ_BOX = {}, {}, {}, {}, {}, {}          # per box: the parameter stacks, the state-occupancy stacks, the quench-distance stack, the state assignment per epoch
cens_rows = []
for b in BOXES_OK:
    gbox = QM[(QM["box"] == b) & np.isfinite(QM["t_sft"])].reset_index(drop=True)
    grp_of = {(int(s), int(i)): GRP_OF[(b, int(s), int(i))] for s, i in zip(gbox["snap"], gbox["gal_id"])
              if (b, int(s), int(i)) in GRP_OF}                                       # no chain in Part 2a -> no route -> not tracked
    sft_of = {(int(s), int(i)): float(t) for s, i, t in zip(gbox["snap"], gbox["gal_id"], gbox["t_sft"])}
    tqt_of = {(int(s), int(i)): float(t) for s, i, t in zip(gbox["snap"], gbox["gal_id"], gbox["t_qt"])}
    trk = {q: {g: [] for g in G3} for q, _, _ in QUANTS}
    occ = {s: {g: [] for g in G3} for s in SKEYS}
    dq  = {g: [] for g in G3}
    sti = {g: [] for g in G3}
    tauq = {g: [] for g in G3}
    cells = {g: {k: 0.0 for k in SKEYS + [d for d, _, _ in DIAG] + ["all"]} for g in G3}     # the raw snapshot census, before any interpolation
    with h5py.File(HIST_FILE.format(box=b), "r") as f:
        for grpname in f:
            gh = f[grpname]; a = int(grpname.replace("snap", ""))
            t = gh["t_yr"][:] / 1e9; order = np.argsort(t); t = t[order]
            ms, sf, mg, mbh, fedd = (gh[c][:][order] for c in ("mstar", "sfr", "mgas", "mbh", "fedd"))
            md = gh["mdust"][:][order] if HAVE_MD[b] else None
            mh = gh["mh2"][:][order] if HAVE_MH2[b] else None
            S, D, X = agn_states(ms, mg, mbh, fedd)
            with np.errstate(all="ignore"):
                den   = np.where(X["exist"], ms, np.nan)
                ssfr  = np.clip(np.where(X["exist"], np.clip(sf, 0.0, None) / den, np.nan), SSFR_FLOOR, None)
                QT = {"lmdot": np.log10(np.clip(X["mdot"], MDOT_FLOOR, None)),
                      "lfedd": np.log10(np.clip(np.where(X["exist"], np.nan_to_num(fedd), np.nan), FEDD_FLOOR, None)),
                      "lmbh":  np.log10(np.where(X["exist"] & np.isfinite(mbh) & (mbh > 0), mbh, np.nan)),
                      "lssfr": np.log10(ssfr)}
                if HAVE_MD[b]:
                    QT["lfd"] = np.log10(np.clip(np.where(X["exist"], md / den, np.nan), 10 ** P9_FLOOR, None))
                if HAVE_MH2[b]:
                    QT["lfh2"] = np.log10(np.where(X["exist"] & (mh > 0), HE * mh / den, np.nan))
                QT["dq"] = np.log10(ssfr) - np.log10(PASSIVE_FACTOR / (t[:, None] * 1e9))    # the distance from the 0.2 / t_H quenching curve, evaluated at each snapshot's own t_H
            for j, gid in enumerate(gh["gal_ids"][:]):
                key = (a, int(gid)); g = grp_of.get(key)
                if g is None:
                    continue
                trel = t - sft_of[key]
                for q, _, _ in QUANTS:
                    trk[q][g].append(to_grid(trel, QT[q][:, j]))
                dq[g].append(to_grid(trel, QT["dq"][:, j]))
                si = np.vstack([to_grid(trel, S[s][:, j]) for s in SKEYS])                   # the four indicators interpolated: they still sum to 1 wherever the galaxy is sampled
                for si_row, s in zip(si, SKEYS):
                    occ[s][g].append(si_row)
                good = np.isfinite(si).any(axis=0)
                sti[g].append(np.where(good, np.argmax(np.where(np.isfinite(si), si, -np.inf), axis=0), -1))
                if np.isfinite(tqt_of.get(key, np.nan)):
                    tauq[g].append(tqt_of[key] - sft_of[key])
                for k, v in list(S.items()) + list(D.items()):
                    cells[g][k] += float(np.nansum(v[:, j]))
                cells[g]["all"] += float(np.isfinite(S["dormant"][:, j]).sum())
    TRK[b] = {q: {g: np.array(v) for g, v in trk[q].items()} for q, _, _ in QUANTS}
    OCC[b] = {s: {g: np.array(v) for g, v in occ[s].items()} for s in SKEYS}
    DQ[b]  = {g: np.array(v) for g, v in dq.items()}
    STIDX[b] = {g: np.array(v) for g, v in sti.items()}
    NG[b]  = {g: len(v) for g, v in dq.items()}
    TAUQ = {g: (np.median(v) if len(v) else np.nan) for g, v in tauq.items()}
    TAUQ_BOX[b] = TAUQ
    print(f"{'=' * 112}\n{BOX_LABEL[b]}: " + " + ".join(f"{NG[b][g]} {GSHORT[g]}" for g in G3)
          + f" chains on the SFT clock (of {int((QM['box'] == b).sum())} selected; the rest carry no SFT or no chain); "
          + "median tau_q = t_QT - t_SFT: " + ", ".join(f"{GSHORT[g]} {TAUQ[g]:.2f}" for g in G3) + " Gyr")
    print(f"  the snapshot census over the whole tracked history (share of all main-branch snapshots of the route):")
    print(f"  {'state':12s} " + "  ".join(f"{GSHORT[g]:>17s}" for g in G3))
    for s in SKEYS:
        print(f"  {s:12s} " + "  ".join(f"{100 * cells[g][s] / max(cells[g]['all'], 1):16.1f}%" for g in G3))
    print(f"  inside those states ('{'/'.join(SKEYS)}' stay four; this is only what they are made of):")
    for dkey, parent, what in DIAG:
        print(f"    {parent:10s} <- {dkey:7s} {what:44s} " + "  ".join(f"{100 * cells[g][dkey] / max(cells[g][parent], 1e-9):6.1f}% of {parent}" for g in G3))
    for g in G3:
        cens_rows.append(dict(box=b, route=g, n_chains=NG[b][g], n_snap_cells=cells[g]["all"], tau_q_med=TAUQ[g],
                              **{f"f_{s}": cells[g][s] / max(cells[g]["all"], 1) for s in SKEYS},
                              **{f"f_{d}_of_{parent}": cells[g][d] / max(cells[g][parent], 1e-9) for d, parent, _ in DIAG}))
    print()

STCENS = pd.DataFrame(cens_rows)
STCENS.to_csv(os.path.join(OUT, "agn_states_snapshot_census.csv"), index=False)
print(f"snapshot-level state census -> {os.path.join(OUT, 'agn_states_snapshot_census.csv')}")


## Part 3 — the median tracks on the SFT clock, one curve per route, box by box

Three figures on the same grid — one column per box, $t - t_{\rm SFT}$ on every $x$-axis, **the three routes of Part 2a as three median tracks** with bootstrap 16–84 % bands, and the number of contributing galaxies drawn as a histogram behind each panel (all three routes on one shared count scale; the peak $N$ is printed in the corner). A track is blanked wherever fewer than `NMIN` galaxies reach that epoch, so the ends of the clock fade out honestly rather than turning into one galaxy's history.

* **(A) the parameters** — $\log 1.36\,M_{\rm H_2}/M_\star$, $\log \dot M_{\rm BH}$, $\log f_{\rm Edd}$, $\log M_{\rm BH}$, $\log$ sSFR, $\log M_{\rm dust}/M_\star$. The three BH rows are exactly the quantities the state machine is built from, so this panel is the state definition made continuous; the per-epoch AUC of each contrast in `GPAIRS` (firing vs dust-poor, timing vs dust-poor, firing vs timing) is tabulated underneath at a handful of epochs.
* **(B) the state occupancy** — the fraction of each route in each of the four states, state by state, so the between-route contrast can be read directly off one panel.
* **(C) the same occupancy stacked** — one panel per (box, route): the composition of that route's AGN history from 3 Gyr before the transition to 5 Gyr after it.

The vertical dashed line is the SFT; the dotted vertical is the route's median QT ($t_{\rm SFT} + \tau_q$), i.e. where its median galaxy has finished crossing into the quenched region.

**What the split now asks.** The old dusty-vs-pure version asked whether the AGN history differs between galaxies that end up dusty and galaxies that do not. Splitting on the routes asks the sharper question: the *timing-explained* dusty route and the *dust-poor* route differ in their AGN history **by construction** (that is what the route definition encodes), so the panel to read is the **firing-dusty track against the dust-poor track** — two sets of galaxies that were both fully dosed and are both being dosed at the anchor, and differ only in whether the dust survived.


In [ ]:
# ── Part 3A — the parameter tracks: the median of each of Part 2a's THREE ROUTES per box, with the source counts behind them ──
rows3 = []
TCHECK = [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]


def qt_marks(ax, b):
    for g in G3:
        tq = TAUQ_BOX.get(b, {}).get(g, np.nan)
        if np.isfinite(tq) and TG[0] <= tq <= TG[-1]:
            ax.axvline(tq, color=GRPS[g]["color"], lw=1.0, ls=":", alpha=0.8, zorder=2)


def epoch_auc(stacks, rows, **meta):
    """Per-epoch AUC for each contrast of GPAIRS, appended to rows as series 'AUC_<a> vs <b>'."""
    for ga, gb in GPAIRS:
        A1, A0 = stacks.get(ga), stacks.get(gb)
        if A1 is None or A0 is None or not len(A1) or not len(A0):
            continue
        for it, tt in enumerate(TG):
            auc, pv, n1, n0 = mw_auc(np.r_[A1[:, it], A0[:, it]], np.r_[np.ones(len(A1), bool), np.zeros(len(A0), bool)])
            rows.append(dict(series=f"AUC {GSHORT[ga]} vs {GSHORT[gb]}", t=float(tt), n=n1 + n0, val=auc, lo=np.nan, hi=np.nan, p_value=pv, **meta))


fig, axs = plt.subplots(len(QUANTS), len(BOXES_OK), figsize=(4.5 * len(BOXES_OK), 2.75 * len(QUANTS)), squeeze=False)
for ci, b in enumerate(BOXES_OK):
    for qi, (q, ylab, ylim) in enumerate(QUANTS):
        ax = axs[qi, ci]
        counts = {}
        for g in G3:
            cent, lo, hi, n_t = med_band(TRK[b][q][g])
            counts[g] = n_t
            ax.fill_between(TG, lo, hi, color=GRPS[g]["color"], alpha=0.16, lw=0, zorder=3)
            ax.plot(TG, cent, color=GRPS[g]["color"], lw=GRPS[g]["lw"], ls=GRPS[g]["ls"], zorder=5)
            for tt, vv, l_, h_, nn in zip(TG, cent, lo, hi, n_t):
                rows3.append(dict(box=b, quantity=q, series=g, t=float(tt), n=int(nn), val=float(vv), lo=float(l_), hi=float(h_), p_value=np.nan))
        epoch_auc(TRK[b][q], rows3, box=b, quantity=q)
        count_bg(ax, counts)
        ax.axvline(0, color="0.35", lw=1.0, ls="--", zorder=2); qt_marks(ax, b)
        ax.set_xlim(TG[0], TG[-1]); ax.set_ylim(*ylim); ax.grid(alpha=0.25, lw=0.5)
        ax.set_ylabel(ylab if ci == 0 else ""); ax.set_xlabel(r"$t - t_{\rm SFT}$ [Gyr]" if qi == len(QUANTS) - 1 else "")
        if qi == 0:
            ax.set_title(BOX_LABEL[b] + "\n" + " / ".join(f"{NG[b][g]} {GSHORT[g]}" for g in G3) + " chains", fontsize=9.5)
_h3 = [Line2D([], [], color=GRPS[g]["color"], lw=GRPS[g]["lw"], label=GRPS[g]["label"]) for g in G3]
_h3 += [Line2D([], [], color="0.35", lw=1.0, ls="--", label="SFT"), Line2D([], [], color="0.35", lw=1.0, ls=":", label=r"median QT ($t_{\rm SFT} + \tau_q$)"),
        Line2D([], [], color="0.6", lw=6, alpha=0.35, label="galaxies contributing (histogram behind)")]
fig.legend(handles=_h3, loc="lower center", ncol=3, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.012))
fig.suptitle("the AGN history of the three dust routes: median tracks on the SFT clock", y=0.995, fontsize=11.5)
fig.tight_layout(rect=(0, 0.035, 1, 0.975))
paper_save(fig, "agn_states_parameter_tracks")
plt.show()

R3 = pd.DataFrame(rows3)
print("per-epoch AUC  ('*' = p < 0.01, '-' = fewer than NMIN either side):")
for b in BOXES_OK:
    print(f"  {BOX_LABEL[b]}")
    for ga, gb in GPAIRS:
        ser = f"AUC {GSHORT[ga]} vs {GSHORT[gb]}"
        print(f"    {ser}")
        for q, _, _ in QUANTS:
            S = R3[(R3["box"] == b) & (R3["quantity"] == q) & (R3["series"] == ser)]
            cells = []
            for tc in TCHECK:
                r = S[np.abs(S["t"] - tc) < 0.01]
                cells.append(f"{tc:+.0f}: " + (f"{r['val'].iloc[0]:.2f}" + ("*" if r["p_value"].iloc[0] < 0.01 else " ")
                             if len(r) and np.isfinite(r["val"].iloc[0]) else "  -  "))
            print(f"      {q:6s} " + "  ".join(cells))


In [ ]:
# ── Part 3B — the state occupancy: the fraction of each ROUTE in each AGN state, and (C) the same stacked ──
rows3b = []
fig, axs = plt.subplots(len(SKEYS), len(BOXES_OK), figsize=(4.5 * len(BOXES_OK), 2.75 * len(SKEYS)), squeeze=False)
for ci, b in enumerate(BOXES_OK):
    for si, s in enumerate(SKEYS):
        ax = axs[si, ci]
        counts = {}
        for g in G3:
            cent, lo, hi, n_t = med_band(OCC[b][s][g], stat="mean")
            counts[g] = n_t
            ax.fill_between(TG, lo, hi, color=GRPS[g]["color"], alpha=0.16, lw=0, zorder=3)
            ax.plot(TG, cent, color=GRPS[g]["color"], lw=GRPS[g]["lw"], zorder=5)
            for tt, vv, l_, h_, nn in zip(TG, cent, lo, hi, n_t):
                rows3b.append(dict(box=b, state=s, series=g, t=float(tt), n=int(nn), val=float(vv), lo=float(l_), hi=float(h_), p_value=np.nan))
        epoch_auc(OCC[b][s], rows3b, box=b, state=s)
        count_bg(ax, counts)
        ax.axvline(0, color="0.35", lw=1.0, ls="--", zorder=2); qt_marks(ax, b)
        ax.set_xlim(TG[0], TG[-1]); ax.set_ylim(-0.02, 1.02); ax.grid(alpha=0.25, lw=0.5)
        ax.set_ylabel(f"fraction\n{SLAB[s]}" if ci == 0 else "", fontsize=9)
        ax.set_xlabel(r"$t - t_{\rm SFT}$ [Gyr]" if si == len(SKEYS) - 1 else "")
        if ci == len(BOXES_OK) - 1:
            ax.text(1.015, 0.5, s, transform=ax.transAxes, rotation=270, va="center", ha="left", fontsize=9, color=SCOL[s], fontweight="bold")
        if si == 0:
            ax.set_title(BOX_LABEL[b], fontsize=9.5)
fig.legend(handles=[Line2D([], [], color=GRPS[g]["color"], lw=GRPS[g]["lw"], label=GRPS[g]["label"]) for g in G3]
                   + [Line2D([], [], color="0.35", lw=1.0, ls="--", label="SFT"), Line2D([], [], color="0.35", lw=1.0, ls=":", label="median QT")],
           loc="lower center", ncol=2, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.012))
fig.suptitle("the AGN state occupancy of the three dust routes on the SFT clock", y=0.995, fontsize=11.5)
fig.tight_layout(rect=(0, 0.05, 1, 0.975))
paper_save(fig, "agn_states_occupancy")
plt.show()

R3B = pd.DataFrame(rows3b)
pd.concat([R3, R3B.rename(columns={"state": "quantity"})], ignore_index=True).to_csv(os.path.join(OUT, "agn_states_tracks.csv"), index=False)
print(f"parameter + occupancy tracks and per-epoch AUCs -> {os.path.join(OUT, 'agn_states_tracks.csv')}\n")

# (C) the same occupancy stacked: what each route's AGN history is made of, epoch by epoch
fig, axs = plt.subplots(len(G3), len(BOXES_OK), figsize=(4.5 * len(BOXES_OK), 3.0 * len(G3)), squeeze=False)
for ci, b in enumerate(BOXES_OK):
    for gi, g in enumerate(G3):
        ax = axs[gi, ci]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            fr = np.array([np.nanmean(OCC[b][s][g], axis=0) if len(OCC[b][s][g]) else np.full(len(TG), np.nan) for s in SKEYS])
        n_t = np.isfinite(OCC[b][SKEYS[0]][g]).sum(axis=0) if len(OCC[b][SKEYS[0]][g]) else np.zeros(len(TG), int)
        keep = n_t >= NMIN
        if keep.any():
            ax.stackplot(TG[keep], np.nan_to_num(fr[:, keep]), colors=[SCOL[s] for s in SKEYS], lw=0)
        ax.axvline(0, color="k", lw=1.1, ls="--")
        tq = TAUQ_BOX.get(b, {}).get(g, np.nan)
        if np.isfinite(tq) and TG[0] <= tq <= TG[-1]:
            ax.axvline(tq, color="k", lw=1.0, ls=":")
        ax.set_xlim(TG[0], TG[-1]); ax.set_ylim(0, 1)
        ax.set_ylabel(f"{GSHORT[g]}\nfraction of the route" if ci == 0 else "", fontsize=9.5, color=GRPS[g]["color"], fontweight="bold")
        ax.set_xlabel(r"$t - t_{\rm SFT}$ [Gyr]" if gi == len(G3) - 1 else "")
        ax.text(0.015, 0.06, f"{g}, N = {NG[b][g]}", transform=ax.transAxes, fontsize=8.5, color="w", fontweight="bold")
        if gi == 0:
            ax.set_title(BOX_LABEL[b], fontsize=9.5)
fig.legend(handles=[Line2D([], [], color=SCOL[s], lw=8, label=SLAB[s]) for s in SKEYS], loc="lower center", ncol=4, frameon=False,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("the composition of the AGN history: the four states stacked, per box and route", y=0.995, fontsize=11.5)
fig.tight_layout(rect=(0, 0.055, 1, 0.975))
paper_save(fig, "agn_states_composition")
plt.show()


## Part 4 — the same tracks with the distance from the $0.2/t_{\rm H}$ curve on the $y$-axis

The quenched definition used throughout is ${\rm sSFR} < 0.2/t_{\rm H}$, so the natural single axis for "how quenched was this galaxy at that moment" is the **signed distance from that curve in dex**,

$$\Delta_{\rm q}(t) \;=\; \log_{10}{\rm sSFR}(t) \;-\; \log_{10}\!\frac{0.2}{t_{\rm H}(t)},$$

evaluated at each snapshot's own cosmic time, so the moving threshold is followed rather than frozen at the anchor. $\Delta_{\rm q} > 0$ is star-forming, $\Delta_{\rm q} = 0$ is the quenching line itself (the QT crossing), $\Delta_{\rm q} < 0$ is quenched; the floor is `SSFR_FLOOR` and moves with the threshold, from $\Delta_{\rm q} \approx -2.2$ dex at $z = 0$ to $\approx -2.8$ at $z = 2$ — it is a floor, not a measurement.

Same grid as Part 3 — one column per box, the SFT clock, **the three routes**, the source counts behind. The rows are

* **all** — each route's median trajectory across the quenching line;
* **one row per AGN state** — the median $\Delta_{\rm q}$ restricted to the galaxies that are *in that state at that epoch* (the state assignment is the interpolated indicator's argmax, so every galaxy contributes to exactly one row at each epoch, and the row's own count histogram shows how many that is). This is the crossing read state by state: it separates "the route was falling towards the line" from "the galaxies of that route that were being X-ray heated were falling towards the line".


In [ ]:
# ── Part 4 — the quenching distance Delta_q = log sSFR - log(0.2 / t_H): overall and per AGN state, one curve per ROUTE ──
DQ_YLIM = (-3.05, 1.2)
ROWS4 = [("all", "all galaxies")] + [(s, SLAB[s]) for s in SKEYS]
rows4 = []

fig, axs = plt.subplots(len(ROWS4), len(BOXES_OK), figsize=(4.5 * len(BOXES_OK), 2.75 * len(ROWS4)), squeeze=False)
for ci, b in enumerate(BOXES_OK):
    for ri, (key, lab) in enumerate(ROWS4):
        ax = axs[ri, ci]
        stacks, counts = {}, {}
        for g in G3:
            A = DQ[b][g]
            if key != "all" and len(A):
                A = np.where(STIDX[b][g] == SKEYS.index(key), A, np.nan)      # only the galaxies sitting in this state at this epoch
            stacks[g] = A
            cent, lo, hi, n_t = med_band(A)
            counts[g] = n_t
            ax.fill_between(TG, lo, hi, color=GRPS[g]["color"], alpha=0.16, lw=0, zorder=3)
            ax.plot(TG, cent, color=GRPS[g]["color"], lw=GRPS[g]["lw"], zorder=5)
            for tt, vv, l_, h_, nn in zip(TG, cent, lo, hi, n_t):
                rows4.append(dict(box=b, row=key, series=g, t=float(tt), n=int(nn), val=float(vv), lo=float(l_), hi=float(h_), p_value=np.nan))
        epoch_auc(stacks, rows4, box=b, row=key)
        count_bg(ax, counts)
        ax.axhline(0, color="k", lw=1.2, zorder=2)                     # the 0.2 / t_H quenching curve itself
        ax.axvline(0, color="0.35", lw=1.0, ls="--", zorder=2); qt_marks(ax, b)
        ax.set_xlim(TG[0], TG[-1]); ax.set_ylim(*DQ_YLIM); ax.grid(alpha=0.25, lw=0.5)
        ax.set_ylabel((r"$\Delta_{\rm q}$ [dex]" + f"\n{lab}") if ci == 0 else "", fontsize=9)
        ax.set_xlabel(r"$t - t_{\rm SFT}$ [Gyr]" if ri == len(ROWS4) - 1 else "")
        if ci == len(BOXES_OK) - 1 and key != "all":
            ax.text(1.015, 0.5, key, transform=ax.transAxes, rotation=270, va="center", ha="left", fontsize=9, color=SCOL[key], fontweight="bold")
        if ri == 0:
            ax.set_title(BOX_LABEL[b], fontsize=9.5)
_h4 = [Line2D([], [], color=GRPS[g]["color"], lw=GRPS[g]["lw"], label=GRPS[g]["label"]) for g in G3]
_h4 += [Line2D([], [], color="k", lw=1.2, label=r"the $0.2/t_{\rm H}$ curve ($\Delta_{\rm q} = 0$)"), Line2D([], [], color="0.35", lw=1.0, ls="--", label="SFT"),
        Line2D([], [], color="0.35", lw=1.0, ls=":", label="median QT"), Line2D([], [], color="0.6", lw=6, alpha=0.35, label="galaxies contributing")]
fig.legend(handles=_h4, loc="lower center", ncol=3, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.012))
fig.suptitle(r"the distance from the quenching curve, $\Delta_{\rm q} = \log\,{\rm sSFR} - \log\,(0.2/t_{\rm H})$: overall and per AGN state, by dust route",
             y=0.995, fontsize=11.5)
fig.tight_layout(rect=(0, 0.045, 1, 0.975))
paper_save(fig, "agn_states_quenching_distance")
plt.show()

R4 = pd.DataFrame(rows4)
R4.to_csv(os.path.join(OUT, "agn_states_quenching_distance.csv"), index=False)
print(f"quenching-distance tracks and per-epoch AUCs -> {os.path.join(OUT, 'agn_states_quenching_distance.csv')}\n")
print("median Delta_q at the key epochs (" + " | ".join(GSHORT[g] for g in G3) + "):")
for b in BOXES_OK:
    print(f"  {BOX_LABEL[b]}")
    for key, lab in ROWS4:
        cells = []
        for tc in TCHECK:
            v = {g: R4[(R4["box"] == b) & (R4["row"] == key) & (R4["series"] == g) & (np.abs(R4["t"] - tc) < 0.01)] for g in G3}
            if all(len(v[g]) and np.isfinite(v[g]["val"].iloc[0]) for g in G3):
                cells.append(f"{tc:+.0f}: " + "|".join(f"{v[g]['val'].iloc[0]:+.2f}" for g in G3))
            else:
                cells.append(f"{tc:+.0f}:      -      ")
        print(f"    {key:10s} " + "  ".join(cells))


## Part 8 — the same data read as an implication: exposure is *necessary* for dust destruction, not sufficient

Part 7 asks a correlation question and gets a weak answer. That is the right answer to the wrong question. The relation between the X-ray channel and the dust is **not a dose–response, it is a one-way implication**, and a rank correlation is the wrong instrument for one: it measures a gradient where the data has a step, and it says nothing at all about the direction the implication runs in. This part measures the implication directly.

**(A) The dose–response is a step at the foot of the axis, then a plateau.** The dusty *fraction* (with its Wilson interval, not a median with a percentile spread) against $E_x$, the never-exposed carried as their own point off the axis. What kills the apparent correlation is visible: essentially all of the drop happens in the first bin, and past a fraction of a Gyr the curve is flat. The same statement is made numerically by the **saturation scan** — $\rho(\log f_{\rm dust}, E_x)$ over the exposed, re-measured above a rising floor of $E_x$. If the physics were a dose, $\rho$ would hold as the floor rises; if it is a threshold, $\rho$ collapses once the floor clears it.

**(B) Both directions of the implication, with their base rates.** Four conditionals, each with a Wilson 68 % interval and its raw counts:

| | reading |
|---|---|
| $P(\text{ever exposed} \mid \text{pure})$ | **necessity** — can a galaxy end up dust-poor *without* the channel ever firing? |
| $P(\text{pure} \mid \text{ever exposed})$ | **sufficiency** — does the channel, once fired, finish the job? |
| $P(\text{dusty} \mid \text{never exposed})$ | the rule as usually stated |
| $P(\text{never exposed} \mid \text{dusty})$ | **the converse** — what knowing a galaxy is dusty actually buys you |

The last two are the same fact read in opposite directions, and they are *not* each other's mirror: the never-exposed are a few per cent of the population, so even a rule that holds almost perfectly accounts for only a small share of the dusty galaxies. The base rates are drawn as the dotted lines so the asymmetry can be read as what it is — arithmetic, not a weakening of the rule.

**(C) What the never-exposed actually are.** A rule that separates a population is only interesting if the label is doing the work. Every catalogue and history property is scored by AUC between the never-exposed and the exposed, so it is visible whether "never exposed" is a channel statement or just a restatement of *young*, *low-mass*, *small black hole* or *gas-rich*. The printed census adds the piece the AUCs cannot show: how many of the never-exposed **did launch a jet** at some point (the gas gate or the velocity gate shut the X-ray off) versus never grew a black hole at all — the difference between a channel that was never armed and one that was armed and never fired.

**The censoring caveat is tested, not asserted.** "Never exposed" over 1 Gyr of tracked history is a much weaker statement than "never exposed" over 8. So the four conditionals are re-printed over the **qualified** sample — chains tracked over at least `P8_SPAN_FRAC` (0.5) of the box's median exposed span — and the printed line says how many of the never-exposed survive that cut. A rule that holds on the full sample and dissolves on the qualified one was a statement about short chains.

Outputs: `agn_states_exposure_implication.{png,pdf}` plus `agn_states_exposure_implication.csv` (the conditionals and their intervals per box), `agn_states_exposure_saturation.csv` (the $E_x$-floor scan) and `agn_states_exposure_neverexposed.csv` (the property contrast). Pure post-processing of Part 7's `DOSE` frame — no new reads.

In [ ]:
# ── Part 8 — the implication, not the correlation: exposure is NECESSARY for dust destruction, not sufficient, and the converse fails on base rates ──
P8_NBIN     = 8          # the E_x bins of the dose-response row (quantile bins of the exposed)
P8_NMIN_BIN = {"cis100": 15, "cis50": 8, "cis25": 5}    # galaxies a fraction point needs, per box
P8_NMIN_DEF = 8
P8_SCAN     = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]     # the saturation scan: the E_x floors the trend is re-measured above
P8_SPAN_FRAC = 0.5       # the QUALIFIED sample: a chain must be tracked over at least this fraction of the box's median exposed span.
                         # A galaxy that was 'never exposed' over 1 Gyr of history is not the same statement as one never exposed over 8 -
                         # the four conditionals are re-printed over the qualified galaxies as the robustness check
P8_CONTRAST = [("age", "age [Gyr]"), ("log_mstar", r"$\log M_\star$"), ("log_mbh", r"$\log M_{\rm BH}$"),
               ("lfh2", r"$\log M_{\rm H_2}/M_\star$"),
               ("lm_acc", r"$\log \int \dot M_{\rm BH}{\rm d}t$"), ("t_span", "tracked span [Gyr]")]
p8_nmin = lambda b: int(P8_NMIN_BIN.get(b, P8_NMIN_DEF)) if isinstance(P8_NMIN_BIN, dict) else int(P8_NMIN_BIN)


def wilson(k, n, z=1.0):
    """The Wilson score interval on k/n (z = 1 -> 68 %). A fraction of 0 or 1 out of a handful is not 0 or 1."""
    if n <= 0:
        return np.nan, np.nan, np.nan
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    # the interval is clamped to contain p itself: at k = n the Wilson end point is 1 only up to rounding, and a
    # 1e-16 overshoot is a NEGATIVE yerr to matplotlib. (Which is what a dust cut where every never-exposed
    # galaxy is dusty — k = n — used to crash on.)
    return p, min(p, max(0.0, c - h)), max(p, min(1.0, c + h))


if "DOSE" not in globals() or not len(DOSE):
    print("Part 8: Part 7 built no records — nothing to test.")
else:
    D8 = DOSE.copy()
    D8["dusty"] = D8["lfd"].to_numpy(float) >= DUSTY_CUT
    D8["never"] = D8["n_x"].to_numpy(int) == 0          # never in the X-ray mode over the whole integration window
    P8ROWS, P8SCAN, P8CON = [], [], []

    print("Part 8 — the same data read as an IMPLICATION. Four conditionals, each with its Wilson 68 % interval:")
    print("  necessity   P(ever exposed | pure)  — can a galaxy end up dust-poor WITHOUT the channel?")
    print("  sufficiency P(pure | ever exposed)  — does the channel, once fired, finish the job?")
    print("  the rule    P(dusty | never exposed)")
    print("  the converse P(never exposed | dusty) — what knowing a galaxy is dusty buys you\n")
    for b in BOXES_OK:
        G = D8[D8["box"] == b]
        if not len(G):
            continue
        n, nv, du = len(G), int(G["never"].sum()), int(G["dusty"].sum())
        E, N_, DU, PU = G[~G["never"]], G[G["never"]], G[G["dusty"]], G[~G["dusty"]]
        r = dict(box=b, n=n, n_never=nv, n_dusty=du, f_never=nv / n, f_dusty=du / n)
        for tag, k_, n_ in (("necessity_exposed_given_pure", int((~PU["never"]).sum()), len(PU)),
                            ("sufficiency_pure_given_exposed", int((~E["dusty"]).sum()), len(E)),
                            ("rule_dusty_given_never", int(N_["dusty"].sum()), len(N_)),
                            ("converse_never_given_dusty", int(DU["never"].sum()), len(DU))):
            p_, lo_, hi_ = wilson(k_, n_)
            r[tag], r[tag + "_lo"], r[tag + "_hi"], r[tag + "_k"], r[tag + "_n"] = p_, lo_, hi_, k_, n_
        span_min = P8_SPAN_FRAC * float(np.nanmedian(E["t_span"])) if len(E) else np.nan
        Q = G[G["t_span"].to_numpy(float) >= span_min]                      # the qualified sample
        r["span_min"], r["n_qualified"], r["n_never_qualified"] = span_min, len(Q), int(Q["never"].sum())
        for tag, k_, n_ in (("q_necessity", int((~Q[~Q["dusty"]]["never"]).sum()), int((~Q["dusty"]).sum())),
                            ("q_rule", int(Q[Q["never"]]["dusty"].sum()), int(Q["never"].sum())),
                            ("q_converse", int(Q[Q["dusty"]]["never"].sum()), int(Q["dusty"].sum()))):
            r[tag], r[tag + "_lo"], r[tag + "_hi"], r[tag + "_k"], r[tag + "_n"] = (*wilson(k_, n_), k_, n_)
        P8ROWS.append(r)
        print(f"  {BOX_LABEL[b]}: N = {n}, never exposed {nv} ({100 * nv / n:.1f} %), dusty {du} ({100 * du / n:.1f} %)")
        for tag, lab in (("necessity_exposed_given_pure", "P(ever exposed | pure)      "),
                         ("sufficiency_pure_given_exposed", "P(pure | ever exposed)      "),
                         ("rule_dusty_given_never", "P(dusty | never exposed)    "),
                         ("converse_never_given_dusty", "P(never exposed | dusty)    ")):
            print(f"    {lab} = {r[tag]:.3f} [{r[tag + '_lo']:.3f}, {r[tag + '_hi']:.3f}]   ({r[tag + '_k']}/{r[tag + '_n']})")
        print(f"    qualified on the tracked span (>= {span_min:.2f} Gyr = {P8_SPAN_FRAC:g} x the median exposed span): "
              f"{len(Q)}/{n} galaxies, {int(Q['never'].sum())}/{nv} of the never-exposed survive it — "
              f"P(ever exposed | pure) = {r['q_necessity']:.3f} ({r['q_necessity_k']}/{r['q_necessity_n']}), "
              f"P(dusty | never) = {r['q_rule']:.3f} ({r['q_rule_k']}/{r['q_rule_n']}), "
              f"P(never | dusty) = {r['q_converse']:.3f} ({r['q_converse_k']}/{r['q_converse_n']})")

        # ── the saturation scan: is there a dose-response ABOVE the threshold, or only a step at its foot? ──
        for f_ in P8_SCAN:
            S = E[E["e_x"].to_numpy(float) > f_]
            rho_, pv_ = spearmanr(S["e_x"], S["lfd"]) if len(S) >= 12 else (np.nan, np.nan)
            P8SCAN.append(dict(box=b, e_x_floor=f_, n=len(S), rho=rho_, p=pv_,
                               lfd_med=float(np.median(S["lfd"])) if len(S) else np.nan,
                               f_dusty=float(S["dusty"].mean()) if len(S) else np.nan))
        print(f"    the dose-response among the EXPOSED, re-measured above an E_x floor "
              f"(if it is a threshold and not a dose, rho collapses as the floor rises):")
        for s_ in [x for x in P8SCAN if x["box"] == b]:
            print(f"      E_x > {s_['e_x_floor']:4.2f} Gyr: N = {s_['n']:4d}, rho = {s_['rho']:+.3f} (p = {s_['p']:.2g}), "
                  f"median lfd = {s_['lfd_med']:.2f}, dusty {100 * s_['f_dusty']:.0f} %")

        # ── what the never-exposed ARE: is 'never exposed' doing work beyond being young with a small black hole? ──
        for c, _ in P8_CONTRAST:
            if c not in G.columns:
                continue
            auc, pv, na, nb_ = mw_auc(G[c], G["never"].to_numpy(bool))
            P8CON.append(dict(box=b, prop=c, auc_never_gt_exposed=auc, p=pv, n_never=na, n_exposed=nb_,
                              med_never=float(np.nanmedian(N_[c])) if len(N_) else np.nan,
                              med_exposed=float(np.nanmedian(E[c])) if len(E) else np.nan))
        print(f"    the never-exposed against the exposed (AUC > 0.5 = the never-exposed are the higher ones):")
        for x_ in [x for x in P8CON if x["box"] == b]:
            print(f"      {x_['prop']:10s} never {x_['med_never']:8.2f}  exposed {x_['med_exposed']:8.2f}   "
                  f"AUC = {x_['auc_never_gt_exposed']:.2f} (p = {x_['p']:.2g})")
        nobh = int((~np.isfinite(N_["log_mbh"])).sum())
        print(f"      of the {nv} never exposed: {int((N_['m_acc_jet'] > 0).sum())} DID launch a jet at some point (the gas gate or the "
              f"velocity gate shut the X-ray off), {nobh} never grew a black hole at all.\n")

    P8 = pd.DataFrame(P8ROWS); SCAN8 = pd.DataFrame(P8SCAN); CON8 = pd.DataFrame(P8CON)

    # ── the figure: (A) the dose-response as a FRACTION, (B) the four conditionals, (C) what the never-exposed are ──
    fig, axs = plt.subplots(3, len(BOXES_OK), figsize=(4.6 * len(BOXES_OK), 10.4), squeeze=False,
                            constrained_layout=True, gridspec_kw=dict(height_ratios=[1.15, 1.0, 1.0]))
    CONDS = [("rule_dusty_given_never", "P(dusty | never exposed)", "#d7191c"),
             ("converse_never_given_dusty", "P(never exposed | dusty)", "#f4a582"),
             ("necessity_exposed_given_pure", "P(ever exposed | pure)", "#2c7bb6"),
             ("sufficiency_pure_given_exposed", "P(pure | ever exposed)", "#92c5de")]
    for ci, b in enumerate(BOXES_OK):
        G = D8[D8["box"] == b]
        if not len(G):
            continue
        E, N_ = G[~G["never"]], G[G["never"]]
        r = P8[P8["box"] == b].iloc[0]

        # (A) the dusty fraction against the exposure, the never-exposed as their own point off the axis
        ax = axs[0, ci]
        xs, ys, lo, hi, ns = [], [], [], [], []
        if len(E) >= 2 * p8_nmin(b):
            nb8 = int(np.clip(len(E) // p8_nmin(b), 2, P8_NBIN))
            e = np.unique(np.percentile(E["e_x"], np.linspace(0, 100, nb8 + 1)))
            for i in range(len(e) - 1):
                m = (E["e_x"] >= e[i]) & ((E["e_x"] <= e[i + 1]) if i == len(e) - 2 else (E["e_x"] < e[i + 1]))
                S = E[m]
                if len(S) >= p8_nmin(b):
                    p_, l_, h_ = wilson(int(S["dusty"].sum()), len(S))
                    xs.append(float(np.median(S["e_x"]))); ys.append(p_); lo.append(l_); hi.append(h_); ns.append(len(S))
        xhi = float(np.percentile(E["e_x"], 99)) if len(E) else 1.0
        sx = -0.13 * max(xhi, 1e-6)
        if xs:
            ax.errorbar(xs, ys, yerr=[np.array(ys) - np.array(lo), np.array(hi) - np.array(ys)], fmt="-o",
                        color="k", lw=2.0, ms=5, capsize=3, zorder=4, label=r"exposed, binned in $E_x$")
        if len(N_):
            p_, l_, h_ = wilson(int(N_["dusty"].sum()), len(N_))
            ax.errorbar([sx], [p_], yerr=[[p_ - l_], [h_ - p_]], fmt="s", color="#d7191c", ms=8, lw=2.0, capsize=3, zorder=5,
                        label=f"never exposed ({len(N_)})")
            ax.axvline(0.5 * sx, color="0.5", lw=1.0, ls=":", zorder=1)
        ax.axhline(float(r["f_dusty"]), color="0.45", lw=1.2, ls="--", zorder=2)
        ax.text(0.985, float(r["f_dusty"]), " the whole sample", transform=ax.get_yaxis_transform(), ha="right", va="bottom", fontsize=7, color="0.4")
        ax.set_ylim(-0.03, 1.05); ax.set_xlim(1.35 * sx, xhi * 1.04)
        ax.set_xticks([t for t in ax.get_xticks() if t >= 0.25 * sx]); ax.set_xlim(1.35 * sx, xhi * 1.04)
        ax.set_xlabel(r"$E_x = \int_{\rm X-ray}{\rm d}t$  [Gyr]", fontsize=9)
        ax.set_ylabel("fraction dusty" + (rf"  ($\log M_{{\rm dust}}/M_\star \geq {DUSTY_CUT:g}$)" if ci == 0 else ""), fontsize=9)
        ax.grid(alpha=0.25, lw=0.5); ax.set_title(BOX_LABEL[b], fontsize=9.5)
        ax.legend(frameon=False, fontsize=7.5, loc="upper right")
        s0 = SCAN8[(SCAN8["box"] == b) & (SCAN8["e_x_floor"] == 0.0)]
        s1 = SCAN8[(SCAN8["box"] == b) & (SCAN8["e_x_floor"] == 0.5)]
        if len(s0) and len(s1):
            ax.text(0.035, 0.045, rf"$\rho(\log f_{{\rm dust}}, E_x)$ over the exposed $= {float(s0['rho'].iloc[0]):+.2f}$" + "\n"
                    + rf"above $E_x > 0.5$ Gyr $= {float(s1['rho'].iloc[0]):+.2f}$ ($p = {float(s1['p'].iloc[0]):.1g}$)",
                    transform=ax.transAxes, fontsize=7.2, va="bottom", ha="left", bbox=dict(fc="w", ec="0.8", lw=0.5, alpha=0.9, pad=2.2))

        # (B) the four conditionals, both directions of the implication side by side
        ax = axs[1, ci]
        for k, (tag, lab, col) in enumerate(CONDS):
            p_, l_, h_ = float(r[tag]), float(r[tag + "_lo"]), float(r[tag + "_hi"])
            ax.barh(k, p_, height=0.62, color=col, alpha=0.85, zorder=3)
            ax.errorbar([p_], [k], xerr=[[p_ - l_], [h_ - p_]], fmt="none", ecolor="k", lw=1.4, capsize=3, zorder=4)
            if p_ >= 0.42:                                                 # a long bar carries its own label
                ax.text(0.018, k, f"{p_:.2f}", va="center", ha="left", fontsize=8.5, fontweight="bold", color="w", zorder=5)
                ax.text(0.145, k, f"({int(r[tag + '_k'])}/{int(r[tag + '_n'])})", va="center", ha="left", fontsize=7.5, color="w", zorder=5)
            else:                                                          # a short one: the label sits clear of the interval
                ax.text(h_ + 0.022, k, f"{p_:.2f}", va="center", ha="left", fontsize=8.5, fontweight="bold", color="0.15", zorder=5)
                ax.text(h_ + 0.105, k, f"({int(r[tag + '_k'])}/{int(r[tag + '_n'])})", va="center", ha="left", fontsize=7.5, color="0.4", zorder=5)
        ax.set_yticks(range(len(CONDS))); ax.set_yticklabels([l for _, l, _ in CONDS] if ci == 0 else [""] * len(CONDS), fontsize=8)
        ax.set_xlim(0, 1.0); ax.invert_yaxis(); ax.grid(alpha=0.25, lw=0.5, axis="x")
        ax.set_xlabel("probability (Wilson 68 %)", fontsize=9)
        ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
        ax.axvline(float(r["f_dusty"]), color="#d7191c", lw=1.1, ls=":", zorder=2)
        ax.axvline(1 - float(r["f_dusty"]), color="#2c7bb6", lw=1.1, ls=":", zorder=2)
        ax.text(0.5, 1.012, f"base rates: dusty {100 * float(r['f_dusty']):.0f} %, never exposed {100 * float(r['f_never']):.1f} % (dotted)",
                transform=ax.transAxes, ha="center", va="bottom", fontsize=7.2, color="0.35")

        # (C) what the never-exposed are
        ax = axs[2, ci]
        C = CON8[CON8["box"] == b].set_index("prop")
        labs, aucs, ps = [], [], []
        for c, lab in P8_CONTRAST:
            if c in C.index and np.isfinite(C.loc[c, "auc_never_gt_exposed"]):
                labs.append(lab); aucs.append(float(C.loc[c, "auc_never_gt_exposed"])); ps.append(float(C.loc[c, "p"]))
        if not labs:
            ax.text(0.5, 0.5, f"only {int(r['n_never'])} never-exposed galax"
                    + ("y" if int(r["n_never"]) == 1 else "ies") + " in this box —\nno contrast to draw",
                    transform=ax.transAxes, ha="center", va="center", fontsize=9, color="0.4")
            ax.set_yticks([]); ax.set_xlim(0, 1)
            ax.set_xlabel("AUC(never exposed $>$ exposed)", fontsize=9)
            continue
        yy = np.arange(len(labs))
        ax.barh(yy, np.array(aucs) - 0.5, left=0.5, height=0.6, color=["#d7191c" if a > 0.5 else "#2c7bb6" for a in aucs], alpha=0.85, zorder=3)
        for k, (a_, p_) in enumerate(zip(aucs, ps)):
            t_ = ("*" if p_ < 0.05 else "") + f"{a_:.2f}"
            xt = a_ + (0.014 if a_ > 0.5 else -0.014)
            if xt < 0.09 or xt > 0.91:                                     # the bar reaches an edge: the label goes inside it
                ax.text(a_ + (-0.014 if a_ > 0.5 else 0.014), k, t_, va="center", ha="right" if a_ > 0.5 else "left",
                        fontsize=7.5, color="w", fontweight="bold", zorder=5)
            else:
                ax.text(xt, k, t_, va="center", ha="left" if a_ > 0.5 else "right", fontsize=7.5, zorder=5)
        ax.axvline(0.5, color="k", lw=1.2, zorder=4)
        ax.set_yticks(yy); ax.set_yticklabels(labs if ci == 0 else [""] * len(labs), fontsize=8)
        ax.set_xlim(0, 1); ax.invert_yaxis(); ax.grid(alpha=0.25, lw=0.5, axis="x")
        ax.set_xlabel("AUC(never exposed $>$ exposed)   * $p < 0.05$", fontsize=9)
    fig.suptitle("exposure is NECESSARY for dust destruction, not sufficient — and the converse fails on base rates\n"
                 r"(A) the dusty fraction against the exposure: a STEP at the foot of $E_x$, then a plateau — "
                 r"(B) both directions of the implication — (C) what the never-exposed are", fontsize=11.5)
    paper_save(fig, "agn_states_exposure_implication")
    plt.show()

    P8.to_csv(os.path.join(OUT, "agn_states_exposure_implication.csv"), index=False)
    SCAN8.to_csv(os.path.join(OUT, "agn_states_exposure_saturation.csv"), index=False)
    CON8.to_csv(os.path.join(OUT, "agn_states_exposure_neverexposed.csv"), index=False)
    print(f"conditionals -> {os.path.join(OUT, 'agn_states_exposure_implication.csv')}; the saturation scan -> ..._saturation.csv; "
          f"the never-exposed contrast -> ..._neverexposed.csv")

## Part 9 — the figure: the channel decides whether dust *can* be destroyed, not whether it *is*

Parts 7, 8 and 8b are the working. This part is the argument, on one page, for one box.

**Top row — what the AGN history does explain.** (a) The channel is *necessary*: essentially every dust-poor QG has been through it ($P(\text{ever exposed} \mid \text{dust-poor})$ printed in the panel), and the never-exposed are almost all dusty — but the curve then flattens after a fraction of a Gyr, so having more of it changes nothing. (b) And when it stops, the dust comes back: the dusty fraction climbs with $t_{\rm quiet}$, well above the galaxies still being dosed. (c) Those two facts are a *timing* story, and panel (c) asks how far it goes. Every dusty QG is put in exactly one box:

| category | meaning |
|---|---|
| **never armed** | the channel never fired |
| **under-dosed** | it fired, but for less than `PF_EX` |
| **dosed, then quiet** | fully dosed, then off for at least `PF_TQ` — the dust had time to regrow |
| **dosed and still firing** | fully dosed, being dosed *now*, and dusty anyway |

The first three are explained by when the AGN fired. The fourth is not, and it is not a fringe.

**Bottom rows — the residual, one row per box.** Those "dosed and still firing" galaxies, dusty against dust-poor, on every axis the AGN history offers: age, exposure, integrated coupling, the mass accreted through the channel, black hole mass, stellar mass, sSFR, gas fraction, molecular fraction — and then, in the framed panel, the dust-to-gas ratio. Every box that has at least `PF_MIN_CLASS` galaxies on both sides gets its own row (`PF_ROWS` trims the list), so the comparison is not made once and asserted to generalise: it is drawn at each resolution and the reader can see whether it holds. The individual galaxies are always rugged along the bottom of each panel — in the smaller boxes a class can be a handful of objects, and a seven-galaxy histogram must not be read as a distribution.

**Adding an axis** is one entry in `PF_PROPS`. Any column of Part 7's `DOSE` works directly; anything the cell cannot find there it looks up in the Part 0 catalogue frame `QM` and merges on `(box, snap, gal_id)` — which is how `lssfr` gets in, since Part 7 integrates histories and never carried the anchor sSFR across. `PF_DERIVED` holds the recipes built from other columns (`dg`, and `dh2` / `sfe` if wanted). A name that is in none of the three raises with all three lists printed, rather than failing on a missing key deep in the drawing. **The last entry is the punchline** — it is the one drawn framed in red, so whatever you want highlighted goes at the end of the list. (`dg` holds that slot because it is the observable; move `dtm` there if you would rather frame the physical one.)

That contrast is the whole point, and it is why the figure is built this way rather than as another correlation plot: **this is a matched comparison, not a fit.** These galaxies have the same age, the same dose, the same black hole and the same gas reservoir. If the dust content followed from the AGN history, they could not differ — and they differ by more than a dex in $M_{\rm dust}/M_{\rm gas}$, at a gas fraction that is identical between them.

**The metallicity and dust-to-metal panels are there to close the obvious escape route.** A dust-to-gas difference has two possible causes: the galaxies differ in *how many metals they have*, or in *how much of those metals sit in grains*. So $\log Z_{\rm gas}$ is drawn immediately before $\log$ (dust-to-metal), and the pair decides between them. If the split were enrichment, $Z_{\rm gas}$ would carry it and the dust-to-metal ratio would not. `PF_Z_HAS_DUST` states the one assumption in that ratio: SIMBA's dust model depletes grain metals *out of* the gas phase, so the metal budget is taken as $Z_{\rm gas}M_{\rm gas} + M_{\rm dust}$; set it `True` if your catalogue's `zgas` is already the total. The metallicity comes from `zgas` at the anchor, which **Part 7 records** — a `DOSE` built before that column existed raises with the instruction to re-run Part 7 rather than failing on a missing key.

Two things to keep honest about it. The categories are cuts, so `PF_EX` and `PF_TQ` move the shares — the cell prints the full table for **every** box, and the residual's AUCs are reported per box, so the reader can see that the split reproduces at three resolutions rather than taking the drawn box on faith. And the last two panels' AUCs are not free measurements: the sample is split at $\log M_{\rm dust}/M_\star = $ `DUSTY_CUT`, and both $M_{\rm dust}/M_{\rm gas}$ and the dust-to-metal ratio share that numerator. **The argument therefore rests on the denominators, not on them.** Gas, molecular gas and metallicity are all independent of the split axis, and all three show nothing — so the separation cannot be placed in the gas reservoir or in the enrichment, which leaves the grains. Read the row left to right and that is the whole inference: the AGN axes are coin flips, the gas is a coin flip, the metals are a coin flip, and what is left is how much of the metal budget is locked in dust.

Outputs: `agn_states_the_argument.{png,pdf}`, `agn_states_the_argument_categories.csv` (the decomposition, every box) and `agn_states_the_argument_residual_auc.csv` (the matched comparison, every box and property). `PF_BOX` chooses the box of the top row, `PF_ROWS` the boxes that get a residual row, `PF_PROPS` the axes.

In [ ]:
# ── Part 9 — THE FIGURE: the AGN channel gates the dust, the timing explains most of the dusty QGs, and the rest is not an AGN story at all ──
from matplotlib.gridspec import GridSpec
import matplotlib.patheffects as pe
PF_BOX   = None       # the box the figure is drawn for (None = the one with the most galaxies); the others are printed and written to the CSV
# PF_EX ('fully dosed'), PF_TQ ('still firing'), CATS and the route_cat / route_grp rules now live in PART 2a — the three
# routes are defined there because Parts 2b-4 are split on them.  Change them there and this figure follows.
PF_NBIN  = 8          # the E_x bins of panel (a)
PF_ROWS  = None       # the boxes that get a residual row of their own (None = every box with at least PF_MIN_CLASS on both sides,
                      # so the reproduction across resolutions is on the page; ["cis100", "cis50"] to trim it)
PF_MIN_CLASS = 3      # a residual row needs this many dusty AND this many dust-poor
PF_NHIST = 20         # under this many galaxies a class is still drawn as a histogram, but the individual galaxies are always
                      # rugged along the bottom of the panel — a 7-galaxy histogram must not be read as a distribution
PF_NMIN  = {"cis100": 15, "cis50": 8, "cis25": 5}
# The bottom row. Any column of Part 7's DOSE works; anything else is looked up in the Part 0 catalogue frame QM and merged on
# (box, snap, gal_id) — so "lssfr", "lsig", "tau_q", "t_sft"... are all one line here. PF_DERIVED holds the ones built from others.
# THE LAST ENTRY IS THE PUNCHLINE and is drawn framed in red: keep the axis you want highlighted at the end of the list.
PF_PROPS = [("age", "age [Gyr]"), ("e_x", r"$E_x$ [Gyr]"), ("k_time_act", r"$\int w_{\rm jet}{\rm d}t$ [Gyr]"),
            ("lm_acc_x", r"$\log \int_{\rm X-ray}\dot M_{\rm BH}{\rm d}t$"), ("log_mbh", r"$\log M_{\rm BH}$"),
            ("log_mstar", r"$\log M_\star$"), ("lssfr", r"$\log$ sSFR [yr$^{-1}$]"),
            ("lfh2", r"$\log M_{\rm H_2}/M_\star$"),
            ("lz", r"$\log Z_{\rm gas}$"),
            ("dtm", r"$\log$ dust-to-metal"),
            ("dh2", r"$\log M_{\rm dust}/M_{\rm H_2}$")]
# Whether caesar's gas metallicity already counts the metals locked in grains. SIMBA's dust model DEPLETES them from the gas
# phase, so the default (False) treats zgas as gas-phase only and the total metal budget as Z_gas M_gas + M_dust. If your
# catalogue's zgas is the TOTAL metal fraction instead, set this True and the dust is not added a second time.
PF_Z_HAS_DUST = False


def _pf_dtm(d):
    """log10 of M_dust / M_metals, the metal budget being Z_gas M_gas (+ M_dust, unless zgas already carries it).
    Everything is per unit M*, so the stellar mass cancels and lfd / lfgas can be used directly."""
    fd, fg, z = (10 ** d["lfd"].to_numpy(float), 10 ** d["lfgas"].to_numpy(float), d["zgas"].to_numpy(float))
    met = np.where(np.isfinite(z) & (z > 0), z * fg, np.nan)          # metals in the gas phase, per unit M*
    tot = met if PF_Z_HAS_DUST else met + fd                          # ... plus the metals already in grains
    return np.log10(np.where((tot > 0) & np.isfinite(tot), fd / np.where(tot > 0, tot, np.nan), np.nan))


PF_DERIVED = {"dg": lambda d: d["lfd"].to_numpy(float) - d["lfgas"].to_numpy(float),          # the dust-to-gas ratio
              "dh2": lambda d: d["lfd"].to_numpy(float) - d["lfh2"].to_numpy(float),          # dust per unit H2
              "sfe": lambda d: d["lssfr"].to_numpy(float) - d["lfh2"].to_numpy(float),        # the molecular depletion rate
              "lz": lambda d: np.log10(np.where(d["zgas"].to_numpy(float) > 0, d["zgas"].to_numpy(float), np.nan)),
              "dtm": _pf_dtm}
pf_nmin = lambda b: int(PF_NMIN.get(b, 8)) if isinstance(PF_NMIN, dict) else int(PF_NMIN)

if "DOSE" not in globals() or not len(DOSE):
    print("Part 9: Part 7 built no records — nothing to draw.")
else:
    if "wilson" not in globals():                      # Part 8 defines it; keep this cell runnable on its own
        def wilson(k, n, z=1.0):
            if n <= 0:
                return np.nan, np.nan, np.nan
            p = k / n; d = 1 + z * z / n
            c = (p + z * z / (2 * n)) / d
            h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
            return p, min(p, max(0.0, c - h)), max(p, min(1.0, c + h))    # clamped to contain p: see Part 8

    F = DOSE.copy()
    F["dusty"] = F["lfd"].to_numpy(float) >= DUSTY_CUT
    F["never"] = F["n_x"].to_numpy(int) == 0
    # ── resolve every axis the figure asks for: DOSE first, then the catalogue frame QM, then the derived recipes ──
    _want = [c for c, _ in PF_PROPS]
    _from_qm = [c for c in _want if c not in F.columns and c not in PF_DERIVED and c in QM.columns]
    if _from_qm:
        F = F.merge(QM[KEY + _from_qm].drop_duplicates(subset=KEY), on=KEY, how="left")
        print(f"Part 9: {', '.join(_from_qm)} taken from the Part 0 catalogue frame (not carried in Part 7's integrals).")
    for c in _want:
        if c not in F.columns and c in PF_DERIVED:
            F[c] = PF_DERIVED[c](F)
    _needz = [c for c in _want if c in ("lz", "dtm")]
    if _needz and "zgas" not in F.columns:
        raise KeyError(f"Part 9: {_needz} needs the anchor gas metallicity, which Part 7 records as 'zgas' — this DOSE was built "
                       f"before that column existed. Re-run Part 7 (one pass over the histories caches) and this cell will pick it up. "
                       f"To draw the figure without it, drop {_needz} from PF_PROPS.")
    _missing = [c for c in _want if c not in F.columns]
    if _missing:
        raise KeyError(f"Part 9: {_missing} is in PF_PROPS but is neither a column of DOSE nor of QM nor a PF_DERIVED recipe. "
                       f"Available in DOSE: {sorted(DOSE.columns)}; in QM: {sorted(set(QM.columns) - set(DOSE.columns))}; "
                       f"derived: {sorted(PF_DERIVED)}.")
    F["cat"] = route_cat(F)                                # Part 2a's rule, verbatim
    BOX = PF_BOX or F["box"].value_counts().idxmax()

    # ── the numbers, every box, before the drawing ──
    print(f"Part 9 — the dusty QGs split by what the AGN history can explain "
          f"('fully dosed' = E_x >= {PF_EX:g} Gyr, 'still firing' = t_quiet < {PF_TQ:g} Gyr).")
    frows = []
    for b in BOXES_OK:
        G = F[F["box"] == b]
        if not len(G):
            continue
        nd = int(G["dusty"].sum())
        print(f"  {BOX_LABEL[b]}: {len(G)} quenched, {nd} dusty ({100 * nd / len(G):.0f} %)")
        for c, _, what in CATS:
            S = G[G["cat"] == c]
            k = int(S["dusty"].sum())
            print(f"    {c:24s} ({what:46s}): {len(S):4d} galaxies, {k:3d} dusty "
                  f"({100 * k / max(len(S), 1):3.0f} % of them) = {100 * k / max(nd, 1):3.0f} % of the box's dusty QGs")
            frows.append(dict(box=b, cat=c, n=len(S), n_dusty=k, f_dusty_in_cat=k / max(len(S), 1), share_of_dusty=k / max(nd, 1)))
    # the residual: the same test on every axis
    print(f"\n  THE RESIDUAL — 'dosed and still firing', dusty against dust-poor. Every axis, AUC (0.5 = indistinguishable):")
    arows = []
    for b in BOXES_OK:
        R = F[(F["box"] == b) & (F["cat"] == CATS[3][0])]
        if int(R["dusty"].sum()) < 3 or int((~R["dusty"]).sum()) < 3:
            print(f"    {BOX_LABEL[b]}: too few on one side ({int(R['dusty'].sum())} dusty, {int((~R['dusty']).sum())} pure)")
            continue
        print(f"    {BOX_LABEL[b]} (N = {len(R)}: {int(R['dusty'].sum())} dusty, {int((~R['dusty']).sum())} dust-poor)")
        for c, _ in PF_PROPS:
            auc, pv, n1, n0 = mw_auc(R[c], R["dusty"].to_numpy(bool))
            arows.append(dict(box=b, prop=c, auc=auc, p=pv, n_dusty=n1, n_pure=n0,
                              med_dusty=float(np.nanmedian(R[R["dusty"]][c])), med_pure=float(np.nanmedian(R[~R["dusty"]][c]))))
            print(f"      {c:12s} AUC = {auc:.3f} (p = {pv:8.1g})   dusty {float(np.nanmedian(R[R['dusty']][c])):7.2f}  "
                  f"dust-poor {float(np.nanmedian(R[~R['dusty']][c])):7.2f}")
    PF_CAT, PF_AUC = pd.DataFrame(frows), pd.DataFrame(arows)

    # ── the figure ──
    G = F[F["box"] == BOX]
    E = G[~G["never"]]
    ROWS = [b for b in (PF_ROWS if PF_ROWS is not None else BOXES_OK)
            if int(F[(F["box"] == b) & (F["cat"] == CATS[3][0])]["dusty"].sum()) >= PF_MIN_CLASS
            and int((~F[(F["box"] == b) & (F["cat"] == CATS[3][0])]["dusty"]).sum()) >= PF_MIN_CLASS]
    _skip = [b for b in (PF_ROWS if PF_ROWS is not None else BOXES_OK) if b not in ROWS]
    if _skip:
        print(f"\n  residual rows skipped (fewer than {PF_MIN_CLASS} on one side): " + ", ".join(_skip))
    fig = plt.figure(figsize=(max(15.0, 2.2 * len(PF_PROPS)), 4.6 + 3.05 * len(ROWS)))
    gs = GridSpec(2 + len(ROWS), len(PF_PROPS) * 3, figure=fig,           # row 1 is an empty spacer: the caption lives there
                  height_ratios=[1.18, 0.16] + [1.0] * len(ROWS), hspace=0.55, wspace=0.55)
    nper = len(PF_PROPS)

    def frac_vs(ax, T, col, edges_from, xlab, extra=None, extra_lab=None, extra_col="#7b3294"):
        xs, ys, lo, hi = [], [], [], []
        x, d = T[col].to_numpy(float), T["dusty"].to_numpy(bool)
        if len(T) >= 2 * pf_nmin(BOX):
            nb_ = int(np.clip(len(T) // pf_nmin(BOX), 2, PF_NBIN))
            e = np.unique(np.percentile(x, np.linspace(0, 100, nb_ + 1)))
            for i in range(len(e) - 1):
                m = (x >= e[i]) & ((x <= e[i + 1]) if i == len(e) - 2 else (x < e[i + 1]))
                if m.sum() >= pf_nmin(BOX):
                    p_, l_, h_ = wilson(int(d[m].sum()), int(m.sum()))
                    xs.append(float(np.median(x[m]))); ys.append(p_); lo.append(l_); hi.append(h_)
        xhi = float(np.percentile(x, 99)) if len(x) else 1.0
        sx = -0.135 * max(xhi, 1e-6)
        if xs:
            ax.errorbar(xs, ys, yerr=[np.array(ys) - np.array(lo), np.array(hi) - np.array(ys)], fmt="-o", color="k",
                        lw=2.4, ms=6, capsize=3.5, zorder=4)
        if extra is not None and len(extra):
            p_, l_, h_ = wilson(int(extra["dusty"].sum()), len(extra))
            ax.errorbar([sx], [p_], yerr=[[p_ - l_], [h_ - p_]], fmt="s", color=extra_col, ms=10, lw=2.4, capsize=4, zorder=5)
            ax.axvline(0.5 * sx, color="0.55", lw=1.1, ls=":", zorder=1)
            up = p_ <= 0.5                                        # the label goes on whichever side of the marker has room
            ax.text(sx, p_ + (0.055 if up else -0.055), f"{extra_lab}\n({len(extra)})", ha="center",
                    va="bottom" if up else "top", fontsize=8.5, color=extra_col, fontweight="bold", zorder=6)
        ax.axhline(float(G["dusty"].mean()), color="0.45", lw=1.3, ls="--", zorder=2)
        ax.set_ylim(0, 1.02); ax.set_xlim(1.85 * sx, xhi * 1.05)      # the extra room on the left is for the censored point's label
        ax.set_xticks([t for t in ax.get_xticks() if t >= 0.25 * sx]); ax.set_xlim(1.85 * sx, xhi * 1.05)
        ax.set_xlabel(xlab, fontsize=10); ax.grid(alpha=0.25, lw=0.5)

    # (a) the channel is necessary, and it saturates
    ax = fig.add_subplot(gs[0, 0:nper])
    frac_vs(ax, E, "e_x", None, r"$E_x = \int_{\rm X-ray}{\rm d}t$   [Gyr]",
            extra=G[G["never"]], extra_lab="never\nexposed")
    ax.set_ylabel("fraction dusty", fontsize=11)
    npure = int((~G["dusty"]).sum()); nexp = int((~G[~G["dusty"]]["never"]).sum())
    ax.set_title("(a)  the channel is necessary — and then it saturates", fontsize=11, pad=30)
    ax.text(0.97, 0.93, f"$P$(ever exposed | dust-poor)\n$= {nexp}/{npure} = {nexp / npure:.3f}$",
            transform=ax.transAxes, ha="right", va="top", fontsize=9.5,
            bbox=dict(fc="w", ec="#2c7bb6", lw=1.2, alpha=0.95, pad=3.5))

    # (b) once it stops, the dust comes back
    ax = fig.add_subplot(gs[0, nper:2 * nper])
    Q = E[E["t_quiet"] > 1e-9]
    frac_vs(ax, Q, "t_quiet", None, r"$t_{\rm quiet}$ since the last X-ray episode   [Gyr]",
            extra=E[E["t_quiet"] <= 1e-9], extra_lab="firing at\nthe anchor", extra_col="#d7191c")
    ax.set_title("(b)  the longer it has been off, the more of them are dusty", fontsize=11, pad=30)

    # (c) so what does the timing explain?
    ax = fig.add_subplot(gs[0, 2 * nper:])
    fres_top = np.nan
    for bi, b in enumerate(BOXES_OK):
        Gb = F[F["box"] == b]
        nd = int(Gb["dusty"].sum())
        if not nd:
            continue
        left, y = 0.0, -bi
        for c, col, what in CATS:
            k = int(Gb[Gb["cat"] == c]["dusty"].sum())
            w = k / nd
            ax.barh([y], [w], left=left, height=0.52, color=col, edgecolor="w", lw=1.4,
                    hatch="///" if c == CATS[3][0] else None, zorder=3)
            if w > 0.075:
                ax.text(left + w / 2, y, f"{100 * w:.0f}", ha="center", va="center", fontsize=9.5, fontweight="bold",
                        color="w" if c != CATS[2][0] else "0.12", zorder=5,
                        path_effects=[pe.withStroke(linewidth=2.2, foreground="0.15" if c != CATS[2][0] else "w")])
            if bi == 0:                                     # the category names are written once, over the top bar
                yl = 0.40 if CATS.index((c, col, what)) % 2 == 0 else 0.82
                ax.plot([left + w / 2, left + w / 2], [0.28, yl - 0.03], color=col, lw=0.9, zorder=2)
                ax.text(left + w / 2, yl, f"{c}", ha="center", va="bottom", fontsize=8.2, color=col, fontweight="bold")
            left += w
        if bi == 0:
            fres_top = float(Gb[Gb["cat"] == CATS[3][0]]["dusty"].sum()) / nd
    ax.set_xlim(0, 1); ax.set_ylim(-len(BOXES_OK) + 0.45, 1.72)
    ax.set_yticks([-i for i in range(len(BOXES_OK))])
    ax.set_yticklabels([f"{b}\n({int(F[(F['box'] == b)]['dusty'].sum())} dusty)" for b in BOXES_OK], fontsize=8)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0]); ax.set_xticklabels(["0", "25", "50", "75", "100 %"])
    ax.set_xlabel("share of the box's dusty QGs", fontsize=10)
    ax.set_title("(c)  what the AGN timing explains — and what it does not", fontsize=11, pad=30)
    if np.isfinite(fres_top):
        ax.annotate("", xy=(0, 1.34), xytext=(1.0 - fres_top, 1.34), arrowprops=dict(arrowstyle="<->", color="0.45", lw=1.2))
        ax.text((1.0 - fres_top) / 2, 1.40, "the AGN timing explains these", ha="center", va="bottom", fontsize=8.5, color="0.35")
        ax.annotate("", xy=(1.0, 1.34), xytext=(1.0 - fres_top, 1.34), arrowprops=dict(arrowstyle="<->", color="#d7191c", lw=1.8))
        ax.text(1.0 - fres_top / 2, 1.40, "and not these", ha="center", va="bottom", fontsize=8.5, color="#d7191c", fontweight="bold")

    _yres = gs[2, 0].get_position(fig).y1                     # the top of the first residual row: the caption hangs off it
    # ── one residual row per box: the same nine axes, the same matched comparison ──
    for ri, b in enumerate(ROWS):
        R = F[(F["box"] == b) & (F["cat"] == CATS[3][0])]
        d_all = R["dusty"].to_numpy(bool)
        for i, (c, lab) in enumerate(PF_PROPS):
            ax = fig.add_subplot(gs[2 + ri, 3 * i:3 * i + 3])
            last = (c == PF_PROPS[-1][0])
            x = R[c].to_numpy(float); d = d_all
            fin = np.isfinite(x)
            if fin.sum() > 5:
                b_ = np.linspace(np.percentile(x[fin], 1), np.percentile(x[fin], 99), 17)
                for m, col in ((d & fin, "#d7191c"), (~d & fin, "#2c7bb6")):
                    if m.sum() >= 3:
                        ax.hist(x[m], bins=b_, density=True, color=col, alpha=0.34, zorder=2)
                        ax.hist(x[m], bins=b_, density=True, histtype="step", color=col, lw=2.0, zorder=3)
                        ax.axvline(np.median(x[m]), color=col, lw=1.6, ls="--", zorder=4)
                        # the galaxies themselves, always: a seven-object histogram is not a distribution
                        ax.plot(x[m], np.full(int(m.sum()), 0.035 if col == "#d7191c" else 0.085), "|",
                                color=col, ms=5.5, mew=1.1, alpha=0.85, transform=ax.get_xaxis_transform(), zorder=5,
                                clip_on=True)
            a_ = PF_AUC[(PF_AUC["box"] == b) & (PF_AUC["prop"] == c)]
            auc = float(a_["auc"].iloc[0]) if len(a_) else np.nan
            ax.set_yticks([]); ax.grid(alpha=0.22, lw=0.5, axis="x")
            ax.set_xlabel(lab if ri == len(ROWS) - 1 else "", fontsize=9.5,
                          color="#d7191c" if last else "k", fontweight="bold" if last else "normal")
            ax.set_title(f"AUC = {auc:.2f}", fontsize=10.5, color="#d7191c" if last else "0.3",
                         fontweight="bold" if last else "normal")
            if last:
                for sp in ax.spines.values():
                    sp.set_edgecolor("#d7191c"); sp.set_linewidth(2.2)
            if i == 0:
                ax.set_ylabel(f"{BOX_LABEL[b]}\n" + rf"{int(d.sum())} dusty vs {int((~d).sum())} dust-poor",
                              fontsize=9, fontweight="bold")
    fig.legend(handles=[Line2D([], [], color="#d7191c", lw=2.6, label="dusty"),
                        Line2D([], [], color="#2c7bb6", lw=2.6, label="dust-poor"),
                        Line2D([], [], color="0.4", lw=0, marker="|", ms=7, mew=1.2, label="the individual galaxies")],
               frameon=False, fontsize=10.5, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 0.001))
    fig.text(0.5, _yres + 0.012, rf"THE RESIDUAL — the galaxies that are FULLY DOSED ($E_x \geq {PF_EX:g}$ Gyr) AND BEING DOSED NOW "
             rf"($t_{{\rm quiet}} < {PF_TQ:g}$ Gyr): dusty against dust-poor, one row per box, "
             r"on every axis the AGN history offers — and then on the dust-to-H$_2$ ratio",
             ha="center", va="bottom", fontsize=11.5, fontweight="bold")
    def _gap(prop):                                                   # the median separation of the two populations on one axis
        a_ = PF_AUC[(PF_AUC["box"] == BOX) & (PF_AUC["prop"] == prop)]
        return abs(float(a_["med_dusty"].iloc[0]) - float(a_["med_pure"].iloc[0])) if len(a_) else np.nan
    _have_z = "lz" in [c for c, _ in PF_PROPS] and "dtm" in [c for c, _ in PF_PROPS]
    fig.suptitle(r"the X-ray channel decides whether dust $can$ be destroyed, not whether it $is$:  "
                 + (rf"same age, same dose, same $M_{{\rm BH}}$, same H$_2$, same METALS — and {_gap('dtm'):.1f} dex apart in dust-to-metal"
                    if _have_z and np.isfinite(_gap("dtm")) else
                    rf"same age, same dose, same $M_{{\rm BH}}$, same H$_2$ — and {_gap('dh2'):.1f} dex apart in dust-to-H$_2$"),
                 fontsize=13.5, y=0.985)
    paper_save(fig, "agn_states_the_argument")
    plt.show()

    PF_CAT.to_csv(os.path.join(OUT, "agn_states_the_argument_categories.csv"), index=False)
    PF_AUC.to_csv(os.path.join(OUT, "agn_states_the_argument_residual_auc.csv"), index=False)
    print(f"\ncategories -> {os.path.join(OUT, 'agn_states_the_argument_categories.csv')}; the residual AUCs -> ..._residual_auc.csv")

## Part 11 — the dust tracks of the three routes: destroyed, held, or made?

Every comparison so far has been made *at the anchor*, which can only say where the three populations ended up. This part puts them on the quenching clock and asks how they got there, which is the question that separates the mechanisms. Take the three routes of Part 9 — the **unexplained residual** (dusty, fully dosed, being dosed now), the **dusty the timing does explain**, and the **dust-poor** — and follow $M_{\rm dust}/M_{\rm gas}$ and the dust-to-metal ratio back along each galaxy's main branch on $t - t_{\rm SFT}$.

The prediction to test is sharp. If the residual group is dusty because dust is being **made** in it after quenching, its D/G must *rise* while the dust-poor curve falls. If it is dusty because its dust is being **kept**, the curve is flat. If it merely started higher and is on the same downward road, the curve falls like the others — only from a higher place, or more slowly.

**Why D/G alone cannot answer it, and D/M is drawn beside it.** Dust-to-gas can rise for two quite different reasons: more of the metals condense into grains, or the gas simply becomes more metal-rich. The $Z_{\rm gas}$ row settles which — and the dust-to-metal row is the version of the question that enrichment cannot fake, since a rising D/M means grains are being built out of a fixed metal budget. The gas fraction and sSFR rows are there as the control: if the three routes are on identical gas and star-formation tracks, then whatever separates them is not the fuel supply.

**The slopes are the measurement, not the eye.** For every track the cell fits $d/dt$ over `P11_FIT` (0 to 3 Gyr after the SFT, drawn as the shaded strip) inside a bootstrap over the galaxies, and labels each as **RISES** / **FALLS** / **holds** according to whether the 16–84 interval clears zero. That is what makes "the dusty ones hold their dust" a testable claim rather than an impression from a noisy curve.

**Two things to keep in mind reading it.** The routes are defined *at the anchor*, so these are the histories of galaxies selected by where they ended — the curves are conditioned on the outcome, and a population that is dusty at the anchor was necessarily dustier before. And each chain stops at its own anchor, so the right-hand end of every track is built from progressively fewer galaxies; the count histogram behind each panel is the guard against reading a late excursion as a trend. Output: `agn_states_dust_tracks_routes.{png,pdf}` and `agn_states_dust_tracks_routes_slopes.csv` (every box, quantity and route, with the bootstrap interval).

In [ ]:
# ── Part 11 — the D/G and D/M TRACKS on the SFT clock, one curve per route: is the dust being destroyed, held, or made? ──
# The anchor comparisons say where the three routes ended up. This says how they got there, which is the discriminating question:
# if the dust of the unexplained group is made in situ after quenching, its D/G must RISE while the dust-poor curve falls.
P11_ROWS = [("dh2",   r"$\log M_{\rm dust}/M_{\rm H_2}$"),
            ("dtm",   r"$\log$ dust-to-metal"),
            ("lz",    r"$\log Z_{\rm gas}$"),
            ("lfd",   r"$\log M_{\rm dust}/M_\star$"),
            ("lfh2",  r"$\log 1.36\,M_{\rm H_2}/M_\star$"),
            ("lssfr", r"$\log$ sSFR [yr$^{-1}$]")]
P11_FIT   = (0.0, 3.0)     # the window, in t - t_SFT [Gyr], over which each track's SLOPE is measured and printed
P11_NMIN  = 1              # a track epoch needs this many galaxies
P11_GRPS  = [("dusty, dosed & firing",   "#d7191c", "the unexplained residual"),
             ("dusty, timing-explained", "#f0a860", "dusty, but the timing explains it"),
             ("dust-poor",               "#2c7bb6", "everything below DUSTY_CUT")]
G11 = [g for g, _, _ in P11_GRPS]
G11C = {g: c for g, c, _ in P11_GRPS}
P11_ZDUST = bool(globals().get("PF_Z_HAS_DUST", False))     # Part 9's convention for the metal budget, reused verbatim

if "DOSE" not in globals() or "F" not in globals() or not len(DOSE):
    print("Part 11: needs Part 9's frame F — run Parts 7 and 9 first.")
else:
    M11 = F.copy()
    M11["grp"] = np.where(M11["dusty"] & (M11["cat"] == CATS[3][0]), G11[0], np.where(M11["dusty"], G11[1], G11[2]))
    grp_of = {(b_, int(s_), int(g_)): v for b_, s_, g_, v in zip(M11["box"], M11["snap"], M11["gal_id"], M11["grp"])}
    sft11 = {(b_, int(s_), int(g_)): float(v_) for b_, s_, g_, v_ in zip(QM["box"], QM["snap"], QM["gal_id"], QM["t_sft"])}

    TRK11, N11 = {}, {}
    for b in BOXES_OK:
        want = {k[1:]: v for k, v in grp_of.items() if k[0] == b}
        trk = {q: {g: [] for g in G11} for q, _ in P11_ROWS}
        with h5py.File(HIST_FILE.format(box=b), "r") as f:
            for grpname in f:
                a = int(grpname.replace("snap", ""))
                if not any(k[0] == a for k in want):
                    continue
                gh = f[grpname]
                t = gh["t_yr"][:] / 1e9; order = np.argsort(t); t = t[order]
                ms11, sf11, mg11 = (gh[c][:][order] for c in ("mstar", "sfr", "mgas"))
                md11 = gh["mdust"][:][order] if "mdust" in gh else None
                mh11 = gh["mh2"][:][order] if "mh2" in gh else None
                zg11 = gh["zgas"][:][order] if "zgas" in gh else None
                if md11 is None:
                    continue
                with np.errstate(all="ignore"):
                    ex11 = np.isfinite(ms11) & (ms11 > 0)
                    den = np.where(ex11, ms11, np.nan)
                    fd11 = np.where(ex11, md11 / den, np.nan)
                    fg11 = np.where(ex11, mg11 / den, np.nan)                       # the TOTAL gas: only the metal budget below uses it
                    fh11 = np.where(ex11 & (mh11 > 0), HE * mh11 / den, np.nan) if mh11 is not None else np.full_like(fd11, np.nan)
                    met = (zg11 * fg11) if zg11 is not None else np.full_like(fg11, np.nan)
                    tot = met if P11_ZDUST else met + fd11
                    Q11 = {"lfd":   np.log10(np.clip(fd11, 10 ** P9_FLOOR, None)),
                           "lfh2":  np.log10(fh11),
                           "dh2":   np.log10(np.where(fh11 > 0, HE * fd11 / fh11, np.nan)),      # per unit BARE H2 mass
                           "dtm":   np.log10(np.where(tot > 0, fd11 / np.where(tot > 0, tot, np.nan), np.nan)),
                           "lz":    np.log10(np.where((zg11 if zg11 is not None else np.nan) > 0, zg11, np.nan)) if zg11 is not None else np.full_like(fd11, np.nan),
                           "lssfr": np.log10(np.clip(np.where(ex11, np.clip(sf11, 0, None) / den, np.nan), SSFR_FLOOR, None))}
                for j, gid in enumerate(gh["gal_ids"][:]):
                    key = (a, int(gid))
                    if key not in want:
                        continue
                    ts = sft11.get((b, a, int(gid)), np.nan)
                    if not np.isfinite(ts):
                        continue
                    trel = t - ts
                    for q, _ in P11_ROWS:
                        trk[q][want[key]].append(to_grid(trel, Q11[q][:, j]))
        TRK11[b] = {q: {g: np.array(v) for g, v in trk[q].items()} for q, _ in P11_ROWS}
        N11[b] = {g: len(trk[P11_ROWS[0][0]][g]) for g in G11}
        print(f"{BOX_LABEL[b]}: " + ", ".join(f"{g} {N11[b][g]}" for g in G11) + " chains on the SFT clock")

    # ── the slopes: the quantitative form of "declines / holds / rises" ──
    print(f"\nthe slope of each track over t - t_SFT in [{P11_FIT[0]:g}, {P11_FIT[1]:g}] Gyr, in dex per Gyr, with a bootstrap 16-84 "
          f"band over the galaxies (a track that HOLDS has a slope consistent with zero):")
    srows = []
    win = (TG >= P11_FIT[0]) & (TG <= P11_FIT[1])
    rng11 = np.random.default_rng(11)
    for b in BOXES_OK:
        print(f"  {BOX_LABEL[b]}")
        for q, lab in P11_ROWS:
            line = f"    {q:6s}"
            for g in G11:
                A = TRK11[b][q][g]
                if not len(A):
                    line += f"   {g.split(',')[0][:6]}: n/a"; continue
                sl = []
                for _ in range(200):
                    S = A[rng11.integers(0, len(A), len(A))]
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore", RuntimeWarning)
                        m = np.nanmedian(S, axis=0)
                    ok = win & np.isfinite(m) & (np.isfinite(S).sum(axis=0) >= P11_NMIN)
                    if ok.sum() >= 3:
                        sl.append(np.polyfit(TG[ok], m[ok], 1)[0])
                if len(sl) < 20:
                    line += f"   {g.split(',')[0][:6]}: n/a"; continue
                lo, md_, hi = np.percentile(sl, [16, 50, 84])
                sig = "RISES " if lo > 0 else ("FALLS " if hi < 0 else "holds ")
                line += f"   {g.split(',')[0][:6]}: {md_:+.3f} [{lo:+.3f},{hi:+.3f}] {sig}"
                srows.append(dict(box=b, q=q, grp=g, slope=md_, lo=lo, hi=hi, n=len(A)))
            print(line)
    SL11 = pd.DataFrame(srows)

    # ── the figure ──
    fig, axs = plt.subplots(len(P11_ROWS), len(BOXES_OK), figsize=(4.9 * len(BOXES_OK), 2.65 * len(P11_ROWS)),
                            squeeze=False, constrained_layout=True, sharex=True)
    for ci, b in enumerate(BOXES_OK):
        for ri, (q, lab) in enumerate(P11_ROWS):
            ax = axs[ri, ci]
            axb = ax.twinx(); axb.set_zorder(ax.get_zorder() - 1); ax.patch.set_visible(False); axb.set_yticks([])
            top = 1.0
            for g in G11:
                A = TRK11[b][q][g]
                if not len(A):
                    continue
                cent, lo, hi, nt = med_band(A)
                top = max(top, float(np.nanmax(nt)) if len(nt) else 1.0)
                axb.bar(TG, nt, width=0.9 * (TG[1] - TG[0]), color=G11C[g], alpha=0.10, lw=0, zorder=1)
                ax.fill_between(TG, lo, hi, color=G11C[g], alpha=0.20, lw=0, zorder=2)
                ax.plot(TG, cent, "-", color=G11C[g], lw=2.4, zorder=3,
                        label=(f"{g} ({len(A)})" if ri == 0 and ci == 0 else None))
            axb.set_ylim(0, top * 2.9)
            ax.axvline(0, color="k", lw=1.2, ls="--", alpha=0.85, zorder=1)
            ax.axvspan(P11_FIT[0], P11_FIT[1], color="0.85", alpha=0.30, zorder=0)
            if q == "dh2":
                for g in G11:
                    s_ = SL11[(SL11["box"] == b) & (SL11["q"] == q) & (SL11["grp"] == g)]
                    if len(s_):
                        ax.text(0.985, 0.06 + 0.13 * G11.index(g), f"{float(s_['slope'].iloc[0]):+.2f} dex/Gyr",
                                transform=ax.transAxes, ha="right", va="bottom", fontsize=7.6, color=G11C[g], fontweight="bold")
            ax.grid(alpha=0.22, lw=0.5)
            ax.set_ylabel(lab if ci == 0 else "", fontsize=9.5)
            if ri == 0:
                ax.set_title(BOX_LABEL[b], fontsize=10)
            if ri == len(P11_ROWS) - 1:
                ax.set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]", fontsize=10)
            if ri == 0 and ci == 0:
                ax.legend(frameon=False, fontsize=7.6, loc="best")
    fig.suptitle(r"the dust tracks of the three routes on the quenching clock — destroyed, held, or made?"
                 + "\n" + rf"median with a bootstrap 16-84 band, the source count behind each panel; the shaded strip is the "
                 rf"slope window $t-t_{{\rm SFT}} \in [{P11_FIT[0]:g}, {P11_FIT[1]:g}]$ Gyr", fontsize=11.5)
    paper_save(fig, "agn_states_dust_tracks_routes")
    plt.show()
    SL11.to_csv(os.path.join(OUT, "agn_states_dust_tracks_routes_slopes.csv"), index=False)
    print(f"\nslopes -> {os.path.join(OUT, 'agn_states_dust_tracks_routes_slopes.csv')}")
    print("READ IT LIKE THIS: the three routes are defined AT THE ANCHOR, so these are the histories of galaxies selected by where "
          "they ended. A rising M_dust/M_H2 in the unexplained group means its dust was BUILT after the SFT; a flat one means it was "
          "kept; a falling one that it is on its way down and merely started higher. The D/M row separates growth from enrichment: "
          "the dust-to-H2 ratio can rise on metallicity alone, D/M cannot.")

### Part 12 — the plain version: fixed $t_{\rm sq}$ bins, one point per bin, the median and its error

No tracks, no binned planes, no clouds of points. The sample is cut into fixed bins of $t_{\rm sq} = t_{\rm anchor} - t_{\rm SFT}$, and each route contributes **one point per bin**: the median $t_{\rm sq}$ of the galaxies in it, the median of the property, and the bootstrap error on that median. That is the whole figure, and it is in the form a measured sample can be dropped straight onto.

**The bins are log-spaced** (`P12_LOG`), because the interesting range runs from a couple of hundred Myr to several Gyr and a linear grid spends most of its resolution where nothing is happening. `P12_EDGES` takes explicit edges if a data set needs matching.

**Nothing is thrown away.** `P12_NMIN = 1`, so a bin holding a single galaxy is kept — but it is drawn as an **open symbol with no error bar**, because a single galaxy is a galaxy, not a median, and an error bar on it would be a fiction. The bin count is in the printed table and the CSV for every point.

**The error is on the median, not the spread.** The bar is the bootstrap 16–84 interval on the median of the bin, which is what you compare against a measured median. The population's own 16–84 spread is carried in the CSV as `<prop>_p16` / `<prop>_p84` — for a figure about how *wide* each route is rather than where its centre sits, plot that instead.

**The properties, and the two conventions inside them.** The notebook's `lfh2` carries a helium factor of 1.36; the dust-to-H$_2$ and gas-to-H$_2$ ratios take it back out, so both are ratios to the bare molecular mass. And $12+\log({\rm O/H})$ is *scaled from the total metal mass fraction* — $8.69 + \log(Z/0.0134)$, Asplund+09 — which assumes solar abundance ratios, since the catalogue carries $Z$ and not oxygen. The dashed line is solar. Both `M_{\rm H_2}/M_\star` and `M_{\rm gas}/M_{\rm H_2}` are drawn, so whichever of the two "gas–H$_2$" quantities was meant is on the page.

Output: `agn_states_tsq_binned_properties.{png,pdf}` and `agn_states_tsq_binned_properties.csv` — per box, route and bin: $N$, the median $t_{\rm sq}$, and for every property the median, its bootstrap error and the 16–84 spread.

In [ ]:
# ── Part 12 — the simple version: fixed t_sq bins, one point per bin per route, median and its error. Nothing else. ──
P12_NBIN   = 17            # the number of t_sq bins
P12_LOG    = True         # log-spaced bins in t_sq (the early times are where everything happens, so they get the resolution)
P12_EDGES  = None         # explicit t_sq edges [Gyr]; None -> P12_NBIN bins over the pooled range, log- or linearly spaced
P12_NMIN   = 1            # a bin of ONE galaxy is kept, drawn open and without an error bar — it is a galaxy, not a median
P12_NBOOT  = 2000
P12_DX     = 0.012        # a small horizontal offset per route, in log10(t_sq), so three coincident points stay separable
ZSUN, OH_SUN = 0.0134, 8.69      # Asplund+09: the solar metal mass fraction and 12+log(O/H)
HE12 = float(globals().get("HE", 1.36))

# (key, y label, the horizontal reference line to draw, or None)
P12_PROPS = [("dh2",   r"$\log M_{\rm dust}/M_{\rm H_2}$",                None),
             ("dtm",   r"$\log$ dust-to-metal",                           None),
             ("oh",    r"$12 + \log_{10}({\rm O/H})$",                    OH_SUN),
             ("lfh2",  r"$\log M_{\rm H_2}/M_\star$",                     None),
             ("lssfr", r"$\log$ sSFR  [yr$^{-1}$]",                       None),
             ("log_mstar", r"$\log M_\star$  [$M_\odot$]",                None)]
P12_GRPS = globals().get("P11_GRPS", [("dusty, dosed & firing", "#d7191c", ""), ("dusty, timing-explained", "#f0a860", ""),
                                      ("dust-poor", "#2c7bb6", "")])
G12 = [g for g, _, _ in P12_GRPS]
G12C = {g: c for g, c, _ in P12_GRPS}


def p12_build(d):
    """The six properties, all from anchor columns. lfh2 carries the notebook's helium factor; dust-to-H2 takes it back out, so it
    is a ratio to the BARE molecular mass.  The total gas survives only inside the metal budget of the dust-to-metal ratio."""
    out = {}
    lhe = np.log10(HE12)
    out["dh2"] = d["lfd"].to_numpy(float) - d["lfh2"].to_numpy(float) + lhe          # M_dust / M_H2
    out["lfh2"] = d["lfh2"].to_numpy(float)
    out["lssfr"] = d["lssfr"].to_numpy(float)
    out["log_mstar"] = d["log_mstar"].to_numpy(float)
    z = d["zgas"].to_numpy(float)
    out["oh"] = OH_SUN + np.log10(np.where(z > 0, z, np.nan) / ZSUN)                 # scaled solar: 12+log(O/H) from the total Z
    fd, fg = 10 ** d["lfd"].to_numpy(float), 10 ** d["lfgas"].to_numpy(float)
    met = z * fg
    tot = met if bool(globals().get("PF_Z_HAS_DUST", False)) else met + fd
    out["dtm"] = np.log10(np.where(tot > 0, fd / np.where(tot > 0, tot, np.nan), np.nan))
    return out


if "DOSE" not in globals() or "F" not in globals() or not len(DOSE):
    print("Part 12: needs Part 9's frame F — run Parts 7 and 9 first.")
else:
    P = F.copy()
    P["grp"] = np.where(P["dusty"] & (P["cat"] == CATS[3][0]), G12[0], np.where(P["dusty"], G12[1], G12[2]))
    if "lssfr" not in P.columns:
        P = P.merge(QM[KEY + ["lssfr"]].drop_duplicates(subset=KEY), on=KEY, how="left")
    if "t_sft" not in P.columns:
        P = P.merge(QM[KEY + ["t_sft"]].drop_duplicates(subset=KEY), on=KEY, how="left")
    P["t_sq"] = P["t_anchor"].to_numpy(float) - P["t_sft"].to_numpy(float)
    for k, v in p12_build(P).items():
        P[k] = v
    P = P[np.isfinite(P["t_sq"]) & (P["t_sq"] > 0)].copy()

    if P12_EDGES is not None:
        E12 = np.asarray(P12_EDGES, float)
    elif P12_LOG:
        E12 = np.logspace(np.log10(P["t_sq"].min() * 0.999), np.log10(P["t_sq"].max() * 1.001), P12_NBIN + 1)
    else:
        E12 = np.linspace(P["t_sq"].min() * 0.999, P["t_sq"].max() * 1.001, P12_NBIN + 1)
    print(f"Part 12 — {len(P)} galaxies with t_sq = t_anchor - t_SFT > 0, in {len(E12) - 1} "
          + ("log-spaced" if P12_LOG and P12_EDGES is None else "fixed") + " bins: "
          + ", ".join(f"{E12[i]:.2f}-{E12[i + 1]:.2f}" for i in range(len(E12) - 1)) + " Gyr.")
    print(f"  x is the MEDIAN t_sq of the galaxies in the bin, y the median of the property, the error bar the bootstrap "
          f"({P12_NBOOT} draws) 16-84 on that median. A bin of one galaxy is kept and drawn open with no bar.")
    print(f"  12+log(O/H) is scaled from the total metal mass fraction, 12+log(O/H) = {OH_SUN:g} + log(Z/{ZSUN:g}), i.e. solar "
          f"abundance ratios assumed; the dashed line is solar.")

    rng12 = np.random.default_rng(12)
    rows = []
    for b in BOXES_OK:
        for g in G12:
            S = P[(P["box"] == b) & (P["grp"] == g)]
            for i in range(len(E12) - 1):
                m = (S["t_sq"] >= E12[i]) & (S["t_sq"] < E12[i + 1] if i < len(E12) - 2 else S["t_sq"] <= E12[i + 1])
                B = S[m]
                if len(B) < P12_NMIN:
                    continue
                r = dict(box=b, grp=g, bin_lo=E12[i], bin_hi=E12[i + 1], n=len(B), t_sq=float(np.median(B["t_sq"])))
                for q, _, _ in P12_PROPS:
                    v = B[q].to_numpy(float); v = v[np.isfinite(v)]
                    if not len(v):
                        r[q], r[q + "_elo"], r[q + "_ehi"], r[q + "_p16"], r[q + "_p84"], r[q + "_n"] = (np.nan,) * 6
                        continue
                    med = float(np.median(v))
                    if len(v) >= 2:
                        bs = np.median(v[rng12.integers(0, len(v), (P12_NBOOT, len(v)))], axis=1)
                        elo, ehi = med - float(np.percentile(bs, 16)), float(np.percentile(bs, 84)) - med
                    else:
                        elo = ehi = np.nan
                    r[q], r[q + "_elo"], r[q + "_ehi"] = med, elo, ehi
                    r[q + "_p16"], r[q + "_p84"] = float(np.percentile(v, 16)), float(np.percentile(v, 84))
                    r[q + "_n"] = len(v)
                rows.append(r)
    P12 = pd.DataFrame(rows)

    fig, axs = plt.subplots(len(P12_PROPS), len(BOXES_OK), figsize=(4.3 * len(BOXES_OK), 2.15 * len(P12_PROPS)),
                            squeeze=False, constrained_layout=True, sharex=True)
    for ri, (q, ylab, href) in enumerate(P12_PROPS):
        for ci, b in enumerate(BOXES_OK):
            ax = axs[ri, ci]
            if href is not None:
                ax.axhline(href, color="k", lw=1.2, ls="--", alpha=0.75, zorder=1)
                if ci == len(BOXES_OK) - 1:
                    ax.text(0.995, href, " solar", transform=ax.get_yaxis_transform(), ha="right", va="bottom",
                            fontsize=7.5, color="0.3")
            for gi, g in enumerate(G12):
                S = P12[(P12["box"] == b) & (P12["grp"] == g)].sort_values("t_sq")
                if not len(S):
                    continue
                x = S["t_sq"].to_numpy(float) * 10 ** (P12_DX * (gi - 1))       # the small offset, applied in log x
                y = S[q].to_numpy(float)
                solo = S["n"].to_numpy(int) < 2
                ax.errorbar(x[~solo], y[~solo], yerr=[S[q + "_elo"].to_numpy(float)[~solo], S[q + "_ehi"].to_numpy(float)[~solo]],
                            fmt="o", color=G12C[g], ms=6, lw=1.8, capsize=3, zorder=4,
                            label=(g if (ri == 0 and ci == 0) else None))
                ax.scatter(x[solo], y[solo], s=44, facecolors="none", edgecolors=G12C[g], lw=1.6, zorder=4)
            if P12_LOG:
                ax.set_xscale("log")
                ax.set_xticks([0.3, 0.5, 1, 2, 3, 5, 7]); ax.set_xticklabels(["0.3", "0.5", "1", "2", "3", "5", "7"])
            ax.grid(alpha=0.22, lw=0.5)
            ax.set_ylabel(ylab if ci == 0 else "", fontsize=9)
            if ri == 0:
                ax.set_title(BOX_LABEL[b], fontsize=9.5)
            if ri == len(P12_PROPS) - 1:
                ax.set_xlabel(r"$t_{\rm sq} = t_{\rm anchor} - t_{\rm SFT}$   [Gyr]", fontsize=10)
            if ri == 0 and ci == 0:
                ax.legend(frameon=False, fontsize=7.2, loc="best")
    fig.suptitle(r"the three routes in fixed $t_{\rm sq}$ bins: the median of each property and the error on that median"
                 + "\n" + r"open symbols are bins holding a single galaxy (kept, but they are a galaxy and not a median)", fontsize=11.5)
    paper_save(fig, "agn_states_tsq_binned_properties")
    plt.show()

    P12.to_csv(os.path.join(OUT, "agn_states_tsq_binned_properties.csv"), index=False)
    print(f"\nthe table -> {os.path.join(OUT, 'agn_states_tsq_binned_properties.csv')} "
          f"(per box, route and bin: n, median t_sq, and for each property the median, its bootstrap error and the 16-84 spread)")
    for b in BOXES_OK:
        T = P12[P12["box"] == b]
        if not len(T):
            continue
        print(f"\n  {BOX_LABEL[b]}")
        for g in G12:
            S = T[T["grp"] == g].sort_values("t_sq")
            if not len(S):
                continue
            print(f"    {g}")
            print(f"      {'t_sq':>6s} {'n':>4s} " + " ".join(f"{q:>12s}" for q, _, _ in P12_PROPS))
            for r_ in S.itertuples():
                print(f"      {r_.t_sq:6.2f} {r_.n:4d} " + " ".join(f"{getattr(r_, q):12.2f}" for q, _, _ in P12_PROPS))

## Part 13 — the m25 close-up: the histories, the radial anatomy and the K-S plane of the three routes

Everything from Part 9 on is decided at the *catalogue* level: one number per galaxy per anchor. This part takes the three routes into the **particle-level** products of the m25 box — the only one where the ISM and $\kappa_{\rm rot}$ are resolved — and asks whether the split that Part 9 makes on the AGN history survives when the galaxies are opened up. The three routes are Part 11's, verbatim and re-derived from Part 9's frame `F` so the cell runs on its own:

| route | definition | what it means |
|---|---|---|
| **dusty, dosed & firing** | $\log M_{\rm dust}/M_\star \geq$ `DUSTY_CUT`, $E_x \geq$ `PF_EX`, $t_{\rm quiet} <$ `PF_TQ` | the unexplained residual: the AGN timing cannot account for it |
| **dusty, timing-explained** | dusty, in any other category of Part 9 (never armed / under-dosed / dosed-then-quiet) | dusty, and the timing says why |
| **dust-poor** | everything below `DUSTY_CUT` | the pure quenched galaxies |

**(A) The histories** — $\log M_{\rm dust}/M_\star$, $\log 1.36\,M_{\rm H_2}/M_\star$ and $\log$ sSFR on the $t - t_{\rm SFT}$ clock, one median track with a bootstrap 16–84 band per route and the individual chains behind them. This is Part 11 restricted to m25, plus the molecular leg Part 11 does not carry: it is here so the three blocks are read on the same galaxies, and so the H$_2$ track can be laid against the resolved $f_{\rm H_2}$ profile of (B).

**(B) The radial anatomy** — $A_V$, $M_{\rm dust}/M_\star$, $M_{\rm gas}/M_\star$, $M_{\rm H_2}/M_\star$, stellar age and $\kappa_{\rm rot}$ (gas / H$_2$ / stars) per annulus (0–1, 1–3.2, 3.2–10, 10–32 kpc), with the m25 star-forming controls as the grey reference. Each galaxy contributes the median over its sightlines, so a galaxy counts once per annulus; the band is a bootstrap of the group median over galaxies. If the *dusty, dosed & firing* route is dusty because its grains sit in a compact, shielded core, that is where it shows: the same integrated $M_{\rm dust}/M_\star$ can be a bright inner kpc or a smooth disc, and the annuli tell them apart.

**(C) The K-S plane** at $r < R_{50}({\rm H_2})$ — the sft $\to$ qt $\to$ end track of each route and where its end points sit with respect to B08. The routes have the same gas (Part 9's $f_{\rm gas}$ / $f_{\rm H_2}$ panels are coin flips), so if they also sit on the same $\Sigma_{\rm H_2}$–$\Sigma_{\rm SFR}$ locus then nothing about the star formation distinguishes them either, and the separation is in the grains alone.

**This is the one part that is not a pure read of the `paper_ism_prediction_boxes` products.** (A) reads the same history cache as everything else, but (B) needs the m25 notebook's `output/cis25/tables/{annulus_ism_truth,aperture_truth}.fits` (plus, optionally, `annulus_av_allincl.fits`, `annulus_kinematics.fits` and `powderday_quenched_selection.fits`) and (C) needs `output/cis25/ks_tracks_pt/ks_tracks.fits` and the `ks_tracks_lib` module. **Every block gates on its own file and prints what is missing instead of failing**, so the cell runs unchanged off the cluster and each panel appears as its product lands. `P13_NMIN` is lower than the notebook's `NMIN` on purpose: in m25 the unexplained residual is a handful of objects, and the count is printed on every point rather than hidden. The routes carry the names Parts 11–12 give them, so the three parts can be read side by side.

Outputs: `agn_states_routes_m25_{histories,profiles,ks}.{png,pdf}` and `agn_states_routes_m25.csv` (every block, quantity, route and point, with its count).

In [ ]:
# ── Part 13 — the m25 close-up of the three routes: (A) the histories, (B) the radial anatomy, (C) the K-S plane ──
# The only part that reaches outside the ism_prediction products: (B) and (C) read the m25 notebook's own particle-level
# tables. Each block gates on its file and prints what is missing, so the cell runs unchanged off the cluster.
P13_BOX  = "cis25"
P13_NMIN = 3            # galaxies a drawn epoch / point needs — deliberately below the notebook's NMIN: in m25 the
                        # unexplained residual is a handful of chains, and the count is printed rather than hidden
P13_GRPS = globals().get("P11_GRPS", [("dusty, dosed & firing",   "#d7191c", "the unexplained residual"),
                                      ("dusty, timing-explained", "#f0a860", "dusty, but the timing explains it"),
                                      ("dust-poor",               "#2c7bb6", "everything below DUSTY_CUT")])
G13   = [g for g, _, _ in P13_GRPS]
C13   = {g: c for g, c, _ in P13_GRPS}
M25   = os.path.join(os.getcwd(), "output", P13_BOX)
ANN13 = os.path.join(M25, "tables", "annulus_ism_truth.fits")                 # m25 Part 8a: M_dust / M_gas / M_H2 per (annulus, sightline)
APT13 = os.path.join(M25, "tables", "aperture_truth.fits")                    # m25 Part 7e: M*, age per (annulus, sightline)
AV13  = os.path.join(M25, "tables", "annulus_av_allincl.fits")                # m25 Part 8b: A_V per (annulus, sightline) — optional panel
KIN13 = os.path.join(M25, "tables", "annulus_kinematics.fits")                # m25 Part 8j0: kappa_rot per spherical zone — optional panels
SEL13 = os.path.join(M25, "tables", "powderday_quenched_selection.fits")      # m25 Part 3: pop -> the SF-controls line — optional
KS13  = os.path.join(M25, "ks_tracks_pt", "ks_tracks.fits")                   # ks_tracks_quenched_m25 Part 4a (run tag 'pt')
ANNULI13, NSTAR_AP_MIN13 = [("ap1kpc", "0-1"), ("ann3kpc", "1-3.2"), ("ann10kpc", "3.2-10"), ("ann32kpc", "10-32")], 20
P13_ZMAX = None         # THE ANCHORS THAT ENTER. None = every anchor Part 2a could route (DOSE_ALL). Part 9's frame F is only
                        # the z <= P7_ZMAX (0.5) subset, and inheriting it here threw away three quarters of the m25 sample
                        # for no reason: this close-up is about the particle-level anatomy, not about comparable dose clocks.
                        # Set a float to restrict it again (P7_ZMAX reproduces Part 9's sample exactly).


def _band13(A, nmin=None, seed=13):
    """A (n_gal, n_t) -> the median track, the bootstrap 16-84 band of that median and the per-epoch count; under nmin blanked.
    Part 11's med_band with a lower floor — an m25 route can be four chains and med_band's NMIN would blank all of it."""
    A = np.asarray(A, float)
    nmin = P13_NMIN if nmin is None else nmin
    if A.ndim != 2 or not len(A):
        return (np.full(len(TG), np.nan),) * 3 + (np.zeros(len(TG), int),)
    n_t = np.isfinite(A).sum(axis=0)
    rng = np.random.default_rng(seed)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        cent = np.nanmedian(A, axis=0)
        bs = np.array([np.nanmedian(A[rng.integers(0, len(A), len(A))], axis=0) for _ in range(NBOOT)])
        lo, hi = np.nanpercentile(bs, 16, axis=0), np.nanpercentile(bs, 84, axis=0)
    ok = n_t >= nmin
    return np.where(ok, cent, np.nan), np.where(ok, lo, np.nan), np.where(ok, hi, np.nan), n_t


def _frame13(path, cols):
    """A FITS table -> a DataFrame of the named columns, masked values as NaN and byte strings decoded."""
    t = Table.read(path); d = {}
    for c in cols:
        v = t[c]; v = v.filled(np.nan) if hasattr(v, "filled") else v
        v = np.asarray(v)
        d[c] = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in v]) if v.dtype.kind in "SU" else v.astype(float)
    return pd.DataFrame(d)


if "DOSE_ALL" not in globals() or not len(DOSE_ALL):
    print("Part 13: Part 2a built no routes — run it first.")
elif P13_BOX not in BOXES_OK:
    print(f"Part 13: {P13_BOX} has no histories cache — nothing to draw.")
else:
    _src13 = DOSE_ALL if P13_ZMAX is None else DOSE_ALL[DOSE_ALL["z"] <= float(P13_ZMAX) + 1e-9]
    M13 = _src13[_src13["box"] == P13_BOX].copy()
    M13["grp"] = route_grp(M13)
    KEY13 = {(int(s_), int(i_)): g_ for s_, i_, g_ in zip(M13["snap"], M13["gal_id"], M13["grp"])}
    _q13  = QM[QM["box"] == P13_BOX]
    SFT13 = {(int(s_), int(i_)): float(t_) for s_, i_, t_ in zip(_q13["snap"], _q13["gal_id"], _q13["t_sft"])}
    rows13 = []
    print(f"Part 13 — {BOX_LABEL[P13_BOX]}, the three routes: "
          + ", ".join(f"{g} {int((M13['grp'] == g).sum())}" for g in G13)
          + f"  (of {len(M13)} quenched galaxy-anchors, "
          + ("every anchor" if P13_ZMAX is None else f"z <= {float(P13_ZMAX):g}")
          + f", z = {M13['z'].min():.2f}-{M13['z'].max():.2f}; each block below skips cleanly where its product file is absent)")

    # ── (A) the histories: log M_dust/M*, log 1.36 M_H2/M*, log sSFR on the t - t_SFT clock ──
    TRK13 = {q: {g: [] for g in G13} for q in ("lfd", "lfh2", "lssfr")}
    has_md13, has_mh13 = False, False
    with h5py.File(HIST_FILE.format(box=P13_BOX), "r") as f:
        for grpname in f:
            gh = f[grpname]; a = int(grpname.replace("snap", ""))
            if not any(k[0] == a for k in KEY13):
                continue
            t = gh["t_yr"][:] / 1e9; order = np.argsort(t); t = t[order]
            ms13, sf13 = (gh[c][:][order] for c in ("mstar", "sfr"))
            md13 = gh["mdust"][:][order] if "mdust" in gh else None
            mh13 = gh["mh2"][:][order] if "mh2" in gh else None
            has_md13 |= md13 is not None; has_mh13 |= mh13 is not None
            for j, gid in enumerate(gh["gal_ids"][:]):
                key = (a, int(gid)); g = KEY13.get(key); ts = SFT13.get(key, np.nan)
                if g is None or not np.isfinite(ts):
                    continue
                with np.errstate(all="ignore"):
                    ex13 = np.isfinite(ms13[:, j]) & (ms13[:, j] > 0)
                    den = np.where(ex13, ms13[:, j], np.nan)
                    trel = t - ts
                    TRK13["lssfr"][g].append(to_grid(trel, np.log10(np.clip(np.clip(sf13[:, j], 0.0, None) / den, SSFR_FLOOR, None))))
                    if mh13 is not None:
                        TRK13["lfh2"][g].append(to_grid(trel, np.log10(np.where(mh13[:, j] > 0, HE * mh13[:, j] / den, np.nan))))
                    if md13 is not None:                                        # zero dust at the floor, never dropped (Part 0's convention)
                        TRK13["lfd"][g].append(to_grid(trel, np.log10(np.clip(md13[:, j] / den, 10 ** P9_FLOOR, None))))
    QA13 = (["lfd"] if has_md13 else []) + (["lfh2"] if has_mh13 else []) + ["lssfr"]
    LAB13 = {"lfd": r"$\log M_{\rm dust}/M_\star$", "lfh2": r"$\log 1.36\,M_{\rm H_2}/M_\star$",
             "lssfr": r"$\log$ sSFR [yr$^{-1}$]"}
    fig, axs = plt.subplots(1, len(QA13), figsize=(4.9 * len(QA13), 4.0), squeeze=False)
    for ax, q in zip(axs[0], QA13):
        axb = ax.twinx(); axb.set_zorder(ax.get_zorder() - 1); ax.patch.set_visible(False); axb.set_yticks([])
        top13 = 1.0
        for g in G13:
            A = np.array(TRK13[q][g])
            if not len(A):
                continue
            for row in A:
                ax.plot(TG, row, color=C13[g], lw=0.5, alpha=0.16, zorder=2)
            cent, lo, hi, n_t = _band13(A)
            top13 = max(top13, float(np.nanmax(n_t)) if len(n_t) else 1.0)
            axb.bar(TG, n_t, width=0.9 * (TG[1] - TG[0]), color=C13[g], alpha=0.10, lw=0, zorder=1)
            ax.fill_between(TG, lo, hi, color=C13[g], alpha=0.18, lw=0, zorder=3)
            ax.plot(TG, cent, "-", color=C13[g], lw=2.4, zorder=5, label=f"{g} ({len(A)})")
            for x_, m_, l_, h_, nn in zip(TG, cent, lo, hi, n_t):
                rows13.append(dict(block="history", quantity=q, group=g, x=float(x_), n_gal=int(nn),
                                   med=float(m_), lo=float(l_), hi=float(h_)))
        axb.set_ylim(0, top13 * 2.9)
        ax.axvline(0, color="k", lw=1.1, ls="--", alpha=0.85, zorder=1)
        if q == "lfd":
            ax.axhline(DUSTY_CUT, color="0.4", lw=1.0, ls=":", zorder=1)
        ax.set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]"); ax.set_ylabel(LAB13[q]); ax.grid(alpha=0.25, lw=0.5)
    axs[0, 0].legend(frameon=False, fontsize=8.5, loc="lower left")
    fig.suptitle(f"{BOX_LABEL[P13_BOX]}: the three routes on the t - t_SFT clock — median with a bootstrap 16-84 band, "
                 f"the chains behind them, the count per epoch shaded", fontsize=10.5)
    fig.tight_layout()
    paper_save(fig, "agn_states_routes_m25_histories")
    plt.show()
    if not has_md13:
        print("  (A) no 'mdust' in the histories cache — rm ism_prediction_histories_*.hdf5 and re-run "
              "paper_ism_prediction_boxes Part 6a for the dust leg")
    if not has_mh13:
        print("  (A) no 'mh2' in the histories cache — the molecular panel is dropped; the same rm + re-run adds it")

    # ── (B) the radial anatomy: A_V, the ISM fractions, the age and the rotation support per annulus, the SF controls as the reference ──
    if not (os.path.exists(ANN13) and os.path.exists(APT13)):
        print(f"\n(B) radial profiles: {ANN13} or {APT13} missing — the m25 notebook's Parts 8a / 7e (cluster)")
    else:
        K4 = ["snap", "gal_id", "incl", "aperture"]
        RF13 = _frame13(ANN13, K4 + ["M_dust", "M_H2"]).merge(_frame13(APT13, K4 + ["nstar_ap", "mstar", "age_m_star_myr"]),
                                                                      on=K4, how="inner")
        if os.path.exists(AV13):
            RF13 = RF13.merge(_frame13(AV13, K4 + ["A_V"]), on=K4, how="left")
        RF13 = RF13[RF13["aperture"].isin([a for a, _ in ANNULI13])].reset_index(drop=True)
        SF13 = set()
        if os.path.exists(SEL13):
            _s13 = _frame13(SEL13, ["snap", "gal_id", "pop"])
            SF13 = {(int(s_), int(i_)) for s_, i_, p_ in zip(_s13["snap"], _s13["gal_id"], _s13["pop"]) if p_ == "SF"}
        RF13["grp"] = [KEY13.get((int(s_), int(i_)), "star_forming" if (int(s_), int(i_)) in SF13 else None)
                       for s_, i_ in zip(RF13["snap"], RF13["gal_id"])]
        RF13 = RF13[RF13["grp"].notna()].reset_index(drop=True)
        if os.path.exists(KIN13):
            RF13 = RF13.merge(_frame13(KIN13, ["snap", "gal_id", "aperture", "kappa_H2", "kappa_star"]),
                              on=["snap", "gal_id", "aperture"], how="left")
        for _f, _n in ((AV13, "A_V (m25 8b)"), (KIN13, "kappa (m25 8j0)"), (SEL13, "the SF-controls line (m25 Part 3)")):
            if not os.path.exists(_f):
                print(f"  (B) {_n}: {_f} missing — panel / line skipped")
        _ms13 = np.where(RF13["nstar_ap"].to_numpy(float) >= NSTAR_AP_MIN13, RF13["mstar"].to_numpy(float), np.nan)   # the m25 7e sampling floor
        with np.errstate(divide="ignore", invalid="ignore"):
            for c, num in (("fdust_ap", "M_dust"), ("fH2_ap", "M_H2")):
                v = RF13[num].to_numpy(float)
                RF13[c] = np.where(np.isfinite(v) & (v > 0) & (_ms13 > 0), v / _ms13, np.nan)      # a zero mass is NaN here, never a floor
            RF13["age_ap"] = np.where(np.isfinite(_ms13), RF13["age_m_star_myr"].to_numpy(float) / 1e3, np.nan)
            if "A_V" in RF13.columns:
                _av13 = RF13["A_V"].to_numpy(float)
                RF13["A_V"] = np.where(np.isfinite(_av13) & (_av13 > 0), _av13, np.nan)
        QUANT13 = ([("A_V", r"$A_V$ [mag]", "log")] if "A_V" in RF13.columns else []) \
                + [("fdust_ap", r"$M_{\rm dust}/M_\star$", "log"),
                   ("fH2_ap", r"$M_{\rm H_2}/M_\star$", "log"), ("age_ap", "stellar age [Gyr]", "lin")] \
                + ([(k, lab, "kappa") for k, lab in (("kappa_H2", r"$\kappa_{\rm rot}^{\rm H_2}$"),
                                                     ("kappa_star", r"$\kappa_{\rm rot}^{\star}$"))] if "kappa_H2" in RF13.columns else [])
        GRP13B = (["star_forming"] if len(SF13) else []) + G13
        C13B = dict(C13, star_forming="0.6")
        _x13 = np.arange(len(ANNULI13))
        _nc13 = (len(QUANT13) + 1) // 2
        rng13 = np.random.default_rng(13)
        NMAX13 = {g: 0 for g in GRP13B}
        fig, axs = plt.subplots(2, _nc13, figsize=(3.5 * _nc13, 7.1), squeeze=False)
        for qi, (col, lab, kind) in enumerate(QUANT13):
            ax = axs[qi // _nc13, qi % _nc13]
            for gi, g in enumerate(GRP13B):
                m_, l_, h_, n_ = [], [], [], []
                for a_, _ in ANNULI13:
                    sub = RF13[(RF13["grp"] == g) & (RF13["aperture"] == a_)]
                    v = sub[col].to_numpy(float); ok = np.isfinite(v)
                    per = {}
                    for s_, i_, vv in zip(sub["snap"].to_numpy(float)[ok], sub["gal_id"].to_numpy(float)[ok], v[ok]):
                        per.setdefault((int(s_), int(i_)), []).append(vv)                     # the sightline rows -> one value per galaxy
                    pg = np.array([np.median(vv) for vv in per.values()])
                    if len(pg) >= P13_NMIN:
                        bs = np.median(pg[rng13.integers(0, len(pg), (NBOOT, len(pg)))], axis=1)
                        m_.append(float(np.median(pg))); l_.append(float(np.percentile(bs, 16))); h_.append(float(np.percentile(bs, 84)))
                    else:
                        m_.append(np.nan); l_.append(np.nan); h_.append(np.nan)
                    n_.append(len(pg))
                    rows13.append(dict(block="profile", quantity=col, group=g, x=a_, n_gal=len(pg), med=m_[-1], lo=l_[-1], hi=h_[-1]))
                NMAX13[g] = max(NMAX13[g], max(n_))
                off = (gi - (len(GRP13B) - 1) / 2) * 0.08
                m_, l_, h_ = np.array(m_), np.array(l_), np.array(h_)
                ok_ = np.isfinite(m_)
                if ok_.any():
                    ax.fill_between(_x13[ok_] + off, l_[ok_], h_[ok_], color=C13B[g], alpha=0.15, lw=0)
                    ax.plot(_x13[ok_] + off, m_[ok_], "o-", color=C13B[g], lw=1.3 if g == "star_forming" else 1.9, ms=4.5,
                            mec="k", mew=0.4)
            if kind == "log":
                ax.set_yscale("log")
            if kind == "kappa":
                ax.axhline(0.3, color="0.3", lw=1.0, ls="--"); ax.set_ylim(0, 1)
            if col == "fdust_ap":
                ax.axhline(10 ** DUSTY_CUT, color="0.4", lw=0.9, ls=":")
            ax.set_xticks(_x13); ax.set_xticklabels([r_ for _, r_ in ANNULI13], fontsize=8)
            if qi // _nc13 == 1:
                ax.set_xlabel("annulus [kpc]")
            ax.set_ylabel(lab); ax.grid(alpha=0.25, lw=0.5)
        for qi in range(len(QUANT13), 2 * _nc13):
            axs[qi // _nc13, qi % _nc13].axis("off")
        axs[0, 0].legend(handles=[Line2D([], [], color=C13B[g], marker="o", ms=4.5, mec="k", mew=0.4,          # every route, including
                                         lw=1.3 if g == "star_forming" else 1.9,                                # one that never clears
                                         label=f"{g} (N <= {NMAX13[g]})") for g in GRP13B],                # P13_NMIN and draws no line
                         frameon=False, fontsize=8)
        fig.suptitle(f"{BOX_LABEL[P13_BOX]}: the radial anatomy of the three routes (+ the SF controls) — per-galaxy sightline medians, "
                     f"bootstrap 16-84 of the group median"
                     + ("" if "kappa_H2" not in RF13.columns else "; kappa in the spherical zones (sightline-independent)"), fontsize=10.5)
        fig.tight_layout()
        paper_save(fig, "agn_states_routes_m25_profiles")
        plt.show()

    # ── (C) the K-S plane: the R50_H2 tracks (sft -> qt -> end) of the three routes, the end points against B08 ──
    kl = None
    if not os.path.exists(KS13):
        print(f"\n(C) K-S plane: {KS13} missing — ks_tracks_quenched_m25 Part 4 (run tag 'pt', cluster)")
    else:
        try:
            import ks_tracks_lib as kl                                   # the B08 / K98 relations and the He factor, verbatim
        except ImportError as e:
            print(f"\n(C) K-S plane: {KS13} is there but ks_tracks_lib is not importable ({e}) — run the notebook from the repo root")
    if kl is not None:
        TR13 = Table.read(KS13)
        _s = lambda a: np.array([x.decode() if isinstance(x, bytes) else str(x) for x in np.asarray(a)])
        ap13, pop13, stg13, gk13 = _s(TR13["ap_label"]), _s(TR13["pop"]), _s(TR13["stage"]), _s(TR13["gkey"])
        x13, y13, ul13 = np.asarray(TR13["logSigmaH2"], float), np.asarray(TR13["logSigmaSFR"], float), np.asarray(TR13["is_ul"], bool)
        grp13 = np.array([KEY13.get(tuple(int(v) for v in g_.split("_"))) for g_ in gk13], dtype=object)
        STAGES13, MRK13 = ["sft", "qt", "end"], {"sft": "^", "qt": "s", "end": "o"}
        base13 = ap13 == "R50_H2"
        band13 = float(kl.RELATIONS["B08"]["sig"])

        fig, ax = plt.subplots(figsize=(6.8, 6.3))
        sf_ = base13 & (pop13 == "SF")
        ax.scatter(x13[sf_], y13[sf_], s=7, c="0.65", alpha=0.3, lw=0, zorder=1, rasterized=True)
        xs13 = np.linspace(-1.0, 4.1, 60)
        for rk, lw_, alpha_ in (("B08", 1.4, 1.0), ("K98", 0.9, 0.55)):
            rel = kl.RELATIONS[rk]
            if rk == "B08":
                ax.fill_between(xs13, rel["A"] + rel["N"] * xs13 - rel["sig"], rel["A"] + rel["N"] * xs13 + rel["sig"],
                                color=rel["color"], alpha=0.10, lw=0, zorder=0)
            ax.plot(xs13, rel["A"] + rel["N"] * xs13, color=rel["color"], lw=lw_, ls=rel["ls"], alpha=alpha_, zorder=2)
        print(f"\n(C) the track-end regions w.r.t. B08 +- {band13:g} dex (the R50_H2 aperture):")
        for g in G13:
            mk = base13 & (pop13 == "Q") & (grp13 == g) & np.isin(stg13, STAGES13)
            for g_ in np.unique(gk13[mk]):
                r_ = np.flatnonzero(mk & (gk13 == g_))
                r_ = r_[np.argsort([STAGES13.index(s_) for s_ in stg13[r_]])]
                ok = np.isfinite(x13[r_]) & np.isfinite(y13[r_])
                if ok.sum() >= 2:
                    ax.plot(x13[r_][ok], y13[r_][ok], "-", color=C13[g], lw=0.8, alpha=0.4, zorder=3)
            medx, medy = [], []
            for st_ in STAGES13:
                ms_ = mk & (stg13 == st_) & ~ul13 & np.isfinite(x13) & np.isfinite(y13)
                if ms_.sum() >= P13_NMIN:
                    medx.append(float(np.median(x13[ms_]))); medy.append(float(np.median(y13[ms_])))
                    ax.scatter(medx[-1], medy[-1], s=95, marker=MRK13[st_], c=C13[g], edgecolors="k", linewidths=0.8, zorder=6)
                    rows13.append(dict(block="ks_track", quantity="stage_median", group=g, x=st_, n_gal=int(ms_.sum()),
                                       med=medy[-1], lo=np.nan, hi=np.nan, logSigmaH2=medx[-1]))
            if len(medx) >= 2:
                ax.plot(medx, medy, "-", color=C13[g], lw=2.4, zorder=5)
            me_ = mk & (stg13 == "end")
            with np.errstate(invalid="ignore"):
                offr = y13[me_] - kl.relation_y("B08", x13[me_])
            fin_ = np.isfinite(offr) & ~ul13[me_]
            cnt = dict(n=int(len(np.unique(gk13[mk]))), below=int((fin_ & (offr < -band13)).sum()),
                       within=int((fin_ & (np.abs(offr) <= band13)).sum()), above=int((fin_ & (offr > band13)).sum()),
                       censored=int((~fin_).sum()))
            rows13.append(dict(block="ks_region", quantity="end_vs_B08", group=g, x="", n_gal=cnt["n"], med=np.nan, lo=np.nan,
                               hi=np.nan, **{c: cnt[c] for c in ("below", "within", "above", "censored")}))
            print(f"  {g:24s} N={cnt['n']:3d}: below {cnt['below']}, on {cnt['within']}, above {cnt['above']}, "
                  f"censored/undefined {cnt['censored']}")
        _h13  = [Line2D([], [], color=C13[g], lw=2.2, label=f"{g} median track") for g in G13]
        _h13 += [Line2D([], [], ls="", marker=MRK13[st_], ms=8, mfc="0.5", mec="k",
                        label={"sft": "SFT (quench start)", "qt": "QT (quench end)", "end": "track end"}[st_]) for st_ in STAGES13]
        _h13 += [Line2D([], [], ls="", marker="o", ms=5, mfc="0.65", mec="none", label="m25 SF controls"),
                 Line2D([], [], color=kl.RELATIONS["B08"]["color"], lw=1.4, ls=kl.RELATIONS["B08"]["ls"],
                        label=kl.RELATIONS["B08"]["label"] + r" $\pm$ scatter"),
                 Line2D([], [], color=kl.RELATIONS["K98"]["color"], lw=0.9, ls=kl.RELATIONS["K98"]["ls"], alpha=0.55,
                        label=kl.RELATIONS["K98"]["label"] + " (reference)")]
        ax.legend(handles=_h13, loc="lower right", frameon=False, fontsize=8)
        ax.set_xlabel(r"$\log(\Sigma_{\rm H_2}/M_\odot\,{\rm pc}^{-2})$  [$r<R_{50}({\rm H_2})$, $\times$" + f"{kl.HE_FACTOR:g} He]")
        ax.set_ylabel(r"$\log(\Sigma_{\rm SFR}/M_\odot\,{\rm yr}^{-1}\,{\rm kpc}^{-2})$")
        ax.set_xlim(-0.5, 3.6); ax.set_ylim(-4.6, 1.1)
        ax.tick_params(direction="in", top=True, right=True)
        ax.set_title(f"{BOX_LABEL[P13_BOX]}: the three routes on the K-S plane", fontsize=10.5)
        fig.tight_layout()
        paper_save(fig, "agn_states_routes_m25_ks")
        plt.show()

    if rows13:
        R13_CSV = os.path.join(OUT, "agn_states_routes_m25.csv")
        pd.DataFrame(rows13).to_csv(R13_CSV, index=False)
        print(f"\nthe m25 close-up tables -> {R13_CSV}")

## Part 14 — the critical-point sequences of the three routes (noxray_analogues Part 8, re-cut)

The same m25 galaxies followed through their **critical points** — AGN ignition $\to$ jet-mode onset $\to$ SFT $\to$ QT $\to$ track end — in the two spherical zones of the K-S notebook's Part 3b (core $r<3.2$ kpc, outskirt $3.2$–$10$ kpc): $\kappa_{\rm rot}^{\rm H_2}$, $\kappa_{\rm rot}^\star$, sSFR, stellar age, $M_{\rm dust}/M_\star$ and $M_{\rm H_2}/M_\star$ per stage, **one curve per route**. The ISM in each zone comes from the tracks' projected apertures (core $=$ `ap3kpc`, outskirt $=$ `ap10kpc` $-$ `ap3kpc`); the kinematics and the age come from the rule-free `ks_stage_kinematics` cache. Zero dust or H$_2$ is NaN here, never a floor; zero SFR sits at `SSFR_FLOOR`, because the quenching is the point.

The pink bands are the min–max range of the ALMA-C11 detections (dust detections for age / $f_{\rm dust}$ / sSFR, CO detections for $f_{\rm H_2}$, divided by 1.36 so bare molecular masses are compared).

**The question.** If the firing-dusty route is a distinct *route* — dust that came through the X-ray phase — the curves should already separate at **SFT/QT**. If they only separate at `end`, it is a late *state*, not a route.

Like Part 13 this is not a pure read of the `ism_prediction` products: it gates on the m25 notebook's own `ks_tracks_pt/ks_tracks.fits` and `ks_tracks/ks_stage_kinematics.fits` and prints what is missing.


In [ ]:
# ── Part 14 — the critical-point sequences of the three routes: core / outskirt at ignition -> jet -> SFT -> QT -> end ──
# noxray_analogues Part 8 (itself the X-ray notebook's Section 4, lean), re-cut on Part 2a's routes. Like Part 13 this is
# NOT a pure read of the ism_prediction products: it reads the m25 notebook's own K-S tables and gates on each of them.
P14_BOX    = "cis25"
P14_STAGES_ALL = ["agn_ign", "jet_on", "sft", "qt", "end"]
P14_ZONES  = [("ap3kpc", "core, $r < 3.2$ kpc"), ("ann10kpc", "outskirt, 3.2-10 kpc")]
P14_NMIN   = int(globals().get("P13_NMIN", 3))      # galaxies a drawn stage point needs — the m25 residual is a handful of objects
P14_ZMAX   = globals().get("P13_ZMAX", None)       # the anchors that enter — Part 13's knob, for the same reason: None = every anchor
                                                   # Part 2a could route, NOT Part 9's z <= P7_ZMAX subset
P14_NSTAR_MIN, P14_NBOOT = 20, 400
M25_14   = os.path.join(os.getcwd(), "output", P14_BOX)
KS14     = globals().get("KS13", os.path.join(M25_14, "ks_tracks_pt", "ks_tracks.fits"))   # ks_tracks_quenched_m25 Part 4a (run tag 'pt')
KINST14  = os.path.join(M25_14, "ks_tracks", "ks_stage_kinematics.fits")                   # its Part 3b: kappa + stellar age per spherical zone, rule-free
OBS14    = os.path.join(os.getcwd(), "obs_data", "almac11", "almac11_gas_dust.csv")
P14_QUANT = [("kappa_H2", r"$\kappa_{\rm rot}^{\rm H_2}$", "kappa"), ("kappa_star", r"$\kappa_{\rm rot}^{\star}$", "kappa"),
             ("ssfr", r"sSFR [yr$^{-1}$]", "log"), ("age_mw_gyr", "stellar age [Gyr]" + "\n" + r"(sim: MASS-weighted)", "lin"),
             ("fdust", r"$M_{\rm dust}/M_\star$", "log"), ("fH2", r"$M_{\rm H_2}/M_\star$", "log")]
# ── THE AGE AXIS IS NOT LIKE FOR LIKE, and the band on it is drawn hatched to say so ──
# The simulation's age is caesar's `ages.mass_weighted`. The ALMA-C11 `logage` is CIGALE's sfhdelayedbq `age`, i.e. the
# ONSET of star formation — a formation time, weighted by nothing. For a delayed-tau SFH the mass-weighted age sits
# roughly 2*tau BELOW the onset age (tau = 0.3-1 Gyr -> 0.6-1.8 Gyr lower at these ages), and the table carries no tau,
# so the conversion cannot be done here. The observed age band is therefore an UPPER bound on the like-for-like value.
P14_OBS_AGE_LIKE = False        # set True only if OBS14 ever carries a mass-weighted age column
P14_TICK  = {"agn_ign": "ign.", "jet_on": "jet", "sft": "SFT", "qt": "QT", "end": "end"}

if "DOSE_ALL" not in globals() or not len(DOSE_ALL):
    print("Part 14: Part 2a built no routes — run it first.")
elif P14_BOX not in BOXES_OK:
    print(f"Part 14: {P14_BOX} has no histories cache — nothing to draw.")
elif not (os.path.exists(KS14) and os.path.exists(KINST14)):
    print(f"Part 14: {KS14 if not os.path.exists(KS14) else KINST14} missing — "
          "ks_tracks_quenched_m25 Part 4a (run tag 'pt') and Part 3b build them (cluster); skipped")
else:
    _src14 = DOSE_ALL if P14_ZMAX is None else DOSE_ALL[DOSE_ALL["z"] <= float(P14_ZMAX) + 1e-9]
    M14 = _src14[_src14["box"] == P14_BOX].copy()
    M14["grp"] = route_grp(M14)
    KEY14 = {(int(s_), int(i_)): g_ for s_, i_, g_ in zip(M14["snap"], M14["gal_id"], M14["grp"])}

    T14 = Table.read(KS14)
    _s14 = lambda a: np.array([x.decode() if isinstance(x, bytes) else str(x) for x in np.asarray(a)])
    stg, pop, ap, gk = _s14(T14["stage"]), _s14(T14["pop"]), _s14(T14["ap_label"]), _s14(T14["gkey"])
    P14_STAGES = [st for st in P14_STAGES_ALL if st in set(stg[pop == "Q"])]
    _q = (pop == "Q") & np.isin(stg, P14_STAGES)
    EP14 = pd.DataFrame(dict(gkey=gk[_q], stage=stg[_q], snap=np.asarray(T14["snap"], float)[_q].astype(int),
                             gx=np.asarray(T14["gx"], float)[_q].astype(int))).drop_duplicates(["gkey", "stage"])
    _kin = Table.read(KINST14)
    K14 = pd.DataFrame({c: (_s14(_kin[c]) if c == "zone" else np.asarray(_kin[c], float))
                        for c in ("snap", "gx", "zone", "kappa_H2", "kappa_star", "age_mw_gyr")})
    K14["snap"], K14["gx"] = K14["snap"].astype(int), K14["gx"].astype(int)
    K14 = EP14.merge(K14[K14["zone"].isin([z for z, _ in P14_ZONES])], on=["snap", "gx"], how="inner")

    # the zone ISM at every critical point, from the tracks' projected apertures: core = ap3kpc, outskirt = ap10kpc - ap3kpc
    _m = _q & np.isin(ap, ["ap3kpc", "ap10kpc"])
    TD14 = pd.DataFrame(dict(gkey=gk[_m], stage=stg[_m], ap=ap[_m],
                             m_dust=np.asarray(T14["m_dust"], float)[_m], m_H2=np.asarray(T14["m_H2"], float)[_m],
                             sfr=np.asarray(T14["sfr100"], float)[_m], m_star=np.asarray(T14["m_star"], float)[_m],
                             n_star=np.asarray(T14["n_star"], float)[_m])).drop_duplicates(["gkey", "stage", "ap"])
    _pv = TD14.pivot_table(index=["gkey", "stage"], columns="ap", values=["m_dust", "m_H2", "sfr", "m_star", "n_star"], aggfunc="first")
    _ism = []
    for (gk_, st_), r in _pv.iterrows():
        for zone, sel in (("ap3kpc", lambda c: r[(c, "ap3kpc")]), ("ann10kpc", lambda c: r[(c, "ap10kpc")] - r[(c, "ap3kpc")])):
            ms_, ns_ = sel("m_star"), sel("n_star")
            ok_ = np.isfinite(ms_) and ms_ > 0 and ns_ >= P14_NSTAR_MIN
            d = dict(gkey=gk_, stage=st_, zone=zone)
            for c, num in (("fdust", "m_dust"), ("fH2", "m_H2")):
                v = sel(num)
                d[c] = (v / ms_) if (ok_ and np.isfinite(v) and v > 0) else np.nan          # zero dust / H2 -> NaN, never a floor
            v = sel("sfr")
            d["ssfr"] = max(v / ms_, SSFR_FLOOR) if (ok_ and np.isfinite(v) and v >= 0) else np.nan   # zero SFR at the floor: quenching is the point
            _ism.append(d)
    K14 = K14.merge(pd.DataFrame(_ism), on=["gkey", "stage", "zone"], how="left")
    K14["grp"] = [KEY14.get(tuple(int(v) for v in g_.split("_"))) for g_ in K14["gkey"]]
    K14 = K14[K14["grp"].notna()].reset_index(drop=True)
    print(f"Part 14 — {BOX_LABEL[P14_BOX]}, {len(M14)} routed anchors ("
          + ("every anchor" if P14_ZMAX is None else f"z <= {float(P14_ZMAX):g}") + f"), of which "
          f"{K14['gkey'].nunique()} are in the K-S tracks; the critical points " + " -> ".join(P14_STAGES) + ": "
          + ", ".join(f"{GSHORT[g]} {K14.loc[(K14['grp'] == g) & (K14['zone'] == 'ap3kpc'), 'gkey'].nunique()} galaxies" for g in G3)
          + ("" if len(P14_STAGES) == len(P14_STAGES_ALL)
             else f"; stages {[s for s in P14_STAGES_ALL if s not in P14_STAGES]} are not in the K-S tracks"))

    # the observed bands: ALMA-C11 dust detections -> age / f_dust / sSFR, CO detections -> f_H2 (/1.36, to compare bare H2 masses)
    B14 = {}
    if os.path.exists(OBS14):
        _o = pd.read_csv(OBS14)
        _d = _o[_o["det_dust"].astype(bool)]
        if len(_d):
            B14["age_mw_gyr"] = (float((10 ** _d["logage"] / 1e9).min()), float((10 ** _d["logage"] / 1e9).max()))
            B14["fdust"] = (float((10 ** (_d["logMdust"] - _d["logMstar"])).min()), float((10 ** (_d["logMdust"] - _d["logMstar"])).max()))
            B14["ssfr"] = (float((_d["SFR"] / 10 ** _d["logMstar"]).min()), float((_d["SFR"] / 10 ** _d["logMstar"]).max()))
        _c = _o[_o["det_co"].astype(bool)]
        if len(_c):
            B14["fH2"] = (float((_c["MH2_fid"] / 1.36 / 10 ** _c["logMstar"]).min()), float((_c["MH2_fid"] / 1.36 / 10 ** _c["logMstar"]).max()))
    else:
        print(f"  (the ALMA-C11 bands need {OBS14} — drawn without them)")

    def _bmed14(v, n=P14_NBOOT, seed=14):
        v = np.asarray(v, float); v = v[np.isfinite(v)]
        if len(v) < P14_NMIN:
            return np.nan, np.nan, np.nan, len(v)
        bs = np.median(v[np.random.default_rng(seed).integers(0, len(v), (n, len(v)))], axis=1)
        return float(np.median(v)), float(np.percentile(bs, 16)), float(np.percentile(bs, 84)), len(v)

    rows14 = []
    fig, axs = plt.subplots(len(P14_ZONES), len(P14_QUANT), figsize=(2.95 * len(P14_QUANT), 3.4 * len(P14_ZONES)), squeeze=False)
    for zi, (zone, zlab) in enumerate(P14_ZONES):
        for qi, (col, lab, kind) in enumerate(P14_QUANT):
            ax = axs[zi, qi]
            for gi, g in enumerate(G3):
                off = (gi - (len(G3) - 1) / 2) * 0.18                      # a small stage offset per route, so three coincident points stay separable
                xx, yy, lo, hi = [], [], [], []
                for i, st_ in enumerate(P14_STAGES):
                    med, l16, h84, n = _bmed14(K14.loc[(K14["grp"] == g) & (K14["zone"] == zone) & (K14["stage"] == st_), col])
                    rows14.append(dict(zone=zone, quantity=col, route=g, stage=st_, n_gal=n, med=med, lo=l16, hi=h84))
                    if np.isfinite(med):
                        xx.append(i + off); yy.append(med); lo.append(med - l16); hi.append(h84 - med)
                if len(xx):
                    ax.errorbar(xx, yy, yerr=np.array([lo, hi]), color=GRPS[g]["color"], lw=2.0, elinewidth=1.3, capsize=2.5,
                                marker="o", ms=5.5, mec="k", mew=0.5, ls="-", zorder=5,
                                label=GRPS[g]["label"] if (zi == 0 and qi == 0) else None)
            _mismatch = (col == "age_mw_gyr") and not P14_OBS_AGE_LIKE
            if col in B14 and np.all(np.isfinite(B14[col])):
                ax.axhspan(B14[col][0], B14[col][1], color="#e7298a", alpha=0.12, lw=0, zorder=0,
                           hatch="///" if _mismatch else None, edgecolor="#e7298a")
                if _mismatch and zi == 0:
                    ax.text(0.5, 0.985, "obs = CIGALE ONSET age, not mass-weighted", transform=ax.transAxes,
                            ha="center", va="top", fontsize=6.6, color="#8c1b52", fontweight="bold")
            if kind == "log":
                ax.set_yscale("log")
            if kind == "kappa":
                ax.axhline(0.3, color="0.3", lw=0.9, ls="--"); ax.set_ylim(0, 1)
            if col == "ssfr":
                ax.axhline(1e-11, color="0.3", lw=0.9, ls="--")
            if col == "fdust":
                ax.axhline(10 ** DUSTY_CUT, color="0.4", lw=0.9, ls=":")     # the notebook's own dust split, for scale
            ax.set_xlim(-0.6, len(P14_STAGES) - 0.4)
            ax.set_xticks(range(len(P14_STAGES))); ax.set_xticklabels([P14_TICK.get(s_, s_) for s_ in P14_STAGES], fontsize=8.5)
            ax.set_ylabel((zlab + "\n" if qi == 0 else "") + lab, fontsize=9)
            ax.grid(alpha=0.25, lw=0.5)
    axs[0, 0].legend(frameon=False, fontsize=7.5, loc="best")
    if B14:
        axs[0, 2].legend(handles=[Line2D([], [], color="#e7298a", lw=6, alpha=0.25)], labels=["ALMA-C11 detections (min-max)"],
                         frameon=False, fontsize=7.5, loc="best")
        print("  CAVEAT — the age panel is NOT like for like: the simulation's age is caesar's mass-weighted age, the "
              "ALMA-C11 age is CIGALE's sfhdelayedbq ONSET age (the start of star formation, weighted by nothing).")
        print("  For a delayed-tau SFH the mass-weighted age sits ~2 tau below the onset age, so with tau = 0.3-1 Gyr the "
              "observed band should move 0.6-1.8 Gyr LEFT to be comparable; the table carries no tau, so it is drawn")
        print("  hatched as an UPPER bound instead of being corrected. Every other panel (kappa, sSFR, f_dust, f_H2) is "
              "like for like.")
    fig.suptitle(f"{BOX_LABEL[P14_BOX]}: the three routes at the critical points (median + galaxy-bootstrap 16-84; core / outskirt = the "
                 f"K-S Part 3b spherical zones; the ISM per zone from the tracks' projected apertures)", fontsize=10)
    #fig.tight_layout()
    paper_save(fig, "agn_states_routes_m25_sequences")
    plt.show()
    SEQ14 = pd.DataFrame(rows14)
    SEQ14.to_csv(os.path.join(OUT, "agn_states_routes_m25_sequences.csv"), index=False)
    print(f"stage medians -> {os.path.join(OUT, 'agn_states_routes_m25_sequences.csv')}")
    print("READ IT LIKE THIS: if the firing-dusty route is a distinct ROUTE — dust that came through the X-ray phase — the "
          "curves should already separate at SFT/QT. If they only separate at 'end', it is a late STATE, not a route.")


## Part 15 — the ISM grid under the route split (noxray_analogues Part 10, re-cut): where the routes run through the observed planes

The figure-H grid per box: $\log M_{\rm dust}/M_\star$ (zero dust at the `P9_FLOOR`) and $\log 1.36\,M_{\rm H_2}/M_\star$ against **age**, $\Sigma_{\rm e}$ and **sSFR** — running-median tracks of the three routes with bootstrap 16–84 bands, the *all quenched with a route* dotted track, the channel-off `cis50nox` all-quenched dashed track, and the **ALMA-C11 quenched galaxies as points** (dust and CO upper limits arrowed). The H$_2$ row compares the observed $\alpha_{\rm CO}$ H$_2$ (helium included) to the simulated $1.36\,M_{\rm H_2}$ — like for like.

The printed **scorecard** gives the median (measured $-$ track) offset of the *detections* that fall inside each track's own $x$ range: the direct test of **which route carries the observed dusty QGs**. The route whose offset is closest to zero is the one the observations sit on; the dust-poor track should sit far below them.

Two things to keep in mind reading it. The offsets are computed only where a track has $x$ coverage, so the count per entry is printed with it; and the routes are defined at the anchor, so a track is the locus of galaxies *that ended up* in that route, not an evolutionary path.


In [ ]:
# ── Part 15 — the ISM grid under the route split: lfd and lfh2 against age / Sigma_e / sSFR, with the ALMA-C11 points ──
# noxray_analogues Part 10 (the X-ray notebook's Section 5b, lean), re-cut on Part 2a's routes. The direct test of whether
# the OBSERVED dusty QGs sit on the firing-dusty track or on the dust-poor one.
# NB the AGE column is NOT like for like (see G15_LIKE below): the simulation's age is caesar's mass-weighted age, the
# observed one is CIGALE's sfhdelayedbq ONSET age. The other two x axes and both y axes are like for like.
G15_X = [("age", "stellar age [Gyr]" + "\n" + r"sim: mass-weighted;  obs: CIGALE onset (an upper bound)"),
         ("lsig", r"$\log \Sigma_{\rm e}$ [$M_\odot$ kpc$^{-2}$]"), ("lssfr", r"$\log$ sSFR [yr$^{-1}$]")]
G15_LIKE = {"age": False, "lsig": True, "lssfr": True}     # is the observed x the SAME quantity as the simulated one?
G15_Y = [("lfd", r"$\log M_{\rm dust}/M_\star$"), ("lfh2", r"$\log 1.36\,M_{\rm H_2}/M_\star$ (sim; obs: $\alpha_{\rm CO}$)")]
G15_NBIN, G15_NMIN, G15_NBOOT, G15_POINTS_MAX = 7, 3, 400, 150
G15_NOX  = globals().get("NOX_BOX", "cis50nox")     # the channel-off reference track: all its quenched galaxies, no route (there is no X-ray to date)
OBS15_GD, OBS15_ST = (os.path.join(os.getcwd(), "obs_data", "almac11", f) for f in ("almac11_gas_dust.csv", "age_sersic_sigma.csv"))
C_OBS15, C_OBS15_EDGE = "#e7298a", "#3b0f2a"


def run_track15(x, y, nbins=G15_NBIN, nmin=G15_NMIN, nmin_small=5, n=G15_NBOOT, seed=15):
    """Running median of y in equal-count x bins + a bootstrap 16-84 of that median (a sample under 2 x nmin falls back to 4 bins)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
    if len(x) < 2 * nmin:
        nbins, nmin = 4, min(nmin, nmin_small)
    nb_ = min(nbins, len(x) // max(nmin, 1))
    if nb_ < 2:
        return pd.DataFrame(columns=["x", "y", "lo", "hi", "n"])
    e = np.quantile(x, np.linspace(0, 1, nb_ + 1))
    ib = np.clip(np.searchsorted(e, x, side="right") - 1, 0, nb_ - 1)
    rng, out = np.random.default_rng(seed), []
    for k_ in range(nb_):
        yy = y[ib == k_]
        bs = np.median(yy[rng.integers(0, len(yy), (n, len(yy)))], axis=1)
        out.append((float(np.median(x[ib == k_])), float(np.median(yy)), float(np.percentile(bs, 16)), float(np.percentile(bs, 84)), int(len(yy))))
    return pd.DataFrame(out, columns=["x", "y", "lo", "hi", "n"])


if "DOSE_ALL" not in globals() or not len(DOSE_ALL):
    print("Part 15: Part 2a built no routes — nothing to draw.")
else:
    # the observed points, built once: the ALMA-C11 QGs (the ctrl rows dropped)
    OBS15 = pd.DataFrame()
    if os.path.exists(OBS15_GD) and os.path.exists(OBS15_ST):
        _gd = pd.read_csv(OBS15_GD); _st = pd.read_csv(OBS15_ST).drop_duplicates("id").set_index("id")
        _ro = []
        for _, r in _gd.iterrows():
            sid = str(r["id"])
            if sid.startswith("ctrl") or sid not in _st.index:
                continue
            Mo = 10 ** float(r["logMstar"]); re_ = float(_st.loc[sid, "re_kpc"])
            _ro.append(dict(id=sid, age=10 ** float(r["logage"]) / 1e9,
                            lsig=np.log10(0.5 * Mo / (np.pi * re_ ** 2)) if re_ > 0 else np.nan,
                            lssfr=np.log10(max(float(r["SFR"]) / Mo, SSFR_FLOOR)),
                            lfd=float(r["log_fdust"]), lfd_ul=bool(r["dust_ul"]),
                            lfh2=float(r["logMH2"]) - float(r["logMstar"]), lfh2_ul=bool(r["co_ul"])))
        OBS15 = pd.DataFrame(_ro)
        print(f"Part 15 — observed points: {len(OBS15)} ALMA-C11 QGs ({int(OBS15['lfd_ul'].sum())} dust ULs, {int(OBS15['lfh2_ul'].sum())} CO ULs; "
              "the H2 row compares the observed alpha_CO H2, helium included, to the simulated 1.36 M_H2 — like for like)")
    else:
        print(f"Part 15 — observed points: {OBS15_GD} or {OBS15_ST} missing; tracks only")

    Q15 = QM.merge(DOSE_ALL[KEY + ["grp"]], on=KEY, how="left")             # the route of every galaxy Part 2a could integrate
    NOX15 = Q15[Q15["box"] == G15_NOX] if (Q15["box"] == G15_NOX).any() else Q15.iloc[0:0]
    rows15 = []
    for b in BOXES_OK:
        g = Q15[(Q15["box"] == b) & Q15["grp"].notna()].reset_index(drop=True)
        if not len(g):
            continue
        tracks = [(k, g[g["grp"] == k], dict(color=GRPS[k]["color"], ls="-", lw=2.4, band=True,
                                             points=int((g["grp"] == k).sum()) <= G15_POINTS_MAX)) for k in G3]
        tracks.append(("all", g, dict(color="0.15", ls=(0, (1.2, 1.6)), lw=1.8, band=False, points=False)))
        if len(NOX15) >= G15_NMIN:
            tracks.append((G15_NOX, NOX15, dict(color="#4daf4a", ls="--", lw=2.0, band=False, points=False)))
        print(f"\n  {BOX_LABEL[b]}: " + ", ".join(f"{GSHORT[k]} {int((g['grp'] == k).sum())}" for k in G3)
              + (f"; the {G15_NOX} reference is {len(NOX15)} quenched galaxies" if len(NOX15) >= G15_NMIN else ""))
        fig, axs = plt.subplots(len(G15_Y), len(G15_X), figsize=(4.9 * len(G15_X), 4.0 * len(G15_Y)), squeeze=False)
        for i, (yc, ylab) in enumerate(G15_Y):
            for j, (xc, xlab) in enumerate(G15_X):
                ax = axs[i, j]
                for name, gg, st in tracks:
                    ok = np.isfinite(gg[xc].to_numpy(float)) & np.isfinite(gg[yc].to_numpy(float))
                    if st["points"]:
                        ax.scatter(gg.loc[ok, xc], gg.loc[ok, yc], s=16, c=st["color"], edgecolors="k", linewidths=0.3,
                                   alpha=0.65, zorder=3, rasterized=True)
                    tr = run_track15(gg[xc], gg[yc])
                    for _, r in tr.iterrows():
                        rows15.append(dict(kind="track", box=b, panel=f"{xc}_{yc}", group=name, x=r["x"], y=r["y"],
                                           lo=r["lo"], hi=r["hi"], n=r["n"], offset_dex=np.nan))
                    if len(tr):
                        if st["band"]:
                            ax.fill_between(tr["x"], tr["lo"], tr["hi"], color=st["color"], alpha=0.13, lw=0, zorder=2)
                        ax.plot(tr["x"], tr["y"], color=st["color"], ls=st["ls"], lw=st["lw"], marker="o", ms=5, mec="k", mew=0.4, zorder=4)
                    # the scorecard: the observed DETECTIONS inside the track's x range, against the track
                    if len(OBS15) and len(tr) >= 2:
                        O = OBS15[~OBS15[yc + "_ul"].astype(bool) & np.isfinite(OBS15[xc]) & np.isfinite(OBS15[yc])
                                  & OBS15[xc].between(tr["x"].min(), tr["x"].max())]
                        if len(O) >= 3:
                            d_ = O[yc].to_numpy(float) - np.interp(O[xc].to_numpy(float), tr["x"].to_numpy(float), tr["y"].to_numpy(float))
                            rows15.append(dict(kind="obs_offset", box=b, panel=f"{xc}_{yc}", group=name, x=np.nan, y=np.nan,
                                               lo=float(np.percentile(d_, 16)), hi=float(np.percentile(d_, 84)),
                                               n=len(O), offset_dex=float(np.median(d_)),
                                               x_like_for_like=bool(G15_LIKE.get(xc, True))))
                for _, o in OBS15.iterrows():
                    if not (np.isfinite(o[xc]) and np.isfinite(o[yc])):
                        continue
                    ax.scatter(o[xc], o[yc], s=70, marker="o", c=C_OBS15, edgecolors=C_OBS15_EDGE, linewidths=1.1, zorder=6)
                    if o[yc + "_ul"]:
                        ax.annotate("", (o[xc], o[yc]), xytext=(0, -13), textcoords="offset points",
                                    arrowprops=dict(arrowstyle="<|-", color=C_OBS15_EDGE, lw=1.1, mutation_scale=9), zorder=7)
                if not G15_LIKE.get(xc, True):                                          # say it on the panel, not only in the caption
                    ax.text(0.015, 0.02, "x not like for like", transform=ax.transAxes, ha="left", va="bottom",
                            fontsize=6.8, color="#8c1b52", fontweight="bold")
                if yc == "lfd":
                    ax.axhline(P9_FLOOR + 0.25, color="0.6", lw=0.8, ls=":", zorder=1)      # just above the zero-dust floor: below this is the spike
                    ax.axhline(DUSTY_CUT, color="0.4", lw=0.9, ls=":", zorder=1)
                    ax.set_ylim(P9_FLOOR - 0.4, -1.8)
                ax.set_xlabel(xlab if i == len(G15_Y) - 1 else "")
                ax.set_ylabel(ylab if j == 0 else "")
                ax.grid(alpha=0.25, lw=0.5)
        _h15  = [Line2D([], [], color=GRPS[k]["color"], lw=2.4, marker="o", ms=5, mec="k", label=GRPS[k]["label"]) for k in G3]
        _h15 += [Line2D([], [], color="0.15", lw=1.8, ls=(0, (1.2, 1.6)), label="all quenched with a route")]
        if len(NOX15) >= G15_NMIN:
            _h15 += [Line2D([], [], color="#4daf4a", lw=2.0, ls="--", label=f"{BOX_LABEL.get(G15_NOX, G15_NOX)}, all quenched")]
        if len(OBS15):
            _h15 += [Line2D([], [], ls="", marker="o", ms=8, mfc=C_OBS15, mec=C_OBS15_EDGE, label="ALMA-C11 QGs (arrow = upper limit)")]
        fig.legend(handles=_h15, loc="lower center", ncol=3, frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, -0.005))
        fig.suptitle(f"{BOX_LABEL[b]}: the ISM grid under the route split (running medians, band = bootstrap 16-84; "
                     f"zero dust at the {P9_FLOOR:g} floor)", y=0.995, fontsize=11)
        fig.tight_layout(rect=(0, 0.06, 1, 0.975))
        paper_save(fig, f"agn_states_routes_ismgrid_{b}")
        plt.show()
    R15 = pd.DataFrame(rows15)
    R15.to_csv(os.path.join(OUT, "agn_states_routes_ismgrid.csv"), index=False)
    print(f"\ntracks + observed offsets -> {os.path.join(OUT, 'agn_states_routes_ismgrid.csv')}")
    _o15 = R15[R15["kind"] == "obs_offset"] if len(R15) else R15
    if len(_o15):
        _lk = _o15[_o15["x_like_for_like"]] if "x_like_for_like" in _o15.columns else _o15
        print("median (measured - track) [dex] over the detections inside the track's x range, pooled over the x panels whose")
        print("  x axis IS like for like (Sigma_e, sSFR — the AGE panel is excluded: sim mass-weighted vs obs CIGALE onset)")
        print("  — the route the observed dusty QGs actually sit on is the one closest to 0:")
        for b in BOXES_OK:
            for yc, _ in G15_Y:
                parts = []
                for name in G3 + ["all", G15_NOX]:
                    S_ = _lk[(_lk["box"] == b) & (_lk["group"] == name) & _lk["panel"].str.endswith("_" + yc)]
                    if len(S_):
                        parts.append(f"{GSHORT.get(name, name)} {S_['offset_dex'].median():+.2f} ({int(S_['n'].sum())} det)")
                if parts:
                    print(f"  {b:8s} {yc:5s}: " + " | ".join(parts))
        print("  (the CSV keeps every panel, with x_like_for_like flagging the age column.)")
        print("  THE AGE AXIS: the simulation's age is caesar's mass-weighted age; the ALMA-C11 age is CIGALE's sfhdelayedbq")
        print("  ONSET age. For a delayed-tau SFH the mass-weighted age sits ~2 tau below the onset (0.6-1.8 Gyr at these")
        print("  ages for tau = 0.3-1 Gyr), so the observed points on that panel sit too far RIGHT; the table carries no tau,")
        print("  so they are drawn uncorrected and the panel is marked. Sigma_e and sSFR are like for like.")


## Part 16 — the DGR under two divisions of the same sample, and the quenching-time distribution behind them

The three routes are an **AGN-timing** classification. The division a quenched sample is more usually cut with is the **speed of the quenching itself**,

$$r_q \;=\; \log_{10}\frac{\tau_q}{\tau(z)}\ ,\qquad \tau_q \;=\; t_{\rm QT}-t_{\rm SFT}\ ,$$

*fast* below `P16_TSQ_CUT` $=-1.25$ and *slow* at or above it — with the default $\tau(z)$ this is **exactly the baseline dichotomy of `paper_xray_classes_m25`** (its `FAST_CUT`, $\log_{10}(\tau_q/t_{\rm QT}) < -1.25$), so the fast/slow numbers here and there are the same division of the same galaxies. **(a)** the dust-to-gas ratio cut by route, **(b)** the same histogram cut by quenching speed, on the same galaxies and the same axis; **(c)** the distribution of $r_q$ itself, which is what the cut is drawn on.

**What $\tau(z)$ is.** `t_sft` and `t_qt` come out of Part 6a of the boxes notebook as **cosmic** times, so the age of the universe at the quenching epoch *is* $t_{\rm QT}$ and the default needs no cosmology at all — the convention of `ks_tracks_lib`'s `tau_q_over_tH`. The printed clock check, median$(t_{\rm anchor} - t_{\rm H}(z_{\rm anchor}))$, is what says the branch times and the catalogue's age are the same clock. `P16_TAU` switches $\tau(z)$ to the anchor's $t_{\rm H}$ (the catalogue's `t_H_gyr`, the clock of the $0.2/t_{\rm H}$ passive cut) or to the Hubble time $1/H(z_{\rm QT})$ — the other reading of "$\tau(z)$", which sits 0.08–0.17 dex above the age over $z=0.3$–2 and therefore moves the cut, not the ordering.

**Which DGR.** `P16_DGR = "dh2"` is $\log M_{\rm dust}/M_{\rm H_2}$, the ratio to the **bare** molecular mass (the helium factor is taken back out) — the H$_2$-only convention every other measured axis in this notebook obeys. `"dg"` switches to $\log M_{\rm dust}/M_{\rm gas}$ over the **total** gas, halo gas inside the caesar aperture included; the figure stem carries the choice, so the two can sit side by side on disk.

**(a) is not a discovery, (b) is the test.** The route split has `DUSTY_CUT` inside it, so of course it separates the dust — (a) is there to set the scale of a real separation on this axis. The quenching speed knows nothing about the dust: if (b)'s two histograms sit on top of each other while (a)'s are 1–2 dex apart, the dust-to-gas ratio of a quenched galaxy is not set by how fast it quenched. The AUCs are that statement as a number — the best route pair and fast-vs-slow are annotated on the panels — the cross-tab prints how each route splits across the cut and what each speed class is made of, and $\rho(r_q, {\rm DGR})$ is the version of the question that needs no cut at all.

**(c) is the sanity check on the division.** A fast/slow dichotomy is only worth drawing if $\tau_q/\tau(z)$ really is two populations, so the panel draws the distribution and tests it two ways: the **density's interior minima** (a Gaussian KDE — assumption-free: a genuine trough between two peaks, reported with its depth relative to the lower of the two flanking peaks) and **$\Delta$BIC$(2\,\text{vs}\,1)$ of a 1-D Gaussian mixture** fitted by EM in the cell (no `sklearn` needed). "Bimodal" is printed only when *both* say so ($\Delta$BIC $>$ `P16_DBIC` $=6$ **and** a real trough). Three things are drawn with it: the **parent population** (`P16_PARENT`) — every galaxy of that box in the Part 6a class table, with none of the notebook's mass, age, redshift or quenched cuts — so a mode killed by the selection is visible; the cut; and the **snapshot-cadence floor**, median $\log_{10}(\Delta t_{\rm snap}/t_{\rm QT})$, below which a quenching faster than one snapshot cannot be resolved. Panel (c) counts **one point per quenching event**, not per anchor: `P16_UNIQ` (0.02 Gyr) merges anchors of the same box whose $(t_{\rm SFT}, t_{\rm QT})$ agree to that tolerance, because the branch is re-interpolated at every anchor and the same galaxy's times then repeat only to $\sim10^{-3}$ Gyr. (a) and (b) keep every anchor — they are ISM measurements, not events.

**Everything on the same footing.** `P16_MATCH = True` keeps only galaxies with **both** a finite DGR and an SFT/QT pair, so the two divisions are scored on identical galaxies (the toll is printed); the bins are shared by every box and by both splits; and nothing is dropped at the ends — the 1–99 percentile range is the axis, the tails are clipped into the end bins and counted.

Output: `agn_states_dgr_<dgr>_vs_quenching_speed.{png,pdf}`, `agn_states_dgr_vs_quenching_speed.csv` (per box, division and group: $N$, the median DGR and its 16–84, the dusty and fast fractions, the median $r_q$ and $\tau_q$), `agn_states_dgr_vs_quenching_speed_contrasts.csv` (AUC, $\Delta$median, Mann–Whitney and KS $p$ for every route pair and for fast vs slow) and `agn_states_quenching_time_bimodality.csv` (per box, for the sample and for the parent: $N$ events, median and 16–84 of $r_q$, the fast fraction, the KDE trough and its depth, $\Delta$BIC and both mixture components).

In [ ]:
# ── Part 16 — the DGR under two divisions of the same sample, and the quenching-time distribution behind them ──
# The three routes are an AGN-TIMING classification (Part 2a). The other division a quenched sample is usually cut with is the
# SPEED of the quenching itself: r_q = log10(tau_q / tau(z)) with tau_q = t_QT - t_SFT, fast below P16_TSQ_CUT and slow above it.
# (a) and (b) are the SAME galaxies on the SAME DGR axis, cut the two ways: if the speed division cut the dust the way the routes
# do, the routes would be a restatement of it. (c) is the sanity check on the division itself — the distribution of r_q, which
# is the quantity the fast / slow literature reports as BIMODAL, drawn for this sample and for the parent population of every
# classified galaxy in the box, with two tests on it (the density's interior minima, and BIC(1) - BIC(2) of a Gaussian mixture).
# tau(z): t_SFT and t_QT are COSMIC times, so the age of the universe at the quenching epoch IS t_QT and no cosmology is needed
# (the convention of ks_tracks_lib's tau_q_over_tH). "anchor" and "hubble" are the two other readings of "tau(z)", one knob away.
# WITH THE DEFAULTS THIS IS EXACTLY paper_xray_classes_m25's BASELINE DICHOTOMY — its FAST_CUT, log10(tau_q / t_QT) < -1.25 = fast —
# so the two notebooks cut the sample the same way and the fast/slow numbers here are comparable with the ones printed there.
P16_DGR     = "dg"     # the DGR axis: "dh2" = log M_dust/M_H2 (the notebook's molecular D/G) | "dg" = log M_dust/M_gas (the TOTAL gas)
P16_TSQ_CUT = -1.5     # THE division: r_q below this = FAST quencher, at or above it = SLOW (= paper_xray_classes_m25's FAST_CUT)
P16_TAU     = "qt"      # tau(z) = "qt": the age of the universe at QT (= t_QT) | "anchor": t_H at the anchor (the catalogue's t_H_gyr,
                        # the clock of the 0.2/t_H passive cut) | "hubble": the Hubble time 1/H(z_QT), Planck15
P16_NBIN    = 22        # the DGR bins, shared by every box and by both splits so the panels are read against each other
P16_RANGE   = None      # the DGR axis; None -> the pooled 1-99 percentiles, everything outside CLIPPED into the end bins (nothing dropped)
P16_DENSITY = False      # each group normalised to unit area — the groups are wildly unequal in number; the counts are in the legend
P16_MATCH   = True      # keep only galaxies with BOTH a finite DGR and a finite r_q, so the two splits are scored on the SAME sample
P16_SPEED   = [("fast", "#5e3c99"), ("slow", "#008837")]        # the colours of the speed classes (deliberately not the route colours)
P16_UNIQ    = 0.02      # panel (c) only: anchors of the same box whose (t_SFT, t_QT) agree within this [Gyr] are ONE quenching event
                        # (the branch is re-interpolated at every anchor, so the same galaxy's times repeat only to ~1e-3 Gyr).
                        # 0 or None = every galaxy-anchor counts. (a) and (b) always use every anchor: they are ISM measurements.
P16_PARENT  = True      # draw the parent population behind (c): EVERY galaxy of that box in the Part 6a class table, no notebook cuts
P16_DBIC    = 6.0       # BIC(1) - BIC(2) above this = the mixture prefers two components (the usual "strong evidence" threshold)

from scipy.stats import ks_2samp, gaussian_kde

if "DOSE" not in globals() or not len(DOSE):
    print("Part 16: needs Part 2a's DOSE frame (the routes and the anchor columns) — run Part 2a first.")
else:
    DGR_LAB = {"dh2": r"$\log M_{\rm dust}/M_{\rm H_2}$", "dg": r"$\log M_{\rm dust}/M_{\rm gas}$"}[P16_DGR]
    SP16  = [s for s, _ in P16_SPEED]
    SP16C = dict(P16_SPEED)

    if P16_TAU == "hubble":                                                              # the age <-> Hubble-time table, built once
        from astropy.cosmology import Planck15 as COSMO16
        _zg16 = np.linspace(0.0, 30.0, 6001)
        _ag16 = COSMO16.age(_zg16).value                                                 # the age of the universe [Gyr], decreasing in z
        _hg16 = (1.0 / COSMO16.H(_zg16)).to("Gyr").value                                 # the Hubble time 1/H(z) [Gyr]
    TAU_LAB = {"qt": r"t_{\rm H}(z_{\rm QT})", "anchor": r"t_{\rm H}(z_{\rm anchor})", "hubble": r"1/H(z_{\rm QT})"}[P16_TAU]

    def _rq16(t_sft, t_qt, t_H=None):
        """log10(tau_q / tau(z)) under P16_TAU, for any frame carrying t_SFT / t_QT [Gyr, cosmic]."""
        tau = np.asarray(t_qt, float) - np.asarray(t_sft, float)
        if P16_TAU == "anchor":
            tH = np.asarray(t_H, float)
        elif P16_TAU == "hubble":
            tH = np.interp(np.asarray(t_qt, float), _ag16[::-1], _hg16[::-1], left=np.nan, right=np.nan)
        else:
            tH = np.asarray(t_qt, float)                                                 # the age at QT IS t_QT
        with np.errstate(all="ignore"):
            return np.log10(np.where((tau > 0) & (tH > 0), tau / np.where(tH > 0, tH, np.nan), np.nan))

    def _events16(df, tol=P16_UNIQ):
        """One row per quenching EVENT: within a box, anchors whose (t_SFT, t_QT) agree within tol [Gyr] are the same galaxy seen
        again at a later anchor. tol = 0 / None -> every galaxy-anchor kept."""
        if not tol:
            return df
        out = []
        for _b, S in df.groupby("box", sort=False):
            S = S.sort_values(["t_sft", "t_qt"])
            ts, tq = S["t_sft"].to_numpy(float), S["t_qt"].to_numpy(float)
            m, lts, ltq = np.zeros(len(S), bool), -np.inf, -np.inf
            for i in range(len(S)):
                if abs(ts[i] - lts) < tol and abs(tq[i] - ltq) < tol:
                    continue
                m[i], lts, ltq = True, ts[i], tq[i]
            out.append(S[m])
        return pd.concat(out, ignore_index=True) if out else df

    def _gmm16(v, k, iters=400):
        """A 1-D k-component Gaussian mixture by EM, and its BIC. Hand-rolled: no sklearn needed on the cluster kernel."""
        v = np.asarray(v, float); n = len(v)
        if n < 12:
            return None
        if k == 1:
            mu, sd, w = np.array([v.mean()]), np.array([max(v.std(), 1e-6)]), np.array([1.0])
        else:
            mu, sd, w = np.quantile(v, np.linspace(0.2, 0.8, k)).astype(float), np.full(k, max(v.std(), 1e-6)), np.full(k, 1.0 / k)
            for _ in range(iters):
                P = w * np.exp(-0.5 * ((v[:, None] - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
                s = P.sum(1, keepdims=True); s[s <= 0] = 1e-300
                R = P / s; nk = R.sum(0)
                w = nk / n
                mu = (R * v[:, None]).sum(0) / np.maximum(nk, 1e-12)
                sd = np.sqrt(np.maximum((R * (v[:, None] - mu) ** 2).sum(0) / np.maximum(nk, 1e-12), 1e-8))
        ll = float(np.log(np.maximum((w * np.exp(-0.5 * ((v[:, None] - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))).sum(1), 1e-300)).sum())
        o = np.argsort(mu)
        return dict(w=w[o], mu=mu[o], sd=sd[o], bic=float((3 * k - 1) * np.log(n) - 2 * ll), n=n)

    def _bimod16(v):
        """Is this distribution bimodal? Two tests: the DENSITY's interior minima (assumption-free — a real dip between two peaks)
        and BIC(1) - BIC(2) of a Gaussian mixture (a fit-based one). Returns the numbers, the KDE curve and a verdict."""
        v = np.asarray(v, float); v = v[np.isfinite(v)]
        r = {k: np.nan for k in ("med", "p16", "p84", "trough", "depth", "dbic", "mu1", "mu2", "sd1", "sd2", "w1", "w2")}
        r["n"] = int(len(v)); r["fast_frac"] = float(np.mean(v < P16_TSQ_CUT)) if len(v) else np.nan
        if len(v) < 12:
            return r, None, "too few galaxies to test"
        r.update(med=float(np.median(v)), p16=float(np.percentile(v, 16)), p84=float(np.percentile(v, 84)))
        xg = np.linspace(v.min(), v.max(), 512)
        kd = gaussian_kde(v)(xg)
        best = None
        for i in range(1, len(xg) - 1):                                                  # the deepest interior minimum of the density
            if kd[i] <= kd[i - 1] and kd[i] < kd[i + 1]:
                d = kd[i] / max(min(kd[:i].max(), kd[i + 1:].max()), 1e-300)
                if best is None or d < best[1]:
                    best = (float(xg[i]), float(d))
        if best is not None:
            r["trough"], r["depth"] = best
        g1, g2 = _gmm16(v, 1), _gmm16(v, 2)
        if g1 is not None and g2 is not None:
            r.update(dbic=g1["bic"] - g2["bic"], mu1=g2["mu"][0], mu2=g2["mu"][1],
                     sd1=g2["sd"][0], sd2=g2["sd"][1], w1=g2["w"][0], w2=g2["w"][1])
        dip, two = np.isfinite(r["trough"]), (np.isfinite(r["dbic"]) and r["dbic"] > P16_DBIC)
        if dip and two:
            v_ = f"BIMODAL — a trough at {r['trough']:+.2f} ({r['depth']:.2f} x the lower peak) and dBIC(2 vs 1) = {r['dbic']:+.1f}"
        elif two:
            v_ = f"NOT bimodal — the mixture prefers two components (dBIC {r['dbic']:+.1f}) but the density has no interior minimum: one skewed peak"
        elif dip:
            v_ = f"a weak dip at {r['trough']:+.2f} ({r['depth']:.2f} x the lower peak), but the mixture does not prefer two components (dBIC {r['dbic']:+.1f})"
        else:
            v_ = f"UNIMODAL — no interior minimum in the density, and dBIC(2 vs 1) = {r['dbic']:+.1f}"
        return r, (xg, kd), v_

    # ── the frame: Part 2a's routed anchors + the quenching timings, and the two derived axes ──
    D16 = DOSE.copy()
    _miss16 = [c for c in ("t_sft", "t_qt", "t_H_gyr") if c not in D16.columns and c in QM.columns]
    if _miss16:
        D16 = D16.merge(QM[KEY + _miss16].drop_duplicates(subset=KEY), on=KEY, how="left")
    _lhe16 = np.log10(float(globals().get("HE", 1.36)))
    with np.errstate(all="ignore"):
        D16["dh2"] = D16["lfd"].to_numpy(float) - D16["lfh2"].to_numpy(float) + _lhe16   # per unit BARE H2 mass (Part 11's convention)
        D16["dg"]  = D16["lfd"].to_numpy(float)# - D16["lfgas"].to_numpy(float)           # the TOTAL gas: the halo gas inside the aperture is in it
    D16["tau_q"] = D16["t_qt"].to_numpy(float) - D16["t_sft"].to_numpy(float)
    D16["r_q"]   = _rq16(D16["t_sft"], D16["t_qt"], D16.get("t_H_gyr"))
    _rq16v = D16["r_q"].to_numpy(float)
    D16["speed"] = np.where(np.isfinite(_rq16v), np.where(_rq16v < P16_TSQ_CUT, SP16[0], SP16[1]), "no SFT/QT")

    _n16_0, _x16 = len(D16), D16[P16_DGR].to_numpy(float)
    _keep16 = np.isfinite(_x16) & (np.isfinite(_rq16v) if P16_MATCH else np.ones(len(D16), bool))
    _lost_dgr, _lost_rq = int((~np.isfinite(_x16)).sum()), int((np.isfinite(_x16) & ~np.isfinite(_rq16v)).sum())
    D16 = D16[_keep16].reset_index(drop=True)
    _x16 = D16[P16_DGR].to_numpy(float)
    lo16, hi16 = ((float(np.nanpercentile(_x16, 1)), float(np.nanpercentile(_x16, 99))) if P16_RANGE is None
                  else (float(P16_RANGE[0]), float(P16_RANGE[1])))
    E16 = np.linspace(lo16, hi16, P16_NBIN + 1)
    D16["_x"] = np.clip(_x16, lo16 + 1e-6, hi16 - 1e-6)                                  # the tails pile into the end bins; nothing is dropped
    _nclip16 = int(np.sum((_x16 < lo16) | (_x16 > hi16)))

    # ── the parent population of panel (c): every galaxy of the box in the Part 6a class table, none of the notebook's cuts ──
    PAR16 = None
    if P16_PARENT and "CLS" in globals():
        PAR16 = CLS[np.isfinite(CLS["t_sft"].to_numpy(float)) & np.isfinite(CLS["t_qt"].to_numpy(float))].copy()
        if P16_TAU == "anchor" and "CAT" in globals() and "t_H_gyr" in CAT.columns:
            PAR16 = PAR16.merge(CAT[["box", "snap", "t_H_gyr"]].drop_duplicates(["box", "snap"]), on=["box", "snap"], how="left")
        PAR16["r_q"] = _rq16(PAR16["t_sft"], PAR16["t_qt"], PAR16.get("t_H_gyr"))

    print(f"Part 16 — the DGR ({P16_DGR}) distribution under TWO divisions of the same {len(D16)} galaxy-anchors (the three AGN "
          f"routes of Part 2a, and the quenching SPEED), and the distribution of the quenching time itself.")
    print(f"  the speed axis: r_q = log10( (t_QT - t_SFT) / tau(z) ),  tau(z) = {P16_TAU} "
          + {"qt": "= the age of the universe at QT, which IS t_QT (t_SFT / t_QT are cosmic times)",
             "anchor": "= the catalogue's t_H_gyr at the anchor redshift",
             "hubble": "= the Hubble time 1/H(z_QT), Planck15"}[P16_TAU]
          + f";  the cut at r_q = {P16_TSQ_CUT:g}.")
    if "t_H_gyr" in D16.columns:
        _chk16 = np.nanmedian(D16["t_anchor"].to_numpy(float) - D16["t_H_gyr"].to_numpy(float))
        print(f"  the clock check: median(t_anchor - t_H(z_anchor)) = {_chk16:+.3f} Gyr — the branch times and the catalogue's "
              f"age of the universe are the same clock, so t_H(z_QT) = t_QT holds.")
    print(f"  {_n16_0} routed anchors -> {len(D16)} used"
          + (f"  (no finite {P16_DGR}: {_lost_dgr}" if _lost_dgr else "  (")
          + (f"{'; ' if _lost_dgr else ''}no SFT/QT pair: {_lost_rq}" if _lost_rq and P16_MATCH else "")
          + f"{'; ' if (_lost_dgr or _lost_rq) else ''}clipped into the end bins: {_nclip16})")
    print(f"  the DGR axis: [{lo16:.2f}, {hi16:.2f}] in {P16_NBIN} bins, shared by every panel"
          + ("; each group normalised to unit area." if P16_DENSITY else "; raw counts."))
    print(f"  panel (c) counts ONE ROW PER QUENCHING EVENT: anchors of the same box agreeing to {P16_UNIQ:g} Gyr in (t_SFT, t_QT) "
          f"are the same galaxy seen again" if P16_UNIQ else "  panel (c) counts every galaxy-anchor (P16_UNIQ = 0).")

    # ── the numbers: per box, the DGR of each group under both divisions, the contrasts, the cross-tab, then the r_q distribution ──
    def _stat16(v):
        v = np.asarray(v, float); v = v[np.isfinite(v)]
        if not len(v):
            return dict(n=0, med=np.nan, p16=np.nan, p84=np.nan)
        return dict(n=int(len(v)), med=float(np.median(v)), p16=float(np.percentile(v, 16)), p84=float(np.percentile(v, 84)))

    rows16, con16, bim16, best16, KDE16 = [], [], [], {}, {}
    for b in BOXES_OK:
        B = D16[D16["box"] == b]
        if not len(B):
            continue
        print(f"\n  {BOX_LABEL[b]} — {len(B)} anchors")
        print(f"    {'division':10s} {'group':26s} {'N':>5s} {'med DGR':>9s} {'16-84':>15s} {'dusty':>7s} {'fast':>7s}")
        for split, groups, col in (("route", G3, "grp"), ("speed", SP16, "speed")):
            for g in groups:
                S = B[B[col] == g]
                st = _stat16(S["_x"])
                fr_d = float(np.mean(S["dusty"].to_numpy(bool))) if len(S) else np.nan
                fr_f = float(np.mean((S["speed"] == SP16[0]).to_numpy(bool))) if len(S) else np.nan
                rows16.append(dict(box=b, split=split, group=g, dgr=P16_DGR, **st, dusty_frac=fr_d, fast_frac=fr_f,
                                   r_q_med=float(np.nanmedian(S["r_q"].to_numpy(float))) if len(S) else np.nan,
                                   tau_q_med=float(np.nanmedian(S["tau_q"].to_numpy(float))) if len(S) else np.nan))
                _rs16 = "[%.2f, %.2f]" % (st["p16"], st["p84"])
                print(f"    {split:10s} {g[:26]:26s} {st['n']:5d} {st['med']:9.2f} {_rs16:>15s} "
                      f"{100 * fr_d:6.0f}% {100 * fr_f:6.0f}%")
        print(f"    the contrasts on the DGR (AUC = P(group 1 > group 0), Mann-Whitney p, two-sample KS p):")
        for split, pairs in (("route", GPAIRS), ("speed", [(SP16[0], SP16[1])])):
            col = "grp" if split == "route" else "speed"
            for g1, g0 in pairs:
                P_ = B[B[col].isin([g0, g1])]
                auc, pmw, n1, n0 = mw_auc(P_["_x"].to_numpy(float), (P_[col] == g1).to_numpy(bool))
                a_, b_ = (P_.loc[P_[col] == g1, "_x"].to_numpy(float), P_.loc[P_[col] == g0, "_x"].to_numpy(float))
                pks = float(ks_2samp(a_[np.isfinite(a_)], b_[np.isfinite(b_)]).pvalue) if (len(a_) > 2 and len(b_) > 2) else np.nan
                dmed = (np.nanmedian(a_) - np.nanmedian(b_)) if (len(a_) and len(b_)) else np.nan
                con16.append(dict(box=b, split=split, group1=g1, group0=g0, n1=n1, n0=n0, auc=auc, p_mw=pmw, p_ks=pks, dmed=dmed))
                print(f"      {split:6s} {GSHORT.get(g1, g1)[:14]:>14s} vs {GSHORT.get(g0, g0)[:14]:<14s} "
                      f"AUC {auc:5.2f}   dmed {dmed:+5.2f} dex   p_MW {pmw:9.2e}   p_KS {pks:9.2e}")
        _r = [c for c in con16 if c["box"] == b and c["split"] == "route" and np.isfinite(c["auc"])]
        _s = [c for c in con16 if c["box"] == b and c["split"] == "speed" and np.isfinite(c["auc"])]
        if _r and _s:
            _rb = max(_r, key=lambda c: abs(c["auc"] - 0.5))
            best16[b] = (_rb, _s[0])
            print(f"    -> the routes separate the DGR at AUC {_rb['auc']:.2f} "
                  f"({GSHORT.get(_rb['group1'], _rb['group1'])} vs {GSHORT.get(_rb['group0'], _rb['group0'])}), "
                  f"the fast/slow cut at AUC {_s[0]['auc']:.2f}.")
        print(f"    the cross-tab (each route's split by quenching speed, and each speed class's route composition):")
        for g in G3:
            S = B[B["grp"] == g]
            if len(S):
                print(f"      {g[:26]:26s} n {len(S):4d}   " + "  ".join(f"{s} {100 * np.mean((S['speed'] == s).to_numpy(bool)):3.0f}%" for s in SP16))
        for s in SP16:
            S = B[B["speed"] == s]
            if len(S):
                print(f"      {s:26s} n {len(S):4d}   " + "  ".join(f"{GSHORT.get(g, g)} {100 * np.mean((S['grp'] == g).to_numpy(bool)):3.0f}%" for g in G3))
        _rho, _p = spearmanr(B["r_q"].to_numpy(float), B["_x"].to_numpy(float), nan_policy="omit")
        print(f"      the continuous version: rho(r_q, DGR) = {float(_rho):+.2f} (p {float(_p):.1e}) over all {len(B)} of them")

        # ── (c): the quenching time itself — is it bimodal, in this sample and in the parent population? ──
        Sev = _events16(B)
        st_s, kde_s, verd_s = _bimod16(Sev["r_q"].to_numpy(float))
        KDE16[b] = {"sample": kde_s}
        bim16.append(dict(box=b, sample="this notebook's sample", n_anchor=len(B), **st_s))
        _fl = np.nanmedian(np.log10(B["dt_samp"].to_numpy(float) / B["t_qt"].to_numpy(float))) if "dt_samp" in B.columns else np.nan
        print(f"    THE QUENCHING TIME ITSELF — {st_s['n']} events (from {len(B)} anchors), median r_q {st_s['med']:+.2f}, "
              f"16-84 [{st_s['p16']:+.2f}, {st_s['p84']:+.2f}], {100 * st_s['fast_frac']:.0f} % fast")
        print(f"      sample: {verd_s}")
        if np.isfinite(st_s["dbic"]):
            print(f"        the two-component fit: w {st_s['w1']:.2f} at {st_s['mu1']:+.2f} (sd {st_s['sd1']:.2f})  +  "
                  f"w {st_s['w2']:.2f} at {st_s['mu2']:+.2f} (sd {st_s['sd2']:.2f})")
        if np.isfinite(_fl):
            print(f"        the snapshot cadence floor: median log10(dt_snap / t_QT) = {_fl:+.2f} — a quenching faster than one "
                  f"snapshot cannot be resolved below that, and the cut sits {P16_TSQ_CUT - _fl:+.2f} dex from it")
        if PAR16 is not None and (PAR16["box"] == b).any():
            Pev = _events16(PAR16[PAR16["box"] == b].copy())
            st_p, kde_p, verd_p = _bimod16(Pev["r_q"].to_numpy(float))
            KDE16[b]["parent"] = kde_p
            bim16.append(dict(box=b, sample="parent (every classified galaxy)", n_anchor=int((PAR16["box"] == b).sum()), **st_p))
            print(f"      parent ({st_p['n']} events, no mass / age / z / quenched cut): median r_q {st_p['med']:+.2f}, "
                  f"{100 * st_p['fast_frac']:.0f} % fast — {verd_p}")

    # ── the figure ──
    _rqa = np.concatenate([D16["r_q"].to_numpy(float)] + ([PAR16["r_q"].to_numpy(float)] if PAR16 is not None else []))
    _rqa = _rqa[np.isfinite(_rqa)]
    Eq16 = (np.linspace(float(np.percentile(_rqa, 0.5)), float(np.percentile(_rqa, 99.5)), P16_NBIN + 1) if len(_rqa) > 4
            else np.linspace(-2.0, 0.0, 11))                                   # the speed axis, shared by every box
    fig, axs = plt.subplots(len(BOXES_OK), 3, figsize=(4.9 * 3, 3.05 * len(BOXES_OK)), squeeze=False, constrained_layout=True)
    for ri, b in enumerate(BOXES_OK):
        B = D16[D16["box"] == b]
        # (a) the DGR, split by the three routes
        ax = axs[ri, 0]
        for g in G3:
            v = B.loc[B["grp"] == g, "_x"].to_numpy(float); v = v[np.isfinite(v)]
            if not len(v):
                continue
            ax.hist(v, bins=E16, density=P16_DENSITY, histtype="stepfilled", color=GRPS[g]["color"], alpha=0.25, lw=0, zorder=2)
            ax.hist(v, bins=E16, density=P16_DENSITY, histtype="step", color=GRPS[g]["color"], lw=2.1, zorder=3,
                    label=f"{GSHORT.get(g, g)}  ({len(v)})")
            ax.axvline(np.median(v), color=GRPS[g]["color"], lw=1.5, ls="--", alpha=0.9, zorder=4)
        ax.legend(frameon=False, fontsize=8, loc="upper left")
        ax.set_ylabel(f"{BOX_LABEL[b]}\n" + ("density" if P16_DENSITY else "N"), fontsize=9.5)
        if b in best16:
            ax.text(0.985, 0.965, f"best pair AUC {best16[b][0]['auc']:.2f}", transform=ax.transAxes, ha="right", va="top",
                    fontsize=8.5, color="0.25", fontweight="bold", bbox=dict(boxstyle="round,pad=0.2", fc="w", ec="none", alpha=0.72))
        # (b) the same DGR, split by the quenching speed
        ax = axs[ri, 1]
        for s in SP16:
            v = B.loc[B["speed"] == s, "_x"].to_numpy(float); v = v[np.isfinite(v)]
            if not len(v):
                continue
            _sym16 = "<" if s == SP16[0] else r"$\geq$"
            ax.hist(v, bins=E16, density=P16_DENSITY, histtype="stepfilled", color=SP16C[s], alpha=0.25, lw=0, zorder=2)
            ax.hist(v, bins=E16, density=P16_DENSITY, histtype="step", color=SP16C[s], lw=2.1, zorder=3,
                    label=rf"{s}  ($r_q$ {_sym16} {P16_TSQ_CUT:g})  ({len(v)})")
            ax.axvline(np.median(v), color=SP16C[s], lw=1.5, ls="--", alpha=0.9, zorder=4)
        ax.legend(frameon=False, fontsize=8, loc="upper left")
        if b in best16:
            ax.text(0.985, 0.965, f"fast vs slow AUC {best16[b][1]['auc']:.2f}", transform=ax.transAxes, ha="right", va="top",
                    fontsize=8.5, color="0.25", fontweight="bold", bbox=dict(boxstyle="round,pad=0.2", fc="w", ec="none", alpha=0.72))
        # (c) the quenching time itself: the distribution the fast / slow division is drawn on
        ax = axs[ri, 2]
        _bim = [r for r in bim16 if r["box"] == b]
        Sev = _events16(B)
        v = Sev["r_q"].to_numpy(float); v = v[np.isfinite(v)]
        ax.axvspan(Eq16[0], P16_TSQ_CUT, color=SP16C[SP16[0]], alpha=0.07, lw=0, zorder=0)
        ax.axvspan(P16_TSQ_CUT, Eq16[-1], color=SP16C[SP16[1]], alpha=0.07, lw=0, zorder=0)
        if len(v):
            ax.hist(np.clip(v, Eq16[0] + 1e-6, Eq16[-1] - 1e-6), bins=Eq16, density=True, histtype="stepfilled",
                    color="0.45", alpha=0.30, lw=0, zorder=2)
            ax.hist(np.clip(v, Eq16[0] + 1e-6, Eq16[-1] - 1e-6), bins=Eq16, density=True, histtype="step",
                    color="0.15", lw=2.0, zorder=3, label=f"this sample ({len(v)} events)")
        if KDE16.get(b, {}).get("sample") is not None:
            xg, kd = KDE16[b]["sample"]
            ax.plot(xg, kd, "-", color="k", lw=1.5, alpha=0.85, zorder=4)
        if KDE16.get(b, {}).get("parent") is not None:
            xg, kd = KDE16[b]["parent"]
            _np_ = [r for r in _bim if r["sample"].startswith("parent")]
            ax.plot(xg, kd, "--", color="#b15928", lw=2.0, alpha=0.95, zorder=4,
                    label=f"parent, no cuts ({int(_np_[0]['n']) if _np_ else 0} events)")
        ax.axvline(P16_TSQ_CUT, color="k", lw=1.6, zorder=5)
        ax.text(P16_TSQ_CUT, 0.02, f"  {P16_TSQ_CUT:g}", transform=ax.get_xaxis_transform(), ha="left", va="bottom",
                fontsize=8.5, color="k", fontweight="bold")
        if "dt_samp" in B.columns:
            _fl = np.nanmedian(np.log10(B["dt_samp"].to_numpy(float) / B["t_qt"].to_numpy(float)))
            if np.isfinite(_fl) and Eq16[0] <= _fl <= Eq16[-1]:
                ax.axvline(_fl, color="0.45", lw=1.2, ls=":", zorder=5,
                           label=("one snapshot" if ri == 0 else None))
        _ss = [r for r in _bim if r["sample"].startswith("this")]
        if _ss and np.isfinite(_ss[0]["dbic"]):
            _lab = ("bimodal" if (np.isfinite(_ss[0]["trough"]) and _ss[0]["dbic"] > P16_DBIC) else "unimodal")
            ax.text(0.985, 0.965, f"{_lab}:  $\\Delta$BIC(2 vs 1) = {_ss[0]['dbic']:+.1f}", transform=ax.transAxes,
                    ha="right", va="top", fontsize=8.5, color="0.25", fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="w", ec="none", alpha=0.72))
        ax.legend(frameon=False, fontsize=8, loc="upper left")
        ax.set_xlim(Eq16[0], Eq16[-1])
        for ci in range(3):
            axs[ri, ci].grid(alpha=0.22, lw=0.5)
            if ci < 2:
                axs[ri, ci].set_xlim(lo16, hi16)
            if ri == len(BOXES_OK) - 1:
                axs[ri, ci].set_xlabel([DGR_LAB, DGR_LAB, rf"$r_q = \log_{{10}}(\tau_q / {TAU_LAB})$"][ci], fontsize=10)
        if ri == 0:
            for ci, t in enumerate([f"(a) {DGR_LAB} — the three AGN routes", f"(b) {DGR_LAB} — fast vs slow quenching",
                                    "(c) the quenching time itself — bimodal?"]):
                axs[ri, ci].set_title(t, fontsize=10)
    fig.suptitle(rf"the dust-to-gas ratio under two divisions of the same sample: the AGN-timing routes, and the quenching speed "
                 rf"$r_q = \log_{{10}}(\tau_q / {TAU_LAB})$ cut at ${P16_TSQ_CUT:g}$"
                 + "\n" + r"(c) is the distribution that cut is drawn on — one point per quenching EVENT, the parent population "
                          r"(none of the notebook's cuts) dashed; $\tau_q = t_{\rm QT} - t_{\rm SFT}$", fontsize=11.5)
    paper_save(fig, f"agn_states_dgr_{P16_DGR}_vs_quenching_speed")
    plt.show()

    R16, C16, BM16 = pd.DataFrame(rows16), pd.DataFrame(con16), pd.DataFrame(bim16)
    R16.to_csv(os.path.join(OUT, "agn_states_dgr_vs_quenching_speed.csv"), index=False)
    C16.to_csv(os.path.join(OUT, "agn_states_dgr_vs_quenching_speed_contrasts.csv"), index=False)
    BM16.to_csv(os.path.join(OUT, "agn_states_quenching_time_bimodality.csv"), index=False)
    print(f"\nper box and group -> {os.path.join(OUT, 'agn_states_dgr_vs_quenching_speed.csv')}")
    print(f"the contrasts     -> {os.path.join(OUT, 'agn_states_dgr_vs_quenching_speed_contrasts.csv')}")
    print(f"the (c) tests     -> {os.path.join(OUT, 'agn_states_quenching_time_bimodality.csv')}")
    print("READ IT LIKE THIS: (a) and (b) are the SAME galaxies and the same axis, cut two ways. The route division is a dust cut "
          "by construction (DUSTY_CUT enters it), so its separation in (a) is not a discovery — what (b) asks is whether the "
          "quenching SPEED, which knows nothing about the dust, reproduces it. (c) is the check on the division itself: the "
          "fast / slow dichotomy is worth drawing only if r_q is really two populations. Where (c) comes out unimodal, the cut at "
          f"{P16_TSQ_CUT:g} is a slice through one continuous distribution, not a boundary between two — and a null result in (b) "
          "then says the dust does not care about the quenching speed ANYWHERE along that distribution, which is the stronger "
          "reading, not a weaker one (rho(r_q, DGR) above is the version that needs no cut at all).")

## Part 17 — one snapshot: the quenched galaxies binned by age, and inside each age bin their AGN histories split by the **observed** $f_{\rm dust}$

Everything from Part 2a on is built on thresholds *we* chose — `PF_EX`, `PF_TQ`, `DUSTY_CUT` — applied to a frame that stacks ten anchors, in which one galaxy can appear five times. This part drops all of it and asks the question in the rawest form available:

**one snapshot; every galaxy exactly once; bin by stellar age; inside each age bin split by the dust an observer would measure ($\log M_{\rm dust}/M_\star$); draw what the black holes were doing.** The clock is *lookback from the snapshot itself*, so nothing here depends on $t_{\rm SFT}$ or $t_{\rm QT}$ — which matters twice over while those are being re-measured (Part 16 and the `QT_SMOOTH` fix upstream).

**What is drawn per age bin (a column), one coloured line per dust bin.** The X-ray state occupancy; the jet criterion; the fraction with the X-ray gate open ($f_{\rm gas} < $ `XRAY_FGAS_MAX`); $\log M_{\rm gas}/M_\star$, the gate variable itself — the one place besides the metal budget where the total gas is allowed in this notebook, because it *is* SIMBA's criterion; $\log \dot M_{\rm BH}$; $\log f_{\rm Edd}$; and $E_x$ accumulated between that lookback and the snapshot. Bands are bootstrap 16–84 over the galaxies, and an epoch is blank where fewer than `P17_NMIN` chains reach it.

**Two things the binning cannot fake.** The dust bins are quantiles *inside* each age bin, so the split is never the age–dust correlation; and the printed $\rho$ table repeats the whole question with the dust left **continuous** — Spearman $\rho(\log M_{\rm dust}/M_\star,\ X)$ inside each age bin for $E_x$, the duty cycle, $t_{\rm quiet}$, X-ray-on-now, the gate, $\log M_{\rm BH}$, $\log M_\star$ (control) and $\log M_{\rm gas}/M_\star$ (the gate variable). No choice of bin edge can manufacture that number.

**The table under the figure answers "are the dusty ones still being dosed?" directly** — per (age, dust) cell: $N$, the medians, the fraction **in the X-ray state at the snapshot**, the fraction with the gate open, the fraction that never fired, and the median $E_x$ / $t_{\rm quiet}$.

**A caveat this part will print at you, and it is not small.** The histories cache was built by `paper_ism_prediction_boxes` Part 6a for **its** quenched pool, with **its** constants (`PASSIVE_FACTOR = 0.2`, `DUST_TO_H2_MIN = 1e-4`). Where this notebook's Part 0 constants are looser, the extra galaxies have no main branch at all — so every history-based part here, Parts 2a–16 included, silently runs on the galaxies that *do* have one, and that subset is selected along the dust axis. Part 17 therefore restricts to the galaxies with a chain **before** it bins, and prints the toll; Part 0 now prints the same coverage per box. If the two samples have drifted far apart, either bring Part 0's constants back to the boxes notebook's or rebuild the histories there with the looser ones.

`P17_BOX` / `P17_SNAP` (`None` = the richest anchor), `P17_CUTS` (`"no_age"` = the quenched rule + `MASS_MIN` only, since `AGE_MIN` would truncate the very axis we bin on; `"part0"` = `QM` verbatim), `P17_NAGE` / `P17_AGE_EDGES`, `P17_NFD` / `P17_FD_EDGES`, `P17_TMAX` / `P17_DT`, `P17_NMIN`, `P17_ROWS`.

Output: `agn_states_snapshot_agnhist_<box>_snap<NNN>.{png,pdf}` and, beside it, `_tracks.csv` (every line of the figure), `_cells.csv` (the snapshot table), `_rho.csv` (the unbinned control) and `_galaxies.csv` (one row per galaxy: the bins, the anchor-epoch AGN summaries and the catalogue axes).

In [ ]:
# ── Part 17 — ONE SNAPSHOT: the quenched galaxies binned by age, and inside each age bin their AGN histories split by the OBSERVED dust ──
# The routes of Part 2a are built from thresholds WE chose (PF_EX, PF_TQ, DUSTY_CUT) on a frame that mixes ten anchors, and a
# galaxy can enter it several times. This part throws all of that away and asks the question in the rawest form there is:
# take ONE snapshot, bin its quenched galaxies by stellar age, and inside each age bin split them by the dust an observer would
# measure. Then draw what the black holes of each group were doing, on the ONLY clock that needs no quenching times: lookback
# from the snapshot itself. No SFT, no QT, no exposure threshold, no route. Every galaxy appears exactly once.
# TWO CONTROLS ARE BUILT IN. (1) P17_SPLIT chooses WHICH dust variable the colours are cut on: log M_dust/M* is largely a
# statement about how much gas is left, so "dtm" (dust-to-metal) and "dh2" (per unit H2) are the grain-survival versions of the
# same question — the printed rho table scores all three whatever the colours show. (2) P17_BOXES may list the CHANNEL-OFF run
# cis50nox: if a dust spread at fixed age survives there, where the X-ray mode never fired at all, no AGN history can explain it.
P17_BOXES    = ["cis100", "cis50nox"]   # one figure per box; cis50nox = the X-ray-off control (its AGN-state rows are WOULD-BE states)
P17_SNAP     = None        # the anchor snapshot; None = the one with the most quenched galaxies in that box
P17_CUTS     = "no_age"    # "no_age" = Part 0's quenched rule + MASS_MIN only, since AGE_MIN would truncate the very axis we bin on
                           # "part0"  = QM exactly as Part 0 declared it (age- and z-cut), for continuity with Parts 1-16
P17_SPLIT    = "lfd"       # the dust variable the colour bins are cut on: "lfd" = log M_dust/M* (what an observer reports),
                           # "dh2" = log M_dust/M_H2, "dtm" = log dust-to-metal (M_dust / (Z_gas M_gas + M_dust)) — the survival versions
P17_NAGE     = 3           # age bins; P17_AGE_EDGES overrides with explicit edges [Gyr]
P17_AGE_EDGES = None
P17_NFD      = 3           # dust bins INSIDE each age bin (quantiles there, so the split is not the age-dust correlation)
P17_FD_EDGES = None        # explicit edges of P17_SPLIT, shared by every age bin, if you want fixed dust bins instead
P17_TMAX     = 4.0         # how far back the histories are drawn [Gyr of lookback from the snapshot]
P17_DT       = 0.1         # the lookback grid step [Gyr]; a galaxy contributes the nearest snapshot within this distance
P17_NMIN     = 5           # an epoch is drawn only where this many galaxies of the group contribute
P17_NBOOT    = 200
P17_CMAP     = "plasma"    # the dust bins, dust-poor -> dusty

P17_ROWS = [("xray",  r"in the X-ray state   [fraction]",      "frac"),
            ("jet",   r"jet criterion met   [fraction]",       "frac"),
            ("gate",  rf"$f_{{\rm gas}} < {XRAY_FGAS_MAX:g}$ (the X-ray gate)   [fraction]", "frac"),
            ("lfgas", r"$\log M_{\rm gas}/M_\star$   (the gate variable)", "med"),
            ("lmdot", r"$\log \dot M_{\rm BH}$   [$M_\odot$ yr$^{-1}$]", "med"),
            ("lfedd", r"$\log f_{\rm Edd}$",                   "med"),
            ("excum", r"$E_x$ accumulated since then   [Gyr]", "med")]
P17_DUSTVARS = [("lfd", r"$\log M_{\rm dust}/M_\star$"), ("dh2", r"$\log M_{\rm dust}/M_{\rm H_2}$"),
                ("dtm", r"$\log$ dust-to-metal")]
P17_TESTS = [("e_x", "E_x [Gyr]"), ("f_x", "duty f_x"), ("t_quiet", "t_quiet"), ("xray_now", "X-ray now"),
             ("gate_now", "gate now"), ("log_mbh", "log M_BH"), ("log_mstar", "log M* (ctrl)"),
             ("lfgas_now", "log fgas = GATE"), ("lfh2", "log fH2 (ctrl)")]

if "CAT" not in globals():
    print("Part 17: needs Part 0's catalogue cache — run Part 0 first.")
else:
    TL17 = np.arange(0.0, P17_TMAX + 1e-9, P17_DT)
    _boxes17 = [P17_BOXES] if isinstance(P17_BOXES, str) else list(P17_BOXES)
    ALL17 = {"cells": [], "rho": [], "tracks": [], "gal": []}

    def _agg17(A, kind):
        A = np.asarray(A, float)
        if not len(A):
            return (np.full(len(TL17), np.nan),) * 3 + (np.zeros(len(TL17), int),)
        n = np.isfinite(A).sum(axis=0)
        fun = np.nanmean if kind == "frac" else np.nanmedian
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            cent = fun(A, axis=0)
            rng = np.random.default_rng(17)
            bs = np.array([fun(A[rng.integers(0, len(A), len(A))], axis=0) for _ in range(P17_NBOOT)])
            lo, hi = np.nanpercentile(bs, 16, axis=0), np.nanpercentile(bs, 84, axis=0)
        ok = n >= P17_NMIN
        return np.where(ok, cent, np.nan), np.where(ok, lo, np.nan), np.where(ok, hi, np.nan), n

    def p17_run(box):
        _p = HIST_FILE.format(box=box)
        if not os.path.exists(_p) or not (CAT["box"] == box).any():
            print(f"\nPart 17 [{box}]: no catalogue rows or no histories cache — skipped."); return
        # ── the sample: ONE snapshot, the quenched rule, and (by default) no age cut ──
        if P17_CUTS == "part0" and "QM" in globals() and (QM["box"] == box).any():
            B = QM[QM["box"] == box].copy(); rule = "Part 0's QM verbatim (mass, redshift AND age cut)"
        else:
            _c = CAT[CAT["box"] == box].copy()
            _ms, _sfr = _c["mstar"].to_numpy(float), _c["sfr"].to_numpy(float)
            with np.errstate(all="ignore"):
                _ssfr = np.where(_ms > 0, _sfr / _ms, np.nan)
                _lm = np.log10(np.where(_ms > 0, _ms, np.nan))
            _q = ((_lm > MASS_FLOOR) & (_ssfr < PASSIVE_FACTOR / (_c["t_H_gyr"].to_numpy(float) * 1e9))
                  & (_c["ngas"].to_numpy(float) >= NGAS_MIN) & (_c["nstar"].to_numpy(float) >= NSTAR_MIN)
                  & (_c["mdust"].to_numpy(float) >= DUST_TO_H2_MIN * _c["mh2"].to_numpy(float)) & (_lm > MASS_MIN))
            B = _c[_q].copy()
            rule = f"the m25 quenched rule + log M* > {MASS_MIN:g}, NO age or redshift cut (one snapshot, so z is fixed anyway)"
        with np.errstate(all="ignore"):
            _m = np.where(B["mstar"].to_numpy(float) > 0, B["mstar"].to_numpy(float), np.nan)
            B["log_mstar"] = np.log10(_m)
            _fd = B["mdust"].to_numpy(float) / _m
            B["lfd"] = np.where(_fd > 0, np.log10(np.where(_fd > 0, _fd, 1.0)), P9_FLOOR)
            B["lfh2"] = np.log10(np.where(B["mh2"].to_numpy(float) > 0, HE * B["mh2"].to_numpy(float) / _m, np.nan))
            B["lfgas"] = np.log10(np.where(B["mgas"].to_numpy(float) > 0, B["mgas"].to_numpy(float) / _m, np.nan))
            B["dh2"] = np.log10(np.where((B["mh2"].to_numpy(float) > 0) & (B["mdust"].to_numpy(float) > 0),
                                         B["mdust"].to_numpy(float) / B["mh2"].to_numpy(float), np.nan))
            _mbh = B["mbh"].to_numpy(float)
            B["log_mbh"] = np.where(_mbh > 0, np.log10(np.where(_mbh > 0, _mbh, 1.0)), np.nan)
        snaps = B.groupby("snap").size().sort_values(ascending=False)
        if not len(snaps):
            print(f"\nPart 17 [{box}]: the quenched rule leaves nothing — skipped."); return
        snap = int(P17_SNAP) if P17_SNAP is not None else int(snaps.index[0])
        B = B[B["snap"] == snap].reset_index(drop=True)
        z17 = float(B["z"].iloc[0]) if len(B) else np.nan
        nox = (box == globals().get("NOX_BOX", "cis50nox"))
        print(f"\nPart 17 [{box}] — ONE SNAPSHOT: {BOX_LABEL.get(box, box)}, snapshot {snap} (z = {z17:.2f}), "
              f"{len(B)} quenched galaxies, each exactly once."
              + ("\n  THE X-RAY CHANNEL WAS OFF IN THIS RUN: its AGN-state rows are the WOULD-BE states, i.e. what SIMBA would "
                 "have fired had the channel been on. Any dust spread here is NOT made by X-ray feedback." if nox else ""))
        print(f"  the sample: {rule}"
              + ("" if P17_SNAP is not None else f"; snapshot chosen as the richest ({dict(list(snaps.items())[:4])} ...)"))

        # ── only the galaxies whose main branch is in the cache, and the binning happens AFTER that restriction ──
        with h5py.File(_p, "r") as f:
            grp = f"snap{snap:03d}"
            if grp not in f:
                print(f"  snapshot {snap} is not in the histories cache (it has {sorted(f)}) — skipped."); return
            g = f[grp]
            t17 = g["t_yr"][:] / 1e9; o17 = np.argsort(t17); t17 = t17[o17]
            ms17, mg17, mb17, fe17 = (g[c][:][o17] for c in ("mstar", "mgas", "mbh", "fedd"))
            zg17 = g["zgas"][:][o17] if "zgas" in g else np.full_like(ms17, np.nan)
            ids17 = np.asarray(g["gal_ids"][:], dtype=np.int64)
            S17, _, X17 = agn_states(ms17, mg17, mb17, fe17)
            dtw17 = np.gradient(t17)
            with np.errstate(all="ignore"):
                Q17 = {"xray":  np.nan_to_num(S17["xray"]),
                       "jet":   np.nan_to_num(S17["xray"]) + np.nan_to_num(S17["choked"]),
                       "gate":  (X17["fgas"] < XRAY_FGAS_MAX).astype(float),
                       "lfgas": np.log10(np.clip(X17["fgas"], FGAS_FLOOR, None)),   # the gate is a criterion of the RUN, not an ISM axis
                       "lmdot": np.log10(np.clip(X17["mdot"], MDOT_FLOOR, None)),
                       "lfedd": np.log10(np.clip(fe17, FEDD_FLOOR, None))}
            n_all = len(B)
            B = B[B["gal_id"].astype(np.int64).isin(set(ids17))].reset_index(drop=True)
            print(f"  with a main branch in the histories cache: {len(B)} of {n_all}"
                  + (f"  ({n_all - len(B)} dropped — the cache holds the {len(ids17)} galaxies of the boxes notebook's own "
                     f"quenched pool at this anchor, built with ITS constants)" if len(B) < n_all else "")
                  + ".  THE BINS BELOW ARE BUILT ON THESE GALAXIES ONLY.")
            if len(B) < 3 * P17_NAGE * P17_NFD:
                print(f"  {len(B)} galaxies is too few for {P17_NAGE} x {P17_NFD} groups — lower them or pick a richer P17_SNAP."); return
            col_of = {int(v): j for j, v in enumerate(ids17)}
            TRK, ANC = {q: {} for q, _, _ in P17_ROWS}, {}
            for gid in B["gal_id"].astype(np.int64):
                j = col_of[int(gid)]
                ex = X17["exist"][:, j]
                if not ex.any():
                    continue
                k_anc = int(np.flatnonzero(ex)[-1]); t_anc = float(t17[k_anc])
                lb = t_anc - t17
                xr = (Q17["xray"][:, j] > 0.5) & ex
                ecum = np.cumsum((dtw17 * xr.astype(float))[::-1])[::-1]
                ecum = np.where(ex, ecum - ecum[k_anc] + dtw17[k_anc] * xr[k_anc], np.nan)
                for q, _, _ in P17_ROWS:
                    y = ecum if q == "excum" else np.where(ex, Q17[q][:, j], np.nan)
                    out = np.full(len(TL17), np.nan)
                    for i, L in enumerate(TL17):
                        d = np.abs(lb - L); d[~ex] = np.inf
                        k = int(np.argmin(d))
                        if d[k] <= P17_DT:
                            out[i] = y[k]
                    TRK[q][int(gid)] = out
                ANC[int(gid)] = dict(e_x=float(np.sum(dtw17[xr])), f_x=float(xr.sum() / max(1, int(ex.sum()))),
                                     xray_now=float(xr[k_anc]), gate_now=float(X17["fgas"][k_anc, j] < XRAY_FGAS_MAX),
                                     t_quiet=float(t_anc - t17[xr][-1]) if xr.any() else np.nan,
                                     n_x=int(xr.sum()), t_span=float(t_anc - t17[ex][0]),
                                     lfgas_now=float(np.log10(max(X17["fgas"][k_anc, j], FGAS_FLOOR))),
                                     zgas_anc=float(zg17[k_anc, j]) if np.isfinite(zg17[k_anc, j]) else np.nan)
        A = pd.DataFrame.from_dict(ANC, orient="index"); A.index.name = "gal_id"
        B = B.merge(A.reset_index(), on="gal_id", how="left")
        with np.errstate(all="ignore"):                                   # the metal budget needs zgas, which only the branch carries
            _met = B["zgas_anc"].to_numpy(float) * B["mgas"].to_numpy(float)
            _tot = _met + B["mdust"].to_numpy(float)
            B["dtm"] = np.log10(np.where(_tot > 0, B["mdust"].to_numpy(float) / np.where(_tot > 0, _tot, np.nan), np.nan))
        print(f"  the clock is lookback from the snapshot, so nothing here depends on t_SFT / t_QT; the anchor summaries "
              f"(E_x, f_x, t_quiet, X-ray now) run over the whole tracked branch (median {np.nanmedian(B['t_span']):.1f} Gyr).")

        # ── the two binnings: age, then the chosen dust variable INSIDE each age bin ──
        SPL = P17_SPLIT if P17_SPLIT in B.columns else "lfd"
        SPL_LAB = dict(P17_DUSTVARS).get(SPL, SPL)
        age = B["age"].to_numpy(float)
        AE = (np.asarray(P17_AGE_EDGES, float) if P17_AGE_EDGES is not None
              else np.quantile(age[np.isfinite(age)], np.linspace(0, 1, P17_NAGE + 1)))
        AE[0], AE[-1] = AE[0] - 1e-6, AE[-1] + 1e-6
        B["age_bin"] = np.clip(np.digitize(age, AE) - 1, 0, len(AE) - 2)
        sv = B[SPL].to_numpy(float)
        if P17_FD_EDGES is not None:
            FE = {a: np.asarray(P17_FD_EDGES, float) for a in range(len(AE) - 1)}
        else:
            FE = {}
            for a in range(len(AE) - 1):
                v = sv[(B["age_bin"].to_numpy(int) == a) & np.isfinite(sv)]
                FE[a] = np.quantile(v, np.linspace(0, 1, P17_NFD + 1)) if len(v) >= P17_NFD else np.array([-np.inf, np.inf])
        fb = np.full(len(B), -1, int)
        for a in range(len(AE) - 1):
            m = B["age_bin"].to_numpy(int) == a
            e = FE[a].copy(); e[0], e[-1] = e[0] - 1e-6, e[-1] + 1e-6
            fb[m] = np.clip(np.digitize(sv[m], e) - 1, 0, len(e) - 2)
        B["fd_bin"] = fb
        NFD = max(1, max(len(FE[a]) - 1 for a in FE))
        CM = plt.get_cmap(P17_CMAP)
        FCOL = {k: CM(0.08 + 0.84 * k / max(1, NFD - 1)) for k in range(NFD)}
        print(f"  the colour split is {SPL} ({SPL_LAB}); age bins [Gyr]: "
              + ", ".join(f"{AE[i]:.2f}-{AE[i+1]:.2f} (n {int((B['age_bin'] == i).sum())})" for i in range(len(AE) - 1)))
        for a in range(len(AE) - 1):
            print(f"    age bin {a}: {SPL} edges " + ", ".join(f"{e:.2f}" for e in FE[a])
                  + " -> n " + ", ".join(str(int(((B['age_bin'] == a) & (B['fd_bin'] == k)).sum())) for k in range(len(FE[a]) - 1)))

        # ── the figure ──
        NA = len(AE) - 1
        fig, axs = plt.subplots(len(P17_ROWS), NA, figsize=(4.6 * NA, 2.35 * len(P17_ROWS)), squeeze=False,
                                constrained_layout=True, sharex=True)
        for ci in range(NA):
            for ri, (q, ylab, kind) in enumerate(P17_ROWS):
                ax = axs[ri, ci]
                for k in range(len(FE[ci]) - 1):
                    Sg = B[(B["age_bin"] == ci) & (B["fd_bin"] == k)]
                    gids = [int(v) for v in Sg["gal_id"]]
                    cent, lo, hi, n = _agg17([TRK[q][g_] for g_ in gids if g_ in TRK[q]], kind)
                    if not np.isfinite(cent).any():
                        continue
                    ax.fill_between(TL17, lo, hi, color=FCOL[k], alpha=0.18, lw=0, zorder=2)
                    ax.plot(TL17, cent, "-", color=FCOL[k], lw=2.2, zorder=3,
                            label=(f"{FE[ci][k]:.2f} to {FE[ci][k+1]:.2f}  ({len(gids)})" if ri == 0 else None))
                    for i, L in enumerate(TL17):
                        ALL17["tracks"].append(dict(box=box, snap=snap, split=SPL, age_bin=ci, age_lo=AE[ci], age_hi=AE[ci+1],
                                                    fd_bin=k, fd_lo=FE[ci][k], fd_hi=FE[ci][k+1], lookback=L, q=q,
                                                    value=cent[i], lo=lo[i], hi=hi[i], n=int(n[i])))
                if kind == "frac":
                    ax.set_ylim(-0.03, 1.03)
                ax.grid(alpha=0.22, lw=0.5); ax.set_xlim(TL17[0], TL17[-1]); ax.invert_xaxis()
                if ci == 0:
                    ax.set_ylabel(ylab, fontsize=9)
                if ri == 0:
                    ax.set_title(f"age {AE[ci]:.2f}-{AE[ci+1]:.2f} Gyr   (n = {int((B['age_bin'] == ci).sum())})", fontsize=10)
                    ax.legend(frameon=False, fontsize=7.6, loc="upper left", title=SPL_LAB, title_fontsize=7.6)
                if ri == len(P17_ROWS) - 1:
                    ax.set_xlabel(r"lookback from the snapshot   [Gyr]", fontsize=10)
        fig.suptitle(rf"one snapshot, no route and no quenching clock: {BOX_LABEL.get(box, box)} snapshot {snap} "
                     rf"($z = {z17:.2f}$), {len(B)} quenched galaxies, each once"
                     + ("   —   THE X-RAY CHANNEL WAS OFF IN THIS RUN (would-be states)" if nox else "")
                     + "\n" + rf"binned by stellar age (columns), and inside each age bin by the OBSERVED {SPL_LAB} "
                              rf"(colour); bands are bootstrap 16-84", fontsize=11.5)
        paper_save(fig, f"agn_states_snapshot_agnhist_{box}_snap{snap:03d}_{SPL}")
        plt.show()

        # ── the snapshot table and the unbinned control, for ALL THREE dust variables ──
        print(f"  AT THE SNAPSHOT (the question 'are the dusty ones still being dosed?'):")
        print(f"    {'age bin':>14s} {SPL + ' bin':>16s} {'N':>4s} {'age':>5s} {'split':>6s} {'logM*':>6s} {'logMBH':>7s} "
              f"{'X-ray NOW':>10s} {'gate now':>9s} {'never fired':>12s} {'E_x':>6s} {'t_quiet':>8s}")
        for a in range(NA):
            for k in range(len(FE[a]) - 1):
                Sg = B[(B["age_bin"] == a) & (B["fd_bin"] == k)]
                if not len(Sg):
                    continue
                r = dict(box=box, snap=snap, z=z17, split=SPL, age_bin=a, age_lo=AE[a], age_hi=AE[a+1], fd_bin=k,
                         fd_lo=FE[a][k], fd_hi=FE[a][k+1], n=len(Sg), age_med=float(np.nanmedian(Sg["age"])),
                         split_med=float(np.nanmedian(Sg[SPL])), lfd_med=float(np.nanmedian(Sg["lfd"])),
                         dtm_med=float(np.nanmedian(Sg["dtm"])), dh2_med=float(np.nanmedian(Sg["dh2"])),
                         log_mstar_med=float(np.nanmedian(Sg["log_mstar"])), log_mbh_med=float(np.nanmedian(Sg["log_mbh"])),
                         xray_now=float(np.nanmean(Sg["xray_now"])), gate_now=float(np.nanmean(Sg["gate_now"])),
                         never=float(np.nanmean(Sg["n_x"].to_numpy(float) == 0)), e_x_med=float(np.nanmedian(Sg["e_x"])),
                         f_x_med=float(np.nanmedian(Sg["f_x"])), t_quiet_med=float(np.nanmedian(Sg["t_quiet"])))
                ALL17["cells"].append(r)
                print(f"    {AE[a]:6.2f}-{AE[a+1]:5.2f} {FE[a][k]:7.2f} to {FE[a][k+1]:5.2f} {r['n']:4d} "
                      f"{r['age_med']:5.2f} {r['split_med']:6.2f} {r['log_mstar_med']:6.2f} {r['log_mbh_med']:7.2f} "
                      f"{100*r['xray_now']:9.0f}% {100*r['gate_now']:8.0f}% {100*r['never']:11.0f}% "
                      f"{r['e_x_med']:6.2f} {r['t_quiet_med']:8.2f}")
        print(f"  THE UNBINNED CONTROL — Spearman rho between each CONTINUOUS dust variable and each AGN quantity, INSIDE each "
              f"age bin (no dust bins at all, so no edge can make it; * = p < 0.05):")
        print(f"    {'age bin':>14s} {'dust var':>10s} {'N':>4s} " + " ".join(f"{lab[:13]:>14s}" for _, lab in P17_TESTS))
        for a in range(NA):
            Sg = B[B["age_bin"] == a]
            for dv, _dvlab in P17_DUSTVARS:
                if dv not in Sg.columns:
                    continue
                line = f"    {AE[a]:6.2f}-{AE[a+1]:5.2f} {dv:>10s} {len(Sg):4d} "
                for qq, lab in P17_TESTS:
                    x, y = Sg[dv].to_numpy(float), Sg[qq].to_numpy(float) if qq in Sg.columns else np.full(len(Sg), np.nan)
                    m = np.isfinite(x) & np.isfinite(y)
                    if m.sum() < 8 or np.nanstd(y[m]) == 0:
                        line += f"{'n/a':>14s}"; rho = p = np.nan
                    else:
                        rho, p = spearmanr(x[m], y[m])
                        line += f"{rho:+7.2f}{'*' if p < 0.05 else ' ':1s}{f'({p:.0e})' if p < 0.05 else '       ':>6s}"
                    ALL17["rho"].append(dict(box=box, snap=snap, age_bin=a, age_lo=AE[a], age_hi=AE[a+1], dust_var=dv,
                                             prop=qq, n=int(m.sum()), rho=float(rho) if np.isfinite(rho) else np.nan,
                                             p=float(p) if np.isfinite(p) else np.nan))
                print(line)
        B["box"], B["split_var"] = box, SPL
        ALL17["gal"].append(B)

    for _b17 in _boxes17:
        p17_run(_b17)
    if ALL17["cells"]:
        _st = f"agn_states_snapshot_agnhist_{P17_SPLIT}"
        pd.DataFrame(ALL17["tracks"]).to_csv(os.path.join(OUT, _st + "_tracks.csv"), index=False)
        pd.DataFrame(ALL17["cells"]).to_csv(os.path.join(OUT, _st + "_cells.csv"), index=False)
        pd.DataFrame(ALL17["rho"]).to_csv(os.path.join(OUT, _st + "_rho.csv"), index=False)
        P17G = pd.concat(ALL17["gal"], ignore_index=True)
        P17G.to_csv(os.path.join(OUT, _st + "_galaxies.csv"), index=False)
        print(f"\ntracks / cells / rho / galaxies -> {os.path.join(OUT, _st)}_{{tracks,cells,rho,galaxies}}.csv "
              f"(P17G is left in memory for Part 18)")
    print("READ IT LIKE THIS: every galaxy is drawn once, at one redshift, and the x-axis is lookback from that snapshot — no "
          "SFT, no QT, no route, no exposure threshold. Inside an age column the galaxies differ ONLY in the dust an observer "
          "would measure. If the coloured lines lie on top of each other, the AGN history does not know about the dust at fixed "
          "age and the Part 2a routes are our binning. The rho block repeats it with the dust CONTINUOUS and for all three dust "
          "variables: log M_dust/M* is partly a statement about how much gas is left (watch it against log fgas), while "
          "dust-to-metal is the grain-survival version. And cis50nox is the control that cannot be argued with: the X-ray "
          "channel never fired there.")

## Part 18 — matched pairs: at the same age, mass, gas and metallicity, what actually differs between a dusty and a dust-poor QG?

Part 17 says the AGN history barely knows about the dust above ~2.4 Gyr, so the difference has to be somewhere else. This part stops classifying and starts **controlling**. Every *dusty* galaxy (top `P18_QTOP` of the chosen dust variable) is paired with the *dust-poor* one (bottom `P18_QBOT`) nearest to it in **age, $\log M_\star$, $\log M_{\rm gas}/M_\star$ and $\log Z_{\rm gas}$** — the four things that trivially set a dust mass — and the pair is then scored on three competing explanations:

- **(A) destruction** — the grains were sputtered away: the dusty member should have spent *less* time gas-poor and X-ray exposed ($T_{\rm gas-poor}$, $E_x$, $f_x$, $t_{\rm quiet}$, time since the gate shut).
- **(B) resupply** — it was refilled: $M_{\rm gas}$, $M_{\rm H_2}$ and $M_\star$ rose over the last 0.5–1 Gyr, or it is past a recent gas peak.
- **(C) noise** — it is not physics at all: the "dusty" galaxies are the ones whose dust is carried by fewest particles. `lngas`, `lnstar` and the **gas particle mass** (a pure resolution variable with no dust in it) test that directly, and re-running with `P18_BOX = "cis25"` tests it at 8× better mass resolution.
- **(D) recency** is printed separately and is **not** a hypothesis test: the dust level and its recent change are two views of the same number, so those two rows are semi-circular by construction. They are there to *date* the difference — a large positive $\Delta\log M_{\rm dust}$ says the dusty member **grew** its dust inside the window rather than having held it since quenching.

**The matching is auditable.** The right-hand panel shows the standardised mean difference of every matching axis before and after pairing (|SMD| < 0.1 is a good match); `P18_MAXD` caps the total distance and `P18_CALIPER` caps each axis separately — the caliper is what keeps $Z_{\rm gas}$ honest, since dust-to-metal has $Z$ in its denominator. `P18_POOL` decides where pairing happens: inside one anchor (identical redshift, the cleanest) or across all anchors with $z$ added to the matching axes — `"auto"` picks the second when the anchors hold fewer than `P18_MIN_PER_SNAP` galaxies, which is the case for every box except m100n1024. If fewer than 20 pairs survive, the cell says so and the scorecard should not be read.

Everything is measured on lookback from the anchor, so **no quantity here depends on $t_{\rm SFT}$ or $t_{\rm QT}$**. The caveat of Part 17 carries over: the histories cache covers only the boxes notebook's own quenched pool, so the dust-poor member of a pair is the dust-poor end *of that pool*.

Output: `agn_states_matched_pairs_<box>_<split>.{png,pdf}` (the forest plot of AUC per property, coloured by block, with the median paired difference beside it, and the matching-balance panel), `_scorecard.csv` (per property: $N$ pairs, median paired difference, Wilcoxon $p$, AUC, the bootstrapped fraction of pairs with a positive difference) and `_galaxies.csv` (both members of every pair, tagged `role`).

In [ ]:
# ── Part 18 — MATCHED PAIRS: at the same age, mass, gas and metallicity, what actually differs between a dusty and a dust-poor QG? ──
# Part 17 says the AGN history barely knows about the dust above ~2.4 Gyr, so the difference has to be somewhere else. This part
# stops classifying and starts controlling: inside one anchor, every DUSTY galaxy (top P18_QTOP of the chosen dust variable) is
# paired with the DUST-POOR galaxy (bottom P18_QBOT) nearest to it in age, log M*, log M_gas/M* and log Z_gas — the four things
# that trivially set a dust mass — and then three competing explanations are scored on the pairs:
#   (A) DESTRUCTION — the grains were sputtered away: the galaxy sat gas-poor / X-ray exposed for longer.
#   (B) RESUPPLY    — the dusty one got fresh gas and metals recently: M_gas, M_H2, M_dust rose in the last Gyr.
#   (C) NOISE       — it is not physics: the dust mass is carried by a handful of particles (ngas, nstar at the anchor), so the
#                     'dusty' galaxies are the ones whose dust is least well sampled. Run P18_BOX = 'cis25' to see it at 8x mass
#                     resolution: an effect that survives there and dies in cis100 is resolution, and vice versa.
# Every quantity is measured on the SAME clock (lookback from the anchor) and needs no SFT / QT.
# CAVEAT the cell prints: the histories cache only covers the boxes notebook's own quenched pool (its DUST_TO_H2_MIN), so the
# dust-poor member of each pair is the dust-poor end OF THAT POOL, not of the catalogue.
P18_BOX     = "cis25"     # one box; rerun with "cis25" for the resolution arm of (C), "cis50nox" for the channel-off arm
P18_SNAP    = None         # None = every anchor of the box, matched WITHIN each anchor and then pooled (a galaxy can recur across anchors:
                           # the printed count says how many unique branches the pairs come from); an int restricts to one snapshot
P18_SPLIT   = "dtm"        # what "dusty" means here: "dtm" (dust-to-metal, the grain-survival variable) | "dh2" | "lfd"
P18_QBOT, P18_QTOP = 0.33, 0.67     # the bottom / top quantile of P18_SPLIT inside the anchor
P18_MATCH   = ["age", "log_mstar", "lfgas", "lzgas"]      # matched on these, standardised inside the anchor
P18_MAXD    = 0.75         # the largest standardised distance allowed inside a pair (in units of the anchor's own sigma)
P18_CALIPER = 0.5          # ... and no single matching variable may differ by more than this many sigma (the caliper that keeps
                           # the metallicity honest: dust-to-metal has Z in its denominator, so a sloppy Z match would fake the result)
P18_WIN     = (0.5, 1.0)   # the resupply windows [Gyr] before the anchor
P18_POOL    = "auto"       # where the pairing happens: "snap" = inside one anchor (cleanest: identical redshift) | "box" = all
                           # anchors together, with z ADDED to the matching axes | "auto" = per anchor if the anchors are rich
                           # enough (P18_MIN_PER_SNAP), pooled otherwise — the small boxes have ~15 quenched galaxies per anchor
P18_MIN_PER_SNAP = 40
P18_NBOOT   = 500
P18_BLOCKS  = {"A destruction": [("t_gaspoor", "T gas-poor [Gyr]"), ("e_x", "E_x [Gyr]"), ("f_x", "duty cycle f_x"),
                                 ("t_quiet", "t since X-ray [Gyr]"), ("t_gate", "t since gate shut [Gyr]"), ("lfgas_now", "log fgas now")],
               "B resupply":   [("dlmgas_0.5", "dlog Mgas 0.5 Gyr"), ("dlmgas_1.0", "dlog Mgas 1 Gyr"), ("dlmh2_1.0", "dlog MH2 1 Gyr"),
                                ("dlmstar_1.0", "dlog M* 1 Gyr"), ("gas_rise", "gas peak - now [dex]")],
               "C noise":      [("lngas", "log N gas part."), ("lnstar", "log N star part."), ("lmgas_part", "log gas particle mass")],
               # NOT a hypothesis test - the dust level and its recent change are two views of the same number, so these are
               # SEMI-CIRCULAR by construction. They are here to date the difference: a large positive dlog Mdust says the dusty
               # member GREW its dust inside the window rather than having held it since quenching.
               "D recency (semi-circular)": [("dlmdust_0.5", "dlog Mdust 0.5 Gyr"), ("dlmdust_1.0", "dlog Mdust 1 Gyr")]}

from scipy.stats import wilcoxon

if "CAT" not in globals() or not os.path.exists(HIST_FILE.format(box=P18_BOX)):
    print(f"Part 18: needs Part 0's catalogue cache and the {P18_BOX} histories — run Part 0 first.")
else:
    # ── the frame: the quenched rule at every anchor of the box, the four matching axes and the three blocks ──
    _c18 = CAT[CAT["box"] == P18_BOX].copy()
    _ms18, _sfr18 = _c18["mstar"].to_numpy(float), _c18["sfr"].to_numpy(float)
    with np.errstate(all="ignore"):
        _ss18 = np.where(_ms18 > 0, _sfr18 / _ms18, np.nan)
        _lm18 = np.log10(np.where(_ms18 > 0, _ms18, np.nan))
    _q18 = ((_lm18 > MASS_FLOOR) & (_ss18 < PASSIVE_FACTOR / (_c18["t_H_gyr"].to_numpy(float) * 1e9))
            & (_c18["ngas"].to_numpy(float) >= NGAS_MIN) & (_c18["nstar"].to_numpy(float) >= NSTAR_MIN)
            & (_c18["mdust"].to_numpy(float) >= DUST_TO_H2_MIN * _c18["mh2"].to_numpy(float)) & (_lm18 > MASS_MIN))
    D18 = _c18[_q18].copy()
    if P18_SNAP is not None:
        D18 = D18[D18["snap"] == int(P18_SNAP)]
    with np.errstate(all="ignore"):
        _m = np.where(D18["mstar"].to_numpy(float) > 0, D18["mstar"].to_numpy(float), np.nan)
        D18["log_mstar"] = np.log10(_m)
        _fd = D18["mdust"].to_numpy(float) / _m
        D18["lfd"] = np.where(_fd > 0, np.log10(np.where(_fd > 0, _fd, 1.0)), P9_FLOOR)
        D18["lfgas"] = np.log10(np.where(D18["mgas"].to_numpy(float) > 0, D18["mgas"].to_numpy(float) / _m, np.nan))
        D18["dh2"] = np.log10(np.where((D18["mh2"].to_numpy(float) > 0) & (D18["mdust"].to_numpy(float) > 0),
                                       D18["mdust"].to_numpy(float) / D18["mh2"].to_numpy(float), np.nan))
        D18["lngas"], D18["lnstar"] = np.log10(D18["ngas"].to_numpy(float)), np.log10(D18["nstar"].to_numpy(float))
        D18["lmgas_part"] = np.log10(np.where(D18["mgas"].to_numpy(float) > 0,                       # a PURE resolution variable:
                                              D18["mgas"].to_numpy(float) / np.maximum(D18["ngas"].to_numpy(float), 1), np.nan))   # no dust in it

    # ── the history-derived quantities of the three blocks, one pass per anchor ──
    rows18 = []
    with h5py.File(HIST_FILE.format(box=P18_BOX), "r") as f:
        for grp in sorted(f):
            a = int(grp.replace("snap", ""))
            sub = D18[D18["snap"] == a]
            if not len(sub):
                continue
            g = f[grp]
            t = g["t_yr"][:] / 1e9; o = np.argsort(t); t = t[o]
            ms, mg, mb, fe = (g[c][:][o] for c in ("mstar", "mgas", "mbh", "fedd"))
            md = g["mdust"][:][o] if "mdust" in g else np.full_like(ms, np.nan)
            mh = g["mh2"][:][o] if "mh2" in g else np.full_like(ms, np.nan)
            zg = g["zgas"][:][o] if "zgas" in g else np.full_like(ms, np.nan)
            ids = np.asarray(g["gal_ids"][:], dtype=np.int64)
            col = {int(v): j for j, v in enumerate(ids)}
            S, _, X = agn_states(ms, mg, mb, fe)
            dtw = np.gradient(t)
            gate = X["fgas"] < XRAY_FGAS_MAX
            for gid in sub["gal_id"].astype(np.int64):
                j = col.get(int(gid))
                if j is None:
                    continue
                ex = X["exist"][:, j]
                if ex.sum() < 3:
                    continue
                k = int(np.flatnonzero(ex)[-1]); t_anc = float(t[k])
                xr = (np.nan_to_num(S["xray"][:, j]) > 0.5) & ex
                gt = gate[:, j] & ex
                r = dict(gal_id=int(gid), snap=a, lzgas=np.log10(zg[k, j]) if zg[k, j] > 0 else np.nan,
                         t_gaspoor=float(np.sum(dtw[gt])), e_x=float(np.sum(dtw[xr])),
                         f_x=float(xr.sum() / max(1, int(ex.sum()))),
                         t_quiet=float(t_anc - t[xr][-1]) if xr.any() else np.nan,
                         lfgas_now=float(np.log10(max(X["fgas"][k, j], FGAS_FLOOR))),
                         t_span=float(t_anc - t[ex][0]), xray_now=float(xr[k]))
                _open = np.flatnonzero(ex & ~gt)                                  # the last epoch the gate was OPEN
                r["t_gate"] = float(t_anc - t[_open[-1]]) if len(_open) else r["t_span"]
                for W in P18_WIN:                                                 # the resupply block: log changes over the window
                    d = np.abs((t_anc - t) - W); d[~ex] = np.inf
                    k0 = int(np.argmin(d))
                    if d[k0] > 0.5 * W:
                        for q in ("mgas", "mh2", "mdust", "mstar"):
                            r[f"dl{q}_{W}"] = np.nan
                    else:
                        for q, arr in (("mgas", mg), ("mh2", mh), ("mdust", md), ("mstar", ms)):
                            a1, a0 = arr[k, j], arr[k0, j]
                            r[f"dl{q}_{W}"] = float(np.log10(a1 / a0)) if (np.isfinite(a1) and np.isfinite(a0) and a1 > 0 and a0 > 0) else np.nan
                w2 = ex & ((t_anc - t) <= 2.0)                                     # was there a gas peak in the last 2 Gyr?
                _gv = mg[w2, j]; _gv = _gv[np.isfinite(_gv) & (_gv > 0)]
                r["gas_rise"] = float(np.log10(np.max(_gv) / max(mg[k, j], 1e-30))) if (len(_gv) and mg[k, j] > 0) else np.nan
                rows18.append(r)
    H18 = pd.DataFrame(rows18)
    D18 = D18.merge(H18, on=["gal_id", "snap"], how="inner")
    with np.errstate(all="ignore"):
        _met = (10 ** D18["lzgas"].to_numpy(float)) * D18["mgas"].to_numpy(float)
        _tot = _met + D18["mdust"].to_numpy(float)
        D18["dtm"] = np.log10(np.where(_tot > 0, D18["mdust"].to_numpy(float) / np.where(_tot > 0, _tot, np.nan), np.nan))
    SPL18 = P18_SPLIT if P18_SPLIT in D18.columns else "lfd"
    print(f"Part 18 — MATCHED PAIRS in {BOX_LABEL.get(P18_BOX, P18_BOX)}"
          + (f", snapshot {P18_SNAP}" if P18_SNAP is not None else ", every anchor (matched WITHIN each, then pooled)")
          + f": {len(D18)} quenched galaxy-anchors with a main branch."
          + "\n  NOTE the histories cache covers only the boxes notebook's own quenched pool, so the 'dust-poor' member of a "
            "pair is the dust-poor end OF THAT POOL (its DUST_TO_H2_MIN floor), not of the catalogue.")

    # ── the pairing: dusty vs dust-poor at the same age, mass, gas and metallicity ──
    _per = D18.groupby("snap").size()
    _pool = (P18_POOL == "box") or (P18_POOL == "auto" and (not len(_per) or float(np.median(_per)) < P18_MIN_PER_SNAP))
    MATCH18 = list(P18_MATCH) + (["z"] if (_pool and "z" not in P18_MATCH and "z" in D18.columns) else [])
    print(f"  pairing {'across all anchors together, with z added to the matching axes' if _pool else 'inside each anchor'} "
          f"({'P18_POOL=' + str(P18_POOL)}; median {float(np.median(_per)) if len(_per) else 0:.0f} galaxies per anchor); "
          f"matched on {MATCH18}.")
    pairs, bal_rows = [], []
    for a, S in ([("all", D18)] if _pool else list(D18.groupby("snap"))):
        v = S[SPL18].to_numpy(float)
        ok = np.isfinite(v)
        if ok.sum() < 20:
            continue
        lo_c, hi_c = np.nanquantile(v, P18_QBOT), np.nanquantile(v, P18_QTOP)
        M = S[MATCH18].to_numpy(float)
        good = np.isfinite(M).all(axis=1) & ok
        mu, sd = np.nanmean(M[good], axis=0), np.nanstd(M[good], axis=0)
        sd = np.where(sd > 0, sd, 1.0)
        Z = (M - mu) / sd
        idx_d = np.flatnonzero(good & (v >= hi_c))
        idx_p = list(np.flatnonzero(good & (v <= lo_c)))
        for i in idx_d:
            if not idx_p:
                break
            dv = np.abs(Z[idx_p] - Z[i])
            d = np.sqrt(np.nansum(dv ** 2, axis=1))
            d = np.where((dv <= P18_CALIPER).all(axis=1), d, np.inf)          # the per-variable caliper
            k = int(np.argmin(d))
            if not np.isfinite(d[k]) or d[k] > P18_MAXD:
                continue
            jp = idx_p.pop(k)
            rd, rp = S.iloc[i], S.iloc[jp]
            pairs.append((rd, rp, float(d[k]), a))
        for nm, arr in (("before", (S.loc[good & (v >= hi_c), MATCH18].to_numpy(float),
                                    S.loc[good & (v <= lo_c), MATCH18].to_numpy(float))),):
            for ci, cname in enumerate(MATCH18):
                x1, x0 = arr[0][:, ci], arr[1][:, ci]
                bal_rows.append(dict(stage=nm, var=cname, smd=float((np.nanmean(x1) - np.nanmean(x0)) / sd[ci])))
    if not pairs:
        print("  no pair survived the matching distance — loosen P18_MAXD or widen the quantiles.")
    else:
        PD = pd.DataFrame([p[0] for p in pairs]).reset_index(drop=True)
        PP = pd.DataFrame([p[1] for p in pairs]).reset_index(drop=True)
        PD["pair_d"], PP["pair_d"] = [p[2] for p in pairs], [p[2] for p in pairs]
        for ci, cname in enumerate(MATCH18):
            x1, x0 = PD[cname].to_numpy(float), PP[cname].to_numpy(float)
            s_ = np.nanstd(np.concatenate([x1, x0])) or 1.0
            bal_rows.append(dict(stage="after", var=cname, smd=float((np.nanmean(x1) - np.nanmean(x0)) / s_)))
        BAL = pd.DataFrame(bal_rows).groupby(["stage", "var"], as_index=False)["smd"].mean()
        print(f"  {len(PD)} pairs from {PD['gal_id'].nunique()} distinct dusty galaxies over {PD['snap'].nunique()} anchors; "
              f"median pairing distance {np.median([p[2] for p in pairs]):.2f} sigma.")
        if len(PD) < 20:
            print(f"  ONLY {len(PD)} PAIRS — the scorecard below is not worth reading. The small boxes have few quenched "
                  f"galaxies per anchor and five matching axes: loosen P18_CALIPER ({P18_CALIPER:g}) / P18_MAXD ({P18_MAXD:g}), "
                  f"widen the quantiles (P18_QBOT/P18_QTOP), or drop a matching axis.")
        print(f"  the split: {SPL18} >= its {P18_QTOP:g} quantile (dusty, median {np.nanmedian(PD[SPL18]):.2f}) vs "
              f"<= {P18_QBOT:g} (dust-poor, median {np.nanmedian(PP[SPL18]):.2f}) — a gap of "
              f"{np.nanmedian(PD[SPL18]) - np.nanmedian(PP[SPL18]):+.2f} dex.")
        print(f"  the match (standardised mean difference, dusty - dust-poor; |SMD| < 0.1 is a good match):")
        for cname in MATCH18:
            b_ = BAL[(BAL["var"] == cname) & (BAL["stage"] == "before")]["smd"]
            a_ = BAL[(BAL["var"] == cname) & (BAL["stage"] == "after")]["smd"]
            print(f"    {cname:12s} before {float(b_.iloc[0]) if len(b_) else np.nan:+.2f}  ->  after "
                  f"{float(a_.iloc[0]) if len(a_) else np.nan:+.2f}")

        # ── the scorecard: which block separates the pair? ──
        rng18 = np.random.default_rng(18)
        sc = []
        print(f"\n  THE SCORECARD — paired difference (dusty - dust-poor), Wilcoxon signed-rank p, and the unpaired AUC "
              f"(0.5 = the two are indistinguishable):")
        for blk, props in P18_BLOCKS.items():
            print(f"    {blk}")
            for q, lab in props:
                if q not in PD.columns:
                    print(f"      {lab:24s} n/a (column missing)"); continue
                x1, x0 = PD[q].to_numpy(float), PP[q].to_numpy(float)
                m = np.isfinite(x1) & np.isfinite(x0)
                if m.sum() < 8:
                    print(f"      {lab:24s} n/a ({int(m.sum())} usable pairs)"); continue
                dif = x1[m] - x0[m]
                try:
                    w = float(wilcoxon(dif).pvalue) if np.nanstd(dif) > 0 else np.nan
                except ValueError:
                    w = np.nan
                auc, pmw, n1, n0 = mw_auc(np.concatenate([x1[m], x0[m]]), np.r_[np.ones(m.sum(), bool), np.zeros(m.sum(), bool)])
                bs = [np.mean(rng18.choice(dif, len(dif)) > 0) for _ in range(P18_NBOOT)]
                sc.append(dict(box=P18_BOX, block=blk, prop=q, label=lab, n=int(m.sum()), dmed=float(np.median(dif)),
                               p_wilcoxon=w, auc=auc, frac_pos=float(np.mean(dif > 0)),
                               frac_pos_lo=float(np.percentile(bs, 16)), frac_pos_hi=float(np.percentile(bs, 84))))
                print(f"      {lab:24s} n {m.sum():4d}   dmed {np.median(dif):+8.3f}   AUC {auc:5.2f}   "
                      f"p(Wilcoxon) {w:9.2e} {'*' if (np.isfinite(w) and w < 0.05) else ' '}")
        SC18 = pd.DataFrame(sc)

        # ── the figure: the scorecard as a forest plot, and the balance of the match ──
        fig, axs = plt.subplots(1, 2, figsize=(13.0, 0.42 * max(8, len(SC18)) + 1.6),
                                gridspec_kw=dict(width_ratios=[2.2, 1.0]), constrained_layout=True)
        BCOL = {"A destruction": "#d7191c", "B resupply": "#2c7bb6", "C noise": "#7b3294", "D recency (semi-circular)": "0.45"}
        ax = axs[0]
        yy = np.arange(len(SC18))[::-1]
        for y, r in zip(yy, SC18.itertuples()):
            c = BCOL.get(r.block, "0.4")
            ax.plot([r.auc], [y], "o", color=c, ms=7, zorder=3)
            ax.plot([0.5, r.auc], [y, y], "-", color=c, lw=2.0, alpha=0.55, zorder=2)
            ax.text(1.015, y, f"{r.dmed:+.3f}" + ("*" if (np.isfinite(r.p_wilcoxon) and r.p_wilcoxon < 0.05) else " "),
                    va="center", fontsize=8, color=c, transform=ax.get_yaxis_transform(), clip_on=False)
        ax.axvline(0.5, color="k", lw=1.3)
        ax.set_yticks(yy); ax.set_yticklabels([f"{r.block.split()[0]}  {r.label}" for r in SC18.itertuples()], fontsize=8.5)
        ax.set_xlabel("AUC — P(dusty > dust-poor) at matched age, mass, gas and metallicity", fontsize=10)
        ax.set_xlim(0.0, 1.0); ax.grid(alpha=0.22, lw=0.5, axis="x")
        ax.set_title(f"{BOX_LABEL.get(P18_BOX, P18_BOX)} — {len(PD)} matched pairs, split on {SPL18}", fontsize=10.5)
        ax.text(1.015, 1.015, "median\npaired diff", transform=ax.transAxes, fontsize=8, color="0.3", ha="left", va="bottom")
        ax = axs[1]
        for st, mk, c in (("before", "o", "0.55"), ("after", "D", "#1a9641")):
            sb = BAL[BAL["stage"] == st]
            ax.plot([sb[sb["var"] == v]["smd"].mean() for v in MATCH18], np.arange(len(MATCH18)), mk, color=c, ms=7, label=st)
        ax.axvline(0, color="k", lw=1.2)
        for xv in (-0.1, 0.1):
            ax.axvline(xv, color="0.6", lw=1.0, ls=":")
        ax.set_yticks(np.arange(len(MATCH18))); ax.set_yticklabels(MATCH18, fontsize=9)
        ax.set_xlabel("standardised mean difference", fontsize=10); ax.legend(frameon=False, fontsize=8.5)
        ax.set_title("did the matching work?", fontsize=10.5); ax.grid(alpha=0.22, lw=0.5, axis="x")
        fig.suptitle("Part 18 — what differs between a dusty and a dust-poor quenched galaxy when age, mass, gas and metals are held fixed",
                     fontsize=11.5)
        paper_save(fig, f"agn_states_matched_pairs_{P18_BOX}_{SPL18}")
        plt.show()
        SC18.to_csv(os.path.join(OUT, f"agn_states_matched_pairs_{P18_BOX}_{SPL18}_scorecard.csv"), index=False)
        pd.concat([PD.assign(role="dusty"), PP.assign(role="dust-poor")], ignore_index=True).to_csv(
            os.path.join(OUT, f"agn_states_matched_pairs_{P18_BOX}_{SPL18}_galaxies.csv"), index=False)
        print(f"\n  scorecard -> {os.path.join(OUT, f'agn_states_matched_pairs_{P18_BOX}_{SPL18}_scorecard.csv')}; "
              f"the paired galaxies beside it.")
        print("READ IT LIKE THIS: the pairs are matched on the four things that trivially set a dust mass, so a block that still "
              "separates them is a real difference. (A) large and positive means the dusty one was LESS exposed / less time "
              "gas-poor — destruction. (B) positive means it was refilled — resupply. (C) means the 'dusty' galaxies are simply "
              "the ones with fewest particles carrying the dust, i.e. the effect is resolution: rerun with P18_BOX = 'cis25' "
              "(8x better mass resolution) and see whether the same block survives.")

## Part 19 — thermal sputtering, measured particle by particle (m25)

Parts 17 and 18 can only ask *when* the AGN fired. SIMBA does not destroy dust with "an AGN state" — it destroys it by **thermal sputtering**, grain by grain, at a rate set by the local gas density and temperature (Tsai & Mathews 1995; the form implemented in Li, Narayanan & Davé 2019):

$$\tau_{\rm sp} \;=\; 0.17\ {\rm Gyr}\ \left(\frac{a}{0.1\,\mu{\rm m}}\right)\left(\frac{10^{-27}}{\rho_{\rm gas}}\right)\left[\left(\frac{T_0}{T}\right)^{\omega}+1\right],\qquad T_0 = 2\times10^{6}\,{\rm K},\ \ \omega = 2.5 .$$

The 25 Mpc box's **reduced particle files** carry `m_dust`, `temp` and — since the `nh` field was added to `build_reduced_particles_job.py` on 2026-09-02 — the gas density, for every gas particle inside a 100 kpc sphere. So $\tau_{\rm sp}$ can be evaluated **per particle** and collapsed onto the only aggregate that means anything:

$$\tau_{\rm eff} \;=\; \frac{M_{\rm dust}}{\sum_i m_{{\rm dust},i}/\tau_{{\rm sp},i}}$$

— the e-folding time of the galaxy's *whole* dust reservoir under the model's own destruction term. Not a proxy: this is the number SIMBA applies.

**Three questions, three panels.** (1) does the dust content follow $\tau_{\rm eff}$ at fixed age; (2) *where* the dust actually sits on the $(T, n_{\rm H})$ plane the law is written on, with the $\tau_{\rm sp}$ contours drawn on it and the points coloured by dust-to-metal; (3) does the **AGN exposure predict that environment at all** — $E_x$ against $\tau_{\rm eff}$. The printed tables give the same in Spearman form: dust variable vs sputtering variable inside each age bin, then AGN quantity vs sputtering quantity. If the dust follows $\tau_{\rm eff}$ while $\tau_{\rm eff}$ does not follow $E_x$, then the AGN decides *when* the gas turns hot and thin, but what survives is decided by the ISM phase structure at the time.

**Per galaxy it writes**: `tau_eff`, the dust-weighted median $\tau_{\rm sp}$, the dust-weighted $\log T$ and $\log n_{\rm H}$, the dust fraction with $\tau_{\rm sp} <$ `P19_TFAST`, the dust fractions above $10^5$ and $10^6$ K, the dust fraction in star-forming gas (where it *grows*), and the particle-sum dust and gas masses — with a printed cross-check of $M_{\rm dust}$(particles)$/M_{\rm dust}$(caesar), which is 1.0 when the selection is the member list.

**Knobs.** `P19_SEL` — `"member"` (the caesar galaxy list, so the dust mass matches the catalogue), `"aperture"` (everything in the file's 100 kpc sphere: ISM *and* CGM), or a radius in kpc. `P19_AGRAIN` (SIMBA follows a single grain size; $\tau_{\rm sp}$ is linear in it), `P19_TAU0` / `P19_T0` / `P19_OMEGA` (the law), `P19_TFAST`, `P19_NAGE`.

**m25 only, and on purpose**: it is the box with the particle products *and* the resolution — the thing being measured is the ISM phase structure the dust lives in.

**If no file carries `nh` yet** the cell prints the exact backfill command instead of running. `nh` is an additive field: re-submitting a plan appends *only* that dataset to the existing files — no geometry redo, one snapshot column read per snapshot:

```
sbatch --array=0-15 submit_reduced_particles.sh output/cis25/caesar_sfh/prof_kstracks/dust_profile_plan_kstracks.hdf5
sbatch --array=0-15 submit_reduced_particles.sh output/cis25/caesar_sfh/prof_boxcomp/dust_profile_plan_boxcomp.hdf5
```

Output: `agn_states_sputtering_cis25.{png,pdf}`, `_galaxies.csv` (the per-galaxy table — it is what feeds Part 18: add `tau_eff` / `f_dust_fast` to `P18_BLOCKS["A destruction"]` and merge this table into `D18` to turn block A from proxies into the measured destruction rate) and `_rho.csv`.

In [ ]:
# ── Part 19 — THE SPUTTERING MEASUREMENT (m25): the term of SIMBA's dust model that no catalogue part can see ──
# Parts 17-18 can only ask WHEN the AGN fired. SIMBA does not destroy dust with "an AGN state": it destroys it with THERMAL
# SPUTTERING, grain by grain, at a rate set by the local gas density and temperature (Tsai & Mathews 1995; Li, Narayanan &
# Dave 2019):        tau_sp = 0.17 Gyr (a / 0.1 um) (1e-27 / rho_gas) [ (T_0 / T)^omega + 1 ],  T_0 = 2e6 K, omega = 2.5.
# The reduced particle files of the 25 Mpc box carry m_dust, temp and (since 2026-09-02) nh for every gas particle inside a
# 100 kpc sphere, so tau_sp can be evaluated PARTICLE BY PARTICLE and collapsed onto the quantity that actually matters:
#     tau_eff = M_dust / sum_i (m_dust,i / tau_sp,i)      — the e-folding time of the galaxy's whole dust reservoir.
# That is the destruction rate the model applies. The question Part 18 could only proxy becomes measurable: is a dusty QG
# dusty because its dust sits where sputtering is slow, and does the AGN history predict that environment at all?
# m25 ONLY: it is the box with the particle products and the resolution (the ISM phase structure is what is being measured).
P19_BOX     = "cis25"
P19_PDIR    = os.path.join(os.getcwd(), "output", "cis25", "reduced_particles")   # build_reduced_particles_job.py products
P19_PREFIX  = "m25n512"       # the file prefix the job derives from the sim name
P19_SEL     = "member"        # which gas carries the dust: "member" = the caesar galaxy member list (so M_dust matches the
                              # catalogue) | "aperture" = every particle in the file's 100 kpc sphere (ISM + CGM) | a float = kpc
P19_AGRAIN  = 0.1             # grain radius [micron]: SIMBA follows a single grain size, and tau_sp is linear in it
P19_TAU0, P19_T0, P19_OMEGA = 0.17, 2.0e6, 2.5     # the sputtering law's coefficient [Gyr], T_0 [K] and exponent
P19_HOT     = (1.0e5, 1.0e6)  # the two thresholds the dust-in-hot-gas fractions are quoted at
P19_TFAST   = 1.0             # "fast" sputtering: tau_sp below this many Gyr
P19_NAGE    = 2               # age bins for the fixed-age correlations (m25 is small)
P19_NMIN_RHO = 8              # the smallest n a Spearman is printed for
P19_DUSTVARS = [("dtm", r"$\log$ dust-to-metal"), ("lfd", r"$\log M_{\rm dust}/M_\star$"), ("dh2", r"$\log M_{\rm dust}/M_{\rm H_2}$")]
MP_G, XH_C = 1.672622e-24, 0.76        # m_p [g] and the caesar hydrogen fraction behind nh (rho = nh m_p / XH)


def p19_tau_sp(T, nh):
    """The thermal-sputtering timescale [Gyr] of a 0.1-um grain in gas of temperature T [K] and hydrogen density nh [cm^-3]."""
    rho = np.asarray(nh, float) * MP_G / XH_C
    with np.errstate(all="ignore"):
        return P19_TAU0 * (P19_AGRAIN / 0.1) * (1e-27 / rho) * ((P19_T0 / np.asarray(T, float)) ** P19_OMEGA + 1.0)


if "CAT" not in globals():
    print("Part 19: needs Part 0's catalogue cache — run Part 0 first.")
elif not os.path.isdir(P19_PDIR):
    print(f"Part 19: {P19_PDIR} does not exist — the reduced particle files are built on the cluster by\n"
          f"  sbatch --array=0-15 submit_reduced_particles.sh output/cis25/caesar_sfh/prof_kstracks/dust_profile_plan_kstracks.hdf5")
else:
    # ── the sample: the m25 quenched galaxies of this notebook, and the files that exist for them ──
    B19 = (QM[QM["box"] == P19_BOX].copy() if (P19_BOX in set(QM["box"])) else pd.DataFrame())
    if not len(B19):
        print(f"Part 19: no {P19_BOX} galaxy in Part 0's sample — nothing to measure.")
    else:
        def _pfile(snap, gid):
            return os.path.join(P19_PDIR, f"snap_{int(snap):03d}", f"{P19_PREFIX}_snap{int(snap):03d}_gal{int(gid):06d}.h5")
        B19["pfile"] = [_pfile(s_, g_) for s_, g_ in zip(B19["snap"], B19["gal_id"])]
        B19["has_file"] = [os.path.exists(p) for p in B19["pfile"]]
        _n_nh = 0
        for p_ in B19.loc[B19["has_file"], "pfile"]:
            try:
                with h5py.File(p_, "r") as h:
                    _n_nh += int("nh" in h["gas"])
            except OSError:
                pass
        print(f"Part 19 — the sputtering measurement in {BOX_LABEL.get(P19_BOX, P19_BOX)}: {len(B19)} quenched galaxies in "
              f"Part 0's sample, {int(B19['has_file'].sum())} with a reduced particle file, {_n_nh} of those carrying `nh`.")
        if _n_nh == 0:
            print("  NO FILE CARRIES `nh` YET. It is a BACKFILL field (added to build_reduced_particles_job.py on 2026-09-02): "
                  "re-submitting a plan appends ONLY that dataset — no geometry redo, one snapshot column read per snapshot:\n"
                  "    sbatch --array=0-15 submit_reduced_particles.sh output/cis25/caesar_sfh/prof_kstracks/dust_profile_plan_kstracks.hdf5\n"
                  "    sbatch --array=0-15 submit_reduced_particles.sh output/cis25/caesar_sfh/prof_boxcomp/dust_profile_plan_boxcomp.hdf5\n"
                  "  (the second only covers snapshots 105 and 134). Then re-run this cell.")
        else:
            # ── the per-particle measurement, collapsed per galaxy ──
            rows19 = []
            for r in B19[B19["has_file"]].itertuples():
                try:
                    with h5py.File(r.pfile, "r") as h:
                        g = h["gas"]
                        if "nh" not in g:
                            continue
                        md, mg = g["m_dust"][:].astype(float), g["m_gas"][:].astype(float)
                        T, nh = g["temp"][:].astype(float), g["nh"][:].astype(float)
                        sfr = g["sfr"][:].astype(float) if "sfr" in g else np.zeros(len(md))
                        mem = g["member"][:].astype(bool) if "member" in g else np.ones(len(md), bool)
                        pos = g["pos"][:].astype(float) if "pos" in g else None
                except OSError:
                    continue
                if isinstance(P19_SEL, str) and P19_SEL == "member":
                    sel = mem
                elif isinstance(P19_SEL, str):
                    sel = np.ones(len(md), bool)
                else:
                    sel = (np.sqrt(np.sum(pos ** 2, axis=1)) <= float(P19_SEL)) if pos is not None else np.ones(len(md), bool)
                sel &= np.isfinite(md) & (md > 0) & np.isfinite(T) & (T > 0) & np.isfinite(nh) & (nh > 0)
                if sel.sum() < 5:
                    continue
                mdd, Td, nhd, sfd = md[sel], T[sel], nh[sel], sfr[sel]
                tau = p19_tau_sp(Td, nhd)
                M = float(mdd.sum())
                w = mdd / M
                o = np.argsort(tau); cw = np.cumsum(mdd[o])
                # THE SPLIT THAT DECIDES WHAT THE NUMBER MEANS: star-forming particles sit on SIMBA's effective EOS, where
                # `temp` is the sub-grid pressure floor and NOT a thermal temperature — tau_sp evaluated there is an artefact
                # of the EOS (and that gas is where dust GROWS, not where it is sputtered). The non-star-forming dust is the
                # only part of the reservoir whose sputtering time means what it says.
                _phase = {}
                for _lab, _m in (("sf", sfd > 0), ("nsf", sfd <= 0)):
                    if mdd[_m].sum() > 0:
                        _w = mdd[_m] / mdd[_m].sum()
                        _phase[f"f_dust_{_lab}"] = float(mdd[_m].sum() / M)
                        _phase[f"tau_eff_{_lab}"] = float(mdd[_m].sum() / np.sum(mdd[_m] / tau[_m]))
                        _phase[f"logT_dw_{_lab}"] = float(np.sum(_w * np.log10(Td[_m])))
                        _phase[f"lognh_dw_{_lab}"] = float(np.sum(_w * np.log10(nhd[_m])))
                    else:
                        for _k in ("f_dust", "tau_eff", "logT_dw", "lognh_dw"):
                            _phase[f"{_k}_{_lab}"] = np.nan if _k != "f_dust" else 0.0
                rec = dict(box=P19_BOX, snap=int(r.snap), gal_id=int(r.gal_id), n_part=int(sel.sum()), **_phase,
                           M_dust_p=M, M_gas_p=float(mg[sel].sum()),
                           # THE number: the e-folding time of the whole reservoir, sum_i m_i/tau_i
                           tau_eff=float(M / np.sum(mdd / tau)),
                           tau_med=float(tau[o][np.searchsorted(cw, 0.5 * cw[-1])]),
                           logT_dw=float(np.sum(w * np.log10(Td))), lognh_dw=float(np.sum(w * np.log10(nhd))),
                           f_dust_fast=float(mdd[tau < P19_TFAST].sum() / M),
                           f_dust_hot5=float(mdd[Td > P19_HOT[0]].sum() / M),
                           f_dust_hot6=float(mdd[Td > P19_HOT[1]].sum() / M),
                           f_dust_member=float(mdd[mem[sel]].sum() / M) if mem is not None else np.nan)
                rows19.append(rec)
            P19T = pd.DataFrame(rows19)
            if not len(P19T):
                print("  no galaxy produced a measurement (files present but empty of dust-bearing gas).")
            else:
                P19 = B19.merge(P19T, on=["box", "snap", "gal_id"], how="inner")
                with np.errstate(all="ignore"):
                    _fdp = P19["M_dust_p"].to_numpy(float) / np.maximum(P19["mdust"].to_numpy(float), 1e-30)
                print(f"  measured {len(P19)} galaxies over {P19['snap'].nunique()} anchors, selection = {P19_SEL!r}; "
                      f"median {int(np.median(P19['n_part']))} dust-bearing particles each.")
                print(f"  cross-check against the catalogue dust mass: median M_dust(particles)/M_dust(caesar) = "
                      f"{np.nanmedian(_fdp):.2f} (1.0 = the same gas; the caesar member list is the like-for-like selection).")
                print(f"  the reservoir: median tau_eff = {np.nanmedian(P19['tau_eff']):.2f} Gyr, dust-weighted "
                      f"log T = {np.nanmedian(P19['logT_dw']):.2f}, log nh = {np.nanmedian(P19['lognh_dw']):.2f}; "
                      f"median dust fraction above 1e5 K = {np.nanmedian(P19['f_dust_hot5']):.3f}, "
                      f"above 1e6 K = {np.nanmedian(P19['f_dust_hot6']):.3f}, in star-forming gas = {np.nanmedian(P19['f_dust_sf']):.2f}.")
                print(f"  THE PHASE SPLIT (this is what decides whether tau means anything): the star-forming particles sit on "
                      f"SIMBA's effective EOS, where `temp` is the sub-grid pressure floor, not a thermal temperature — and that "
                      f"gas is where dust GROWS. Only the non-star-forming dust has a sputtering time that means what it says.")
                print(f"    star-forming dust:      median f = {np.nanmedian(P19['f_dust_sf']):.2f}, "
                      f"log T = {np.nanmedian(P19['logT_dw_sf']):.2f}, log nh = {np.nanmedian(P19['lognh_dw_sf']):+.2f}, "
                      f"tau_eff = {np.nanmedian(P19['tau_eff_sf']):.2f} Gyr   [EOS: read as a density ordering, not a temperature]")
                print(f"    NON-star-forming dust:  median f = {np.nanmedian(P19['f_dust_nsf']):.2f}, "
                      f"log T = {np.nanmedian(P19['logT_dw_nsf']):.2f}, log nh = {np.nanmedian(P19['lognh_dw_nsf']):+.2f}, "
                      f"tau_eff = {np.nanmedian(P19['tau_eff_nsf']):.2f} Gyr   [THE physical sputtering number]")

                # ── the dust variables and the AGN summaries, straight from this box's histories (no dependence on Part 17) ──
                with np.errstate(all="ignore"):
                    P19["dh2"] = np.log10(np.where((P19["mh2"].to_numpy(float) > 0) & (P19["mdust"].to_numpy(float) > 0),
                                                   P19["mdust"].to_numpy(float) / P19["mh2"].to_numpy(float), np.nan))
                AGN19 = [("e_x", "E_x [Gyr]"), ("t_quiet", "t_quiet [Gyr]"), ("f_x", "duty f_x"), ("xray_now", "X-ray now")]
                _anc19 = {}
                _hp19 = HIST_FILE.format(box=P19_BOX)
                if os.path.exists(_hp19):
                    with h5py.File(_hp19, "r") as _f19:
                        for _a19 in sorted(set(P19["snap"].astype(int))):
                            _k19 = f"snap{int(_a19):03d}"
                            if _k19 not in _f19:
                                continue
                            _g19 = _f19[_k19]
                            _t19 = _g19["t_yr"][:] / 1e9; _o19 = np.argsort(_t19); _t19 = _t19[_o19]
                            _ms19, _mg19, _mb19, _fe19 = (_g19[c][:][_o19] for c in ("mstar", "mgas", "mbh", "fedd"))
                            _zg19 = _g19["zgas"][:][_o19] if "zgas" in _g19 else np.full_like(_ms19, np.nan)
                            _S19, _, _X19 = agn_states(_ms19, _mg19, _mb19, _fe19)
                            _dt19 = np.gradient(_t19)
                            for _j19, _gid19 in enumerate(np.asarray(_g19["gal_ids"][:], dtype=np.int64)):
                                _ex19 = _X19["exist"][:, _j19]
                                if not _ex19.any():
                                    continue
                                _k = int(np.flatnonzero(_ex19)[-1])
                                _xr19 = (np.nan_to_num(_S19["xray"][:, _j19]) > 0.5) & _ex19
                                _anc19[(int(_a19), int(_gid19))] = dict(
                                    e_x=float(np.sum(_dt19[_xr19])), f_x=float(_xr19.sum() / max(1, int(_ex19.sum()))),
                                    xray_now=float(_xr19[_k]),
                                    t_quiet=float(_t19[_ex19][-1] - _t19[_xr19][-1]) if _xr19.any() else np.nan,
                                    zgas_anc=float(_zg19[_k, _j19]) if np.isfinite(_zg19[_k, _j19]) else np.nan)
                for _c19 in ["e_x", "f_x", "xray_now", "t_quiet", "zgas_anc"]:
                    P19[_c19] = [(_anc19.get((int(s_), int(g_)), {}) or {}).get(_c19, np.nan)
                                 for s_, g_ in zip(P19["snap"], P19["gal_id"])]
                with np.errstate(all="ignore"):                     # the metal budget: Z_gas M_gas + M_dust
                    _met19 = P19["zgas_anc"].to_numpy(float) * P19["mgas"].to_numpy(float)
                    _tot19 = _met19 + P19["mdust"].to_numpy(float)
                    P19["dtm"] = np.log10(np.where(_tot19 > 0, P19["mdust"].to_numpy(float) / np.where(_tot19 > 0, _tot19, np.nan), np.nan))
                SPUT19 = [("tau_eff", "tau_eff [Gyr]"), ("tau_eff_nsf", "tau_eff(non-SF)"), ("f_dust_fast", f"f_dust(tau<{P19_TFAST:g})"),
                          ("logT_dw", "log T (dust-w)"), ("lognh_dw", "log nh (dust-w)"), ("lognh_dw_nsf", "log nh (non-SF)"),
                          ("f_dust_hot5", "f_dust(T>1e5)"), ("f_dust_sf", "f_dust(SF gas)")]
                _age = P19["age"].to_numpy(float)
                AE19 = np.quantile(_age[np.isfinite(_age)], np.linspace(0, 1, P19_NAGE + 1))
                AE19[0], AE19[-1] = AE19[0] - 1e-6, AE19[-1] + 1e-6
                P19["age_bin"] = np.clip(np.digitize(_age, AE19) - 1, 0, len(AE19) - 2)
                print(f"\n  (1) DOES THE SPUTTERING ENVIRONMENT EXPLAIN THE DUST? Spearman rho inside each age bin "
                      f"(* = p < 0.05):")
                rho19 = []
                for a in range(len(AE19) - 1):
                    S = P19[P19["age_bin"] == a]
                    print(f"    age {AE19[a]:.2f}-{AE19[a+1]:.2f} Gyr  (n {len(S)})")
                    for dv, _ in P19_DUSTVARS:
                        if dv not in S.columns or not np.isfinite(S[dv]).sum():
                            continue
                        line = f"      {dv:>4s} vs "
                        for q, lab in SPUT19:
                            x, y = S[dv].to_numpy(float), S[q].to_numpy(float)
                            m = np.isfinite(x) & np.isfinite(y)
                            if m.sum() < P19_NMIN_RHO or np.nanstd(y[m]) == 0:
                                line += f"{lab.split()[0][:11]:>12s} n/a "; continue
                            rr, pp = spearmanr(x[m], y[m])
                            rho19.append(dict(box=P19_BOX, age_bin=a, dust_var=dv, prop=q, n=int(m.sum()), rho=float(rr), p=float(pp)))
                            line += f"{lab.split()[0][:11]:>12s} {rr:+.2f}{'*' if pp < 0.05 else ' '}"
                        print(line)
                print(f"\n  (2) DOES THE AGN HISTORY EXPLAIN THE ENVIRONMENT? the same rho, AGN quantity vs sputtering quantity, "
                      f"over all {len(P19)} galaxies (age partialled out by the bins above where n allows):")
                for q, lab in AGN19:
                    if q not in P19.columns:
                        continue
                    line = f"    {lab:>16s} vs "
                    for sq, slab in SPUT19:
                        x, y = P19[q].to_numpy(float), P19[sq].to_numpy(float)
                        m = np.isfinite(x) & np.isfinite(y)
                        if m.sum() < P19_NMIN_RHO or np.nanstd(y[m]) == 0 or np.nanstd(x[m]) == 0:
                            line += f"{slab.split()[0][:11]:>12s} n/a "; continue
                        rr, pp = spearmanr(x[m], y[m])
                        rho19.append(dict(box=P19_BOX, age_bin=-1, dust_var=q, prop=sq, n=int(m.sum()), rho=float(rr), p=float(pp)))
                        line += f"{slab.split()[0][:11]:>12s} {rr:+.2f}{'*' if pp < 0.05 else ' '}"
                    print(line)
                R19 = pd.DataFrame(rho19)

                # ── the figure ──
                fig, axs = plt.subplots(1, 3, figsize=(15.0, 4.3), constrained_layout=True)
                _dv = "dtm" if np.isfinite(P19["dtm"]).sum() > 8 else "lfd"
                _dvlab = dict(P19_DUSTVARS)[_dv]
                ax = axs[0]
                sc = ax.scatter(P19["tau_eff"], P19[_dv], c=P19["age"], s=40, cmap="viridis", lw=0.4, edgecolor="k", zorder=3)
                ax.set_xscale("log"); ax.set_xlabel(r"$\tau_{\rm eff}$ of the dust reservoir   [Gyr]", fontsize=10)
                ax.set_ylabel(_dvlab, fontsize=10); ax.grid(alpha=0.22, lw=0.5)
                plt.colorbar(sc, ax=ax, label="stellar age [Gyr]")
                _m = np.isfinite(P19["tau_eff"]) & np.isfinite(P19[_dv])
                if _m.sum() > 8:
                    _r, _p = spearmanr(np.log10(P19.loc[_m, "tau_eff"]), P19.loc[_m, _dv])
                    ax.set_title(rf"the dust vs its own destruction time   ($\rho = {_r:+.2f}$, $p$ = {_p:.1e})", fontsize=10)
                ax = axs[1]
                _Tv, _nv = 10 ** P19["logT_dw"].to_numpy(float), 10 ** P19["lognh_dw"].to_numpy(float)
                _Tv, _nv = _Tv[np.isfinite(_Tv)], _nv[np.isfinite(_nv)]
                _Tlo, _Thi = (np.nanmin(_Tv) / 5, np.nanmax(_Tv) * 5) if len(_Tv) else (1e3, 1e8)
                _nlo, _nhi = (np.nanmin(_nv) / 20, np.nanmax(_nv) * 20) if len(_nv) else (1e-5, 1e1)
                TT = np.logspace(np.log10(_Tlo), np.log10(_Thi), 200); NN = np.logspace(np.log10(_nlo), np.log10(_nhi), 200)
                TG, NG = np.meshgrid(TT, NN)
                CS = ax.contour(TG, NG, np.log10(p19_tau_sp(TG, NG)), levels=[-1, 0, 1, 2], colors="0.5", linewidths=1.0)
                ax.clabel(CS, fmt=lambda v: rf"$\tau_{{\rm sp}} = 10^{{{v:g}}}$ Gyr", fontsize=7.5)
                sc = ax.scatter(10 ** P19["logT_dw"], 10 ** P19["lognh_dw"], c=P19[_dv], s=40, cmap="plasma", lw=0.4,
                                edgecolor="k", zorder=3)
                ax.set_xscale("log"); ax.set_yscale("log")
                ax.set_xlim(_Tlo, _Thi); ax.set_ylim(_nlo, _nhi)
                ax.set_xlabel(r"$T$ of the dust-bearing gas (dust-weighted)   [K]", fontsize=10)
                ax.set_ylabel(r"$n_{\rm H}$ (dust-weighted)   [cm$^{-3}$]", fontsize=10)
                plt.colorbar(sc, ax=ax, label=_dvlab)
                ax.set_title("where the dust actually sits, on the sputtering plane", fontsize=10)
                ax.grid(alpha=0.22, lw=0.5)
                ax = axs[2]
                _x = "e_x" if "e_x" in P19.columns and np.isfinite(P19["e_x"]).sum() > 8 else None
                if _x:
                    sc = ax.scatter(P19[_x], P19["tau_eff"], c=P19[_dv], s=40, cmap="plasma", lw=0.4, edgecolor="k", zorder=3)
                    ax.set_yscale("log"); ax.set_xlabel(r"$E_x$ — the X-ray exposure of the whole branch   [Gyr]", fontsize=10)
                    ax.set_ylabel(r"$\tau_{\rm eff}$   [Gyr]", fontsize=10)
                    plt.colorbar(sc, ax=ax, label=_dvlab)
                    _m2 = np.isfinite(P19[_x]) & np.isfinite(P19["tau_eff"])
                    _r2, _p2 = spearmanr(P19.loc[_m2, _x], np.log10(P19.loc[_m2, "tau_eff"]))
                    ax.set_title(rf"does the dose set the environment?   ($\rho = {_r2:+.2f}$, $p$ = {_p2:.1e})", fontsize=10)
                else:
                    ax.text(0.5, 0.5, "no AGN summary in memory —\nrun Part 17 first (it leaves P17G)", ha="center", va="center",
                            transform=ax.transAxes, fontsize=10, color="0.4")
                ax.grid(alpha=0.22, lw=0.5)
                fig.suptitle(rf"Part 19 — thermal sputtering measured particle by particle in {BOX_LABEL.get(P19_BOX, P19_BOX)}: "
                             rf"$\tau_{{\rm sp}} = {P19_TAU0:g}\,$Gyr$\,(a/0.1\,\mu m)(10^{{-27}}/\rho)[(T_0/T)^{{{P19_OMEGA:g}}}+1]$, "
                             rf"{len(P19)} galaxies", fontsize=11.5)
                paper_save(fig, f"agn_states_sputtering_{P19_BOX}")
                plt.show()
                P19.to_csv(os.path.join(OUT, f"agn_states_sputtering_{P19_BOX}_galaxies.csv"), index=False)
                R19.to_csv(os.path.join(OUT, f"agn_states_sputtering_{P19_BOX}_rho.csv"), index=False)
                print(f"\n  per galaxy -> {os.path.join(OUT, f'agn_states_sputtering_{P19_BOX}_galaxies.csv')}  (feeds Part 18: add "
                      f"'tau_eff' / 'f_dust_fast' to P18_BLOCKS['A destruction'] and merge this table into D18)")
                print("READ IT LIKE THIS: tau_eff is the time SIMBA itself would take to sputter the galaxy's dust away at the "
                      "density and temperature the dust is actually sitting at — the destruction term of the model, measured "
                      "rather than proxied. Panel 1 asks whether the dust content follows it; panel 2 shows where the dust sits "
                      "on the (T, nh) plane the law is written on; panel 3 asks whether the AGN exposure predicts that "
                      "environment at all. If the dust follows tau_eff but tau_eff does not follow E_x, then the AGN sets when "
                      "the gas becomes hot and thin, but what survives is decided by the ISM phase structure at the time.")

## Part 20 — the dust budget along the track: sputtering **and** grain growth, at every critical epoch (m25)

Part 19 measured the destruction term *at the anchor* and found nothing being destroyed — no dust above $10^5$ K, and the non-star-forming dust sitting where $\tau_{\rm sp}\sim60$ Gyr. But that is the environment of the **survivors**. If sputtering ever mattered it mattered earlier: at the jet ignition, at the SFT, around the QT, at the gas minimum. So this part walks the same measurement along the KS-track stages (`ks_track_epochs.fits` gives the snapshot and the galaxy index at each) and adds the other half of SIMBA's budget, the **growth** term — grains accrete metals at a rate $\propto \rho Z$:

$$\tau_{\rm acc} \;=\; \tau_{\rm acc,0}\,\left(\frac{a}{0.1\,\mu{\rm m}}\right)\left(\frac{n_{\rm ref}}{n_{\rm H}}\right)\left(\frac{Z_\odot}{Z}\right)\left(\frac{T_{\rm ref}}{T}\right)^{1/2}$$

so at every epoch the question becomes **which of the two clocks is shorter** — and `f_dust_growing`, the dust fraction with $\tau_{\rm acc} < \tau_{\rm sp}$, answers it directly. The coefficient sets the scale, not the ordering; both clocks carry the same grain size, and the comparison is per particle.

$\tau_{\rm acc}$ needs the per-particle metallicity — the `zgas` field added to `build_reduced_particles_job.py` beside `nh`. If it is not in the files yet, the cell measures the sputtering half and prints the one-line backfill:

```
sbatch --array=0-15 submit_reduced_particles.sh output/cis25/caesar_sfh/prof_kstracks/dust_profile_plan_kstracks.hdf5
```

**The phase split matters here too**: star-forming particles sit on SIMBA's effective EOS, where `temp` is a pressure floor rather than a temperature, so `tau_sp_nsf` — the non-star-forming dust — is the row whose clock means what it says. A Hubble-time line is drawn on it: if it never comes down to that at any stage, sputtering never had a chance and the budget is decided by growth and by how much dense gas survives.

Tracks are split by the dust the galaxy **ends** with (`P20_SPLIT`, top vs bottom tercile at the anchor), so the panels read as "what did the galaxies that ended dusty do differently, and when". `P20_STAGES` sets the epochs and their order, `P20_MAXFILES` caps the read for a smoke test.

Output: `agn_states_dustbudget_track_cis25.{png,pdf}` and `agn_states_dustbudget_track_cis25.csv` (per galaxy and stage: both clocks, the hot-dust fractions, the dust-weighted $T$, $n_{\rm H}$, $Z$, and the particle-sum masses).

In [ ]:
# ── Part 20 — THE DUST BUDGET ALONG THE TRACK (m25): sputtering AND grain growth, measured at every critical epoch ──
# Part 19 measured the destruction term AT THE ANCHOR and found nothing being destroyed: no dust above 1e5 K, and the non-star-
# forming dust sitting where tau_sp ~ 60 Gyr. But that is the environment of the SURVIVORS. If sputtering ever mattered it
# mattered EARLIER — at the jet ignition, at the SFT, around the QT, at the gas minimum. This part therefore walks the same
# measurement along the KS-track stages (ks_track_epochs gives the snapshot and the galaxy index at each one) and adds the
# other half of SIMBA's dust budget, the GROWTH term: grains accrete metals at a rate ∝ rho * Z, so
#     tau_acc = TAU_ACC0 (a/0.1um) (n_ref/n_H) (Z_ref/Z) (T_ref/T)^(1/2)
# and the question at every epoch becomes which of the two clocks is shorter. tau_acc needs the per-particle metallicity, the
# `zgas` field added to build_reduced_particles_job.py on 2026-09-02 — the cell says so if it is not there yet.
# THE PHASE SPLIT MATTERS HERE TOO: star-forming particles sit on SIMBA's effective EOS, where `temp` is a pressure floor and
# not a temperature; their tau_sp is an EOS artefact, and it is the NON-star-forming dust whose sputtering time means anything.
P20_BOX     = "cis25"
P20_KSDIR   = os.path.join(os.getcwd(), "output", "cis25", "ks_tracks")
P20_PDIR    = os.path.join(os.getcwd(), "output", "cis25", "reduced_particles")
P20_PREFIX  = "m25n512"
P20_STAGES  = ["sf_peak", "agn_ign", "jet_on", "sft", "qt", "post_quench", "gas_min", "anchor"]   # in time order
P20_SEL     = "member"
P20_SPLIT   = "dh2"          # the anchor outcome the tracks are split on (top vs bottom tercile)
P20_ZSUN    = 0.0134         # Asplund+09 solar metal mass fraction, the reference of the growth law
P20_TAU_ACC0, P20_N_REF, P20_T_REF = 0.2, 100.0, 20.0    # tau_acc = TAU_ACC0 (n_ref/n)(Z_sun/Z)(T_ref/T)^0.5 [Gyr]
                             # THE COEFFICIENT SETS THE SCALE, NOT THE ORDERING: the comparison that matters below is
                             # tau_acc vs tau_sp per particle, and both scale with the same grain size.
P20_MAXFILES = int(os.environ.get("P20_MAXFILES", "0"))  # 0 = every stage row; a number caps it (a smoke test)

if "p19_tau_sp" not in globals():
    MP_G, XH_C = 1.672622e-24, 0.76
    def p19_tau_sp(T, nh, a=0.1, tau0=0.17, T0=2.0e6, om=2.5):
        with np.errstate(all="ignore"):
            return tau0 * (a / 0.1) * (1e-27 / (np.asarray(nh, float) * MP_G / XH_C)) * ((T0 / np.asarray(T, float)) ** om + 1.0)


def p20_tau_acc(T, nh, Z):
    """The grain-growth (metal accretion) timescale [Gyr]: tau_acc ∝ 1 / (n_H Z sqrt(T))."""
    with np.errstate(all="ignore"):
        return (P20_TAU_ACC0 * (P20_N_REF / np.asarray(nh, float)) * (P20_ZSUN / np.asarray(Z, float))
                * np.sqrt(P20_T_REF / np.asarray(T, float)))


_ep20 = os.path.join(P20_KSDIR, "ks_track_epochs.fits")
if "CAT" not in globals():
    print("Part 20: needs Part 0 — run it first.")
elif not (os.path.exists(_ep20) and os.path.isdir(P20_PDIR)):
    print(f"Part 20: needs {_ep20} (ks_tracks_quenched_m25.ipynb) and {P20_PDIR} (submit_reduced_particles.sh).")
else:
    E20 = Table.read(_ep20).to_pandas()
    for c in E20.columns:
        if E20[c].dtype == object:
            E20[c] = E20[c].map(lambda v: v.decode() if isinstance(v, bytes) else v)
    E20 = E20[(E20["pop"] == "Q") & E20["stage"].isin(P20_STAGES)].copy()
    if P20_MAXFILES:
        E20 = E20.groupby("stage", group_keys=False).head(max(4, P20_MAXFILES // max(1, len(P20_STAGES))))
    print(f"Part 20 — the dust budget along the track in {BOX_LABEL.get(P20_BOX, P20_BOX)}: {len(E20)} (galaxy, stage) rows "
          f"over {E20['gal_id'].nunique()} galaxies and {len(P20_STAGES)} stages; reading their reduced particle files.")
    rows20, _nz20, _nmiss = [], 0, 0
    for r in E20.itertuples():
        p = os.path.join(P20_PDIR, f"snap_{int(r.snap):03d}", f"{P20_PREFIX}_snap{int(r.snap):03d}_gal{int(r.gx):06d}.h5")
        if not os.path.exists(p):
            _nmiss += 1; continue
        try:
            with h5py.File(p, "r") as h:
                g = h["gas"]
                if "nh" not in g:
                    continue
                md, T, nh = (g[k][:].astype(float) for k in ("m_dust", "temp", "nh"))
                sfr = g["sfr"][:].astype(float) if "sfr" in g else np.zeros(len(md))
                mem = g["member"][:].astype(bool) if "member" in g else np.ones(len(md), bool)
                Zp = g["zgas"][:].astype(float) if "zgas" in g else None
                mg = g["m_gas"][:].astype(float)
        except OSError:
            _nmiss += 1; continue
        _nz20 += int(Zp is not None)
        sel = (mem if P20_SEL == "member" else np.ones(len(md), bool))
        sel &= np.isfinite(md) & (md > 0) & np.isfinite(T) & (T > 0) & np.isfinite(nh) & (nh > 0)
        if sel.sum() < 5:
            continue
        mdd, Td, nhd, sfd = md[sel], T[sel], nh[sel], sfr[sel]
        tsp = p19_tau_sp(Td, nhd)
        M = float(mdd.sum())
        rec = dict(gal_id=int(r.gal_id), anchor_snap=int(r.anchor_snap), stage=r.stage, snap=int(r.snap), gx=int(r.gx),
                   z=float(r.z), t_stage=float(r.t_stage_gyr), n_part=int(sel.sum()), M_dust_p=M, M_gas_p=float(mg[sel].sum()),
                   tau_sp=float(M / np.sum(mdd / tsp)),
                   f_dust_hot5=float(mdd[Td > 1e5].sum() / M), f_dust_hot6=float(mdd[Td > 1e6].sum() / M),
                   f_dust_sf=float(mdd[sfd > 0].sum() / M),
                   logT_dw=float(np.sum(mdd / M * np.log10(Td))), lognh_dw=float(np.sum(mdd / M * np.log10(nhd))))
        _ns = sfd <= 0
        if mdd[_ns].sum() > 0:
            rec["tau_sp_nsf"] = float(mdd[_ns].sum() / np.sum(mdd[_ns] / tsp[_ns]))
            rec["logT_dw_nsf"] = float(np.sum(mdd[_ns] / mdd[_ns].sum() * np.log10(Td[_ns])))
        if Zp is not None:                                        # the growth term, once `zgas` is in the files
            Zd = Zp[sel]
            ok = np.isfinite(Zd) & (Zd > 0)
            if ok.sum() >= 5:
                tac = p20_tau_acc(Td[ok], nhd[ok], Zd[ok])
                w = mdd[ok]
                rec["tau_acc"] = float(w.sum() / np.sum(w / tac))
                rec["f_dust_growing"] = float(w[tac < tsp[ok]].sum() / w.sum())     # growth clock shorter than destruction
                rec["logZ_dw"] = float(np.sum(w / w.sum() * np.log10(Zd[ok])))
        rows20.append(rec)
    T20 = pd.DataFrame(rows20)
    if not len(T20):
        print(f"  no stage row could be measured ({_nmiss} files missing). If the files exist but carry no `nh`, run the "
              f"backfill:\n    sbatch --array=0-15 submit_reduced_particles.sh "
              f"output/cis25/caesar_sfh/prof_kstracks/dust_profile_plan_kstracks.hdf5")
    else:
        _has_acc = "tau_acc" in T20.columns and np.isfinite(T20["tau_acc"]).any()
        print(f"  measured {len(T20)} (galaxy, stage) rows ({_nmiss} files missing); the growth term "
              + ("IS available (`zgas` present)." if _has_acc else
                 "is NOT available: `zgas` is missing from the reduced files. It backfills like `nh` did —\n"
                 "    sbatch --array=0-15 submit_reduced_particles.sh output/cis25/caesar_sfh/prof_kstracks/dust_profile_plan_kstracks.hdf5"))
        # ── split the tracks by the ANCHOR outcome ──
        ANC20 = T20[T20["stage"] == "anchor"][["gal_id", "anchor_snap"]].drop_duplicates()
        _cat = CAT[CAT["box"] == P20_BOX].copy()
        with np.errstate(all="ignore"):
            _cat["dh2"] = np.log10(np.where((_cat["mh2"].to_numpy(float) > 0) & (_cat["mdust"].to_numpy(float) > 0),
                                            _cat["mdust"].to_numpy(float) / _cat["mh2"].to_numpy(float), np.nan))
            _cat["lfd"] = np.log10(np.where(_cat["mdust"].to_numpy(float) > 0,
                                            _cat["mdust"].to_numpy(float) / _cat["mstar"].to_numpy(float), np.nan))
        _key = _cat[["snap", "gal_id", P20_SPLIT]].rename(columns={"snap": "anchor_snap", P20_SPLIT: "outcome"})
        ANC20 = ANC20.merge(_key, on=["anchor_snap", "gal_id"], how="left")
        _lo, _hi = np.nanquantile(ANC20["outcome"], 0.33), np.nanquantile(ANC20["outcome"], 0.67)
        ANC20["grp"] = np.where(ANC20["outcome"] >= _hi, "dusty at the anchor",
                                np.where(ANC20["outcome"] <= _lo, "dust-poor at the anchor", "middle"))
        T20 = T20.merge(ANC20[["gal_id", "anchor_snap", "outcome", "grp"]], on=["gal_id", "anchor_snap"], how="left")
        print(f"  the tracks are split on the anchor {P20_SPLIT}: dusty >= {_hi:.2f} ({int((ANC20['grp'] == 'dusty at the anchor').sum())} "
              f"galaxies) vs dust-poor <= {_lo:.2f} ({int((ANC20['grp'] == 'dust-poor at the anchor').sum())}).")
        Q20 = [("tau_sp_nsf", r"$\tau_{\rm sp}$ of the NON-SF dust  [Gyr]", True),
               ("f_dust_hot5", r"dust fraction above $10^5$ K", False),
               ("f_dust_sf", r"dust fraction in SF gas", False),
               ("lognh_dw", r"$\log n_{\rm H}$ (dust-weighted)", False)]
        if _has_acc:
            Q20 = [("tau_acc", r"$\tau_{\rm acc}$ — the GROWTH clock  [Gyr]", True),
                   ("f_dust_growing", r"dust fraction with $\tau_{\rm acc} < \tau_{\rm sp}$", False)] + Q20
        print(f"\n  {'stage':13s} {'n':>4s} " + " ".join(f"{q[:16]:>18s}" for q, _, _ in Q20))
        for st in P20_STAGES:
            S = T20[T20["stage"] == st]
            if not len(S):
                continue
            print(f"  {st:13s} {len(S):4d} " + " ".join(f"{np.nanmedian(S[q]):18.3f}" if q in S.columns else f"{'n/a':>18s}"
                                                        for q, _, _ in Q20))
        print(f"\n  the same, split by the anchor outcome (median per stage):")
        for q, lab, _lg in Q20:
            if q not in T20.columns:
                continue
            print(f"    {lab[:44]:44s}")
            for g in ("dusty at the anchor", "dust-poor at the anchor"):
                S = T20[T20["grp"] == g]
                print(f"      {g:26s} " + " ".join(f"{st[:9]:>10s} {np.nanmedian(S.loc[S['stage'] == st, q]):8.3f}"
                                                   for st in P20_STAGES if (S["stage"] == st).any()))
        # ── the figure: the budget along the track ──
        _x20 = {st: i for i, st in enumerate(P20_STAGES)}
        fig, axs = plt.subplots(len(Q20), 1, figsize=(9.2, 2.5 * len(Q20)), squeeze=False, constrained_layout=True, sharex=True)
        COL20 = {"dusty at the anchor": "#d7191c", "dust-poor at the anchor": "#2c7bb6", "middle": "0.6"}
        for ri, (q, lab, logy) in enumerate(Q20):
            ax = axs[ri, 0]
            if q not in T20.columns:
                ax.text(0.5, 0.5, "not available", transform=ax.transAxes, ha="center"); continue
            for g, c in COL20.items():
                S = T20[T20["grp"] == g]
                xs, ys, lo, hi = [], [], [], []
                for st in P20_STAGES:
                    v = S.loc[S["stage"] == st, q].to_numpy(float); v = v[np.isfinite(v)]
                    if len(v) < 3:
                        continue
                    xs.append(_x20[st]); ys.append(np.median(v))
                    lo.append(np.percentile(v, 25)); hi.append(np.percentile(v, 75))
                if not xs:
                    continue
                ax.fill_between(xs, lo, hi, color=c, alpha=0.15, lw=0)
                ax.plot(xs, ys, "-o", color=c, lw=2.2, ms=6, label=(g if ri == 0 else None))
            if logy:
                ax.set_yscale("log")
            ax.set_ylabel(lab, fontsize=9.5); ax.grid(alpha=0.22, lw=0.5)
            if ri == 0:
                ax.legend(frameon=False, fontsize=8.5, loc="best")
            if q == "tau_sp_nsf":
                ax.axhline(13.8, color="0.4", lw=1.2, ls=":")
                ax.text(0.01, 13.8, " a Hubble time", transform=ax.get_yaxis_transform(), fontsize=8, color="0.4", va="bottom")
        axs[-1, 0].set_xticks(list(_x20.values())); axs[-1, 0].set_xticklabels(P20_STAGES, rotation=25, ha="right", fontsize=9)
        fig.suptitle(rf"Part 20 — the dust budget along the track ({BOX_LABEL.get(P20_BOX, P20_BOX)}, {T20['gal_id'].nunique()} "
                     rf"galaxies): was the dust ever in a sputtering-fast phase, and when did growth win?" + "\n"
                     + r"median and 25-75 range per stage; the split is the dust-to-H$_2$ the galaxy ends with", fontsize=11.5)
        paper_save(fig, f"agn_states_dustbudget_track_{P20_BOX}")
        plt.show()
        T20.to_csv(os.path.join(OUT, f"agn_states_dustbudget_track_{P20_BOX}.csv"), index=False)
        print(f"\n  per (galaxy, stage) -> {os.path.join(OUT, f'agn_states_dustbudget_track_{P20_BOX}.csv')}")
        print("READ IT LIKE THIS: the top row is the destruction clock of the dust that is NOT on the star-forming EOS — the only "
              "one whose temperature is physical. If it never drops near a Hubble time at any stage, sputtering never had a "
              "chance to matter and the dust budget of these galaxies is decided by the growth term and by how much dense gas "
              "survives. The 'dust above 1e5 K' row is the direct version of the same statement, and the growth rows say when "
              "and where the grains were actually being rebuilt.")

## Part 21 — the chain: AGN history → gas structure → dust

Parts 17–19 killed the direct link. At fixed age the AGN history barely predicts the dust ($\rho\sim-0.06$ in the old bin); nothing is being sputtered (no dust above $10^5$ K, $\tau_{\rm sp}\sim60$ Gyr for the non-SF dust); and what *does* predict the dust is the **density of the gas it sits in**. The hypothesis that survives is the one this part tests:

> the AGN does not destroy the grains — it decides whether the galaxy can still hold **dense, rotating molecular gas**, and dust grows wherever that gas is.

So the chain is **AGN history → gas structure ($\Sigma_{\rm H_2}$, $\Sigma_{\rm dust}$, extent, $\kappa_{\rm rot}$, core vs disc) → dust**, and it is tested *as* a chain:

1. **AGN vs structure** — the link the AGN is allowed to have: not with the grains, with the gas.
2. **structure vs dust** — the link we already know is strong.
3. **mediation** — $\rho({\rm AGN},{\rm dust})$ against the partial $\rho({\rm AGN},{\rm dust}\mid S)$ for the best single mediator *and* against the whole structural block held fixed at once (`p21_partial_multi`, rank-residualised). If the AGN acts *through* the structure, (1) and (2) are strong and the partial collapses toward zero.
4. **cause before effect** — the exposure accumulated **up to** a stage against the structure measured **at** that stage, for `P21_STAGES` (qt → post_quench → gas_min → anchor), with the structure at each stage also scored against the dust the galaxy ends with.

**Circularity is excluded explicitly.** $\Sigma_{\rm dust}$ and D/H$_2$ per zone are the outcome seen through a smaller aperture; they stay in block (2) — flagged — because they say *where* the dust is, but `P21_CIRCULAR` bars them from being mediators. Only the H$_2$ and kinematic structure is allowed to mediate.

**Everything is a pure read** of the m25 KS-track products — `ks_track_measurements.fits` (apertures `ap1/ap3/ap10kpc` with areas, so $\Sigma$ in the core and in the **disc annulus** = `ap10` minus `ap3`, plus $R_{50}$(H$_2$), $R_{50}$(star)), `ks_stage_kinematics.fits` (zones `ap3kpc` / `ann10kpc`: $\kappa_{\rm rot}$ of H$_2$, gas and stars and the H$_2$–star alignment) and `ks_track_epochs.fits` (the stage map, $t_{\rm SFT}$, $t_{\rm QT}$, $\tau_q$) — joined to this notebook's own AGN histories, from which $E_x$, the post-QT exposure, $f_x$, $t_{\rm quiet}$ and the current state are rebuilt.

Output: `agn_states_chain_cis25.{png,pdf}` — the AGN × structure correlation matrix, the structure → dust forest, the mediation panel (raw vs structure-controlled), and the two planes the story lives on ($\Sigma_{\rm H_2}$(disc) vs dust coloured by $E_x$; $\kappa_{\rm rot}$(H$_2$) vs $\Sigma_{\rm H_2}$ coloured by dust) — plus `_galaxies.csv` (every structural, AGN and dust column per anchor) and `_rho.csv` (all four blocks).

In [ ]:
# ── Part 21 — THE CHAIN: does the AGN history decide what kind of gas structure a galaxy can keep, and does THAT decide the dust? ──
# Parts 17-19 killed the direct link: at fixed age the AGN history barely predicts the dust (rho ~ -0.06), nothing is being
# sputtered (no dust above 1e5 K, tau_sp of the non-star-forming dust ~ 60 Gyr), and what does predict the dust is the DENSITY of
# the gas it sits in. The hypothesis this part tests is the one that survives: the AGN does not destroy the grains, it decides
# whether the galaxy can still hold a dense, extended, rotating H2 disc — and dust grows wherever that gas is.
# So the chain is  AGN history  ->  GAS STRUCTURE (Sigma_H2, Sigma_dust, extent, kappa_rot, core vs disc)  ->  DUST,
# and it is tested as a chain: (1) AGN vs structure, (2) structure vs dust, (3) MEDIATION — the partial correlation
# rho(AGN, dust | structure) against rho(AGN, structure | dust). If the AGN acts THROUGH the structure, (1) and (2) are strong
# and rho(AGN, dust | structure) collapses towards zero while rho(AGN, structure | dust) survives.
# (4) The temporal version: the exposure accumulated BEFORE a stage against the structure AT that stage — cause before effect.
# Pure reads of the m25 KS-track products (ks_track_measurements / ks_stage_kinematics / ks_track_epochs, built by
# ks_tracks_quenched_m25.ipynb from the same reduced particles Part 19 measures) plus this notebook's own AGN histories.
P21_BOX     = "cis25"
P21_KSDIR   = os.path.join(os.getcwd(), "output", "cis25", "ks_tracks")
P21_CORE, P21_DISC_IN, P21_DISC_OUT = "ap3kpc", "ap3kpc", "ap10kpc"   # core aperture; the disc = OUT minus IN (an annulus)
P21_STAGES  = ["qt", "post_quench", "gas_min", "anchor"]              # the temporal test, in time order
P21_DUST    = "dtm"          # the outcome: "dtm" (dust-to-metal) | "dh2" | "lfd"
# A mediator may NOT be made of dust: Sigma_dust and D/H2 per zone are the outcome seen through a smaller aperture, so their
# correlation with the dust is circular. They are still printed in block (2) — flagged — because they say WHERE the dust is;
# only the H2 / kinematic structure is allowed to mediate.
P21_CIRCULAR = ("lsig_dust_core", "lsig_dust_disc", "dh2_core", "dh2_disc")
P21_NMIN    = 12


def p21_partial(x, y, z):
    """Partial Spearman rho(x, y | z): rank-transform, then the usual first-order partial correlation."""
    x, y, z = (np.asarray(v, float) for v in (x, y, z))
    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    if m.sum() < 8:
        return np.nan, np.nan, int(m.sum())
    from scipy.stats import rankdata, t as _t
    rx, ry, rz = (rankdata(v[m]) for v in (x, y, z))
    def _r(a, b):
        a, b = a - a.mean(), b - b.mean()
        d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
        return float((a * b).sum() / d) if d > 0 else np.nan
    rxy, rxz, ryz = _r(rx, ry), _r(rx, rz), _r(ry, rz)
    den = np.sqrt(max(1e-12, (1 - rxz ** 2) * (1 - ryz ** 2)))
    pr = (rxy - rxz * ryz) / den
    n = int(m.sum())
    tt = pr * np.sqrt(max(n - 3, 1) / max(1e-12, 1 - pr ** 2))
    return float(pr), float(2 * _t.sf(abs(tt), max(n - 3, 1))), n


def p21_partial_multi(x, y, Z):
    """Rank-based partial correlation rho(x, y | Z) with Z a (n, k) block of controls: rank everything, residualise x and y on
    Z by least squares, correlate the residuals. This is the honest version of "hold the gas structure fixed"."""
    from scipy.stats import rankdata, t as _t
    x, y, Z = np.asarray(x, float), np.asarray(y, float), np.asarray(Z, float)
    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(Z).all(axis=1)
    n = int(m.sum())
    if n < 3 * Z.shape[1] + 8:
        return np.nan, np.nan, n
    rx, ry = rankdata(x[m]), rankdata(y[m])
    RZ = np.column_stack([rankdata(Z[m, j]) for j in range(Z.shape[1])] + [np.ones(n)])
    bx, _, _, _ = np.linalg.lstsq(RZ, rx, rcond=None)
    by, _, _, _ = np.linalg.lstsq(RZ, ry, rcond=None)
    ex, ey = rx - RZ @ bx, ry - RZ @ by
    d = np.sqrt((ex ** 2).sum() * (ey ** 2).sum())
    pr = float((ex * ey).sum() / d) if d > 0 else np.nan
    k = Z.shape[1]
    tt = pr * np.sqrt(max(n - k - 2, 1) / max(1e-12, 1 - pr ** 2))
    return pr, float(2 * _t.sf(abs(tt), max(n - k - 2, 1))), n


_need21 = [os.path.join(P21_KSDIR, f) for f in ("ks_track_measurements.fits", "ks_stage_kinematics.fits", "ks_track_epochs.fits")]
if "CAT" not in globals():
    print("Part 21: needs Part 0 — run it first.")
elif not all(os.path.exists(p) for p in _need21):
    print("Part 21: the m25 KS-track products are missing — ks_tracks_quenched_m25.ipynb builds them:\n  "
          + "\n  ".join(p for p in _need21 if not os.path.exists(p)))
else:
    def _s21(t):
        d = t.to_pandas() if hasattr(t, "to_pandas") else t
        for c in d.columns:
            if d[c].dtype == object:
                d[c] = d[c].map(lambda v: v.decode() if isinstance(v, bytes) else v)
        return d
    MEA = _s21(Table.read(_need21[0]))
    KIN = _s21(Table.read(_need21[1]))
    EPO = _s21(Table.read(_need21[2]))
    EPO = EPO[EPO["pop"] == "Q"].copy()

    # ── the structure block: surface densities in the core and in the disc annulus, extent, and rotation ──
    def p21_structure(snap_gx):
        """(snap, gx) frame -> one row per galaxy with the structural block."""
        A = MEA[MEA["ap_label"] == P21_CORE].merge(snap_gx, on=["snap", "gx"], how="inner")
        B = MEA[MEA["ap_label"] == P21_DISC_OUT][["snap", "gx", "m_H2", "m_dust", "m_gas", "m_star", "area_kpc2", "sfr_inst"]]
        B = B.rename(columns={c: c + "_out" for c in ("m_H2", "m_dust", "m_gas", "m_star", "area_kpc2", "sfr_inst")})
        S = A.merge(B, on=["snap", "gx"], how="left")
        with np.errstate(all="ignore"):
            for q in ("H2", "dust", "gas"):
                _in, _out = S[f"m_{q}"].to_numpy(float), S[f"m_{q}_out"].to_numpy(float)
                _ai, _ao = S["area_kpc2"].to_numpy(float), S["area_kpc2_out"].to_numpy(float)
                S[f"lsig_{q}_core"] = np.log10(np.where(_in > 0, _in / _ai, np.nan))
                _d, _da = _out - _in, _ao - _ai
                S[f"lsig_{q}_disc"] = np.log10(np.where((_d > 0) & (_da > 0), _d / np.maximum(_da, 1e-9), np.nan))
                S[f"conc_{q}"] = S[f"lsig_{q}_core"] - S[f"lsig_{q}_disc"]          # core-heavy (+) vs extended (-)
            S["dh2_core"] = np.log10(np.where((S["m_H2"] > 0) & (S["m_dust"] > 0), S["m_dust"] / S["m_H2"], np.nan))
            _dd = (S["m_dust_out"] - S["m_dust"]).to_numpy(float); _hd = (S["m_H2_out"] - S["m_H2"]).to_numpy(float)
            S["dh2_disc"] = np.log10(np.where((_dd > 0) & (_hd > 0), _dd / np.maximum(_hd, 1e-30), np.nan))
            S["r50_H2_over_star"] = S["r50_H2"].to_numpy(float) / np.where(S["r50_star"].to_numpy(float) > 0,
                                                                          S["r50_star"].to_numpy(float), np.nan)
        for zone, tag in ((P21_CORE, "core"), ("ann10kpc", "disc")):
            Kz = KIN[KIN["zone"] == zone][["snap", "gx", "kappa_H2", "kappa_star", "kappa_gas", "cos_H2_star", "age_mw_gyr"]]
            Kz = Kz.rename(columns={c: f"{c}_{tag}" for c in ("kappa_H2", "kappa_star", "kappa_gas", "cos_H2_star", "age_mw_gyr")})
            S = S.merge(Kz, on=["snap", "gx"], how="left")
        return S

    P21_STRUCT = [("lsig_H2_core", "log Sig_H2 core"), ("lsig_H2_disc", "log Sig_H2 disc"),
                  ("lsig_dust_core", "log Sig_d core"), ("lsig_dust_disc", "log Sig_d disc"),
                  ("conc_H2", "H2 concentration"), ("r50_H2", "R50(H2) [kpc]"), ("r50_H2_over_star", "R50 H2 / star"),
                  ("kappa_H2_core", "kappa_H2 core"), ("kappa_H2_disc", "kappa_H2 disc"),
                  ("kappa_star_core", "kappa_* core"), ("cos_H2_star_disc", "cos(H2, *) disc")]
    P21_AGN = [("e_x", "E_x [Gyr]"), ("e_x_post", "E_x since QT"), ("f_x", "duty f_x"), ("t_quiet", "t_quiet [Gyr]"),
               ("xray_now", "X-ray now"), ("tau_q_gyr", "tau_q [Gyr]"), ("xstr_quench", "x_str at quench")]

    # ── the anchor sample: structure + the AGN history from this notebook's own cis25 histories ──
    ANC = EPO[EPO["stage"] == "anchor"][["snap", "gx", "gal_id", "anchor_snap", "z", "t_stage_gyr", "t_sft_gyr", "t_qt_gyr",
                                         "tau_q_gyr", "agn_class", "xstr_quench", "log_mstar_anchor", "mdust_cat", "mh2_cat",
                                         "mgas_cat", "r50star_cat"]].copy()
    P21 = p21_structure(ANC[["snap", "gx"]].drop_duplicates()).merge(ANC, on=["snap", "gx"], how="inner")
    _agn21, _hp21 = {}, HIST_FILE.format(box=P21_BOX)
    if os.path.exists(_hp21):
        with h5py.File(_hp21, "r") as _f:
            for _a in sorted(set(P21["snap"].astype(int))):
                _k = f"snap{int(_a):03d}"
                if _k not in _f:
                    continue
                _g = _f[_k]
                _t = _g["t_yr"][:] / 1e9; _o = np.argsort(_t); _t = _t[_o]
                _ms, _mg, _mb, _fe = (_g[c][:][_o] for c in ("mstar", "mgas", "mbh", "fedd"))
                _zg = _g["zgas"][:][_o] if "zgas" in _g else np.full_like(_ms, np.nan)
                _S, _, _X = agn_states(_ms, _mg, _mb, _fe)
                _dt = np.gradient(_t)
                for _j, _gid in enumerate(np.asarray(_g["gal_ids"][:], dtype=np.int64)):
                    _ex = _X["exist"][:, _j]
                    if not _ex.any():
                        continue
                    _k2 = int(np.flatnonzero(_ex)[-1])
                    _xr = (np.nan_to_num(_S["xray"][:, _j]) > 0.5) & _ex
                    _agn21[(int(_a), int(_gid))] = dict(
                        e_x=float(np.sum(_dt[_xr])), f_x=float(_xr.sum() / max(1, int(_ex.sum()))),
                        xray_now=float(_xr[_k2]), t_anchor=float(_t[_k2]),
                        t_quiet=float(_t[_ex][-1] - _t[_xr][-1]) if _xr.any() else np.nan,
                        zgas_anc=float(_zg[_k2, _j]) if np.isfinite(_zg[_k2, _j]) else np.nan,
                        _t=_t, _xr=_xr, _dt=_dt)
    def _ag(row, key):
        d = _agn21.get((int(row["snap"]), int(row["gal_id"])), None)
        return np.nan if d is None else d.get(key, np.nan)
    for c in ("e_x", "f_x", "xray_now", "t_quiet", "zgas_anc"):
        P21[c] = [ _ag(r, c) for _, r in P21.iterrows() ]
    _exp = []                                                        # the exposure accumulated SINCE QT (the post-quench dose)
    for _, r in P21.iterrows():
        d = _agn21.get((int(r["snap"]), int(r["gal_id"])), None)
        if d is None or not np.isfinite(r.get("t_qt_gyr", np.nan)):
            _exp.append(np.nan); continue
        m = d["_xr"] & (d["_t"] >= float(r["t_qt_gyr"]))
        _exp.append(float(np.sum(d["_dt"][m])))
    P21["e_x_post"] = _exp
    with np.errstate(all="ignore"):
        _ms21 = 10 ** P21["log_mstar_anchor"].to_numpy(float)
        P21["lfd"] = np.log10(np.where(P21["mdust_cat"] > 0, P21["mdust_cat"] / _ms21, np.nan))
        P21["dh2"] = np.log10(np.where((P21["mh2_cat"] > 0) & (P21["mdust_cat"] > 0), P21["mdust_cat"] / P21["mh2_cat"], np.nan))
        _met = P21["zgas_anc"].to_numpy(float) * P21["mgas_cat"].to_numpy(float)
        _tot = _met + P21["mdust_cat"].to_numpy(float)
        P21["dtm"] = np.log10(np.where(_tot > 0, P21["mdust_cat"].to_numpy(float) / np.where(_tot > 0, _tot, np.nan), np.nan))
    DUST21 = P21_DUST if np.isfinite(P21[P21_DUST]).sum() >= P21_NMIN else "dh2"
    print(f"Part 21 — THE CHAIN in {BOX_LABEL.get(P21_BOX, P21_BOX)}: {len(P21)} quenched anchor rows "
          f"({P21['gal_id'].nunique()} distinct galaxies over {P21['anchor_snap'].nunique()} anchors), structure from the KS-track "
          f"products, AGN history from this notebook's own histories, outcome = {DUST21}.")
    print(f"  core = {P21_CORE} (area {np.nanmedian(P21['area_kpc2']):.1f} kpc^2), disc = {P21_DISC_OUT} minus {P21_DISC_IN} "
          f"(annulus); kappa_rot from the KS zone kinematics (core = {P21_CORE}, disc = ann10kpc).")

    # ── (1) AGN -> structure, (2) structure -> dust, (3) mediation ──
    def _rho21(d, x, y):
        m = np.isfinite(d[x]) & np.isfinite(d[y])
        if m.sum() < P21_NMIN or np.nanstd(d.loc[m, x]) == 0 or np.nanstd(d.loc[m, y]) == 0:
            return np.nan, np.nan, int(m.sum())
        r, p = spearmanr(d.loc[m, x], d.loc[m, y])
        return float(r), float(p), int(m.sum())
    rows21 = []
    print(f"\n  (1) AGN HISTORY vs GAS STRUCTURE  (* = p < 0.05)")
    print(f"    {'':18s}" + "".join(f"{lab[:15]:>17s}" for _, lab in P21_STRUCT))
    for q, lab in P21_AGN:
        if q not in P21.columns:
            continue
        line = f"    {lab[:18]:18s}"
        for s, slab in P21_STRUCT:
            r, p, n = _rho21(P21, q, s)
            rows21.append(dict(block="agn_vs_structure", x=q, y=s, rho=r, p=p, n=n))
            line += (f"{r:+11.2f}{'*' if (np.isfinite(p) and p < 0.05) else ' '}     " if np.isfinite(r) else f"{'n/a':>17s}")
        print(line)
    print(f"\n  (2) GAS STRUCTURE vs THE DUST ({DUST21})")
    for s, slab in P21_STRUCT:
        r, p, n = _rho21(P21, s, DUST21)
        rows21.append(dict(block="structure_vs_dust", x=s, y=DUST21, rho=r, p=p, n=n))
        _circ = "   [CIRCULAR: this IS the dust, seen through one aperture — not a mediator]" if s in P21_CIRCULAR else ""
        print((f"    {slab[:22]:22s} rho {r:+.2f}{'*' if (np.isfinite(p) and p < 0.05) else ' '}  (p {p:.1e}, n {n}){_circ}")
              if np.isfinite(r) else f"    {slab[:22]:22s} n/a")
    _best = max([r for r in rows21 if r["block"] == "structure_vs_dust" and np.isfinite(r["rho"])
                 and r["x"] not in P21_CIRCULAR], key=lambda r: abs(r["rho"]), default=None)
    print(f"\n  (3) MEDIATION — is the AGN acting THROUGH the structure? mediator = "
          f"{_best['x'] if _best else 'n/a'} (the strongest NON-CIRCULAR structural predictor of the dust: the dust-made "
          f"variables of block (2) are excluded, they are the outcome itself)")
    if _best is not None:
        S_ = _best["x"]
        _ctrl = [c for c, _ in P21_STRUCT if c not in P21_CIRCULAR and c in P21.columns
                 and np.isfinite(P21[c]).sum() > 0.6 * len(P21)]          # the structural block, minus the dust-made variables
        print(f"    controls for the joint column: {_ctrl}")
        print(f"    {'AGN quantity':18s} {'rho(A,dust)':>13s} {'rho(A,d|S)':>12s} {'rho(A,d|ALL)':>14s} {'rho(A,S)':>10s} "
              f"{'rho(S,d|A)':>12s}   verdict")
        for q, lab in P21_AGN:
            if q not in P21.columns:
                continue
            r_ad, p_ad, n1 = _rho21(P21, q, DUST21)
            pr_ad, pp_ad, _ = p21_partial(P21[q], P21[DUST21], P21[S_])
            r_as, p_as, _ = _rho21(P21, q, S_)
            pr_sd, pp_sd, _ = p21_partial(P21[S_], P21[DUST21], P21[q])
            if not np.isfinite(r_ad):
                continue
            pr_all, pp_all, n_all = p21_partial_multi(P21[q], P21[DUST21], P21[_ctrl].to_numpy(float))
            _ref = pr_all if np.isfinite(pr_all) else pr_ad
            verdict = ("no link to mediate" if abs(r_ad) <= 0.15 else
                       ("MEDIATED by the gas structure" if abs(_ref) < 0.5 * abs(r_ad) else
                        ("partly mediated" if abs(_ref) < 0.8 * abs(r_ad) else "survives the structure")))
            rows21.append(dict(block="mediation", x=q, y=DUST21, mediator=S_, rho=r_ad, rho_partial=pr_ad,
                               rho_partial_all=pr_all, n_all=n_all, rho_agn_struct=r_as,
                               rho_struct_dust_partial=pr_sd, p=p_ad, n=n1))
            print(f"    {lab[:18]:18s} {r_ad:+13.2f} {pr_ad:+12.2f} {pr_all:+14.2f} {r_as:+10.2f} {pr_sd:+12.2f}   {verdict}")
    # ── (4) cause before effect: the exposure accumulated BEFORE a stage vs the structure AT that stage ──
    print(f"\n  (4) CAUSE BEFORE EFFECT — E_x accumulated up to each stage vs the structure measured AT that stage:")
    stage_rows = []
    for st in P21_STAGES:
        Est = EPO[EPO["stage"] == st][["snap", "gx", "gal_id", "anchor_snap", "t_stage_gyr", "t_qt_gyr"]]
        if not len(Est):
            continue
        St = p21_structure(Est[["snap", "gx"]].drop_duplicates()).merge(Est, on=["snap", "gx"], how="inner")
        _ex_to = []
        for _, r in St.iterrows():
            d = _agn21.get((int(r["anchor_snap"]), int(r["gal_id"])), None)
            if d is None or not np.isfinite(r["t_stage_gyr"]):
                _ex_to.append(np.nan); continue
            m = d["_xr"] & (d["_t"] <= float(r["t_stage_gyr"]))
            _ex_to.append(float(np.sum(d["_dt"][m])))
        St["e_x_to_stage"] = _ex_to
        St = St.merge(P21[["gal_id", "anchor_snap", DUST21]].rename(columns={DUST21: "dust_final"}),
                      on=["gal_id", "anchor_snap"], how="left")
        line = f"    {st:12s} n {len(St):4d}  "
        for s, slab in [("lsig_H2_disc", "Sig_H2 disc"), ("lsig_dust_disc", "Sig_d disc"), ("kappa_H2_disc", "kappa_H2 disc"),
                        ("r50_H2", "R50(H2)")]:
            r, p, n = _rho21(St, "e_x_to_stage", s)
            stage_rows.append(dict(block="stage", stage=st, x="e_x_to_stage", y=s, rho=r, p=p, n=n))
            line += f"{slab} {r:+.2f}{'*' if (np.isfinite(p) and p < 0.05) else ' '}  " if np.isfinite(r) else f"{slab} n/a  "
        r, p, n = _rho21(St, "lsig_H2_disc", "dust_final")
        stage_rows.append(dict(block="stage", stage=st, x="lsig_H2_disc", y="dust_final", rho=r, p=p, n=n))
        line += f"|  Sig_H2(disc) -> final dust {r:+.2f}{'*' if (np.isfinite(p) and p < 0.05) else ' '}" if np.isfinite(r) else ""
        print(line)
    R21 = pd.DataFrame(rows21 + stage_rows)

    # ── the figure ──
    fig = plt.figure(figsize=(15.5, 8.6), constrained_layout=True)
    gs = fig.add_gridspec(2, 3)
    ax = fig.add_subplot(gs[0, :2])
    Mx = np.full((len([q for q, _ in P21_AGN if q in P21.columns]), len(P21_STRUCT)), np.nan)
    _agnl = [(q, l) for q, l in P21_AGN if q in P21.columns]
    for i, (q, _) in enumerate(_agnl):
        for j, (s, _) in enumerate(P21_STRUCT):
            v = [r for r in rows21 if r["block"] == "agn_vs_structure" and r["x"] == q and r["y"] == s]
            Mx[i, j] = v[0]["rho"] if v else np.nan
    im = ax.imshow(Mx, cmap="RdBu_r", vmin=-0.6, vmax=0.6, aspect="auto")
    ax.set_xticks(range(len(P21_STRUCT))); ax.set_xticklabels([l for _, l in P21_STRUCT], rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(len(_agnl))); ax.set_yticklabels([l for _, l in _agnl], fontsize=8.5)
    for i in range(Mx.shape[0]):
        for j in range(Mx.shape[1]):
            if np.isfinite(Mx[i, j]):
                ax.text(j, i, f"{Mx[i, j]:+.2f}", ha="center", va="center", fontsize=7,
                        color="w" if abs(Mx[i, j]) > 0.35 else "0.2")
    plt.colorbar(im, ax=ax, label=r"Spearman $\rho$")
    ax.set_title("(1) does the AGN history set the gas structure?", fontsize=10.5)
    ax = fig.add_subplot(gs[0, 2])
    sd = [r for r in rows21 if r["block"] == "structure_vs_dust" and np.isfinite(r["rho"]) and r["x"] not in P21_CIRCULAR]
    yy = np.arange(len(sd))[::-1]
    lab21 = dict(P21_STRUCT)
    for y, r in zip(yy, sd):
        c = "#d7191c" if r["rho"] > 0 else "#2c7bb6"
        ax.plot([0, r["rho"]], [y, y], "-", color=c, lw=2.0, alpha=0.6); ax.plot([r["rho"]], [y], "o", color=c, ms=7)
    ax.axvline(0, color="k", lw=1.2); ax.set_yticks(yy); ax.set_yticklabels([lab21.get(r["x"], r["x"]) for r in sd], fontsize=8)
    ax.set_xlabel(rf"$\rho$ with {DUST21}", fontsize=10); ax.grid(alpha=0.22, lw=0.5, axis="x")
    ax.set_title("(2) does the structure set the dust?", fontsize=10.5)
    ax = fig.add_subplot(gs[1, 0])
    med = [r for r in rows21 if r["block"] == "mediation"]
    if med:
        yy = np.arange(len(med))[::-1]
        for y, r in zip(yy, med):
            ax.plot([0, r["rho"]], [y + 0.16, y + 0.16], "-o", color="0.35", lw=2.0, ms=5)
            _pa = r.get("rho_partial_all", np.nan)
            ax.plot([0, _pa if np.isfinite(_pa) else r["rho_partial"]], [y - 0.16, y - 0.16], "-o", color="#e66101", lw=2.0, ms=5)
        ax.set_yticks(yy); ax.set_yticklabels([dict(P21_AGN).get(r["x"], r["x"]) for r in med], fontsize=8)
        ax.axvline(0, color="k", lw=1.2)
        ax.plot([], [], "-o", color="0.35", label=rf"$\rho$(AGN, {DUST21})")
        ax.plot([], [], "-o", color="#e66101", label=rf"$\rho$(AGN, {DUST21} $|$ the whole gas structure)")
        ax.legend(frameon=False, fontsize=8, loc="best")
    ax.set_xlabel(r"$\rho$", fontsize=10); ax.grid(alpha=0.22, lw=0.5, axis="x")
    ax.set_title("(3) mediation: what is left of the AGN link\nonce the structure is held fixed", fontsize=10.5)
    ax = fig.add_subplot(gs[1, 1])
    _c = P21["e_x"] if np.isfinite(P21["e_x"]).sum() > P21_NMIN else P21["age_mw_gyr_core"]
    sc = ax.scatter(P21["lsig_H2_disc"], P21[DUST21], c=_c, s=42, cmap="viridis", lw=0.4, edgecolor="k")
    plt.colorbar(sc, ax=ax, label=r"$E_x$ [Gyr]")
    ax.set_xlabel(r"$\log \Sigma_{\rm H_2}$ of the DISC annulus   [$M_\odot$ kpc$^{-2}$]", fontsize=10)
    ax.set_ylabel(DUST21, fontsize=10); ax.grid(alpha=0.22, lw=0.5)
    _r, _p, _n = _rho21(P21, "lsig_H2_disc", DUST21)
    ax.set_title(rf"the disc gas and the dust  ($\rho = {_r:+.2f}$)" if np.isfinite(_r) else "the disc gas and the dust", fontsize=10.5)
    ax = fig.add_subplot(gs[1, 2])
    sc = ax.scatter(P21["kappa_H2_disc"], P21["lsig_H2_disc"], c=P21[DUST21], s=42, cmap="plasma", lw=0.4, edgecolor="k")
    plt.colorbar(sc, ax=ax, label=DUST21)
    ax.set_xlabel(r"$\kappa_{\rm rot}$(H$_2$) in the disc annulus", fontsize=10)
    ax.set_ylabel(r"$\log \Sigma_{\rm H_2}$ disc", fontsize=10); ax.grid(alpha=0.22, lw=0.5)
    ax.set_title("a rotating, dense H$_2$ disc — and its dust", fontsize=10.5)
    fig.suptitle(rf"Part 21 — the chain: AGN history $\rightarrow$ gas structure $\rightarrow$ dust, "
                 rf"{len(P21)} quenched anchors of {BOX_LABEL.get(P21_BOX, P21_BOX)}", fontsize=12)
    paper_save(fig, f"agn_states_chain_{P21_BOX}")
    plt.show()
    P21.to_csv(os.path.join(OUT, f"agn_states_chain_{P21_BOX}_galaxies.csv"), index=False)
    R21.to_csv(os.path.join(OUT, f"agn_states_chain_{P21_BOX}_rho.csv"), index=False)
    print(f"\n  per galaxy -> {os.path.join(OUT, f'agn_states_chain_{P21_BOX}_galaxies.csv')}; the correlations beside it.")
    print("READ IT LIKE THIS: (1) is the link the AGN is allowed to have — not with the grains, with the GAS. (2) is the link we "
          "already know is strong. (3) is the test that matters: if rho(AGN, dust) collapses when the structure is held fixed "
          "while rho(AGN, structure) survives, then the AGN sets what kind of gas the galaxy can keep and the dust simply "
          "follows the gas — the story the earlier parts point at. (4) puts the cause before the effect: the exposure "
          "accumulated BEFORE a stage against the structure measured AT it.")

## Part 22 — the picture: the AGN dose, the shape of the H$_2$, and the dust

Part 21 established the chain with correlations. **This part draws it.** Nothing new is measured — the same m25 KS-track products for the structure, the same histories for the AGN — but everything is turned into pictures, because a correlation table is not a result anyone can see. The dust axis is `P22_DUST`, by default $\log M_{\rm dust}/M_{\rm H_2}$ (`lfd` and `dg` are the other two).

`agn_states_dustclock_boxes` — *where the dust is*, **one column per box** (`P22_BOXES`: m100, m50, m25) so the three resolutions can be compared side by side, two rows:

* **(a) the fan** — the main panel. The dust ratio on each galaxy's own quenching clock $t - t_{\rm QT}$, the sample split four ways by **AGN dose $\times$ gas geometry**, each box at its own medians. The four curves lie on top of each other before $t_{\rm QT}$ and fan out after it: *one galaxy before quenching, four after*.
* **(b) the dose** — the dust against how much of the post-star-forming era was spent in the X-ray mode, one line per **third** of the geometry, `P22_DBINS` equal-count bins **inside each third** (so every point carries the same number of galaxies whatever that third's own dose distribution looks like). Three falling lines that never cross: the dose lowers the dust at every geometry, the geometry decides which line you are on. In m25 the two axes are close to independent ($\rho \approx -0.3$) and each survives holding the other fixed.

**The geometry axis is not the same quantity in every column, and the titles say which.** Only m25 has the KS-track products, so only there is the split the H$_2$ **extent relative to the stars**, $-\log R_{50}({\rm H_2})/R_{50}(\star)$ (`P22_GEOM = \"compactness\"`: *compact* H$_2$ above the split, *extended* H$_2$ below — see *why the geometry axis is no longer $\kappa_{\rm rot}$* further down); the coarse boxes are split on the catalogue's **whole-galaxy** $\kappa_{\rm rot}$ of the gas (`P22_GEOM_CAT`, caesar's `rotation.gas_kappa_rot`) over Part 0's sample, with $t_{\rm SFT}$ / $t_{\rm QT}$ from the boxes notebook's classes and the same de-duplication and dose as the m25 frame. `P22_LIKE = True` cuts the m25 column the same way, so the three are strictly like for like — at the price of the KS-track geometry that Figs 2-3 and Part 23 are built on. Each box is written to its own `agn_states_dustclock_<box>.csv`.

`agn_states_dustsplit_cis25` — *when the two halves part company*, low-dose half against high-dose half (m25 only: this needs the resolved core)

* **(a) how much H$_2$ is in the core** and **(b) the geometry axis itself** (`P22_GEOM`), at each critical point of the quenching (`P22_STAGES`: `agn_ign`, `jet_on`, `sft`, `qt`, `anchor`; `sf_peak` is dropped by default), drawn as **binned points, not curves** — the stages are epochs of a sample, not a trajectory, and each galaxy reaches them at its own time. (a): the two halves are indistinguishable the whole way, anchor included — *the dose does not decide how much molecular gas is left*. (b): with the old $\kappa_{\rm rot}$ axis the two halves were identical until $t_{\rm QT}$ and then separated, the low-dose half climbing back to $\kappa \approx 0.3$ while the high-dose half stayed at $\approx 0.16$ — both *below* the isotropic value $1/3$, so what looked like 're-settling' was less radial motion, not a disc. With the extent axis the panel reads how compact the H$_2$ is at each critical point (re-run for the numbers).
* **(c) the dust-gas plane, walked in time since quenching.** $\Sigma_{\rm dust}$ against $\Sigma_{\rm H_2}$ in the core. Behind everything, **the whole scatter**: every critical point and the anchor of every galaxy of the sample, red for the low-dose half and blue for the high-dose half (`P22_SCAT_STAGES = None` = every stage row; a list restricts it). Over it, both halves walking the **same bins of $t - t_{\rm QT}$** (only bins where both halves have `P22_NMIN` galaxies, each galaxy entering a bin once, so the two are comparable point by point). At the bottom right of each **high-dose** point a small box gives the time since quenching (the bin centre) and the number of galaxies behind the pair, low / high dose. They fall together until quenching and then split *vertically*: at the same $\Sigma_{\rm H_2}$ the high-dose half ends more than a dex below in $\Sigma_{\rm dust}$. The grey dashed line is constant dust-to-gas, so the vertical distance from it *is* the dust-to-gas ratio. `P22_PLANE = "ks"` gives the Kennicutt-Schmidt version of the same walk.

`agn_states_dustgas_ssfr_cis25` — **the rejuvenation test**

The **same walked plane as Fig 2 (c)** — but stepped through the **critical points**, `jet_on` $\to$ `sft` $\to$ `qt` $\to$ `anchor` (`P22_SSTAGE`, in time order, arrows on the segments for the direction of time), instead of bins of $t - t_{\rm QT}$ — with one panel per bin of **the galaxy's global sSFR at the anchor** (`P22_SEDGES`, terciles of the galaxies by default). The panel is a property of the **galaxy**, not of the epoch: a whole track stays inside one panel, so each panel follows a *fixed subsample* through its own history and the tracks are genuine evolutionary tracks rather than a different set of galaxies at every stage. If the dusty galaxies were simply the ones still — or again — forming stars at the end, the two halves of the dose would separate *across* the panels and lie on top of each other *inside* them.

A quenched galaxy that formed **no star in the last 100 Myr** at the anchor has no $\log$ sSFR at all. That is not missing data — it is the deadest galaxy in the sample — so it goes into the **bottom panel** rather than being dropped, and the panel edges are quantiles of the galaxies that *do* have a measurable sSFR, so they stay finite whatever fraction is undetected. The cell prints how many those are, and panel (a)'s title carries the count.

Each point is the **median galaxy at that critical point** with its bootstrap error, drawn where `P22_NMIN3` galaxies contribute — a panel $\times$ half $\times$ epoch cell is thinner than a Fig 2 epoch, so this figure carries its own floor. The grey dashed line is constant dust-to-gas and is **the same line in every panel**, so the three panels are directly comparable: the vertical distance from it *is* $\Sigma_{\rm dust}/\Sigma_{\rm H_2}$. The cell prints that ratio, with its N, at every drawn point — so the figure now shows not only *that* the two halves end apart but **where along the sequence they part company**, and whether that place moves when the galaxy's own sSFR is held fixed. *(The earlier version of this figure scored one star per half per panel and gave a 1.07 dex gap in the lowest-sSFR panel and none in the other two; re-run the cell for the per-critical-point numbers.)*

**Two things this part settles, both printed:**

* *"Time since the last X-ray episode" is not an axis in this sample* — 83 % of these quenched galaxies are **in** the X-ray state at the anchor, and over their post-quenching snapshots only ~13 % are out of it, and those sit at one end of the geometry axis (with $\kappa_{\rm rot}$ it was ~72 % in its top third): getting *out* of the state needs re-accretion back over SIMBA's gas gate. The AGN axis that works is **how much of the post-star-forming era was spent in the X-ray mode** (`f_x_post`) — a duty fraction, not an elapsed time, and unlike $E_x$ [Gyr] it carries no redshift trend ($\rho(f_x^{\rm post}, z) = 0.00$ against $-0.39$).
* *The X-ray gate is not doing the work.* SIMBA fires the X-ray channel only below $f_{\rm gas} = $ `XRAY_FGAS_MAX`, so one might fear the dose axis is the gas-fraction axis in disguise. It is not: 99 % of this sample is already below the gate (median $f_{\rm gas} = 0.03$), the dose is flat in it ($\rho = -0.04$), and the picture survives on the dust-to-**gas** ratio (`P22_DUST = "dg"`), which has the gas divided out.

**Why the geometry axis is the H$_2$ extent and no longer $\kappa_{\rm rot}$ (2026-09-04).** The 'rotating' / 'dispersion-supported' split was $\kappa_{\rm rot}$(H$_2$) of the core above or below its median, and that median is **0.21**. For an isotropic velocity field $\kappa_{\rm rot} = K_{\rm rot}/K = 1/3$ (one of three components is azimuthal); a disc sits above $\sim 0.5$. Only 31 % of the sample is above $1/3$ and 21 % above $0.5$, so *nobody* rotated: the label was the upper half of a non-rotating distribution, and a median below $1/3$ means the H$_2$ velocities are dominated by *radial* motion, not by rotation. Worse, the signal lived in the resolution floor: these quenched galaxies hold a median of 128 member H$_2$ particles (43 % below 100), and below 100 particles $\kappa$ scatters from 0.10 to 0.67 with the compact few-particle knots scoring highest. Among the galaxies with $\geq 100$ particles $\rho(\kappa, {\rm dust}) = +0.14$ and the low-dose / high-dose contrast flips sign ($+0.64$ / $-0.52$ dex). The size ratio keeps its sign in both halves there ($-0.71$ / $-0.38$ dex, extended minus compact), keeps its strength at fixed $M_{\rm H_2}$ ($\rho = -0.39$), is **independent of the dose** ($\rho = +0.09$, against $-0.29$ for $\kappa$), and is what the observations measure — $R_{\rm CO}/R_e$, which the ALMA-C11 QGs span from $-0.4$ to $+0.1$ in the log, the range of the simulated bulk (16-84: $-0.43$ to $+0.04$). $\log \Sigma_{\rm H_2}$ inside $R_{50}({\rm H_2})$ is the strongest single dust predictor ($\rho = +0.50$; $+0.49$ at fixed $M_{\rm H_2}$), but it *is* the K-S abscissa, so it is an option (`"sigma_R50"`) and not the default. The two-dimensional view ($M_{\rm H_2} \times$ extent) shows SIMBA's quenched galaxies dusty in **two** places: the ultra-compact H$_2$-poor remnants (extent $< -0.6$, $M_{\rm H_2} < 2 \times 10^8\,M_\odot$: $\log M_{\rm dust}/M_{\rm H_2} \approx -1.4$) and the H$_2$-rich extended discs ($M_{\rm H_2} > 10^9$, $R_{50} \approx 2.7$ kpc, $\kappa \approx 0.45$: $\approx -1.7$); the bulk between them is dust-poor at any $\kappa$. The observed QGs, with $M_{\rm H_2} \sim 10^{9.6-10.1}$ and 4-9 kpc CO discs, belong with the second, of which m25 holds about ten. The cell prints this check for whichever axis is chosen (`P22_NH2CHK`), and `P22_NH2MIN` drops the few-particle galaxies from the split altogether.

**Knobs.** `P22_BOXES` the columns of Fig 1, `P22_GEOM_CAT` / `P22_LIKE` the geometry axis of the boxes without KS-track products and whether m25 is cut the same way; `P22_GEOM` chooses the m25 ISM-geometry axis (`compactness` $= -\log R_{50}({\rm H_2})/R_{50}(\star)$, the default; `sigma_R50` $= \log \Sigma_{\rm H_2}$ inside $R_{50}({\rm H_2})$, He included, the K-S abscissa; `kappa_H2` / `kappa_gas`, the Sales+12 $\kappa_{\rm rot}$ in `P22_CORE`, whose group names then carry the cut itself instead of 'rotating'; `sigma_H2` of the core, which only tracks the particle count, $\rho = +0.89$), `P22_GCUT` where it is split (`\"median\"`, or a number — $1/3$ is the isotropic $\kappa$, $0.5$ rotation-supported), `P22_NH2MIN` / `P22_NH2CHK` the resolution floor of the split and the printed check. `P22_DOSE` the AGN axis, `P22_DUST` the outcome, `P22_CORE` the aperture, `P22_STAGES` the critical points of Fig 2 (a-b), `P22_TQT` the time bins and `P22_SCAT_STAGES` the scatter of Fig 2 (c), `P22_SSTAGE` / `P22_SEDGES` / `P22_NMIN3` the critical points, the sSFR panels and the floor of Fig 3, `P22_DBINS` the dose bins, `P22_PLANE` the plane, `P22_NMIN` the floor everywhere else.

**One track per galaxy.** An anchor row is not a galaxy: a galaxy that is a quenched anchor at several snapshots contributes several overlapping chains — the same early history up to five times. The chains are therefore de-duplicated on the main branch itself (a fingerprint of $M_\star(t)$ at three fixed times) and only the **latest** anchor of each galaxy is kept: in m25, 262 anchor rows $\to$ 175 distinct galaxies. Bands and error bars are 16-84 bootstraps over galaxies, and an epoch or a bin is drawn only where `P22_NMIN` of them contribute.

In [ ]:
# ── Part 22 — THE PICTURE: the AGN dose, the shape of the molecular gas, and the dust, drawn instead of scored ──
# Part 21 established the chain with correlations. This part draws it. Nothing new is measured: the same m25 KS-track products
# (structure) and the same histories (AGN), turned into pictures a reader can take in at a glance —
#   FIG 1  ONE COLUMN PER BOX (P22_BOXES: m100 / m50 / m25), two rows —
#          (a) THE FAN     f_dust on each galaxy's own quenching clock, the sample split four ways by AGN dose x gas geometry:
#                          one curve before t_QT, four after it;
#          (b) THE DOSE    the dust against how much of the post-star-forming era was spent in the X-ray mode, one line per
#                          third of the geometry, equal-count bins WITHIN each third.
#          The geometry axis is the KS-track kappa_rot(H2) of the core where those products exist (m25 only) and the
#          catalogue's whole-galaxy kappa_rot of the gas (P22_GEOM_CAT) where they do not; P22_LIKE = True cuts every column,
#          m25 included, on the catalogue axis over Part 0's sample, so the three columns are strictly like for like.
#   FIG 2  (a) how much H2 the two halves of the dose have at each critical point, (b) how that H2 moves — both as binned
#          points, because the stages are epochs of a sample, not a curve (P22_STAGES; sf_peak is out) — and (c) the
#          dust-gas plane: EVERY point of the sample as a scatter (every critical point and the anchor of every galaxy, low
#          and high dose), the two walked tracks in bins of TIME SINCE QUENCHING over it, and at the bottom right of each
#          high-dose point a box with the time since quenching and the number of galaxies behind the pair of points.
#   FIG 3  THE REJUVENATION TEST: the SAME walked plane as (c), but stepped through the CRITICAL POINTS
#          (jet_on -> sft -> qt -> anchor) instead of bins of time, one panel per bin of the galaxy's sSFR AT THE
#          ANCHOR -- the panel is a property of the GALAXY, not of the epoch, so a whole track stays inside one panel.
#          If the dusty galaxies were simply the ones still (or again) forming stars, the two halves of the dose
#          would separate ACROSS the panels and lie on top of each other inside them.
# ONE TRACK PER GALAXY: a galaxy that is a quenched anchor at several snapshots contributes several overlapping chains, so the
# chains are de-duplicated on the main branch itself (M*(t) at three fixed times) and only the LATEST anchor of each is kept.
P22_BOX    = "cis25"           # the box of Figs 2-3 and of Part 23: the one with KS-track products
P22_BOXES  = ["cis100", "cis50", "cis25"]   # Fig 1: one column per box, in this order (a box without a histories cache is skipped)
P22_KSDIR  = globals().get("P22_KSDIR_OVERRIDE", os.path.join(os.getcwd(), "output", "cis25", "ks_tracks"))
P22_CORE   = "ap3kpc"          # the core aperture the geometry is measured in (the disc annulus is empty for a quarter of them)
P22_GEOM   = "compactness"     # the m25 ISM-geometry axis: "compactness" (= -log R50(H2)/R50(*), the H2 extent relative to the stars —
                               # compact above the split, extended below; the default since 2026-09-04, see the markdown) | "sigma_R50"
                               # (log Sigma_H2 inside R50(H2), He included = the K-S abscissa) | "kappa_H2" | "kappa_gas" (Sales+12
                               # kappa_rot in P22_CORE — its median here is 0.21, BELOW the isotropic 1/3: nobody rotates, and its dust
                               # signal lives in the galaxies with < 100 H2 particles) | "sigma_H2" (core; tracks the particle count)
P22_GCUT   = "median"          # where the geometry axis is split: "median" | a number (for kappa: 1/3 = isotropic, no net rotation
                               # below it; 0.5 = rotation-supported)
P22_NH2MIN = 0                 # a galaxy needs this many member H2 particles for its geometry to count in the split (0 = every galaxy);
                               # the cell prints the picture at P22_NH2CHK regardless
P22_NH2CHK = 100               # the resolution check printed for the chosen axis: the same split inside n_H2 >= this
P22_GEOM_CAT = "kappa_gas"     # Fig 1, the boxes WITHOUT KS-track products: the catalogue's whole-galaxy axis ("kappa_gas" | "kappa_star")
P22_LIKE   = False             # Fig 1: True = the m25 column too is cut on P22_GEOM_CAT over Part 0's sample (strictly like-for-like
                               # columns); False = m25 keeps the KS-track geometry (P22_GEOM in P22_CORE) and sample of Figs 2-3 / Part 23
P22_DOSE   = "f_x_post"        # the AGN axis: "f_x_post" (X-ray duty fraction SINCE SFT) | "f_x" | "e_x" [Gyr] | "xray_now"
P22_DUST   = "dh2"             # the dust axis of (a) and (b): "dh2" = log M_dust/M_H2 | "lfd" = log M_dust/M* | "dg" = log M_dust/M_gas
P22_STAGES = ["agn_ign", "jet_on", "sft", "qt", "anchor"]   # the critical points of Fig 2 (a) and (b) (sf_peak dropped; put it back to see it)
P22_PLANE  = "dust"            # Fig 2 (c): "dust" = the Sigma_dust-Sigma_H2 plane | "ks" = the Kennicutt-Schmidt plane
P22_SCAT_STAGES = None         # Fig 2 (c): the stage rows drawn as the scatter (None = every critical point and the anchor of every galaxy)
P22_NMIN   = 8                 # an epoch / a bin is drawn only where this many galaxies contribute
P22_NMIN3  = 5                 # Fig 3's own floor: a panel x half x critical point is a thinner cell than a Fig 2 epoch
P22_TG     = np.arange(-2.0, 2.01, 0.2)                  # the quenching clock of Fig 1 (a), t - t_QT [Gyr]
P22_DBINS  = 6                 # Fig 1 (b): equal-count dose bins inside each third of the geometry
P22_TQT    = [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 4.0]  # Fig 2 (c): the bin edges of t - t_QT [Gyr] the plane is walked in
P22_SSTAGE = ["jet_on", "sft", "qt", "anchor"]      # Fig 3: the critical points the plane is walked through, in time order
P22_SEDGES = None              # Fig 3: the log sSFR panel edges (the ANCHOR sSFR, per galaxy); None = terciles of the galaxies
P22_NBOOT  = 300
P22_GLAB   = {"kappa_H2": (r"$\kappa_{\rm rot}$(H$_2$), core", "higher-$\kappa$ H$_2$", "lower-$\kappa$ H$_2$"),
              "kappa_gas": (r"$\kappa_{\rm rot}$(gas), core", "higher-$\kappa$ gas", "lower-$\kappa$ gas"),
              "sigma_H2": (r"$\log \Sigma_{\rm H_2}$, core", "dense H$_2$", "diffuse H$_2$"),
              "sigma_R50": (r"$\log \Sigma_{\rm H_2}(<R_{50})$", "dense H$_2$", "diffuse H$_2$"),
              "compactness": (r"$-\log\,R_{50}({\rm H_2})/R_{50}(\star)$", "compact H$_2$", "extended H$_2$")}
P22_GTITLE = {"kappa_H2": "(b) how that gas moves", "kappa_gas": "(b) how that gas moves", "sigma_H2": "(b) how dense that gas is",
              "sigma_R50": "(b) how dense that H$_2$ is inside its own $R_{50}$",
              "compactness": "(b) how compact that H$_2$ is, relative to the stars"}

def _glab22(geom, gcut, vals):
    """The axis label and the names of its two sides. A kappa axis names the cut itself — the split is NOT 'rotating' against
    'dispersion-supported': kappa_rot = 1/3 is an isotropic velocity field, a disc sits above ~0.5 — the other axes keep their words."""
    lab, hi, lo = P22_GLAB[geom]
    if geom.startswith("kappa"):
        c = float(np.nanmedian(vals)) if gcut == "median" else float(gcut)
        what = "H$_2$" if geom == "kappa_H2" else "gas"
        hi, lo = what + rf" $\kappa_{{\rm rot}} > {c:.2f}$", what + rf" $\kappa_{{\rm rot}} \leq {c:.2f}$"
    return lab, hi, lo
P22_GLAB_CAT = {"kappa_gas": (r"$\kappa_{\rm rot}$(gas), whole galaxy", "rotating gas", "dispersion-supported gas"),
                "kappa_star": (r"$\kappa_{\rm rot}$(stars), whole galaxy", "rotating stars", "dispersion-supported stars")}
P22_DLAB   = {"f_x_post": r"X-ray duty fraction since SFT", "f_x": r"X-ray duty fraction", "e_x": r"$E_x$  [Gyr]",
              "xray_now": "X-ray on at the anchor"}
P22_ULAB   = {"lfd": r"$\log\, M_{\rm dust}/M_\star$", "dg": r"$\log\, M_{\rm dust}/M_{\rm gas}$",
              "dh2": r"$\log\, M_{\rm dust}/M_{\rm H_2}$"}

_need22 = [os.path.join(P22_KSDIR, f) for f in ("ks_track_measurements.fits", "ks_stage_kinematics.fits", "ks_track_epochs.fits")]
if "CAT" not in globals():
    print("Part 22: needs Part 0 — run it first.")
elif not all(os.path.exists(p) for p in _need22):
    print("Part 22: the m25 KS-track products are missing — ks_tracks_quenched_m25.ipynb builds them:\n  "
          + "\n  ".join(p for p in _need22 if not os.path.exists(p)))
elif not os.path.exists(HIST_FILE.format(box=P22_BOX)):
    print(f"Part 22: {HIST_FILE.format(box=P22_BOX)} is missing — paper_ism_prediction_boxes Part 6a builds it.")
else:
    def _s22(t):
        d = t.to_pandas() if hasattr(t, "to_pandas") else t
        for c in d.columns:
            if d[c].dtype == object:
                d[c] = d[c].map(lambda v: v.decode() if isinstance(v, bytes) else v)
        return d

    def _band22(A, nmin=None):
        """A (n_gal, n_x) -> median track, its 16-84 bootstrap band over GALAXIES, and the per-column count."""
        A = np.asarray(A, float)
        nmin = P22_NMIN if nmin is None else nmin
        if A.ndim != 2 or not len(A):
            return (np.full(A.shape[-1] if A.ndim == 2 else 1, np.nan),) * 3 + (np.zeros(1, int),)
        n = np.isfinite(A).sum(axis=0)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            cent = np.nanmedian(A, axis=0)
            rng = np.random.default_rng(22)
            bs = np.array([np.nanmedian(A[rng.integers(0, len(A), len(A))], axis=0) for _ in range(P22_NBOOT)])
            lo, hi = np.nanpercentile(bs, 16, axis=0), np.nanpercentile(bs, 84, axis=0)
        ok = n >= nmin
        return np.where(ok, cent, np.nan), np.where(ok, lo, np.nan), np.where(ok, hi, np.nan), n

    def _pt22(v, nmin=None):
        """One bin of per-galaxy values -> median and half the 16-84 bootstrap width (nan below the floor)."""
        v = np.asarray(v, float); v = v[np.isfinite(v)]
        if len(v) < (P22_NMIN if nmin is None else nmin):
            return np.nan, np.nan, len(v)
        rng = np.random.default_rng(22)
        bs = np.array([np.median(v[rng.integers(0, len(v), len(v))]) for _ in range(P22_NBOOT)])
        return float(np.median(v)), float(0.5 * (np.percentile(bs, 84) - np.percentile(bs, 16))), len(v)

    MEA22, KIN22, EPO22 = (_s22(Table.read(p)) for p in _need22)
    EPO22 = EPO22[EPO22["pop"] == "Q"].copy()
    EPO22["key"] = list(zip(EPO22["anchor_snap"].astype(int), EPO22["gal_id"].astype(int)))
    _C22 = MEA22[MEA22["ap_label"] == P22_CORE].set_index(["snap", "gx"])
    _K22 = KIN22[KIN22["zone"] == P22_CORE].set_index(["snap", "gx"])

    def _geom22(snaps, gxs):
        """The structural block of a set of (snap, gx) rows, in the core aperture."""
        ix = list(zip(np.asarray(snaps, int), np.asarray(gxs, int)))
        g = lambda df, c: df[c].reindex(ix).to_numpy(float) if c in df.columns else np.full(len(ix), np.nan)
        with np.errstate(all="ignore"):
            mh2, md, mst, ar, mgs = g(_C22, "m_H2"), g(_C22, "m_dust"), g(_C22, "m_star"), g(_C22, "area_kpc2"), g(_C22, "m_gas")
            sf100, sf_tot, ms_tot = g(_C22, "sfr100"), g(_C22, "sfr100_tot"), g(_C22, "m_star_tot")
            out = dict(sigma_H2=np.log10(np.where(mh2 > 0, mh2 / ar, np.nan)),
                       sigma_SFR=np.log10(np.where(sf100 > 0, sf100 / ar, np.nan)),
                       sigma_dust=np.log10(np.where(md > 0, md / ar, np.nan)),
                       lfd_core=np.log10(np.where((md > 0) & (mst > 0), md / np.where(mst > 0, mst, np.nan), np.nan)),
                       dg_core=np.log10(np.where((md > 0) & (mgs > 0), md / np.where(mgs > 0, mgs, np.nan), np.nan)),
                       dh2_core=np.log10(np.where((md > 0) & (mh2 > 0), md / np.where(mh2 > 0, mh2, np.nan), np.nan)),
                       kappa_H2=g(_K22, "kappa_H2"), kappa_gas=g(_K22, "kappa_gas"), kappa_star=g(_K22, "kappa_star"),
                       lssfr_tot=np.log10(np.where((sf_tot > 0) & (ms_tot > 0), sf_tot / np.where(ms_tot > 0, ms_tot, np.nan), np.nan)),
                       compactness=-np.log10(g(_C22, "r50_H2") / np.where(g(_C22, "r50_star") > 0, g(_C22, "r50_star"), np.nan)),
                       sigma_R50=np.log10(np.where((g(_C22, "m_H2_tot") > 0) & (g(_C22, "r50_H2") > 0),          # the K-S abscissa:
                                                   1.36 * 0.5 * g(_C22, "m_H2_tot") / (np.pi * g(_C22, "r50_H2") ** 2) / 1e6, np.nan)),
                       n_H2_tot=g(_C22, "n_H2_tot"))                                                              # He x 0.5 M / pi R50^2 [Msun/pc^2]
        return pd.DataFrame(out)

    # ── the chains, the dose and the de-duplication: the same three steps for every box ──
    def _chains22(box, want=None):
        """The main-branch chains of one box's histories cache, keyed (anchor_snap, gal_id): the AGN clock, the dust ratio, the
        sSFR and f_gas along the branch, plus the branch fingerprint (log M* at three fixed times) the de-duplication uses.
        `want` restricts to a set of keys — the coarse boxes hold thousands of chains per anchor."""
        CH, FP = {}, {}
        with h5py.File(HIST_FILE.format(box=box), "r") as f:
            for k in f.keys():
                g, a = f[k], int(k[4:])
                t = g["t_yr"][:] / 1e9; o = np.argsort(t); t = t[o]
                gids = np.asarray(g["gal_ids"][:], dtype=np.int64)
                js = [j for j, gid in enumerate(gids) if want is None or (a, int(gid)) in want]
                if not js:
                    continue
                ms, mg, mb, fe = (g[c][:][o] for c in ("mstar", "mgas", "mbh", "fedd"))
                sf = g["sfr"][:][o] if "sfr" in g else np.full_like(ms, np.nan)
                md = g["mdust"][:][o] if "mdust" in g else np.full_like(ms, np.nan)
                mh = g["mh2"][:][o] if "mh2" in g else np.full_like(ms, np.nan)
                S, _, X = agn_states(ms, mg, mb, fe)
                dt = np.gradient(t)
                for j in js:
                    gid, ex = int(gids[j]), X["exist"][:, j]
                    if ex.sum() < 3:
                        continue
                    xr = (np.nan_to_num(S["xray"][:, j]) > 0.5) & ex
                    with np.errstate(all="ignore"):
                        rat = lambda num, den: np.log10(np.where((num > 0) & (den > 0), num / np.where(den > 0, den, np.nan), np.nan))
                        fd = {"lfd": rat(md[:, j], ms[:, j]), "dg": rat(md[:, j], mg[:, j]), "dh2": rat(md[:, j], mh[:, j])}[P22_DUST]
                        ss = np.log10(np.where((sf[:, j] > 0) & (ms[:, j] > 0), sf[:, j] / np.where(ms[:, j] > 0, ms[:, j], np.nan), np.nan))
                    last = np.maximum.accumulate(np.where(xr, t, -np.inf))       # the last X-ray snapshot up to each step
                    tsx = np.where(np.isfinite(last), t - last, np.nan)          # time since the last X-ray episode, along the branch
                    CH[(a, gid)] = dict(t=t, ex=ex, xr=xr, fd=fd, dt=dt, tsx=tsx, ssfr=ss, fgas=X["fgas"][:, j])
                    FP[(a, gid)] = tuple(np.round(np.log10(np.interp([0.8, 1.5, 2.5], t[ex], ms[ex, j],
                                                                     left=np.nan, right=np.nan)), 5))   # the branch fingerprint
        return CH, FP

    def _dose22(CH, keys, t_sft):
        """The AGN dose block of a set of anchors, from their chains: E_x, f_x, f_x_post (the duty fraction SINCE SFT), the
        state at the anchor, t_quiet, f_gas at the anchor and the dust ratio there."""
        rows = []
        for k, ts in zip(keys, np.asarray(t_sft, float)):
            d = CH[k]; t, ex, xr, dtc = d["t"], d["ex"], d["xr"], d["dt"]
            ka = int(np.flatnonzero(ex)[-1])
            post = ex & (t >= ts) if np.isfinite(ts) else ex
            Tp, Ta = float(np.sum(dtc[post])), float(np.sum(dtc[ex]))
            rows.append(dict(e_x=float(np.sum(dtc[xr])), f_x=float(np.sum(dtc[xr]) / Ta) if Ta > 0 else np.nan,
                             f_x_post=float(np.sum(dtc[xr & post]) / Tp) if Tp > 0 else np.nan,
                             xray_now=float(xr[ka]), t_quiet=float(t[ka] - t[xr][-1]) if xr.any() else np.nan,
                             f_gas_anchor=float(d["fgas"][ka]), dust=float(d["fd"][ka])))
        return pd.DataFrame(rows)

    def _dedup22(A, FP):
        """One chain per galaxy: anchors sharing a main-branch fingerprint are the same galaxy — keep the LATEST anchor."""
        A = A.copy(); A["fp"] = [FP[k] for k in A["key"]]
        return A.sort_values("anchor_snap").groupby("fp", as_index=False).tail(1).reset_index(drop=True)

    def _fan22(A, CH):
        """The dust track of every anchor on its own quenching clock t - t_QT, interpolated onto P22_TG."""
        TR = []
        for _, r in A.iterrows():
            d = CH[r["key"]]; ok = d["ex"] & np.isfinite(d["fd"])
            TR.append(np.interp(P22_TG, d["t"][ok] - r["t_qt_gyr"], d["fd"][ok], left=np.nan, right=np.nan)
                      if (ok.sum() >= 3 and np.isfinite(r["t_qt_gyr"])) else np.full(len(P22_TG), np.nan))
        return np.asarray(TR, float).reshape(len(A), len(P22_TG))

    # ── the m25 frame Figs 2-3 and Part 23 build on: the KS-track anchors, the KS-track geometry, one chain per galaxy ──
    _CH22, _fp22 = _chains22(P22_BOX)
    ANC22 = EPO22[EPO22["stage"] == "anchor"][["key", "snap", "gx", "gal_id", "anchor_snap", "z", "t_sft_gyr", "t_qt_gyr",
                                               "log_mstar_anchor"]].copy()
    ANC22 = ANC22[[k in _CH22 for k in ANC22["key"]]].copy()
    _n_rows22 = len(ANC22)
    ANC22 = _dedup22(ANC22, _fp22)                                                       # one per galaxy
    ANC22 = pd.concat([ANC22.reset_index(drop=True), _geom22(ANC22["snap"], ANC22["gx"]).reset_index(drop=True)], axis=1)
    ANC22 = pd.concat([ANC22, _dose22(_CH22, ANC22["key"], ANC22["t_sft_gyr"])], axis=1)
    _dose_v, _geom_v = ANC22[P22_DOSE].to_numpy(float), ANC22[P22_GEOM].to_numpy(float)
    _nh2_v = ANC22["n_H2_tot"].to_numpy(float)
    if P22_NH2MIN > 0:                                   # the resolution floor: too few H2 particles -> no geometry, no group
        _geom_v = np.where(_nh2_v >= P22_NH2MIN, _geom_v, np.nan)
    _dust_v = ANC22["dust"].to_numpy(float)
    _glab, _ghi, _glo = _glab22(P22_GEOM, P22_GCUT, _geom_v)
    _dcut = float(np.nanmedian(_dose_v))
    _gcut = float(np.nanmedian(_geom_v)) if P22_GCUT == "median" else float(P22_GCUT)
    ANC22["hi_dose"] = _dose_v > _dcut
    ANC22["hi_geom"], ANC22["lo_geom"] = _geom_v > _gcut, _geom_v <= _gcut          # both False where the geometry is missing
    _gq = np.nanquantile(_geom_v[np.isfinite(_geom_v)], [0.0, 1 / 3, 2 / 3, 1.0]); _gq[0] -= 1e-9
    _gterc = np.where(np.isfinite(_geom_v), np.digitize(_geom_v, _gq[1:3]), -1)      # 0 / 1 / 2 = the thirds of the geometry (-1 = none)
    _gcols = ["#2166ac", "#7f7f7f", "#b2182b"]
    _gnames = ["lowest third", "middle third", "highest third"]
    print(f"Part 22 — the picture, {BOX_LABEL.get(P22_BOX, P22_BOX)}: {_n_rows22} quenched anchor rows -> {len(ANC22)} distinct "
          f"galaxies (one chain each, the latest anchor).  AGN axis = {P22_DLAB[P22_DOSE]}, split at {_dcut:.2f};  "
          f"ISM-geometry axis = {P22_GEOM}" + (f" in {P22_CORE}" if P22_GEOM in ("kappa_H2", "kappa_gas", "sigma_H2") else " (galaxy-wide)")
          + f", split at {_gcut:.2f}" + (" (the median)" if P22_GCUT == "median" else "")
          + (f", geometry only above {P22_NH2MIN} H2 particles ({int(np.isfinite(_geom_v).sum())} galaxies)" if P22_NH2MIN > 0 else "") + ".")
    # ── the resolution check of the geometry axis (2026-09-04): the old kappa signal lived in the few-particle galaxies ──
    _k22 = ANC22["kappa_H2"].to_numpy(float)
    print(f"  kappa_rot(H2) in {P22_CORE}: median {np.nanmedian(_k22):.2f}; {100 * np.nanmean(_k22 > 1 / 3):.0f} % above the isotropic "
          f"1/3 (net rotation), {100 * np.nanmean(_k22 > 0.5):.0f} % above 0.5 (rotation-supported). Member H2 particles: median "
          f"{np.nanmedian(_nh2_v):.0f}, {100 * np.nanmean(_nh2_v < P22_NH2CHK):.0f} % below {P22_NH2CHK}.")
    _hdv22 = _dose_v > _dcut
    for _nm22, _sel22 in (("every galaxy", np.ones(len(ANC22), bool)), (f"n_H2 >= {P22_NH2CHK}", _nh2_v >= P22_NH2CHK)):
        _fanv22 = []
        for _h22 in (False, True):
            _m22 = _sel22 & (_hdv22 == _h22) & np.isfinite(_geom_v) & np.isfinite(_dust_v)
            _a22, _b22 = _m22 & (_geom_v > _gcut), _m22 & (_geom_v <= _gcut)
            _fanv22.append((np.nanmedian(_dust_v[_a22]) - np.nanmedian(_dust_v[_b22]))
                          if (_a22.sum() >= P22_NMIN and _b22.sum() >= P22_NMIN) else np.nan)
        _ok22 = _sel22 & np.isfinite(_geom_v) & np.isfinite(_dust_v) & (_nh2_v > 0)
        print(f"  {P22_GEOM} vs {P22_DUST}, {_nm22:14s} (N {int(_ok22.sum()):3d}): Spearman "
              f"{spearmanr(_geom_v[_ok22], _dust_v[_ok22])[0]:+.2f}, with log n_H2 {spearmanr(_geom_v[_ok22], np.log10(_nh2_v[_ok22]))[0]:+.2f}; "
              f"upper side minus lower side of the split: low dose {_fanv22[0]:+.2f}, high dose {_fanv22[1]:+.2f} dex")

    GRP22 = [("lo_hi", (~ANC22["hi_dose"]) & ANC22["hi_geom"],  "#b2182b", "-",  f"low dose $+$ {_ghi}"),
             ("lo_lo", (~ANC22["hi_dose"]) & ANC22["lo_geom"],  "#ef8a62", "--", f"low dose $+$ {_glo}"),
             ("hi_hi", ANC22["hi_dose"] & ANC22["hi_geom"],     "#67a9cf", "--", f"high dose $+$ {_ghi}"),
             ("hi_lo", ANC22["hi_dose"] & ANC22["lo_geom"],     "#2166ac", "-",  f"high dose $+$ {_glo}")]

    # ── FIGURE 1: the fan (a) and the dose (b), one column per box ──
    # The m25 column is ANC22 itself (the KS-track geometry, Fig 2-3's frame) unless P22_LIKE. The coarse boxes have no KS-track
    # products, so their anchors are Part 0's sample (QM, with t_SFT / t_QT from the boxes notebook's classes) and their geometry
    # axis is the catalogue's whole-galaxy kappa_rot (P22_GEOM_CAT) — de-duplicated and dosed exactly as the m25 frame, each
    # box split at ITS OWN medians.
    FIG1_22 = {}
    for _b in P22_BOXES:
        if _b == P22_BOX and not P22_LIKE:
            _A = ANC22.assign(geom=_geom_v); _CHb = _CH22                     # the floored axis, the same split as the frame
            _lab3, _gsrc = (_glab, _ghi, _glo), (f"{P22_GEOM} in {P22_CORE}" if P22_GEOM in ("kappa_H2", "kappa_gas", "sigma_H2")
                                                  else f"{P22_GEOM}, galaxy-wide") + " (KS-track products)"
            print(f"  Fig 1 {_b}: the frame above; geometry = {_gsrc}")
        else:
            if not os.path.exists(HIST_FILE.format(box=_b)):
                print(f"  Fig 1 {_b}: no histories cache ({HIST_FILE.format(box=_b)}) — column skipped"); continue
            if not (HAVE_MD.get(_b, False) and (P22_DUST != "dh2" or HAVE_MH2.get(_b, False))):
                print(f"  Fig 1 {_b}: the histories carry no mdust{' / mh2' if P22_DUST == 'dh2' else ''} — no dust axis, column skipped"); continue
            if not {"t_sft", "t_qt", P22_GEOM_CAT} <= set(QM.columns) or not (QM["box"] == _b).any():
                print(f"  Fig 1 {_b}: QM has no rows for it or lacks t_sft / t_qt / {P22_GEOM_CAT} — column skipped"); continue
            _Q = QM.loc[QM["box"] == _b, ["snap", "gal_id", "z", "t_sft", "t_qt", "age", "log_mstar", P22_GEOM_CAT]].copy()
            _Q["key"] = list(zip(_Q["snap"].astype(int), _Q["gal_id"].astype(int)))
            _CHb, _FPb = (_CH22, _fp22) if _b == P22_BOX else _chains22(_b, want=set(_Q["key"]))
            _A = _Q[[k in _CHb for k in _Q["key"]]].rename(columns={"t_sft": "t_sft_gyr", "t_qt": "t_qt_gyr"}).copy()
            _A["anchor_snap"] = _A["snap"].astype(int)
            _nrow = len(_A)
            _A = _dedup22(_A, _FPb)
            _A = pd.concat([_A.reset_index(drop=True), _dose22(_CHb, _A["key"], _A["t_sft_gyr"])], axis=1)
            _A["geom"] = _A[P22_GEOM_CAT].to_numpy(float)
            _lab3, _gsrc = P22_GLAB_CAT[P22_GEOM_CAT], f"catalogue {P22_GEOM_CAT} (whole galaxy)"
            print(f"  Fig 1 {_b}: {_nrow} quenched anchor rows with a chain -> {len(_A)} distinct galaxies "
                  f"(Part 0's sample); geometry = {_gsrc}")
        _dv, _gv, _uv = _A[P22_DOSE].to_numpy(float), _A["geom"].to_numpy(float), _A["dust"].to_numpy(float)
        if np.isfinite(_dv).sum() < P22_NMIN or np.isfinite(_gv).sum() < P22_NMIN or np.isfinite(_uv).sum() < P22_NMIN:
            print(f"  Fig 1 {_b}: fewer than {P22_NMIN} galaxies with a dose, a geometry and a dust value — column skipped"); continue
        _dc = float(np.nanmedian(_dv))
        _gc = _gcut if (_b == P22_BOX and not P22_LIKE) else float(np.nanmedian(_gv))   # the m25 frame keeps its own cut (P22_GCUT)
        _q3 = np.nanquantile(_gv[np.isfinite(_gv)], [0.0, 1 / 3, 2 / 3, 1.0]); _q3[0] -= 1e-9
        FIG1_22[_b] = dict(A=_A, TR=_fan22(_A, _CHb), dose=_dv, geom=_gv, dust=_uv, dcut=_dc, gcut=_gc,
                           hi_dose=_dv > _dc, hi_geom=_gv > _gc, lo_geom=_gv <= _gc,
                           terc=np.where(np.isfinite(_gv), np.digitize(_gv, _q3[1:3]), -1), lab=_lab3, src=_gsrc)
        print(f"           dose split at {_dc:.2f}, geometry split at {_gc:.2f}")
    _bx22 = [b for b in P22_BOXES if b in FIG1_22]
    if _bx22:
        fig, axs = plt.subplots(2, len(_bx22), figsize=(5.5 * len(_bx22), 9.2), sharey="row", squeeze=False,
                                constrained_layout=True)
        for _j, _b in enumerate(_bx22):
            F1 = FIG1_22[_b]; _gl1, _gh1, _glo1 = F1["lab"]
            _grp = [((~F1["hi_dose"]) & F1["hi_geom"],    "#b2182b", "-",  f"low dose $+$ {_gh1}"),
                    ((~F1["hi_dose"]) & F1["lo_geom"],    "#ef8a62", "--", f"low dose $+$ {_glo1}"),
                    (F1["hi_dose"] & F1["hi_geom"],       "#67a9cf", "--", f"high dose $+$ {_gh1}"),
                    (F1["hi_dose"] & F1["lo_geom"],       "#2166ac", "-",  f"high dose $+$ {_glo1}")]
            # (a) the fan
            ax = axs[0, _j]
            for m, c, ls, lab in _grp:
                cen, lo, hi, n = _band22(F1["TR"][np.asarray(m, bool)])
                ax.fill_between(P22_TG, lo, hi, color=c, alpha=0.18, lw=0)
                ax.plot(P22_TG, cen, ls, color=c, lw=2.6, label=f"{lab}  ({int(np.sum(m))})")
            ax.axvline(0, color="k", lw=1.4, ls=":")
            ax.annotate("quenched", xy=(0, 1.0), xycoords=("data", "axes fraction"), xytext=(3, -11), textcoords="offset points",
                        fontsize=8.5, color="0.35")
            ax.set_xlabel(r"$t - t_{\rm QT}$   [Gyr]")
            if _j == 0:
                ax.set_ylabel(P22_ULAB[P22_DUST])
            ax.legend(frameon=False, fontsize=8.2, loc="lower left", bbox_to_anchor=(-0.01, -0.02)); ax.grid(alpha=0.22, lw=0.5)
            ax.set_title(f"(a) {BOX_LABEL.get(_b, _b)}\n{len(F1['A'])} quenched galaxies, geometry = {_gl1}", fontsize=10)
            # (b) the dust against the dose, equal-count bins INSIDE each third of the geometry
            ax = axs[1, _j]
            for _t3 in range(3):
                _gm = (F1["terc"] == _t3) & np.isfinite(F1["dose"]) & np.isfinite(F1["dust"])
                if _gm.sum() < P22_NMIN:
                    continue
                _q = np.nanquantile(F1["dose"][_gm], np.linspace(0, 1, P22_DBINS + 1)); _q[0] -= 1e-9
                _x, _y, _e = [], [], []
                for _i in range(P22_DBINS):
                    _m = _gm & (F1["dose"] > _q[_i]) & (F1["dose"] <= _q[_i + 1])
                    _c, _s, _n = _pt22(F1["dust"][_m])
                    if np.isfinite(_c):
                        _x.append(float(np.nanmedian(F1["dose"][_m]))); _y.append(_c); _e.append(_s)
                ax.errorbar(_x, _y, yerr=_e, marker="o", ms=6.5, lw=2.2, capsize=3, color=_gcols[_t3],
                            label=_gnames[_t3] + rf"  of {_gl1}")
            ax.set_xlabel(P22_DLAB[P22_DOSE])
            if _j == 0:
                ax.set_ylabel("median " + P22_ULAB[P22_DUST])
            ax.legend(frameon=False, fontsize=8.4, loc="lower left"); ax.grid(alpha=0.22, lw=0.5)
            ax.set_title(f"(b) {BOX_LABEL.get(_b, _b)}", fontsize=10)
        fig.suptitle("Part 22 — (a) the dust on each galaxy's own quenching clock, split by AGN dose x gas geometry: one galaxy "
                     "before quenching, four after;\n(b) the dust against the dose, one line per third of the geometry "
                     f"({P22_DBINS} equal-count bins inside each third)", fontsize=11.5)
        paper_save(fig, "agn_states_dustclock_boxes")
        plt.show()
        for _b in _bx22:
            FIG1_22[_b]["A"].drop(columns=[c for c in ("key", "fp") if c in FIG1_22[_b]["A"].columns]).to_csv(
                os.path.join(OUT, f"agn_states_dustclock_{_b}.csv"), index=False)
    else:
        print("  Fig 1: no box could be drawn.")

    # ── FIGURE 2: the two halves of the dose at the critical points, and the plane they walk in time ──
    _SG22 = EPO22[EPO22["key"].isin(set(ANC22["key"]))].copy()
    _SG22 = pd.concat([_SG22.reset_index(drop=True), _geom22(_SG22["snap"], _SG22["gx"]).reset_index(drop=True)], axis=1)
    _hd22 = {k: v for k, v in zip(ANC22["key"], ANC22["hi_dose"])}
    _qt22 = {k: v for k, v in zip(ANC22["key"], ANC22["t_qt_gyr"])}
    _SG22["hi_dose"] = [_hd22.get(k, np.nan) for k in _SG22["key"]]
    _SG22["dtq"] = _SG22["t_stage_gyr"].to_numpy(float) - np.array([_qt22.get(k, np.nan) for k in _SG22["key"]], float)
    _st_ok = [s for s in P22_STAGES if (_SG22["stage"] == s).sum() >= P22_NMIN]
    _halves = [(False, "#b2182b", "low dose"), (True, "#2166ac", "high dose")]
    fig = plt.figure(figsize=(14.6, 7.6), constrained_layout=True)
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.25])
    for _p, (_c22, _yl, _tt) in enumerate([("sigma_H2", r"$\log\, \Sigma_{\rm H_2}$  [$M_\odot$ kpc$^{-2}$]",
                                            "(a) how much H$_2$ is in the core"),
                                           (P22_GEOM, _glab, P22_GTITLE.get(P22_GEOM, "(b) the geometry axis"))]):
        ax = fig.add_subplot(gs[_p, 0])
        for _k3, (_hd, _col, _lab) in enumerate(_halves):
            _keys = set(ANC22.loc[ANC22["hi_dose"] == _hd, "key"])
            _x, _y, _e = [], [], []
            for _i22, s_ in enumerate(_st_ok):
                _v = _SG22.loc[(_SG22["stage"] == s_) & (_SG22["key"].isin(_keys)), _c22].to_numpy(float)
                _c, _s, _n = _pt22(_v)
                _x.append(_i22 + (0.09 if _k3 else -0.09)); _y.append(_c); _e.append(_s)
            ax.errorbar(_x, _y, yerr=_e, ls="none", marker="o", ms=8, capsize=3.5, lw=2.0, color=_col,
                        label=f"{_lab}  ({int((ANC22['hi_dose'] == _hd).sum())})")
        ax.set_xticks(range(len(_st_ok))); ax.set_xticklabels([s.replace("_", "\n") for s in _st_ok], fontsize=9)
        ax.set_xlim(-0.5, len(_st_ok) - 0.5)
        if "qt" in _st_ok:
            ax.axvline(_st_ok.index("qt"), color="k", lw=1.2, ls=":")
        ax.set_ylabel(_yl); ax.grid(alpha=0.22, lw=0.5, axis="y"); ax.set_title(_tt)
        ax.margins(y=0.14)
        if _p == 0:
            ax.legend(frameon=False, fontsize=9, loc="upper left")
    # (c) the plane: every point of the sample behind, the two walked tracks over it
    ax = fig.add_subplot(gs[:, 1])
    _PL = {"dust": ("sigma_H2", "sigma_dust", r"$\log\, \Sigma_{\rm H_2}$  [$M_\odot$ kpc$^{-2}$]",
                    r"$\log\, \Sigma_{\rm dust}$  [$M_\odot$ kpc$^{-2}$]", "(c) the dust-gas plane, walked in time since quenching"),
           "ks":   ("sigma_H2", "sigma_SFR", r"$\log\, \Sigma_{\rm H_2}$  [$M_\odot$ kpc$^{-2}$]",
                    r"$\log\, \Sigma_{\rm SFR}$  [$M_\odot$ yr$^{-1}$ kpc$^{-2}$]",
                    "(c) the KS plane, walked in time since quenching")}[P22_PLANE]
    _cent = [0.5 * (P22_TQT[_i] + P22_TQT[_i + 1]) for _i in range(len(P22_TQT) - 1)]
    _SC22 = (_SG22 if P22_SCAT_STAGES is None else _SG22[_SG22["stage"].isin(P22_SCAT_STAGES)]).dropna(subset=[_PL[0], _PL[1]])
    _nsc = {}
    for _hd, _col, _lab in _halves:                     # the scatter: every critical point and the anchor of every galaxy in the half
        _s = _SC22[_SC22["key"].isin(set(ANC22.loc[ANC22["hi_dose"] == _hd, "key"]))]
        _nsc[_hd] = len(_s)
        ax.scatter(_s[_PL[0]], _s[_PL[1]], s=12, color=_col, alpha=0.30, lw=0, zorder=1, rasterized=True)
    _walk = {}
    for _hd, _col, _lab in _halves:
        _keys = set(ANC22.loc[ANC22["hi_dose"] == _hd, "key"])
        _w = {}
        for _i22 in range(len(P22_TQT) - 1):
            _sub = _SG22[(_SG22["key"].isin(_keys)) & (_SG22["dtq"] > P22_TQT[_i22]) & (_SG22["dtq"] <= P22_TQT[_i22 + 1])]
            _per = _sub.groupby("key")[[_PL[0], _PL[1]]].median()             # one value per galaxy first
            _per = _per[np.isfinite(_per[_PL[0]]) & np.isfinite(_per[_PL[1]])]
            if len(_per) >= P22_NMIN:
                _w[_i22] = (float(_per[_PL[0]].median()), float(_per[_PL[1]].median()), int(len(_per)))
        _walk[_hd] = _w
    _common = sorted(set(_walk[False]) & set(_walk[True]))      # the same time bins for both halves, or the tracks are not comparable
    _all = np.array([_walk[h][i][:2] for h in (False, True) for i in _common], float).reshape(-1, 2)
    if len(_SC22):                       # the frame FIRST (the boxes below are placed in display space, so the limits must be final):
        _px = np.nanpercentile(_SC22[_PL[0]].to_numpy(float), [1, 99]); _py = np.nanpercentile(_SC22[_PL[1]].to_numpy(float), [1, 99])
        if len(_all):                    # the bulk of the scatter, never tighter than the tracks
            _px = [min(_px[0], _all[:, 0].min() - 0.3), max(_px[1], _all[:, 0].max() + 0.3)]
            _py = [min(_py[0], _all[:, 1].min() - 0.3), max(_py[1], _all[:, 1].max() + 0.3)]
        ax.set_xlim(_px[0] - 0.15, _px[1] + 0.15); ax.set_ylim(_py[0] - 0.15, _py[1] + 0.15)
        _nout = int(((_SC22[_PL[0]] < _px[0] - 0.15) | (_SC22[_PL[0]] > _px[1] + 0.15) | (_SC22[_PL[1]] < _py[0] - 0.15)
                     | (_SC22[_PL[1]] > _py[1] + 0.15)).sum())
        if _nout:
            print(f"  Fig 2 (c): {_nout} of {len(_SC22)} scatter points lie outside the frame (1-99 % of the scatter, padded)")
    if len(_all):
        _gx = np.linspace(np.nanmin(_all[:, 0]) - 0.15, np.nanmax(_all[:, 0]) + 0.15, 8)
        _off = np.nanmedian(_all[:, 1] - _all[:, 0])
        ax.plot(_gx, _gx + _off, ls=(0, (5, 4)), color="0.72", lw=1.1, zorder=0)
        ax.annotate("constant dust-to-gas", xy=(_gx[0], _gx[0] + _off), xytext=(4, 6), textcoords="offset points",
                    fontsize=8, color="0.55", ha="left")
    for _hd, _col, _lab in _halves:
        _xx = [_walk[_hd][_i22][0] for _i22 in _common]; _yy = [_walk[_hd][_i22][1] for _i22 in _common]
        ax.plot(_xx, _yy, "-", color=_col, lw=2.2, alpha=0.9, zorder=3)
        ax.scatter(_xx, _yy, s=np.linspace(150, 45, len(_xx)), color=_col, edgecolor="k", lw=0.6, zorder=4,
                   label=f"{_lab}  ({_nsc[_hd]} points, {int((ANC22['hi_dose'] == _hd).sum())} galaxies)")
        for _i22 in range(len(_xx) - 1):
            ax.annotate("", xy=(_xx[_i22 + 1], _yy[_i22 + 1]), xytext=(_xx[_i22], _yy[_i22]), zorder=3,
                        arrowprops=dict(arrowstyle="-|>", color=_col, lw=1.5, alpha=0.9))
    # the boxes: the time since quenching and the galaxies behind the pair, at the bottom right of each HIGH-DOSE point. Two
    # points a fraction of a dex apart would put their boxes on top of each other, so a box that lands on an earlier one is
    # stacked underneath it (worked out in points, after the layout is final).
    if _common:
        fig.canvas.draw()
        _xxh = [_walk[True][i][0] for i in _common]; _yyh = [_walk[True][i][1] for i in _common]
        _disp = ax.transData.transform(np.column_stack([_xxh, _yyh])) * 72.0 / fig.dpi          # the points, in points
        _BOXW, _BOXH, _GAP = 48.0, 22.0, 2.5                    # the footprint of one box (two lines of 7 pt text + padding)
        _placed = []                                            # (left, right, top, bottom) of every box drawn so far
        for _i22, _b22 in enumerate(_common):
            _left, _top = _disp[_i22, 0] + 9.0, _disp[_i22, 1] - 9.0
            for _ in range(12):                                 # push down until it overlaps none of the boxes already placed
                _hit = [b for b in _placed if _left < b[1] and _left + _BOXW > b[0] and _top > b[3] - _GAP and _top - _BOXH < b[2] + _GAP]
                if not _hit:
                    break
                _top = min(b[3] for b in _hit) - _GAP
            ax.annotate(f"{_cent[_b22]:+.1f} Gyr\nN {_walk[False][_b22][2]} / {_walk[True][_b22][2]}",
                        xy=(_xxh[_i22], _yyh[_i22]), xytext=(9.0, _top - _disp[_i22, 1]), textcoords="offset points",
                        ha="left", va="top", fontsize=7.0, color="#2166ac", zorder=7,
                        bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="#2166ac", lw=0.7, alpha=0.92))
            _placed.append((_left, _left + _BOXW, _top, _top - _BOXH))
    ax.set_xlabel(_PL[2]); ax.set_ylabel(_PL[3])
    ax.legend(frameon=False, fontsize=9, loc="upper left",
              title=r"boxes: $t - t_{\rm QT}$ [Gyr] at the bin centre, N galaxies low / high dose", title_fontsize=8)
    ax.grid(alpha=0.22, lw=0.5); ax.set_title(_PL[4])
    fig.suptitle(rf"Part 22 — the two halves of the dose through their own quenching ({BOX_LABEL.get(P22_BOX, P22_BOX)}, "
                 rf"core = {P22_CORE})", fontsize=12.5)
    paper_save(fig, f"agn_states_dustsplit_{P22_BOX}")
    plt.show()

    # ── FIGURE 3: the rejuvenation test — the SAME walked plane as Fig 2 (c), stepped through the CRITICAL POINTS ──
    # Fig 2 (c) walks the dust-gas plane in bins of t - t_QT; this one walks the very same plane through the critical points
    # jet_on -> sft -> qt -> anchor (P22_SSTAGE, in time order), one panel per bin of the galaxy's OWN global sSFR
    # (sfr100_tot / m_star_tot) AT THAT EPOCH, so a galaxy moves between panels as it quenches. If the dusty galaxies were
    # simply the ones still (or again) forming stars, the two halves of the dose would separate ACROSS the panels and lie on
    # top of each other INSIDE them. Each point is the median galaxy at that critical point (one value per galaxy first) with
    # its bootstrap error, the arrows give the direction of time, and the grey dashed line — the SAME line in every panel —
    # is constant dust-to-gas, so the vertical distance from it is the dust-to-gas ratio itself.
    S3_22 = _SG22[_SG22["stage"].isin(P22_SSTAGE)].dropna(subset=["sigma_H2", "sigma_dust"]).copy()
    # the panel is a property of the GALAXY, not of the epoch: its global sSFR AT THE ANCHOR, so one track stays in one panel.
    # A galaxy with a measured M* but no star formed in the last 100 Myr has no log sSFR at all — it is not missing data, it is
    # the deadest galaxy there is, so it goes to the bottom panel (-inf) instead of being dropped; the edges are quantiles of
    # the galaxies that DO have a measurable sSFR, so they stay finite whatever fraction of the sample is undetected.
    _ixa22 = list(zip(ANC22["snap"].astype(int), ANC22["gx"].astype(int)))
    _gcol22 = lambda c: (_C22[c].reindex(_ixa22).to_numpy(float) if c in _C22.columns else np.full(len(ANC22), np.nan))
    _sfa22, _msa22 = _gcol22("sfr100_tot"), _gcol22("m_star_tot")
    _lsa22 = ANC22["lssfr_tot"].to_numpy(float).copy()
    _nod22 = (~np.isfinite(_lsa22)) & (_msa22 > 0) & np.isfinite(_sfa22) & (_sfa22 <= 0)
    _lsa22[_nod22] = -np.inf
    _fin22 = np.isfinite(_lsa22)
    _sed22 = np.asarray(P22_SEDGES, float) if P22_SEDGES is not None else np.nanquantile(_lsa22[_fin22], [1 / 3, 2 / 3])
    _sed22 = np.atleast_1d(_sed22)[np.isfinite(np.atleast_1d(_sed22))] if _fin22.sum() >= 3 else np.array([])
    _drw22 = _fin22 | np.isneginf(_lsa22)                       # the galaxies that get a panel at all
    _pan22 = np.where(_drw22, np.digitize(np.where(_drw22, _lsa22, 0.0), _sed22), -1)
    _PAN22 = {k: int(p) for k, p in zip(ANC22["key"], _pan22)}
    S3_22["panel"] = [_PAN22.get(k, -1) for k in S3_22["key"]]
    _pe22 = np.concatenate([[-np.inf], _sed22, [np.inf]])
    MRK3 = {"sf_peak": "*", "agn_ign": "^", "jet_on": "v", "sft": "s", "qt": "D", "post_quench": "P", "gas_min": "X",
            "anchor": "o"}
    LAB3 = {"sf_peak": "SF peak", "agn_ign": "ignition", "jet_on": "jet", "sft": "SFT", "qt": "QT",
            "post_quench": "post-q", "gas_min": "gas min", "anchor": "anchor"}
    _np22 = len(_pe22) - 1
    _W3 = {}
    for _p22 in range(_np22):
        for _hd, _col, _lab in _halves:
            _keys = set(ANC22.loc[ANC22["hi_dose"] == _hd, "key"])
            _g22 = S3_22[(S3_22["panel"] == _p22) & (S3_22["key"].isin(_keys))]
            _w = {}
            for _st22 in P22_SSTAGE:                                  # one row per galaxy per stage, so the median is per galaxy
                _gs22 = _g22[_g22["stage"] == _st22]
                _cx, _ex22, _n22 = _pt22(_gs22.groupby("key")["sigma_H2"].median().to_numpy(float), P22_NMIN3)
                _cy, _ey22, _ = _pt22(_gs22.groupby("key")["sigma_dust"].median().to_numpy(float), P22_NMIN3)
                if np.isfinite(_cx) and np.isfinite(_cy):
                    _w[_st22] = (_cx, _cy, _ex22, _ey22, _n22)
            _W3[(_p22, _hd)] = _w
    _pts3 = np.array([v[:2] for _k3 in _W3 for v in _W3[_k3].values()], float).reshape(-1, 2)
    fig, axs = plt.subplots(1, _np22, figsize=(5.0 * _np22, 5.0), sharex=True, sharey=True, constrained_layout=True)
    axs = np.atleast_1d(axs)
    if len(_pts3) >= 2:                                               # ONE constant-dust-to-gas line for all panels: they compare
        _gx3 = np.linspace(_pts3[:, 0].min() - 0.35, _pts3[:, 0].max() + 0.35, 8)
        _off3 = float(np.median(_pts3[:, 1] - _pts3[:, 0]))
    else:
        _gx3, _off3 = np.linspace(0, 1, 8), 0.0
    print(f"\n  the rejuvenation test — the dust-gas plane walked through the critical points, the galaxies split by their own "
          f"sSFR AT THE ANCHOR\n  ({int(_nod22.sum())} of {len(ANC22)} formed no star in the last 100 Myr there and sit in "
          f"panel a; {int((_pan22 < 0).sum())} have no anchor measurement and are dropped).\n"
          f"  log Sigma_dust/Sigma_H2 of the median galaxy, N in brackets:")
    for _p22 in range(_np22):
        ax = axs[_p22]
        ax.plot(_gx3, _gx3 + _off3, ls=(0, (5, 4)), color="0.72", lw=1.1, zorder=0)
        for _hd, _col, _lab in _halves:
            _w = _W3[(_p22, _hd)]
            _st3 = [s_ for s_ in P22_SSTAGE if s_ in _w]
            _xx = np.array([_w[s_][0] for s_ in _st3], float); _yy = np.array([_w[s_][1] for s_ in _st3], float)
            if len(_st3) >= 2:
                ax.plot(_xx, _yy, "-", color=_col, lw=2.2, alpha=0.85, zorder=4)
                for _i22 in range(len(_st3) - 1):            # the head at the MIDPOINT: the markers are large and would hide it
                    ax.annotate("", xy=(0.5 * (_xx[_i22] + _xx[_i22 + 1]), 0.5 * (_yy[_i22] + _yy[_i22 + 1])),
                                xytext=(_xx[_i22], _yy[_i22]), zorder=5,
                                arrowprops=dict(arrowstyle="-|>", color=_col, lw=1.6, alpha=0.9, mutation_scale=16,
                                                shrinkA=6, shrinkB=0))
            for _i22, _s3 in enumerate(_st3):
                ax.errorbar(_xx[_i22], _yy[_i22], xerr=_w[_s3][2], yerr=_w[_s3][3], marker=MRK3.get(_s3, "o"), ms=11,
                            color=_col, mec="k", mew=0.7, capsize=3, lw=1.5, zorder=6)
                if not _hd:                                           # the stage order is the same on both tracks: label one
                    ax.annotate(LAB3.get(_s3, _s3), xy=(_xx[_i22], _yy[_i22]), xytext=(8, -12), textcoords="offset points",
                                fontsize=8.2, color=_col)
            print(f"    panel {chr(97 + _p22)}  {_lab:9s}: "
                  + (",  ".join(f"{LAB3.get(s_, s_)} {_w[s_][1] - _w[s_][0]:+.2f} ({_w[s_][4]})" for s_ in _st3)
                     if _st3 else f"nothing above N = {P22_NMIN3}"))
        _lo22 = "" if not np.isfinite(_pe22[_p22]) else f"{_pe22[_p22]:.2f} < "
        _hi_tex = "" if not np.isfinite(_pe22[_p22 + 1]) else r" $\leq$ " + f"{_pe22[_p22 + 1]:.2f}"
        _ngp22 = int((_pan22 == _p22).sum())
        _nod_t = f", {int((_nod22 & (_pan22 == _p22)).sum())} of them with no detected SF" if _p22 == 0 else ""
        ax.set_title(f"({chr(97 + _p22)})  {_lo22}" + r"$\log$ sSFR$_{\rm anchor}$" + f"{_hi_tex}"
                     + f"\n{_ngp22} galaxies{_nod_t}", fontsize=10.5)
        ax.set_xlabel(r"$\log\, \Sigma_{\rm H_2}$  [$M_\odot$ kpc$^{-2}$]"); ax.grid(alpha=0.22, lw=0.5)
    axs[0].set_ylabel(r"$\log\, \Sigma_{\rm dust}$  [$M_\odot$ kpc$^{-2}$]")
    _h3 = [Line2D([], [], color=c, lw=2.2, marker="o", ms=8, mec="k", mew=0.6, label=l) for _, c, l in _halves]
    _h3 += [Line2D([], [], ls=(0, (5, 4)), color="0.72", lw=1.1, label="constant dust-to-gas")]
    axs[0].legend(handles=_h3, frameon=False, fontsize=8.6, loc="upper left")
    _h3b = [Line2D([], [], ls="", marker=MRK3[s_], ms=8, mfc="0.6", mec="k", mew=0.5, label=LAB3.get(s_, s_))
            for s_ in P22_SSTAGE if s_ in MRK3]
    axs[-1].legend(handles=_h3b, frameon=False, fontsize=8, loc="lower right", ncol=2, title="critical point",
                   title_fontsize=8)
    fig.suptitle(r"Part 22 — is the dust just rejuvenation? the dust-gas plane walked through the critical points, the "
                 rf"galaxies split by their sSFR at the anchor ({BOX_LABEL.get(P22_BOX, P22_BOX)}, core = {P22_CORE})",
                 fontsize=12.5)
    paper_save(fig, f"agn_states_dustgas_ssfr_{P22_BOX}")
    plt.show()

    ANC22.drop(columns=["key", "fp"]).to_csv(os.path.join(OUT, f"agn_states_dustclock_{P22_BOX}.csv"), index=False)
    _fg22 = ANC22["f_gas_anchor"].to_numpy(float)
    _n_out, _n_all, _n_rot = 0, 0, 0                     # why "time since the last X-ray episode" is not an axis anywhere here
    for _i22, _r22 in ANC22.iterrows():
        _d = _CH22[_r22["key"]]
        if not _d["xr"].any() or not np.isfinite(_r22["t_qt_gyr"]):
            continue
        _m = _d["ex"] & np.isfinite(_d["tsx"]) & (_d["t"] >= _r22["t_qt_gyr"])
        _n_all += int(_m.sum()); _n_out += int((_m & (_d["tsx"] > 0)).sum())
        _n_rot += int((_m & (_d["tsx"] > 0)).sum()) if _gterc[_i22] == 2 else 0
    print(f"  \"time since the last X-ray episode\" is thin by construction: {100 * np.nanmean(ANC22['xray_now']):.0f} % of them "
          f"are IN the X-ray state at the anchor, and over the post-quenching snapshots only "
          f"{100 * _n_out / max(1, _n_all):.0f} % are out of it — {100 * _n_rot / max(1, _n_out):.0f} % of those belong to the top "
          f"third of {P22_GEOM}, so getting OUT of the state is something the galaxies at that end of the geometry axis manage.")
    print(f"  the X-ray gate is not what separates them: {100 * np.nanmean(_fg22 < XRAY_FGAS_MAX):.0f} % of these quenched "
          f"galaxies are already below f_gas = {XRAY_FGAS_MAX:g} at the anchor (median {np.nanmedian(_fg22):.3f}), and the dose "
          f"axis is flat in it (Spearman {spearmanr(_dose_v, np.log10(np.where(_fg22 > 0, _fg22, np.nan)), nan_policy='omit')[0]:+.2f}).")
    _lo_end = np.nanmedian(ANC22.loc[~ANC22["hi_dose"], "dust"]); _hi_end = np.nanmedian(ANC22.loc[ANC22["hi_dose"], "dust"])
    print(f"  the two halves of the dose end {_lo_end - _hi_end:+.2f} dex apart in {P22_DUST} ({_lo_end:.2f} vs {_hi_end:.2f}); "
          f"per galaxy -> {os.path.join(OUT, f'agn_states_dustclock_{P22_BOX}.csv')}")

## Part 23 — the same division in the resolved products: the radial anatomy and the K-S plane

Part 22 divides the quenched galaxies by two catalogue numbers — the X-ray duty fraction since SFT and the ISM-geometry axis `P22_GEOM` (by default the H$_2$ extent relative to the stars, $-\log R_{50}({\rm H_2})/R_{50}(\star)$: *compact* against *extended* H$_2$) — and shows the dust follows that division. This part puts the **same four groups** into the m25 *particle-level* products, the ones that correspond to what a telescope measures. **It is not a pure read of the `paper_ism_prediction_boxes` products**: it needs the m25 notebook's own tables and the `ks_tracks_quenched_m25` run, and each block gates on its file and prints what is missing.

The division is re-derived for **every quenched anchor row** (`P23_ALLANCH`), not Part 22's one-per-galaxy set, because the resolved tables are keyed on the anchor — but with Part 22's cuts (`_gcut`, `_dcut`, `P22_NH2MIN`), not re-medianed, so the groups mean the same thing.

`agn_states_dosegeom_profiles_cis25` — **the radial anatomy**, per annulus (0-1, 1-3.2, 3.2-10, 10-32 kpc), each galaxy contributing the median over its sightlines, the m25 star-forming galaxies in grey:

* $A_V$, $M_{\rm dust}/M_\star$ and $M_{\rm dust}/M_{\rm H_2}$ separate the **low-dose, upper-side** group from the other three by an order of magnitude *in the inner kpc* and converge outwards — the dust that distinguishes them is a **core** phenomenon (this was the reading with the $\kappa$ axis; re-run for the extent axis).
* $M_{\rm H_2}/M_\star$ does **not** separate them at any radius. Same molecular gas, different dust, resolved this time.
* The **resolved sSFR** — `P23_SFRCOL` (`sfr100`, the 100 Myr window, the same convention as the K-S plane; `sfr25` is the other) over the *same region's* $M_\star$ — answers the obvious objection to the dust panels: whether the group that keeps its dust is simply the one still forming stars, radius by radius rather than in the integrated numbers. An annulus with no star formation at all sits **on** the `P23_SSFRMIN` floor, drawn as a dotted line marked *no SF*, instead of vanishing off a log axis; Part 0's `SSFR_FLOOR` $= 10^{-15}$ would stretch the panel over five decades to show one dead annulus. The medians and their bootstrap bands go to the CSV like every other panel.
* $\kappa_{\rm rot}^\star$ is flat and identical for all four groups: the stellar body is *not* what differs. The $\kappa$ panels now carry the **isotropic value $1/3$** (dashed) and $0.5$ (dotted, rotation-supported) as their reference lines: every quenched group sits *below* $1/3$ in $\kappa_{\rm rot}^{\rm H_2}$ — radial motion, not rotation. With `P22_GEOM = "kappa_H2"` the core $\kappa^{\rm H_2}$ panel is partly circular (it is then the split variable) and the cell says so; with the extent axis no annulus panel is.

`agn_states_dosegeom_ks_cis25` — **the K-S plane, with the ALMA-C11 resolution applied**, one panel. The **intrinsic tracks are the solid ones** (`P23_KS_MAIN = "intrinsic"`: $R = R_{50}({\rm H_2})$, the physics); where ALMA-C11 *would measure* each group is the faded, dashed overlay (`P23_KS_OVERLAY`), with a slope-1 arrow on a stage point when more than half of its galaxies come out unresolved.

* **The resolution rule (2026-09-04, corrected the same day).** The observed $\Sigma$ are $0.5 M / \pi R^2$ with $R$ the CO(3-2) size from a *uv-plane* Gaussian fit, and a uv fit measures **below the beam**: the smallest size the ALMA-C11 fits recovered is 1.6 kpc (hc9) against beam half-widths $b_{\rm maj}/2$ of 3.3-4.5 kpc (beams of 1.3-1.6 arcsec at $z = 0.35$-$0.42$). Only when the fit fails does a source get $R < b_{\rm maj}/2$ and both $\Sigma$ become lower limits (the open observed points). So the rule is **two-tier**: a simulated galaxy with $R_{50}({\rm H_2}) \geq$ `P23_UV_MIN_KPC` (1.6 kpc) keeps its own size, one below it gets the `P23_BEAM_KPC` limit (3.8 kpc, the median $b_{\rm maj}/2$ of the three unresolved sources). The first version used $R = \max(R_{50}, 3.8)$ — every galaxy treated as if its fit had failed — and that collapsed all four groups onto one point (0.67-0.80 in $\log \Sigma_{\rm H_2}$); `P23_UV_MIN_KPC = None` gives that one-tier rule back. Under the two-tier rule the groups keep their order and the **compact groups are the ones that come out unresolved** — a prediction for the sample:

  | group at `end` | intrinsic $\log \Sigma_{\rm H_2}$ | as observed | unresolved |
  |---|---|---|---|
  | low dose + compact | 1.84 | 0.77 | 66 % |
  | low dose + extended | 1.35 | 1.16 | 20 % |
  | high dose + compact | 1.57 | 1.09 | 43 % |
  | high dose + extended | 1.26 | 1.23 | 9 % |

  The simulated anchors run to $z = 2$, where 1 arcsec is 8 kpc rather than 4.6, so a fixed *angular* beam would penalise the higher-$z$ anchors more, not less; the correction that matters is the uv-fit tier, not the beam value.
* **The galaxies larger than the beam** (`P23_TAIL`): every quenched galaxy with $R_{50}({\rm H_2}) >$ `P23_BEAM_KPC` at `end` is drawn as its own cross, coloured by group, and listed with its $R_{50}$, $\kappa_{\rm rot}$ and group. These few (six of 263 anchor rows above 4.5 kpc, none above 8 kpc) are the simulation's counterparts of the observed rotating CO discs of 8-9 kpc (hc3, hc5, hc7, the bottom-left points, all dust-detected), and they *are* the rotating ones ($\kappa$ 0.6-0.8, thousands of H$_2$ particles): in the simulation as in the data, **where rotation is measurable, rotating = extended = low $\Sigma_{\rm H_2}$**. Their scarcity is the result — SIMBA's quenched galaxies keep compact nuggets, the ALMA-C11 QGs hold $10^{9.6-10.1}\,M_\odot$ discs.
* **What the H$_2$ convention can and cannot do.** $\alpha_{\rm CO}$ and $r_{31}$ move *every* observed point sideways by the same amount ($\alpha_{\rm CO}$ 4.36 → 0.8: $-0.74$ dex; $r_{31}$ 0.5 → 0.7 / 1.0: $-0.15$ / $-0.30$ dex), so they cannot reorder the observed rotating and non-rotating sources; a size upper limit moves a point up-right along slope 1, parallel to Bigiel+08 (slope 0.96), which preserves the offset from the relation and pushes the *compact* sources further right. Neither can produce the old plot's sign, where the simulated 'rotating' groups ended to the right of the others — that came from the $\kappa$ split (Part 22's section on the geometry axis), and with the extent axis the compact group ends to the right *by construction* ($\Sigma_{\rm H_2} = M/2\pi R^2$ at fixed mass) — which is exactly what the resolution rule turns into "the compact ones are the unresolved ones".
* Only the **binned tracks** (per-group stage medians through the **same critical points as Part 22's Fig 3**, `jet_on` $\to$ `sft` $\to$ `qt` $\to$ `end`, with arrows for the direction of time; `end` is the anchor as the K-S products define it, the last snapshot still holding H$_2$) against **one** relation, the molecular Bigiel+08; the total-gas K98 is deliberately not drawn. Put `"sf_peak"` / `"agn_ign"` back into `P23_KS_STG` to see the pre-quenching burst itself. The frame (`P23_KS_XLIM` $= 0$-$4$, `P23_KS_YLIM` $= -3.2$-$2$) is opened to the left for the as-observed end points.
* The offset from that relation at each critical point — the test of *were they post-starbursts?* — is **printed and written to the CSV** on the intrinsic apertures (a slope-1 shift changes the offset from a slope-0.96 relation by 4 % of the shift, so the resolution rule is immaterial there). With the $\kappa$ split all four groups sat $\approx 1$ dex **above** B08 through `jet_on` and `sft`, the lower-$\kappa$ groups quenched faster ($\tau_q$ 0.21-0.22 Gyr against 0.28-0.31), and by `end` the low-dose groups were back *on* the relation while the high-dose ones had fallen *below* it; re-run for the extent split.

**Knobs.** `P23_ALLANCH`, `P23_ANNULI`, `P23_SFRCOL` / `P23_SSFRMIN` (the sSFR panel), `P23_KS_AP`, `P23_KS_STG`, `P23_KS_SFR`, `P23_BEAM_KPC` / `P23_UV_MIN_KPC` (the two-tier resolution rule), `P23_KS_MAIN` / `P23_KS_OVERLAY` (which track is solid, whether the other is drawn), `P23_TAIL` (the galaxies larger than the beam), `P23_KS_XLIM` / `P23_KS_YLIM` (the frame), `P23_REL` (the one relation drawn), `P23_SFCLOUD` (re-add the star-forming point cloud), `P23_NMIN`, and `P23_M25` / `P23_OBS` for the product and observation paths.


In [ ]:
# ── Part 23 — the dose x geometry division taken into the m25 PARTICLE-LEVEL products: the radial anatomy and the K-S plane ──
# Part 22 divides the quenched galaxies by two catalogue-level numbers — the X-ray duty fraction since SFT and kappa_rot(H2) of
# the core — and shows the dust follows that division. This part asks the same question of the resolved products, which are the
# ones a telescope would see: (A) the radial anatomy per annulus (A_V, M_dust/M*, M_H2/M*, sSFR, D/H2, stellar age, kappa_rot) with the
# m25 star-forming galaxies as the grey reference, and (B) the K-S plane at r < R50(H2) with the ALMA-C11 quiescent galaxies on
# it, ONE panel cropped to where the tracks and the observations are (P23_KS_XLIM / P23_KS_YLIM). (C) answers the question the
# fan raises: the dispersion-supported galaxies carry MORE dust just before quenching and lose it fastest — were they
# post-starbursts? On the K-S plane that is a testable statement, so (C) measures the offset from the reference relation at each
# critical point (a burst sits ABOVE it before quenching) — PRINTED and written to the CSV, no longer a panel. Both blocks step
# through the SAME critical points as Part 22's Fig 3 (jet_on -> sft -> qt -> end), and the tracks carry the direction of time
# as arrows.
# NOT a pure read of the ism_prediction products: this needs the m25 notebook's own tables and the ks_tracks_quenched_m25 run.
# Every block gates on its file and prints what is missing, so the cell runs unchanged off the cluster.
P23_BOX     = "cis25"
P23_M25     = globals().get("P23_M25_OVERRIDE", os.path.join(os.getcwd(), "output", P23_BOX))
P23_OBS     = globals().get("P23_OBS_OVERRIDE", os.path.join(os.getcwd(), "obs_data", "almac11", "ks_table.csv"))   # the ALMA-C11 K-S table (see obs_data/almac11/README)
P23_NMIN    = 5             # galaxies a drawn point needs (below the notebook's NMIN: a dose x geometry cell is ~40 galaxies)
P23_ANNULI  = [("ap1kpc", "0-1"), ("ann3kpc", "1-3.2"), ("ann10kpc", "3.2-10"), ("ann32kpc", "10-32")]
P23_NSTAR   = 20            # the m25 Part 7e sampling floor on an annulus
P23_SFRCOL  = "sfr100"      # the SFR column of aperture_truth.fits behind the resolved sSFR panel ("sfr25" is the
                            # other one); sSFR = that SFR over the SAME region's M*
P23_SSFRMIN = 1e-13         # an annulus with no star formation sits ON this floor (drawn as a dotted line) instead
                            # of vanishing off a log axis; Part 0's SSFR_FLOOR = 1e-15 would stretch the panel over
                            # five decades to show one dead annulus
P23_KS_AP   = "R50_H2"      # the K-S aperture: 0.5 M / pi R50(H2)^2, the convention the observed points are measured in
P23_KS_STG  = ["jet_on", "sft", "qt", "end"]                 # the critical points of the K-S track — Part 22's Fig 3 set ('end' is the
                                                             # anchor as the K-S products define it: the last snapshot still holding H2).
                                                             # Add "sf_peak" / "agn_ign" back to see the pre-quenching burst itself
P23_KS_SFR  = "logSigmaSFR"                                  # the SFR surface density column of ks_tracks.fits (100 Myr, observed convention)
P23_REL     = "B08"         # the ONLY relation drawn (the H2 one). K98 is a total-gas relation and is deliberately not shown
P23_KS_XLIM = (0.0, 4.0)    # the K-S frame, log Sigma_H2: where the beam-matched tracks and the observed points are
P23_KS_YLIM = (-3.2, 2.0)   # ... and log Sigma_SFR
P23_SFCLOUD = False         # True re-adds the m25 star-forming galaxies as a grey point cloud behind the tracks
P23_ALLANCH = True          # True = every quenched anchor row gets a group (the resolved tables are per anchor); False = Part 22's one-per-galaxy set
P23_BEAM_KPC = 3.8          # the ALMA-C11 beam half-width [kpc]: the size limit (R < b_maj/2) an UNRESOLVED source gets, and with it the Sigma
                            # lower limit 0.5 M / pi (b_maj/2)^2 — exactly what the open observed points are. 3.8 = the median b_maj/2 of
                            # the three unresolved ALMA-C11 sources (3.29 / 3.80 / 4.52 kpc, i.e. beams of 1.3-1.6 arcsec at z = 0.35-0.42,
                            # obs_data/almac11/README). None = no "as observed" positions at all
P23_UV_MIN_KPC = 1.6        # the smallest size the ALMA-C11 uv-plane fits recovered (hc9, 1.6 kpc): a uv fit measures BELOW the beam, so a
                            # simulated galaxy with R50(H2) at or above this keeps its own size; below it the fit would fail and the
                            # observation returns the P23_BEAM_KPC limit (two-tier rule). None = the one-tier rule R = max(R50, beam),
                            # which treats every galaxy as if its fit had failed (too conservative — the 2026-09-04 first version)
P23_KS_MAIN = "intrinsic"   # the track drawn solid: "intrinsic" (R = R50(H2), the physics) | "observed" (where ALMA-C11 would place it)
P23_KS_OVERLAY = True       # draw the other representation too, faded and dashed
P23_TAIL    = True          # the extended-H2 tail: every quenched galaxy LARGER than the beam at 'end' (R50(H2) > P23_BEAM_KPC), its own point

ANN23 = os.path.join(P23_M25, "tables", "annulus_ism_truth.fits")            # m25 Part 8a: M_dust / M_gas / M_H2 per (annulus, sightline)
APT23 = os.path.join(P23_M25, "tables", "aperture_truth.fits")               # m25 Part 7e: M*, age per (annulus, sightline)
AV23  = os.path.join(P23_M25, "tables", "annulus_av_allincl.fits")           # m25 Part 8b: A_V per (annulus, sightline) — optional panel
KIN23 = os.path.join(P23_M25, "tables", "annulus_kinematics.fits")           # m25 Part 8j0: kappa_rot per spherical zone — optional panels
SEL23 = os.path.join(P23_M25, "tables", "powderday_quenched_selection.fits") # m25 Part 3: pop -> the SF-controls line — optional
KS23  = os.path.join(P23_M25, "ks_tracks_pt", "ks_tracks.fits")              # ks_tracks_quenched_m25 Part 4a (run tag 'pt')

if "ANC22" not in globals():
    print("Part 23: needs Part 22 — run it first (it defines the dose x geometry division this part carries into the products).")
else:
    from matplotlib.lines import Line2D as _L23
    import re as _re23
    # ── the division, re-derived for EVERY quenched anchor row (the resolved tables are keyed on the anchor) ──
    A23 = EPO22[EPO22["stage"] == "anchor"][["key", "snap", "gx", "gal_id", "anchor_snap", "z", "t_sft_gyr", "t_qt_gyr",
                                             "tau_q_gyr"]].copy()
    A23 = A23[[k in _CH22 for k in A23["key"]]].reset_index(drop=True)
    if not P23_ALLANCH:
        A23 = A23[A23["key"].isin(set(ANC22["key"]))].reset_index(drop=True)
    A23 = pd.concat([A23, _geom22(A23["snap"], A23["gx"]).reset_index(drop=True)], axis=1)
    _d23 = []
    for _, _r23 in A23.iterrows():
        _d = _CH22[_r23["key"]]; _t, _ex, _xr, _dtc = _d["t"], _d["ex"], _d["xr"], _d["dt"]
        _post = _ex & (_t >= _r23["t_sft_gyr"]) if np.isfinite(_r23["t_sft_gyr"]) else _ex
        _Tp = float(np.sum(_dtc[_post]))
        _d23.append(float(np.sum(_dtc[_xr & _post]) / _Tp) if _Tp > 0 else np.nan)
    A23["f_x_post"] = _d23
    A23["hi_dose"] = A23["f_x_post"].to_numpy(float) > _dcut          # the SAME cuts as Part 22, not re-medianed
    _g23v = A23[P22_GEOM].to_numpy(float)
    if P22_NH2MIN > 0:                                                  # ... and the same resolution floor
        _g23v = np.where(A23["n_H2_tot"].to_numpy(float) >= P22_NH2MIN, _g23v, np.nan)
    A23["hi_geom"], A23["lo_geom"] = _g23v > _gcut, _g23v <= _gcut     # both False where the geometry is missing
    _plain = lambda t: _re23.sub(r"\\[a-zA-Z]+|[{}$]", "", t.replace("$_2$", "2")).replace("  ", " ").strip()   # a legend label -> a plain group name
    G23 = [("low dose + " + _plain(_ghi),  (~A23["hi_dose"]) & A23["hi_geom"],   "#b2182b", "low dose $+$ " + _ghi),
           ("low dose + " + _plain(_glo),  (~A23["hi_dose"]) & A23["lo_geom"],   "#ef8a62", "low dose $+$ " + _glo),
           ("high dose + " + _plain(_ghi), A23["hi_dose"] & A23["hi_geom"],      "#67a9cf", "high dose $+$ " + _ghi),
           ("high dose + " + _plain(_glo), A23["hi_dose"] & A23["lo_geom"],      "#2166ac", "high dose $+$ " + _glo)]
    NAM23 = [n for n, _, _, _ in G23]; COL23 = {n: c for n, _, c, _ in G23}; LAB23 = {n: l for n, _, _, l in G23}
    A23["grp"] = None
    for _n23, _m23, _, _ in G23:
        A23.loc[_m23.to_numpy(), "grp"] = _n23
    KEY23 = {(int(a_), int(g_)): n_ for a_, g_, n_ in zip(A23["anchor_snap"], A23["gal_id"], A23["grp"]) if n_ is not None}
    print(f"Part 23 — the dose x geometry division in the m25 products: {len(KEY23)} quenched anchors "
          + ("(every anchor row" if P23_ALLANCH else "(Part 22's one-per-galaxy set") + "), "
          + ", ".join(f"{n} {int(m.sum())}" for n, m, _, _ in G23))

    def _fr23(path, cols):
        """A FITS table -> a DataFrame of the named columns, masked values as NaN and byte strings decoded."""
        t = Table.read(path); d = {}
        for c in cols:
            v = t[c]; v = v.filled(np.nan) if hasattr(v, "filled") else v
            v = np.asarray(v)
            d[c] = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in v]) if v.dtype.kind in "SU" else v.astype(float)
        return pd.DataFrame(d)

    def _gband23(vals, seed=23):
        """Per-galaxy values -> median and its bootstrap 16-84 (nan below P23_NMIN)."""
        v = np.asarray(vals, float); v = v[np.isfinite(v)]
        if len(v) < P23_NMIN:
            return np.nan, np.nan, np.nan, len(v)
        rng = np.random.default_rng(seed)
        bs = np.median(v[rng.integers(0, len(v), (P22_NBOOT, len(v)))], axis=1)
        return float(np.median(v)), float(np.percentile(bs, 16)), float(np.percentile(bs, 84)), len(v)

    R23 = []
    # ── (A) the radial anatomy per annulus ──
    if not (os.path.exists(ANN23) and os.path.exists(APT23)):
        print(f"\n(A) radial profiles: {ANN23} or {APT23} missing — the m25 notebook's Parts 8a / 7e (cluster)")
    else:
        K4 = ["snap", "gal_id", "incl", "aperture"]
        _APC23 = set(Table.read(APT23).colnames)
        _sfr23 = P23_SFRCOL if P23_SFRCOL in _APC23 else None      # the resolved sSFR panel needs an SFR column
        if _sfr23 is None:
            print(f"  (A) '{P23_SFRCOL}' is not in {os.path.basename(APT23)} "
                  f"({', '.join(sorted(c for c in _APC23 if 'sfr' in c.lower())) or 'no SFR column at all'}) "
                  f"— the sSFR panel is skipped")
        RF = _fr23(ANN23, K4 + ["M_dust", "M_H2", "M_gas"]).merge(
             _fr23(APT23, K4 + ["nstar_ap", "mstar", "age_m_star_myr"] + ([_sfr23] if _sfr23 else [])),
             on=K4, how="inner")
        if os.path.exists(AV23):
            RF = RF.merge(_fr23(AV23, K4 + ["A_V"]), on=K4, how="left")
        RF = RF[RF["aperture"].isin([a for a, _ in P23_ANNULI])].reset_index(drop=True)
        SFK = set()
        if os.path.exists(SEL23):
            _s23 = _fr23(SEL23, ["snap", "gal_id", "pop"])
            SFK = {(int(s_), int(i_)) for s_, i_, p_ in zip(_s23["snap"], _s23["gal_id"], _s23["pop"]) if p_ == "SF"}
        RF["grp"] = [KEY23.get((int(s_), int(i_)), "star-forming controls" if (int(s_), int(i_)) in SFK else None)
                     for s_, i_ in zip(RF["snap"], RF["gal_id"])]
        RF = RF[RF["grp"].notna()].reset_index(drop=True)
        if os.path.exists(KIN23):
            RF = RF.merge(_fr23(KIN23, ["snap", "gal_id", "aperture", "kappa_H2", "kappa_star"]),
                          on=["snap", "gal_id", "aperture"], how="left")
        for _f23, _n23 in ((AV23, "A_V (m25 8b)"), (KIN23, "kappa (m25 8j0)"), (SEL23, "the SF-controls line (m25 Part 3)")):
            if not os.path.exists(_f23):
                print(f"  (A) {_n23}: {_f23} missing — panel / line skipped")
        _ms23 = np.where(RF["nstar_ap"].to_numpy(float) >= P23_NSTAR, RF["mstar"].to_numpy(float), np.nan)
        with np.errstate(all="ignore"):
            for _c23, _num in (("fdust_ap", "M_dust"), ("fH2_ap", "M_H2")):
                _v23 = RF[_num].to_numpy(float)
                RF[_c23] = np.where(np.isfinite(_v23) & (_v23 > 0) & (_ms23 > 0), _v23 / _ms23, np.nan)
            _mh23 = RF["M_H2"].to_numpy(float); _md23 = RF["M_dust"].to_numpy(float)
            RF["dh2_ann"] = np.where((_mh23 > 0) & (_md23 > 0), _md23 / np.where(_mh23 > 0, _mh23, np.nan), np.nan)
            RF["age_ap"] = np.where(np.isfinite(_ms23), RF["age_m_star_myr"].to_numpy(float) / 1e3, np.nan)
            if _sfr23:                                              # the RESOLVED sSFR: this region's SFR over its M*
                _sv23 = RF[_sfr23].to_numpy(float)
                RF["ssfr_ap"] = np.where(np.isfinite(_sv23) & (_ms23 > 0),
                                         np.clip(_sv23 / np.where(_ms23 > 0, _ms23, np.nan), P23_SSFRMIN, None),
                                         np.nan)
            if "A_V" in RF.columns:
                _av23 = RF["A_V"].to_numpy(float)
                RF["A_V"] = np.where(np.isfinite(_av23) & (_av23 > 0), _av23, np.nan)
        QUANT23 = ([("A_V", r"$A_V$ [mag]", "log")] if "A_V" in RF.columns else []) \
                + [("fdust_ap", r"$M_{\rm dust}/M_\star$", "log"), ("fH2_ap", r"$M_{\rm H_2}/M_\star$", "log")] \
                + ([("ssfr_ap", rf"sSFR$_{{{P23_SFRCOL[3:]}}}$ [yr$^{{-1}}$]", "log")] if _sfr23 else []) \
                + [("dh2_ann", r"$M_{\rm dust}/M_{\rm H_2}$", "log"), ("age_ap", "stellar age [Gyr]", "lin")] \
                + ([(k, lab, "kappa") for k, lab in (("kappa_H2", r"$\kappa_{\rm rot}^{\rm H_2}$"),
                                                     ("kappa_star", r"$\kappa_{\rm rot}^{\star}$"))] if "kappa_H2" in RF.columns else [])
        GRPA = (["star-forming controls"] if len(SFK) else []) + NAM23
        COLA = dict(COL23, **{"star-forming controls": "0.62"})
        _x23 = np.arange(len(P23_ANNULI)); _nc23 = (len(QUANT23) + 1) // 2
        NMAX23 = {g: 0 for g in GRPA}
        fig, axs = plt.subplots(2, _nc23, figsize=(3.6 * _nc23, 7.3), squeeze=False)
        for _qi, (_col23, _lab23, _kind) in enumerate(QUANT23):
            ax = axs[_qi // _nc23, _qi % _nc23]
            for _gi, _g23 in enumerate(GRPA):
                _m, _l, _h, _n = [], [], [], []
                for _a23, _ in P23_ANNULI:
                    _sub = RF[(RF["grp"] == _g23) & (RF["aperture"] == _a23)]
                    _v = _sub[_col23].to_numpy(float); _ok = np.isfinite(_v)
                    _per = {}
                    for _s_, _i_, _vv in zip(_sub["snap"].to_numpy(float)[_ok], _sub["gal_id"].to_numpy(float)[_ok], _v[_ok]):
                        _per.setdefault((int(_s_), int(_i_)), []).append(_vv)          # the sightlines -> one value per galaxy
                    _c, _lo, _hi, _nn = _gband23([np.median(vv) for vv in _per.values()])
                    _m.append(_c); _l.append(_lo); _h.append(_hi); _n.append(_nn)
                    R23.append(dict(block="profile", quantity=_col23, group=_g23, x=_a23, n_gal=_nn, med=_c, lo=_lo, hi=_hi))
                NMAX23[_g23] = max(NMAX23[_g23], max(_n))
                _off23 = (_gi - (len(GRPA) - 1) / 2) * 0.075
                _m, _l, _h = np.array(_m), np.array(_l), np.array(_h)
                _okm = np.isfinite(_m)
                if _okm.any():
                    ax.fill_between(_x23[_okm] + _off23, _l[_okm], _h[_okm], color=COLA[_g23], alpha=0.15, lw=0)
                    ax.plot(_x23[_okm] + _off23, _m[_okm], "o-", color=COLA[_g23], ms=4.5, mec="k", mew=0.4,
                            lw=1.3 if _g23.startswith("star-forming") else 1.9)
            if _kind == "log":
                ax.set_yscale("log")
            if _col23 == "ssfr_ap":                                  # say where the floor is, and stop above it
                ax.axhline(P23_SSFRMIN, color="0.45", lw=0.9, ls=":")
                ax.annotate("no SF", xy=(0.02, P23_SSFRMIN), xycoords=("axes fraction", "data"), xytext=(0, 2),
                            textcoords="offset points", fontsize=7, color="0.45", va="bottom")
                ax.set_ylim(bottom=0.55 * P23_SSFRMIN)
            if _kind == "kappa":                                     # the physical reference lines, not an arbitrary 0.3
                ax.axhline(1 / 3, color="0.3", lw=1.0, ls="--"); ax.axhline(0.5, color="0.3", lw=0.8, ls=":"); ax.set_ylim(0, 1)
                ax.annotate("isotropic (1/3)", xy=(0.98, 1 / 3), xycoords=("axes fraction", "data"), xytext=(0, 2),
                            textcoords="offset points", fontsize=6.5, color="0.3", ha="right", va="bottom")
                ax.annotate("rotation-supported (0.5)", xy=(0.98, 0.5), xycoords=("axes fraction", "data"), xytext=(0, 2),
                            textcoords="offset points", fontsize=6.5, color="0.3", ha="right", va="bottom")
            ax.set_xticks(_x23); ax.set_xticklabels([r_ for _, r_ in P23_ANNULI], fontsize=8)
            if _qi // _nc23 == 1:
                ax.set_xlabel("annulus [kpc]")
            ax.set_ylabel(_lab23); ax.grid(alpha=0.25, lw=0.5)
        for _qi in range(len(QUANT23), 2 * _nc23):
            axs[_qi // _nc23, _qi % _nc23].axis("off")
        axs[0, 0].legend(handles=[_L23([], [], color=COLA[g], marker="o", ms=4.5, mec="k", mew=0.4,
                                       lw=1.3 if g.startswith("star-forming") else 1.9, label=f"{g} (N $\\leq$ {NMAX23[g]})")
                                  for g in GRPA], frameon=False, fontsize=7.6)
        fig.suptitle(f"Part 23 — {BOX_LABEL.get(P23_BOX, P23_BOX)}: the radial anatomy of the dose x geometry groups "
                     f"(+ the SF controls); per-galaxy sightline medians, bootstrap 16-84 of the group median", fontsize=11)
        if P22_GEOM in ("kappa_H2", "kappa_gas"):
            print(f"  (A) NOTE the {P22_GEOM} panel is PARTLY CIRCULAR inside {P22_CORE}: the groups are split on it there. "
                  f"The outer annuli, kappa_star and every other panel are not.")
        else:
            print(f"  (A) the groups are split on {P22_GEOM} (galaxy-wide), so no annulus panel here is circular. In the kappa panels the "
                  f"dashed line is the isotropic value 1/3 (no net rotation below it), the dotted one 0.5 (rotation-supported).")
        fig.tight_layout()
        paper_save(fig, f"agn_states_dosegeom_profiles_{P23_BOX}")
        plt.show()

    # ── (B) the K-S plane, one panel; (C) the offset from the relation at each critical point, printed ──
    if not os.path.exists(KS23):
        print(f"\n(B) K-S plane: {KS23} missing — ks_tracks_quenched_m25 Part 4 (run tag 'pt', cluster)")
    else:
        try:
            import ks_tracks_lib as _kl23
            REL23 = dict(_kl23.RELATIONS[P23_REL]); _HE23 = float(getattr(_kl23, "HE_FACTOR", 1.36))
        except Exception:                                            # the relation, verbatim, so the cell runs outside the repo root
            REL23 = dict(A=-2.06 - 0.96, N=0.96, sig=0.20, color="#1b9e77", ls="--", label="Bigiel+08 (H$_2$)"); _HE23 = 1.36
            print("  (B) ks_tracks_lib not importable — the reference relation is used from its literal definition")
        T23 = Table.read(KS23)
        _sa = lambda a: np.array([x.decode() if isinstance(x, bytes) else str(x) for x in np.asarray(a)])
        ap23, pop23, stg23, gk23 = _sa(T23["ap_label"]), _sa(T23["pop"]), _sa(T23["stage"]), _sa(T23["gkey"])
        x23 = np.asarray(T23["logSigmaH2"], float); y23 = np.asarray(T23[P23_KS_SFR], float)
        ul23 = np.asarray(T23["is_ul"], bool)
        # ── the beam-matched coordinates: R = max(R50(H2), beam); an unresolved galaxy gets the lower limit the observation would give ──
        _sfrtot23 = {"logSigmaSFR": "sfr100_tot", "logSigmaSFR_sfr25": "sfr25_tot", "logSigmaSFR_inst": "sfr_inst_tot"}.get(P23_KS_SFR, "sfr100_tot")
        r50_23 = np.asarray(T23["r50_H2"], float); mh2t23 = np.asarray(T23["m_H2_tot"], float); sft23 = np.asarray(T23[_sfrtot23], float)
        if P23_BEAM_KPC and P23_UV_MIN_KPC:                          # two-tier: a uv fit measures down to P23_UV_MIN_KPC, below it the limit
            _Rb23 = np.where(r50_23 >= P23_UV_MIN_KPC, r50_23, P23_BEAM_KPC)
            unres23 = np.isfinite(r50_23) & (r50_23 < P23_UV_MIN_KPC)
        elif P23_BEAM_KPC:                                           # one-tier: every galaxy below the beam is unresolved
            _Rb23 = np.maximum(r50_23, P23_BEAM_KPC)
            unres23 = np.isfinite(r50_23) & (r50_23 < P23_BEAM_KPC)
        else:
            _Rb23 = r50_23; unres23 = np.zeros(len(r50_23), bool)
        with np.errstate(all="ignore"):
            xb23 = np.where((mh2t23 > 0) & (_Rb23 > 0), np.log10(_HE23 * 0.5 * mh2t23 / (np.pi * _Rb23 ** 2) / 1e6), np.nan)
            yb23 = np.where((sft23 > 0) & (_Rb23 > 0), np.log10(0.5 * sft23 / (np.pi * _Rb23 ** 2)), np.nan)
        if not P23_BEAM_KPC:
            xb23, yb23 = x23, y23                                    # no observed positions: both representations are the intrinsic one
        _RULE23 = (rf"$R = R_{{50}}$ if $\geq {P23_UV_MIN_KPC:g}$ kpc (uv fit), else the ${P23_BEAM_KPC:g}$ kpc limit"
                   if (P23_BEAM_KPC and P23_UV_MIN_KPC) else (rf"$R = \max(R_{{50}}, {P23_BEAM_KPC:g}\,$kpc$)$" if P23_BEAM_KPC else ""))
        _main_obs = (P23_KS_MAIN == "observed") and bool(P23_BEAM_KPC)
        _XM, _YM = (xb23, yb23) if _main_obs else (x23, y23)         # the solid track
        _XO, _YO = (x23, y23) if _main_obs else (xb23, yb23)         # the overlay
        grp23 = np.array([KEY23.get(tuple(int(v) for v in g_.split("_"))) for g_ in gk23], dtype=object)
        base23 = (ap23 == P23_KS_AP)
        MRK23 = {"sf_peak": "*", "agn_ign": "^", "jet_on": "v", "sft": "s", "qt": "D", "end": "o", "anchor": "o",
                 "gas_min": "X"}                                 # the same marker per epoch as Part 22's Fig 3
        LAB23S = {"sf_peak": "SF peak", "agn_ign": "ignition", "jet_on": "jet", "sft": "SFT", "qt": "QT",
                  "post_quench": "post-q", "gas_min": "gas min", "anchor": "anchor", "end": "end"}
        OBS23 = None
        if os.path.exists(P23_OBS):
            _o23 = pd.read_csv(P23_OBS)
            OBS23 = _o23[_o23["co_det"].astype(bool) & np.isfinite(_o23["logSigmaH2"])].reset_index(drop=True)
        else:
            print(f"  (B) {P23_OBS} missing — the observed points are skipped")

        fig, ax = plt.subplots(figsize=(9.6, 7.0), constrained_layout=True)
        _xs23 = np.linspace(P23_KS_XLIM[0] - 1.0, P23_KS_XLIM[1] + 1.0, 60)
        ax.fill_between(_xs23, REL23["A"] + REL23["N"] * _xs23 - REL23["sig"], REL23["A"] + REL23["N"] * _xs23 + REL23["sig"],
                        color=REL23["color"], alpha=0.10, lw=0, zorder=0)
        ax.plot(_xs23, REL23["A"] + REL23["N"] * _xs23, color=REL23["color"], lw=1.5, ls=REL23.get("ls", "--"), zorder=2)
        if P23_SFCLOUD:
            _sf23 = base23 & (pop23 == "SF") & np.isfinite(x23) & np.isfinite(y23)
            ax.scatter(x23[_sf23], y23[_sf23], s=7, c="0.72", alpha=0.30, lw=0, zorder=1, rasterized=True)
        _END23 = {}                                                  # per group at 'end': the numbers printed under the figure
        _lim23 = lambda ax_, x_, y_, c_, a_, s_: ax_.annotate("", xy=(x_ + 0.22, y_ + 0.22), xytext=(x_, y_), zorder=6,   # a slope-1 lower-limit arrow
                                                              arrowprops=dict(arrowstyle="-|>", color=c_, lw=1.1, alpha=a_, mutation_scale=9,
                                                                              shrinkA=s_, shrinkB=0))
        for _n23, _, _c23, _ in G23:
            _mk = base23 & (pop23 == "Q") & (grp23 == _n23) & np.isin(stg23, P23_KS_STG)
            _mx, _my, _ox, _oy, _fu = [], [], [], [], []
            for _st23 in P23_KS_STG:
                _ms23b = _mk & (stg23 == _st23) & ~ul23 & np.isfinite(_XM) & np.isfinite(_YM)
                if _ms23b.sum() >= P23_NMIN:
                    _mx.append(float(np.median(_XM[_ms23b]))); _my.append(float(np.median(_YM[_ms23b])))
                    _oo = _mk & (stg23 == _st23) & ~ul23 & np.isfinite(_XO) & np.isfinite(_YO)
                    _ox.append(float(np.median(_XO[_oo])) if _oo.sum() >= P23_NMIN else np.nan)
                    _oy.append(float(np.median(_YO[_oo])) if _oo.sum() >= P23_NMIN else np.nan)
                    _fu.append(float(np.mean(unres23[_ms23b])))
                    ax.scatter(_mx[-1], _my[-1], s=110 if _st23 != "sf_peak" else 200, marker=MRK23.get(_st23, "o"),
                               c=_c23, edgecolors="k", linewidths=0.8, zorder=6)
                    if _main_obs and _fu[-1] > 0.5:                  # most of the group unresolved: the observed point is a lower limit
                        _lim23(ax, _mx[-1], _my[-1], _c23, 0.9, 7)
                    _xi23, _yi23 = (_ox[-1], _oy[-1]) if _main_obs else (_mx[-1], _my[-1])
                    _xb23, _yb23 = (_mx[-1], _my[-1]) if _main_obs else (_ox[-1], _oy[-1])
                    R23.append(dict(block="ks_track", quantity="stage_median", group=_n23, x=_st23,
                                    n_gal=int(_ms23b.sum()), med=_yi23, lo=np.nan, hi=np.nan, logSigmaH2=_xi23,
                                    logSigmaH2_observed=_xb23, logSigmaSFR_observed=_yb23, f_unresolved=_fu[-1],
                                    r50_H2_kpc=float(np.median(r50_23[_ms23b]))))
                    if _st23 == "end":
                        _END23[_n23] = (_xb23, _xi23, _fu[-1], float(np.median(r50_23[_ms23b])), int(_ms23b.sum()))
            if len(_mx) >= 2:
                ax.plot(_mx, _my, "-", color=_c23, lw=2.6, zorder=5)
                for _ai23 in range(len(_mx) - 1):                    # the direction of time, as in Part 22's walked planes
                    ax.annotate("", xy=(0.5 * (_mx[_ai23] + _mx[_ai23 + 1]), 0.5 * (_my[_ai23] + _my[_ai23 + 1])),
                                xytext=(_mx[_ai23], _my[_ai23]), zorder=5,     # the head at the midpoint, clear of the markers
                                arrowprops=dict(arrowstyle="-|>", color=_c23, lw=1.6, alpha=0.9, mutation_scale=16,
                                                shrinkA=6, shrinkB=0))
                if P23_KS_OVERLAY and P23_BEAM_KPC:                  # the other representation, faded: the size of the resolution effect
                    _okO = np.isfinite(_ox) & np.isfinite(_oy)
                    ax.plot(np.asarray(_ox)[_okO], np.asarray(_oy)[_okO], "--", color=_c23, lw=1.2, alpha=0.45, zorder=3)
                    ax.scatter(np.asarray(_ox)[_okO], np.asarray(_oy)[_okO], s=26, marker="o", c=_c23, alpha=0.45, lw=0, zorder=3)
                    if not _main_obs:                                # the overlay IS the observed one: its lower-limit arrows, faded
                        for _xo_, _yo_, _fu_ in zip(_ox, _oy, _fu):
                            if np.isfinite(_xo_) and _fu_ > 0.5:
                                _lim23(ax, _xo_, _yo_, _c23, 0.45, 3)
        # ── the extended tail: the quenched galaxies the beam WOULD resolve at 'end', each its own point ──
        _tm23 = np.zeros(len(r50_23), bool)
        if P23_TAIL and P23_BEAM_KPC:
            _tm23 = base23 & (pop23 == "Q") & (stg23 == "end") & np.isfinite(r50_23) & (r50_23 > P23_BEAM_KPC) & np.isfinite(xb23)
            _KAP23 = {(int(a_), int(g_)): (k_, c_) for a_, g_, k_, c_ in zip(A23["anchor_snap"], A23["gal_id"], A23["kappa_H2"],
                                                                             A23[P22_GEOM])}
            for _i23 in np.flatnonzero(_tm23):
                _yt = yb23[_i23] if np.isfinite(yb23[_i23]) else P23_KS_YLIM[0] + 0.15        # no SF at all: parked on the frame's floor
                ax.scatter(xb23[_i23], _yt, s=75, marker="P", c=COL23.get(grp23[_i23], "0.5") if grp23[_i23] else "0.5",
                           edgecolors="k", linewidths=0.7, zorder=7)
                _k23, _c23g = _KAP23.get(tuple(int(v) for v in gk23[_i23].split("_")), (np.nan, np.nan))
                R23.append(dict(block="ks_tail", quantity="galaxy", group=grp23[_i23] if grp23[_i23] else "ungrouped", x=gk23[_i23],
                                n_gal=1, med=float(yb23[_i23]), lo=np.nan, hi=np.nan, logSigmaH2=float(xb23[_i23]),
                                r50_H2_kpc=float(r50_23[_i23]), kappa_H2=float(_k23), geom=float(_c23g)))
        _leg23 = []
        if OBS23 is not None:
            _CO, _CE = "#e7298a", "#3b0f2a"
            for _, _r23 in OBS23.iterrows():
                _xo, _yo = float(_r23["logSigmaH2"]), float(_r23["logSigmaSFR"])
                if bool(_r23["sigma_is_ll"]):
                    ax.annotate("", xy=(_xo + 0.3, _yo + 0.3), xytext=(_xo, _yo),
                                arrowprops=dict(arrowstyle="-|>", color=_CE, lw=1.2, mutation_scale=10), zorder=8)
                    ax.scatter(_xo, _yo, s=110, marker="o", facecolors="white", edgecolors=_CE, linewidths=1.4, zorder=9)
                    ax.scatter(_xo, _yo, s=40, marker="o", c=_CO, edgecolors="none", zorder=10)
                else:
                    _elo = float(_r23["logSigmaSFR_elo"]) if np.isfinite(_r23["logSigmaSFR_elo"]) else 0.0
                    _ehi = float(_r23["logSigmaSFR_ehi"]) if np.isfinite(_r23["logSigmaSFR_ehi"]) else 0.0
                    _xe = float(_r23["logSigmaH2_err"]) if np.isfinite(_r23["logSigmaH2_err"]) else 0.0
                    ax.errorbar(_xo, _yo, xerr=_xe, yerr=[[min(_elo, _yo + 4.5)], [_ehi]], fmt="none", ecolor=_CE,
                                elinewidth=1.0, zorder=8)
                    ax.scatter(_xo, _yo, s=110, marker="o", c=_CO, edgecolors=_CE, linewidths=1.4, zorder=9)
            _leg23 += [_L23([], [], marker="o", ls="", ms=9, mfc=_CO, mec=_CE,
                            label=rf"ALMA-C11 QGs, $z\approx0.4$ (N={len(OBS23)})"),
                       _L23([], [], marker="o", ls="", ms=9, mfc="white", mec=_CE, label=r"unresolved: $\Sigma$ lower limits")]
        _leg23 = [_L23([], [], color=_c23, lw=2.4, label=LAB23[_n23]) for _n23, _, _c23, _ in G23] \
               + [_L23([], [], ls="", marker=MRK23.get(s_, "o"), ms=8, mfc="0.55", mec="k",
                        label=LAB23S.get(s_, s_.replace("_", " "))) for s_ in P23_KS_STG] \
               + [_L23([], [], color=REL23["color"], lw=1.5, ls=REL23.get("ls", "--"),
                       label=REL23.get("label", P23_REL) + r" $\pm$ scatter")] \
               + ([_L23([], [], color="0.4", lw=2.0 if _main_obs else 1.2, ls="-" if _main_obs else "--", alpha=1.0 if _main_obs else 0.5,
                         marker="o", ms=6 if _main_obs else 4, mec="k" if _main_obs else "none",
                         label="as ALMA-C11 would measure it (arrow: mostly unresolved)")]
                  + ([_L23([], [], color="0.4", lw=1.2 if _main_obs else 2.0, ls="--" if _main_obs else "-", alpha=0.5 if _main_obs else 1.0,
                            marker="o", ms=4 if _main_obs else 6, mec="none" if _main_obs else "k", label=r"intrinsic: $R = R_{50}({\rm H_2})$")]
                     if (P23_KS_OVERLAY or not _main_obs) else [])
                  + ([_L23([], [], ls="", marker="P", ms=8, mfc="0.6", mec="k",
                            label=rf"larger than the beam: $R_{{50}}({{\rm H_2}}) > {P23_BEAM_KPC:g}$ kpc (N={int(_tm23.sum())})")]
                     if P23_TAIL else [])
                  if P23_BEAM_KPC else []) + _leg23
        ax.legend(handles=_leg23, loc="upper left", frameon=False, fontsize=7.6, ncol=2)
        ax.set_xlabel(r"$\log(\Sigma_{\rm H_2}/M_\odot\,{\rm pc}^{-2})$   [$0.5M/\pi R^2$, He included; solid: "
                      + ("as observed" if _main_obs else r"$R = R_{50}({\rm H_2})$") + "]")
        ax.set_ylabel(r"$\log(\Sigma_{\rm SFR}/M_\odot\,{\rm yr}^{-1}\,{\rm kpc}^{-2})$")
        ax.set_xlim(*P23_KS_XLIM); ax.set_ylim(*P23_KS_YLIM); ax.tick_params(direction="in", top=True, right=True)
        ax.grid(alpha=0.15, lw=0.6)
        ax.set_title(("solid: as observed; faded: intrinsic" if _main_obs else "solid: intrinsic $R_{50}$; faded: as observed")
                     + (" — " + _RULE23 + r"; $+$: larger than the beam" if P23_BEAM_KPC else ""), fontsize=9.5)
        fig.suptitle(f"Part 23 — {BOX_LABEL.get(P23_BOX, P23_BOX)}: the dose x geometry groups on the K-S plane, against the ALMA-C11 QGs",
                     fontsize=11)
        paper_save(fig, f"agn_states_dosegeom_ks_{P23_BOX}")
        plt.show()
        if P23_BEAM_KPC and _END23:
            print(f"\n(B) at 'end', as ALMA-C11 would measure it ("
                  + (f"uv floor {P23_UV_MIN_KPC:g} kpc, limit {P23_BEAM_KPC:g} kpc" if P23_UV_MIN_KPC else f"R = max(R50, {P23_BEAM_KPC:g} kpc)")
                  + "): log Sigma_H2 observed | intrinsic; unresolved fraction; median R50(H2):")
            for _n23 in NAM23:
                if _n23 in _END23:
                    _e = _END23[_n23]
                    print(f"    {_n23:34s} {_e[0]:5.2f} | {_e[1]:5.2f};  {100 * _e[2]:3.0f} % unresolved;  R50 {_e[3]:.2f} kpc  (N={_e[4]})")
            if P23_TAIL:
                _tl = [r for r in R23 if r["block"] == "ks_tail"]
                print(f"    the galaxies larger than the beam (R50(H2) > {P23_BEAM_KPC:g} kpc at 'end'): {len(_tl)} of "
                      f"{int((base23 & (pop23 == 'Q') & (stg23 == 'end') & np.isfinite(r50_23)).sum())} anchor rows"
                      + (":" if _tl else ""))
                for r in sorted(_tl, key=lambda r: -r["r50_H2_kpc"]):
                    print(f"      {r['x']:>10s}  {r['group']:34s} R50 {r['r50_H2_kpc']:5.2f} kpc  kappa_H2 {r['kappa_H2']:.2f}  "
                          f"{P22_GEOM} {r['geom']:+.2f}  log Sigma_H2 {r['logSigmaH2']:.2f}  log Sigma_SFR "
                          + (f"{r['med']:.2f}" if np.isfinite(r['med']) else "no SF"))

        # ── (C) were they starbursts? the offset from the relation at each critical point — printed + tabulated, not drawn ──
        _off23 = y23 - (REL23["A"] + REL23["N"] * x23)
        print(f"\n(C) the offset from {REL23.get('label', P23_REL)} at each critical point (median over galaxies, "
              f"the censored SFRs dropped; intrinsic R50 apertures — a slope-1 beam shift moves the offset from a slope-{REL23['N']:.2f} "
              f"relation by {100 * abs(1 - REL23['N']):.0f} % of the shift):")
        for _gi, (_n23, _, _c23, _) in enumerate(G23):
            _yy = []
            for _si, _st23 in enumerate(P23_KS_STG):
                _m23 = base23 & (pop23 == "Q") & (grp23 == _n23) & (stg23 == _st23) & ~ul23 & np.isfinite(_off23)
                _c, _l, _h, _n = _gband23(_off23[_m23])
                _yy.append(_c)
                R23.append(dict(block="ks_offset", quantity="dlog_rel", group=_n23, x=_st23, n_gal=_n, med=_c, lo=_l, hi=_h))
            print(f"    {_n23:34s} " + "  ".join(f"{LAB23S.get(s_, s_)} {v:+.2f}" if np.isfinite(v)
                                                 else f"{LAB23S.get(s_, s_)} n/a" for s_, v in zip(P23_KS_STG, _yy)))
        print(f"    (above the relation = a burst; below = a dying disc; +- {REL23['sig']:.2f} dex is the relation's own scatter)")
        _tq23 = A23.groupby("grp")["tau_q_gyr"].median()
        _bst = {}                       # the burst itself: peak sSFR in the Gyr before SFT over the median of the 3 Gyr before that
        for _, _r23 in A23.iterrows():
            _d = _CH22.get(_r23["key"]); _ts = _r23["t_sft_gyr"]
            if _d is None or not np.isfinite(_ts) or "ssfr" not in _d:
                continue
            _late = _d["ex"] & (_d["t"] > _ts - 1.0) & (_d["t"] <= _ts)
            _base = _d["ex"] & (_d["t"] > _ts - 4.0) & (_d["t"] <= _ts - 1.0)
            if _late.sum() >= 2 and _base.sum() >= 3:
                with np.errstate(all="ignore"):
                    _bst.setdefault(_r23["grp"], []).append(float(np.nanmax(_d["ssfr"][_late]) - np.nanmedian(_d["ssfr"][_base])))
        print("  quenching time tau_q [Gyr], median per group: "
              + ",  ".join(f"{n} {_tq23.get(n, np.nan):.2f}" for n in NAM23))
        if _bst:
            print("  the burst before SFT (peak log sSFR in the last Gyr minus the median of the 3 Gyr before it, dex): "
                  + ",  ".join(f"{n} {np.nanmedian(_bst[n]):+.2f}" for n in NAM23 if n in _bst))

    if R23:
        _p23 = os.path.join(OUT, f"agn_states_dosegeom_{P23_BOX}.csv")
        pd.DataFrame(R23).to_csv(_p23, index=False)
        print(f"\n  the tables behind both figures -> {_p23}")

## Part 24 — the pictures: one galaxy per class, at each critical point

Parts 22 and 23 divide the quenched galaxies four ways and *score* the division. This part **shows** it: one representative galaxy per class, rendered at the same four critical points the tracks are walked through — `jet_on` $\to$ `sft` $\to$ `qt` $\to$ `anchor`.

**The two dials at the top of the cell**, flagged with `◄──`, because they are the ones you actually reach for:

* **`P24_SIZE`** — *how big each image is*, **in kpc a side**: `60` gives a 60 $\times$ 60 kpc box around each galaxy. One number for the whole grid, so every panel is on the same physical scale and a galaxy shrinking or growing across a row is real, not a zoom. The cutouts hold particles out to `rmax_kpc` $=$ 100 kpc from the centre, so past ~200 the corners are genuinely empty and the cell says so once.
* **`P24_FS`** — *how big the text is*: one multiplier over every label, title, tick and number the part draws.

**Where the pixels come from.** The renders read the **reduced particle cutouts** (`build_reduced_particles_job.py` $\to$ `output/cis25/reduced_particles/snap_NNN/…`) through `simbanator.visualization.rendering.ReducedRender` — the same particles the $\Sigma$-profile and K-S products of Parts 13 and 23 are measured from, so a panel is a picture *of* the numbers in those tables and not an independent re-measurement. It needs only `py-sphviewer` and `h5py`: no snapshot and no Caesar catalogue, so the whole grid renders in seconds where `RenderRGB` / `SingleRender` would load a full box per panel. The trade is that only the fields the reduced job stored exist (gas, HI, H$_2$, dust, SFR, stars, velocities), and only out to 100 kpc.

Face-on by default (`P24_SPOS`), and **the whole cutout** rather than the catalogue members (`P24_MEMBERS = False`). That is not only about seeing the CGM and the neighbours: py-sphviewer estimates each particle's smoothing length from its 32 nearest *loaded* neighbours, so rendering members alone inflates hsml in the ISM and the galaxy comes out blobbier, smaller and lower-resolution than it is.

### Which four galaxies — seeded on the low-dust end, then matched

A gallery only says something about the *classes* if the four galaxies are otherwise alike, so they are chosen in one dependent pass rather than class by class:

1. **The seed** is the class that has to carry the argument's low-dust end — `P24_SEEDCLS = "hi_lo"`, high dose $+$ dispersion-supported — restricted to $\log M_{\rm dust}/M_{\rm H_2} <$ `P24_SEEDMAX` $= -3$, and among those the **best-resolved** galaxy (most gas $+$ star particles). This is the only place resolution decides anything. If no member of the class is below the cut, the cell says so and uses the three lowest instead.
2. **The other three** are then whichever member of their class is **closest to the seed in mass-weighted stellar age and $\log M_\star$**. Matching is all that decides them: `P24_TOLAGE` (0.4 Gyr) and `P24_TOLM` (0.12 dex) only set how age trades against mass — the cost is measured in units of them, so a cost of 1 means "off by one tolerance" — and `P24_RESW`, the weight resolution carries in *this* choice, is **0** by default, so a well-matched galaxy beats a better-sampled one.

**The seed's anchor is a preference, not a cage.** `P24_SAMEANCHOR` still looks there first, but one anchor holds only a handful of drawable galaxies per class, and insisting on it produced quadruples off by 2.9 Gyr in age and 1.0 dex in mass. So a class the seed's anchor cannot match to within `P24_COSTMAX` is taken from whichever anchor matches best, and the cell names it: *a matched pair at two redshifts says more about the classes than a mismatched pair at one*, and every panel carries its own redshift anyway. For the same reason the **seed itself is searched**: the best-resolved low-dust galaxy is the first choice, but if the other three cannot be matched to it, the next `P24_SEEDTRY` in resolution order are tried and the one with the least-bad worst match is kept. The printout says which seed won and why.

`P24_DGR` picks which ratio the seed cut and every reported number mean: `"dh2"` (the default, $M_{\rm dust}/M_{\rm H_2}$, which is what `P22_DUST` means) or `"dg"` ($M_{\rm dust}/M_{\rm gas}$). The cell prints the seed and why it won, then every galaxy with its **match cost**, and warns when a cost exceeds 1 — the spread is reported rather than hidden, because with only a handful of drawable candidates per class at one anchor it may well exceed the tolerance. Anchors are restricted to $z >$ `P24_ZMIN`.

With `P24_ALLANCH = True` the pool is **every anchor row** (Part 23's `A23`, if Part 23 has run) rather than Part 22's one-per-galaxy set, which matters here: `ANC22` keeps only each galaxy's *latest* anchor, so a redshift floor on it throws away most of the sample. `P24_PICK` names a galaxy by hand for any class, seed included.

**Where the ages come from.** Age and $\log M_\star$ are read from Part 0's **`CAT`** — the *full* catalogue cache, every galaxy above $\log M_\star = 9.5$ per box and anchor — and deliberately **not** from `QM`. `QM` is `CAT` after the quenched rule and the mass, age and history cuts, and those are not the K-S products' cuts, so a perfectly drawable anchor row can be missing from `QM` and carry no age at all: that is what once left a class labelled `nan`. A galaxy that still has no age cannot be matched, so it is held out of the pool and counted in the printout; $\log M_\star$ falls back to the K-S table's own `log_mstar_anchor`; and if that leaves a class with nothing, it is drawn unmatched and best-resolved with an explicit message. Nothing is printed or drawn as `nan` — a missing number reads `n/a`.

**The numbers on the figure.** In the **`P24_SIDE` margin (right by default), for every row**, horizontally: the class name large and bold in its own colour, then that galaxy's $\log M_\star$, age and $\log M_{\rm dust}/M_{\rm H_2}$ **at the anchor**, and its match cost (the seed is marked *(seed)*) — so the reader can check for themselves that the four are alike in everything except the class. Each *panel* then carries, in a small white box, its own redshift and its own $\log M_{\rm dust}/M_{\rm H_2}$ from the member particles at that epoch, so the ratio can be watched moving along the row: that quantity is what Parts 22 and 23 are about, and this is the same number drawn instead of plotted. `P24_MARGIN` sets the margin's width and `P24_WSPACE` the gutter between panels, which is nearly closed — they are images, not plots.

### The figures

`agn_states_gallery_rgb_cis25` — **the composite.** One channel per component (`P24_CHAN`, default stars $+$ gas $+$ dust), each painted with its own black-to-hue colormap — stars white, gas blue, dust orange — and Screen-blended, so the three stay distinguishable instead of merging into one warm blob. The first panel names the channels in their own colours.

The stretch is `log_stretch`: **`P24_DYN` decades below the `P24_PMAX` percentile of the positive pixels, per channel**, and that choice matters. `RenderRGB`'s percentile clip (still available as `P24_STRETCH = "percentile"`) saturates whenever the source fills less than $100 - v_{\rm max}$ per cent of the frame — every galaxy pixel then sits above the clip and comes out at the *same* value, which is the uniform bright blob on a black field. A fixed dynamic range does not care how much empty sky surrounds the source. Raise `P24_DYN` to bring out faint outskirts, lower it for a high-contrast core. `P24_TAU > 0` additionally attenuates every channel by the dust column, as `RenderRGB` does; with dust drawn as its own channel that double-counts it, so it is off.

`agn_states_gallery_dust_cis25`, `agn_states_gallery_H2_cis25`, `agn_states_gallery_temp_cis25` — **the physical maps**, one figure per `P24_MAPS` field, on a **shared colour scale across all 16 panels**, so any panel can be read against any other. Each carries the **mass-weighted gas velocity field as streamlines**: face-on, a rotating disc closes its streamlines into circles and a dispersion-supported one does not — $\kappa_{\rm rot}$, drawn instead of scored.

`P24_MAPDEF` says what each field *is*, because they do not all project the same way:

* **`"sigma"`** — a surface density (`dust`, `H2`, `HI`, `gas`, `star`, `sfr`) through `ReducedRender.sigma_map`, calibrated so that $\Sigma \times$ pixel area sums to the mass inside the frame, i.e. in $M_\odot\,{\rm kpc^{-2}}$ whatever py-sphviewer's internal normalisation is (particles outside the frame still smear in at the edges, so that equality is a calibration, not a budget).
* **`"weighted"`** — the **mass-weighted line-of-sight mean** of a per-particle gas quantity through `.weighted_map`, which is what a temperature needs: $T$ is not a column and must not be summed along the sightline. `temp` gives $\langle T \rangle_M$ in K, and the same machinery already carries `zgas` and `vlos` (the latter on a symmetric diverging scale about zero) — add them to `P24_MAPS` to draw them.

Each entry is `(kind, label, unit, log?, cmap, symmetric?)`; `P24_CMAP` overrides just the colour. A log map is scaled between the `P24_PCT` percentiles but shows **at most `P24_MAPDYN` decades** below its top: with the whole region loaded the empty outskirts otherwise drag the low end down and wash the galaxy out. The map panels take the colormap's own dark end as their background, so the empty pixels, the white streamlines and the white labels all read against each other.

**If `P24_SIZE` is `None`** the frame is derived instead: the half-width becomes `P24_EXTFAC` $\times$ the median $R_{50}(\star)$ of the drawn epochs, and only *then* does `P24_EXTLIM` matter — as a **guard on that automatic value rather than a size control**: raising its upper bound changes nothing unless the automatic value was already being clipped by it. Either way the cell prints which one decided.

**Streamlines.** The velocity field behind the streamlines is built from the gas `P24_STREAM_SOURCE` selects, in the panel's own frame: `"slab"` (default) takes every gas particle of the loaded region within ±`P24_STREAM_DEPTH` kpc (3) of the galaxy's mid-plane along the line of sight — enough particles for smooth flows, without the CGM in front and behind; `"members"` takes the CAESAR members only (too few for nice flows); `"region"` everything loaded (the CGM and every neighbour in the frame, which is what the field used before). The velocity subtracted first is the galaxy's own systemic velocity (`P24_STREAM_BULK = "members"`: its member gas, its member stars if it has no gas) rather than the mean of whatever was selected. `P24_STREAM_WEIGHT` picks the weight (`"m_gas"`, `"m_H2"` for the molecular ISM alone, `"m_dust"`); the cell prints how many particles each panel's field used of those in the frame, and the CSV carries the counts.

**Knobs.** `P24_SIZE` / `P24_FS` the two above, `P24_STAGES` the columns, `P24_SEEDCLS` / `P24_SEEDMAX` / `P24_DGR` / `P24_SAMEANCHOR` / `P24_COSTMAX` / `P24_SEEDTRY` / `P24_ZMIN` / `P24_TOLAGE` / `P24_TOLM` / `P24_RESW` / `P24_ALLANCH` / `P24_PICK` the four galaxies, `P24_SIDE` / `P24_MARGIN` the label margin, `P24_WSPACE` the gutter, `P24_CHAN` / `P24_DYN` / `P24_PMAX` / `P24_STRETCH` / `P24_TAU` the composite, `P24_MAPS` / `P24_MAPDEF` / `P24_CMAP` / `P24_PCT` / `P24_MAPDYN` the maps, `P24_STREAM` / `P24_NGRID` the velocity field, `P24_EXTFAC` / `P24_EXTLIM` the automatic frame, `P24_SPOS` / `P24_MEMBERS` / `P24_XSIZE` what is rendered, `P24_REDUCED` / `P24_PREFIX` the cutout tree.

`agn_states_gallery_cis25.csv` records, per panel, the ids, the epoch, the redshift, the frame size, and **both** sets of masses: `frame_*` = what is actually drawn in the panel (the region inside the frame) and `member_*` = the catalogue galaxy itself, which reproduces `ks_track_epochs` exactly — so the picture can always be tied back to the tables even when the whole region is rendered.

Needs Part 22 to have run (it defines the classes and the galaxy set) and the reduced cutouts on disk; each is gated with a message.

**The projection frame (2026-09-04).** `faceon` / `edgeon` used to refer to the stored *stellar principal frame* — the inertia tensor of the member stars — which for a spheroidal quenched galaxy is nearly degenerate and unrelated to where its gas disc lies, so the switch changed nothing visible. `P24_FRAME = "H2_L"` now builds the frame from the **angular momentum of the member H$_2$ inside `P24_FRAME_RMAX` (10 kpc)** — the ISM's own spin, members only whatever `P24_MEMBERS` says, so the CGM and the neighbours cannot set it — with `faceon` looking down that spin and `edgeon` along the disc (`ReducedRender(frame=...)`, tested in `tests/test_rendering_frame.py`). `"gas_L"` / `"dust_L"` / `"star_L"` are the other spins, `"stellar"` the old frame. A file without velocities, or with fewer than 20 weighted particles inside the radius, falls back to the stellar frame and the cell prints which panels did, with the tilt between the H$_2$ spin and the stellar minor axis for the rest.

**Why the first H$_2$-spin gallery still looked tilted (2026-09-04, later).** The frame was applied to all 16 panels, but the four galaxies of that pick have $\kappa_{\rm rot}$(H$_2$) of 0.1-0.2 at every epoch — below the isotropic $1/3$ — so their spin vectors are noise and there is no disc to align: a random tilt is the physical answer. Two things changed. The renderer now reports the spin's **coherence**, $|\sum \mathbf{l}_i| / \sum |\mathbf{l}_i|$ (1 for a disc, $\sim 1/\sqrt{n}$ for random motions), and `P24_FRAME_STAGE = "sft"` gives every row **one** frame, the spin measured at that critical point (`ReducedRender.set_evecs`), so the panels of a row no longer jump against each other; the report prints each panel's coherence, its own epoch's coherence, and the tilt against the stellar minor axis. And `P24_PICK` now names the **rotating quadruple** 116/61, 116/119, 116/42, 110/71 ($\kappa$ 0.77 / 0.64 / 0.69 / 0.49; age 3.5-4.0 Gyr, $\log M_\star$ 10.17-10.64, $\log M_{\rm dust}/M_\star$ $-3.38$ / $-4.74$ / $-4.26$ / $-5.31$, 200-850 gas particles, three at anchor 116) — rotation is rare in the quenched sample (12 of 36 low-dose compact, 4 of 34 low-dose extended, 7 of 70 high-dose compact, 2 of 39 high-dose extended above $\kappa = 0.5$), and this is the only matched set of four that has it. The tighter-matched non-rotating set stays in `P24_PICK_DUSTCONTRAST`.

In [ ]:
# ── Part 24 — THE PICTURES: one representative galaxy per class, at each critical point ──
# Parts 22 and 23 divide the quenched galaxies four ways (AGN dose x H2 geometry) and score the division.
# This part SHOWS it: one representative galaxy per class, rendered at jet_on -> sft -> qt -> anchor, the same
# four critical points the tracks are walked through.
#   FIG 1  the RGB composite: stars + gas, Screen-blended and attenuated by the dust column
#   FIG 2+ one physical map per P24_MAPS field (Sigma_dust, Sigma_H2 by default) on a SHARED log colour scale,
#          with the mass-weighted GAS VELOCITY FIELD as streamlines: face-on, a rotating disc closes its
#          streamlines into circles and a dispersion-supported one does not — kappa_rot drawn instead of scored.
# The renders read the REDUCED PARTICLE CUTOUTS (build_reduced_particles_job.py), i.e. the very particles the KS
# and profile products are measured from, through simbanator.visualization.rendering.ReducedRender — no snapshot
# and no Caesar catalogue, so the whole grid is seconds rather than one full-box load per panel.
# The representative of a class is the galaxy CLOSEST TO ITS CLASS MEDIAN in the three axes that define the
# picture (dose, geometry, dust), among those whose four epochs all have a reduced file; P24_PICK overrides it.
# ══ THE TWO DIALS YOU ARE MOST LIKELY TO WANT ═══════════════════════════════════════════════════════════════════
P24_SIZE    = 15.0             # ◄── HOW BIG EACH IMAGE IS, in kpc a side (physical).  60 = a 60 x 60 kpc box around
                               #     each galaxy.  The cutouts hold particles out to 100 kpc from the centre, so
                               #     anything past ~200 has empty corners and the cell says so.  None = derive it
                               #     from the galaxies themselves (P24_EXTFAC x the median R50(*), inside P24_EXTLIM).
P24_FS      = 1.5              # ◄── HOW BIG THE TEXT IS: one multiplier for every label, title and number drawn here.
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════
P24_BOX     = "cis25"
P24_REDUCED = globals().get("P24_REDUCED_OVERRIDE", os.path.join(os.getcwd(), "output", P24_BOX, "reduced_particles"))
P24_PREFIX  = "m25n512"        # the reduced files' name stem: <prefix>_snapNNN_galGGGGGG.h5
P24_STAGES  = ["jet_on", "sft", "qt", "anchor"]     # the columns, in time order
P24_SPOS    = "faceon"         # "faceon" (looking down the spin axis of P24_FRAME) | "edgeon" (along its disc)
P24_FRAME   = "H2_L"           # what faceon / edgeon refer to: "H2_L" (the angular momentum of the member H2 inside P24_FRAME_RMAX —
                               # the ISM's own spin, 2026-09-04) | "gas_L" | "dust_L" | "star_L" | "stellar" (the stored stellar
                               # principal frame: the inertia tensor of the member stars, nearly degenerate for a spheroid, which is
                               # why faceon / edgeon did nothing visible before). A file without velocities, or with fewer than 20
                               # weighted particles inside the radius, falls back to "stellar" and the cell says so.
P24_FRAME_RMAX = 4.0          # [kpc] the radius the frame's angular momentum is summed inside
P24_FRAME_MEMBERS = True       # ... over the CAESAR members only, whatever P24_MEMBERS says: the CGM must not set the orientation
P24_FRAME_STAGE = None        # ONE frame per row: the spin measured at this critical point orients every panel of the row (None =
                               # each panel its own spin, which jumps from panel to panel when the gas barely rotates). The report
                               # prints each spin's COHERENCE |sum l| / sum |l| (1 = a disc, ~1/sqrt(n) = random): below ~0.3 there
                               # is no disc to orient and any frame is a random tilt — the quenched H2 of most of the sample.
# the frame, in full: P24_SIZE above is the whole story unless it is None, in which case the half-width becomes
# P24_EXTFAC x the median R50(*) of the drawn epochs and P24_EXTLIM is a GUARD ON THAT AUTOMATIC VALUE — raising its
# upper bound does nothing unless the automatic value was already being clipped by it.  The cell prints which decided.
P24_EXTFAC  = 3.5              # P24_SIZE is None: the half-width in units of the median R50(*) of the drawn epochs
P24_EXTLIM  = (8.0, 20.0)      # P24_SIZE is None: the bounds that automatic half-width is clipped to [kpc]
P24_EXT     = None if P24_SIZE is None else 0.5 * float(P24_SIZE)      # the half-width the code works in
P24_SIDE    = "right"          # which margin carries the row labels + numbers: "right" | "left"
P24_MARGIN  = 0.21             # the fraction of the figure width that margin gets (the class names are long)
P24_WSPACE  = 0.015            # the gap between panels, as a fraction of a panel (both directions)
P24_XSIZE   = 400              # render size [pixels] of the composites and maps
P24_MEMBERS = False            # False = the WHOLE 100 kpc cutout (CGM + neighbours), which is also what makes the
                               # smoothing lengths right: with members only, hsml is estimated from the members alone
                               # and the ISM comes out blobbier and smaller than it is. True = catalogue members only.
P24_CHAN    = ("star", "gas", "dust")   # the composite's channels, each painted with its own black-to-hue colormap
P24_DYN     = (1.0, 2.5, 2.5)  # decades shown per channel (log_stretch): more = fainter outskirts visible
P24_PMAX    = 99.9             # the percentile of the POSITIVE pixels that saturates to white, per channel
P24_STRETCH = "dyn"            # "dyn" = log_stretch (fixed dynamic range); "percentile" = RenderRGB's clip, which
                               # saturates a small source into a uniform blob — kept only for comparison
# THE FOUR GALAXIES: seeded on the class that has to carry the low-dust end, then matched to it in age and mass.
P24_SEEDCLS = "hi_lo"          # the class the quadruple is seeded from ("hi_lo" = high dose + the lower side of P22_GEOM)
P24_DGR     = "dh2"            # THE ratio, for the seed cut and for every number reported: "dh2" = M_dust/M_H2 (what
                               # P22_DUST means), "dg" = M_dust/M_gas
P24_SEEDMAX = -3.0             # the seed must be below this in log of that ratio — the low-dust end of its class;
                               # None = no cut.  Among those, the best-resolved galaxy wins (the only place resolution
                               # decides anything)
P24_SAMEANCHOR = True          # PREFER the seed's own anchor for the other three — but only as a preference: a class
P24_COSTMAX = 1.0              # that cannot be matched there to within P24_COSTMAX is taken from whichever anchor
                               # matches best, and the cell says so.  A matched pair at two redshifts says more about
                               # the classes than a mismatched pair at one
P24_SEEDTRY = 5                # how far down the seed's resolution order to walk looking for one the other three CAN
                               # be matched to; the least-bad seed is kept if none of them can
P24_ZMIN    = 0.5              # the anchor must be above this redshift
P24_TOLAGE  = 0.4              # Gyr:  how age trades against mass in the match; the cost is in units of it, so a cost
P24_TOLM    = 0.12             # dex:  of 1 means "off by one tolerance" — the cell prints each galaxy's cost
P24_RESW    = 0.0              # the weight resolution carries when matching the other three: 0 = matching only, which
                               # is the point — a well-matched galaxy beats a better-sampled one
P24_ALLANCH = True             # draw from every anchor row (Part 23's A23, a bigger pool) when Part 23 has run;
                               # False, or Part 23 not run, = Part 22's one-per-galaxy set
P24_MAPS    = ["dust", "H2", "temp"]     # the physical maps drawn, one figure each: any key of P24_MAPDEF
#      name -> (kind, label, unit, log?, cmap, symmetric?).  "sigma" = a surface density through ReducedRender.sigma_map;
#      "weighted" = the mass-weighted line-of-sight mean of a per-particle gas quantity through .weighted_map, which is
#      what a temperature (or a velocity, or a metallicity) needs — a temperature does not project as a column.
P24_MAPDEF  = {
    "dust": ("sigma",    r"\Sigma_{\rm dust}",       r"M_\odot\,{\rm kpc^{-2}}",    True,  "magma",   False),
    "H2":   ("sigma",    r"\Sigma_{\rm H_2}",        r"M_\odot\,{\rm kpc^{-2}}",    True,  "viridis", False),
    "HI":   ("sigma",    r"\Sigma_{\rm HI}",         r"M_\odot\,{\rm kpc^{-2}}",    True,  "cividis", False),
    "gas":  ("sigma",    r"\Sigma_{\rm gas}",        r"M_\odot\,{\rm kpc^{-2}}",    True,  "bone",    False),
    "star": ("sigma",    r"\Sigma_\star",            r"M_\odot\,{\rm kpc^{-2}}",    True,  "afmhot",  False),
    "sfr":  ("sigma",    r"\Sigma_{\rm SFR}",        r"M_\odot\,{\rm yr^{-1}\,kpc^{-2}}", True, "inferno", False),
    "temp": ("weighted", r"\langle T \rangle_M",     r"{\rm K}",                    True,  "inferno", False),
    "zgas": ("weighted", r"\langle Z_{\rm gas} \rangle_M", "",                      True,  "cubehelix", False),
    "vlos": ("weighted", r"\langle v_{\rm los} \rangle_M", r"{\rm km\,s^{-1}}",     False, "RdBu_r",  True),
}
P24_STREAM  = False             # overlay the gas velocity field as streamlines
P24_NGRID   = 8               # the velocity field grid (per side)
P24_STREAM_DENSITY = 1.0       # HOW MANY streamlines: matplotlib seeds them on its own 30 x 30 base grid scaled by this
                               #   (0.5 = half as many lines, 2 = twice) — P24_NGRID only sets the RESOLUTION of the
                               #   velocity field the lines follow, not their number
P24_STREAM_SOURCE = "slab"     # WHICH gas feeds the velocity field: "slab" = every gas particle of the loaded region within
                               #   +-P24_STREAM_DEPTH kpc of the galaxy's mid-plane along the line of sight (enough particles for
                               #   smooth flows, without the CGM in front and behind) | "members" = the CAESAR members only (too few
                               #   for nice flows) | "region" = everything loaded (the CGM and every neighbour in the frame too)
P24_STREAM_DEPTH  = 2.0        # [kpc] the half-thickness of that slab along the line of sight (face-on: the disc plane +- this)
P24_STREAM_BULK   = "members"  # the velocity subtracted before drawing: "members" = the galaxy's own systemic velocity (its member gas,
                               #   its member stars if it has no gas) | "slab" = the mass-weighted mean of the particles used
P24_STREAM_WEIGHT = "m_gas"    # what weights the field: "m_gas" (all the gas) | "m_H2" (the molecular ISM only) | "m_dust"
P24_STREAM_LW = 0.85           # the streamline width
P24_STREAM_ARROW = 0.8         # the arrow size
P24_TAU     = 0.0              # >0 also attenuates every channel by the dust column; with dust drawn as its own
                               # channel that double-counts it, so it is off
P24_CMAP    = {}               # per-field colormap overrides of P24_MAPDEF, e.g. {"temp": "turbo"}
P24_PICK    = {"lo_hi": (116, 61), "lo_lo": (116, 119), "hi_hi": (116, 42), "hi_lo": (110, 71)}
                               # {"lo_hi": (anchor_snap, gal_id), ...} names the representatives by hand. {} = the seeded match below.
                               # THE ROTATING QUADRUPLE (2026-09-04): the only one whose H2 has a disc to orient — kappa_rot(H2) 0.77 /
                               # 0.64 / 0.69 / 0.49 (the median quenched galaxy sits at 0.2, below the isotropic 1/3, and no frame can
                               # align a spin that is noise); age 4.01 / 3.62 / 4.02 / 3.48 Gyr, log M* 10.46 / 10.17 / 10.64 / 10.39,
                               # log M_dust/M* -3.38 / -4.74 / -4.26 / -5.31 (0.9-1.9 dex below the dusty one), 201 / 527 / 405 / 853
                               # gas particles, three at anchor 116 (z 0.70). The tighter-matched but NON-rotating set (kappa 0.1-0.2,
                               # age 1.29-1.55, log M* 10.34-10.53, dust -3.40 / -4.05 / -5.77 / -4.24) is
P24_PICK_DUSTCONTRAST = {"lo_hi": (100, 45), "lo_lo": (90, 43), "hi_hi": (95, 60), "hi_lo": (90, 31)}   # P24_PICK = P24_PICK_DUSTCONTRAST
P24_PCT     = (2.0, 99.5)      # the shared colour scale of a map grid, as percentiles of its finite pixels
P24_MAPDYN  = 4.0              # ... but a LOG map shows at most this many decades below its top: with the whole region
                               # loaded the empty outskirts drag the low percentile down and wash the galaxy out.
                               # None = the percentile alone

if "ANC22" not in globals():
    print("Part 24: needs Part 22 — run it first (it defines the dose x geometry classes and the galaxy set).")
elif not os.path.isdir(P24_REDUCED):
    print(f"Part 24: {P24_REDUCED} is missing — build_reduced_particles_job.py writes the cutouts (cluster).")
else:
    _fs24 = lambda v: float(v) * float(P24_FS)     # one dial for every label in this part
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    import matplotlib.patheffects as pe
    try:
        from simbanator.visualization.rendering import ReducedRender
    except Exception as _e24:
        ReducedRender = None
        print(f"Part 24: simbanator.visualization.rendering.ReducedRender not importable ({_e24}) — "
              f"it needs py-sphviewer + h5py.")
    if ReducedRender is not None:
        # ── where every galaxy is at every critical point, and which of those have a cutout on disk ──
        _EP24 = {(k, s): (int(sn), int(gx)) for k, s, sn, gx
                 in zip(EPO22["key"], EPO22["stage"], EPO22["snap"], EPO22["gx"])}
        _path24 = lambda sn, gx: ReducedRender.epoch_path(P24_REDUCED, sn, gx, P24_PREFIX)

        def _stream24(rr):
            """The velocity field behind the streamlines, from the gas P24_STREAM_SOURCE selects (in rr's CURRENT frame), the bulk
            velocity P24_STREAM_BULK names subtracted -> ((X, Y, VX, VY, den), (n_used, n_in_frame))."""
            g = rr.parts.get("gas", {})
            if "pos" not in g or "vel" not in g:
                return None, (0, 0)
            xyz = rr.project("gas", P24_SPOS); vel = rr.project("gas", P24_SPOS, what="vel")
            w = np.asarray(g.get(P24_STREAM_WEIGHT, g.get("m_gas")), float)
            inframe = (np.abs(xyz[:, 0]) <= _EXT24) & (np.abs(xyz[:, 1]) <= _EXT24)
            memb = np.asarray(g["member"], bool) if "member" in g else np.ones(len(w), bool)
            keep = (memb if P24_STREAM_SOURCE == "members" else
                    (np.abs(xyz[:, 2]) <= float(P24_STREAM_DEPTH)) if P24_STREAM_SOURCE == "slab" else np.ones(len(w), bool))
            keep &= np.isfinite(w) & (w > 0) & np.all(np.isfinite(vel), axis=1)
            # the systemic velocity: the galaxy's own (member gas, else member stars), or the slab's mean
            v0 = None
            if P24_STREAM_BULK == "members":
                mm = memb & np.isfinite(w) & (w > 0)
                if mm.sum() >= 5:
                    v0 = np.average(vel[mm], axis=0, weights=w[mm])
                elif "star" in rr.parts and "vel" in rr.parts["star"]:
                    sv = rr.project("star", P24_SPOS, what="vel"); sw = np.asarray(rr.parts["star"]["m_star"], float)
                    sm = (np.asarray(rr.parts["star"]["member"], bool) if "member" in rr.parts["star"] else np.ones(len(sw), bool)) & (sw > 0)
                    if sm.sum() >= 5:
                        v0 = np.average(sv[sm], axis=0, weights=sw[sm])
            if v0 is None:
                v0 = np.average(vel[keep], axis=0, weights=w[keep]) if keep.sum() else np.zeros(3)
            saved = rr.parts["gas"]
            rr.parts["gas"] = {k: np.asarray(v)[keep] for k, v in saved.items()}          # the selection, in place, briefly
            try:
                X, Y, VX, VY, den = rr.velocity_field(ex=_EXT24, spos=P24_SPOS, weight=P24_STREAM_WEIGHT, ngrid=P24_NGRID,
                                                      subtract_bulk=False)
            finally:
                rr.parts["gas"] = saved
            VX = np.where(den > 0, VX - v0[0], 0.0); VY = np.where(den > 0, VY - v0[1], 0.0)
            return (X, Y, VX, VY, den), (int(np.sum(keep & inframe)), int(np.sum(inframe)))

        def _chain24(key):
            """The (snap, gx) of `key` at every P24_STAGES epoch, or None if one is missing on disk."""
            out = {}
            for s_ in P24_STAGES:
                sg = _EP24.get((key, s_))
                if sg is None or not os.path.exists(_path24(*sg)):
                    return None
                out[s_] = sg
            return out

        # ── the four galaxies: SEED ON THE LOW-DUST END, then MATCH THE OTHER THREE IN AGE AND MASS ──
        # A gallery only says something about the classes if the four galaxies are otherwise alike, so they are chosen
        # in one dependent pass rather than class by class:
        #   1. the SEED is the class P24_SEEDCLS (high dose + the lower side of the geometry axis by default) — the one that has to
        #      carry the low-dust end of the argument — restricted to log M_dust/M_H2 < P24_SEEDMAX, and among those
        #      the BEST-RESOLVED galaxy (most gas + star particles), because only there does resolution decide;
        #   2. the other three are then whichever member of their class is CLOSEST TO THE SEED in mass-weighted stellar
        #      age and log M*, from the seed's own anchor when P24_SAMEANCHOR.  Matching is all that decides them:
        #      P24_TOLAGE / P24_TOLM only set how age trades against mass (the cost is in units of them, so 1 = off by
        #      one tolerance), and P24_RESW, the weight resolution carries in that choice, is 0 by default.
        _ANCROW24 = EPO22[EPO22["stage"] == "anchor"].drop_duplicates("key").set_index("key")
        _CAND24 = (A23 if (P24_ALLANCH and "A23" in globals() and "hi_dose" in getattr(A23, "columns", []))
                   else ANC22).copy()
        _pool24 = "every anchor row (Part 23's set)" if _CAND24 is not ANC22 else "Part 22's one-per-galaxy set"
        if P24_ALLANCH and _CAND24 is ANC22:
            _pool24 += " — run Part 23 first for the full per-anchor pool"
        # the numbers the match, the seed cut and the figure are made of, all at the anchor
        # the age comes from the FULL Part 0 catalogue (CAT: every galaxy above log M* 9.5 per box and anchor), not
        # from QM — QM is CAT after the quenched rule and the mass / age / history cuts, and those are not the K-S
        # products' cuts, so a perfectly drawable anchor row can be missing from QM and have no age at all.
        _SRC24 = ("CAT" if "CAT" in globals() else ("QM" if "QM" in globals() else None))
        _Q24 = globals()[_SRC24][globals()[_SRC24]["box"] == P24_BOX].copy() if _SRC24 else None
        if _Q24 is not None:
            with np.errstate(all="ignore"):
                if "log_mstar" not in _Q24.columns:
                    _mst24 = _Q24["mstar"].to_numpy(float)
                    _Q24["log_mstar"] = np.log10(np.where(_mst24 > 0, _mst24, np.nan))
            _Q24 = _Q24[~_Q24.duplicated(subset=["snap", "gal_id"])]
            _Q24 = _Q24.set_index([_Q24["snap"].astype(int), _Q24["gal_id"].astype(int)])
        for _c24, _src24 in (("_age", "age"), ("_lm", "log_mstar")):
            _CAND24[_c24] = (_Q24[_src24].reindex(_CAND24["key"]).to_numpy(float)
                             if (_Q24 is not None and _src24 in _Q24.columns) else np.full(len(_CAND24), np.nan))
        if "log_mstar_anchor" in _ANCROW24.columns:            # the KS table always has the mass; QM does not always
            _CAND24["_lm"] = np.where(np.isfinite(_CAND24["_lm"].to_numpy(float)), _CAND24["_lm"].to_numpy(float),
                                      _ANCROW24["log_mstar_anchor"].reindex(_CAND24["key"]).to_numpy(float))
        for _c24, _src24 in (("_ngas", "ngas_cat"), ("_nstar", "nstar_cat"), ("_z", "z"),
                             ("_mdust", "mdust_cat"), ("_mh2", "mh2_cat"), ("_mgas", "mgas_cat")):
            _CAND24[_c24] = (_ANCROW24[_src24].reindex(_CAND24["key"]).to_numpy(float)
                             if _src24 in _ANCROW24.columns else np.full(len(_CAND24), np.nan))
        with np.errstate(all="ignore"):                          # THE ratio: dust over H2 by default, as in P22_DUST
            _num24 = _CAND24["_mdust"].to_numpy(float)
            _den24 = _CAND24["_mh2" if P24_DGR == "dh2" else "_mgas"].to_numpy(float)
            _CAND24["_dgr"] = np.log10(np.where((_num24 > 0) & (_den24 > 0),
                                                _num24 / np.where(_den24 > 0, _den24, np.nan), np.nan))
        _CAND24["_anchor"] = [int(k[0]) for k in _CAND24["key"]]
        _CAND24["_ok"] = [(_chain24(k) is not None) for k in _CAND24["key"]]       # all four epochs on disk
        _CAND24["_rank"] = 0.5 * (_CAND24["_ngas"].rank(pct=True) + _CAND24["_nstar"].rank(pct=True))

        _GNAME24 = [_n24 for _n24, _, _, _, _ in GRP22]                            # "lo_hi" = low dose + high geometry
        _GINFO24 = {_n24: (_c24, _lab24) for _n24, _, _c24, _, _lab24 in GRP22}
        def _inclass24(df, name):
            _d, _g = [p == "hi" for p in name.split("_")]                          # the way Part 22 builds GRP22
            return (df["hi_dose"].to_numpy(bool) == _d) & (df["hi_geom"].to_numpy(bool) == _g)

        _base24 = _CAND24[_CAND24["_ok"].to_numpy(bool)].copy()
        _zok24 = _base24[_base24["_z"].to_numpy(float) > float(P24_ZMIN)]
        if len(_zok24) == 0:
            print(f"  no candidate anchor is above z = {P24_ZMIN:g} — the redshift floor is ignored")
            _zok24 = _base24
        # a galaxy with no age cannot be matched, so it is kept out of the pool rather than silently drawn blank
        _nz24 = len(_zok24)                                        # above the redshift floor, before the age cut
        _hasam24 = np.isfinite(_zok24["_age"].to_numpy(float)) & np.isfinite(_zok24["_lm"].to_numpy(float))
        _nomatch24 = _zok24[~_hasam24]
        if int((~_hasam24).sum()):
            print(f"  {int((~_hasam24).sum())} of {len(_zok24)} drawable candidates have no age in Part 0's "
                  f"{_SRC24} — they are kept out of the match")
        _zok24 = _zok24[_hasam24]

        def _hand24(name):
            """The galaxy P24_PICK names for this class, if it named one and it is drawable."""
            if name not in P24_PICK:
                return None
            _w = tuple(P24_PICK[name])
            _h = _base24[[k == _w for k in _base24["key"]]]
            if not len(_h):
                print(f"  P24_PICK['{name}'] = {_w} has no cutout at every critical point — ignored")
                return None
            return _h.iloc[0]

        REP24, _miss24, _pick24, _note24 = [], [], {}, ""
        _seedcls24 = P24_SEEDCLS if P24_SEEDCLS in _GNAME24 else _GNAME24[-1]

        def _cost24(row, age, lm):
            """How far a candidate is from (age, log M*), in units of the two tolerances."""
            _d = np.hypot((float(row["_age"]) - age) / float(P24_TOLAGE),
                          (float(row["_lm"]) - lm) / float(P24_TOLM))
            return float(_d) if np.isfinite(_d) else np.inf

        def _others24(seed):
            """The best match to `seed` in each of the other classes -> (picks, costs, crossed, unmatched)."""
            _a0, _m0 = float(seed["_age"]), float(seed["_lm"])
            _ok0 = np.isfinite(_a0) and np.isfinite(_m0)
            _p, _c, _x, _u = {}, {}, [], []
            for _n in _GNAME24:
                if _n == _seedcls24:
                    continue
                _h = _hand24(_n)
                if _h is not None:
                    _p[_n], _c[_n] = _h, (_cost24(_h, _a0, _m0) if _ok0 else np.nan)
                    continue
                _all = _zok24[_inclass24(_zok24, _n)]
                if not len(_all):                                  # nobody in this class has an age at all
                    _fb = _nomatch24[_inclass24(_nomatch24, _n)]
                    if len(_fb):
                        _p[_n], _c[_n] = _fb.iloc[int(np.argmax(np.nan_to_num(_fb["_rank"].to_numpy(float))))], np.nan
                        _u.append(_n)
                    continue
                if not _ok0:
                    _p[_n], _c[_n] = _all.iloc[int(np.argmax(np.nan_to_num(_all["_rank"].to_numpy(float))))], np.nan
                    continue
                _pool = _all
                if P24_SAMEANCHOR:
                    _same = _all[_all["_anchor"].to_numpy(int) == int(seed["_anchor"])]
                    _pool = _same if len(_same) else _all
                _score = lambda df: (np.array([_cost24(r, _a0, _m0) for _, r in df.iterrows()])
                                     - float(P24_RESW) * np.nan_to_num(df["_rank"].to_numpy(float)))
                _best = _pool.iloc[int(np.argmin(_score(_pool)))]
                # the seed's own anchor is a preference, not a cage: if it cannot match this class to within
                # P24_COSTMAX and another anchor can do better, take that and say so — a matched pair at two
                # redshifts says more about the classes than a mismatched pair at one.
                if P24_SAMEANCHOR and len(_pool) < len(_all) and _cost24(_best, _a0, _m0) > float(P24_COSTMAX):
                    _alt = _all.iloc[int(np.argmin(_score(_all)))]
                    if _cost24(_alt, _a0, _m0) < _cost24(_best, _a0, _m0):
                        _best, _ = _alt, _x.append(_n)
                _p[_n], _c[_n] = _best, _cost24(_best, _a0, _m0)
            return _p, _c, _x, _u

        _seed24, _why24 = _hand24(_seedcls24), "named by P24_PICK"
        _cost24s, _cross24, _unm24 = {}, [], []
        if _seed24 is not None:
            _pick24[_seedcls24] = _seed24
            _o, _cost24s, _cross24, _unm24 = _others24(_seed24)
            _pick24.update(_o)
        else:
            _sp24 = _zok24[_inclass24(_zok24, _seedcls24)]                 # the seed must be matchable itself
            _cut24 = _sp24[_sp24["_dgr"].to_numpy(float) < float(P24_SEEDMAX)] if P24_SEEDMAX is not None else _sp24
            _why24 = (f"log M_dust/M_{'H2' if P24_DGR == 'dh2' else 'gas'} < {P24_SEEDMAX:g}, "
                      f"{len(_cut24)} of {len(_sp24)}")
            if not len(_cut24) and len(_sp24):
                _cut24 = _sp24.nsmallest(3, "_dgr")
                _why24 = (f"NO member of this class is below {P24_SEEDMAX:g} — the three lowest were used instead "
                          f"(lowest {np.nanmin(_sp24['_dgr']):.2f})")
            # the best-resolved low-dust galaxy is the FIRST choice, but a seed the other three cannot be matched to
            # is a bad seed: walk down the resolution order until one of them can be matched to within P24_COSTMAX,
            # and if none can, keep the one whose worst match is least bad.
            _cut24 = _cut24.sort_values("_rank", ascending=False)
            _try24, _best24 = min(int(P24_SEEDTRY), len(_cut24)), None
            for _i24 in range(_try24):
                _cand24 = _cut24.iloc[_i24]
                _o, _c, _x, _u = _others24(_cand24)
                _worst = max([v for v in _c.values() if np.isfinite(v)], default=np.inf)
                if _best24 is None or _worst < _best24[0]:
                    _best24 = (_worst, _i24, _cand24, _o, _c, _x, _u)
                if _worst <= float(P24_COSTMAX):
                    break
            if _best24 is not None:
                _worst, _i24, _seed24, _o, _cost24s, _cross24, _unm24 = _best24
                _pick24[_seedcls24] = _seed24
                _pick24.update(_o)
                _why24 += ("; best resolved of those" if _i24 == 0 else
                           f"; #{_i24 + 1} in resolution order — the {_i24} better-resolved ones could not be "
                           f"matched to within {P24_COSTMAX:g}")
                if not np.isfinite(_worst) or _worst > float(P24_COSTMAX):
                    _why24 += f", and neither could this one (worst match {_worst:.2f})"
        for _n24 in _cross24:
            print(f"  '{_GINFO24[_n24][1]}' matches better at another anchor than the seed's — taken from there "
                  f"(P24_SAMEANCHOR is a preference, P24_COSTMAX the threshold)")
        for _n24 in _unm24:
            print(f"  '{_GINFO24[_n24][1]}' has no member with an age — drawn unmatched, best resolved")

        for _n24 in _GNAME24:
            if _n24 not in _pick24:
                _miss24.append(_GINFO24[_n24][1])
                continue
            _r24 = _pick24[_n24]
            _col24, _lab24 = _GINFO24[_n24]
            REP24.append(dict(group=_n24, label=_lab24, color=_col24, key=_r24["key"], chain=_chain24(_r24["key"]),
                              seed=(_n24 == _seedcls24), dose=float(_r24[P22_DOSE]), geom=float(_r24[P22_GEOM]),
                              z=float(_r24["_z"]), age=float(_r24["_age"]), lmstar=float(_r24["_lm"]),
                              ldgr=float(_r24["_dgr"]), ngas=float(_r24["_ngas"]), nstar=float(_r24["_nstar"]),
                              cost=(0.0 if _n24 == _seedcls24 else float(_cost24s.get(_n24, np.nan)))))
        for _lb in _miss24:
            print(f"  no galaxy of '{_lb}' survives the cuts with all of {', '.join(P24_STAGES)} on disk "
                  f"— that row is dropped")
        if not REP24:
            print("Part 24: nothing to draw — no class has a galaxy with a cutout at every critical point.")
        _DGLAB24 = (r"\log\,M_{\rm dust}/M_{\rm H_2}" if P24_DGR == "dh2" else r"\log\,M_{\rm dust}/M_{\rm gas}")
        _DGSHORT24 = (r"\log\,M_{\rm d}/M_{\rm H_2}" if P24_DGR == "dh2" else r"\log\,M_{\rm d}/M_{\rm g}")
        _fignote24 = ""
        if REP24:
            _ages24 = [_r["age"] for _r in REP24 if np.isfinite(_r["age"])]
            _lms24 = [_r["lmstar"] for _r in REP24 if np.isfinite(_r["lmstar"])]
            _anc24 = sorted({_r["key"][0] for _r in REP24})
            _note24 = ("seeded on " + _GINFO24[_seedcls24][1].replace("$", "").replace("_", "")
                       + f", then matched in age and mass;  anchor"
                       + (f" snap {_anc24[0]} (z = {REP24[0]['z']:.2f})" if len(_anc24) == 1
                          else "s " + ", ".join(str(a) for a in _anc24))
                       + (f";  age {min(_ages24):.2f}-{max(_ages24):.2f} Gyr (spread {max(_ages24) - min(_ages24):.2f}, "
                          f"tolerance {P24_TOLAGE:g})" if _ages24 else "")
                       + (f", log M* {min(_lms24):.2f}-{max(_lms24):.2f} (spread {max(_lms24) - min(_lms24):.2f}, "
                          f"tolerance {P24_TOLM:g})" if _lms24 else ""))
            _fignote24 = ("seeded on " + _GINFO24[_seedcls24][1] + r", matched in age and $M_\star$"
                          + (rf";  anchor {_anc24[0]}, $z = {REP24[0]['z']:.2f}$" if len(_anc24) == 1 else "")
                          + (rf";  spread {max(_ages24) - min(_ages24):.2f} Gyr" if _ages24 else "")
                          + (rf", {max(_lms24) - min(_lms24):.2f} dex" if _lms24 else ""))
        print(f"Part 24 — the pictures, {BOX_LABEL.get(P24_BOX, P24_BOX)}: {len(REP24)} classes x {len(P24_STAGES)} "
              f"critical points, {P24_SPOS}, "
              f"{'catalogue members only' if P24_MEMBERS else 'the whole 100 kpc cutout'}.")
        print(f"  the pool: {_pool24}, {len(_base24)} with all four cutouts, {_nz24} of them above z = "
              f"{P24_ZMIN:g}, {len(_zok24)} of those with an age to match on.  "
              f"The seed is '{_GINFO24[_seedcls24][1]}' ({_why24}).")
        if _note24:
            print("  " + _note24 + ".")
        for _r in REP24:
            _f24n = lambda v, f="{:.2f}": (f.format(v) if np.isfinite(v) else "n/a")
            print(f"    {'SEED' if _r['seed'] else '    '} {_r['label']:42s} anchor_snap {_r['key'][0]:3d} "
                  f"gal {_r['key'][1]:5d}  z {_r['z']:.2f}  age {_f24n(_r['age'])} Gyr  "
                  f"log M* {_f24n(_r['lmstar'])}  "
                  f"{'dh2' if P24_DGR == 'dh2' else 'D/G'} {_f24n(_r['ldgr'])}  "
                  + ("" if not np.isfinite(_r["cost"]) else f"match cost {_r['cost']:.2f}  ")
                  + f"({_r['ngas']:.0f} gas + {_r['nstar']:.0f} star particles)")
        if REP24 and any(np.isfinite(_r["cost"]) and _r["cost"] > 1 for _r in REP24):
            print("  NOTE a match cost above 1 means that galaxy is more than one tolerance from the seed in the "
                  "(age, log M*) plane — loosen P24_ZMIN, raise P24_SEEDMAX, or accept the spread.")

        # ── the frame: one half-width for the whole grid, and a line saying which knob set it ──
        if P24_EXT is not None:
            _EXT24 = float(P24_EXT)
            _why24 = (f"P24_SIZE = {2 * _EXT24:g} kpc a side (P24_EXTFAC and P24_EXTLIM are not used)")
        else:
            _r50_24 = (EPO22.groupby(["snap", "gx"])["r50star_cat"].first()      # (snap, gx) repeats across anchors
                       if "r50star_cat" in EPO22.columns else None)
            _rr24 = [float(_r50_24.get(sg, np.nan)) for r in REP24 for sg in r["chain"].values()] \
                if _r50_24 is not None else []
            _rr24 = [v for v in _rr24 if np.isfinite(v) and v > 0]
            _raw24 = round(P24_EXTFAC * float(np.median(_rr24))) if _rr24 else 15.0
            _EXT24 = float(np.clip(_raw24, *P24_EXTLIM))
            _why24 = (f"P24_EXTFAC {P24_EXTFAC:g} x median R50(*) "
                      + (f"{np.median(_rr24):.1f} kpc = {_raw24:g}" if _rr24 else "unavailable -> 15")
                      + (f", CLIPPED by P24_EXTLIM {tuple(P24_EXTLIM)}" if _EXT24 != _raw24 else
                         f"; P24_EXTLIM {tuple(P24_EXTLIM)} did not bind — SET P24_SIZE (kpc a side) for a larger "
                         f"region"))
        print(f"  every panel is {2 * _EXT24:.0f} kpc across (half-width {_EXT24:g} kpc from {_why24}).")

        # ── the renders: one file open per panel, every product taken from it ──
        REN24, ROW24, _warn24 = {}, [], False
        _frames24 = []                                            # (class, stage, frame_info) per panel: which frame was used
        for _r in REP24:
            _rowframe = None                                      # the row's common frame: the spin at P24_FRAME_STAGE
            if P24_FRAME_STAGE and P24_FRAME != "stellar" and P24_FRAME_STAGE in _r["chain"]:
                _rf = ReducedRender.from_epoch(P24_REDUCED, *_r["chain"][P24_FRAME_STAGE], P24_PREFIX, members_only=P24_MEMBERS,
                                               frame=P24_FRAME, frame_rmax=P24_FRAME_RMAX, frame_members=P24_FRAME_MEMBERS)
                if _rf.frame != "stellar":
                    _rowframe = (_rf.evecs.copy(), dict(_rf.frame_info))
                del _rf
            for _s24 in P24_STAGES:
                _sn, _gx = _r["chain"][_s24]
                _rr = ReducedRender.from_epoch(P24_REDUCED, _sn, _gx, P24_PREFIX, members_only=P24_MEMBERS,
                                               frame=P24_FRAME, frame_rmax=P24_FRAME_RMAX, frame_members=P24_FRAME_MEMBERS)
                _own = dict(getattr(_rr, "frame_info", {}))
                if _rowframe is not None:
                    _rr.set_evecs(_rowframe[0], label=f"{P24_FRAME}@{P24_FRAME_STAGE}", info=_rowframe[1])
                    _own = dict(_rr.frame_info, own_coherence=_own.get("coherence", np.nan), own_n=_own.get("n_used", 0))
                _frames24.append((_r.get("name", _r.get("cls", "")) if isinstance(_r, dict) else "", _s24, _own))
                if np.isfinite(_rr.rmax_kpc) and _EXT24 > _rr.rmax_kpc and not _warn24:
                    print(f"  NOTE the frame half-width {_EXT24:g} kpc runs past the cutout's own rmax_kpc = "
                          f"{_rr.rmax_kpc:g}: the corners are empty because there are no particles there, not because "
                          f"the sky is.")
                    _warn24 = True
                _cell = dict(z=_rr.redshift, ext=[-_EXT24, _EXT24, -_EXT24, _EXT24],
                             rgb=_rr.rgb(ex=_EXT24, spos=P24_SPOS, channels=P24_CHAN, dyn=P24_DYN, pmax=P24_PMAX,
                                         stretch=P24_STRETCH, tau_max=P24_TAU, xsize=P24_XSIZE)[0],
                             tot=_rr.totals(ex=_EXT24, spos=P24_SPOS),
                             memb=_rr.totals(ex=None, spos=P24_SPOS, members=True))
                for _f24 in P24_MAPS:
                    _cell[f"map_{_f24}"] = (
                        _rr.sigma_map(_f24, ex=_EXT24, spos=P24_SPOS, xsize=P24_XSIZE)[0]
                        if P24_MAPDEF[_f24][0] == "sigma" else
                        _rr.weighted_map(_f24, ex=_EXT24, spos=P24_SPOS, xsize=P24_XSIZE)[0])
                if P24_STREAM:
                    _cell["vel"], _cell["vel_n"] = _stream24(_rr)
                REN24[(_r["group"], _s24)] = _cell
                _row = dict(group=_r["group"], label=_r["label"], stage=_s24, anchor_snap=_r["key"][0],
                            gal_id=_r["key"][1], snap=_sn, gx=_gx, z=_rr.redshift, ext_kpc=_EXT24,
                            loaded="members" if P24_MEMBERS else "region",
                            stream_gas=(f"slab +-{P24_STREAM_DEPTH:g} kpc" if P24_STREAM_SOURCE == "slab" else P24_STREAM_SOURCE) if P24_STREAM else "",
                            stream_n_used=_cell.get("vel_n", (0, 0))[0], stream_n_in_frame=_cell.get("vel_n", (0, 0))[1])
                _row.update({f"frame_{k}": v for k, v in _cell["tot"].items()})       # what is drawn in the panel
                _row.update({f"member_{k}": v for k, v in _cell["memb"].items()})     # the catalogue galaxy itself
                ROW24.append(_row)

        _stlab24 = {"jet_on": "Jet start", "sft": "Quenching", "qt": "Quenched", "anchor": "Observed", "agn_ign": "ignition",
                    "sf_peak": "SF peak", "post_quench": "post-q", "gas_min": "gas min", "end": "end"}

        def _grid24(draw, title, fname, cbar=None, cbar_label=None, fg="w", bg="k"):
            """The 4 x 4 frame every figure of this part shares: classes down, critical points across."""
            if not REP24:
                return
            if P24_STREAM and REN24 and "vel_n" in next(iter(REN24.values())):
                _nu = [c["vel_n"] for c in REN24.values()]
                print(f"  streamlines: the {P24_STREAM_WEIGHT}-weighted velocity field of "
                      + {"slab": f"the gas within +-{P24_STREAM_DEPTH:g} kpc of the mid-plane along the line of sight (P24_STREAM_SOURCE 'slab')",
                         "members": "the galaxy's CAESAR member gas only", "region": "EVERY gas particle in the frame (CGM and neighbours included)"}[P24_STREAM_SOURCE]
                      + f", the {'galaxy' if P24_STREAM_BULK == 'members' else 'selection'}'s bulk velocity subtracted"
                      + f" — per panel {min(n for n, _ in _nu)}-{max(n for n, _ in _nu)} particles used of {min(m for _, m in _nu)}-{max(m for _, m in _nu)} in the frame")
            if _frames24:                                             # the frame every panel was drawn in
                _nfb = sum(1 for _, _, fi in _frames24 if fi.get("frame") == "stellar")
                print(f"  projection frame P24_FRAME = {P24_FRAME}" + (f" measured at {P24_FRAME_STAGE} for the whole row" if P24_FRAME_STAGE else "")
                      + f": {len(_frames24) - _nfb} of {len(_frames24)} panels in that frame"
                      + (f", {_nfb} fell back to the stellar frame" if _nfb else "") + " (tilt = angle between that spin and the stellar minor axis):")
                for _lb24, _st24, fi in _frames24:
                    print(f"    {str(_lb24)[:34]:34s} {_st24:7s} {fi.get('frame', '?'):10s} n={fi.get('n_used', 0):5d}"
                          + (f"  coherence {fi['coherence']:.2f}" if np.isfinite(fi.get("coherence", np.nan)) else "")
                          + (f"  (this epoch's own spin: coherence {fi['own_coherence']:.2f}, n={fi['own_n']})" if "own_coherence" in fi else "")
                          + (f"  tilt {fi['tilt_vs_stellar_deg']:5.1f} deg" if "tilt_vs_stellar_deg" in fi else f"  ({fi.get('reason', '')})"))
            fig, axs = plt.subplots(len(REP24), len(P24_STAGES),
                                    figsize=(2.9 * len(P24_STAGES) / (1 - P24_MARGIN), 3.05 * len(REP24) + 0.9),
                                    squeeze=False)
            for _i, _r in enumerate(REP24):
                for _j, _s24 in enumerate(P24_STAGES):
                    ax = axs[_i][_j]
                    _cell = REN24[(_r["group"], _s24)]
                    draw(ax, _cell, _r, _s24)
                    ax.set_xticks([]); ax.set_yticks([])
                    for _sp in ax.spines.values():
                        _sp.set_color(_r["color"]); _sp.set_linewidth(2.0)
                    _dg24 = (_cell["memb"].get("m_dust", np.nan)
                             / max(_cell["memb"].get("m_H2" if P24_DGR == "dh2" else "m_gas", np.nan), 1e-30))
                    _ann24 = f"$z = {_cell['z']:.2f}$" + (f"\n${_DGSHORT24} = {np.log10(_dg24):.2f}$"
                                                         if np.isfinite(_dg24) and _dg24 > 0 else "")
                    ax.text(0.03, 0.97, _ann24, transform=ax.transAxes, ha="left", va="top", linespacing=1.5,
                            fontsize=_fs24(8.5), color="0.1", zorder=12,
                            bbox=dict(boxstyle="round,pad=0.32", facecolor="white", alpha=0.82, edgecolor="none"))
                    if _i == 0:
                        ax.set_title(_stlab24.get(_s24, _s24), fontsize=_fs24(12))
                    if _j == 0:
                        _bar = 5.0 if _EXT24 <= 25 else (10.0 if _EXT24 <= 60 else 25.0)   # once per row
                        _f24b = _bar / (2 * _EXT24)                   # its length as a fraction of the panel
                        ax.plot([0.06, 0.06 + _f24b], [0.065, 0.065], "-", transform=ax.transAxes, color=fg, lw=2.6,
                                zorder=12, path_effects=[pe.withStroke(linewidth=4.2, foreground=bg)])
                        ax.text(0.06 + 0.5 * _f24b, 0.105, f"{_bar:.0f} kpc", transform=ax.transAxes, color=fg,
                                ha="center", va="bottom", fontsize=_fs24(8), zorder=12,
                                path_effects=[pe.withStroke(linewidth=2.2, foreground=bg)])
            #fig.suptitle(title, fontsize=_fs24(13) if "\n" not in title else 11.5)
            _rt24 = P24_SIDE == "right"
            fig.tight_layout(rect=(0 if _rt24 else P24_MARGIN, 0.075 if cbar is not None else 0,
                                   1 - P24_MARGIN if _rt24 else 1, 0.985))
            fig.subplots_adjust(wspace=P24_WSPACE, hspace=P24_WSPACE)     # images: the grid wants almost no gutter
            for _i, _r in enumerate(REP24):                      # written after the layout, so nothing is clipped
                _bb = (axs[_i][-1] if _rt24 else axs[_i][0]).get_position()
                _x, _ha = (_bb.x1 + 0.010, "left") if _rt24 else (_bb.x0 - 0.010, "right")
                _y = _bb.y0 + 0.5 * _bb.height
                _v24 = lambda k: (f"{_r[k]:.2f}" if np.isfinite(_r.get(k, np.nan)) else "n/a")
                _num24 = [rf"$\log M_\star = $ {_v24('lmstar')}", rf"age $= $ {_v24('age')} Gyr",
                          "$" + _DGLAB24 + "$ $=$ " + _v24("ldgr")]
                # if np.isfinite(_r.get("cost", np.nan)) and not _r.get("seed"):
                #     _num24.append(rf"match cost {_r['cost']:.2f}")
                fig.text(_x, _y + 0.010, _r["label"].replace(" $+$ ", "\n"), ha=_ha, va="bottom", fontsize=_fs24(12),
                         color=_r["color"], linespacing=1.35, fontweight="bold")
                fig.text(_x, _y - 0.012, "\n".join(_num24), ha=_ha,
                         va="top", fontsize=_fs24(9), color="0.2", linespacing=1.6)
            if cbar is not None:
                _cax = fig.add_axes([0.30, 0.022, 0.42, 0.013])
                _cb24 = fig.colorbar(cbar, cax=_cax, orientation="horizontal")
                _cb24.ax.tick_params(labelsize=_fs24(9))
                _cax.set_title(cbar_label, fontsize=_fs24(9.5), pad=4)
            paper_save(fig, fname)
            plt.show()

        # ── FIGURE 1: the composite ──
        def _drawrgb24(ax, cell, r, s):
            ax.imshow(cell["rgb"], extent=cell["ext"], origin="lower")
            if r is REP24[0] and s == P24_STAGES[0]:                 # name the channels once, in their own colours
                for _k24, _nm24 in enumerate(P24_CHAN):
                    ax.text(0.965, 0.955 - 0.08 * _k24, _nm24, transform=ax.transAxes, ha="right", va="top",
                            fontsize=_fs24(9.5), weight="bold",
                            color=ReducedRender.CHANNEL_CMAPS.get(_nm24, cm.Greys_r)(0.88),
                            path_effects=[pe.withStroke(linewidth=2.2, foreground="k")])

        _grid24(_drawrgb24,
                rf"Part 24 — one galaxy per class through its own quenching ({BOX_LABEL.get(P24_BOX, P24_BOX)}, "
                rf"{P24_SPOS}, {' $+$ '.join(P24_CHAN)}, {P24_STRETCH} stretch)"
                + ("\n" + _fignote24 if _fignote24 else ""),
                f"agn_states_gallery_rgb_{P24_BOX}")

        # ── FIGURE 2+: the physical maps, one per field, shared colour scale, velocity field on top ──
        for _f24 in (P24_MAPS if REP24 else []):
            _kind24, _lab24, _unit24, _log24, _cm24, _sym24 = P24_MAPDEF[_f24]
            _cm24 = P24_CMAP.get(_f24, _cm24)
            _all24 = np.concatenate([REN24[(r["group"], s)][f"map_{_f24}"].ravel()
                                     for r in REP24 for s in P24_STAGES])
            _all24 = _all24[np.isfinite(_all24)]
            if _log24:
                _all24 = _all24[_all24 > 0]
            if not len(_all24):
                print(f"  the {_f24} maps are empty everywhere — figure skipped")
                continue
            if _sym24:                                     # a diverging quantity: one symmetric scale about zero
                _q24 = float(np.percentile(np.abs(_all24), P24_PCT[1])); _vmn, _vmx = -_q24, _q24
            else:
                _vmn, _vmx = (float(np.percentile(_all24, P24_PCT[0])), float(np.percentile(_all24, P24_PCT[1])))
                if _log24:
                    _vmn, _vmx = np.log10(_vmn), np.log10(_vmx)
                    if P24_MAPDYN is not None:
                        _vmn = max(_vmn, _vmx - float(P24_MAPDYN))
            _pre24 = r"\log\," if _log24 else ""
            _cbl24 = "$" + _pre24 + _lab24 + "$" + ("  [$" + _unit24 + "$]" if _unit24 else "")
            _sm24 = cm.ScalarMappable(norm=mcolors.Normalize(_vmn, _vmx), cmap=_cm24)

            def _draw24(ax, cell, r, s, _f=_f24, _lg=_log24, _c=_cm24, _lo=_vmn, _hi=_vmx):
                ax.set_facecolor(plt.get_cmap(_c)(0.0))    # empty pixels take the colormap's own low end, so the
                _m = cell[f"map_{_f}"]                     # white streamlines and labels read everywhere
                with np.errstate(divide="ignore", invalid="ignore"):
                    _d = np.log10(np.where(_m > 0, _m, np.nan)) if _lg else np.where(np.isfinite(_m), _m, np.nan)
                ax.imshow(_d, extent=cell["ext"], origin="lower", vmin=_lo, vmax=_hi, cmap=_c)
                if P24_STREAM and cell.get("vel") is not None:
                    _X, _Y, _VX, _VY, _den = cell["vel"]
                    if np.any(np.hypot(_VX, _VY) > 0):
                        ax.streamplot(_X, _Y, _VX, _VY, color="w", density=P24_STREAM_DENSITY, linewidth=P24_STREAM_LW, arrowsize=P24_STREAM_ARROW,
                                      zorder=4)
                ax.set_xlim(cell["ext"][0], cell["ext"][1]); ax.set_ylim(cell["ext"][2], cell["ext"][3])

            _grid24(_draw24,
                    "Part 24 — $" + _lab24 + "$ through the critical points, one galaxy per class "
                    + f"({BOX_LABEL.get(P24_BOX, P24_BOX)}, {P24_SPOS}"
                    + ((f", the gas velocity field within $\\pm${P24_STREAM_DEPTH:g} kpc of the mid-plane as streamlines)" if P24_STREAM_SOURCE == "slab" else
                        ", the member gas velocity field as streamlines)" if P24_STREAM_SOURCE == "members" else ", the region's gas velocity field as streamlines)") if P24_STREAM else ")")
                    + ("\n" + _fignote24 if _fignote24 else ""),
                    f"agn_states_gallery_{_f24}_{P24_BOX}", cbar=_sm24, cbar_label=_cbl24,
                    fg="w", bg="k")

        if ROW24:
            _p24 = os.path.join(OUT, f"agn_states_gallery_{P24_BOX}.csv")
            pd.DataFrame(ROW24).to_csv(_p24, index=False)
            print(f"  what was drawn, panel by panel (ids, epochs, masses inside the frame) -> {_p24}")

## Part 25 — where the dust went, and whether the H$_2$ came back: the budget along the branch since SFT, core → CGM

Parts 22-24 divide the quenched galaxies by AGN dose × H$_2$ extent and show the dust follows the division. This part asks the two questions that division raises, on **each galaxy's own clock $t - t_{\rm SFT}$** and with the same four groups (Part 22's frame, one chain per galaxy): *did the dust move from the ISM into the CGM, or vanish in place?* and *did the central H$_2$ of the compact galaxies arrive after quenching (re-accretion), or is it what was left?* Three blocks, each gated on its inputs:

* **(A) the branch itself** — from the histories cache, every snapshot: the galaxy's total $M_{\rm dust}$, $M_{\rm H_2}$ and their ratio against $t - t_{\rm SFT}$ per group; the **X-ray occupancy** (the fraction of each group in the X-ray state at each moment since SFT); the distribution of **$t_{\rm quiet}$**, the time since the last exposure at the anchor (never-exposed galaxies counted apart); and — panel (f) — **where the dust is**: the CGM share of the 100 kpc dust on the continuous clock (the same track as (C)'s panel (c), moved here on 2026-09-04 so the branch figure reads end to end; the regrowth scatter it replaced is the fallback when the budget products are absent). Per galaxy the **regrowth** — how far above its post-SFT minimum the H$_2$ and the dust stand at the anchor (`P25_REGROW` $= 0.3$ dex = "came back") — is printed: a galaxy whose H$_2$ climbs back after its minimum accreted; one whose dust climbs back regrew it.
* **(B) the critical points, member particles** — from the K-S track measurements at `sft` → `qt` → `post_quench` → `gas_min` → `anchor`, each group's median $M_{\rm dust}$ and $M_{\rm H_2}$ in the **core** (0-1 kpc), the **ring** (1-3.2 kpc) and the **outskirt** (3.2-10 kpc), placed at the group's own median $t - t_{\rm SFT}$ of each stage, plus the core's share of the $<10$ kpc mass. *Projected annuli cannot see a 3-D hole; the 3-D core here is a sphere.* The 3-D check on the anchors (m25 8j0 kinematics table, every particle): the two **extended** groups hold a median of **zero H$_2$ particles inside 1 kpc** (5 gas particles) against 21 (23) for the low-dose compact group — their H$_2$ sits in the 1-3 kpc ring. The extended groups do have a central cavity; the compact low-dose group does not.
* **(C) the whole 100 kpc cutout** — from `ks_tracks/ks_stage_budget.fits` (ks_tracks_quenched_m25 Part 3c, every particle, spherical shells core / ring / outskirt / `cgm_in` 10-30 / `cgm_out` 30-100 kpc, radial velocities about the stellar systemic velocity): the dust in the ISM ($<10$ kpc) against the dust in the CGM, the CGM share, the dust-weighted $v_r$ of the outskirt and of the inner CGM (positive = outflow), the share of the CGM dust **above the local escape speed** (panel (e), solid — the sharpened 'outflowing' of 2026-09-04: $v_r$ above the NFW escape speed of the *parent* halo at the particle's radius, caesar $M_{200c}$ / $R_{200c}$ with the Dutton & Macciò 2014 concentration, so a slow drift no longer counts; the loose $v_r > 0$ share is the dashed comparison) and the share in gas above $10^5$ K, and the core H$_2$ with its particle count. At the critical points always; **on a continuous clock** once Part 2b's post-SFT cutouts exist (`ks_postsft_map.csv`), binned in `P25_TB`. **The population (first cluster run, 2026-09-04, critical points, 1335 cutouts):** the CGM share of the 100 kpc dust rises from 0.4 at SFT to 0.8-0.9 at the anchor in every group; the ISM dust falls by 1.2-2.3 dex while the CGM dust falls by only 0.3-0.7 dex — *the dust that survives is the dust in the CGM, the ISM dust is what disappears*. The CGM dust is **outflowing, not falling back**: $v_r$(10-30 kpc) jumps from $+12$ km/s at ignition to $+150$ at the jet onset and SFT ($f_{\rm out} \approx 0.9$), decelerates through QT ($+100$) and reaches $\approx 0$ (high dose: $-4$, $-5$) or stays mildly outflowing (low-dose compact: $+98$) at the anchor; only 24 % of the 10-30 kpc rows have $v_r < 0$. The one shell with a population-level inflow is the **outskirt** (3.2-10 kpc) of the two high-dose groups after QT ($-40$ to $-70$ km/s, in the CSV as `vr_dust_outskirt`). The Part 24 low-dose compact example, whose shells were inflowing at its anchor, is not representative.

* **(D) where the material of each zone is going** — `agn_states_budget_flows`: for the ring (1-3.2 kpc), the outskirt (3.2-10 kpc) and the CGM (10-100 kpc), the share of the zone's dust (top row) and H$_2$ (bottom row) on particles moving **outward** ($v_r > 0$, solid) and **escaping** ($v_r > v_{\rm esc}$, dashed, open markers); the inward share is the complement. Above the 0.5 line more of the zone's material heads for the CGM than for the core. On the continuous clock where the post-SFT cutouts exist, else at the critical points; the anchor values are printed.

**Knobs.** `P25_TG` (the clock of A), `P25_STG` (the critical points of B/C), `P25_TB` (the bins of the continuous C), `P25_REGROW`, `P25_NMIN`, `P25_KSDIR` (the K-S track products).


In [ ]:
# ── Part 25 — where the dust went, and whether the H2 came back: the budget along the branch since SFT, core -> CGM ──
# (A) the histories: the galaxy's total M_dust / M_H2 on t - t_SFT per group, the X-ray occupancy, t_quiet, and — panel (f) — the CGM
#     share of the dust on the continuous clock (the regrowth scatter is the fallback when the budget products are absent; the
#     regrowth statistics are printed either way).
# (B) the critical points, member particles: M_dust / M_H2 in the core (0-1), the ring (1-3.2) and the outskirt (3.2-10 kpc).
# (C) the 100 kpc cutouts (ks_stage_budget.fits, every particle): ISM against CGM dust, the CGM share, the radial flows, the ESCAPING
#     share (v_r above the parent halo's NFW escape speed — the sharpened 'outflowing', 2026-09-04) and the core H2 — at the critical
#     points, and on a continuous clock once the post-SFT cutouts exist (ks_postsft_map.csv).
# (D) where the dust and the H2 of each zone are going: the outward (v_r > 0), inward (v_r < 0) and escaping shares in the ring,
#     the outskirt and the CGM, per group — is the disc material heading for the core or for the CGM?
P25_BOX    = "cis25"
P25_TG     = np.arange(-1.0, 3.01, 0.1)                       # the clock of (A), t - t_SFT [Gyr]
P25_STG    = ["sft", "qt", "post_quench", "gas_min", "anchor"] # the critical points of (B) and (C), in the order they are joined
P25_TB     = np.arange(-0.5, 3.01, 0.25)                      # the bins of the continuous (C) / (D) tracks
P25_REGROW = 0.3                                              # dex above the post-SFT minimum at the anchor = "came back"
P25_NMIN   = 8
P25_KSDIR  = globals().get("P25_KSDIR_OVERRIDE", P22_KSDIR if "P22_KSDIR" in globals() else os.path.join(os.getcwd(), "output", P25_BOX, "ks_tracks"))
P25_BUDGET = os.path.join(P25_KSDIR, "ks_stage_budget.fits")
P25_MAP    = os.path.join(P25_KSDIR, "ks_postsft_map.csv")
P25_ZONES  = [("core", "ap1kpc", None, "core 0-1 kpc"), ("ring", "ap3kpc", "ap1kpc", "ring 1-3.2 kpc"),
              ("outskirt", "ap10kpc", "ap3kpc", "outskirt 3.2-10 kpc")]
MRK25 = {"sft": "s", "qt": "D", "post_quench": "P", "gas_min": "X", "anchor": "o"}
LAB25 = {"sft": "SFT", "qt": "QT", "post_quench": "post-q", "gas_min": "gas min", "anchor": "anchor"}

if "GRP22" not in globals():
    print("Part 25: needs Part 22 — run it first (it defines the dose x geometry groups, the chains and the K-S track tables).")
else:
    R25 = []
    G25 = [(lab, np.asarray(m, bool), col) for _, m, col, _, lab in GRP22]
    NAM25 = [g[0] for g in G25]; COL25 = {g[0]: g[2] for g in G25}
    _key25 = list(ANC22["key"]); _tsft25 = ANC22["t_sft_gyr"].to_numpy(float)
    _grp_of25 = {}
    for lab, m, _ in G25:
        for k, mm in zip(_key25, m):
            if mm:
                _grp_of25[k] = lab
    _lg = lambda v: np.log10(np.where(np.isfinite(v) & (v > 0), v, np.nan))
    _h25 = [Line2D([], [], color=COL25[l], lw=2.0, marker="o", ms=6, mec="k", label=l) for l in NAM25] \
         + [Line2D([], [], ls="", marker=MRK25[s], ms=7, mfc="0.6", mec="k", label=LAB25[s]) for s in P25_STG]

    # ── (A) the data: the histories cache, sorted in time like _chains22 ──
    _HA = {}
    with h5py.File(HIST_FILE.format(box=P25_BOX), "r") as f:
        for k in f.keys():
            g, a = f[k], int(k[4:])
            t = g["t_yr"][:] / 1e9; o = np.argsort(t)
            gids = np.asarray(g["gal_ids"][:], np.int64)
            arr = {c: (g[c][:][o] if c in g else None) for c in ("mdust", "mh2", "mgas", "mstar", "sfr")}
            for j, gid in enumerate(gids):
                key = (a, int(gid))
                if key in _CH22:
                    _HA[key] = {c: (arr[c][:, j] if arr[c] is not None else np.full(len(t), np.nan)) for c in arr}
    _have_md = any(np.isfinite(v["mdust"]).any() for v in _HA.values())
    TRK = {c: np.full((len(ANC22), len(P25_TG)), np.nan) for c in ("mdust", "mh2", "dh2", "xray")}
    REG = pd.DataFrame(index=range(len(ANC22)), columns=["d_h2", "d_dust", "t_min_h2", "t_min_dust", "h2_anchor", "dust_anchor"], dtype=float)
    for i, (key, ts) in enumerate(zip(_key25, _tsft25)):
        if key not in _HA or not np.isfinite(ts):
            continue
        d, h = _CH22[key], _HA[key]
        x, ex, xr = d["t"] - ts, d["ex"], d["xr"]
        with np.errstate(all="ignore"):
            ld, lh = _lg(h["mdust"]), _lg(h["mh2"])
        for c, v in (("mdust", ld), ("mh2", lh), ("dh2", ld - lh)):
            ok = ex & np.isfinite(v)
            if ok.sum() >= 3:
                TRK[c][i] = np.interp(P25_TG, x[ok], v[ok], left=np.nan, right=np.nan)
        if ex.sum() >= 3:
            TRK["xray"][i] = np.interp(P25_TG, x[ex], xr[ex].astype(float), left=np.nan, right=np.nan)
        post = ex & (x >= 0)
        ka = int(np.flatnonzero(ex)[-1])
        for c, v in (("h2", lh), ("dust", ld)):
            okp = post & np.isfinite(v)
            if okp.sum() >= 2 and np.isfinite(v[ka]):
                jm = int(np.flatnonzero(okp)[np.argmin(v[okp])])
                REG.loc[i, f"d_{c}"] = float(v[ka] - v[jm]); REG.loc[i, f"t_min_{c}"] = float(x[jm]); REG.loc[i, f"{c}_anchor"] = float(v[ka])

    # ── the stage rows of the four groups, keyed for (B), (C) and (D) ──
    _ST25 = EPO22[(EPO22["stage"].isin(P25_STG)) & (EPO22["key"].isin(set(_key25)))].copy()
    _ST25["grp"] = [_grp_of25.get(k) for k in _ST25["key"]]
    _ST25 = _ST25[_ST25["grp"].notna()].reset_index(drop=True)
    _ST25["dt"] = _ST25["t_stage_gyr"].to_numpy(float) - _ST25["t_sft_gyr"].to_numpy(float)

    # ── (C) the data: the 100 kpc cutouts, at the critical points (C25) and on the continuous clock (CT) ──
    _ZC = ["core", "ring", "outskirt", "cgm_in", "cgm_out"]
    HAVE_BUD, HAVE_ESC, C25, CT = False, False, None, pd.DataFrame()
    if os.path.exists(P25_BUDGET):
        _BT = Table.read(P25_BUDGET)
        _BD = pd.DataFrame({c: (np.char.strip(np.asarray(_BT[c]).astype(str)) if np.asarray(_BT[c]).dtype.kind in "SU" else np.asarray(_BT[c]))
                            for c in _BT.colnames}).set_index(["zone", "snap", "gx"])
        HAVE_BUD = True
        HAVE_ESC = ("m_dust_esc" in _BD.columns) and bool(np.isfinite(_BD["m_dust_esc"].to_numpy(float)).any())

        def _budget_cols(df):
            """rows with snap, gx -> the per-row budget quantities from the 5 shells (NaN where a shell row or a column is missing)."""
            ix = list(zip(df["snap"].astype(int), df["gx"].astype(int)))
            g = lambda z, c: (np.array([_BD[c].get((z, s, x), np.nan) if (z, s, x) in _BD.index else np.nan for s, x in ix], float)
                              if c in _BD.columns else np.full(len(ix), np.nan))
            md = {z: g(z, "m_dust") for z in _ZC}; mh = {z: g(z, "m_H2") for z in _ZC}
            ism_d = md["core"] + md["ring"] + md["outskirt"]; cgm_d = md["cgm_in"] + md["cgm_out"]
            out = {}
            with np.errstate(all="ignore"):
                _r = lambda num, den: np.where(den > 0, num / np.where(den > 0, den, np.nan), np.nan)
                out["ldust_ism"] = _lg(ism_d); out["ldust_cgm"] = _lg(cgm_d)
                out["f_cgm_dust"] = _r(cgm_d, ism_d + cgm_d)
                out["vr_dust_outskirt"] = g("outskirt", "vr_dust"); out["vr_dust_cgm_in"] = g("cgm_in", "vr_dust")
                out["f_hot_cgm_dust"] = _r(g("cgm_in", "m_dust_hot") + g("cgm_out", "m_dust_hot"), cgm_d)
                out["lh2_core_all"] = _lg(mh["core"]); out["n_gas_core"] = g("core", "n_gas")
                out["lh2_ring_all"] = _lg(mh["ring"]); out["vr_H2_ring"] = g("ring", "vr_H2")
                out["f_member_ism_dust"] = _r(g("core", "m_dust_member") + g("ring", "m_dust_member") + g("outskirt", "m_dust_member"), ism_d)
                for zn, zs in (("ring", ["ring"]), ("outskirt", ["outskirt"]), ("cgm", ["cgm_in", "cgm_out"])):   # (D): where each zone's material goes
                    for q, tag in (("m_dust", "dust"), ("m_H2", "H2")):
                        tot = sum(g(z, q) for z in zs)
                        out[f"fout_{tag}_{zn}"] = _r(sum(g(z, q + "_out") for z in zs), tot)
                        out[f"fin_{tag}_{zn}"] = _r(sum(g(z, q + "_in") for z in zs), tot)
                        out[f"fesc_{tag}_{zn}"] = _r(sum(g(z, q + "_esc") for z in zs), tot)
                out["f_out_cgm_dust"] = out["fout_dust_cgm"]; out["f_esc_cgm_dust"] = out["fesc_dust_cgm"]
            return pd.DataFrame(out, index=df.index)

        C25 = pd.concat([_ST25[["key", "grp", "stage", "dt", "snap", "gx"]], _budget_cols(_ST25)], axis=1)
        if os.path.exists(P25_MAP):
            _MP = pd.read_csv(P25_MAP)
            _MP["key"] = list(zip(_MP["anchor_snap"].astype(int), _MP["gal_id"].astype(int)))
            _MP = _MP[_MP["key"].isin(set(_key25))].copy()
            _MP["grp"] = [_grp_of25.get(k) for k in _MP["key"]]
            _MP = _MP[_MP["grp"].notna()].reset_index(drop=True)
            _MP["dt"] = _MP["dt_sft_gyr"].to_numpy(float)
            CT = pd.concat([_MP[["key", "grp", "dt", "snap", "gx"]], _budget_cols(_MP)], axis=1)
            CT = CT[np.isfinite(CT["ldust_ism"]) | np.isfinite(CT["ldust_cgm"])].reset_index(drop=True)
    _cent = 0.5 * (P25_TB[1:] + P25_TB[:-1])

    def _track_panel(ax, c, ls="-", band=True, block="C_track"):
        """The binned group tracks of one quantity on the continuous clock (one value per galaxy per bin); rows -> R25."""
        for lab in NAM25:
            xs, ys, ls_, hs = [], [], [], []
            for b0, b1, xc in zip(P25_TB[:-1], P25_TB[1:], _cent):
                s = CT[(CT["grp"] == lab) & (CT["dt"] > b0) & (CT["dt"] <= b1)]
                per = s.groupby("key")[c].median().to_numpy(float)          # one value per galaxy per bin
                cen, hw, n = _pt22(per, P25_NMIN)
                if np.isfinite(cen):
                    xs.append(xc); ys.append(cen); ls_.append(cen - hw); hs.append(cen + hw)
                    R25.append(dict(block=block, quantity=c, group=lab, x=float(xc), n_gal=n, med=cen, lo=cen - hw, hi=cen + hw))
            if xs:
                if band:
                    ax.fill_between(xs, ls_, hs, color=COL25[lab], alpha=0.16, lw=0)
                ax.plot(xs, ys, ls, color=COL25[lab], lw=2.2 if ls == "-" else 1.5, marker="o", ms=4.5 if ls == "-" else 4,
                        mfc=COL25[lab] if ls == "-" else "white", mec="k" if ls == "-" else COL25[lab], mew=0.4 if ls == "-" else 1.0,
                        label=lab if ls == "-" else None)

    def _stage_tracks(df, cols, block):
        """df: one row per (galaxy, stage) with `grp`, `stage`, `dt` and the quantity columns -> per group x stage the median dt and
        the median of each column (bootstrap half-width, nan below P25_NMIN); rows appended to R25."""
        out = {}
        for lab in NAM25:
            for st in P25_STG:
                s = df[(df["grp"] == lab) & (df["stage"] == st)]
                if len(s) < P25_NMIN:
                    continue
                rec = {"dt": float(np.nanmedian(s["dt"]))}
                for c in cols:
                    v = s[c].to_numpy(float); cen, hw, n = _pt22(v, P25_NMIN)
                    rec[c] = (cen, hw, n)
                    R25.append(dict(block=block, quantity=c, group=lab, x=st, n_gal=n, med=cen, lo=cen - hw, hi=cen + hw, dt=rec["dt"]))
                out[(lab, st)] = rec
        return out

    def _draw_stage_panel(ax, TR, c, yl, tt, ls="-", hollow=False):
        """The group tracks of one stage quantity; ls / hollow draw a second quantity into the same panel (dashed, open markers)."""
        for lab in NAM25:
            pts = [(TR[(lab, st)]["dt"], TR[(lab, st)][c][0], TR[(lab, st)][c][1], st) for st in P25_STG
                   if (lab, st) in TR and c in TR[(lab, st)] and np.isfinite(TR[(lab, st)][c][0])]
            if not pts:
                continue
            pts.sort(key=lambda p: p[0])
            xs, ys, es, sts = zip(*pts)
            ax.plot(xs, ys, ls, color=COL25[lab], lw=2.0 if ls == "-" else 1.5, alpha=0.9 if ls == "-" else 0.8, zorder=3)
            for x_, y_, e_, st in pts:
                ax.errorbar(x_, y_, yerr=e_, marker=MRK25.get(st, "o"), ms=8 if not hollow else 7, color=COL25[lab],
                            mfc="white" if hollow else COL25[lab], mec=COL25[lab] if hollow else "k", mew=1.2 if hollow else 0.6,
                            capsize=2.5, lw=1.2, zorder=4)
        ax.axvline(0, color="k", lw=1.0, ls=":"); ax.set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]"); ax.set_ylabel(yl); ax.set_title(tt, fontsize=10)
        ax.grid(alpha=0.22, lw=0.5)

    _two = lambda a, b: [Line2D([], [], color="0.3", lw=2.0, marker="o", ms=5, mec="k", label=a),
                         Line2D([], [], color="0.3", lw=1.5, ls="--", marker="o", ms=5, mfc="white", mec="0.3", mew=1.0, label=b)]

    # ── FIGURE A: the branch ──
    fig, axs = plt.subplots(2, 3, figsize=(16.5, 9.0), constrained_layout=True)
    for ax, (c, yl, tt) in zip(axs.flat[:3], (("mdust", r"$\log\,M_{\rm dust}$  [$M_\odot$]", "(a) the dust of the whole galaxy"),
                                              ("mh2", r"$\log\,M_{\rm H_2}$  [$M_\odot$]", "(b) the molecular gas of the whole galaxy"),
                                              ("dh2", r"$\log\,M_{\rm dust}/M_{\rm H_2}$", "(c) their ratio"))):
        for lab, m, col in G25:
            cen, lo, hi, n = _band22(TRK[c][m], P25_NMIN)
            ax.fill_between(P25_TG, lo, hi, color=col, alpha=0.16, lw=0)
            ax.plot(P25_TG, cen, "-", color=col, lw=2.4, label=f"{lab}  ({int(m.sum())})")
            for xg, yc, yl_, yh, nn in zip(P25_TG, cen, lo, hi, n):
                R25.append(dict(block="A_track", quantity=c, group=lab, x=float(xg), n_gal=int(nn), med=yc, lo=yl_, hi=yh))
        ax.axvline(0, color="k", lw=1.2, ls=":"); ax.set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]"); ax.set_ylabel(yl); ax.set_title(tt, fontsize=10.5)
        ax.grid(alpha=0.22, lw=0.5)
    axs[0, 0].legend(frameon=False, fontsize=8)
    if not _have_md:
        axs[0, 0].text(0.5, 0.5, "no mdust in the histories cache\n(rebuild paper_ism_prediction_boxes Part 6a)", transform=axs[0, 0].transAxes,
                       ha="center", va="center", fontsize=9, color="0.4")
    ax = axs[1, 0]                                                # (d) the X-ray occupancy since SFT
    for lab, m, col in G25:
        occ = np.nanmean(TRK["xray"][m], axis=0); nn = np.isfinite(TRK["xray"][m]).sum(axis=0)
        occ = np.where(nn >= P25_NMIN, occ, np.nan)
        ax.plot(P25_TG, occ, "-", color=col, lw=2.4)
        for xg, yc, n_ in zip(P25_TG, occ, nn):
            R25.append(dict(block="A_xray_occupancy", quantity="f_xray", group=lab, x=float(xg), n_gal=int(n_), med=yc, lo=np.nan, hi=np.nan))
    ax.axvline(0, color="k", lw=1.2, ls=":"); ax.set_ylim(0, 1.02); ax.set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]")
    ax.set_ylabel("fraction in the X-ray state"); ax.set_title("(d) the X-ray occupancy along the branch", fontsize=10.5); ax.grid(alpha=0.22, lw=0.5)
    ax = axs[1, 1]                                                # (e) t_quiet at the anchor
    _tq25 = ANC22["t_quiet"].to_numpy(float)
    for lab, m, col in G25:
        v = _tq25[m]; fin = np.isfinite(v)
        if fin.sum() >= 3:
            xs = np.sort(v[fin]); ax.step(xs, np.arange(1, len(xs) + 1) / len(xs), where="post", color=col, lw=2.2,
                                          label=f"{lab}: never exposed {int((~fin).sum())}/{int(m.sum())}")
            R25.append(dict(block="A_tquiet", quantity="t_quiet_median", group=lab, x=np.nan, n_gal=int(fin.sum()), med=float(np.median(xs)),
                            lo=float(np.percentile(xs, 16)), hi=float(np.percentile(xs, 84)), f_never=float((~fin).mean())))
    ax.set_xlabel(r"$t_{\rm quiet}$ = time since the last X-ray episode at the anchor  [Gyr]"); ax.set_ylabel("cumulative fraction")
    ax.set_title("(e) time since the last exposure", fontsize=10.5); ax.legend(frameon=False, fontsize=7.8, loc="lower right"); ax.grid(alpha=0.22, lw=0.5)
    ax = axs[1, 2]                                                # (f) where the dust is, on the continuous clock (fallback: the regrowth)
    if len(CT) and np.isfinite(CT["f_cgm_dust"]).any():
        _track_panel(ax, "f_cgm_dust", block="A_cgm_share")
        ax.axvline(0, color="k", lw=1.2, ls=":"); ax.set_ylim(0, 1.02); ax.set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]")
        ax.set_ylabel("CGM share of the 100 kpc dust"); ax.set_title("(f) where the dust is: the CGM share, every post-SFT cutout", fontsize=10.5)
        ax.legend(frameon=False, fontsize=7.8, loc="lower right")
    else:
        for lab, m, col in G25:
            r = REG[m]
            ax.scatter(r["d_h2"], r["d_dust"], s=22, color=col, alpha=0.75, lw=0.4, edgecolors="k", label=lab)
        ax.axvline(P25_REGROW, color="0.4", lw=0.9, ls="--"); ax.axhline(P25_REGROW, color="0.4", lw=0.9, ls="--")
        ax.set_xlabel(r"$\Delta \log M_{\rm H_2}$ = anchor $-$ post-SFT minimum"); ax.set_ylabel(r"$\Delta \log M_{\rm dust}$ = anchor $-$ post-SFT minimum")
        ax.set_title("(f) did the H$_2$ and the dust come back?  (no budget products yet)", fontsize=10.5); ax.legend(frameon=False, fontsize=7.8)
    ax.grid(alpha=0.22, lw=0.5)
    print("Part 25 (A) — the regrowth since the post-SFT minimum (anchor minus minimum, dex; 'came back' = above "
          f"{P25_REGROW:g} dex), per group: H2 | dust; and the median time of the H2 minimum after SFT [Gyr]")
    for lab, m, col in G25:
        r = REG[m]; ok = np.isfinite(r["d_h2"]) & np.isfinite(r["d_dust"])
        if ok.sum():
            print(f"    {lab:34s} N={int(ok.sum()):3d}  H2 back {100 * np.mean(r['d_h2'][ok] > P25_REGROW):3.0f} % (median {np.nanmedian(r['d_h2'][ok]):+.2f})"
                  f"  dust back {100 * np.mean(r['d_dust'][ok] > P25_REGROW):3.0f} % (median {np.nanmedian(r['d_dust'][ok]):+.2f})"
                  f"  t_min(H2) {np.nanmedian(r['t_min_h2'][ok]):.2f}")
            R25.append(dict(block="A_regrowth", quantity="f_back", group=lab, x=np.nan, n_gal=int(ok.sum()),
                            med=float(np.mean(r["d_h2"][ok] > P25_REGROW)), lo=float(np.nanmedian(r["d_h2"][ok])),
                            hi=float(np.mean(r["d_dust"][ok] > P25_REGROW)), d_dust_median=float(np.nanmedian(r["d_dust"][ok])),
                            t_min_h2=float(np.nanmedian(r["t_min_h2"][ok]))))
    fig.suptitle(f"Part 25 — {BOX_LABEL.get(P25_BOX, P25_BOX)}: the whole-galaxy dust and H$_2$ along the branch since SFT, the X-ray "
                 "occupancy, the time since the last exposure, and where the dust is", fontsize=12)
    paper_save(fig, f"agn_states_budget_branch_{P25_BOX}")
    plt.show()

    # ── FIGURE B: the critical points, member particles: core / ring / outskirt ──
    _M25 = MEA22.set_index(["ap_label", "snap", "gx"])
    _ix = list(zip(_ST25["snap"].astype(int), _ST25["gx"].astype(int)))
    def _ap(col, ap):
        return np.array([_M25[col].get((ap, s, g), np.nan) if (ap, s, g) in _M25.index else np.nan for s, g in _ix], float)
    B25 = _ST25[["key", "grp", "stage", "dt"]].copy()
    with np.errstate(all="ignore"):
        for q in ("m_dust", "m_H2"):
            tot10 = _ap(q, "ap10kpc")
            for z, ap_out, ap_in, _ in P25_ZONES:
                v = _ap(q, ap_out) - (_ap(q, ap_in) if ap_in else 0.0)
                B25[f"l{q}_{z}"] = _lg(np.clip(v, 0, None))
            B25[f"fcore_{q}"] = np.where(tot10 > 0, _ap(q, "ap1kpc") / np.where(tot10 > 0, tot10, np.nan), np.nan)
    _colsB = [f"l{q}_{z}" for q in ("m_dust", "m_H2") for z, _, _, _ in P25_ZONES] + ["fcore_m_dust", "fcore_m_H2"]
    TRB = _stage_tracks(B25, _colsB, "B_stage")
    fig, axs = plt.subplots(2, 4, figsize=(19, 8.6), constrained_layout=True)
    for r_, (q, qn) in enumerate((("m_dust", r"M_{\rm dust}"), ("m_H2", r"M_{\rm H_2}"))):
        for c_, (z, _, _, zl) in enumerate(P25_ZONES):
            _draw_stage_panel(axs[r_, c_], TRB, f"l{q}_{z}", rf"$\log\,{qn}$  [$M_\odot$]", f"({chr(97 + 4 * r_ + c_)}) {zl}, members")
        _draw_stage_panel(axs[r_, 3], TRB, f"fcore_{q}", rf"${qn}$(core) / ${qn}$($<10$ kpc)", f"({chr(100 + 4 * r_)}) the core's share")
    axs[0, 0].legend(handles=_h25, frameon=False, fontsize=7.6, ncol=2)
    fig.suptitle(f"Part 25 — {BOX_LABEL.get(P25_BOX, P25_BOX)}: the dust and the H$_2$ of the member particles in the core, the ring and "
                 "the outskirt at the critical points, on each group's own $t - t_{\\rm SFT}$", fontsize=12)
    paper_save(fig, f"agn_states_budget_zones_{P25_BOX}")
    plt.show()

    # ── FIGURES C and D: the 100 kpc cutouts ──
    if not HAVE_BUD:
        print(f"\n(C) {P25_BUDGET} missing — ks_tracks_quenched_m25.ipynb Part 3c (cluster) builds it from the existing cutouts")
    else:
        if not HAVE_ESC:
            print(f"\n(C) {os.path.basename(P25_BUDGET)} has no escape columns yet (m_dust_esc): re-run ks_tracks_quenched_m25 Part 3c — "
                  "until then the 'outflowing' panels use v_r > 0 only")
        _eshare = ("f_esc_cgm_dust", "CGM dust above the escape speed", "(e) the escaping share  (dashed: any $v_r > 0$)") if HAVE_ESC else \
                  ("f_out_cgm_dust", "CGM dust on outflowing particles", "(e) the outflowing share ($v_r > 0$; no escape columns yet)")
        _PC = [("ldust_ism", r"$\log\,M_{\rm dust}(<10\,{\rm kpc})$  [$M_\odot$]", "(a) the dust in the ISM (every particle)"),
               ("ldust_cgm", r"$\log\,M_{\rm dust}(10$-$100\,{\rm kpc})$  [$M_\odot$]", "(b) the dust in the CGM"),
               ("f_cgm_dust", "CGM share of the 100 kpc dust", "(c) where the dust is"),
               ("vr_dust_cgm_in", r"$v_r$ of the dust  [km/s]  ($+$ = outflow)", "(d) leaving or falling back?  solid: CGM 10-30 kpc, dashed: outskirt 3.2-10 kpc"),
               _eshare,
               ("lh2_core_all", r"$\log\,M_{\rm H_2}(<1\,{\rm kpc})$, every particle", "(f) the core H$_2$: did it arrive after SFT?")]
        _colsC = [c for c, _, _ in _PC] + ["vr_dust_outskirt", "f_out_cgm_dust", "f_hot_cgm_dust", "n_gas_core", "f_member_ism_dust", "lh2_ring_all", "vr_H2_ring"] \
               + [f"{p}_{t}_{z}" for p in ("fout", "fin", "fesc") for t in ("dust", "H2") for z in ("ring", "outskirt", "cgm")]
        TRC = _stage_tracks(C25, list(dict.fromkeys(_colsC)), "C_stage")
        fig, axs = plt.subplots(2, 3, figsize=(16.5, 9.0), constrained_layout=True)
        for ax, (c, yl, tt) in zip(axs.flat, _PC):
            _draw_stage_panel(ax, TRC, c, yl, tt)
            if c == "vr_dust_cgm_in":                                  # the outskirt shell in the same panel: the comparison
                _draw_stage_panel(ax, TRC, "vr_dust_outskirt", yl, tt, ls="--", hollow=True)
                ax.legend(handles=_two("CGM 10-30 kpc", "outskirt 3.2-10 kpc"), frameon=False, fontsize=8, loc="upper right")
            if c == "f_esc_cgm_dust":                                  # the loose v_r > 0 share behind the escape one
                _draw_stage_panel(ax, TRC, "f_out_cgm_dust", yl, tt, ls="--", hollow=True)
                ax.legend(handles=_two(r"$v_r > v_{\rm esc}$", r"$v_r > 0$"), frameon=False, fontsize=8, loc="upper right")
            if c.startswith("vr"):
                ax.axhline(0, color="0.4", lw=0.9, ls="--")
        axs[0, 0].legend(handles=_h25, frameon=False, fontsize=7.6, ncol=2)
        fig.suptitle(f"Part 25 — {BOX_LABEL.get(P25_BOX, P25_BOX)}: the dust budget of the whole 100 kpc cutout at the critical points — "
                     "ISM against CGM, the radial flows, the core H$_2$ (every particle)", fontsize=12)
        paper_save(fig, f"agn_states_budget_cgm_{P25_BOX}")
        plt.show()
        print("\n(C) at the anchor, per group (medians): dust ISM | CGM | CGM share | v_r(10-30 kpc) | outflowing share | escaping share | hot share | "
              "core H2 (all) | core gas particles | member share of the ISM dust")
        for lab in NAM25:
            r = TRC.get((lab, "anchor"))
            if r:
                v = lambda c: r[c][0] if c in r else np.nan
                print(f"    {lab:34s} {v('ldust_ism'):5.2f} | {v('ldust_cgm'):5.2f} | {v('f_cgm_dust'):.2f} | {v('vr_dust_cgm_in'):+5.0f} | "
                      f"{v('f_out_cgm_dust'):.2f} | {v('f_esc_cgm_dust'):.2f} | {v('f_hot_cgm_dust'):.2f} | {v('lh2_core_all'):5.2f} | {v('n_gas_core'):4.0f} | {v('f_member_ism_dust'):.2f}")
        if not len(CT):
            print(f"  the continuous clock needs the post-SFT cutouts: {P25_MAP} missing or empty (ks_tracks_quenched_m25 Part 2b + its sbatch, then Part 3c)")
        else:
            print(f"  the continuous clock: {len(CT)} measured branch snapshots of {CT['key'].nunique()} galaxies")
            fig, axs = plt.subplots(2, 3, figsize=(16.5, 9.0), constrained_layout=True)
            for ax, (c, yl, tt) in zip(axs.flat, _PC):
                _track_panel(ax, c)
                if c == "vr_dust_cgm_in":
                    _track_panel(ax, "vr_dust_outskirt", ls="--", band=False)
                    ax.legend(handles=_two("CGM 10-30 kpc", "outskirt 3.2-10 kpc"), frameon=False, fontsize=8, loc="upper right")
                if c == "f_esc_cgm_dust":
                    _track_panel(ax, "f_out_cgm_dust", ls="--", band=False)
                    ax.legend(handles=_two(r"$v_r > v_{\rm esc}$", r"$v_r > 0$"), frameon=False, fontsize=8, loc="upper right")
                if c.startswith("vr"):
                    ax.axhline(0, color="0.4", lw=0.9, ls="--")
                ax.axvline(0, color="k", lw=1.0, ls=":"); ax.set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]"); ax.set_ylabel(yl); ax.set_title(tt, fontsize=10)
                ax.grid(alpha=0.22, lw=0.5)
            axs[0, 0].legend(frameon=False, fontsize=7.8)
            fig.suptitle(f"Part 25 — {BOX_LABEL.get(P25_BOX, P25_BOX)}: the same budget on the continuous clock, every post-SFT snapshot "
                         f"with a cutout (bins of {P25_TB[1] - P25_TB[0]:g} Gyr, one value per galaxy per bin)", fontsize=12)
            paper_save(fig, f"agn_states_budget_cgm_track_{P25_BOX}")
            plt.show()

        # ── FIGURE D: where the material of each zone is going — outward (v_r > 0) solid, escaping (v_r > v_esc) dashed, inward = the rest ──
        _ZD = [("ring", "ring 1-3.2 kpc"), ("outskirt", "outskirt 3.2-10 kpc"), ("cgm", "CGM 10-100 kpc")]
        fig, axs = plt.subplots(2, 3, figsize=(16.5, 9.0), constrained_layout=True)
        for r_, (tag, tn) in enumerate((("dust", "dust"), ("H2", "H$_2$"))):
            for c_, (zn, zl) in enumerate(_ZD):
                ax = axs[r_, c_]
                yl = f"share of the zone's {tn} mass"; tt = f"({chr(97 + 3 * r_ + c_)}) {zl}: {tn} moving outward (solid)" + (", escaping (dashed)" if HAVE_ESC else "")
                if len(CT):
                    _track_panel(ax, f"fout_{tag}_{zn}", block="D_track")
                    if HAVE_ESC:
                        _track_panel(ax, f"fesc_{tag}_{zn}", ls="--", band=False, block="D_track")
                    ax.axvline(0, color="k", lw=1.0, ls=":"); ax.set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]"); ax.set_ylabel(yl); ax.set_title(tt, fontsize=10)
                    ax.grid(alpha=0.22, lw=0.5)
                else:
                    _draw_stage_panel(ax, TRC, f"fout_{tag}_{zn}", yl, tt)
                    if HAVE_ESC:
                        _draw_stage_panel(ax, TRC, f"fesc_{tag}_{zn}", yl, tt, ls="--", hollow=True)
                ax.axhline(0.5, color="0.4", lw=0.9, ls="--"); ax.set_ylim(0, 1.02)
        axs[0, 0].legend(handles=[Line2D([], [], color=COL25[l], lw=2.2, marker="o", ms=5, mec="k", label=l) for l in NAM25]
                         + (_two(r"$v_r > 0$ (outward)", r"$v_r > v_{\rm esc}$ (escaping)") if HAVE_ESC else []), frameon=False, fontsize=7.6, ncol=2)
        axs[0, 0].text(0.02, 0.03, "inward share = 1 $-$ outward (particles at $v_r = 0$ aside); the 0.5 line = as much out as in",
                       transform=axs[0, 0].transAxes, fontsize=7.5, color="0.35")
        fig.suptitle(f"Part 25 — {BOX_LABEL.get(P25_BOX, P25_BOX)}: where the dust and the H$_2$ of each zone are going — towards the "
                     "core or towards the CGM, and how much leaves the halo for good" + ("" if len(CT) else " (critical points)"), fontsize=12)
        paper_save(fig, f"agn_states_budget_flows_{P25_BOX}")
        plt.show()
        print("\n(D) at the anchor, per group (medians): the OUTSKIRT 3.2-10 kpc dust moving outward | inward | escaping;  the same for its H2;  "
              "the CGM dust escaping")
        for lab in NAM25:
            r = TRC.get((lab, "anchor"))
            if r:
                v = lambda c: r[c][0] if c in r else np.nan
                print(f"    {lab:34s} dust {v('fout_dust_outskirt'):.2f} | {v('fin_dust_outskirt'):.2f} | {v('fesc_dust_outskirt'):.2f};  "
                      f"H2 {v('fout_H2_outskirt'):.2f} | {v('fin_H2_outskirt'):.2f} | {v('fesc_H2_outskirt'):.2f};  CGM dust escaping {v('fesc_dust_cgm'):.2f}")

    if R25:
        _p25 = os.path.join(OUT, f"agn_states_budget_{P25_BOX}.csv")
        pd.DataFrame(R25).to_csv(_p25, index=False)
        print(f"\n  the tables behind the figures -> {_p25}")


## Part 26 — the dose × extent split of the 100 Mpc box, against the ALMA-C11 sample

Part 25 showed the ALMA-C11 quiescent galaxies map onto a handful of massive, extended, dusty galaxies in m25 — a tail, because the 25 Mpc box holds 18 quenched anchors above $\log M_\star = 10.9$. The 100 Mpc box has the mass. It has no K-S track products, but the **catalogue radii** give the same geometry axis: on m25 the catalogue $\log R_{50}({\rm gas})/R_{50}(\star)$ (all gas, caesar `radii.gas_half_mass` / `radii.stellar_half_mass`) tracks the particle-level H$_2$ extent with Spearman $+0.95$ and puts 92 % of the galaxies on the same side of the median split. This part reads those two radii and the dust / H$_2$ / stellar masses **straight from the caesar catalogues** with h5py for every quenched galaxy of Part 22's Fig 1 frame (`FIG1_22`, the same dose, the same de-duplication), caches them (`agn_states_catalogue_radii_<box>.csv`), splits the box at its own medians of the X-ray duty fraction since SFT and of the extent, and draws the four classes on the observable planes **as population tracks**: in each panel a class is its running median along the abscissa (a sliding window over the `P26_KFRAC` $= 30$ % of the class nearest in $x$, never fewer than `P26_NMIN` galaxies) with the 16–84 range of that window as the dispersion band, computed in the **observed regime** (anchors at $z \leq$ `P26_ZMAX` $= 0.5$ and $\log M_\star \geq$ `P26_LOGM` $= 10.8$; `P26_TRACK_SEL = "zmax"` uses every mass). The planes are age → $M_{\rm dust}/M_\star$, age → $M_{\rm H_2}/M_\star$ (the observed $\alpha_{\rm CO}$ H$_2$ divided by 1.36, SIMBA's catalogue H$_2$ being hydrogen-only), age → extent ($R_{\rm CO}/R_e$ for the data) and $M_{\rm H_2}/M_\star$ → $M_{\rm dust}/M_\star$. The **observed box** (`P26_OBOX`, the sample's 16–84 ranges widened) is the dashed rectangle; the regime galaxies that fall inside it on all four axes are drawn **one by one, numbered**, listed in the legend of panel (b) and in `agn_states_boxes_vs_alma_inbox.csv`; the ten CO-detected ALMA-C11 sources carry their **error bars** — the CIGALE age and $M_{\rm dust}$ errors, the CO-flux $M_{\rm H_2}$ error, the uv-fit size interval combined with the $R_e$ error for the extent — with arrows where the value is an upper limit (dust undetected, CO unresolved: $R_{\rm CO} = b_{\rm maj}/2$). The cell still prints, per class and per selection, the medians and the in-box counts, and per source the counterparts within `P26_TOL`.

On the two age panels each class also gets its **history track** (dashed): the same galaxies followed back along their main branch in Part 22's histories cache (`HIST_FILE`, the chains that set the dose), the median of $M_{\rm dust}/M_\star$ or $M_{\rm H_2}/M_\star$ against the mass-weighted age **along the branch** (`P26_HIST`, grid `P26_HIST_AGE`, nodes with at least `P26_HIST_NMIN` chains; `P26_HIST_BAND` adds the 16–84 range across the chains), with an open circle at the class's median age at SFT. The solid track is where the quenched galaxies of a class sit at their anchors; the dashed one is where they came from — a class whose dashed track passes through the observed box at some age contained ALMA-like galaxies at that stage of its life.

**What the cluster run (2026-09-04) found in m100.** 1625 quenched galaxies, 309 in the observed regime. The mass gap closes and the H$_2$ content is reproduced — $\log M_{\rm H_2}/M_\star$ $-1.55$ to $-1.72$ per class against $-1.45$ observed — but a **dust gap** opens: the class medians sit at $\log M_{\rm dust}/M_\star$ $-4.7$ to $-6.3$ against $-3.0$ observed (the bands are wide because the dust distribution is bimodal: a dust-poor floor and a dusty tail), and only **8 of the 309** (2.6 %) fall in the observed box: 5 low dose + extended (11 % of that class), 2 low dose + compact, 0 high dose + compact, 1 high dose + extended. The observed sample is the dusty upper edge of the massive quenched population, and that edge is populated by the **low-dose** galaxies, the extended ones first. The catalogue extent is compressed relative to the H$_2$ one (all gas: $-0.30$ to $-0.15$ per class), so the split cut is $-0.20$ and the observed $R_{\rm CO}/R_e$ scale is not the same scale — the extent panel compares ranks, not values.

**Knobs.** `P26_BOXES` (`["cis100"]`; add `"cis50"` for the 50 Mpc box, one figure each), `P26_ZMAX`, `P26_LOGM`, `P26_OBOX`, `P26_TOL`, `P26_TRACK_SEL`, `P26_KFRAC`, `P26_NMIN`, `P26_NGRID`, `P26_POINTS` (the regime galaxies as faint points behind the bands), `P26_ULIM` (the arrow length), `P26_HIST` / `P26_HIST_AGE` / `P26_HIST_NMIN` / `P26_HIST_BAND` (the history tracks), `P26_CATDIR` (per-box catalogue directory overrides; default from `~/.simbanator/config.json`), `P26_OBS` (the observed tables).


In [ ]:
# ── Part 26 — the dose x extent split of the 100 Mpc box with the CATALOGUE radii, against the ALMA-C11 sample ──
# Part 22's Fig 1 frame (FIG1_22[box]["A"]: one row per quenched galaxy with the dose, the age, log M*, z) gets the catalogue's
# gas / stellar half-mass radii and dust / H2 / stellar masses read straight from the caesar files (h5py, two datasets each), is
# split at the box's own medians of f_x_post and of log R50(gas)/R50(*), and each of the four classes is drawn as its POPULATION
# TRACK along the observable axes: the running median of the class (a sliding window over the P26_KFRAC nearest galaxies) with
# its 16-84 dispersion band, in the observed regime (z <= P26_ZMAX, log M* >= P26_LOGM). The galaxies that fall inside the
# observed box (P26_OBOX) are drawn and numbered one by one, and the ten CO-detected ALMA-C11 sources carry their error bars
# (age, log M_dust/M*, log M_H2/M*, log R_CO/R_e; arrows for the dust / size upper limits). On the age panels each class also gets
# its HISTORY track (dashed): the median main-branch chain of the quantity against the mass-weighted age ALONG the branch, from
# Part 22's histories cache (HIST_FILE), with a circle at the class's median age at SFT — where the class came from.
P26_BOXES = ["cis100", "cis25"]          # one figure per box; ["cis100", "cis50"] brings the 50 Mpc box back
P26_ZMAX  = 0.5                 # the observed sample sits at z ~ 0.4: anchors at or below this
P26_LOGM  = 10.8                # the observed mass range starts here (log M*): the observed regime = z <= P26_ZMAX and log M* >= this
P26_OBOX  = dict(age=(3.5, 6.5), lfd=(-3.5, -2.5), lfh2=(-1.8, -0.8), extent=(-0.4, 0.15))   # the observed box, 16-84 widened
P26_TOL   = dict(age=0.5, lfd=0.3, lfh2=0.3, extent=0.3)                                     # the per-source counterpart box
P26_TRACK_SEL = "regime"        # the galaxies the tracks are drawn from: "regime" (z <= P26_ZMAX and log M* >= P26_LOGM) | "zmax" (any mass)
P26_KFRAC = 0.30                # the sliding window: each track point is the median of this fraction of the class (nearest in x) ...
P26_NMIN  = 10                  # ... and never fewer than this many galaxies; a class below it draws no track
P26_NGRID = 40                  # track points between the class's 5th and 95th percentiles of x
P26_POINTS = False              # True: the regime galaxies as faint points behind the bands (the dust distribution is bimodal)
P26_ULIM  = dict(lfd=0.3, extent=0.2)   # [dex] the arrow length of the observed upper limits, per axis
P26_HIST  = ("lfd", "lfh2")     # the HISTORY tracks, dashed on the age panels: each class's median main-branch chain of the quantity vs
                                #   the mass-weighted age along the branch (HIST_FILE, the same chains Part 22 dosed); () to skip
P26_HIST_AGE  = np.arange(0.5, 9.01, 0.25)   # the age grid of the history tracks [Gyr]
P26_HIST_NMIN = 10              # a history node drawn where this many chains contribute
P26_HIST_BAND = False           # True: also the 16-84 range across the chains (dotted), on top of the population bands
P26_CATDIR = {}                 # {"cis100": "/path/to/Groups", ...} overrides; default = ~/.simbanator/config.json (catalog_dir + file_format)
P26_OBS   = (os.path.join(os.getcwd(), "obs_data", "almac11", "age_sersic_sigma.csv"), os.path.join(os.getcwd(), "obs_data", "almac11", "ks_table.csv"))
NAME26 = {"lo_hi": "low dose + compact gas", "lo_lo": "low dose + extended gas", "hi_hi": "high dose + compact gas", "hi_lo": "high dose + extended gas"}
COL26  = {"lo_hi": "#b2182b", "lo_lo": "#ef8a62", "hi_hi": "#67a9cf", "hi_lo": "#2166ac"}

if "FIG1_22" not in globals():
    print("Part 26: needs Part 22 — run it first (its Fig 1 frames carry the dose of the coarse boxes).")
elif not all(os.path.exists(p) for p in P26_OBS):
    print(f"Part 26: the observed tables are missing ({[p for p in P26_OBS if not os.path.exists(p)]})")
else:
    R26 = []
    # ── the observed sample on the four axes, with its errors ──
    _o26 = pd.read_csv(P26_OBS[0]).set_index("id"); _k26 = pd.read_csv(P26_OBS[1]).set_index("id")
    for _c in ("logMH2", "logMH2_err", "r_lo_kpc", "r_hi_kpc"):
        _o26[_c] = _k26[_c].reindex(_o26.index) if _c in _k26.columns else np.nan
    _o26["co_det"] = _k26["co_det"].reindex(_o26.index).astype(bool)
    with np.errstate(all="ignore"):
        _o26["lfh2"] = _o26["logMH2"] - np.log10(1.36) - _o26["logMstar"]          # alpha_CO includes He; the catalogue H2 does not
        _o26["lfh2_err"] = _o26["logMH2_err"]
        _o26["extent"] = np.log10(_o26["r_co_kpc"] / _o26["re_kpc"])
        _re_e = (_o26["re_err_kpc"].fillna(0) / (_o26["re_kpc"] * np.log(10))).to_numpy(float)   # the R_e error, in dex
        _lo_c = np.log10(_o26["r_co_kpc"] / _o26["r_lo_kpc"]).to_numpy(float)                   # the CO size interval, in dex
        _hi_c = np.log10(_o26["r_hi_kpc"] / _o26["r_co_kpc"]).to_numpy(float)
        _o26["extent_elo"] = np.sqrt(np.nan_to_num(_lo_c) ** 2 + _re_e ** 2)
        _o26["extent_ehi"] = np.sqrt(np.nan_to_num(_hi_c) ** 2 + _re_e ** 2)
    _o26["extent_ul"] = _o26["r_co_is_ul"].astype(bool) if "r_co_is_ul" in _o26.columns else False   # unresolved CO: R_CO = b_maj/2 is an upper limit
    _o26["lfd_ul"] = _o26["fdust_is_ul"].astype(bool) if "fdust_is_ul" in _o26.columns else False     # undetected dust: M_dust is an upper limit
    OBS26 = _o26[_o26["co_det"]].rename(columns={"age_gyr": "age", "age_err_gyr": "age_err", "log_fdust": "lfd", "log_fdust_err": "lfd_err",
                                                 "logMstar": "lm"})[["z", "age", "age_err", "lm", "lfd", "lfd_err", "lfd_ul", "lfh2", "lfh2_err",
                                                                     "extent", "extent_elo", "extent_ehi", "extent_ul"]].copy()
    _f26 = lambda v: (f"{np.nanmedian(v):+.2f} [{np.nanpercentile(v, 16):+.2f}, {np.nanpercentile(v, 84):+.2f}]" if np.isfinite(v).sum() >= 3 else "n/a")
    print(f"Part 26 — the ALMA-C11 CO-detected sources (N={len(OBS26)}): age {_f26(OBS26['age'])} Gyr, log M* {_f26(OBS26['lm'])}, "
          f"log M_dust/M* {_f26(OBS26['lfd'])}, log M_H2/M* {_f26(OBS26['lfh2'])} (He removed), log R_CO/R_e {_f26(OBS26['extent'])};  "
          f"errors: age {np.nanmedian(OBS26['age_err']):.2f} Gyr, dust {np.nanmedian(OBS26['lfd_err']):.2f} dex ({int(OBS26['lfd_ul'].sum())} upper limits), "
          f"H2 {np.nanmedian(OBS26['lfh2_err']):.2f} dex, extent {np.nanmedian(OBS26['extent_ehi']):.2f} dex ({int(OBS26['extent_ul'].sum())} unresolved)")

    def _catdir26(box):
        """The caesar directory and file pattern of a box: P26_CATDIR, else ~/.simbanator/config.json."""
        import json as _json26
        try:
            cfg = _json26.load(open(os.path.expanduser("~/.simbanator/config.json"))); ent = (cfg.get("simulations", cfg)).get(box, {})
        except Exception:
            ent = {}
        _stem = {"cis100": "m100n1024", "cis50": "m50n512", "cis50nox": "m50n512", "cis25": "m25n512"}.get(box, box)   # the SIMBA file stems, if no config
        return P26_CATDIR.get(box, ent.get("catalog_dir")), ent.get("file_format", "%s_{snap:03d}.hdf5" % _stem)

    def _radii26(box, A):
        """The catalogue radii and masses of A's (snap, gal_id) rows, cached per box."""
        cache = os.path.join(OUT, f"agn_states_catalogue_radii_{box}.csv")
        have = pd.read_csv(cache) if os.path.exists(cache) else pd.DataFrame(columns=["snap", "gal_id"])
        if len(have) and "age_mw" not in have.columns:                 # an older cache without the mass-weighted age: rebuild it
            print(f"  [{box}] the radii cache predates the age column — re-reading the catalogue rows")
            have = pd.DataFrame(columns=["snap", "gal_id"])
        known = set(zip(have["snap"].astype(int), have["gal_id"].astype(int)))
        todo = A[[(int(s), int(g)) not in known for s, g in zip(A["snap"], A["gal_id"])]]
        cdir, fmt = _catdir26(box)
        rows = []
        for snap in sorted(set(todo["snap"].astype(int))):
            fn = os.path.join(cdir or "", fmt.format(snap=int(snap)))
            ids = todo.loc[todo["snap"].astype(int) == snap, "gal_id"].astype(int).to_numpy()
            try:
                with h5py.File(fn, "r") as f:
                    a = float(f["simulation_attributes"].attrs["scale_factor"]); g = f["galaxy_data/dicts"]
                    rg, rs = g["radii.gas_half_mass"][:], g["radii.stellar_half_mass"][:]
                    md, mh, ms, mg = g["masses.dust"][:], g["masses.H2"][:], g["masses.stellar"][:], g["masses.gas"][:]
                    ag = g["ages.mass_weighted"][:] if "ages.mass_weighted" in g else np.full(len(ms), np.nan)   # [Gyr]
                for i in ids:
                    rows.append(dict(snap=int(snap), gal_id=int(i), r50gas_kpc=float(rg[i]) * a, r50star_kpc=float(rs[i]) * a,
                                     mdust=float(md[i]), mh2=float(mh[i]), mstar=float(ms[i]), mgas=float(mg[i]), age_mw=float(ag[i])))
            except Exception as e:
                print(f"  [{box}] snap {snap}: {type(e).__name__}: {e} ({fn}) — {len(ids)} galaxies without radii")
        if rows:
            have = pd.concat([have, pd.DataFrame(rows)], ignore_index=True); have.to_csv(cache, index=False)
            print(f"  [{box}] catalogue radii read for {len(rows)} galaxies -> {cache}")
        return have

    def _track26(x, y):
        """The population track of y along x: at P26_NGRID points between the 5th and 95th percentiles of x, the median and the
        16-84 range of y over the P26_KFRAC nearest galaxies in x (never fewer than P26_NMIN). None below P26_NMIN galaxies."""
        x, y = np.asarray(x, float), np.asarray(y, float); ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
        if len(x) < P26_NMIN:
            return None
        k = int(min(max(int(round(P26_KFRAC * len(x))), P26_NMIN), len(x)))
        xg = np.linspace(np.percentile(x, 5), np.percentile(x, 95), P26_NGRID)
        med, lo, hi = (np.full(P26_NGRID, np.nan) for _ in range(3))
        for i, xc in enumerate(xg):
            v = y[np.argsort(np.abs(x - xc), kind="stable")[:k]]
            med[i], lo[i], hi[i] = np.median(v), np.percentile(v, 16), np.percentile(v, 84)
        return dict(x=xg, med=med, lo=lo, hi=hi, k=k, n=len(x))

    def _hist26(box, keys):
        """The main-branch chains of the wanted (anchor_snap, gal_id) keys from the histories cache: the time, the mass-weighted age
        and log M_dust/M*, log M_H2/M* along the branch (the snapshots where the progenitor exists)."""
        fn, H, want = HIST_FILE.format(box=box), {}, set(keys)
        if not os.path.exists(fn):
            return H
        with h5py.File(fn, "r") as f:
            for k in f.keys():
                g, a = f[k], int(k[4:])
                gids = np.asarray(g["gal_ids"][:], dtype=np.int64)
                js = [j for j, gid in enumerate(gids) if (a, int(gid)) in want]
                if not js or "age" not in g or "mdust" not in g:
                    continue
                t = g["t_yr"][:] / 1e9; o = np.argsort(t); t = t[o]
                age, ms, md = g["age"][:][o], g["mstar"][:][o], g["mdust"][:][o]
                mh = g["mh2"][:][o] if "mh2" in g else np.full_like(ms, np.nan)
                with np.errstate(all="ignore"):
                    for j in js:
                        ok = np.isfinite(ms[:, j]) & (ms[:, j] > 0) & np.isfinite(age[:, j])
                        if ok.sum() < 3:
                            continue
                        H[(a, int(gids[j]))] = dict(t=t[ok], age=age[ok, j],
                                                    lfd=np.log10(np.where(md[ok, j] > 0, md[ok, j] / ms[ok, j], np.nan)),
                                                    lfh2=np.log10(np.where(mh[ok, j] > 0, mh[ok, j] / ms[ok, j], np.nan)))
        return H

    def _onage26(h, q):
        """One chain's quantity on the P26_HIST_AGE grid (the age made monotonic with a running maximum; NaN outside the chain)."""
        ok = np.isfinite(h[q]) & np.isfinite(h["age"])
        if ok.sum() < 3:
            return np.full(len(P26_HIST_AGE), np.nan)
        return np.interp(P26_HIST_AGE, np.maximum.accumulate(h["age"][ok]), h[q][ok], left=np.nan, right=np.nan)

    def _hband26(M):
        """(n_chain, n_age) -> the median, the 16-84 range across the chains and the count, NaN below P26_HIST_NMIN."""
        M = np.asarray(M, float); n = np.isfinite(M).sum(axis=0); ok = n >= P26_HIST_NMIN
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            med, lo, hi = np.nanmedian(M, axis=0), np.nanpercentile(M, 16, axis=0), np.nanpercentile(M, 84, axis=0)
        return dict(med=np.where(ok, med, np.nan), lo=np.where(ok, lo, np.nan), hi=np.where(ok, hi, np.nan), n=n)

    _inobs26 = lambda d: (d["age"].between(*P26_OBOX["age"]) & d["lfd"].between(*P26_OBOX["lfd"]) & d["lfh2"].between(*P26_OBOX["lfh2"])
                          & d["extent"].between(*P26_OBOX["extent"]))
    _ORDER26 = ("lo_hi", "lo_lo", "hi_hi", "hi_lo")
    FR26, INBOX26 = {}, {}
    for _b in P26_BOXES:
        if _b not in FIG1_22:
            print(f"\n[{_b}] not in Part 22's Fig 1 frames — skipped"); continue
        A = FIG1_22[_b]["A"].copy()
        A["snap"] = A["snap"].astype(int); A["gal_id"] = A["gal_id"].astype(int)
        A = A.merge(_radii26(_b, A), on=["snap", "gal_id"], how="left")
        if "age" not in A.columns:                                      # the m25 frame (Part 22's KS-track anchors) carries no age column:
            A["age"] = A["age_mw"]                                      #   the catalogue's mass-weighted age, the same quantity the coarse boxes' QM rows hold
            print(f"  [{_b}] no age column in the Part 22 frame — using the catalogue's ages.mass_weighted ({int(np.isfinite(A['age']).sum())}/{len(A)} finite)")
        with np.errstate(all="ignore"):
            A["extent"] = np.log10(A["r50gas_kpc"] / A["r50star_kpc"]); A["lfd"] = np.log10(A["mdust"] / A["mstar"])
            A["lfh2"] = np.log10(A["mh2"] / A["mstar"]); A["lm"] = np.log10(A["mstar"])
        A = A[np.isfinite(A["extent"]) & np.isfinite(A[P22_DOSE])].copy()
        _dc, _ec = float(np.nanmedian(A[P22_DOSE])), float(np.nanmedian(A["extent"]))
        A["cls"] = [("hi" if fx > _dc else "lo") + "_" + ("hi" if ex <= _ec else "lo") for fx, ex in zip(A[P22_DOSE], A["extent"])]
        A["regime"] = (A["z"] <= P26_ZMAX) & (A["lm"] >= P26_LOGM)
        A["in_obs_box"] = _inobs26(A) & A["regime"]
        FR26[_b] = (A, _dc, _ec)
        print(f"\n[{_b}] {len(A)} quenched galaxies; dose split at {_dc:.2f}, extent split at {_ec:+.2f} (catalogue r50_gas / r50_star)")
        for _sel, _lab in ((np.ones(len(A), bool), "all anchors"), ((A["z"] <= P26_ZMAX).to_numpy(), f"z <= {P26_ZMAX:g}"),
                           (A["regime"].to_numpy(), f"z <= {P26_ZMAX:g} and log M* >= {P26_LOGM:g} (the observed regime)")):
            print(f"  -- {_lab}: N={int(_sel.sum())}")
            for _c in _ORDER26:
                s = A[_sel & (A["cls"] == _c).to_numpy()]
                _nin = int(_inobs26(s).sum()) if len(s) else 0
                print(f"     {NAME26[_c]:26s} N={len(s):4d}: age {_f26(s['age'])}  logM* {_f26(s['lm'])}  lfd {_f26(s['lfd'])}  "
                      f"lfh2 {_f26(s['lfh2'])}  extent {_f26(s['extent'])}  in the observed box {_nin:3d} ({100 * _nin / max(len(s), 1):.0f} %)")
                R26.append(dict(box=_b, selection=_lab, group=NAME26[_c], n=len(s), n_in_obs_box=_nin,
                                **{f"{q}_med": float(np.nanmedian(s[q])) if len(s) else np.nan for q in ("age", "lm", "lfd", "lfh2", "extent")}))
        # ── the galaxies inside the observed box, one by one ──
        _ib = A[A["in_obs_box"]].sort_values(["cls", "age"], key=lambda c: c.map({k: i for i, k in enumerate(_ORDER26)}) if c.name == "cls" else c)
        _ib = _ib.assign(num=np.arange(1, len(_ib) + 1)); INBOX26[_b] = _ib
        print(f"  the {len(_ib)} galaxies of the observed regime inside the observed box (age {P26_OBOX['age']}, lfd {P26_OBOX['lfd']}, "
              f"lfh2 {P26_OBOX['lfh2']}, extent {P26_OBOX['extent']}):")
        for _, r in _ib.iterrows():
            print(f"     #{int(r['num']):d}  snap {int(r['snap']):3d} gal {int(r['gal_id']):5d}  {NAME26[r['cls']]:26s}  z={r['z']:.2f}  age {r['age']:.2f} Gyr  "
                  f"log M* {r['lm']:.2f}  lfd {r['lfd']:+.2f}  lfh2 {r['lfh2']:+.2f}  extent {r['extent']:+.2f}  {P22_DOSE} {r[P22_DOSE]:.2f}")
        Rz = A[A["z"] <= P26_ZMAX]; X = Rz[["age", "lfd", "lfh2", "extent"]].to_numpy(float)
        _sc = np.array([P26_TOL[q] for q in ("age", "lfd", "lfh2", "extent")])
        print(f"  per source (z <= {P26_ZMAX:g} anchors, any mass): counterparts within the tolerance box by group (those above log M* {P26_LOGM:g}) | the nearest")
        for _i, r in OBS26.iterrows():
            x = np.array([r["age"], r["lfd"], r["lfh2"], r["extent"]]); dd = np.abs(X - x) / _sc
            inb = np.all(dd < 1, axis=1); dist = np.sqrt((dd ** 2).sum(1)); j = int(np.argmin(dist)) if len(dist) else -1
            cnt = Rz[inb]["cls"].value_counts().to_dict(); massive = int((inb & (Rz["lm"] >= P26_LOGM).to_numpy()).sum())
            print(f"     {_i:5s}: {cnt if cnt else 'none'} ({massive} above {P26_LOGM:g})"
                  + (f" | nearest d={dist[j]:.1f} {Rz.iloc[j]['cls']} log M* {Rz.iloc[j]['lm']:.2f} (snap {int(Rz.iloc[j]['snap'])}, gal {int(Rz.iloc[j]['gal_id'])})" if j >= 0 else ""))
            R26.append(dict(box=_b, selection="per source", group=str(_i), n=int(inb.sum()), n_in_obs_box=massive,
                            nearest_d=float(dist[j]) if j >= 0 else np.nan, nearest_cls=Rz.iloc[j]["cls"] if j >= 0 else ""))

    # ── the figure, one per box: the four class tracks with their dispersion, the in-box galaxies numbered, the ALMA sources with errors ──
    _PL26 = [("age", "lfd", "age [Gyr]", r"$\log M_{\rm dust}/M_\star$"), ("age", "lfh2", "age [Gyr]", r"$\log M_{\rm H_2}/M_\star$"),
             ("age", "extent", "age [Gyr]", r"$\log R_{50}({\rm gas})/R_{50}(\star)$  |  $\log R_{\rm CO}/R_e$"),
             ("lfh2", "lfd", r"$\log M_{\rm H_2}/M_\star$", r"$\log M_{\rm dust}/M_\star$")]
    _LIM26 = dict(age=((1.0 if P26_HIST else 2.5), 8.5), lfd=(-7.2, -1.8), lfh2=(-3.6, -0.6), extent=(-0.6, 0.6))
    for _b, (A, _dc, _ec) in FR26.items():
        T = A[A["regime"]] if P26_TRACK_SEL == "regime" else A[A["z"] <= P26_ZMAX]
        _tsel = (f"z <= {P26_ZMAX:g}, log M* >= {P26_LOGM:g}" if P26_TRACK_SEL == "regime" else f"z <= {P26_ZMAX:g}, every mass")
        _ib = INBOX26[_b]
        HT = {}                                                   # the history tracks of the classes, on the age grid
        if P26_HIST:
            _keys = list(zip(T["snap"].astype(int), T["gal_id"].astype(int)))
            _H = _hist26(_b, _keys)
            if not _H:
                print(f"  [{_b}] history tracks: no chains found ({HIST_FILE.format(box=_b)} missing, or without age / mdust) — skipped")
            else:
                print(f"  [{_b}] history tracks: {len(_H)} of {len(_keys)} chains of the track selection in {os.path.basename(HIST_FILE.format(box=_b))}; "
                      f"per class the median chain vs the mass-weighted age along the branch (o = the median age at SFT):")
                _tsft = dict(zip(_keys, T["t_sft_gyr"].to_numpy(float))) if "t_sft_gyr" in T.columns else {}
                _nodes = [2.0, 3.0, 4.0, 5.0, 6.0]
                for _c in _ORDER26:
                    ks = [k for k in zip(T.loc[T["cls"] == _c, "snap"].astype(int), T.loc[T["cls"] == _c, "gal_id"].astype(int)) if k in _H]
                    if len(ks) < P26_HIST_NMIN:
                        print(f"     {NAME26[_c]:26s}: {len(ks)} chains — below P26_HIST_NMIN, no history track"); continue
                    _asft = np.array([np.interp(_tsft[k], _H[k]["t"], _H[k]["age"]) for k in ks if np.isfinite(_tsft.get(k, np.nan))])
                    HT[_c] = dict(n=len(ks), age_sft=float(np.nanmedian(_asft)) if len(_asft) else np.nan,
                                  **{q: _hband26([_onage26(_H[k], q) for k in ks]) for q in P26_HIST})
                    _at = lambda q, a: np.interp(a, P26_HIST_AGE, HT[_c][q]["med"])
                    print(f"     {NAME26[_c]:26s}: {len(ks):3d} chains, age at SFT {HT[_c]['age_sft']:.2f} Gyr | "
                          + "  ".join(f"{q} at " + ", ".join(f"{a:.0f}: {_at(q, a):+.2f}" for a in _nodes) for q in P26_HIST))
                    for q in P26_HIST:
                        R26 += [dict(box=_b, selection="history track", group=NAME26[_c], quantity=q, age=float(a), n=int(n_), med=float(m_), lo=float(l_), hi=float(h_))
                                for a, m_, l_, h_, n_ in zip(P26_HIST_AGE, HT[_c][q]["med"], HT[_c][q]["lo"], HT[_c][q]["hi"], HT[_c][q]["n"]) if np.isfinite(m_)]
        fig, axs = plt.subplots(2, 2, figsize=(14, 11), constrained_layout=True)
        for ax, (x, y, xl, yl), _tag in zip(axs.ravel(), _PL26, "abcd"):
            for _c in _ORDER26:
                s = T[T["cls"] == _c]
                if P26_POINTS:
                    ax.scatter(s[x], s[y], s=7, color=COL26[_c], alpha=0.25, lw=0, rasterized=True)
                tr = _track26(s[x], s[y])
                if tr is None:
                    if _tag == "a": print(f"  [{_b}] {NAME26[_c]}: {len(s)} galaxies in the track selection — below P26_NMIN, no track")
                    continue
                ax.fill_between(tr["x"], tr["lo"], tr["hi"], color=COL26[_c], alpha=0.16, lw=0)
                ax.plot(tr["x"], tr["med"], color=COL26[_c], lw=2.4, label=f"{NAME26[_c]} (N={tr['n']}, window {tr['k']})" if _tag == "d" else None)
            # the history tracks: where the class came from, dashed, a circle at the median age at SFT
            if x == "age" and y in P26_HIST:
                for _c, h in HT.items():
                    ax.plot(P26_HIST_AGE, h[y]["med"], color=COL26[_c], lw=1.6, ls="--", zorder=3)
                    if P26_HIST_BAND:
                        ax.plot(P26_HIST_AGE, h[y]["lo"], color=COL26[_c], lw=0.8, ls=":", zorder=3); ax.plot(P26_HIST_AGE, h[y]["hi"], color=COL26[_c], lw=0.8, ls=":", zorder=3)
                    if np.isfinite(h["age_sft"]):
                        ax.plot(h["age_sft"], np.interp(h["age_sft"], P26_HIST_AGE, h[y]["med"]), marker="o", mfc="w", mec=COL26[_c], ms=7.5, mew=1.6, ls="none", zorder=4)
            # the observed box on this plane
            if x in P26_OBOX and y in P26_OBOX:
                ax.add_patch(plt.Rectangle((P26_OBOX[x][0], P26_OBOX[y][0]), np.diff(P26_OBOX[x])[0], np.diff(P26_OBOX[y])[0],
                                           fill=False, ls="--", lw=0.9, ec="0.35", zorder=2))
            # the in-box galaxies, one marker each, numbered
            for _, r in _ib.iterrows():
                ax.scatter(r[x], r[y], s=95, marker="D", color=COL26[r["cls"]], edgecolors="k", lw=0.9, zorder=6)
                ax.annotate(str(int(r["num"])), (r[x], r[y]), xytext=(5, 4), textcoords="offset points", fontsize=8, fontweight="bold", zorder=7)
            # the observed sources with their errors; arrows for the upper limits
            _ex = OBS26[f"{x}_err"].to_numpy(float) if x != "extent" else np.vstack([OBS26["extent_elo"], OBS26["extent_ehi"]]).astype(float)
            _ey = OBS26[f"{y}_err"].to_numpy(float) if y != "extent" else np.vstack([OBS26["extent_elo"], OBS26["extent_ehi"]]).astype(float)
            _ulx = OBS26[f"{x}_ul"].to_numpy(bool) if f"{x}_ul" in OBS26.columns else np.zeros(len(OBS26), bool)
            _uly = OBS26[f"{y}_ul"].to_numpy(bool) if f"{y}_ul" in OBS26.columns else np.zeros(len(OBS26), bool)
            _ex = np.where(_ulx, P26_ULIM.get(x, 0.3), np.nan_to_num(_ex)); _ey = np.where(_uly, P26_ULIM.get(y, 0.3), np.nan_to_num(_ey))
            ax.errorbar(OBS26[x], OBS26[y], xerr=_ex, yerr=_ey, xuplims=_ulx, uplims=_uly, fmt="none", ecolor="#e7298a", elinewidth=1.1, capsize=2.5, zorder=5)
            ax.scatter(OBS26[x], OBS26[y], s=150, marker="*", c="#e7298a", edgecolors="k", lw=0.8, zorder=6,
                       label=f"ALMA-C11 quiescent galaxies (N={len(OBS26)}; arrows = upper limits)" if _tag == "d" else None)
            if y == "extent":
                ax.axhline(_ec, color="0.4", ls=":", lw=1.0); ax.text(0.99, _ec, f"extent split {_ec:+.2f}", color="0.35", fontsize=7.5, ha="right", va="bottom", transform=ax.get_yaxis_transform())
            ax.set_xlim(*_LIM26[x]); ax.set_ylim(*_LIM26[y]); ax.set_xlabel(xl); ax.set_ylabel(yl); ax.grid(alpha=0.2)
            ax.text(0.02, 0.97, f"({_tag})", transform=ax.transAxes, fontsize=11, fontweight="bold", va="top")
        _hd, _ld = axs[1, 1].get_legend_handles_labels()
        if HT:
            _hd += [Line2D([], [], color="0.3", lw=1.6, ls="--", marker="o", mfc="w", mec="0.3", ms=7, label="dashed: the median history along the main branch vs the age along it; o = the median age at SFT")]
        axs[1, 1].legend(handles=_hd, frameon=True, framealpha=0.85, fontsize=7.6, loc="lower left", title="the classes: median track, 16-84 band", title_fontsize=7.8)
        if len(_ib):
            _hh = [Line2D([], [], marker="D", ls="none", ms=7, color=COL26[r["cls"]], markeredgecolor="k",
                          label=f"#{int(r['num'])}: snap {int(r['snap'])} gal {int(r['gal_id'])}  {NAME26[r['cls']]}, {r['age']:.1f} Gyr, log M* {r['lm']:.2f}")
                   for _, r in _ib.iterrows()]
            axs[0, 1].legend(handles=_hh, frameon=True, framealpha=0.85, fontsize=7.0, loc="lower left", title=f"inside the observed box (dashed), N={len(_ib)}", title_fontsize=7.6)
        fig.suptitle(f"Part 26 — {BOX_LABEL.get(_b, _b)}: the quenched galaxies split by X-ray dose x catalogue gas extent ({_tsel})\n"
                     f"each class as its population track (solid, band) and its history (dashed), the galaxies inside the observed box numbered, the ALMA-C11 quiescent galaxies with their errors", fontsize=11)
        paper_save(fig, "agn_states_boxes_vs_alma" if _b == P26_BOXES[0] else f"agn_states_boxes_vs_alma_{_b}")
        plt.show()
    if INBOX26:
        _pi26 = os.path.join(OUT, "agn_states_boxes_vs_alma_inbox.csv")
        pd.concat([v.assign(box=k) for k, v in INBOX26.items()], ignore_index=True).drop(columns=[c for c in ("key", "fp") if c in next(iter(INBOX26.values())).columns]).to_csv(_pi26, index=False)
        print(f"\n  the in-box galaxies -> {_pi26}")
    if R26:
        _p26 = os.path.join(OUT, "agn_states_boxes_vs_alma.csv"); pd.DataFrame(R26).to_csv(_p26, index=False)
        print(f"  the tables -> {_p26}")


## Part 27 — the companions: each ALMA-C11 source's closest simulated galaxy, seen at the source's own inclination and position angle

Part 26 placed the ALMA-C11 sources on the population tracks; this part goes one galaxy at a time. For every CO-detected
source the 100 Mpc regime sample (Part 26's `FR26`: quenched, $z \le$ `P26_ZMAX`, $\log M_* \ge$ `P26_LOGM`, with the
dose $\times$ extent class) is searched for the **closest galaxy in the observable space** — the mass-weighted age, $\log
M_{\rm dust}/M_*$, $\log M_{\rm H_2}/M_*$, the H$_2$/stellar extent and $\log M_*$, each divided by its tolerance
(`P27_AXES`) and summed in quadrature; an observed upper limit (hc2's dust, the unresolved CO sizes) penalises only the
galaxies above it. The closest match and `P27_NBEST` runner-ups are tabulated (`agn_states_companions.csv`).

That companion is then **rendered from the line of sight the observation had** (`ReducedRender`), on a sky with North up
and East left, tilted to the source's inclination and turned so that the projected major axis sits at the source's position
angle — next to the source's own ACS F814W stamp at the same physical scale (`P27_SIZE` kpc a side). With `P27_STYLE =
"contours"` (default) each side shows **its stars with the dust and H$_2$ as contours**: the observed panel is the F814W
stamp with the imaging pipeline's continuum (orange) and CO(3–2) moment-0 (blue) maps at `P27_OBS_LEVELS` × the map rms (the
first level also negative, dashed; the beam drawn in the corner); the simulated panel is $\log\Sigma_*$ with the companion's
own **projected $\Sigma_{\rm dust}$ and $\Sigma_{\rm H_2}$ as they are**, contoured at `P27_SIM_FRAC` (5, 10, 20, 40, 80 %) of
each map's peak (`P27_SIM_LEVELS = "peak"`; the peak values are written on the panel), smoothed with the source's beam only so
the contours trace the distribution rather than the particles (`P27_BEAM`). Not a flux-matched comparison — the shapes and
extents are the point; `"obs"` (each $\sigma$ level turned into M$_\odot$ pc$^{-2}$ through the source's $M_{\rm H_2}/S_{\rm
CO}$, $M_{\rm dust}/S_\nu$ and beam area) or an explicit `dict(H2=…, dust=…)` remain available, and the table still prints, per
source, the data's 3σ level in M$_\odot$ pc$^{-2}$ against the companion's peak for reference. A third row shows the same
companion **face-on, as its gas**: Part 24's composite with the `P27_FACE_CHAN` channels (gas blue, dust orange; no stars).
`"rgb"` restores the stars / gas / dust composite in the geometry row. The observed geometry comes from `obs_data/almac11/almac11_orientation.csv` (built locally by
`make_orientation_table.py`): the COSMOS-Web NIRCam Sérsic axis ratio and angle (the angle is already a sky PA East of North —
it agrees with the CO uv-fit angles and with the second moments of the F814W stamps), the ACS gim2d axis ratio and the stamp
moments where COSMOS-Web has no fit (hc5, hc9, hc10, hc12), the CO uv-fit PA, and 3D-Barolo's inclination where a kinematic
fit converged (hc5, $i = 75°$). The inclination follows `P27_INC`: `"kin_first"` (Barolo where it exists, else $\cos^2 i =
(q^2 - q_0^2)/(1 - q_0^2)$ with `P27_Q0` $= 0.2$), `"optical"`, a fixed number, or `"match_q"` — tilt the simulated galaxy
until its **apparent** axis ratio equals the observed one (edge-on when it cannot get that flat). Whatever the mode, every
companion's apparent $q$ is measured from the projected stellar mass and scanned face-on to edge-on, so the table also says
how flat the simulated body can look at all: **the SIMBA companions are rounder than the ALMA-C11 discs** (edge-on $q
\approx 0.7$ against observed 0.13–0.5).

What the simulated 'face-on' means is `P27_FRAME`: `"stellar"` (default) is the principal frame of the member stars — the
line of sight tilts away from the minor axis, which is what a Sérsic axis ratio measures; `"star_L"` / `"H2_L"` / `"gas_L"`
use a spin instead. `P27_AZIMUTH` picks the in-plane direction of the tilt (0 = toward the intermediate axis, so the major
axis stays on the sky and carries the PA).

**Other boxes.** `P27_BOXES` lists the boxes, one figure each (the first keeps the plain product names, the others get a
`_<box>` suffix): every box needs its Part 26 frame (`P26_BOXES`, re-run Part 26) and its histories cache; the cutout file
stem comes from the config's `file_format` (m100n1024 / m50n512 / m25n512, `P27_PREFIXES` overrides), the anchor cutouts
from `output/<box>/reduced_particles`, the branch cutouts from `output/<box>/filtered_particles`, and the plan goes to
`output/<box>/caesar_sfh/prof_almamatch/`. The 50 Mpc box has 38 regime galaxies and no cutouts yet (the plan is written);
the 25 Mpc box has almost none above the regime mass cut (`P27_POOL = "zmax"` searches every mass) but many reduced cutouts
already, and no merger-tree links, so only anchors with a cutout draw.

**Where the particles come from.** The 100 Mpc anchors (snaps 125 and 134) have no reduced cutouts yet, so the cell writes
a plan for `build_reduced_particles_job.py` with every listed companion (closest + runner-ups) whose anchor cutout is
missing — `output/cis100/caesar_sfh/prof_almamatch/dust_profile_plan_almamatch.hdf5`, the printed `sbatch` line builds
them — and, until those exist, draws the galaxy from the **RT cutout of the same main branch** nearest in time
(`output/cis100/filtered_particles`, the ALMA-C11 analogue campaign's members-only cutouts at snaps 128–132, linked through
the catalogues' `tree_data` and converted on the fly into the reduced schema in `OUT/rt_reduced/`). The panel says which
snapshot it shows and the time offset from the anchor. A companion with no particle data at all is drawn through the
closest runner-up that has some (a **stand-in**, dashed edge, `P27_STANDIN`). After the sbatch, re-running the part
replaces the branch cutouts and the stand-ins with the anchors themselves.

Products: `agn_states_companions.{png,pdf}` (rows: the F814W stamp with the ALMA contours, the companion at the observed
geometry, the same companion face-on as its gas — `P27_FACEON_ROW`) and `agn_states_companions.csv` (per source: the
geometry used and its origin, the closest match and runner-ups with the distance and the observables on both sides, what was
drawn from where, the apparent $q$ and PA of the render, the face-on and edge-on $q$, the map rms and 3σ levels in
M$_\odot$ pc$^{-2}$, the companion's peak $\Sigma_{\rm H_2}$ / $\Sigma_{\rm dust}$ at the beam).

First run (branch cutouts, 6 of 10 sources): the data's CO sits on the stellar bodies at 3–20σ with the dust following it;
the companions' H$_2$ and dust contours are concentric on the stellar body, more compact than the data's, and their face-on
gas is a ring or a few clumps rather than a disc. For reference every companion's beam-smoothed H$_2$ peak (30–80
M$_\odot$ pc$^{-2}$) clears the source's 3σ level (10–35), while the dust peaks (0.01–1.2) mostly stay below the continuum 3σ
(0.35–2) — the dust gap of Part 26.


In [ ]:
# ── Part 27 — THE COMPANIONS: the closest simulated galaxy of every ALMA-C11 source, drawn at the source's own PA and inclination ──
# For each CO-detected ALMA-C11 source Part 26's regime sample (FR26: the quenched 100 Mpc galaxies at z <= P26_ZMAX, log M* >=
# P26_LOGM, with the dose x extent class) is searched for the CLOSEST galaxy in the observable space (P27_AXES: the age, the dust
# and H2 fractions, the H2/stellar extent and the stellar mass, each in units of its tolerance; an observed upper limit only
# penalises galaxies ABOVE it).  That companion is then rendered with simbanator's ReducedRender from a line of sight that
# reproduces the OBSERVED viewing geometry: the inclination from the source's axis ratio (cos^2 i = (q^2 - q0^2) / (1 - q0^2);
# the 3D-Barolo inclination where a kinematic fit exists) and the position angle from the NIRCam / ACS Sersic fit, on a sky
# with North up and East left — next to the source's own ACS F814W stamp at the same physical scale.  P27_STYLE "contours"
# (default) draws each side as ITS STELLAR LIGHT / MASS with the DUST and H2 as CONTOURS: the imaging pipeline's continuum and
# CO(3-2) moment-0 maps at P27_OBS_LEVELS x rms on the stamp, the companion's own Sigma_dust and Sigma_H2 (beam-convolved,
# P27_BEAM) on its log Sigma_* map, at the SAME physical levels (each sigma level turned into Msun/pc^2 through the source's
# M_H2 / S_CO and M_dust / S_nu, P27_SIM_LEVELS "obs"); "rgb" is Part 24's stars / gas / dust composite.  The particle data: the reduced cutout of the anchor itself (output/<box>/reduced_particles, built by
# build_reduced_particles_job.py from the plan THIS cell writes) or, until that exists, the RT cutout of the SAME branch at the
# nearest snapshot that has one (output/<box>/filtered_particles, the ALMA-C11 analogue campaign's members-only cutouts,
# converted on the fly into the reduced schema) — the panel says which.  A companion with no particle data on disk at all
# is drawn through the closest match that HAS some (a stand-in, dashed edge), and listed in the plan.
P27_BOXES   = ["cis25"]      # one figure per box, in this order: ["cis100", "cis50", "cis25"] — every box needs its Part 26 frame (P26_BOXES) and
                              #   its histories cache; the first box keeps the plain product names, the others get a _<box> suffix
P27_PREFIXES = {}             # {"cis50": "m50n512"}: the file stem of a box's cutouts; default = the stem of the config's file_format
                              #   (m100n1024 / m50n512 / m25n512)
P27_OBS_ORI = os.path.join(os.getcwd(), "obs_data", "almac11", "almac11_orientation.csv")     # PA / axis ratio / kinematic i per source
P27_OBS_STAMPS = os.path.join(os.getcwd(), "obs_data", "almac11", "almac11_stamps_f814w.npz")  # the ACS F814W stamps (N up, E left)
P27_AXES    = dict(age=0.5, lfd=0.3, lfh2=0.3, extent=0.3, lm=0.15)   # the match: the axes and the scale of each — a distance of 1
                                                                       #   = "one tolerance away" summed in quadrature over the axes
P27_POOL    = "regime"        # the galaxies searched: "regime" (Part 26's z <= P26_ZMAX and log M* >= P26_LOGM) | "zmax" (any mass)
P27_NBEST   = 3               # runner-ups listed (and planned) per source
P27_STANDIN = True            # the closest match has no particle data yet -> draw the closest one that HAS (dashed edge, 'stand-in')
P27_SOURCE  = "auto"          # "auto" = the reduced cutout at the anchor, else the RT cutout of the same branch nearest in time
                              #   | "reduced" (anchor cutouts only) | "rt" (branch cutouts only)
P27_REDUCED_FMT = os.path.join(os.getcwd(), "output", "{box}", "reduced_particles")     # the anchor cutouts of a box
P27_RT_FMT  = os.path.join(os.getcwd(), "output", "{box}", "filtered_particles")        # <dir>/snap_NNN/<prefix>_snapNNN_galGGGGGG.h5
P27_RT_DT   = 1.5             # [Gyr] how far along the branch (either way) an RT cutout may sit from the anchor (the z = 0.49 anchors' nearest
                              #   cutouts are ~1 Gyr later, at snaps 128-132; the panel prints the offset)
P27_RT_CACHE_FMT = os.path.join(OUT, "rt_reduced", "{box}")          # the converted RT cutouts (reduced schema) live here
P27_PLAN_FMT = os.path.join(os.getcwd(), "output", "{box}", "caesar_sfh", "prof_almamatch", "dust_profile_plan_almamatch.hdf5")
P27_CATDIR  = {}              # {"cis100": "/path/to/Groups"} overrides for the caesar catalogues (the branch links); default P26_CATDIR / config
# the view
P27_Q0      = 0.2             # the intrinsic (edge-on) axis ratio behind cos^2 i = (q^2 - q0^2) / (1 - q0^2); q <= q0 -> 90 deg
P27_INC     = "kin_first"     # "kin_first" (3D-Barolo inclination where it exists, else from q_opt) | "optical" (from q_opt only) | a number (that
                              #   inclination for all) | "match_q": tilt the SIMULATED galaxy until its apparent axis ratio equals the observed q
                              #   (edge-on when it cannot get that flat — the table says how flat it can get)
P27_PA      = "opt"           # "opt" (NIRCam / ACS-stamp position angle) | "co" (the CO uv fit, where its shape is measured) | "kin" (Barolo)
P27_FRAME   = "stellar"       # what the simulated galaxy's 'face-on' is: "stellar" = the principal axes of its member stars (the line of
                              #   sight tilts away from the MINOR axis, what a Sersic axis ratio measures) | "star_L" | "H2_L" | "gas_L" (a spin)
P27_FRAME_RMAX = 10.0         # [kpc] the spin frames' radius
P27_AZIMUTH = 0.0             # [deg] which way the line of sight tilts, inside the plane: 0 = toward the intermediate axis, so the major axis
                              #   stays on the sky and carries the PA; 90 = toward the major axis
P27_SIZE    = 30.0            # [kpc] a side — the simulated panels AND the observed stamps (cut to the same physical size)
P27_MEMBERS = True            # the RT cutouts hold members only anyway; False shows the whole region of a reduced (100 kpc) cutout
P27_CHAN    = ("star", "gas", "dust"); P27_DYN = (2.0, 2.5, 2.5); P27_PMAX = 99.9; P27_STRETCH = "dyn"; P27_XSIZE = 400   # decades per channel
P27_INC_STEP = 2.0            # [deg] the inclination scan of P27_INC = "match_q" (and of the edge-on / face-on q reported for every companion)
P27_STYLE   = "contours"      # "contours" = the STELLAR map with the dust and H2 CONTOURS: observed = the ACS F814W stamp + the imaging pipeline's
                              #   continuum (dust) and CO(3-2) moment-0 (H2) at P27_OBS_LEVELS x rms; simulated = log Sigma_* + its own
                              #   Sigma_dust / Sigma_H2 contours | "rgb" = Part 24's stars / gas / dust composite
P27_OBS_LEVELS = (3, 5, 8, 12, 20)   # the observed contour levels, in units of each map's rms (the first one also drawn negative, dashed)
P27_SIM_LEVELS = "peak"       # the simulated contours: "peak" = the projected Sigma_H2 / Sigma_dust AS THEY ARE, at P27_SIM_FRAC of each map's own
                              #   peak | "obs" = the same physical levels as the data (each sigma level -> Msun/pc^2 through the source's own
                              #   M_H2 / S_CO, M_dust / S_nu and beam area) | dict(H2=(...), dust=(...)) in Msun/pc^2
P27_SIM_FRAC = (0.3, 0.5, 0.9)   # the "peak" levels, fractions of the map maximum
P27_SIM_FALLBACK = dict(H2=(3.0, 10.0, 30.0, 100.0), dust=(0.03, 0.1, 0.3, 1.0))   # [Msun/pc^2] where "obs" has no conversion (dust undetected)
P27_BEAM    = True            # smooth the simulated Sigma_dust / Sigma_H2 maps with the source's ALMA beam before contouring (so the contours
                              #   trace the distribution, not the individual particles)
P27_FACE_CHAN = ("gas", "dust")   # the face-on row shows the GAS: Part 24's composite with these channels (no stars) — "H2" is available too
P27_FACE_DYN = (2.5, 2.5)     # decades per face-on channel
P27_HE      = 1.36            # the simulated H2 (hydrogen only) x this = the alpha_CO H2 of the data (helium included)
P27_STAR_DYN = 2.5            # decades of the simulated stellar map below its 99.9th percentile
P27_COL     = dict(dust="#ff9f43", H2="#4fc3f7")   # the contour colours, both sides
P27_OBS_ROW = True            # the F814W stamp above each companion (P27_OBS_STAMPS)
P27_FACEON_ROW = True         # a further row: the same companion face-on, its gas distribution (P27_FACE_CHAN)
P27_NCOL    = 5               # sources per block of rows
P27_FS      = 1.0             # text size multiplier
P27_MOM_R   = 1.5             # the apparent axis ratio of the render: the star mass moments inside this x the projected R50(*)

if "FR26" not in globals() or "OBS26" not in globals():
    print("Part 27: needs Part 26 (FR26 / OBS26) — run it first.")
elif not os.path.exists(P27_OBS_ORI):
    print(f"Part 27: {P27_OBS_ORI} is missing — obs_data/almac11/make_orientation_table.py builds it (locally) and it is copied here.")
else:
    _STEM27 = {"cis100": "m100n1024", "cis50": "m50n512", "cis50nox": "m50n512", "cis25": "m25n512"}   # the SIMBA file stems, if no config

    def _catdir27(box):
        """(caesar directory, file pattern) of a box: P27_CATDIR / P26_CATDIR override the directory, ~/.simbanator/config.json the rest."""
        import json as _json27
        try:
            cfg = _json27.load(open(os.path.expanduser("~/.simbanator/config.json"))); ent = (cfg.get("simulations", cfg)).get(box, {})
        except Exception:
            ent = {}
        d = P27_CATDIR.get(box) or (globals().get("P26_CATDIR") or {}).get(box) or ent.get("catalog_dir")
        return d, (ent.get("file_format") or "%s_{snap:03d}.hdf5" % P27_PREFIXES.get(box, _STEM27.get(box, box)))

    for P27_BOX in P27_BOXES:
        if P27_BOX not in FR26:
            print(f"\n[{P27_BOX}] not in Part 26's frames — add it to P26_BOXES and re-run Part 26; skipped."); continue
        _cd27, _ff27 = _catdir27(P27_BOX)
        P27_PREFIX = P27_PREFIXES.get(P27_BOX) or (_ff27.split("_{snap")[0] if "_{snap" in _ff27 else _STEM27.get(P27_BOX, P27_BOX))
        P27_REDUCED, P27_RT_DIR = P27_REDUCED_FMT.format(box=P27_BOX), P27_RT_FMT.format(box=P27_BOX)
        P27_RT_CACHE, P27_PLAN = P27_RT_CACHE_FMT.format(box=P27_BOX), P27_PLAN_FMT.format(box=P27_BOX)
        _TREE27 = {}
        _tag27 = "" if P27_BOX == P27_BOXES[0] else f"_{P27_BOX}"
        print(f"\n{'=' * 24} {BOX_LABEL.get(P27_BOX, P27_BOX)}: cutouts <{P27_PREFIX}>, anchors in {os.path.relpath(P27_REDUCED, os.getcwd())}, "
              f"branch RT cutouts in {os.path.relpath(P27_RT_DIR, os.getcwd())} {'=' * 24}")
        import h5py as _h5_27
        import matplotlib.patheffects as _pe27
        from matplotlib.patches import Rectangle as _Rect27
        _fs27 = lambda v: float(v) * float(P27_FS)
        _A27, _dc27, _ec27 = FR26[P27_BOX]
        _POOL27 = (_A27[_A27["regime"]] if P27_POOL == "regime" else _A27[_A27["z"] <= P26_ZMAX]).copy().reset_index(drop=True)
        _CLS27 = {g[0]: (g[2], g[4]) for g in GRP22} if "GRP22" in globals() else {
            "lo_hi": ("#b2182b", "low dose + extended"), "lo_lo": ("#ef8a62", "low dose + compact"),
            "hi_hi": ("#67a9cf", "high dose + extended"), "hi_lo": ("#2166ac", "high dose + compact")}
        _short27 = lambda c: str(_CLS27.get(c, ("0.4", c))[1]).replace("$+$", "+").replace("$", "")

        # ── the observed geometry: inclination and position angle per source ──
        _ORI27 = pd.read_csv(P27_OBS_ORI).set_index("id")
        O27 = OBS26.join(_ORI27.drop(columns=[c for c in ("z",) if c in _ORI27.columns]), how="left")

        def _inc_from_q27(q):
            q = float(q)
            if not np.isfinite(q):
                return np.nan
            if q <= P27_Q0:
                return 90.0
            return float(np.degrees(np.arccos(np.sqrt((q * q - P27_Q0 ** 2) / (1.0 - P27_Q0 ** 2)))))

        _inc27, _incsrc27, _pa27, _pasrc27 = [], [], [], []
        for _sid, _r in O27.iterrows():
            if isinstance(P27_INC, (int, float)):
                _i, _is = float(P27_INC), "fixed"
            elif P27_INC == "match_q":
                _i, _is = (_inc_from_q27(_r.get("q_opt", np.nan)) if np.isfinite(_r.get("q_opt", np.nan)) else np.nan), "match_q"   # a placeholder: set per companion
            elif P27_INC == "kin_first" and np.isfinite(_r.get("inc_kin_deg", np.nan)):
                _i, _is = float(_r["inc_kin_deg"]), "kin"
            else:
                _i, _is = _inc_from_q27(_r.get("q_opt", np.nan)), f"q_{_r.get('q_opt_src', '')}"
            _p, _ps = np.nan, ""
            for _key in ([P27_PA] + [k for k in ("opt", "co", "kin") if k != P27_PA]):
                _v = _r.get({"opt": "pa_opt_deg", "co": "pa_co_deg", "kin": "pa_kin_deg"}[_key], np.nan)
                if np.isfinite(_v):
                    _p, _ps = float(_v) % 180.0, (_key if _key != "opt" else str(_r.get("pa_opt_src", "opt")))
                    break
            _inc27.append(_i); _incsrc27.append(_is); _pa27.append(_p); _pasrc27.append(_ps)
        O27["inc_deg"], O27["inc_src"], O27["pa_deg"], O27["pa_src"] = _inc27, _incsrc27, _pa27, _pasrc27
        O27 = O27[np.isfinite(O27["inc_deg"]) & np.isfinite(O27["pa_deg"])]
        print(f"Part 27 — the companions of the {len(O27)} CO-detected ALMA-C11 sources in {BOX_LABEL.get(P27_BOX, P27_BOX)} "
              f"({P27_POOL}: {len(_POOL27)} galaxies).  Viewing geometry: i from q with q0 = {P27_Q0:g}"
              + (" (3D-Barolo where it exists)" if P27_INC == "kin_first" else "") + f", PA = {P27_PA}; the simulated frame = {P27_FRAME}, "
              f"tilt azimuth {P27_AZIMUTH:g} deg.")
        print("  source      z    q_opt   i [deg] (from)     PA [deg] (from)")
        for _sid, _r in O27.iterrows():
            print(f"  {_sid:6s} {_r['z']:5.2f}  {_r.get('q_opt', np.nan):5.2f}   {_r['inc_deg']:5.1f}  {_r['inc_src']:10s}   "
                  f"{_r['pa_deg']:6.1f}  {_r['pa_src']}")

        # ── the match: the closest galaxy of the pool in the observable space ──
        _ULS27 = {"lfd": "lfd_ul", "extent": "extent_ul"}
        _X27 = {k: _POOL27[k].to_numpy(float) for k in P27_AXES}

        def _dist27(row):
            d2, used = np.zeros(len(_POOL27)), []
            for k, tol in P27_AXES.items():
                v = float(row.get(k, np.nan))
                if not np.isfinite(v) or k not in _X27:
                    continue
                diff = _X27[k] - v
                if k in _ULS27 and bool(row.get(_ULS27[k], False)):
                    diff = np.maximum(diff, 0.0)                     # an upper limit: only galaxies above it are off
                d2 += (diff / tol) ** 2; used.append(k)
            d2[~np.isfinite(d2)] = np.inf
            return np.sqrt(d2), used

        MATCH27 = {}
        for _sid, _r in O27.iterrows():
            _d, _used = _dist27(_r)
            _o = np.argsort(_d)[:max(1, P27_NBEST)]
            MATCH27[_sid] = [dict(_POOL27.iloc[j][["snap", "gal_id", "anchor_snap", "z", "age", "lfd", "lfh2", "extent", "lm", "cls"]].to_dict(),
                                  rank=r + 1, dist=float(_d[j]), axes=",".join(_used)) for r, j in enumerate(_o) if np.isfinite(_d[j])]

        # ── particle data: the reduced cutout at the anchor, else the RT cutout of the same branch nearest in time ──
        try:
            from simbanator.visualization.rendering import ReducedRender as _RR27
        except Exception as _e27:
            _RR27 = None
            print(f"Part 27: simbanator.visualization.rendering.ReducedRender not importable ({_e27}) — no renders; the match table and the plan still follow.")

        _TREE27 = {}

        def _tree27(snap):
            """(main progenitor id at snap-1, descendant id at snap+1) arrays of the catalogue at `snap`; None when unreadable."""
            if snap not in _TREE27:
                _d, _f = _catdir27(P27_BOX)
                fn = os.path.join(_d or "", _f.format(snap=int(snap)))
                try:
                    with _h5_27.File(fn, "r") as f:
                        p = np.asarray(f["tree_data/progen_galaxy_star"][:]); d = np.asarray(f["tree_data/descend_galaxy_star"][:])
                    _TREE27[snap] = (p[:, 0] if p.ndim == 2 else p, d[:, 0] if d.ndim == 2 else d)
                except Exception as e:
                    _TREE27[snap] = None
                    print(f"  [tree] {fn}: {type(e).__name__} {str(e)[:60]}")
            return _TREE27[snap]

        def _branch27(anchor, gid, snaps):
            """{snap: gal_id} of the main branch of (anchor, gid) at the wanted snapshots (backward via progen, forward via descend)."""
            out = {int(anchor): int(gid)}
            g = int(gid)
            for s in range(int(anchor), min(snaps) if snaps else int(anchor), -1):            # back: id at s-1 = progen[s][g]
                t = _tree27(s)
                if t is None or g < 0 or g >= len(t[0]):
                    break
                g = int(t[0][g]); out[s - 1] = g
                if g < 0:
                    break
            g = int(gid)
            for s in range(int(anchor), max(snaps) if snaps else int(anchor)):                # forward: id at s+1 = descend[s][g]
                t = _tree27(s)
                if t is None or g < 0 or g >= len(t[1]):
                    break
                g = int(t[1][g]); out[s + 1] = g
                if g < 0:
                    break
            return {s: g for s, g in out.items() if g >= 0 and s in set(snaps) | {int(anchor)}}

        # the cosmic time of every snapshot (the histories cache carries the snapshot ladder)
        _TSNAP27 = {}
        try:
            with _h5_27.File(HIST_FILE.format(box=P27_BOX), "r") as _hf:
                for _gk in _hf.keys():                                   # every anchor group carries its own ladder (anchor and older)
                    _g0 = _hf[_gk]
                    if "snaps" in _g0 and "t_yr" in _g0:
                        _TSNAP27.update({int(s): float(t) / 1e9 for s, t in zip(_g0["snaps"][:], _g0["t_yr"][:])})
        except Exception as _e27:
            print(f"  (no snapshot ladder from {HIST_FILE.format(box=P27_BOX)}: {type(_e27).__name__}; the branch search uses snapshot numbers)")
        _tsnap27 = lambda s: _TSNAP27.get(int(s), np.nan)

        _redpath27 = lambda s, g: os.path.join(P27_REDUCED, f"snap_{int(s):03d}", f"{P27_PREFIX}_snap{int(s):03d}_gal{int(g):06d}.h5")
        _RT_SNAPS27 = sorted(int(d[5:]) for d in (os.listdir(P27_RT_DIR) if os.path.isdir(P27_RT_DIR) else []) if d.startswith("snap_") and d[5:].isdigit())
        _rtpath27 = lambda s, g: os.path.join(P27_RT_DIR, f"snap_{int(s):03d}", f"{P27_PREFIX}_snap{int(s):03d}_gal{int(g):06d}.h5")

        def _convert27(snap, gid):
            """An RT cutout (Gadget layout, comoving kpc/h, members only) -> a reduced-schema file ReducedRender can open."""
            src, dst = _rtpath27(snap, gid), os.path.join(P27_RT_CACHE, f"snap_{int(snap):03d}", f"{P27_PREFIX}_snap{int(snap):03d}_gal{int(gid):06d}.h5")
            if os.path.exists(dst):
                return dst
            try:
                from simbanator.utils.geometry import shrink_center as _shr27, principal_axes as _pax27
            except Exception:
                _shr27 = lambda p, masses=None: np.average(p, axis=0, weights=masses)
                def _pax27(p, masses=None):
                    c = np.cov(p.T, aweights=masses); ev, evec = np.linalg.eigh(c); o = np.argsort(ev)[::-1]
                    return None, ev[o], evec[:, o], c
            MSUN_G, KPC_CM, MP, XH, NH_MIN = 1.989e33, 3.0857e21, 1.6726e-24, 0.76, 0.13
            with _h5_27.File(src, "r") as f:
                H = f["Header"].attrs; a, hub, z = float(H["Time"]), float(H["HubbleParam"]), float(H["Redshift"])
                L = float(H["BoxSize"]) * a / hub
                wrap = lambda d: d - L * np.round(d / L)
                spos = np.asarray(f["PartType4/Coordinates"][:], float) * a / hub
                sm = np.asarray(f["PartType4/Masses"][:], float) * 1e10 / hub
                ref = np.median(spos, axis=0); spos = ref + wrap(spos - ref)
                center = _shr27(spos, masses=sm)
                _, _, evecs, _ = _pax27(spos - center, masses=sm)
                rec = dict(gas={}, star={})
                rec["star"] = dict(pos=(spos - center).astype(np.float32), m_star=sm.astype(np.float32),
                                   member=np.ones(len(sm), bool), tform=np.asarray(f["PartType4/StellarFormationTime"][:], np.float32),
                                   vel=(np.asarray(f["PartType4/Velocities"][:], float) * np.sqrt(a)).astype(np.float32))
                if "PartType0" in f and "Coordinates" in f["PartType0"]:
                    g = f["PartType0"]
                    gpos = ref + wrap(np.asarray(g["Coordinates"][:], float) * a / hub - ref) - center
                    m = np.asarray(g["Masses"][:], float) * 1e10 / hub
                    rho = np.asarray(g["Density"][:], float) * 1e10 * MSUN_G * hub ** 2 / (KPC_CM * a) ** 3
                    nH = rho * XH / MP
                    fH2 = np.asarray(g["FractionH2"][:], float) if "FractionH2" in g else np.zeros(len(m))
                    fneut = np.asarray(g["NeutralHydrogenAbundance"][:], float) if "NeutralHydrogenAbundance" in g else np.zeros(len(m))
                    fH2 = np.where(nH >= NH_MIN, fH2, 0.0)
                    rec["gas"] = dict(pos=gpos.astype(np.float32), m_gas=m.astype(np.float32),
                                      m_dust=(np.asarray(g["Dust_Masses"][:], float) * 1e10 / hub).astype(np.float32) if "Dust_Masses" in g else np.zeros(len(m), np.float32),
                                      m_H2=(XH * m * fH2).astype(np.float32), m_HI=(XH * m * np.minimum(fneut, 1.0 - fH2)).astype(np.float32),
                                      sfr=np.asarray(g["StarFormationRate"][:], np.float32) if "StarFormationRate" in g else np.zeros(len(m), np.float32),
                                      nh=nH.astype(np.float32), zgas=np.asarray(g["Metallicity"][:, 0], np.float32) if "Metallicity" in g else np.zeros(len(m), np.float32),
                                      member=np.ones(len(m), bool), vel=(np.asarray(g["Velocities"][:], float) * np.sqrt(a)).astype(np.float32))
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with _h5_27.File(dst, "w") as o:
                o.attrs["sim_name"] = P27_PREFIX; o.attrs["snap"] = int(snap); o.attrs["gx"] = int(gid); o.attrs["redshift"] = z
                o.attrs["a"] = a; o.attrs["hub"] = hub; o.attrs["rmax_kpc"] = np.nan; o.attrs["h2_recipe"] = "caesar-v1 (rt cutout, members only)"
                o.attrs["center_kpc"] = np.asarray(center, float); o.attrs["evecs"] = np.asarray(evecs, float); o.attrs["source"] = src
                for grp in ("gas", "star"):
                    if not rec[grp]:
                        continue                                   # no particles of this kind: no group (ReducedRender skips it cleanly)
                    gg = o.require_group(grp)
                    for k, v in rec[grp].items():
                        gg.create_dataset(k, data=v)
            return dst

        def _source27(anchor, gid):
            """('reduced' | 'rt', snap, gal_id, path, dt_gyr) of the best particle data of (anchor, gid) on disk, else None."""
            if _RR27 is None:
                return None
            if P27_SOURCE in ("auto", "reduced"):
                p = _redpath27(anchor, gid)
                if os.path.exists(p):
                    return ("reduced", int(anchor), int(gid), p, 0.0)
            if P27_SOURCE in ("auto", "rt") and _RT_SNAPS27:
                t0 = _tsnap27(anchor)
                cand = [s for s in _RT_SNAPS27 if not np.isfinite(t0) or not np.isfinite(_tsnap27(s)) or abs(_tsnap27(s) - t0) <= P27_RT_DT]
                if cand:
                    br = _branch27(anchor, gid, cand)
                    have = [(abs(_tsnap27(s) - t0) if np.isfinite(t0) and np.isfinite(_tsnap27(s)) else abs(s - int(anchor)), s, g)
                            for s, g in br.items() if s in cand and os.path.exists(_rtpath27(s, g))]
                    if have:
                        dt, s, g = sorted(have)[0]
                        try:
                            return ("rt", int(s), int(g), _convert27(s, g), float(_tsnap27(s) - t0) if np.isfinite(t0) and np.isfinite(_tsnap27(s)) else np.nan)
                        except Exception as e:
                            print(f"  [rt] snap {s} gal {g}: {type(e).__name__} {str(e)[:70]}")
            return None

        # ── the imposed line of sight ──
        def _view27(E, inc_deg, pa_deg, az_deg=0.0):
            """The projection frame [West, North, toward-the-observer] (columns) that shows the galaxy whose face-on frame is E
            (columns: major, intermediate, minor/spin) at inclination `inc_deg`, its apparent major axis at position angle
            `pa_deg` East of North, on a sky with North up and East left.  `az_deg` picks the in-plane direction the line of
            sight tilts toward (0 = the intermediate axis, so the major axis stays in the sky plane)."""
            E = np.asarray(E, float); e0, e1, ez = E[:, 0], E[:, 1], E[:, 2]
            i, az, pa = np.radians(inc_deg), np.radians(az_deg), np.radians(pa_deg)
            u = np.cos(az) * e1 + np.sin(az) * e0                             # the in-plane tilt direction
            n = np.cos(i) * ez + np.sin(i) * u; n /= np.linalg.norm(n)        # toward the observer
            m = np.cross(ez, n)                                               # the apparent major axis (disc plane ∩ sky plane)
            if np.linalg.norm(m) < 1e-6:                                      # face-on: the intrinsic major axis
                m = e0 - np.dot(e0, n) * n
            m /= np.linalg.norm(m)
            p = np.cross(n, m)                                                # the other sky direction; (m, p, n) right-handed
            N = np.cos(pa) * m - np.sin(pa) * p
            Ea = np.sin(pa) * m + np.cos(pa) * p
            V = np.column_stack([-Ea, N, n])                                  # image x = West, image y = North
            return V

        def _apparent27(rr, ex):
            """(q, PA E of N [deg], R50 [kpc]) of the projected member stars in the current frame: the mass moments inside P27_MOM_R x R50."""
            st = rr.parts.get("star", {})
            if "pos" not in st:
                return np.nan, np.nan, np.nan
            xy = rr.project("star", "faceon")[:, :2]; w = np.asarray(st["m_star"], float)
            if "member" in st:
                k = np.asarray(st["member"], bool); xy, w = xy[k], w[k]
            if len(w) < 20:
                return np.nan, np.nan, np.nan
            r = np.hypot(xy[:, 0], xy[:, 1]); o = np.argsort(r); cw = np.cumsum(w[o])
            r50 = float(r[o][np.searchsorted(cw, 0.5 * cw[-1])])
            k = r <= P27_MOM_R * r50
            x, y = xy[k, 0], xy[k, 1]; ww = w[k]
            xc, yc = np.average(x, weights=ww), np.average(y, weights=ww)
            xx, yy, xy_ = (np.average((x - xc) ** 2, weights=ww), np.average((y - yc) ** 2, weights=ww), np.average((x - xc) * (y - yc), weights=ww))
            ee, nn, en = xx, yy, -xy_                                          # East = -x
            d = np.sqrt(0.25 * (ee - nn) ** 2 + en ** 2)
            l1, l2 = 0.5 * (ee + nn) + d, 0.5 * (ee + nn) - d
            return float(np.sqrt(max(l2, 0) / l1)), float(np.degrees(0.5 * np.arctan2(2 * en, nn - ee)) % 180.0), r50

        # ── the observed maps (the stamps file) and the physical scale per source ──
        try:
            from astropy.cosmology import Planck18 as _cosmo27
        except Exception:
            from astropy.cosmology import FlatLambdaCDM as _FL27; _cosmo27 = _FL27(H0=67.7, Om0=0.31)
        _kpc_as27 = lambda z: float(_cosmo27.kpc_proper_per_arcmin(float(z)).value / 60.0)
        _STAMPS27, _ALMA27 = None, {}
        if os.path.exists(P27_OBS_STAMPS):
            _npz = np.load(P27_OBS_STAMPS, allow_pickle=False)
            _STAMPS27 = dict(zip([str(x) for x in _npz["ids"]], _npz["stamps"])); _PIX27 = float(_npz["pix_arcsec"])
            if "alma_ids" in _npz.files:
                for _j, _aid in enumerate([str(x) for x in _npz["alma_ids"]]):
                    _ALMA27[_aid] = {k: (_npz[k][_j] if k in _npz.files else None)
                                     for k in ("cont", "mom0", "cont_pix_arcsec", "mom0_pix_arcsec", "cont_x0_as", "cont_y0_as", "mom0_x0_as", "mom0_y0_as")}
        _beam_pc2 = lambda bmaj, bmin, kpc_as: np.pi * float(bmaj) * float(bmin) / (4.0 * np.log(2.0)) * (kpc_as * 1e3) ** 2

        def _obsconv27(o):
            """Msun/pc^2 per map unit (H2 per Jy/beam km/s, dust per Jy/beam) of one source, NaN without a detection / beam."""
            k = _kpc_as27(o["z"])
            cH2 = float(o.get("msun_per_jykms", np.nan)) / _beam_pc2(o.get("mom0_bmaj_as", np.nan), o.get("mom0_bmin_as", np.nan), k)
            cD = float(o.get("msun_per_ujy_dust", np.nan)) * 1e6 / _beam_pc2(o.get("cont_bmaj_as", np.nan), o.get("cont_bmin_as", np.nan), k)
            return cH2, cD

        def _simlevels27(o):
            """The simulated contour levels [Msun/pc^2] of one source: (H2 levels, dust levels, how they were set)."""
            if isinstance(P27_SIM_LEVELS, dict):
                return list(P27_SIM_LEVELS["H2"]), list(P27_SIM_LEVELS["dust"]), "fixed"
            cH2, cD = _obsconv27(o)
            lH2 = [n * float(o.get("mom0_rms_jybkms", np.nan)) * cH2 for n in P27_OBS_LEVELS] if np.isfinite(cH2) and np.isfinite(o.get("mom0_rms_jybkms", np.nan)) else list(P27_SIM_FALLBACK["H2"])
            lD = [n * float(o.get("cont_rms_jyb", np.nan)) * cD for n in P27_OBS_LEVELS] if np.isfinite(cD) and np.isfinite(o.get("cont_rms_jyb", np.nan)) else list(P27_SIM_FALLBACK["dust"])
            return lH2, lD, ("obs" if np.isfinite(cH2) else "fallback-H2") + ("" if np.isfinite(cD) else ",fallback-dust")

        # ── choose what is drawn for every source ──
        _EX27 = 0.5 * float(P27_SIZE)
        REN27, ROWS27, _plan27 = {}, [], []
        for _sid, _r in O27.iterrows():
            _cands = MATCH27.get(_sid, [])
            if not _cands:
                print(f"  {_sid}: no finite match"); continue
            _drawn, _src = None, None
            for _c in _cands:                                            # in rank order
                if not os.path.exists(_redpath27(_c["anchor_snap"], _c["gal_id"])):
                    _plan27.append((int(_c["anchor_snap"]), int(_c["gal_id"])))     # every listed companion gets its anchor cutout built
                if _drawn is None and (_c["rank"] == 1 or P27_STANDIN):
                    _s = _source27(_c["anchor_snap"], _c["gal_id"])
                    if _s is not None:
                        _drawn, _src = _c, _s
            _best = _cands[0]
            _cell = dict(sid=_sid, best=_best, drawn=_drawn, src=_src, standin=bool(_drawn is not None and _drawn["rank"] > 1))
            if _src is not None:
                try:
                    _rr = _RR27(_src[3], members_only=P27_MEMBERS, frame=P27_FRAME, frame_rmax=P27_FRAME_RMAX, frame_members=True)
                    _E = _rr.evecs.copy(); _finfo = dict(_rr.frame_info)
                    _scan = []                                            # (i, apparent q) of this companion, face-on to edge-on
                    for _ii in np.arange(0.0, 90.0 + 0.5 * P27_INC_STEP, P27_INC_STEP):
                        _rr.set_evecs(_view27(_E, _ii, _r["pa_deg"], P27_AZIMUTH), label="scan")
                        _scan.append((float(_ii), _apparent27(_rr, _EX27)[0]))
                    _scan = np.array(_scan); _cell["q_faceon"], _cell["q_edgeon"] = float(_scan[0, 1]), float(_scan[-1, 1])
                    _inc = float(_r["inc_deg"])
                    if P27_INC == "match_q" and np.isfinite(_r.get("q_opt", np.nan)):
                        _ok = np.isfinite(_scan[:, 1])
                        _inc = float(_scan[_ok][np.argmin(np.abs(_scan[_ok, 1] - float(_r["q_opt"])))][0]) if _ok.any() else _inc
                    _cell["inc_used"] = _inc
                    _V = _view27(_E, _inc, _r["pa_deg"], P27_AZIMUTH)
                    _rr.set_evecs(_V, label=f"i={_inc:.0f} PA={_r['pa_deg']:.0f}", info=_finfo)
                    _kpc = _kpc_as27(_r["z"])
                    _bm = (float(_r.get("mom0_bmaj_as", np.nan)), float(_r.get("mom0_bmin_as", np.nan)))
                    _fwhm_kpc = np.sqrt(_bm[0] * _bm[1]) * _kpc if np.all(np.isfinite(_bm)) else np.nan
                    _cell["beam_kpc"] = (_bm[0] * _kpc, _bm[1] * _kpc, float(_r.get("mom0_bpa_deg", np.nan))) if np.isfinite(_fwhm_kpc) else None

                    def _maps27(tag):
                        """log Sigma_* [Msun/pc^2] and the (beam-convolved) Sigma_dust / Sigma_H2 maps of the current frame."""
                        from scipy.ndimage import gaussian_filter as _gf27
                        out = {}
                        out[f"sig_star{tag}"] = _rr.sigma_map("star", ex=_EX27, spos="faceon", xsize=P27_XSIZE)[0] / 1e6
                        if "gas" in _rr.parts and "m_dust" in _rr.parts["gas"]:
                            d = _rr.sigma_map("dust", ex=_EX27, spos="faceon", xsize=P27_XSIZE)[0] / 1e6
                            h = _rr.sigma_map("H2", ex=_EX27, spos="faceon", xsize=P27_XSIZE)[0] / 1e6 * P27_HE
                            if P27_BEAM and np.isfinite(_fwhm_kpc):
                                sp = _fwhm_kpc / 2.3548 / (2 * _EX27 / P27_XSIZE)
                                d, h = _gf27(d, sp), _gf27(h, sp)
                            out[f"sig_dust{tag}"], out[f"sig_H2{tag}"] = d, h
                        return out

                    if P27_STYLE == "rgb":
                        _cell["rgb"] = _rr.rgb(ex=_EX27, spos="faceon", channels=P27_CHAN, dyn=P27_DYN, pmax=P27_PMAX, stretch=P27_STRETCH, xsize=P27_XSIZE)[0]
                    else:
                        _cell.update(_maps27(""))
                        _cell["peak_H2"] = float(np.nanmax(_cell["sig_H2"])) if "sig_H2" in _cell else np.nan
                        _cell["peak_dust"] = float(np.nanmax(_cell["sig_dust"])) if "sig_dust" in _cell else np.nan
                        if P27_SIM_LEVELS == "peak":
                            _cell["levels"] = ([f * _cell["peak_H2"] for f in P27_SIM_FRAC], [f * _cell["peak_dust"] for f in P27_SIM_FRAC], "peak")
                        else:
                            _cell["levels"] = _simlevels27(_r)
                    _cell["q_app"], _cell["pa_app"], _cell["r50_app"] = _apparent27(_rr, _EX27)
                    _cell["tot"] = _rr.totals(ex=_EX27, spos="faceon"); _cell["z_drawn"] = float(_rr.redshift); _cell["frame"] = _finfo
                    if P27_FACEON_ROW:
                        _rr.set_evecs(_E, label="faceon")
                        _cell["rgb_face"] = _rr.rgb(ex=_EX27, spos="faceon", channels=P27_FACE_CHAN, dyn=P27_FACE_DYN, pmax=P27_PMAX, stretch=P27_STRETCH, xsize=P27_XSIZE)[0]
                        _cell["q_face"] = _apparent27(_rr, _EX27)[0]
                except Exception as _e27:
                    print(f"  {_sid}: render failed on {_src[3]}: {type(_e27).__name__} {str(_e27)[:80]}")
                    _cell["src"] = None; _cell["drawn"] = None; _cell["standin"] = False
            REN27[_sid] = _cell
            for _c in _cands:
                _isd = _cell.get("drawn") is _c and _cell.get("src") is not None
                _sd = _cell["src"] if _isd else None
                ROWS27.append(dict(id=_sid, z_obs=float(_r["z"]), inc_deg=float(_r["inc_deg"]), inc_src=_r["inc_src"], pa_deg=float(_r["pa_deg"]),
                                   pa_src=_r["pa_src"], q_obs=float(_r.get("q_opt", np.nan)), rank=_c["rank"], dist=_c["dist"], axes=_c["axes"],
                                   anchor_snap=int(_c["anchor_snap"]), gal_id=int(_c["gal_id"]), cls=_c["cls"], z_sim=float(_c["z"]), age=float(_c["age"]),
                                   lfd=float(_c["lfd"]), lfh2=float(_c["lfh2"]), extent=float(_c["extent"]), lm=float(_c["lm"]),
                                   obs_age=float(_r["age"]), obs_lfd=float(_r["lfd"]), obs_lfh2=float(_r["lfh2"]), obs_extent=float(_r["extent"]), obs_lm=float(_r["lm"]),
                                   drawn=_isd, data=(_sd[0] if _sd else ""), data_snap=(_sd[1] if _sd else -1), data_gal=(_sd[2] if _sd else -1),
                                   data_dt_gyr=(_sd[4] if _sd else np.nan), inc_used=(_cell.get("inc_used", np.nan) if _isd else np.nan),
                                   q_app=(_cell.get("q_app", np.nan) if _isd else np.nan), pa_app=(_cell.get("pa_app", np.nan) if _isd else np.nan),
                                   q_faceon=(_cell.get("q_faceon", np.nan) if _isd else np.nan), q_edgeon=(_cell.get("q_edgeon", np.nan) if _isd else np.nan),
                                   obs_rms_mom0=float(_r.get("mom0_rms_jybkms", np.nan)), obs_rms_cont=float(_r.get("cont_rms_jyb", np.nan)),
                                   obs_3sig_h2_pc2=(3.0 * float(_r.get("mom0_rms_jybkms", np.nan)) * _obsconv27(_r)[0] if _isd else np.nan),
                                   obs_3sig_dust_pc2=(3.0 * float(_r.get("cont_rms_jyb", np.nan)) * _obsconv27(_r)[1] if _isd else np.nan),
                                   sim_peak_h2_pc2=(_cell.get("peak_H2", np.nan) if _isd else np.nan), sim_peak_dust_pc2=(_cell.get("peak_dust", np.nan) if _isd else np.nan)))

        # the report
        print(f"\n  the closest companions (distance in tolerance units over {'/'.join(P27_AXES)}; tolerances "
              + ", ".join(f"{k} {v:g}" for k, v in P27_AXES.items()) + "):")
        print("  source  | closest match: snap  gal   class                   d    age   lfd   lfh2  extent  lM* | drawn from")
        for _sid, _cell in REN27.items():
            _b = _cell["best"]; _o = O27.loc[_sid]; _sd = _cell.get("src")
            if _sd is None:
                _what = "— (no particle data on disk)"
            elif _sd[0] == "reduced":
                _what = "the anchor cutout"
            else:
                _what = f"the branch RT cutout snap {_sd[1]} gal {_sd[2]}" + (f" (dt {_sd[4]:+.2f} Gyr)" if np.isfinite(_sd[4]) else "")
            if _cell["standin"]:
                _dd = _cell["drawn"]
                _what += f"  STAND-IN: rank {_dd['rank']} snap {int(_dd['snap'])} gal {int(_dd['gal_id'])} ({_short27(_dd['cls'])}, d {_dd['dist']:.2f})"
            print(f"  {_sid:7s} | {int(_b['snap']):4d} {int(_b['gal_id']):5d}  {_short27(_b['cls']):22s} {_b['dist']:4.2f}  "
                  f"{_b['age']:4.1f} {_b['lfd']:5.2f} {_b['lfh2']:5.2f} {_b['extent']:5.2f} {_b['lm']:5.2f} | {_what}")
            print(f"          | observed:                                    {_o['age']:4.1f} {_o['lfd']:5.2f} {_o['lfh2']:5.2f} {_o['extent']:5.2f} {_o['lm']:5.2f} |"
                  + (f" i {_cell['inc_used']:.0f}: apparent q {_cell['q_app']:.2f} (obs {_o.get('q_opt', np.nan):.2f}; face-on {_cell['q_faceon']:.2f} "
                     f"-> edge-on {_cell['q_edgeon']:.2f}), PA {_cell['pa_app']:.0f} (asked {_o['pa_deg']:.0f})"
                     if np.isfinite(_cell.get("q_app", np.nan)) else ""))
        if P27_STYLE != "rgb":
            _lvtxt = (f"the simulation at {', '.join(f'{100 * f:g}' for f in P27_SIM_FRAC)} % of each map's own peak" if P27_SIM_LEVELS == "peak" else
                      "the simulation at the same Msun/pc^2 through each source's M_H2/S_CO and M_dust/S_nu and beam" if P27_SIM_LEVELS == "obs" else
                      f"the simulation at {P27_SIM_LEVELS} Msun/pc^2")
            print(f"\n  the contour levels (P27_OBS_LEVELS {P27_OBS_LEVELS} x rms on the data; {_lvtxt}{'; beam-smoothed' if P27_BEAM else ''}). "
                  f"For reference, the data's 3 sigma in Msun/pc^2 (through M_H2/S_CO, M_dust/S_nu and the beam) against the companion's peak:")
            print("  source   rms mom0 [Jy/b km/s]  rms cont [uJy/b]   3 sigma = Sigma_H2 / Sigma_dust [Msun/pc^2]   companion's peak Sigma_H2 / Sigma_dust")
            for _sid, _cell in REN27.items():
                _o = O27.loc[_sid]; _cH2, _cD = _obsconv27(_o)
                _l3h = 3.0 * float(_o.get("mom0_rms_jybkms", np.nan)) * _cH2; _l3d = 3.0 * float(_o.get("cont_rms_jyb", np.nan)) * 1e6 * _cD / 1e6
                _pk = (f"{_cell['peak_H2']:8.1f} / {_cell['peak_dust']:6.2f}" + ("   above 3 sigma" if _cell["peak_H2"] > _l3h else "   BELOW 3 sigma (would not be detected)")
                       if np.isfinite(_cell.get("peak_H2", np.nan)) and np.isfinite(_l3h) else (f"{_cell['peak_H2']:8.1f} / {_cell['peak_dust']:6.2f}" if np.isfinite(_cell.get("peak_H2", np.nan)) else "—"))
                print(f"  {_sid:7s}  {float(_o.get('mom0_rms_jybkms', np.nan)):8.3f}            {1e6 * float(_o.get('cont_rms_jyb', np.nan)):6.1f}        "
                      f"{_l3h:8.1f} / {_l3d:6.2f}                          {_pk}")
        _flat = [(s, c) for s, c in REN27.items() if np.isfinite(c.get("q_edgeon", np.nan)) and np.isfinite(O27.loc[s].get("q_opt", np.nan))
                 and c["q_edgeon"] > float(O27.loc[s]["q_opt"])]
        if _flat:
            print(f"  {len(_flat)} of {sum(1 for c in REN27.values() if np.isfinite(c.get('q_edgeon', np.nan)))} companions cannot look as flat as their source "
                  f"from ANY direction (edge-on q above the observed q): " + ", ".join(f"{s} {c['q_edgeon']:.2f} vs {float(O27.loc[s]['q_opt']):.2f}" for s, c in _flat)
                  + " — the simulated quenched bodies are rounder than the observed ones.")

        _low = [s for s, r in O27.iterrows() if P27_POOL == "regime" and float(r["lm"]) < P26_LOGM]
        if _low:
            print(f"  NOTE {', '.join(_low)}: log M* below the regime cut P26_LOGM = {P26_LOGM:g}, so the regime pool has no mass counterpart "
                  f"(the distance carries it); P27_POOL = 'zmax' searches every mass at z <= {P26_ZMAX:g}.")

        # the plan for build_reduced_particles_job.py: every listed companion whose anchor cutout is not on disk
        _plan27 = sorted(set(_plan27))
        if _plan27:
            os.makedirs(os.path.dirname(P27_PLAN), exist_ok=True)
            with _h5_27.File(P27_PLAN, "w") as _pf:
                _pf.attrs["note"] = "agn_states_dusty_vs_pure.ipynb Part 27: the ALMA-C11 companions (closest + runner-ups) without an anchor cutout"
                _pf.attrs["sim_name"] = P27_BOX
                _pf.create_dataset("entry_snap", data=np.array([s for s, g in _plan27], np.int32))
                _pf.create_dataset("entry_gx", data=np.array([g for s, g in _plan27], np.int64))
            _ns = len(set(s for s, g in _plan27))
            print(f"\n  plan: {len(_plan27)} (snap, galaxy) pairs over {_ns} snapshot(s) without an anchor cutout -> {os.path.relpath(P27_PLAN, os.getcwd())}\n"
                  f"        sbatch --array=0-{_ns - 1} submit_reduced_particles.sh {os.path.relpath(P27_PLAN, os.getcwd())}\n"
                  f"        then re-run this part: the anchor cutouts replace the branch cutouts and the stand-ins.")

        # ── the figure ──
        from matplotlib.patches import Ellipse as _Ell27
        _sids = [s for s in REN27 if REN27[s].get("src") is not None]
        if not _sids:
            print("  nothing to draw yet (no particle data on disk for any companion): build the plan's cutouts, or point P27_RT_DIR at the RT cutouts.")
        else:
            _nrow_b = int(bool(_STAMPS27) and P27_OBS_ROW) + 1 + int(P27_FACEON_ROW and any("rgb_face" in REN27[s] for s in _sids))
            _blocks = [_sids[i:i + P27_NCOL] for i in range(0, len(_sids), P27_NCOL)]
            _ncol = min(P27_NCOL, len(_sids))
            fig, axs = plt.subplots(_nrow_b * len(_blocks), _ncol, figsize=(2.7 * _ncol + 0.3, 2.7 * _nrow_b * len(_blocks) + 0.9), squeeze=False)
            fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.93, wspace=0.03, hspace=0.05 if _nrow_b == 1 else 0.04)
            _bar = 5.0 if _EX27 <= 25 else 10.0
            _txt = lambda ax, x, y, s, **kw: ax.text(x, y, s, transform=ax.transAxes, fontsize=_fs27(kw.pop("fs", 8)), color=kw.pop("color", "w"),
                                                     path_effects=[_pe27.withStroke(linewidth=2.2, foreground="k")], zorder=12, **kw)
            _starcmap = _RR27.CHANNEL_CMAPS["star"] if _RR27 is not None else "gray"

            def _beam27(ax, beam_kpc, color="w"):
                """The ALMA beam (FWHM) at the upper right of a panel: (bmaj, bmin) kpc, PA East of North."""
                if not beam_kpc or not np.all(np.isfinite(beam_kpc[:2])):
                    return
                cx, cy = _EX27 * 0.78, _EX27 * 0.62
                ax.add_patch(_Ell27((cx, cy), beam_kpc[1], beam_kpc[0], angle=90.0 + (beam_kpc[2] if np.isfinite(beam_kpc[2]) else 0.0),
                                    facecolor="none", edgecolor=color, lw=1.0, ls="-", zorder=11, hatch="////", alpha=0.9))

            def _obs_contours27(ax, sid, o):
                """The imaging pipeline's continuum (dust) and CO moment-0 (H2) contours on the stamp, at P27_OBS_LEVELS x rms."""
                a = _ALMA27.get(sid)
                if a is None:
                    return False
                k = _kpc_as27(o["z"])
                for key, rms_key, col in (("cont", "cont_rms_jyb", P27_COL["dust"]), ("mom0", "mom0_rms_jybkms", P27_COL["H2"])):
                    m = a.get(key); rms = float(o.get(rms_key, np.nan)); sc = float(a.get(f"{key}_pix_arcsec"))
                    if m is None or not np.isfinite(rms) or rms <= 0:
                        continue
                    n = m.shape[0]; h = (n - 1) / 2.0
                    x = (float(a[f"{key}_x0_as"]) + (np.arange(n) - h) * sc) * k; y = (float(a[f"{key}_y0_as"]) + (np.arange(n) - h) * sc) * k
                    X, Y = np.meshgrid(x, y)
                    lv = [float(l) * rms for l in P27_OBS_LEVELS]
                    mm = np.where(np.isfinite(m), m, 0.0)
                    if np.nanmax(mm) > lv[0]:
                        ax.contour(X, Y, mm, levels=lv, colors=col, linewidths=0.9, zorder=8)
                    if np.nanmin(mm) < -lv[0]:
                        ax.contour(X, Y, mm, levels=[-lv[0]], colors=col, linewidths=0.7, linestyles="dashed", zorder=8)
                return True

            def _sim_contours27(ax, cell, tag=""):
                """The companion's own Sigma_dust / Sigma_H2 contours at the source's levels [Msun/pc^2]."""
                if f"sig_H2{tag}" not in cell or not cell.get("levels"):
                    return
                lH2, lD, how = cell["levels"]
                px = 2 * _EX27 / P27_XSIZE
                x = np.linspace(-_EX27 + 0.5 * px, _EX27 - 0.5 * px, P27_XSIZE); X, Y = np.meshgrid(x, x)
                for key, lv, col in ((f"sig_dust{tag}", lD, P27_COL["dust"]), (f"sig_H2{tag}", lH2, P27_COL["H2"])):
                    m = np.nan_to_num(np.asarray(cell[key], float)); lv = [l for l in lv if np.isfinite(l) and l > 0]
                    if lv and np.nanmax(m) > lv[0]:
                        ax.contour(X, Y, m, levels=lv, colors=col, linewidths=0.9, zorder=8)

            def _sim_panel27(ax, cell, tag=""):
                """The companion: log Sigma_* (the stars) with the contours, or the rgb composite."""
                if P27_STYLE == "rgb":
                    ax.imshow(cell[f"rgb{tag}"], origin="lower", extent=[-_EX27, _EX27, -_EX27, _EX27], interpolation="bilinear")
                    return
                m = np.asarray(cell[f"sig_star{tag}"], float); pos = m[np.isfinite(m) & (m > 0)]
                vmax = np.log10(np.percentile(pos, 99.9)) if len(pos) else 0.0
                ax.imshow(np.log10(np.where(m > 0, m, np.nan)), origin="lower", extent=[-_EX27, _EX27, -_EX27, _EX27], cmap=_starcmap,
                          vmin=vmax - P27_STAR_DYN, vmax=vmax, interpolation="bilinear")
                _sim_contours27(ax, cell, tag)
                if P27_BEAM:
                    _beam27(ax, cell.get("beam_kpc"))

            for _bi, _blk in enumerate(_blocks):
                for _ci in range(_ncol):
                    _r0 = _bi * _nrow_b
                    for _k in range(_nrow_b):
                        axs[_r0 + _k][_ci].set_xticks([]); axs[_r0 + _k][_ci].set_yticks([])
                        axs[_r0 + _k][_ci].set_facecolor("k")
                    if _ci >= len(_blk):
                        for _k in range(_nrow_b):
                            axs[_r0 + _k][_ci].axis("off")
                        continue
                    _sid = _blk[_ci]; _cell = REN27[_sid]; _o = O27.loc[_sid]; _row = _r0
                    if _STAMPS27 and P27_OBS_ROW:
                        ax = axs[_row][_ci]; _row += 1
                        if _sid in _STAMPS27:
                            _st = np.asarray(_STAMPS27[_sid], float)
                            _kpc_as = _kpc_as27(_o["z"])
                            _hpx = int(round(_EX27 / _kpc_as / _PIX27)); _c0 = _st.shape[0] // 2
                            _sub = _st[max(0, _c0 - _hpx):_c0 + _hpx, max(0, _c0 - _hpx):_c0 + _hpx]
                            _bg = np.nanmedian(_st[np.abs(np.arange(_st.shape[0]) - _c0) > 0.8 * _c0][:, :]) if _st.shape[0] > 8 else 0.0
                            _im = _sub - _bg; _vmax = np.nanpercentile(_im, 99.7) if np.isfinite(_im).any() else 1.0
                            _im = np.arcsinh(np.clip(_im, 0, None) / (0.03 * max(_vmax, 1e-12))) / np.arcsinh(1 / 0.03)
                            _hk = _sub.shape[1] * _PIX27 * _kpc_as / 2.0, _sub.shape[0] * _PIX27 * _kpc_as / 2.0
                            ax.imshow(np.clip(_im, 0, 1), origin="lower", cmap=_starcmap, vmin=0, vmax=1, extent=[-_hk[0], _hk[0], -_hk[1], _hk[1]], interpolation="nearest")
                            ax.set_xlim(-_EX27, _EX27); ax.set_ylim(-_EX27, _EX27)
                            _txt(ax, 0.03, 0.96, f"{_sid}   $z = {_o['z']:.2f}$", va="top", ha="left", fs=9.5, fontweight="bold")
                            _has = P27_STYLE != "rgb" and _obs_contours27(ax, _sid, _o)
                            if _has:
                                _beam27(ax, (float(_o.get("mom0_bmaj_as", np.nan)) * _kpc_as, float(_o.get("mom0_bmin_as", np.nan)) * _kpc_as, float(_o.get("mom0_bpa_deg", np.nan))))
                                _txt(ax, 0.03, 0.125, f"contours: dust ({P27_OBS_LEVELS[0]:g}, {P27_OBS_LEVELS[1]:g}, ... $\\sigma$), CO(3–2)", va="bottom", ha="left", fs=6.8)
                            _txt(ax, 0.03, 0.05, f"ACS F814W   $q = {_o.get('q_opt', np.nan):.2f}$, PA {_o['pa_deg']:.0f}°", va="bottom", ha="left", fs=7.5)
                            _txt(ax, 0.97, 0.96, "N↑ E←", va="top", ha="right", fs=7.5)
                        else:
                            _txt(ax, 0.5, 0.5, f"{_sid}: no stamp", ha="center", va="center")
                        for _sp in ax.spines.values():
                            _sp.set_color("0.5"); _sp.set_linewidth(1.2)
                    # the companion at the observed geometry
                    ax = axs[_row][_ci]; _row += 1
                    _sim_panel27(ax, _cell)
                    _d = _cell["drawn"]; _col = _CLS27.get(_d["cls"], ("0.6", ""))[0]
                    for _sp in ax.spines.values():
                        _sp.set_color(_col); _sp.set_linewidth(2.2); _sp.set_linestyle("--" if _cell["standin"] else "-")
                    _src = _cell["src"]
                    _hdr = (f"{_sid} → snap {int(_d['snap'])} gal {int(_d['gal_id'])}" + ("  (stand-in)" if _cell["standin"] else ""))
                    _txt(ax, 0.03, 0.96, _hdr, va="top", ha="left", fs=8.5, fontweight="bold")
                    _txt(ax, 0.03, 0.885, f"{_short27(_d['cls'])}   $d = {_d['dist']:.2f}$", va="top", ha="left", fs=7.5, color=_col)
                    _src_lab = (f"$z = {_cell['z_drawn']:.2f}$ (anchor)" if _src[0] == "reduced" else
                                f"$z = {_cell['z_drawn']:.2f}$, branch cutout snap {_src[1]} ({_src[4]:+.2f} Gyr)" if np.isfinite(_src[4]) else f"$z = {_cell['z_drawn']:.2f}$, branch cutout snap {_src[1]}")
                    _txt(ax, 0.03, 0.125, _src_lab, va="bottom", ha="left", fs=6.8)
                    _txt(ax, 0.03, 0.05, f"$i = {_cell.get('inc_used', _o['inc_deg']):.0f}°$, PA {_o['pa_deg']:.0f}°   apparent $q = {_cell.get('q_app', np.nan):.2f}$"
                         + (f"  (edge-on {_cell['q_edgeon']:.2f})" if np.isfinite(_cell.get("q_edgeon", np.nan)) else ""), va="bottom", ha="left", fs=7.2)
                    if P27_STYLE != "rgb" and _cell.get("levels"):
                        _lh, _ld, _how = _cell["levels"]
                        _txt(ax, 0.03, 0.20, (f"peak $\\Sigma_{{\\rm H_2}}$ {_cell['peak_H2']:.3g}, $\\Sigma_{{\\rm d}}$ {_cell['peak_dust']:.2g} M$_\\odot$ pc$^{{-2}}$" if _how == "peak" else
                                              f"contours from $\\Sigma_{{\\rm H_2}}$ {_lh[0]:.3g}, $\\Sigma_{{\\rm d}}$ {_ld[0]:.2g} M$_\\odot$ pc$^{{-2}}$"
                                              + ("" if _how == "obs" else f" ({_how})")), va="bottom", ha="left", fs=6.3)
                    if _ci == 0:                                          # the scale bar: left, between the header and the bottom lines
                        _f = _bar / (2 * _EX27)
                        ax.plot([0.04, 0.04 + _f], [0.78, 0.78], "-", transform=ax.transAxes, color="w", lw=2.4, zorder=12,
                                path_effects=[_pe27.withStroke(linewidth=4, foreground="k")])
                        _txt(ax, 0.06 + _f, 0.78, f"{_bar:g} kpc", ha="left", va="center", fs=7)
                    if P27_FACEON_ROW and "rgb_face" in _cell:
                        ax = axs[_row][_ci]; _row += 1
                        ax.imshow(_cell["rgb_face"], origin="lower", extent=[-_EX27, _EX27, -_EX27, _EX27], interpolation="bilinear")
                        for _sp in ax.spines.values():
                            _sp.set_color(_col); _sp.set_linewidth(2.2); _sp.set_linestyle("--" if _cell["standin"] else "-")
                        _chn = {"gas": "gas (blue)", "H2": "H$_2$ (green)", "dust": "dust (orange)", "star": "stars (white)", "HI": "HI (blue)", "sfr": "SFR (pink)"}
                        _txt(ax, 0.03, 0.96, "face-on: " + " + ".join(_chn.get(c, c) for c in P27_FACE_CHAN), va="top", ha="left", fs=7.5)
                        _txt(ax, 0.03, 0.05, f"stellar apparent $q = {_cell.get('q_face', np.nan):.2f}$", va="bottom", ha="left", fs=7.5)
            _nst = sum(1 for s in _sids if REN27[s]["standin"]); _nrt = sum(1 for s in _sids if REN27[s]["src"][0] == "rt")
            _simlv = (f"its own $\\Sigma_{{\\rm H_2}}$ / $\\Sigma_{{\\rm dust}}$ at {', '.join(f'{100 * f:g}' for f in P27_SIM_FRAC)} % of the peak" if P27_SIM_LEVELS == "peak" else
                      "the same M$_\\odot$ pc$^{{-2}}$ as the data" if P27_SIM_LEVELS == "obs" else "fixed M$_\\odot$ pc$^{{-2}}$ levels")
            _what = ("stars (white), gas (blue), dust (orange)" if P27_STYLE == "rgb" else
                     f"stars (ACS F814W / log $\\Sigma_*$) with dust (orange) and H$_2$ (blue) contours — data: {P27_OBS_LEVELS[0]:g}, {P27_OBS_LEVELS[1]:g}, ... $\\sigma$ of the "
                     f"continuum and CO(3–2) maps; simulation: {_simlv}" + (", beam-smoothed" if P27_BEAM else ""))
            fig.suptitle(f"The closest {BOX_LABEL.get(P27_BOX, P27_BOX)} companion of each ALMA-C11 source, seen at the source's inclination and position angle "
                         f"(North up, East left; {P27_SIZE:g} kpc a side)\n{_what}\nframe = {P27_FRAME}; edge colour = the dose × extent class, dashed = a stand-in"
                         + (f" ({_nst})" if _nst else "") + (f"; {_nrt} drawn from branch RT cutouts, not the anchor" if _nrt else ""),
                         fontsize=_fs27(9.5), y=1.0 - 0.12 / fig.get_size_inches()[1])
            fig.subplots_adjust(top=1.0 - 0.95 / fig.get_size_inches()[1])
            paper_save(fig, "agn_states_companions" + _tag27)
            plt.show()
        pd.DataFrame(ROWS27).to_csv(os.path.join(OUT, f"agn_states_companions{_tag27}.csv"), index=False)
        print(f"  written: agn_states_companions{_tag27}.csv ({len(ROWS27)} rows: every source's closest match and runner-ups, what was drawn, the apparent q and PA, "
              f"the observed 3 sigma levels and the companion's peak surface densities)")


## Part 28 — the dust–age plane of every run against the data, and the exposure behind each box's high / low dose split

Two things the thesis needs side by side, drawn from the same anchor rows of Part 0's sample.

**Left — $\log M_{\rm dust}/M_\star$ against the stellar age**, for **m100, m50, m25 and the channel-off m50 run** (`P28_NOX`), each as two population tracks (Part 26's estimator: the running median over the `P28_KFRAC` nearest galaxies in age, with its 16–84 band): the **dusty** galaxies ($\log M_{\rm dust}/M_\star \geq$ `DUSTY_CUT`, solid) and the **dust-poor** ones (dashed). The ALMA-C11 quiescent galaxies sit on the plane with their errors (dust upper limits arrowed), and the m100 galaxies that fall inside Part 26's observed box are marked and numbered as there (`P28_INBOX`; `INBOX26` when Part 26 has run, otherwise the same rule re-applied to the anchor rows). The split being on the $y$ quantity itself, the two tracks of a run are separated by construction — what the panel shows is *where* each class sits in age, how far apart the classes are in each box, that the channel-off run has essentially no dust-poor class, and which class the data lie on. Note the age axes are not the same quantity on the two sides (the notes: the simulated age is mass-weighted, the observed one is CIGALE's onset age, an upper bound).

**Right — one panel per box with the X-ray channel**: the distribution of the exposure Parts 22–27 split on (`P28_DOSE` = the X-ray duty fraction since SFT, from Part 22's engine run over *every* anchor row), the dusty and dust-poor galaxies drawn separately, the background shaded on either side of the high / low cut (`P28_DCUT` = Part 22's cut of the box, so the halves are the same as in Parts 22–27), and in the two bottom corners **the share of the selected sample** (Part 0: the quenched rule, $\log M_\star >$ `MASS_MIN`, its $z$ / age cuts) that is dusty / dust-poor at low / high dose — `P28_DENOM` "sample" (the four sum to the chain coverage) or "dosed" (they sum to 100 %). **These are anchor rows, not Part 22's de-duplicated galaxies**: a galaxy quenched at several anchors counts once per anchor on both sides of every share, which is why the counts here exceed Part 22's. `P28_LAYOUT = "rows"` gives one dust–age panel per box beside its dose panel instead of the single plane. Products `agn_states_fdust_age_dose.{png,pdf,csv}` (the shares) and `agn_states_fdust_age_tracks.csv` (the tracks).

In [ ]:
# ── Part 28 — THE DUST-AGE PLANE OF THE FOUR RUNS AGAINST THE DATA, and the dose distribution behind every high / low split ──
# LEFT   log M_dust/M* against the mass-weighted stellar age for m100 / m50 / m25 AND the channel-off m50 run, each as TWO
#        population tracks (Part 26's running median with its 16-84 band): the DUSTY galaxies (log M_dust/M* >= DUSTY_CUT) and
#        the DUST-POOR ones, over Part 0's sample; the ALMA-C11 quiescent galaxies with their errors (dust upper limits arrowed)
#        and the m100 galaxies that fall inside Part 26's observed box, numbered as there.
# RIGHT  one panel per box WITH the X-ray channel: the distribution of the exposure Parts 22-27 split on (P28_DOSE = the X-ray
#        duty fraction since SFT), dusty and dust-poor drawn separately, the background shaded on either side of the high / low
#        cut, and in the bottom corners the share of THE SELECTED SAMPLE (Part 0: the quenched rule, log M* > MASS_MIN and its
#        z / age cuts) that is dusty / dust-poor at low / high dose.
# THE FRAME IS ANCHOR ROWS, not Part 22's de-duplicated galaxies: a galaxy quenched at several anchors counts once per anchor,
# on both sides of every fraction, so the counts here exceed Part 22's (cis100 1683 rows -> 1625 galaxies there).
P28_BOXES  = ["cis100", "cis50", "cis25"]      # the dose panels, top to bottom; the same boxes get their two tracks on the left
P28_NOX    = NOX_BOX                            # the channel-off run: two tracks on the left, no dose panel (None = drop it)
P28_LAYOUT = "single"                           # "single" = ONE dust-age panel with every run | "rows" = one dust-age panel per box beside its dose panel (the nox run joins the m50 row)
P28_SEL    = "sample"                           # the galaxies behind the tracks: "sample" = Part 0's QM | "regime" = Part 26's observed regime (z <= P26_ZMAX, log M* >= P26_LOGM)
P28_KFRAC, P28_NMIN, P28_NGRID = 0.30, 10, 40   # the running median: the fraction of the class inside the window (never fewer than P28_NMIN), the track points
P28_BAND   = True                               # the 16-84 dispersion band under each track
P28_DOSE   = globals().get("P22_DOSE", "f_x_post")   # the exposure axis of the dose panels — the one Parts 22-27 split on: "f_x_post" | "f_x" | "e_x" [Gyr]
P28_DCUT   = "part22"                           # the high / low split: "part22" = Part 22's cut of the box (the median of ITS frame; this frame's median where Part 22 skipped the box) | "median" = this frame's own median | a number
P28_DENOM  = "sample"                           # the percentages are of: "sample" = EVERY anchor row of Part 0's sample in the box (the four sum to the chain coverage) | "dosed" = the rows with a dose (they sum to 100)
P28_NBINS  = 20                                 # histogram bins over the dose range
P28_LOGY   = False                              # log counts (a duty fraction piles up at 0 and at 1)
P28_INBOX  = "cis100"                           # the box whose galaxies inside Part 26's observed box are marked one by one (INBOX26 when Part 26 ran, else recomputed here with Part 26's rule); None = none
P28_OBS    = os.path.join(os.getcwd(), "obs_data", "almac11", "age_sersic_sigma.csv")   # the ALMA-C11 sample (kind == main): age, log M_dust/M* and their errors
P28_ULIM   = 0.3                                # [dex] the arrow of a dust upper limit
P28_AGE_LIM, P28_LFD_LIM = None, (-7.2, -1.8)   # the axes (None = from the tracks and the data, padded)
BOXC28 = {"cis100": "#252525", "cis50": "#e6ab02", "cis25": "#1b9e77", "cis50nox": "#7570b3"}   # one hue per run; dusty solid, dust-poor dashed
BOXS28 = {"cis100": "m100", "cis50": "m50", "cis25": "m25", "cis50nox": "m50, X-ray off"}
LOWC28, HIGHC28 = "#b2182b", "#2166ac"          # Part 22's low / high dose colours: the shading of the dose panels
DLAB28 = globals().get("P22_DLAB", {"f_x_post": "X-ray duty fraction since SFT", "f_x": "X-ray duty fraction", "e_x": r"$E_x$  [Gyr]"})

if "QM" not in globals():
    print("Part 28: needs Part 0 — run it first.")
elif "_chains22" not in globals() or "_dose22" not in globals():
    print("Part 28: needs Part 22 — its chain reader and dose engine define the exposure the high / low split is made on. Run it first.")
else:
    def _track28(x, y, kfrac=P28_KFRAC, nmin=P28_NMIN, ngrid=P28_NGRID):
        """The population track of y along x (Part 26's estimator): at `ngrid` points between the 5th and 95th percentiles of x,
        the median and the 16-84 range of y over the `kfrac` nearest galaxies in x (never fewer than `nmin`). None below nmin."""
        x, y = np.asarray(x, float), np.asarray(y, float); ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
        if len(x) < nmin:
            return None
        k = int(min(max(int(round(kfrac * len(x))), nmin), len(x)))
        xg = np.linspace(np.percentile(x, 5), np.percentile(x, 95), ngrid)
        med, lo, hi = (np.full(ngrid, np.nan) for _ in range(3))
        for i, xc in enumerate(xg):
            v = y[np.argsort(np.abs(x - xc), kind="stable")[:k]]
            med[i], lo[i], hi[i] = np.median(v), np.percentile(v, 16), np.percentile(v, 84)
        return dict(x=xg, med=med, lo=lo, hi=hi, k=k, n=len(x))

    def _sel28(box):
        """The galaxies behind the tracks of one run: Part 0's sample, or Part 26's observed regime inside it."""
        Q = QM[QM["box"] == box]
        if P28_SEL == "regime":
            Q = Q[(Q["z"] <= globals().get("P26_ZMAX", 0.5) + 1e-9) & (Q["log_mstar"] >= globals().get("P26_LOGM", 10.8))]
        return Q

    # ── the dose of EVERY anchor row of Part 0's sample, box by box, with Part 22's engine (the m25 chains are Part 22's own) ──
    DOSE28, CUT28, FR28, PCT28, TRK28 = {}, {}, {}, [], []
    for _b in P28_BOXES:
        _Q = QM[QM["box"] == _b].copy()
        if not len(_Q):
            print(f"[{_b}] no rows in Part 0's sample — no dose panel"); continue
        if not os.path.exists(HIST_FILE.format(box=_b)):
            print(f"[{_b}] no histories cache ({HIST_FILE.format(box=_b)}) — no dose panel"); continue
        _keys = list(zip(_Q["snap"].astype(int), _Q["gal_id"].astype(int)))
        _CH = _CH22 if (_b == globals().get("P22_BOX") and "_CH22" in globals()) else _chains22(_b, want=set(_keys))[0]
        _has = np.array([k in _CH for k in _keys], bool)
        if _has.sum() < P28_NMIN:
            print(f"[{_b}] only {int(_has.sum())} of {len(_Q)} anchor rows have a main-branch chain — no dose panel"); continue
        _D = _dose22(_CH, [k for k, h in zip(_keys, _has) if h], _Q["t_sft"].to_numpy(float)[_has] if "t_sft" in _Q.columns else np.full(int(_has.sum()), np.nan))
        _D.insert(0, "gal_id", [k[1] for k, h in zip(_keys, _has) if h]); _D.insert(0, "snap", [k[0] for k, h in zip(_keys, _has) if h])
        DOSE28[_b] = _D
        _Q = _Q.merge(_D, on=["snap", "gal_id"], how="left")
        _dose = _Q[P28_DOSE].to_numpy(float)
        _fin = np.isfinite(_dose)
        if P28_DCUT == "part22" and "FIG1_22" in globals() and _b in FIG1_22:
            _dc, _src = float(FIG1_22[_b]["dcut"]), "Part 22's cut (the median of its de-duplicated frame)"
        elif P28_DCUT in ("part22", "median"):
            _dc, _src = float(np.nanmedian(_dose)), "the median of this frame"
        else:
            _dc, _src = float(P28_DCUT), "set by hand"
        _Q["hi_dose"] = np.where(_fin, _dose > _dc, np.nan)
        CUT28[_b] = _dc; FR28[_b] = _Q
        _dusty = (_Q["pop"].to_numpy() == "dusty")
        _n_all, _n_dosed = len(_Q), int(_fin.sum())
        _den = _n_all if P28_DENOM == "sample" else _n_dosed
        print(f"[{_b}] {_n_all} anchor rows in Part 0's sample, {_n_dosed} with a dose ({100 * _n_dosed / _n_all:.0f} %); "
              f"{DLAB28.get(P28_DOSE, P28_DOSE)} split at {_dc:.2f} = {_src}; percentages of {'the whole sample' if P28_DENOM == 'sample' else 'the dosed rows'} (N = {_den}):")
        for _p in ("dusty", "pure"):
            for _h, _hl in ((False, "low dose"), (True, "high dose")):
                _n = int((_fin & (_dusty == (_p == "dusty")) & ((_dose > _dc) == _h)).sum())
                PCT28.append(dict(box=_b, pop="dusty" if _p == "dusty" else "dust-poor", dose="high" if _h else "low", n=_n, pct=100 * _n / max(_den, 1),
                                  denominator=P28_DENOM, n_denominator=_den, n_sample=_n_all, n_dosed=_n_dosed, dose_axis=P28_DOSE, cut=_dc))
                print(f"     {'dusty' if _p == 'dusty' else 'dust-poor':9s} {_hl:9s}: {_n:5d}  = {100 * _n / max(_den, 1):5.1f} %")
        # why the dosed rows fall short of the sample, when they do: the galaxies without a chain sit at a different dust
        if _n_dosed < _n_all:
            _nc = ~_fin
            print(f"     the {int(_nc.sum())} rows without a dose: median log M_dust/M* {np.nanmedian(_Q.loc[_nc, 'lfd']):.2f} vs {np.nanmedian(_Q.loc[_fin, 'lfd']):.2f} dosed, "
                  f"{100 * np.mean(_dusty[_nc]):.0f} % dusty vs {100 * np.mean(_dusty[_fin]):.0f} % — the histories cover the boxes notebook's own quenched pool (Part 0 prints the coverage)")
    PCT28 = pd.DataFrame(PCT28)

    # ── the observed sample: the ALMA-C11 quiescent galaxies (the controls dropped), dust upper limits flagged ──
    OBS28 = pd.DataFrame()
    if os.path.exists(P28_OBS):
        _o = pd.read_csv(P28_OBS)
        if "kind" in _o.columns:
            _o = _o[_o["kind"].astype(str).str.lower() == "main"]
        OBS28 = pd.DataFrame(dict(id=_o["id"].astype(str), age=_o["age_gyr"].astype(float), age_err=_o["age_err_gyr"].astype(float),
                                  lfd=_o["log_fdust"].astype(float), lfd_err=_o["log_fdust_err"].astype(float),
                                  lfd_ul=_o["fdust_is_ul"].astype(bool) if "fdust_is_ul" in _o.columns else False,
                                  lm=_o["logMstar"].astype(float))).reset_index(drop=True)
        print(f"the observed sample: {len(OBS28)} ALMA-C11 quiescent galaxies, {int(OBS28['lfd_ul'].sum())} dust upper limits; age = CIGALE's onset age (an upper bound on the mass-weighted one, see the notes)")
    else:
        print(f"the observed sample: {P28_OBS} missing — tracks only")

    # ── the galaxies of one box inside Part 26's observed box, numbered as there ──
    IB28, _ibsrc = pd.DataFrame(), ""
    if P28_INBOX:
        if "INBOX26" in globals() and P28_INBOX in INBOX26 and len(INBOX26[P28_INBOX]):
            IB28 = INBOX26[P28_INBOX][[c for c in ("snap", "gal_id", "num", "age", "lfd", "lm", "z", "cls") if c in INBOX26[P28_INBOX].columns]].copy()
            _ibsrc = "Part 26's INBOX26 (de-duplicated galaxies)"
        elif P28_INBOX in set(QM["box"]):
            _ob = globals().get("P26_OBOX", dict(age=(3.5, 6.5), lfd=(-3.5, -2.5), lfh2=(-1.8, -0.8), extent=(-0.4, 0.15)))
            _zm, _lm = globals().get("P26_ZMAX", 0.5), globals().get("P26_LOGM", 10.8)
            _Qi = QM[QM["box"] == P28_INBOX].copy()
            with np.errstate(all="ignore"):
                _rg = _Qi["r_half_gas"].to_numpy(float) if "r_half_gas" in _Qi.columns else np.full(len(_Qi), np.nan)
                _rs = _Qi["r_half_star"].to_numpy(float)
                _Qi["extent"] = np.log10(np.where((_rg > 0) & (_rs > 0), _rg / np.where(_rs > 0, _rs, np.nan), np.nan))
                _Qi["lfh2_bare"] = _Qi["lfh2"].to_numpy(float) - np.log10(HE)          # Part 26's box is on the helium-free ratio
            _m = ((_Qi["z"] <= _zm + 1e-9) & (_Qi["log_mstar"] >= _lm) & _Qi["age"].between(*_ob["age"]) & _Qi["lfd"].between(*_ob["lfd"])
                  & _Qi["lfh2_bare"].between(*_ob["lfh2"]) & _Qi["extent"].between(*_ob["extent"]))
            IB28 = _Qi[_m].sort_values("age")[["snap", "gal_id", "age", "lfd", "log_mstar", "z"]].rename(columns={"log_mstar": "lm"}).reset_index(drop=True)
            IB28["num"] = np.arange(1, len(IB28) + 1)
            _ibsrc = "recomputed here with Part 26's rule on Part 0's anchor rows (a galaxy may repeat across anchors)"
        if len(IB28):
            print(f"the {len(IB28)} {BOXS28.get(P28_INBOX, P28_INBOX)} galaxies inside the observed box — {_ibsrc}: "
                  + ", ".join(f"#{int(r['num'])} snap {int(r['snap'])} gal {int(r['gal_id'])} ({r['age']:.1f} Gyr, {r['lfd']:+.2f})" for _, r in IB28.iterrows()))
        else:
            print(f"no {P28_INBOX} galaxy inside the observed box" + (f" ({_ibsrc})" if _ibsrc else " (Part 26 has not run and QM carries no gas radius)"))

    # ── the tracks: two per run ──
    _runs = [b for b in P28_BOXES if (QM["box"] == b).any()] + ([P28_NOX] if (P28_NOX and (QM["box"] == P28_NOX).any()) else [])
    TR28 = {}
    for _b in _runs:
        _T = _sel28(_b)
        for _p in ("dusty", "pure"):
            _s = _T[_T["pop"] == _p]
            _tr = _track28(_s["age"], _s["lfd"])
            TR28[(_b, _p)] = _tr
            if _tr is None:
                print(f"  [{_b}] {'dusty' if _p == 'dusty' else 'dust-poor'}: {len(_s)} galaxies — below P28_NMIN, no track")
            else:
                TRK28 += [dict(box=_b, pop="dusty" if _p == "dusty" else "dust-poor", age=float(x), lfd_med=float(m), lfd_p16=float(l), lfd_p84=float(h), n=_tr["n"], window=_tr["k"])
                          for x, m, l, h in zip(_tr["x"], _tr["med"], _tr["lo"], _tr["hi"])]
    _lab = lambda b, p: f"{BOXS28.get(b, b)} {'dusty' if p == 'dusty' else 'dust-poor'}  ({TR28[(b, p)]['n']})" if TR28.get((b, p)) else None

    def _plane28(ax, runs, obs=True, inbox=True, legend=True):
        """One dust-age panel: the two tracks of every run in `runs`, the dust cut, the observed points, the in-box galaxies."""
        _x0, _x1 = [], []
        for _b in runs:
            for _p, _ls, _lw in (("pure", "--", 2.0), ("dusty", "-", 2.6)):
                _tr = TR28.get((_b, _p))
                if _tr is None:
                    continue
                if P28_BAND:
                    ax.fill_between(_tr["x"], _tr["lo"], _tr["hi"], color=BOXC28.get(_b, "0.5"), alpha=0.10, lw=0)
                ax.plot(_tr["x"], _tr["med"], ls=_ls, lw=_lw, color=BOXC28.get(_b, "0.5"), label=_lab(_b, _p), zorder=3)
                _x0.append(_tr["x"][0]); _x1.append(_tr["x"][-1])
        ax.axhline(DUSTY_CUT, color="0.35", ls=":", lw=1.0, zorder=2)
        ax.text(0.99, DUSTY_CUT, "dusty above ", color="0.35", fontsize=7.5, ha="right", va="bottom", transform=ax.get_yaxis_transform())
        if inbox and len(IB28):
            ax.scatter(IB28["age"], IB28["lfd"], s=80, marker="D", facecolor="w", edgecolors=BOXC28.get(P28_INBOX, "k"), lw=1.4, zorder=6,
                       label=f"{BOXS28.get(P28_INBOX, P28_INBOX)} galaxies inside the observed box ({len(IB28)})")
            for _, r in IB28.iterrows():
                ax.annotate(str(int(r["num"])), (r["age"], r["lfd"]), xytext=(4, 4), textcoords="offset points", fontsize=7.5, fontweight="bold",
                            color=BOXC28.get(P28_INBOX, "k"), zorder=7)
        if obs and len(OBS28):
            _ul = OBS28["lfd_ul"].to_numpy(bool)
            _ey = np.where(_ul, P28_ULIM, np.nan_to_num(OBS28["lfd_err"].to_numpy(float)))
            ax.errorbar(OBS28["age"], OBS28["lfd"], xerr=np.nan_to_num(OBS28["age_err"].to_numpy(float)), yerr=_ey, uplims=_ul, fmt="none",
                        ecolor="#e7298a", elinewidth=1.1, capsize=2.5, zorder=5)
            ax.scatter(OBS28["age"], OBS28["lfd"], s=140, marker="*", c="#e7298a", edgecolors="k", lw=0.7, zorder=6,
                       label=f"ALMA-C11 quiescent galaxies ({len(OBS28)}; arrows = dust upper limits)")
        if P28_AGE_LIM is not None:
            ax.set_xlim(*P28_AGE_LIM)
        elif _x0:
            _lo = min(_x0 + ([float(np.nanmin(OBS28["age"] - OBS28["age_err"]))] if (obs and len(OBS28)) else []))
            _hi = max(_x1 + ([float(np.nanmax(OBS28["age"] + OBS28["age_err"]))] if (obs and len(OBS28)) else []))
            ax.set_xlim(_lo - 0.3, _hi + 0.3)
        ax.set_ylim(*P28_LFD_LIM)
        ax.set_xlabel("stellar age [Gyr]   (sim: mass-weighted;  obs: CIGALE onset, an upper bound)"); ax.set_ylabel(r"$\log\, M_{\rm dust}/M_\star$")
        ax.grid(alpha=0.2, lw=0.5)
        if legend:
            ax.legend(frameon=False, fontsize=8, ncol=2 if len(runs) > 1 else 1, loc="lower right",
                      title="solid = dusty, dashed = dust-poor; running median, 16-84 band", title_fontsize=8)

    def _dosepanel28(ax, b, xlabel=True):
        """One dose panel: the dusty / dust-poor histograms of the exposure, the shaded halves, the four shares in the corners."""
        _Q, _dc = FR28[b], CUT28[b]
        _dose = _Q[P28_DOSE].to_numpy(float); _fin = np.isfinite(_dose); _dusty = _Q["pop"].to_numpy() == "dusty"
        _xmax = 1.0 if P28_DOSE in ("f_x_post", "f_x") else float(np.nanmax(_dose)) * 1.02
        _edges = np.linspace(0.0, _xmax, P28_NBINS + 1)
        ax.axvspan(0.0, _dc, color=LOWC28, alpha=0.07, lw=0, zorder=0); ax.axvspan(_dc, _xmax, color=HIGHC28, alpha=0.07, lw=0, zorder=0)
        _top = 1.0
        for _p in ("pure", "dusty"):
            _v = _dose[_fin & (_dusty == (_p == "dusty"))]
            _cnt, _ = np.histogram(np.clip(_v, 0.0, _xmax), bins=_edges)
            ax.hist(np.clip(_v, 0.0, _xmax), bins=_edges, histtype="stepfilled", color=POPS[_p]["color"], alpha=0.32, lw=0, zorder=2)
            ax.hist(np.clip(_v, 0.0, _xmax), bins=_edges, histtype="step", color=POPS[_p]["color"], lw=1.8, zorder=3,
                    label=("dusty" if _p == "dusty" else "dust-poor") + f"  ({len(_v)})")
            _top = max(_top, float(_cnt.max()) if len(_cnt) else 1.0)
        ax.axvline(_dc, color="k", lw=1.2, ls="--", zorder=4)
        ax.text(_dc, 0.985, f" split {_dc:.2f}", transform=ax.get_xaxis_transform(), fontsize=7.5, va="top", ha="left", color="0.2")
        _pc = PCT28[PCT28["box"] == b]
        _g = lambda p, d: float(_pc[(_pc["pop"] == p) & (_pc["dose"] == d)]["pct"].iloc[0]) if len(_pc) else np.nan
        _sym = r"$\leq$" if P28_DOSE != "e_x" else r"$\leq$"
        _txt_lo = f"low dose ({_sym} {_dc:.2f})\ndusty  {_g('dusty', 'low'):.1f} %\ndust-poor  {_g('dust-poor', 'low'):.1f} %"
        _txt_hi = f"high dose (> {_dc:.2f})\ndusty  {_g('dusty', 'high'):.1f} %\ndust-poor  {_g('dust-poor', 'high'):.1f} %"
        if P28_LOGY:
            ax.set_yscale("log"); ax.set_ylim(0.7, _top * 3.0)
            _bb = dict(boxstyle="round,pad=0.3", fc="white", alpha=0.92, lw=0.6)
            ax.text(0.02, 0.03, _txt_lo, transform=ax.transAxes, ha="left", va="bottom", fontsize=8, color=LOWC28, bbox=dict(_bb, ec=LOWC28), zorder=8)
            ax.text(0.98, 0.03, _txt_hi, transform=ax.transAxes, ha="right", va="bottom", fontsize=8, color=HIGHC28, bbox=dict(_bb, ec=HIGHC28), zorder=8)
        else:                                            # the histogram floats above a blank band that carries the shares
            ax.set_ylim(-0.46 * _top, 1.18 * _top)
            ax.axhline(0, color="0.3", lw=0.8, zorder=3)
            ax.set_yticks([t for t in ax.get_yticks() if 0 <= t <= 1.18 * _top])
            ax.text(0.02, 0.03, _txt_lo, transform=ax.transAxes, ha="left", va="bottom", fontsize=8, color=LOWC28, zorder=8)
            ax.text(0.98, 0.03, _txt_hi, transform=ax.transAxes, ha="right", va="bottom", fontsize=8, color=HIGHC28, zorder=8)
        _n_all, _n_dosed = len(_Q), int(_fin.sum())
        ax.set_title(f"{BOX_LABEL.get(b, b)}:  {_n_all} selected (log $M_\\star$ > {MASS_MIN:g}), {_n_dosed} with a dose"
                     + (f"  —  shares of the {'sample' if P28_DENOM == 'sample' else 'dosed rows'}" if True else ""), fontsize=9)
        ax.set_xlim(0.0, _xmax); ax.set_ylabel("N anchor rows"); ax.grid(alpha=0.2, lw=0.5, axis="y")
        if xlabel:
            ax.set_xlabel(DLAB28.get(P28_DOSE, P28_DOSE))
        ax.legend(frameon=False, fontsize=8, loc="upper center")

    _bx = [b for b in P28_BOXES if b in FR28]
    if not _bx and not any(v is not None for v in TR28.values()):
        print("Part 28: nothing to draw.")
    else:
        _nr = max(len(_bx), 1)
        if P28_LAYOUT == "rows":
            fig = plt.figure(figsize=(13.6, 4.3 * _nr), constrained_layout=True)
            gs = fig.add_gridspec(_nr, 2, width_ratios=[1.35, 1.0])
            for _i, _b in enumerate(_bx):
                _rn = [_b] + ([P28_NOX] if (P28_NOX and _b == "cis50" and (P28_NOX, "dusty") in TR28) else [])
                _plane28(fig.add_subplot(gs[_i, 0]), _rn, inbox=(_b == P28_INBOX))
                _dosepanel28(fig.add_subplot(gs[_i, 1]), _b, xlabel=(_i == _nr - 1))
        else:
            fig = plt.figure(figsize=(14.5, 3.9 * _nr), constrained_layout=True)
            gs = fig.add_gridspec(_nr, 2, width_ratios=[1.45, 1.0])
            _plane28(fig.add_subplot(gs[:, 0]), _runs)
            for _i, _b in enumerate(_bx):
                _dosepanel28(fig.add_subplot(gs[_i, 1]), _b, xlabel=(_i == _nr - 1))
        fig.suptitle(f"Part 28 — the dust-age plane of every run against the ALMA-C11 sample ({'Part 0 sample' if P28_SEL == 'sample' else 'observed regime'}, "
                     f"dust split at log $M_{{\\rm dust}}/M_\\star$ = {DUSTY_CUT:g}), and the exposure behind the high / low dose split of each box\n"
                     f"(right: {DLAB28.get(P28_DOSE, P28_DOSE)} of every anchor row with a chain; the shares in the corners are of "
                     f"{'the whole selected sample' if P28_DENOM == 'sample' else 'the rows with a dose'})", fontsize=11)
        paper_save(fig, "agn_states_fdust_age_dose")
        plt.show()
        if len(PCT28):
            _p28 = os.path.join(OUT, "agn_states_fdust_age_dose.csv"); PCT28.to_csv(_p28, index=False); print(f"the shares -> {_p28}")
        if TRK28:
            _t28 = os.path.join(OUT, "agn_states_fdust_age_tracks.csv"); pd.DataFrame(TRK28).to_csv(_t28, index=False); print(f"the tracks -> {_t28}")

## Part 29 — the cuts on trial: are the median splits on the dose and on the compactness natural, or arbitrary?

Parts 22–27 cut every box at the **median** of the X-ray duty fraction since SFT and at the **median** of a compactness, and the four quadrants became the four classes. Medians are convenient, not physical, so this part asks whether *the dust knows about those cuts* — on Part 28's anchor rows, with nothing but the dusty fraction ($\log M_{\rm dust}/M_\star \geq$ `DUSTY_CUT`), which is the reading that survives the bimodality of the dust distribution.

**Figure A, the axes.** One row per axis, one column per box. The dose; the compactness Parts 22–26 use, $-\log R_{50}({\rm gas})/R_{50}(\star)$; and the alternative asked for, the **concentration of the gas itself**, $\log R_{80}/R_{20}$ from caesar's `radii.gas_r20` / `gas_r80` (Conselice's $C/5$: a centrally concentrated profile is high, a flat one low; `P29_AXES` also offers it relative to the stellar concentration). In each panel: the dusty fraction in `P29_NBINS` equal-count bins with Wilson intervals, the axis's own distribution behind it, the cut in use (dashed: Part 22's for the dose, the median for a compactness), and the **contrast scan** — the dusty fraction below minus above *every* candidate cut between the `P29_SCAN` quantiles, drawn thin on the same fraction scale with its maximum marked. The reading rule: a step is a threshold, a slope a continuum, and a cut on a flat stretch is arbitrary whatever contrast it happens to have. The printed AUC of each axis for dusty vs dust-poor is the cut-free measure of how much the axis knows at all.

**Figure B, the plane.** Dose $\times$ compactness (one row per compactness axis), cut into `P29_GRID` $\times$ `P29_GRID` equal-count cells drawn in **rank space** — every cell the same size, the axes labelled with the data values at the edges, the medians at 0.5 — and coloured by the dusty fraction, with the four median quadrants (the classes) and their own fractions written in. Four populations would be four flat blocks; one gradient would run across the lines. The spread of the cell fractions inside each quadrant is printed. Products `agn_states_cuts_axes.{png,pdf}`, `agn_states_cuts_plane.{png,pdf}`, and the CSVs `agn_states_cuts_{axes,scan,plane}.csv`; the radii cache `agn_states_gas_concentration_<box>.csv`.

In [ ]:
# ── Part 29 — THE CUTS ON TRIAL: are the median splits on the dose and on the compactness natural, or arbitrary? ──
# Parts 22-27 divide every box at the MEDIAN of the X-ray duty fraction since SFT and at the MEDIAN of a compactness. This part
# asks whether the DUST knows about those cuts, on Part 28's anchor rows, with nothing but the dusty fraction:
#   FIG A  THE AXES, one row per axis (the dose; the compactness of Parts 22-26, -log R50(gas)/R50(*); the gas's own
#          CONCENTRATION log R80/R20 — the r20/r80 alternative, read from caesar's radii.gas_r20 / r80 and cached), one column per
#          box: the dusty fraction in P29_NBINS equal-count bins with Wilson intervals, the axis's own distribution behind it, the
#          cut in use (dashed), and the CONTRAST SCAN — the dusty fraction below minus above every candidate cut between the
#          P29_SCAN quantiles, drawn thin on the same fraction scale, its maximum marked (dotted). A threshold shows as a step, a
#          continuum as a slope, and a cut on a flat stretch is arbitrary whatever its contrast.
#   FIG B  THE PLANE, dose x compactness (one row per compactness axis), cut into P29_GRID x P29_GRID EQUAL-COUNT cells drawn in
#          rank space (every cell the same size, the axes labelled with the data values at the edges, the medians at 0.5) and
#          coloured by the dusty fraction; the four median quadrants = the four classes of Parts 22-26, each with its own
#          fraction written in. Four populations would be four flat blocks; one gradient would run across the lines.
# Printed per box and axis: the AUC of the axis for dusty vs dust-poor (the cut-free measure of what the axis knows), the
# contrast at the cut in use against the best contrast, and the spread of the cell fractions INSIDE each quadrant.
P29_BOXES = ["cis100", "cis50", "cis25"]
P29_DOSE  = globals().get("P28_DOSE", "f_x_post")   # the dose axis (Part 28's frame carries it)
P29_AXES  = ["r50", "conc"]     # the compactness axes drawn, in this order: "r50" = -log R50(gas)/R50(*) (Parts 22-26's catalogue axis)
                                #   | "conc" = log R80/R20 of the gas (Conselice's C / 5: a concentrated profile is HIGH)
                                #   | "conc_rel" = log (R80/R20)_gas - log (R80/R20)_star (the gas concentration relative to the stars')
P29_NBINS = 10                  # equal-count bins of the running dusty fraction along each axis
P29_SCAN  = (0.10, 0.90)        # candidate cuts scanned between these quantiles of the axis
P29_NSCAN = 61
P29_GRID  = 4                   # the plane: equal-count edges per axis (an EVEN number keeps the medians on cell edges)
P29_NMIN  = 8                   # a bin below this is not drawn; a cell below it is hatched
P29_CATDIR = {}                 # {"cis100": "/path/to/Groups", ...} overrides of the config's catalog_dir for the concentration reader
P29_NGAS_CHK = 200              # the resolution guard of a compactness axis: its AUC repeated inside ngas >= this, and Spearman with log ngas
P29_CMAP  = "Reds"
P29_HIST_BINS = 30
CUTC29, BESTC29 = "k", "#d7191c"

if "FR28" not in globals() or "CUT28" not in globals():
    print("Part 29: needs Part 28 — its dosed anchor-row frames (FR28) and the cuts in use (CUT28). Run it first.")
else:
    # ── the gas concentration: caesar's r20 / r50 / r80 of the gas and of the stars for every anchor row, cached per box ──
    def _catdir29(box):
        """The caesar directory and file pattern of a box: P29_CATDIR, else ~/.simbanator/config.json (Part 26's rule)."""
        import json as _json29
        try:
            cfg = _json29.load(open(os.path.expanduser("~/.simbanator/config.json"))); ent = (cfg.get("simulations", cfg)).get(box, {})
        except Exception:
            ent = {}
        _stem = {"cis100": "m100n1024", "cis50": "m50n512", "cis50nox": "m50n512", "cis25": "m25n512"}.get(box, box)
        return P29_CATDIR.get(box, ent.get("catalog_dir")), ent.get("file_format", "%s_{snap:03d}.hdf5" % _stem)

    _RC29 = ["r20_gas_kpc", "r50_gas_kpc", "r80_gas_kpc", "r20_star_kpc", "r50_star_kpc", "r80_star_kpc"]
    _RK29 = ("gas_r20", "gas_half_mass", "gas_r80", "stellar_r20", "stellar_half_mass", "stellar_r80")

    def _radii29(box, Q):
        """The three radii of the gas and of the stars (physical kpc) for Q's (snap, gal_id) rows, read from the catalogue files
        once and cached in OUT; rows the files do not yield stay NaN (and are re-tried next run)."""
        cache = os.path.join(OUT, f"agn_states_gas_concentration_{box}.csv")
        have = pd.read_csv(cache) if os.path.exists(cache) else pd.DataFrame(columns=["snap", "gal_id"] + _RC29)
        known = set(zip(have["snap"].astype(int), have["gal_id"].astype(int)))
        todo = Q[[(int(s), int(g)) not in known for s, g in zip(Q["snap"], Q["gal_id"])]]
        if len(todo):
            cdir, fmt = _catdir29(box); rows, miss = [], []
            for snap in sorted(set(todo["snap"].astype(int))):
                fn = os.path.join(cdir or "", fmt.format(snap=int(snap)))
                ids = todo.loc[todo["snap"].astype(int) == snap, "gal_id"].astype(int).to_numpy()
                try:
                    with h5py.File(fn, "r") as f:
                        a = float(f["simulation_attributes"].attrs["scale_factor"]); g = f["galaxy_data/dicts"]
                        R = {k: g[f"radii.{k}"][:] * a for k in _RK29}                     # caesar radii are comoving kpc: x a
                    for i in ids:
                        rows.append(dict(snap=int(snap), gal_id=int(i), **{c: float(R[k][i]) for c, k in zip(_RC29, _RK29)}))
                except Exception as e:
                    miss.append(f"snap {snap}: {type(e).__name__}: {e} ({fn}) — {len(ids)} rows")
            if rows:
                have = pd.concat([have, pd.DataFrame(rows)], ignore_index=True) if len(have) else pd.DataFrame(rows); have.to_csv(cache, index=False)
                print(f"  [{box}] gas / stellar r20, r50, r80 read for {len(rows)} anchor rows -> {cache}")
            for m in miss:
                print(f"  [{box}] concentration not read, {m}")
        return Q.merge(have, on=["snap", "gal_id"], how="left")

    def _axis29(Q, name):
        """One compactness axis of a frame: the values, the label, the names of its high and low sides; NaN where a radius is missing or zero."""
        with np.errstate(all="ignore"):
            if name == "r50":
                rg, rs = Q["r_half_gas"].to_numpy(float), Q["r_half_star"].to_numpy(float)
                v = -np.log10(np.where((rg > 0) & (rs > 0), rg / np.where(rs > 0, rs, np.nan), np.nan))
                return v, r"$-\log\, R_{50}({\rm gas})/R_{50}(\star)$", "compact", "extended"
            for c in _RC29:
                if c not in Q.columns:
                    return np.full(len(Q), np.nan), name, "", ""
            r20, r80 = Q["r20_gas_kpc"].to_numpy(float), Q["r80_gas_kpc"].to_numpy(float)
            c = np.log10(np.where((r20 > 0) & (r80 > 0), r80 / np.where(r20 > 0, r20, np.nan), np.nan))
            if name == "conc":
                return c, r"$\log\, R_{80}/R_{20}$ (gas)", "concentrated", "flat profile"
            s20, s80 = Q["r20_star_kpc"].to_numpy(float), Q["r80_star_kpc"].to_numpy(float)
            cs = np.log10(np.where((s20 > 0) & (s80 > 0), s80 / np.where(s20 > 0, s20, np.nan), np.nan))
            return c - cs, r"$\log\, (R_{80}/R_{20})_{\rm gas} - \log\, (R_{80}/R_{20})_\star$", "gas more concentrated than the stars", "less"

    def _wilson29(k, n, z=1.0):
        """The Wilson interval of k / n, clamped to contain the point estimate (Part 8's fix)."""
        if n <= 0:
            return np.nan, np.nan, np.nan
        p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
        return p, min(p, max(0.0, c - h)), max(p, min(1.0, c + h))

    def _frac29(x, dusty, nb, nmin=P29_NMIN):
        """The dusty fraction in `nb` equal-count bins of x: (x = the bin's median, p, lo, hi, n) per bin, fewer bins for a thin set."""
        x, dusty = np.asarray(x, float), np.asarray(dusty, bool); ok = np.isfinite(x); x, dusty = x[ok], dusty[ok]
        if len(x) < nmin:
            return pd.DataFrame(columns=["x", "p", "lo", "hi", "n"])
        nb_ = int(max(1, min(nb, len(x) // nmin)))
        e = np.quantile(x, np.linspace(0, 1, nb_ + 1)); e[0] -= 1e-9
        ib = np.clip(np.searchsorted(e, x, side="right") - 1, 0, nb_ - 1)
        out = []
        for i in range(nb_):
            m = ib == i
            if m.sum() >= 1:
                p, lo, hi = _wilson29(int(dusty[m].sum()), int(m.sum()))
                out.append((float(np.median(x[m])), p, lo, hi, int(m.sum())))
        return pd.DataFrame(out, columns=["x", "p", "lo", "hi", "n"])

    def _scan29(x, dusty):
        """The contrast of every candidate cut: the dusty fraction at or below the cut minus above it, over the P29_SCAN quantiles."""
        x, dusty = np.asarray(x, float), np.asarray(dusty, bool); ok = np.isfinite(x); x, dusty = x[ok], dusty[ok]
        if len(x) < 2 * P29_NMIN:
            return np.array([]), np.array([])
        cs = np.quantile(x, np.linspace(P29_SCAN[0], P29_SCAN[1], P29_NSCAN))
        con = np.array([np.mean(dusty[x <= c]) - np.mean(dusty[x > c]) if ((x <= c).sum() >= P29_NMIN and (x > c).sum() >= P29_NMIN) else np.nan for c in cs])
        return cs, con

    # ── the frames: Part 28's dosed anchor rows + the radii ──
    FR29, AX29 = {}, {}
    for _b in P29_BOXES:
        if _b not in FR28:
            print(f"[{_b}] not in Part 28's frames — skipped"); continue
        _Q = FR28[_b].copy()
        if any(a != "r50" for a in P29_AXES):
            _Q = _radii29(_b, _Q)
        _Q["dusty"] = _Q["pop"].to_numpy() == "dusty"
        _ax = {P29_DOSE: (_Q[P29_DOSE].to_numpy(float), globals().get("DLAB28", {}).get(P29_DOSE, P29_DOSE), "high dose", "low dose")}
        for _a in P29_AXES:
            _ax[_a] = _axis29(_Q, _a)
        FR29[_b], AX29[_b] = _Q, _ax
        _d = _Q["dusty"].to_numpy(bool)
        print(f"[{_b}] {len(_Q)} anchor rows, {int(_d.sum())} dusty; the cut-free measure of each axis, AUC = P(dusty above dust-poor):")
        for _k, (_v, _lab, _hi, _lo) in _ax.items():
            _auc, _p, _n1, _n0 = mw_auc(_v, _d)
            _fin = np.isfinite(_v)
            print(f"     {_k:9s}: AUC {_auc:.2f} (p {_p:.1e}; {int(_fin.sum())} finite)  median {np.nanmedian(_v):+.2f}, 16-84 [{np.nanpercentile(_v[_fin], 16):+.2f}, {np.nanpercentile(_v[_fin], 84):+.2f}]"
                  if np.isfinite(_auc) else f"     {_k:9s}: too few finite values ({int(_fin.sum())})")
        if len(P29_AXES) == 2:
            _v1, _v2 = _ax[P29_AXES[0]][0], _ax[P29_AXES[1]][0]; _ok = np.isfinite(_v1) & np.isfinite(_v2)
            if _ok.sum() >= P29_NMIN:
                print(f"     the two compactness axes against each other: Spearman {spearmanr(_v1[_ok], _v2[_ok])[0]:+.2f} (N {int(_ok.sum())})")
        # the resolution guard: a compactness axis must not be the gas particle count in disguise
        if "ngas" in _Q.columns:
            _ng = _Q["ngas"].to_numpy(float); _lng = np.log10(np.where(_ng > 0, _ng, np.nan)); _big = _ng >= P29_NGAS_CHK
            for _a in P29_AXES:
                _v = _ax[_a][0]; _ok = np.isfinite(_v) & np.isfinite(_lng)
                if _ok.sum() < P29_NMIN:
                    continue
                _auc_b = mw_auc(_v[_big], _d[_big])[0] if (_big & _d).sum() >= 3 and (_big & ~_d).sum() >= 3 else np.nan
                print(f"     resolution guard, {_a}: Spearman with log n_gas {spearmanr(_v[_ok], _lng[_ok])[0]:+.2f}, with log M_gas "
                      f"{spearmanr(_v[_ok], np.log10(np.where(_Q['mgas'].to_numpy(float)[_ok] > 0, _Q['mgas'].to_numpy(float)[_ok], np.nan)), nan_policy='omit')[0]:+.2f}; "
                      f"AUC(dusty) inside n_gas >= {P29_NGAS_CHK}: {_auc_b:.2f} ({int(_big.sum())} rows, {int((_big & _d).sum())} dusty)")

    _bx = [b for b in P29_BOXES if b in FR29]
    if not _bx:
        print("Part 29: nothing to draw.")
    else:
        # ── FIGURE A: the axes ──
        _rows = [P29_DOSE] + list(P29_AXES)
        R29, SC29 = [], []
        fig, axs = plt.subplots(len(_rows), len(_bx), figsize=(4.8 * len(_bx), 3.6 * len(_rows)), squeeze=False, constrained_layout=True)
        for _i, _k in enumerate(_rows):
            for _j, _b in enumerate(_bx):
                ax = axs[_i, _j]; _Q = FR29[_b]; _v, _lab, _hi, _lo = AX29[_b][_k]; _d = _Q["dusty"].to_numpy(bool)
                _fin = np.isfinite(_v)
                if _fin.sum() < 2 * P29_NMIN:
                    ax.text(0.5, 0.5, f"{int(_fin.sum())} values — nothing to draw", transform=ax.transAxes, ha="center", va="center", color="0.4"); ax.set_axis_off(); continue
                _cut = CUT28[_b] if _k == P29_DOSE else float(np.nanmedian(_v))                       # the cut in use: Part 22's for the dose, the median else
                _f = _frac29(_v, _d, P29_NBINS); _cs, _con = _scan29(_v, _d)
                # the axis's own distribution behind (a twin axis, no ticks)
                axb = ax.twinx(); axb.set_zorder(ax.get_zorder() - 1); ax.patch.set_visible(False)
                _e = np.linspace(np.nanmin(_v), np.nanmax(_v), P29_HIST_BINS + 1)
                axb.hist(_v[_fin], bins=_e, color="0.55", alpha=0.16, lw=0); axb.set_yticks([]); axb.set_ylim(0, axb.get_ylim()[1] * 2.6)
                # the fraction, the scan, the cuts
                ax.errorbar(_f["x"], _f["p"], yerr=[_f["p"] - _f["lo"], _f["hi"] - _f["p"]], marker="o", ms=5.5, lw=2.0, capsize=2.5, color="0.1",
                            label=f"dusty fraction, {len(_f)} equal-count bins" if _j == 0 else None, zorder=4)
                _best = _cb = _cu = np.nan
                if len(_cs) and np.isfinite(_con).any():
                    ax.plot(_cs, _con, ls=":", lw=1.4, color="0.35", label="contrast of a cut placed here (below $-$ above)" if _j == 0 else None, zorder=3)
                    _ib = int(np.nanargmax(np.abs(_con))); _best = float(_cs[_ib]); _cb = float(_con[_ib])
                    _cu = float(np.interp(_cut, _cs, _con)) if (_cs[0] <= _cut <= _cs[-1]) else np.nan
                    ax.axvline(_best, color=BESTC29, ls=(0, (1.5, 2.5)), lw=1.6, zorder=2)
                    ax.text(_best, 0.985, f" best {_best:.2f} ", transform=ax.get_xaxis_transform(), fontsize=7.2, color=BESTC29, va="top",
                            ha="left" if _best >= _cut else "right")
                ax.axvline(_cut, color=CUTC29, ls="--", lw=1.4, zorder=2)
                ax.text(_cut, 0.985, f" cut {_cut:.2f} ", transform=ax.get_xaxis_transform(), fontsize=7.2, color=CUTC29, va="top",
                        ha="right" if (np.isfinite(_best) and _best >= _cut) else "left")
                ax.axhline(0, color="0.8", lw=0.7, zorder=1)
                _lo_y = min(-0.05, float(np.nanmin(_con)) - 0.05) if (len(_cs) and np.isfinite(_con).any()) else -0.05
                ax.set_ylim(_lo_y, max(0.35, float(_f["hi"].max()) * 1.35) if len(_f) else 1.0)
                ax.set_xlabel(_lab); ax.grid(alpha=0.2, lw=0.5)
                ax.text(0.01, 0.02, f"$\\leftarrow$ {_lo}", transform=ax.transAxes, fontsize=7.5, color="0.4", ha="left", va="bottom")
                ax.text(0.99, 0.02, f"{_hi} $\\rightarrow$", transform=ax.transAxes, fontsize=7.5, color="0.4", ha="right", va="bottom")
                if _j == 0:
                    ax.set_ylabel(f"fraction with log $M_{{\\rm dust}}/M_\\star \\geq$ {DUSTY_CUT:g}")
                if _i == 0:
                    ax.set_title(f"{BOX_LABEL.get(_b, _b)}\n{len(_Q)} anchor rows, {int(_d.sum())} dusty", fontsize=9.5)
                R29.append(dict(box=_b, axis=_k, n=int(_fin.sum()), cut_in_use=_cut, contrast_at_cut=_cu, best_cut=_best, best_contrast=_cb,
                                frac_first_bin=float(_f["p"].iloc[0]) if len(_f) else np.nan, frac_last_bin=float(_f["p"].iloc[-1]) if len(_f) else np.nan,
                                frac_min=float(_f["p"].min()) if len(_f) else np.nan, frac_max=float(_f["p"].max()) if len(_f) else np.nan,
                                auc_dusty=mw_auc(_v, _d)[0]))
                SC29 += [dict(box=_b, axis=_k, cut=float(c), contrast=float(v)) for c, v in zip(_cs, _con)]
                if len(_f):
                    print(f"  [{_b}] {_k:9s}: dusty fraction {100 * _f['p'].iloc[0]:.0f} % ({_lo} end) -> {100 * _f['p'].iloc[-1]:.0f} % ({_hi} end), "
                          f"min {100 * _f['p'].min():.0f} / max {100 * _f['p'].max():.0f} %; contrast at the cut in use {_cut:.2f}: {_cu:+.2f}, best {_cb:+.2f} at {_best:.2f}"
                          + (f"  -> the cut keeps {100 * abs(_cu) / abs(_cb):.0f} % of the best contrast" if (np.isfinite(_cu) and np.isfinite(_cb) and abs(_cb) > 0) else ""))
        axs[0, 0].legend(frameon=False, fontsize=7.6, loc="upper left", bbox_to_anchor=(0.0, 0.93))
        _h29 = [Line2D([], [], color=CUTC29, ls="--", lw=1.4, label="the cut in use (Part 22's for the dose, the median for a compactness)"),
                Line2D([], [], color=BESTC29, ls=(0, (1.5, 2.5)), lw=1.6, label="the cut of largest contrast"),
                plt.Rectangle((0, 0), 1, 1, fc="0.55", alpha=0.25, lw=0, label="the axis's distribution (behind, counts)")]
        axs[min(1, len(_rows) - 1), 0].legend(handles=_h29, frameon=False, fontsize=7.6, loc="center right")
        fig.suptitle("Part 29 A — the cuts on trial: the dusty fraction along each axis the classes are cut on, and the contrast of every candidate cut\n"
                     "a step = a threshold, a slope = a continuum, a cut on a flat stretch = arbitrary (anchor rows of Part 0's sample)", fontsize=11)
        paper_save(fig, "agn_states_cuts_axes")
        plt.show()

        # ── FIGURE B: the plane, in rank space ──
        PL29 = []
        fig, axs = plt.subplots(len(P29_AXES), len(_bx), figsize=(4.9 * len(_bx), 4.4 * len(P29_AXES)), squeeze=False, constrained_layout=True)
        _ims = []
        for _i, _a in enumerate(P29_AXES):
            for _j, _b in enumerate(_bx):
                ax = axs[_i, _j]; _Q = FR29[_b]; _d = _Q["dusty"].to_numpy(bool)
                _x, _xlab = AX29[_b][P29_DOSE][0], AX29[_b][P29_DOSE][1]; _y, _ylab, _yhi, _ylo = AX29[_b][_a]
                _ok = np.isfinite(_x) & np.isfinite(_y)
                _G = int(P29_GRID) if _ok.sum() >= P29_GRID ** 2 * P29_NMIN else 2          # a small box gets the bare quadrants
                _half = _G // 2
                if _ok.sum() < 4 * P29_NMIN:
                    ax.text(0.5, 0.5, f"{int(_ok.sum())} rows — too few for a plane", transform=ax.transAxes, ha="center", va="center", color="0.4"); ax.set_axis_off(); continue
                _ex = np.quantile(_x[_ok], np.linspace(0, 1, _G + 1)); _ey = np.quantile(_y[_ok], np.linspace(0, 1, _G + 1))
                _ix = np.clip(np.searchsorted(_ex, _x[_ok], side="right") - 1, 0, _G - 1); _iy = np.clip(np.searchsorted(_ey, _y[_ok], side="right") - 1, 0, _G - 1)
                _F, _N = np.full((_G, _G), np.nan), np.zeros((_G, _G), int)
                for _p in range(_G):
                    for _q in range(_G):
                        m = (_ix == _p) & (_iy == _q); _N[_q, _p] = int(m.sum())
                        if m.sum() >= P29_NMIN:
                            _F[_q, _p] = float(np.mean(_d[_ok][m]))
                _vmax = max(0.3, float(np.nanmax(_F))) if np.isfinite(_F).any() else 1.0
                im = ax.imshow(_F, origin="lower", extent=(0, 1, 0, 1), cmap=P29_CMAP, vmin=0, vmax=_vmax, aspect="auto", zorder=1)
                _ims.append(im)
                for _p in range(_G):
                    for _q in range(_G):
                        if _N[_q, _p] < P29_NMIN:
                            ax.add_patch(plt.Rectangle((_p / _G, _q / _G), 1 / _G, 1 / _G, fc="none", hatch="////", ec="0.6", lw=0, zorder=2))
                        else:
                            ax.text((_p + 0.5) / _G, (_q + 0.5) / _G, f"{100 * _F[_q, _p]:.0f}\n({_N[_q, _p]})", ha="center", va="center", fontsize=6.8,
                                    color="w" if _F[_q, _p] > 0.6 * _vmax else "k", zorder=4)
                ax.axvline(0.5, color="k", lw=1.6, zorder=3); ax.axhline(0.5, color="k", lw=1.6, zorder=3)
                # the four quadrants = the four classes, their own fraction and the spread of their cells
                _dm = _d[_ok]; _xh = _ix >= _half; _yh = _iy >= _half
                for _qx, _qy, _nm in ((False, True, f"low dose + {_yhi}"), (False, False, f"low dose + {_ylo}"), (True, True, f"high dose + {_yhi}"), (True, False, f"high dose + {_ylo}")):
                    m = (_xh == _qx) & (_yh == _qy)
                    _cells = _F[(_half if _qy else 0):(_G if _qy else _half), (_half if _qx else 0):(_G if _qx else _half)]
                    _pq = float(np.mean(_dm[m])) if m.sum() else np.nan
                    ax.text(0.75 if _qx else 0.25, 0.985 if _qy else 0.485, f"{_nm}: {100 * _pq:.0f} %", transform=ax.transAxes, ha="center", va="top", fontsize=7.4,
                            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.5", lw=0.6, alpha=0.9), zorder=5)
                    PL29.append(dict(box=_b, geom=_a, quadrant=_nm, n=int(m.sum()), dusty_frac=_pq, cell_min=float(np.nanmin(_cells)) if np.isfinite(_cells).any() else np.nan,
                                     cell_max=float(np.nanmax(_cells)) if np.isfinite(_cells).any() else np.nan))
                ax.set_xticks(np.linspace(0, 1, _G + 1)); ax.set_xticklabels([f"{v:.2f}" for v in _ex], fontsize=7.5)
                ax.set_yticks(np.linspace(0, 1, _G + 1)); ax.set_yticklabels([f"{v:+.2f}" for v in _ey], fontsize=7.5)
                ax.set_xlabel(_xlab + "  (equal-count edges)"); ax.set_ylabel(_ylab + "  (equal-count edges)")
                ax.set_title(f"{BOX_LABEL.get(_b, _b)}: {int(_ok.sum())} rows, {_G}x{_G} equal-count cells", fontsize=9)
                _qq = [r for r in PL29 if r["box"] == _b and r["geom"] == _a]
                print(f"  [{_b}] plane dose x {_a}: " + ";  ".join(f"{r['quadrant']} {100 * r['dusty_frac']:.0f} % (cells {100 * r['cell_min']:.0f}-{100 * r['cell_max']:.0f} %)" for r in _qq))
        if _ims:
            fig.colorbar(_ims[0], ax=axs.ravel().tolist(), shrink=0.6, pad=0.01, label=f"fraction with log $M_{{\\rm dust}}/M_\\star \\geq$ {DUSTY_CUT:g}")
        fig.suptitle("Part 29 B — the dose x compactness plane in equal-count cells (rank space), coloured by the dusty fraction; the black lines are the median cuts,\n"
                     "the four quadrants the classes of Parts 22-26 with their own fraction (cells: fraction in %, N in brackets; hatched = under P29_NMIN)", fontsize=11)
        paper_save(fig, "agn_states_cuts_plane")
        plt.show()
        pd.DataFrame(R29).to_csv(os.path.join(OUT, "agn_states_cuts_axes.csv"), index=False)
        pd.DataFrame(SC29).to_csv(os.path.join(OUT, "agn_states_cuts_scan.csv"), index=False)
        pd.DataFrame(PL29).to_csv(os.path.join(OUT, "agn_states_cuts_plane.csv"), index=False)
        print(f"-> {os.path.join(OUT, 'agn_states_cuts_{axes,scan,plane}.csv')}")

## Part 30 — the dose × concentration and the exposure × concentration planes, with the compact / extended division on the concentration

Part 29 found that the compactness axis the dust actually responds to is the **gas's own concentration**, $\log R_{80}/R_{20}$, and that its natural cut sits above the median. This part draws the two planes a class division would be made on, one column per box: the **X-ray duty fraction since SFT** against the concentration (top; Parts 22–27's dose, its cut in use dashed) and the **integrated exposure** $E_x$ against the concentration (bottom; Part 2a's dose, the "fully dosed" `PF_EX` dashed). Every anchor row of Part 29's frame is a point, dusty on top of dust-poor. The **compact / extended division** is the horizontal line at `P30_CCUT` on the concentration — a number is the same physical threshold in every box (the default 0.30 is a uniform disc's $R_{80}/R_{20} = 2$, and Part 29's cuts of largest contrast are 0.29 / 0.29 / 0.41); `"best"` takes Part 29's cut per box, `"median"` the box's median; the box's Part 29 cut is drawn thin in red when it differs. The reference concentrations of simple profiles are ticked on the right (`P30_REFS`: a uniform sphere at 0.20, a uniform disc at 0.30, an exponential disc at 0.56), so the sample's median of 0.21 reads as *a uniform blob* and the concentrated tail as *anything with a real core*. Each quadrant carries its N and its dusty fraction; the table goes to `agn_states_dose_concentration_planes.csv`.

In [ ]:
# ── Part 30 — THE DOSE x CONCENTRATION AND THE EXPOSURE x CONCENTRATION PLANES, with the compact / extended division drawn on the concentration ──
# Part 29 found that the gas's own concentration (log R80/R20, caesar's radii.gas_r20 / r80) is the compactness axis the dust
# responds to, and that the natural cut sits above the median. This part draws the two planes a class division would be made on:
#   TOP     the X-ray duty fraction since SFT (P30_DOSE, Parts 22-27's dose) against the concentration;
#   BOTTOM  the integrated X-ray exposure E_x [Gyr] (Part 2a's dose; PF_EX = 'fully dosed') against the concentration.
# Every anchor row of Part 29's frame is a point (dusty on top of dust-poor), the COMPACT / EXTENDED division is the horizontal
# line at P30_CCUT on the concentration (a number = the same physical threshold in every box; "best" = Part 29's cut of largest
# contrast per box; "median" = the box's median), the vertical line is the dose cut in use (Part 28's for the duty fraction,
# PF_EX for the exposure), and each quadrant carries its N and its dusty fraction. The reference concentrations of simple
# profiles are ticked on the right (P30_REFS): a uniform sphere, a uniform disc, an exponential disc.
P30_BOXES = ["cis100", "cis50", "cis25"]
P30_DOSE  = globals().get("P29_DOSE", "f_x_post")
P30_EXPO  = "e_x"                # the exposure axis: "e_x" [Gyr, the whole branch] (Part 22's _dose22)
P30_CCUT  = 0.30                 # the compact / extended division on log R80/R20: a number | "best" (Part 29's largest-contrast cut per box) | "median"
P30_SHOW_BEST = True             # also draw Part 29's best cut of the box (thin, red) when P30_CCUT is a number or "median"
P30_XCUT  = "part28"             # the vertical line of the top row: "part28" (CUT28, Parts 22-27's cut) | "best" (Part 29's) | a number | None
P30_EXCUT = globals().get("PF_EX", 1.0)   # the vertical line of the bottom row: Part 2a's 'fully dosed' exposure [Gyr]; None = none
P30_REFS  = {"uniform sphere": np.log10((0.8 / 0.2) ** (1 / 3)), "uniform disc": np.log10((0.8 / 0.2) ** 0.5), "exponential disc": np.log10(2.994 / 0.824)}
P30_YLIM  = None                 # the concentration range (None = the 0.5-99.5 percentiles over the boxes, padded)
P30_MS    = (7, 22)              # marker sizes: dust-poor, dusty
P30_EXLOG = False                # True: the exposure axis as symlog (the never-exposed sit at 0)

if "FR29" not in globals() or "_axis29" not in globals():
    print("Part 30: needs Part 29 — its frames carry the concentration and the two dose axes. Run it first.")
else:
    _bx30 = [b for b in P30_BOXES if b in FR29]
    if not _bx30:
        print("Part 30: none of P30_BOXES is in Part 29's frames.")
    else:
        _best29 = {}
        if "R29" in globals():
            for r in R29:
                if r.get("axis") == "conc":
                    _best29[r["box"]] = float(r["best_cut"])

        def _ccut30(box, conc):
            if isinstance(P30_CCUT, str):
                if P30_CCUT == "best" and box in _best29 and np.isfinite(_best29[box]):
                    return _best29[box], "Part 29's cut of largest contrast"
                return float(np.nanmedian(conc)), "the median"
            return float(P30_CCUT), "fixed"

        _allc = np.concatenate([_axis29(FR29[b], "conc")[0] for b in _bx30]); _allc = _allc[np.isfinite(_allc)]
        _yl = P30_YLIM if P30_YLIM is not None else (float(np.percentile(_allc, 0.5)) - 0.04, float(np.percentile(_allc, 99.5)) + 0.06)
        _EXLAB = {"e_x": r"$E_x$ — time in the X-ray state [Gyr]"}
        Q30 = []
        fig, axs = plt.subplots(2, len(_bx30), figsize=(5.0 * len(_bx30), 8.6), squeeze=False, sharey=True, constrained_layout=True)
        for _j, _b in enumerate(_bx30):
            _Q = FR29[_b]; _conc, _clab, _chi, _clo = _axis29(_Q, "conc"); _d = _Q["dusty"].to_numpy(bool)
            _cc, _csrc = _ccut30(_b, _conc)
            for _i, (_xc, _xlab, _vc) in enumerate([(P30_DOSE, globals().get("DLAB28", {}).get(P30_DOSE, P30_DOSE),
                                                     (CUT28.get(_b) if P30_XCUT == "part28" else (_best29.get(_b) if P30_XCUT == "best" else P30_XCUT))
                                                     if "CUT28" in globals() else None),
                                                    (P30_EXPO, _EXLAB.get(P30_EXPO, P30_EXPO), P30_EXCUT)]):
                ax = axs[_i, _j]
                _x = _Q[_xc].to_numpy(float); _ok = np.isfinite(_x) & np.isfinite(_conc)
                ax.scatter(_x[_ok & ~_d], _conc[_ok & ~_d], s=P30_MS[0], color=POPS["pure"]["color"], alpha=0.35, lw=0, rasterized=True, zorder=2,
                           label=None)
                ax.scatter(_x[_ok & _d], _conc[_ok & _d], s=P30_MS[1], color=POPS["dusty"]["color"], alpha=0.85, lw=0.4, edgecolors="k", zorder=3,
                           label=None)
                # the division, the box's own best cut, the reference profiles
                ax.axhline(_cc, color="k", lw=1.8, zorder=4)
                ax.text(0.01, _cc, f" {_chi} above  (log $R_{{80}}/R_{{20}}$ > {_cc:.2f}, {_csrc})", transform=ax.get_yaxis_transform(), fontsize=7.2, va="bottom", ha="left", color="k", zorder=6)
                ax.text(0.01, _cc, f" {_clo} below", transform=ax.get_yaxis_transform(), fontsize=7.2, va="top", ha="left", color="0.3", zorder=6)
                if P30_SHOW_BEST and _b in _best29 and np.isfinite(_best29[_b]) and abs(_best29[_b] - _cc) > 1e-6:
                    ax.axhline(_best29[_b], color="#d7191c", lw=1.0, ls=(0, (1.5, 2.5)), zorder=4)
                    ax.text(0.99, _best29[_b], f"Part 29 best {_best29[_b]:.2f} ", transform=ax.get_yaxis_transform(), fontsize=6.8,
                            va="top" if _best29[_b] < _cc else "bottom", ha="right", color="#d7191c", zorder=6)
                for _nm, _rv in P30_REFS.items():
                    if _yl[0] < _rv < _yl[1]:
                        ax.axhline(_rv, color="0.7", lw=0.7, ls=":", zorder=1)
                        if _j == len(_bx30) - 1:
                            ax.annotate(_nm, xy=(1.0, _rv), xycoords=ax.get_yaxis_transform(), xytext=(3, 0), textcoords="offset points", fontsize=6.8, color="0.45", va="center", ha="left", annotation_clip=False)
                if _vc is not None and np.isfinite(_vc):
                    ax.axvline(_vc, color="k", lw=1.2, ls="--", zorder=4)
                    ax.text(_vc, 0.86, f" cut {_vc:.2f}", transform=ax.get_xaxis_transform(), fontsize=7.2, va="top", ha="left", color="k", zorder=6,
                            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
                # the quadrants: N and the dusty fraction
                _xs = (_x > _vc) if (_vc is not None and np.isfinite(_vc)) else np.zeros(len(_x), bool)
                for _hx, _hy, _px, _py, _ha, _va in ((False, True, 0.02, 0.97, "left", "top"), (True, True, 0.98, 0.97, "right", "top"),
                                                     (False, False, 0.02, 0.03, "left", "bottom"), (True, False, 0.98, 0.03, "right", "bottom")):
                    if _vc is None and _hx:
                        continue
                    m = _ok & ((_xs == _hx) if _vc is not None else True) & ((_conc > _cc) == _hy)
                    _n, _nd = int(m.sum()), int((m & _d).sum())
                    _nm = (("high" if _hx else "low") + (" dose" if _i == 0 else " exposure") + " + " if _vc is not None else "") + (_chi if _hy else _clo)
                    ax.text(_px, _py, f"{_nm}\nN {_n}, dusty {100 * _nd / max(_n, 1):.0f} %", transform=ax.transAxes, ha=_ha, va=_va, fontsize=7.4,
                            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.6", lw=0.6, alpha=0.9), zorder=7)
                    Q30.append(dict(box=_b, plane=_xc, x_cut=_vc, conc_cut=_cc, conc_cut_source=_csrc, quadrant=_nm, n=_n, n_dusty=_nd, dusty_frac=_nd / max(_n, 1)))
                _mc = (float(np.nanmedian(_conc[_ok & _d])) if (_ok & _d).any() else np.nan, float(np.nanmedian(_conc[_ok & ~_d])) if (_ok & ~_d).any() else np.nan)
                ax.set_xlabel(_xlab); ax.grid(alpha=0.2, lw=0.5); ax.set_ylim(*_yl)
                if _xc == P30_DOSE:
                    ax.set_xlim(-0.02, 1.02)
                elif P30_EXLOG:
                    ax.set_xscale("symlog", linthresh=0.1)
                if _j == 0:
                    ax.set_ylabel(_clab + "   (concentration of the gas)")
                if _i == 0:
                    ax.set_title(f"{BOX_LABEL.get(_b, _b)}: {int(_ok.sum())} anchor rows\nmedian concentration dusty {_mc[0]:+.2f}, dust-poor {_mc[1]:+.2f}", fontsize=9.5)
                print(f"  [{_b}] {_xc:8s}: division at log R80/R20 = {_cc:.2f} ({_csrc}); " + ";  ".join(
                    f"{r['quadrant']} N {r['n']} dusty {100 * r['dusty_frac']:.0f} %" for r in Q30 if r["box"] == _b and r["plane"] == _xc))
        _hl30 = [Line2D([], [], ls="", marker="o", ms=4, color=POPS["pure"]["color"], alpha=0.6, label="dust-poor"),
                 Line2D([], [], ls="", marker="o", ms=6, color=POPS["dusty"]["color"], mec="k", mew=0.4, label="dusty")]
        axs[1, 0].legend(handles=_hl30, frameon=True, framealpha=0.9, fontsize=8, loc="center right")
        fig.suptitle("Part 30 — the dose x concentration (top) and the exposure x concentration (bottom) planes, every anchor row of Part 0's sample;\n"
                     "the horizontal line is the compact / extended division on the gas concentration, the dashed vertical the dose cut in use, the boxes the quadrants' N and dusty fraction", fontsize=11)
        paper_save(fig, "agn_states_dose_concentration_planes")
        plt.show()
        pd.DataFrame(Q30).to_csv(os.path.join(OUT, "agn_states_dose_concentration_planes.csv"), index=False)
        print(f"-> {os.path.join(OUT, 'agn_states_dose_concentration_planes.csv')}")

## Part 31 — the removal depth: how much of the dust a galaxy had at quenching is gone, against the channel and against the well

Parts 29–30 left the mass as an unexplained third axis: at a given duty fraction the massive galaxies keep more dust, and no definition of the dose — time in the state, energy delivered, energy per reservoir — absorbs it. This part turns the outcome into a continuous quantity, the **removal depth** $\log M_{\rm dust}({\rm anchor}) / M_{\rm dust}({\rm SFT})$ (`P31_EPOCH`: the branch snapshot nearest $t_{\rm SFT}$, or the branch peak, or $t_{\rm QT}$), the same for the H$_2$ and the total gas, and asks what sets it. The candidates are the channel (the duty fraction) and the **well** the galaxy sits in: the parent halo's $M_{200c}$ and the NFW escape speed at the galaxy's position (caesar's `parent_halo_index` → `virial_quantities.m200c` / `r200c`, the galaxy's distance to the halo centre for a satellite, its stellar half-mass radius for a central, Dutton & Macciò 2014 concentration; read once per box and cached in `agn_states_halo_props_<box>.csv`), against the stellar and black-hole masses, the gas concentration and the reservoir itself.

The figure has one column per box: (1) the depth against the duty fraction, one running median per stellar-mass quartile — the mass offset at every dose; (2) and (3) the depth against the two well axes of `P31_WELL`, one running median per duty quartile — the well at fixed dose; (4) the H$_2$ depth against the first well axis, to see whether the gas goes with the dust. The printed table is the partial correlations without a model: Spearman of each depth with every candidate raw, at fixed duty, at fixed $\log M_\star$, and at fixed both (3 × 3 cells). `P31_CENTRALS` restricts to centrals, whose parent potential is unambiguously their own. Products `agn_states_removal_depth.{png,pdf,csv}` and `_galaxies.csv` (every row with its depths, halo and escape speed).

In [ ]:
# ── Part 31 — THE REMOVAL DEPTH: how much of the dust (and the H2) a galaxy had at quenching is gone, against the channel and against the well ──
# Parts 29-30 showed the dust responds to the TIME spent in the X-ray state, that the response saturates, and that at a given
# duty fraction the massive galaxies keep more dust — while no definition of the dose (time, energy, energy per reservoir)
# absorbs the mass. This part measures the outcome as a continuous quantity instead of the dusty / dust-poor label:
#     depth = log M_dust(anchor) - log M_dust(epoch),   epoch = the branch snapshot nearest t_SFT (P31_EPOCH; "peak" = the branch
#             maximum, "qt" = t_QT),   the same for the H2 (and the total gas),
# and asks what sets it: the duty fraction (the channel), and the WELL the galaxy sits in — the parent halo's M200c and the NFW
# escape speed at the galaxy's position (caesar's parent_halo_index -> virial_quantities.m200c / r200c, the galaxy's distance to
# the halo centre; Dutton & Maccio 2014 concentration; read once per box and cached), against the stellar and black-hole masses.
#   FIG  rows: (1) depth vs the duty fraction, one running median per STELLAR-MASS quartile (the mass offset at every dose);
#              (2) depth vs P31_WELL[0] and (3) vs P31_WELL[1], one running median per DUTY quartile (the well at fixed dose);
#              (4) the H2 depth vs P31_WELL[0], per duty quartile (does the gas go with the dust).  Columns = boxes.
#   PRINTED  Spearman of each depth with every candidate, raw / at fixed duty / at fixed log M* / at fixed both — the partials.
P31_BOXES = ["cis100", "cis25"]
P31_EPOCH = "sft"                # the reference reservoir: "sft" | "peak" | "qt"
P31_DOSE  = globals().get("P28_DOSE", "f_x_post")
P31_WELL  = ["log_m200c", "vesc"]   # the two well axes drawn: "log_m200c" | "vesc" [km/s, NFW at the galaxy's position] | "log_mstar" | "log_mbh" | "log_halo_mtot" | "halo_vc"
P31_CENTRALS = False             # True: centrals only (the parent halo's potential is unambiguously theirs); False: every row, satellites at their own radius in the host
P31_VESC_R = "r50star"           # the radius the escape speed is evaluated at for a central: "r50star" | a number [kpc]; a satellite uses its distance to the halo centre (never less than this)
P31_KFRAC, P31_NMIN, P31_NGRID = 0.30, 10, 30
P31_NQ    = 4                    # the quartiles the lines are cut in
P31_POINTS = True
P31_YLIM  = (-6.0, 1.0)
P31_CATDIR = {}                  # {"cis100": "/path/to/Groups", ...} overrides of the config's catalog_dir (the halo reader)
P31_H     = 0.68
_G_KPC_KMS2 = 4.30091e-6         # G in kpc (km/s)^2 / Msun

if "FR29" not in globals() or "_track28" not in globals():
    print("Part 31: needs Parts 28-29 (the dosed anchor rows and the running-median helper). Run them first.")
else:
    def _catdir31(box):
        import json as _json31
        try:
            cfg = _json31.load(open(os.path.expanduser("~/.simbanator/config.json"))); ent = (cfg.get("simulations", cfg)).get(box, {})
        except Exception:
            ent = {}
        _stem = {"cis100": "m100n1024", "cis50": "m50n512", "cis50nox": "m50n512", "cis25": "m25n512"}.get(box, box)
        return P31_CATDIR.get(box, ent.get("catalog_dir")), ent.get("file_format", "%s_{snap:03d}.hdf5" % _stem)

    _HC31 = ["halo_id", "m200c", "r200c_kpc", "halo_mtot", "halo_vc", "r_in_halo_kpc", "central_cat"]

    def _halo31(box, Q):
        """The parent halo of every (snap, gal_id) row: M200c [Msun], R200c [physical kpc], the halo's total mass and circular
        velocity, the galaxy's distance to the halo centre [physical kpc] and the central flag; read from the catalogue files once,
        cached in OUT."""
        cache = os.path.join(OUT, f"agn_states_halo_props_{box}.csv")
        have = pd.read_csv(cache) if os.path.exists(cache) else pd.DataFrame(columns=["snap", "gal_id"] + _HC31)
        known = set(zip(have["snap"].astype(int), have["gal_id"].astype(int)))
        todo = Q[[(int(s), int(g)) not in known for s, g in zip(Q["snap"], Q["gal_id"])]]
        if len(todo):
            cdir, fmt = _catdir31(box); rows, miss = [], []
            for snap in sorted(set(todo["snap"].astype(int))):
                fn = os.path.join(cdir or "", fmt.format(snap=int(snap)))
                ids = todo.loc[todo["snap"].astype(int) == snap, "gal_id"].astype(int).to_numpy()
                try:
                    with h5py.File(fn, "r") as f:
                        sa = f["simulation_attributes"].attrs; a = float(sa["scale_factor"])
                        box_ckpc = float(sa["boxsize"]) if "boxsize" in sa else np.nan            # caesar: comoving kpc
                        g, h = f["galaxy_data"], f["halo_data"]
                        phi = g["parent_halo_index"][:]; cen = g["central"][:]
                        gpos = g["minpotpos"][:] if "minpotpos" in g else g["pos"][:]
                        hpos = h["minpotpos"][:] if "minpotpos" in h else h["pos"][:]
                        m200, r200 = h["dicts/virial_quantities.m200c"][:], h["dicts/virial_quantities.r200c"][:]
                        mtot = h["dicts/masses.total"][:] if "dicts/masses.total" in h else np.full(len(m200), np.nan)
                        vc = h["dicts/virial_quantities.circular_velocity"][:] if "dicts/virial_quantities.circular_velocity" in h else np.full(len(m200), np.nan)
                    for i in ids:
                        k = int(phi[i]); d = gpos[i] - hpos[k]
                        if np.isfinite(box_ckpc) and box_ckpc > 0:
                            d = d - box_ckpc * np.round(d / box_ckpc)                                  # the minimum image
                        rows.append(dict(snap=int(snap), gal_id=int(i), halo_id=k, m200c=float(m200[k]), r200c_kpc=float(r200[k]) * a, halo_mtot=float(mtot[k]),
                                         halo_vc=float(vc[k]), r_in_halo_kpc=float(np.sqrt((d ** 2).sum())) * a, central_cat=int(cen[i])))
                except Exception as e:
                    miss.append(f"snap {snap}: {type(e).__name__}: {e} ({fn}) — {len(ids)} rows")
            if rows:
                have = pd.concat([have, pd.DataFrame(rows)], ignore_index=True) if len(have) else pd.DataFrame(rows); have.to_csv(cache, index=False)
                print(f"  [{box}] parent-halo properties read for {len(rows)} anchor rows -> {cache}")
            for m in miss:
                print(f"  [{box}] halo not read, {m}")
        return Q.merge(have, on=["snap", "gal_id"], how="left")

    def _nfw_c31(m200c, z):
        """Dutton & Maccio 2014 c200c(M200c, z)."""
        m, z = np.asarray(m200c, float), np.asarray(z, float)
        a = 0.520 + (0.905 - 0.520) * np.exp(-0.617 * z ** 1.21); b = -0.101 + 0.026 * z
        with np.errstate(all="ignore"):
            return 10.0 ** (a + b * np.log10(m * P31_H / 1e12))

    def _vesc31(r_kpc, m200c, r200c_kpc, c):
        """The NFW escape speed at r [km/s] (Phi -> 0 at infinity), vectorised."""
        r = np.maximum(np.asarray(r_kpc, float), 1e-3); rs = np.asarray(r200c_kpc, float) / np.asarray(c, float)
        fc = np.log1p(c) - c / (1.0 + c)
        with np.errstate(all="ignore"):
            phi = -_G_KPC_KMS2 * np.asarray(m200c, float) / (r * fc) * np.log1p(r / rs)
            return np.sqrt(2.0 * np.abs(phi))

    def _reservoir31(box, Q):
        """M_dust, M_H2, M_gas and M* on the branch at the reference epoch (P31_EPOCH) of every row, plus the branch peak of M_dust."""
        want = {(int(s), int(g)): i for i, (s, g) in enumerate(zip(Q["snap"], Q["gal_id"]))}
        t_ref = Q["t_qt"].to_numpy(float) if P31_EPOCH == "qt" else Q["t_sft"].to_numpy(float)
        R = {k: np.full(len(Q), np.nan) for k in ("md_ref", "mh_ref", "mg_ref", "ms_ref", "t_ref", "md_peak", "t_peak")}
        fn = HIST_FILE.format(box=box)
        if not os.path.exists(fn):
            return R
        with h5py.File(fn, "r") as f:
            for gname in f:
                a = int(gname[4:]); g = f[gname]; gids = np.asarray(g["gal_ids"][:], dtype=np.int64)
                js = [(j, want[(a, int(gid))]) for j, gid in enumerate(gids) if (a, int(gid)) in want]
                if not js or "mdust" not in g:
                    continue
                t = g["t_yr"][:] / 1e9; o = np.argsort(t); t = t[o]
                ms, md = g["mstar"][:][o], g["mdust"][:][o]
                mh = g["mh2"][:][o] if "mh2" in g else np.full_like(ms, np.nan); mg = g["mgas"][:][o] if "mgas" in g else np.full_like(ms, np.nan)
                for j, i in js:
                    ex = np.isfinite(ms[:, j]) & (ms[:, j] > 0)
                    if ex.sum() < 3:
                        continue
                    mdd = np.where(ex, md[:, j], np.nan)
                    if np.isfinite(mdd).any():
                        kp = int(np.nanargmax(mdd)); R["md_peak"][i], R["t_peak"][i] = mdd[kp], t[kp]
                    if P31_EPOCH == "peak":
                        k = kp if np.isfinite(mdd).any() else None
                    else:
                        k = int(np.argmin(np.abs(np.where(ex, t, np.inf) - t_ref[i]))) if np.isfinite(t_ref[i]) else None
                    if k is not None and ex[k]:
                        R["md_ref"][i], R["mh_ref"][i], R["mg_ref"][i], R["ms_ref"][i], R["t_ref"][i] = md[k, j], mh[k, j], mg[k, j], ms[k, j], t[k]
        return R

    _lg = lambda v: np.log10(np.where(np.asarray(v, float) > 0, np.asarray(v, float), np.nan))

    def _strat31(x, y, s, nb=5):
        """Spearman(x, y) inside quantile bins of s, pooled by N (a partial correlation without a model)."""
        ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(s); x, y, s = x[ok], y[ok], s[ok]
        if len(x) < nb * P31_NMIN:
            return np.nan
        e = np.quantile(s, np.linspace(0, 1, nb + 1)); e[0] -= 1e-9; ib = np.clip(np.searchsorted(e, s, side="right") - 1, 0, nb - 1)
        num = den = 0.0
        for i in range(nb):
            m = ib == i
            if m.sum() >= P31_NMIN:
                r = spearmanr(x[m], y[m])[0]
                if np.isfinite(r):
                    num += r * m.sum(); den += m.sum()
        return num / den if den else np.nan

    WLAB31 = {"log_m200c": r"$\log\, M_{200c}$ of the parent halo [$M_\odot$]", "vesc": r"$v_{\rm esc}$ at the galaxy (NFW) [km s$^{-1}$]",
              "log_mstar": r"$\log\, M_\star$", "log_mbh": r"$\log\, M_{\rm BH}$", "log_halo_mtot": r"$\log\, M_{\rm halo,tot}$", "halo_vc": r"$V_{\rm c}(R_{200c})$ [km s$^{-1}$]"}
    DLAB31 = {"mdust": r"$\log\, M_{\rm dust}({\rm anchor}) / M_{\rm dust}({\rm %s})$" % P31_EPOCH.upper(), "mh2": r"$\log\, M_{\rm H_2}({\rm anchor}) / M_{\rm H_2}({\rm %s})$" % P31_EPOCH.upper(),
              "mgas": r"$\log\, M_{\rm gas}({\rm anchor}) / M_{\rm gas}({\rm %s})$" % P31_EPOCH.upper()}
    FR31, T31 = {}, []
    for _b in P31_BOXES:
        if _b not in FR29:
            print(f"[{_b}] not in Part 29's frames — skipped"); continue
        _Q = FR29[_b].copy()
        if "conc" not in _Q.columns:                                   # Part 29 keeps the concentration in AX29, not in its frame
            _Q["conc"] = _axis29(_Q, "conc")[0] if "_axis29" in globals() else np.nan
        _Q = _halo31(_b, _Q)
        _R = _reservoir31(_b, _Q)
        for _k, _v in _R.items():
            _Q[_k] = _v
        with np.errstate(all="ignore"):
            _Q["depth_mdust"] = _lg(_Q["mdust"]) - _lg(_Q["md_ref"]); _Q["depth_mh2"] = _lg(_Q["mh2"]) - _lg(_Q["mh_ref"]); _Q["depth_mgas"] = _lg(_Q["mgas"]) - _lg(_Q["mg_ref"])
            _Q["depth_peak"] = _lg(_Q["mdust"]) - _lg(_Q["md_peak"])
            _Q["log_m200c"] = _lg(_Q["m200c"]); _Q["log_halo_mtot"] = _lg(_Q["halo_mtot"])
            _c = _nfw_c31(_Q["m200c"].to_numpy(float), _Q["z"].to_numpy(float))
            _r0 = _Q["r_half_star"].to_numpy(float) if P31_VESC_R == "r50star" else np.full(len(_Q), float(P31_VESC_R))
            _r = np.where(_Q["central_cat"].to_numpy(float) > 0.5, _r0, np.maximum(_Q["r_in_halo_kpc"].to_numpy(float), _r0))
            _Q["vesc"] = _vesc31(_r, _Q["m200c"].to_numpy(float), _Q["r200c_kpc"].to_numpy(float), _c)
            _Q["log_mg_ref"] = _lg(_Q["mg_ref"]); _Q["log_md_ref"] = _lg(_Q["md_ref"])
        if P31_CENTRALS:
            _Q = _Q[_Q["central_cat"] > 0.5].reset_index(drop=True)
        FR31[_b] = _Q
        _d = _Q["dusty"].to_numpy(bool); _fx = _Q[P31_DOSE].to_numpy(float); _lm = _Q["log_mstar"].to_numpy(float)
        _nref = int(np.isfinite(_Q["depth_mdust"]).sum())
        # print(f"[{_b}] {len(_Q)} rows{' (centrals)' if P31_CENTRALS else ''}, {int(np.isfinite(_Q['m200c']).sum())} with a parent halo, {_nref} with a dust depth (epoch {P31_EPOCH}); "
        #       f"median depth: dust {np.nanmedian(_Q['depth_mdust']):+.2f}, H2 {np.nanmedian(_Q['depth_mh2']):+.2f}, gas {np.nanmedian(_Q['depth_mgas']):+.2f} dex; "
        #       f"dusty rows {np.nanmedian(_Q.loc[_d, 'depth_mdust']):+.2f} vs dust-poor {np.nanmedian(_Q.loc[~_d, 'depth_mdust']):+.2f}; "
        #       f"centrals {100 * np.nanmean(_Q['central_cat']):.0f} %, median log M200c {np.nanmedian(_Q['log_m200c']):.2f}, v_esc {np.nanmedian(_Q['vesc']):.0f} km/s")
        _cands = [(P31_DOSE, _fx), ("log M*", _lm), ("log M_BH", _Q["log_mbh"].to_numpy(float)), ("log M200c", _Q["log_m200c"].to_numpy(float)),
                  ("v_esc", _Q["vesc"].to_numpy(float)), ("log M_halo,tot", _Q["log_halo_mtot"].to_numpy(float)), ("V_c(R200c)", _Q["halo_vc"].to_numpy(float)),
                  ("central", _Q["central_cat"].to_numpy(float)), ("conc (R80/R20)", _Q["conc"].to_numpy(float)), (f"log M_gas({P31_EPOCH})", _Q["log_mg_ref"].to_numpy(float)),
                  (f"log M_dust({P31_EPOCH})", _Q["log_md_ref"].to_numpy(float)), ("age", _Q["age"].to_numpy(float)), ("t_quiet", _Q["t_quiet"].to_numpy(float))]
        for _dep, _dl in (("depth_mdust", "dust"), ("depth_mh2", "H2")):
            _y = _Q[_dep].to_numpy(float)
            print(f"   Spearman with the {_dl} depth:   raw | at fixed {P31_DOSE} | at fixed log M* | at fixed both (3x3)")
            for _nm, _x in _cands:
                _ok = np.isfinite(_x) & np.isfinite(_y)
                if _ok.sum() < 2 * P31_NMIN:
                    continue
                _raw = spearmanr(_x[_ok], _y[_ok])[0]
                # both fixed: 3 x 3 cells of duty x mass
                _o2 = _ok & np.isfinite(_fx) & np.isfinite(_lm); _num = _den = 0.0
                if _o2.sum() >= 9 * P31_NMIN:
                    _e1 = np.quantile(_fx[_o2], [0, 1 / 3, 2 / 3, 1]); _e2 = np.quantile(_lm[_o2], [0, 1 / 3, 2 / 3, 1]); _e1[0] -= 1e-9; _e2[0] -= 1e-9
                    _i1 = np.clip(np.searchsorted(_e1, _fx, side="right") - 1, 0, 2); _i2 = np.clip(np.searchsorted(_e2, _lm, side="right") - 1, 0, 2)
                    for _p in range(3):
                        for _q in range(3):
                            _m = _o2 & (_i1 == _p) & (_i2 == _q)
                            if _m.sum() >= P31_NMIN:
                                _rr = spearmanr(_x[_m], _y[_m])[0]
                                if np.isfinite(_rr):
                                    _num += _rr * _m.sum(); _den += _m.sum()
                _both = _num / _den if _den else np.nan
                _s1 = _strat31(_x, _y, _fx) if _nm != P31_DOSE else np.nan; _s2 = _strat31(_x, _y, _lm) if _nm != "log M*" else np.nan
                print(f"     {_nm:18s} {_raw:+.2f}   {_s1:+.2f}   {_s2:+.2f}   {_both:+.2f}")
                T31.append(dict(box=_b, depth=_dl, axis=_nm, n=int(_ok.sum()), rho=_raw, rho_fixed_dose=_s1, rho_fixed_mstar=_s2, rho_fixed_both=_both))

    _bx = [b for b in P31_BOXES if b in FR31]
    if _bx:
        _QC = plt.get_cmap("viridis")(np.linspace(0.1, 0.9, P31_NQ))
        _rows = [("depth_mdust", P31_DOSE, "log_mstar", r"$\log M_\star$ quartile"), ("depth_mdust", P31_WELL[0], P31_DOSE, "duty quartile"),
                 ("depth_mdust", P31_WELL[1], P31_DOSE, "duty quartile"), ("depth_mh2", P31_WELL[0], P31_DOSE, "duty quartile")]
        fig, axs = plt.subplots(len(_rows), len(_bx), figsize=(4.9 * len(_bx), 3.7 * len(_rows)), squeeze=False, sharey="row", constrained_layout=True)
        for _i, (_yc, _xc, _sc, _slab) in enumerate(_rows):
            for _j, _b in enumerate(_bx):
                ax = axs[_i, _j]; _Q = FR31[_b]
                _x, _y, _s = _Q[_xc].to_numpy(float), _Q[_yc].to_numpy(float), _Q[_sc].to_numpy(float)
                _ok = np.isfinite(_x) & np.isfinite(_y) & np.isfinite(_s)
                if _ok.sum() < 2 * P31_NMIN:
                    ax.text(0.5, 0.5, f"{int(_ok.sum())} rows — nothing to draw", transform=ax.transAxes, ha="center", va="center", color="0.4"); ax.set_axis_off(); continue
                if P31_POINTS:
                    ax.scatter(_x[_ok], _y[_ok], s=5, color="0.55", alpha=0.25, lw=0, rasterized=True, zorder=1)
                _e = np.quantile(_s[_ok], np.linspace(0, 1, P31_NQ + 1)); _e[0] -= 1e-9
                _ib = np.clip(np.searchsorted(_e, _s, side="right") - 1, 0, P31_NQ - 1)
                for _q in range(P31_NQ):
                    _m = _ok & (_ib == _q)
                    _tr = _track28(_x[_m], _y[_m], kfrac=P31_KFRAC, nmin=P31_NMIN, ngrid=P31_NGRID)
                    if _tr is None:
                        continue
                    ax.fill_between(_tr["x"], _tr["lo"], _tr["hi"], color=_QC[_q], alpha=0.10, lw=0, zorder=2)
                    ax.plot(_tr["x"], _tr["med"], color=_QC[_q], lw=2.2, zorder=3, label=f"{_slab} {_q + 1}: {_e[_q]:.2f}-{_e[_q + 1]:.2f}  ({int(_m.sum())})")
                ax.axhline(0, color="0.3", lw=0.8, ls=":", zorder=2)
                ax.set_ylim(*P31_YLIM); ax.grid(alpha=0.2, lw=0.5)
                ax.set_xlabel(WLAB31.get(_xc, globals().get("DLAB28", {}).get(_xc, _xc)))
                if _j == 0:
                    ax.set_ylabel(DLAB31.get(_yc.replace("depth_", ""), _yc))
                if _i == 0:
                    ax.set_title(f"{BOX_LABEL.get(_b, _b)}: {int(np.isfinite(_Q['depth_mdust']).sum())} rows with a depth", fontsize=9.5)
                ax.legend(frameon=False, fontsize=6.8, loc="lower right", title=f"lines: {_slab}s, running median + 16-84", title_fontsize=6.8)
                _rr = spearmanr(_x[_ok], _y[_ok])[0]
                ax.text(0.02, 0.97, r"$\rho$" + f" = {_rr:+.2f}", transform=ax.transAxes, fontsize=8, va="top", bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.7", lw=0.5, alpha=0.9))
        fig.suptitle(f"Part 31 — the removal depth since {P31_EPOCH.upper()}: the dust (rows 1-3) and the H$_2$ (row 4) against the channel and against the well"
                     f"{' (centrals only)' if P31_CENTRALS else ''}\n(1) the mass offset at every dose; (2-3) the well at fixed dose; (4) the gas the dust goes with", fontsize=11)
        paper_save(fig, "agn_states_removal_depth")
        plt.show()
        pd.DataFrame(T31).to_csv(os.path.join(OUT, "agn_states_removal_depth.csv"), index=False)
        pd.concat([FR31[b].assign(box=b)[["box", "snap", "gal_id", "z", "log_mstar", "log_mbh", "pop", P31_DOSE, "e_x", "conc", "log_m200c", "r200c_kpc", "r_in_halo_kpc", "central_cat", "vesc",
                                          "t_ref", "log_md_ref", "log_mg_ref", "depth_mdust", "depth_mh2", "depth_mgas", "depth_peak"]] for b in _bx], ignore_index=True
                  ).to_csv(os.path.join(OUT, "agn_states_removal_depth_galaxies.csv"), index=False)
        print(f"-> {os.path.join(OUT, 'agn_states_removal_depth{,_galaxies}.csv')}")

## Part 32 — the division on trial: a score instead of two thresholds

Parts 28–31 asked whether the dusty / dust-poor split (the label at `DUSTY_CUT`) can be made on its causes, and found the dose cut sitting on the flat part of the response (Part 29), the gas concentration as a second axis (Parts 29–30) and the stellar mass as a third that no dose definition absorbs (Part 31: below $\sim 10^{10.9}\,M_\odot$ the dose does not matter, above it the duty fraction and the well decide). This part scores every candidate divider on the same footing and draws the one that wins. The dividers are **single properties** (the Mann–Whitney AUC toward dusty; among them the catalogue whole-galaxy $\kappa_{\rm rot}$ of the gas, `kappa_gas` — the axis Part 22 splits the coarse boxes on, missing from the first version of this ladder), **linear scores** of two to four properties (`P32_SETS`; a ridge logistic regression by Newton steps, hand-rolled — no sklearn on the cluster kernel — judged by the out-of-fold AUC of a repeated stratified $k$-fold, `P32_KFOLD` × `P32_NREP`), and **two-cut AND rules** found by a blind quantile scan on a pair of axes (the largest Youden $J$ = recall − false-positive rate, the model-free counterpart of a depth-2 tree). The threshold of a score is set by the data, not by hand: `P32_THRESH = "prior"` selects the rows whose out-of-fold probability exceeds the box's dusty fraction, `"youden"` the cut of largest $J$, a number a fixed probability.

The figure has one column per box: (1) the `P32_PLANE` plane, duty fraction × $\log M_\star$, every row a point (dusty on top), the drawn score's (`P32_SCORE`, default duty + $\kappa_{\rm gas}$ + $\log M_\star$) iso-probability line at the threshold with the other terms at their medians, the best two-cut rule of the plane as a dashed rectangle with its $N$ / dusty share / recall, the current dose cut dotted and the current rule's members (duty < `CUT28` and concentration > `P32_CCUT`) ringed; (2) the **dusty fraction along each candidate score** (`P32_CURVES`, out-of-fold, equal-count bins of the score's percentile, Wilson bands) — steeper is a better divider, the legend carries the AUC, the dotted line the prior, the dashed one where the drawn score's threshold falls; (3) $\log M_{\rm dust}/M_\star$ against the drawn score's log-odds, running median + 16–84: the gradient the classes cut (in-sample: the score was fitted to the label of this axis, so read it as a display, not a test). Printed and saved: the AUC ladder (`agn_states_division_score.csv`), the rules table — the current rule, Part 22's red group (duty < `CUT28` and $\kappa_{\rm gas}$ above its median), the dose alone, the score at its threshold, the score at the current rule's size, the best two-cut rules on (duty, $M_\star$), (duty, concentration), (concentration, $M_\star$), ($\kappa_{\rm gas}$, $M_\star$), (duty, $\kappa_{\rm gas}$) — with $N$, the dusty share inside, the share of all dusty galaxies recovered and $J$ (`agn_states_division_rules.csv`), and the fitted coefficients, standardised and raw (`agn_states_division_coef.csv`). Products `agn_states_division_score.{png,pdf}`.

**What it says (2026-09-08, with $\kappa_{\rm gas}$ in the ladder).** The first version scored the concentration of Parts 29–31 as the geometry axis and found the stellar mass the missing one. With Part 22's own coarse-box axis in the list the picture is sharper: the whole-galaxy $\kappa_{\rm gas}$ is the strongest single divider in m100 and m50 (AUC $\approx 0.8$), it is uncorrelated with $M_\star$ and holds inside every mass quartile and every gas-particle-count bin (it is not the few-particle artefact of the m25 core $\kappa$, Part 22's section on the geometry axis), $\kappa_{\rm gas} + M_\star$ *without any dose* beats every dose-bearing pair, and the duty fraction adds nothing on top of the two — Part 21's chain (AGN history $\to$ gas structure $\to$ dust) read as a scoreboard: the dose is already written into the gas kinematics. The concentration $R_{80}/R_{20}$ was a weak proxy of the same thing. The blind two-cut on ($\kappa_{\rm gas}$, $M_\star$) lands above the isotropic $1/3$. So Part 22's division is not replaced, it gains a third axis (Part 33 draws it). In m25 the Part 32 frame is Part 0's sample, a fraction of the KS-track frame Parts 22–23 use, and the catalogue $\kappa_{\rm gas}$ there is the whole-galaxy one; re-run for the numbers.

In [ ]:
# ── Part 32 — THE DIVISION ON TRIAL: a score instead of two thresholds ──
# Parts 28-31 asked whether the dusty / dust-poor split can be made on its causes. This part scores every candidate divider on the
# same footing and draws the one that wins. The dividers are (a) single properties, (b) linear scores of two to four properties
# fitted as a logistic regression (hand-rolled Newton steps with a ridge, no sklearn on the cluster kernel), judged by the
# out-of-fold AUC of repeated stratified k-fold (P32_KFOLD x P32_NREP), and (c) two-cut AND rules found by a blind quantile scan
# on a pair of axes (largest Youden J = recall - false-positive rate), the model-free counterpart of the tree.  The threshold of
# a score is set by the data, not by hand: P32_THRESH = "prior" selects the rows whose out-of-fold probability exceeds the box's
# dusty fraction (the class prior), "youden" the cut of largest J, a number a fixed probability.
#   FIG  rows: (1) the P32_PLANE plane (duty fraction x log M*) with the rows as points, the drawn score's iso-probability line at
#              the threshold (P32_SCORE, the other terms at their medians), the best two-cut rule of the plane (dashed
#              rectangle) and the current rule's members ringed;  (2) the dusty fraction along each candidate score
#              (out-of-fold, equal-count bins of the score's percentile — steeper = a better divider; the legend carries the AUC);
#              (3) log M_dust/M* against the drawn score (the log-odds), running median + 16-84: the gradient the classes cut.
#   PRINTED  the AUC ladder per box, the fitted coefficients of the drawn score (standardised and raw), the rules table: the current
#            rule (duty < CUT28 & concentration > P32_CCUT), Part 22's red group (duty < CUT28 & kappa_gas > its median), the dose
#            alone, the score at the threshold, the score at the current rule's size, the best two-cut rules on (duty, M*),
#            (duty, concentration), (concentration, M*), (kappa_gas, M*), (duty, kappa_gas).
#   The candidates include the catalogue whole-galaxy kappa_rot of the gas ("kappa_gas", caesar's rotation.gas_kappa_rot): it is
#   the geometry axis Part 22 splits the coarse boxes on, and it was missing from the first version of this ladder (2026-09-08).
P32_BOXES = ["cis100", "cis50", "cis25"]
P32_DOSE  = globals().get("P31_DOSE", globals().get("P28_DOSE", "f_x_post"))
P32_SETS  = {                    # name -> the columns of a linear score; "dxm" = the (duty - mean) x (log M* - mean) interaction
    "duty": [P32_DOSE], "conc": ["conc"], "M*": ["log_mstar"], "M200c": ["log_m200c"], "age": ["age"],
    "duty+conc (current pair)": [P32_DOSE, "conc"], "duty+M*": [P32_DOSE, "log_mstar"], "duty+M200c": [P32_DOSE, "log_m200c"],
    "duty+M*+dutyxM*": [P32_DOSE, "log_mstar", "dxm"], "duty+conc+M*": [P32_DOSE, "conc", "log_mstar"],
    "duty+conc+M200c": [P32_DOSE, "conc", "log_m200c"], "duty+conc+M*+age": [P32_DOSE, "conc", "log_mstar", "age"],
    "conc+M*+age (no dose)": ["conc", "log_mstar", "age"],
    "kappa_gas": ["kappa_gas"], "duty+kappa_gas (Part 22 pair)": [P32_DOSE, "kappa_gas"], "kappa_gas+M* (no dose)": ["kappa_gas", "log_mstar"],
    "duty+kappa_gas+M*": [P32_DOSE, "kappa_gas", "log_mstar"], "duty+kappa_gas+conc+M*": [P32_DOSE, "kappa_gas", "conc", "log_mstar"],
    "kappa_gas+M*+age (no dose)": ["kappa_gas", "log_mstar", "age"],
    "all": [P32_DOSE, "e_x", "kappa_gas", "conc", "log_mstar", "log_mbh", "log_m200c", "age", "central_cat"]}
P32_SCORE = "duty+kappa_gas+M*"  # the score drawn in rows 1 and 3 and thresholded in the rules table (a key of P32_SETS)
P32_CURVES = ["duty", "duty+conc (current pair)", "duty+kappa_gas (Part 22 pair)", "duty+M*", "duty+conc+M*", "kappa_gas+M* (no dose)", "duty+kappa_gas+M*"]   # row 2
P32_PLANE = (P32_DOSE, "log_mstar")   # the plane of row 1: (x, y); the two-cut rule keeps x below and y above its cuts
P32_KFOLD, P32_NREP = 5, 5       # the repeated stratified k-fold behind every out-of-fold number
P32_RIDGE = 1.0                  # the ridge on the standardised coefficients (sklearn's C = 1 on the summed log-loss)
P32_THRESH = "youden"            # the score's threshold: "prior" (p >= the box's dusty fraction) | "youden" | a probability
P32_CCUT  = P30_CCUT if isinstance(globals().get("P30_CCUT"), (int, float)) else 0.30   # the current rule's concentration cut (Part 30's line)
P32_SCAN, P32_NSCAN, P32_NMIN = (0.10, 0.90), 17, 15   # the quantile grid of the two-cut scan and the smallest selection it may return
P32_NBINS = 10                   # equal-count bins of row 2
P32_MS    = 9
P32_YLIM3 = (-6.5, -1.5)
CURVC32 = {"duty": "#7570b3", "duty+conc (current pair)": "#e6ab02", "duty+kappa_gas (Part 22 pair)": "#e7298a", "duty+M*": "#1b9e77", "duty+conc+M*": "#d95f02",
           "kappa_gas+M* (no dose)": "#66a61e", "duty+kappa_gas+M*": "k", "duty+conc+M*+age": "0.45", "conc+M*+age (no dose)": "0.6"}

if "FR31" not in globals() and "FR29" not in globals():
    print("Part 32: needs Part 31's frames (or Part 29's). Run them first.")
else:
    _wil32 = globals().get("_wilson29")
    if _wil32 is None:
        def _wil32(k, n, z=1.0):
            if n <= 0:
                return np.nan, np.nan, np.nan
            p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
            return p, min(p, max(0.0, c - h)), max(p, min(1.0, c + h))

    def _logit32(X, y, ridge=P32_RIDGE, iters=60):
        """A ridge logistic regression on standardised columns by Newton steps: {beta (intercept first), mu, sd}."""
        X = np.asarray(X, float); y = np.asarray(y, float)
        mu, sd = X.mean(0), X.std(0); sd = np.where(sd > 0, sd, 1.0)
        Z = np.column_stack([np.ones(len(X)), (X - mu) / sd]); k = Z.shape[1]
        beta = np.zeros(k); pen = np.full(k, float(ridge)); pen[0] = 0.0
        for _ in range(iters):
            p = 1.0 / (1.0 + np.exp(-np.clip(Z @ beta, -30, 30))); w = p * (1.0 - p)
            g = Z.T @ (y - p) - pen * beta; H = (Z * w[:, None]).T @ Z + np.diag(pen) + 1e-9 * np.eye(k)
            step = np.linalg.solve(H, g); beta = beta + step
            if np.abs(step).max() < 1e-9:
                break
        return dict(beta=beta, mu=mu, sd=sd)

    def _pred32(m, X):
        Z = np.column_stack([np.ones(len(X)), (np.asarray(X, float) - m["mu"]) / m["sd"]])
        return 1.0 / (1.0 + np.exp(-np.clip(Z @ m["beta"], -30, 30)))

    def _lodds32(m, X):
        Z = np.column_stack([np.ones(len(X)), (np.asarray(X, float) - m["mu"]) / m["sd"]])
        return Z @ m["beta"]

    def _folds32(y, k, seed):
        rng = np.random.default_rng(seed); f = np.empty(len(y), int)
        for c in (0, 1):
            idx = np.flatnonzero(y == c); rng.shuffle(idx); f[idx] = np.arange(len(idx)) % k
        return f

    def _cv32(X, y, k=P32_KFOLD, nrep=P32_NREP):
        """Out-of-fold probabilities (averaged over the repeats) and the mean / sd of the per-repeat AUC."""
        X = np.asarray(X, float); y = np.asarray(y, int); P = np.zeros(len(y)); aucs = []
        for r in range(nrep):
            f = _folds32(y, k, r); p = np.zeros(len(y))
            for i in range(k):
                tr, te = f != i, f == i
                if te.sum() == 0:
                    continue
                if y[tr].min() == y[tr].max():
                    p[te] = y[tr].mean(); continue
                p[te] = _pred32(_logit32(X[tr], y[tr]), X[te])
            aucs.append(mw_auc(p, y > 0)[0]); P += p
        return P / nrep, float(np.nanmean(aucs)), float(np.nanstd(aucs))

    def _rule32(sel, y):
        sel = np.asarray(sel, bool); y = np.asarray(y, bool); n = int(sel.sum()); nd = int((sel & y).sum())
        return dict(n=n, n_dusty=nd, dusty_in=nd / max(n, 1), recall=nd / max(int(y.sum()), 1), fpr=int((sel & ~y).sum()) / max(int((~y).sum()), 1),
                    J=nd / max(int(y.sum()), 1) - int((sel & ~y).sum()) / max(int((~y).sum()), 1))

    def _scan32(a, sa, b, sb, y):
        """The two-cut AND rule (a on its `sa` side of a cut, b on its `sb` side) of largest Youden J on a quantile grid; None if nothing qualifies."""
        ok = np.isfinite(a) & np.isfinite(b); best = None
        for qa in np.linspace(*P32_SCAN, P32_NSCAN):
            ca = np.nanquantile(a[ok], qa); ma = (a > ca) if sa > 0 else (a < ca)
            for qb in np.linspace(*P32_SCAN, P32_NSCAN):
                cb = np.nanquantile(b[ok], qb); sel = ok & ma & ((b > cb) if sb > 0 else (b < cb))
                if sel.sum() < P32_NMIN:
                    continue
                r = _rule32(sel, y)
                if best is None or r["J"] > best["J"]:
                    best = dict(r, cut_a=float(ca), cut_b=float(cb), sel=sel)
        return best

    _sym32 = lambda s: ">" if s > 0 else "<"
    AXL32 = {P32_DOSE: globals().get("DLAB28", {}).get(P32_DOSE, "duty fraction since SFT"), "log_mstar": r"$\log\, M_\star$", "conc": r"$\log\, R_{80}/R_{20}$ (gas)",
             "log_m200c": r"$\log\, M_{200c}$", "age": "age since SFT [Gyr]", "kappa_gas": r"$\kappa_{\rm rot}$(gas), whole galaxy"}
    _short32 = {P32_DOSE: "duty", "log_mstar": "log M*", "conc": "conc", "log_m200c": "log M200c", "age": "age", "e_x": "E_x", "log_mbh": "log M_BH", "central_cat": "central", "dxm": "duty x M*",
                "kappa_gas": "kappa_gas", "kappa_star": "kappa_star"}
    T32, R32, C32, S32 = [], [], [], {}
    for _b in P32_BOXES:
        _src = FR31 if ("FR31" in globals() and _b in FR31) else (FR29 if _b in FR29 else None)
        if _src is None:
            print(f"[{_b}] not in Part 31's / Part 29's frames — skipped"); continue
        _Q = _src[_b].copy()
        if "conc" not in _Q.columns:
            _Q["conc"] = _axis29(_Q, "conc")[0] if "_axis29" in globals() else np.nan
        for _c in ("log_m200c", "e_x", "log_mbh", "central_cat", "age", "kappa_gas", "kappa_star"):
            if _c not in _Q.columns:
                _Q[_c] = np.nan
        with np.errstate(all="ignore"):
            _Q["dxm"] = (_Q[P32_DOSE] - np.nanmean(_Q[P32_DOSE])) * (_Q["log_mstar"] - np.nanmean(_Q["log_mstar"]))
        _y = _Q["dusty"].to_numpy(bool); _prior = float(_y.mean())
        print(f"[{_b}] {len(_Q)} anchor rows, {int(_y.sum())} dusty ({100 * _prior:.1f} %), current dose cut {CUT28.get(_b, np.nan):.2f}, concentration cut {P32_CCUT:.2f}")
        # ── (a) single properties, oriented toward dusty ──
        _sing = [(c, mw_auc(_Q[c].to_numpy(float), _y)[0]) for c in (P32_DOSE, "e_x", "kappa_gas", "kappa_star", "conc", "log_mstar", "log_mbh", "log_m200c", "age", "central_cat") if np.isfinite(_Q[c]).sum() >= 2 * P32_NMIN]
        print("   single properties, AUC toward dusty (> 0.5: high -> dusty): " + "  ".join(f"{_short32.get(c, c)} {a:.2f}" for c, a in _sing))
        for c, a in _sing:
            T32.append(dict(box=_b, divider=_short32.get(c, c), kind="single", columns=c, n=int(np.isfinite(_Q[c]).sum()), auc=a, auc_sd=np.nan))
        # ── (b) the linear scores, out-of-fold ──
        print(f"   linear scores, out-of-fold AUC ({P32_KFOLD}-fold x {P32_NREP}):")
        _P = {}
        for _nm, _cols in P32_SETS.items():
            _ok = np.isfinite(_Q[_cols].to_numpy(float)).all(axis=1)
            if _ok.sum() < 4 * P32_NMIN or _y[_ok].sum() < 5 or (~_y[_ok]).sum() < 5:
                print(f"     {_nm:28s} skipped ({int(_ok.sum())} usable rows)"); continue
            _p, _a, _s = _cv32(_Q.loc[_ok, _cols].to_numpy(float), _y[_ok].astype(int))
            _P[_nm] = (_ok, _p)
            print(f"     {_nm:28s} {_a:.3f} +- {_s:.3f}   (n = {int(_ok.sum())})")
            T32.append(dict(box=_b, divider=_nm, kind="score", columns="+".join(_cols), n=int(_ok.sum()), auc=_a, auc_sd=_s))
        # ── the drawn score: its fit and its threshold ──
        _cols = P32_SETS[P32_SCORE]; _ok = np.isfinite(_Q[_cols].to_numpy(float)).all(axis=1)
        if P32_SCORE not in _P:
            print(f"   the drawn score {P32_SCORE} could not be fitted — no rules, no figure column"); continue
        _m = _logit32(_Q.loc[_ok, _cols].to_numpy(float), _y[_ok].astype(int))
        _braw = _m["beta"][1:] / _m["sd"]; _b0 = _m["beta"][0] - np.sum(_m["beta"][1:] * _m["mu"] / _m["sd"])
        print(f"   the drawn score {P32_SCORE}: log-odds = {_b0:+.2f} " + " ".join(f"{bb:+.2f} x {_short32.get(c, c)}" for bb, c in zip(_braw, _cols))
              + "   (standardised: " + ", ".join(f"{bb:+.2f} {_short32.get(c, c)}" for bb, c in zip(_m["beta"][1:], _cols)) + ")")
        for c, bs, br in zip(_cols, _m["beta"][1:], _braw):
            C32.append(dict(box=_b, score=P32_SCORE, term=c, beta_std=bs, beta_raw=br, intercept_raw=_b0))
        if len(_cols) >= 2:
            print("     one step of 0.1 in the duty fraction is worth " + ", ".join(f"{abs(0.1 * _braw[0] / bb):.2f} in {_short32.get(c, c)}" for bb, c in zip(_braw[1:], _cols[1:]) if bb != 0))
        _pcv = np.full(len(_Q), np.nan); _pcv[_P[P32_SCORE][0]] = _P[P32_SCORE][1]
        if P32_THRESH == "prior":
            _thr = _prior
        elif P32_THRESH == "youden":
            _cand = np.unique(_pcv[np.isfinite(_pcv)]); _thr = max(_cand, key=lambda t: _rule32(_pcv >= t, _y)["J"])
        else:
            _thr = float(P32_THRESH)
        _lo = _lodds32(_m, np.nan_to_num(_Q[_cols].to_numpy(float), nan=0.0)); _lo[~_ok] = np.nan
        S32[_b] = dict(Q=_Q, y=_y, m=_m, cols=_cols, pcv=_pcv, thr=_thr, lodds=_lo, P=_P, prior=_prior)
        # ── (c) the rules table ──
        _dose = _Q[P32_DOSE].to_numpy(float); _conc = _Q["conc"].to_numpy(float); _lm = _Q["log_mstar"].to_numpy(float); _dc = CUT28.get(_b, np.nan)
        _kg = _Q["kappa_gas"].to_numpy(float); _kgc = float(np.nanmedian(_kg)) if np.isfinite(_kg).any() else np.nan
        _rules = [("current: duty < %.2f & conc > %.2f" % (_dc, P32_CCUT), (_dose < _dc) & (_conc > P32_CCUT)),
                  ("Part 22's red group: duty < %.2f & kappa_gas > %.2f (its median)" % (_dc, _kgc), (_dose < _dc) & (_kg > _kgc)),
                  ("dose alone: duty < %.2f" % _dc, _dose < _dc),
                  (f"score {P32_SCORE} >= {P32_THRESH} ({_thr:.2f})", _pcv >= _thr)]
        _ncur = int(_rules[0][1].sum())
        if _ncur >= 1 and np.isfinite(_pcv).sum() >= _ncur:
            _t = np.sort(_pcv[np.isfinite(_pcv)])[::-1][_ncur - 1]; _rules.append((f"score {P32_SCORE}, the current rule's size", _pcv >= _t))
        _xa, _ya = P32_PLANE; _best = {}
        for _lab, (_a, _sa), (_bb, _sb) in ((f"best two-cut on ({_short32.get(_xa, _xa)}, {_short32.get(_ya, _ya)})", (_xa, -1), (_ya, +1)),
                                            ("best two-cut on (duty, conc)", (P32_DOSE, -1), ("conc", +1)), ("best two-cut on (conc, log M*)", ("conc", +1), ("log_mstar", +1)),
                                            ("best two-cut on (kappa_gas, log M*)", ("kappa_gas", +1), ("log_mstar", +1)), ("best two-cut on (duty, kappa_gas)", (P32_DOSE, -1), ("kappa_gas", +1))):
            _r = _scan32(_Q[_a].to_numpy(float), _sa, _Q[_bb].to_numpy(float), _sb, _y)
            if _r is not None:
                _best[_lab] = dict(_r, a=_a, sa=_sa, b=_bb, sb=_sb)
                _rules.append((f"{_lab}: {_short32.get(_a, _a)} {_sym32(_sa)} {_r['cut_a']:.2f} & {_short32.get(_bb, _bb)} {_sym32(_sb)} {_r['cut_b']:.2f}", _r["sel"]))
        S32[_b]["best"] = _best; S32[_b]["current"] = _rules[0][1]
        print("   rules (n selected | dusty share inside | share of all dusty recovered | Youden J):")
        for _lab, _sel in _rules:
            _r = _rule32(np.nan_to_num(_sel.astype(float), nan=0.0) > 0.5, _y)
            print(f"     {_lab:62s} n = {_r['n']:4d} | {100 * _r['dusty_in']:3.0f} % | {100 * _r['recall']:3.0f} % | J = {_r['J']:.2f}")
            R32.append(dict(box=_b, rule=_lab, **{k: v for k, v in _r.items()}))

    _bx = [b for b in P32_BOXES if b in S32]
    if _bx:
        fig, axs = plt.subplots(3, len(_bx), figsize=(4.9 * len(_bx), 3.8 * 3), squeeze=False, constrained_layout=True)
        _xa, _ya = P32_PLANE
        for _j, _b in enumerate(_bx):
            _S = S32[_b]; _Q, _y, _m, _cols = _S["Q"], _S["y"], _S["m"], _S["cols"]
            # ── row 1: the plane ──
            ax = axs[0, _j]; _x, _yy = _Q[_xa].to_numpy(float), _Q[_ya].to_numpy(float); _ok = np.isfinite(_x) & np.isfinite(_yy)
            ax.scatter(_x[_ok & ~_y], _yy[_ok & ~_y], s=P32_MS, color=POPS["pure"]["color"], alpha=0.45, lw=0, zorder=2, rasterized=True, label=f"dust-poor ({int((_ok & ~_y).sum())})")
            ax.scatter(_x[_ok & _y], _yy[_ok & _y], s=P32_MS + 3, color=POPS["dusty"]["color"], alpha=0.85, lw=0, zorder=3, label=f"dusty ({int((_ok & _y).sum())})")
            _cur = _S["current"] & _ok
            if _cur.sum():
                ax.scatter(_x[_cur], _yy[_cur], s=P32_MS * 5, facecolor="none", edgecolors="k", lw=0.7, zorder=4, label=f"current rule ({int(_cur.sum())})")
            _xl = (max(0.0, np.nanmin(_x) - 0.02), min(1.0, np.nanmax(_x) + 0.02)) if _xa == P32_DOSE else (np.nanmin(_x), np.nanmax(_x))
            _yl = (np.nanmin(_yy) - 0.05, np.nanmax(_yy) + 0.12)
            # the drawn score's iso-probability line at the threshold, the other columns at their medians
            if _xa in _cols and _ya in _cols:
                _ix, _iy = _cols.index(_xa), _cols.index(_ya); _braw = _m["beta"][1:] / _m["sd"]; _b0 = _m["beta"][0] - np.sum(_m["beta"][1:] * _m["mu"] / _m["sd"])
                _rest = sum(_braw[k] * np.nanmedian(_Q[c]) for k, c in enumerate(_cols) if k not in (_ix, _iy))
                _xg = np.linspace(*_xl, 100)
                if _braw[_iy] != 0:
                    _yg = (np.log(_S["thr"] / (1 - _S["thr"])) - _b0 - _rest - _braw[_ix] * _xg) / _braw[_iy]
                    _others = ", ".join(f"{_short32.get(c, c)} at its median" for k, c in enumerate(_cols) if k not in (_ix, _iy))
                    ax.plot(_xg, _yg, color="k", lw=2.0, zorder=5, label=f"score = threshold ({_others})" if _others else "score = threshold")
                    ax.fill_between(_xg, _yg, _yl[1] + 1, color="k", alpha=0.05, lw=0, zorder=1)
            _bk = f"best two-cut on ({_short32.get(_xa, _xa)}, {_short32.get(_ya, _ya)})"
            if _bk in _S["best"]:
                _r = _S["best"][_bk]
                ax.axvline(_r["cut_a"], color="0.25", ls="--", lw=1.2, zorder=5); ax.axhline(_r["cut_b"], color="0.25", ls="--", lw=1.2, zorder=5)
                _rx = (_xl[0], _r["cut_a"]) if _r["sa"] < 0 else (_r["cut_a"], _xl[1]); _ry = (_r["cut_b"], _yl[1]) if _r["sb"] > 0 else (_yl[0], _r["cut_b"])
                ax.add_patch(plt.Rectangle((_rx[0], _ry[0]), _rx[1] - _rx[0], _ry[1] - _ry[0], fill=False, ec="0.25", ls="--", lw=1.2, zorder=5))
                ax.text(_rx[0] + 0.01 * (_xl[1] - _xl[0]), _ry[1] - 0.01, f"two-cut rule: {_short32.get(_xa, _xa)} {_sym32(_r['sa'])} {_r['cut_a']:.2f}, {_short32.get(_ya, _ya)} {_sym32(_r['sb'])} {_r['cut_b']:.2f}\n"
                        f"n = {_r['n']}, {100 * _r['dusty_in']:.0f} % dusty, {100 * _r['recall']:.0f} % of the dusty", fontsize=7, ha="left", va="top", zorder=6,
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.7", lw=0.5, alpha=0.9))
            if np.isfinite(CUT28.get(_b, np.nan)):
                ax.axvline(CUT28[_b], color="0.5", ls=":", lw=1.0, zorder=4, label=f"current dose cut {CUT28[_b]:.2f}")
            _rs = _rule32(_S["pcv"] >= _S["thr"], _y)
            ax.text(0.98, 0.03, f"score >= {P32_THRESH} ({_S['thr']:.2f}): n = {_rs['n']}, {100 * _rs['dusty_in']:.0f} % dusty, {100 * _rs['recall']:.0f} % of the dusty",
                    transform=ax.transAxes, fontsize=7, ha="right", va="bottom", bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.7", lw=0.5, alpha=0.9), zorder=6)
            ax.set_xlim(*_xl); ax.set_ylim(*_yl); ax.grid(alpha=0.2, lw=0.5)
            ax.set_xlabel(AXL32.get(_xa, _xa)); ax.set_ylabel(AXL32.get(_ya, _ya))
            ax.set_title(f"{BOX_LABEL.get(_b, _b)}: {len(_Q)} rows, {int(_y.sum())} dusty", fontsize=9.5)
            if _j == 0:
                ax.legend(frameon=True, fontsize=6.5, loc="upper right", framealpha=0.9, edgecolor="0.7")
            # ── row 2: the dusty fraction along each candidate score ──
            ax = axs[1, _j]
            for _nm in P32_CURVES:
                if _nm not in _S["P"]:
                    continue
                _okn, _p = _S["P"][_nm]; _yn = _y[_okn]; _rk = np.argsort(np.argsort(_p)) / max(len(_p) - 1, 1) * 100
                _nb = int(max(2, min(P32_NBINS, len(_p) // P32_NMIN))); _e = np.quantile(_rk, np.linspace(0, 1, _nb + 1)); _e[0] -= 1e-9
                _ib = np.clip(np.searchsorted(_e, _rk, side="right") - 1, 0, _nb - 1)
                _xs, _ps, _ls, _hs = [], [], [], []
                for _i in range(_nb):
                    _mm = _ib == _i
                    if _mm.sum() < 2:
                        continue
                    _pp, _l, _h = _wil32(int(_yn[_mm].sum()), int(_mm.sum())); _xs.append(np.median(_rk[_mm])); _ps.append(_pp); _ls.append(_l); _hs.append(_h)
                _auc = next((t["auc"] for t in T32 if t["box"] == _b and t["divider"] == _nm), np.nan)
                ax.fill_between(_xs, _ls, _hs, color=CURVC32.get(_nm, "0.5"), alpha=0.10, lw=0)
                ax.plot(_xs, _ps, color=CURVC32.get(_nm, "0.5"), lw=2.4 if _nm == P32_SCORE else 1.6, marker="o", ms=3.5, label=f"{_nm}: AUC {_auc:.2f}")
            ax.axhline(_S["prior"], color="0.4", ls=":", lw=1.0); ax.text(99, _S["prior"], f"prior {100 * _S['prior']:.0f} % ", fontsize=7, color="0.35", va="bottom", ha="right")
            _fsel = 100 * (1 - np.nanmean(_S["pcv"] >= _S["thr"]))
            ax.axvline(_fsel, color="k", ls="--", lw=1.0); ax.text(_fsel, 0.98, f" {P32_SCORE} >= {P32_THRESH}", fontsize=7, va="top", ha="left", transform=ax.get_xaxis_transform())
            ax.set_xlim(0, 100); ax.set_ylim(0, 1); ax.grid(alpha=0.2, lw=0.5)
            ax.set_xlabel("percentile of the score (100 = most dusty-like)"); ax.set_ylabel("dusty fraction (out-of-fold)")
            ax.legend(frameon=False, fontsize=6.8, loc="upper left", title="steeper = a better divider", title_fontsize=6.8)
            # ── row 3: log f_dust against the drawn score ──
            ax = axs[2, _j]; _lo = _S["lodds"]; _lfd = _Q["lfd"].to_numpy(float); _ok = np.isfinite(_lo) & np.isfinite(_lfd)
            ax.scatter(_lo[_ok & ~_y], _lfd[_ok & ~_y], s=P32_MS, color=POPS["pure"]["color"], alpha=0.45, lw=0, zorder=2, rasterized=True)
            ax.scatter(_lo[_ok & _y], _lfd[_ok & _y], s=P32_MS + 3, color=POPS["dusty"]["color"], alpha=0.85, lw=0, zorder=3)
            _tr = _track28(_lo[_ok], _lfd[_ok])
            if _tr is not None:
                ax.fill_between(_tr["x"], _tr["lo"], _tr["hi"], color="k", alpha=0.10, lw=0, zorder=4)
                ax.plot(_tr["x"], _tr["med"], color="k", lw=2.2, zorder=5, label="running median, 16-84")
            ax.axhline(DUSTY_CUT, color="0.35", ls=":", lw=1.0, zorder=4); ax.text(0.99, DUSTY_CUT, "dusty above ", color="0.35", fontsize=7.5, ha="right", va="bottom", transform=ax.get_yaxis_transform())
            _lt = np.log(_S["thr"] / (1 - _S["thr"])); ax.axvline(_lt, color="k", ls="--", lw=1.0, zorder=4)
            ax.text(_lt, 0.02, f" threshold ({P32_THRESH})", fontsize=7, va="bottom", ha="left", transform=ax.get_xaxis_transform())
            _rr = spearmanr(_lo[_ok], _lfd[_ok])[0]
            ax.text(0.02, 0.97, r"$\rho$" + f" = {_rr:+.2f}", transform=ax.transAxes, fontsize=8, va="top", bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.7", lw=0.5, alpha=0.9))
            ax.set_ylim(*P32_YLIM3); ax.grid(alpha=0.2, lw=0.5)
            ax.set_xlabel(f"score {P32_SCORE}: log-odds of being dusty"); ax.set_ylabel(r"$\log\, M_{\rm dust}/M_\star$")
            if _j == 0:
                ax.legend(frameon=False, fontsize=7, loc="lower right")
        fig.suptitle(f"Part 32 — the division on trial: (1) the {_short32.get(_xa, _xa)} x {_short32.get(_ya, _ya)} plane with the score's threshold line, the best two-cut rule and the current rule's members; "
                     f"(2) the dusty fraction along each candidate score; (3) the dust against the score", fontsize=10.5)
        paper_save(fig, "agn_states_division_score")
        plt.show()
        pd.DataFrame(T32).to_csv(os.path.join(OUT, "agn_states_division_score.csv"), index=False)
        pd.DataFrame(R32).to_csv(os.path.join(OUT, "agn_states_division_rules.csv"), index=False)
        pd.DataFrame(C32).to_csv(os.path.join(OUT, "agn_states_division_coef.csv"), index=False)
        print(f"-> {os.path.join(OUT, 'agn_states_division_{score,rules,coef}.csv')}")

## Part 33 — Parts 22 and 23 under the divisions: as drawn, the Part 32 cut, the best cut

Part 22 draws the dust on the quenching clock under **dose × geometry** — the catalogue whole-galaxy $\kappa_{\rm rot}$ of the gas in the coarse boxes, the H$_2$ extent relative to the stars in m25 — and Part 23 carries the same four groups into the m25 resolved products. Part 32 then put the division on trial and, once $\kappa_{\rm gas}$ was in its ladder, found the gas kinematics + the stellar mass the strongest pair, the duty fraction adding nothing on top of the two. This part answers *what that changes in the pictures*: it re-cuts the **same frames** (Part 22's `FIG1_22` / `ANC22`, Part 23's `A23`, `RF` and the K-S arrays — nothing is re-measured) under three divisions and draws them side by side.

* **(a) as drawn** — Part 22's dose × geometry at the medians, its own masks and colours.
* **(b) the Part 32 cut** — the drawn score `P32_SCORE` at its threshold (`S32`): *predicted dusty* against *predicted dust-poor*, on the rows Part 32's frame covers. In m100 and m50 that is every row (Part 22's frames *are* Part 32's rows); in m25 Part 32 runs on Part 0's sample, a fraction of the KS-track frame Parts 22–23 use, and the title says how many. `P33_CUT32 = "rule"` takes Part 32's blind two-cut rule on `P32_PLANE` instead (dose and $M_\star$ only, so it covers every row).
* **(c) the best cut** — Part 22's division with a **third axis**, the stellar mass (`P33_THIRD`, split at its median or at `P33_MSPLIT`): dose × geometry × $M_\star$, eight cells, the massive half solid with its band, the light half dotted in the same colours. Not a replacement of the geometry axis: Part 32 shows $\kappa_{\rm gas}$ and $M_\star$ are independent and each carries its own signal.

`agn_states_division_fans` — the Part 22 fan (`P22_DUST` on $t - t_{\rm QT}$), rows (a, b, c) × one column per box; every legend entry carries $N$ and the **dusty share** ($\log M_{\rm dust}/M_\star \geq$ `DUSTY_CUT` at the anchor: QM's `lfd` in the coarse boxes, the KS epochs' catalogue masses in m25), so the purity of each curve is read off the legend rather than inferred from its level. Printed: the level of each median track before and after quenching (`P33_WIN`) and the *opening* of the fan, the spread of the group levels after QT.

`agn_states_division_part23_cis25` — Part 23's resolved anatomy under the same three divisions (needs Part 23's frames in memory): the radial $M_{\rm dust}/M_\star$ and $M_{\rm dust}/M_{\rm H_2}$ (`P33_PROF`; per-galaxy sightline medians, bootstrap 16–84 of the group median, the same aperture set as Part 23) and the **K-S track** through `P33_KS_STG` on the intrinsic $R_{50}({\rm H_2})$ apertures, with each group's end $R_{50}$, unresolved share under Part 23's two-tier rule and offset from Bigiel+08 written under the panel. The 2 × 2 groups of (a) and (c) keep Part 23's colours; in (c) the light half is dotted without bands.

**What to look for.** In the coarse boxes (b) and (c) open the fan wider than (a) — the red curve of (c), *low dose + rotating + massive*, is the dusty cell, every other cell is dust-poor at a few per cent — while the fan before quenching stays closed under every division: one galaxy before quenching, several after, whichever axes are used. In m25 the extent axis is what makes the dust a **core** phenomenon (the inner-kpc contrast of Part 23), and it is not the catalogue concentration Part 32 scored there; the dose ordering of the B08 offsets at the end point survives any second axis.

**Knobs.** `P33_BOXES`, `P33_CUT32`, `P33_THIRD` / `P33_MSPLIT`, `P33_WIN`, `P33_PROF`, `P33_KS_STG`, `P33_NMIN`. Products `agn_states_division_fans.{png,pdf}`, `agn_states_division_part23_cis25.{png,pdf}`, `agn_states_division_compare.csv` (per box / division / group: $N$, the dusty share, the fan levels; for m25 also the inner-kpc ratios and the K-S end point).

In [ ]:
# ── Part 33 — PARTS 22 AND 23 UNDER THE DIVISIONS: the fan and the resolved anatomy, as drawn / the Part 32 cut / the best cut ──
# Part 22 draws the dust on the quenching clock under dose x geometry (the catalogue kappa_gas in the coarse boxes, the H2 extent in
# m25) and Part 23 carries the same four groups into the m25 resolved products. Part 32 then scored every divider and found the gas
# kinematics + the stellar mass the strongest pair, the dose adding nothing on top. This part re-cuts the SAME frames (Part 22's
# FIG1_22 / ANC22, Part 23's A23, RF and the K-S arrays) under three divisions and draws them side by side, so what the division
# changes is visible rather than tabulated:
#   (a) as drawn — Part 22's dose x geometry at the medians (its own masks, its own colours);
#   (b) the Part 32 cut — the drawn score P32_SCORE at its threshold (S32[box]): predicted dusty / predicted dust-poor, on the rows
#       Part 32's frame covers (every row in m100 / m50; in m25 only Part 0's sample, a fraction of the KS-track frame — the title
#       says how many). P33_CUT32 = "rule" takes Part 32's blind two-cut rule on P32_PLANE instead (dose + M* only: every row);
#   (c) the best cut — Part 22's division with a THIRD axis, the stellar mass (P33_THIRD) at its median: dose x geometry x M*, eight
#       cells, the massive half solid with its band, the light half dotted in the same colours.
#   FIG 1  agn_states_division_fans: rows (a, b, c) x one column per box — the Part 22 fan (P22_DUST on t - t_QT), legend = N and
#          the dusty share (log M_dust/M* >= DUSTY_CUT at the anchor: QM's lfd in the coarse boxes, the KS epochs' catalogue masses
#          in m25).
#   FIG 2  agn_states_division_part23_cis25 (needs Part 23's frames): rows (a, b, c) x (the radial M_dust/M*, the radial
#          M_dust/M_H2, the K-S track jet -> SFT -> QT -> end, intrinsic apertures, with the end R50 and the unresolved share).
#   PRINTED + CSV  agn_states_division_compare.csv — per box / division / group: N, the dusty share, the fan level before and after
#          QT (P33_WIN); for m25 also the inner-kpc dust ratios and the K-S end point (log Sigma_H2, offset from B08, unresolved share).
P33_BOXES = list(globals().get("P22_BOXES", ["cis100", "cis50", "cis25"]))
P33_CUT32 = "score"        # (b): "score" = P32_SCORE >= its threshold | "rule" = Part 32's blind two-cut rule on P32_PLANE
P33_THIRD = "log_mstar"    # (c): the third axis (the m25 frame carries it as log_mstar_anchor)
P33_MSPLIT = "median"      # (c): where the third axis is split: "median" | a number
P33_WIN   = ((-2.0, -0.4), (0.6, 2.0))   # the fan windows printed: before QT, after QT [Gyr]
P33_PROF  = [("fdust_ap", r"$M_{\rm dust}/M_\star$"), ("dh2_ann", r"$M_{\rm dust}/M_{\rm H_2}$")]   # the Part 23 radial panels re-drawn
P33_KS_STG = list(globals().get("P23_KS_STG", ["jet_on", "sft", "qt", "end"]))
P33_NMIN  = globals().get("P23_NMIN", 5)   # a Part 23 point needs this many galaxies
P33_LEGFS = 7.2

if "FIG1_22" not in globals() or "S32" not in globals():
    print("Part 33: needs Part 22 (FIG1_22) and Part 32 (S32). Run them first.")
else:
    _C33 = ["#b2182b", "#ef8a62", "#67a9cf", "#2166ac"]; _LS33 = ["-", "--", "--", "-"]
    _CD33, _CP33 = POPS["dusty"]["color"], POPS["pure"]["color"]
    _sh33 = globals().get("_short32", {})
    _win33 = lambda c, lo, hi: (float(np.nanmean(c[(P22_TG >= lo) & (P22_TG <= hi)])) if np.isfinite(c[(P22_TG >= lo) & (P22_TG <= hi)]).any() else np.nan)
    _EA33 = EPO22[EPO22["stage"] == "anchor"].drop_duplicates("key").set_index("key")

    def _lfd33(b, A):
        """log M_dust/M* at the anchor of a Part 22 frame's rows: the KS epochs' catalogue masses (the m25 frame), QM's lfd otherwise."""
        if b == P22_BOX and not P22_LIKE and "key" in A.columns and A["key"].isin(_EA33.index).mean() > 0.5:
            with np.errstate(all="ignore"):
                return np.log10(_EA33["mdust_cat"].reindex(A["key"]).to_numpy(float) / _EA33["mstar_cat"].reindex(A["key"]).to_numpy(float))
        q = QM.loc[QM["box"] == b, ["snap", "gal_id", "lfd"]].astype({"snap": int, "gal_id": int}).drop_duplicates(["snap", "gal_id"])
        return A[["snap", "gal_id"]].astype(int).merge(q, on=["snap", "gal_id"], how="left")["lfd"].to_numpy(float)

    def _lm33(A):
        """The third axis of a frame (P33_THIRD, or its _anchor twin in the m25 frame)."""
        for c in (P33_THIRD, P33_THIRD + "_anchor"):
            if c in A.columns:
                return A[c].to_numpy(float)
        return np.full(len(A), np.nan)

    def _cls33(b, snaps, gids, dose, lm):
        """The Part 32 cut for a set of rows: 1 = predicted dusty, 0 = predicted dust-poor, nan = not in Part 32's frame."""
        if b not in S32:
            return np.full(len(snaps), np.nan)
        S = S32[b]
        if P33_CUT32 == "rule":
            _bk = f"best two-cut on ({_sh33.get(P32_PLANE[0], P32_PLANE[0])}, {_sh33.get(P32_PLANE[1], P32_PLANE[1])})"
            if _bk not in S["best"]:
                return np.full(len(snaps), np.nan)
            r = S["best"][_bk]; d, m = np.asarray(dose, float), np.asarray(lm, float)
            sel = ((d < r["cut_a"]) if r["sa"] < 0 else (d > r["cut_a"])) & ((m > r["cut_b"]) if r["sb"] > 0 else (m < r["cut_b"]))
            return np.where(np.isfinite(d) & np.isfinite(m), sel.astype(float), np.nan)
        v = pd.DataFrame({"snap": S["Q"]["snap"].astype(int).to_numpy(), "gal_id": S["Q"]["gal_id"].astype(int).to_numpy(),
                          "c": np.where(np.isfinite(S["pcv"]), (S["pcv"] >= S["thr"]).astype(float), np.nan)}).drop_duplicates(["snap", "gal_id"])
        return pd.DataFrame({"snap": np.asarray(snaps, int), "gal_id": np.asarray(gids, int)}).merge(v, on=["snap", "gal_id"], how="left")["c"].to_numpy(float)

    _cut32lab = (f"score {P32_SCORE} >= {P32_THRESH}" if P33_CUT32 == "score" else f"Part 32's two-cut rule on ({_sh33.get(P32_PLANE[0], P32_PLANE[0])}, {_sh33.get(P32_PLANE[1], P32_PLANE[1])})")
    ROWS33 = []; MSPLIT33 = {}
    # ── FIG 1: the fan under the three divisions, one column per box ──
    _bx33 = [b for b in P33_BOXES if b in FIG1_22]
    if _bx33:
        fig, axs = plt.subplots(3, len(_bx33), figsize=(5.4 * len(_bx33), 12.6), sharex=True, sharey="row", squeeze=False)
        for _j, _b in enumerate(_bx33):
            F = FIG1_22[_b]; A = F["A"].reset_index(drop=True); TR = F["TR"]
            _lfd = _lfd33(_b, A); _y = np.where(np.isfinite(_lfd), (_lfd >= DUSTY_CUT).astype(float), np.nan)
            _lm = _lm33(A); _ms = float(np.nanmedian(_lm)) if P33_MSPLIT == "median" else float(P33_MSPLIT); MSPLIT33[_b] = _ms
            _hm = _lm > _ms; _hd, _hg, _lg = np.asarray(F["hi_dose"], bool), np.asarray(F["hi_geom"], bool), np.asarray(F["lo_geom"], bool)
            _gl, _gh, _glo = F["lab"]
            _cls = _cls33(_b, A["snap"], A["gal_id"], F["dose"], _lm); _nin = int(np.isfinite(_cls).sum())
            _divs = [(f"(a) as drawn: dose $\\times$ geometry ({_gl})",
                      [(f"low dose + {_gh}", ~_hd & _hg, _C33[0], _LS33[0], False), (f"low dose + {_glo}", ~_hd & _lg, _C33[1], _LS33[1], False),
                       (f"high dose + {_gh}", _hd & _hg, _C33[2], _LS33[2], False), (f"high dose + {_glo}", _hd & _lg, _C33[3], _LS33[3], False)]),
                     (f"(b) the Part 32 cut: {_cut32lab}" + (f"\n({_nin} of {len(A)} rows in Part 32's frame)" if _nin < len(A) else ""),
                      [("predicted dusty", _cls == 1, _CD33, "-", False), ("predicted dust-poor", _cls == 0, _CP33, "-", False)]),
                     (f"(c) the best cut: dose $\\times$ geometry $\\times$ $M_\\star$ (split {_ms:.2f})",
                      [(f"low dose + {_gh} + massive", ~_hd & _hg & _hm, _C33[0], "-", False), (f"low dose + {_glo} + massive", ~_hd & _lg & _hm, _C33[1], "-", False),
                       (f"high dose + {_gh} + massive", _hd & _hg & _hm, _C33[2], "-", False), (f"high dose + {_glo} + massive", _hd & _lg & _hm, _C33[3], "-", False),
                       (f"low dose + {_gh} + light", ~_hd & _hg & ~_hm, _C33[0], ":", True), (f"low dose + {_glo} + light", ~_hd & _lg & ~_hm, _C33[1], ":", True),
                       (f"high dose + {_gh} + light", _hd & _hg & ~_hm, _C33[2], ":", True), (f"high dose + {_glo} + light", _hd & _lg & ~_hm, _C33[3], ":", True)])]
            for _i, (_title, _grp) in enumerate(_divs):
                ax = axs[_i, _j]
                for _lab, _m, _c, _ls, _thin in _grp:
                    _m = np.asarray(_m, bool); _n = int(_m.sum())
                    _cen, _lo, _hi, _ = _band22(TR[_m]) if _n else (np.full(len(P22_TG), np.nan),) * 4
                    _fd = float(np.nanmean(_y[_m])) if np.isfinite(_y[_m]).any() else np.nan
                    if not _thin:
                        ax.fill_between(P22_TG, _lo, _hi, color=_c, alpha=0.15, lw=0)
                    ax.plot(P22_TG, _cen, _ls, color=_c, lw=1.3 if _thin else 2.5, alpha=0.8 if _thin else 1.0,
                            label=None if (_thin or not _n) else f"{_lab} ({_n}, {100 * _fd:.0f} % dusty)")
                    ROWS33.append(dict(box=_b, block="fan", division="abc"[_i], group=_lab, n=_n, f_dusty=_fd,
                                       pre=_win33(_cen, *P33_WIN[0]), post=_win33(_cen, *P33_WIN[1])))
                ax.axvline(0, color="k", lw=1.2, ls=":"); ax.grid(alpha=0.22, lw=0.5)
                ax.legend(frameon=False, fontsize=P33_LEGFS, loc="lower left")
                if _i == 2:
                    ax.text(0.98, 0.97, "dotted, same colours: the light half of each cell", transform=ax.transAxes, fontsize=7.5, ha="right", va="top", color="0.3")
                    ax.set_xlabel(r"$t - t_{\rm QT}$   [Gyr]")
                ax.set_title((f"{BOX_LABEL.get(_b, _b)}: {len(A)} quenched galaxies\n" if _i == 0 else "") + _title, fontsize=9.5)
                if _j == 0:
                    ax.set_ylabel(P22_ULAB[P22_DUST])
        fig.suptitle("Part 33 — the Part 22 fan under the division as drawn (a), the Part 32 cut (b) and the best cut (c); "
                     f"legend: N, dusty share ($\\log M_{{\\rm dust}}/M_\\star \\geq {DUSTY_CUT}$ at the anchor)", fontsize=11)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        paper_save(fig, "agn_states_division_fans")
        plt.show()
        print("Part 33 — the fan: the level of each group's median track before QT [%.1f, %.1f] and after QT [%.1f, %.1f] Gyr (%s):" % (*P33_WIN[0], *P33_WIN[1], P22_DUST))
        for _b in _bx33:
            _R = [r for r in ROWS33 if r["box"] == _b and r["block"] == "fan"]
            for _d in "abc":
                _Rd = [r for r in _R if r["division"] == _d]
                _po = [r["post"] for r in _Rd if np.isfinite(r["post"])]
                print(f"  [{_b}] ({_d}) fan opening after QT {max(_po) - min(_po) if len(_po) > 1 else np.nan:.2f} dex: "
                      + "; ".join(f"{r['group']} n {r['n']} {100 * r['f_dusty']:.0f} % dusty, post {r['post']:+.2f}" for r in _Rd if r["n"]))

    # ── FIG 2: Part 23's resolved anatomy under the same three divisions (m25) ──
    if not ("A23" in globals() and "RF" in globals() and "x23" in globals()):
        print("Part 33 — Part 23's frames (A23 / RF / the K-S arrays) are not in memory: run Part 23 (and its products) first — figure 2 skipped.")
    else:
        from matplotlib.lines import Line2D as _L33
        _A = A23.copy(); _A["lm"] = _EA33["log_mstar_anchor"].reindex(_A["key"]).to_numpy(float)
        with np.errstate(all="ignore"):
            _A["lfd"] = np.log10(_EA33["mdust_cat"].reindex(_A["key"]).to_numpy(float) / _EA33["mstar_cat"].reindex(_A["key"]).to_numpy(float))
        _y23 = np.where(np.isfinite(_A["lfd"]), (_A["lfd"] >= DUSTY_CUT).astype(float), np.nan)
        _ms = MSPLIT33.get(P23_BOX, float(np.nanmedian(_A["lm"])) if P33_MSPLIT == "median" else float(P33_MSPLIT))
        _hm = _A["lm"].to_numpy(float) > _ms; _hd, _hg, _lg = _A["hi_dose"].to_numpy(bool), _A["hi_geom"].to_numpy(bool), _A["lo_geom"].to_numpy(bool)
        _cls = _cls33(P23_BOX, _A["anchor_snap"], _A["gal_id"], _A["f_x_post"], _A["lm"]); _nin = int(np.isfinite(_cls).sum())
        _pl = lambda t: _re23.sub(r"\\[a-zA-Z]+|[{}$]", "", t.replace("$_2$", "2")).replace("  ", " ").strip()
        _ghp, _glp = _pl(_ghi), _pl(_glo)
        _DIV23 = [("(a) as drawn: dose $\\times$ H$_2$ extent", [(n, (_A["grp"] == n).to_numpy(), COL23[n], "-", False) for n in NAM23]),
                  (f"(b) the Part 32 cut: {_cut32lab}" + (f" ({_nin} of {len(_A)} anchor rows)" if _nin < len(_A) else ""),
                   [("predicted dusty", _cls == 1, _CD33, "-", False), ("predicted dust-poor", _cls == 0, _CP33, "-", False)]),
                  (f"(c) the best cut: dose $\\times$ H$_2$ extent $\\times$ $M_\\star$ (split {_ms:.2f})",
                   [(f"low dose + {_ghp} + massive", ~_hd & _hg & _hm, _C33[0], "-", False), (f"low dose + {_glp} + massive", ~_hd & _lg & _hm, _C33[1], "-", False),
                    (f"high dose + {_ghp} + massive", _hd & _hg & _hm, _C33[2], "-", False), (f"high dose + {_glp} + massive", _hd & _lg & _hm, _C33[3], "-", False),
                    (f"low dose + {_ghp} + light", ~_hd & _hg & ~_hm, _C33[0], ":", True), (f"low dose + {_glp} + light", ~_hd & _lg & ~_hm, _C33[1], ":", True),
                    (f"high dose + {_ghp} + light", _hd & _hg & ~_hm, _C33[2], ":", True), (f"high dose + {_glp} + light", _hd & _lg & ~_hm, _C33[3], ":", True)])]
        _keyA = list(zip(_A["anchor_snap"].astype(int), _A["gal_id"].astype(int)))
        _rfkey = list(zip(RF["snap"].astype(int), RF["gal_id"].astype(int)))
        _gkey = [tuple(int(v) for v in g_.split("_")) for g_ in gk23]
        _mk33 = globals().get("MRK23", {"sf_peak": "*", "agn_ign": "^", "jet_on": "v", "sft": "s", "qt": "D", "end": "o"})
        _lb33 = globals().get("LAB23S", {"jet_on": "jet", "sft": "SFT", "qt": "QT", "end": "end"})
        _xs = np.linspace(P23_KS_XLIM[0] - 1, P23_KS_XLIM[1] + 1, 60); _x23n = np.arange(len(P23_ANNULI))
        fig, axs = plt.subplots(3, len(P33_PROF) + 1, figsize=(5.0 * (len(P33_PROF) + 1), 12.6), squeeze=False)
        for _i, (_title, _grp) in enumerate(_DIV23):
            _map = {}
            for _lab, _m, _, _, _ in _grp:
                for _k, _in in zip(_keyA, np.asarray(_m, bool)):
                    if _in:
                        _map[_k] = _lab
            _rfg = np.array([_map.get(k) for k in _rfkey], dtype=object); _ksg = np.array([_map.get(k) for k in _gkey], dtype=object)
            for _q, (_col, _qlab) in enumerate(P33_PROF):
                ax = axs[_i, _q]
                for _gi, (_lab, _m, _c, _ls, _thin) in enumerate(_grp):
                    _med, _lo, _hi, _nn = [], [], [], []
                    for _a23, _ in P23_ANNULI:
                        _sub = RF[(_rfg == _lab) & (RF["aperture"] == _a23).to_numpy()]
                        _per = _sub.groupby(["snap", "gal_id"])[_col].median().to_numpy(float); _per = _per[np.isfinite(_per)]
                        _cc, _l, _h, _n = _gband23(_per) if len(_per) else (np.nan, np.nan, np.nan, 0)
                        _med.append(_cc); _lo.append(_l); _hi.append(_h); _nn.append(_n)
                    _med, _lo, _hi = np.array(_med), np.array(_lo), np.array(_hi); _ok = np.isfinite(_med)
                    _off = (_gi % 4 - 1.5) * 0.07
                    if _ok.any():
                        if not _thin:
                            ax.fill_between(_x23n[_ok] + _off, _lo[_ok], _hi[_ok], color=_c, alpha=0.15, lw=0)
                        ax.plot(_x23n[_ok] + _off, _med[_ok], _ls, marker="o" if not _thin else None, color=_c, ms=4.5, mec="k", mew=0.4,
                                lw=1.2 if _thin else 1.9, alpha=0.8 if _thin else 1.0, label=None if _thin else f"{_lab} ({int(np.sum(np.asarray(_m, bool)))})")
                    if _q == 0:
                        _mm = np.asarray(_m, bool)
                        ROWS33.append(dict(box=P23_BOX, block="part23", division="abc"[_i], group=_lab, n=int(_mm.sum()),
                                           f_dusty=float(np.nanmean(_y23[_mm])) if np.isfinite(_y23[_mm]).any() else np.nan,
                                           inner_fdust=_med[0] if _col == "fdust_ap" else np.nan))
                    elif _col == "dh2_ann":
                        for r in ROWS33:
                            if r["box"] == P23_BOX and r["block"] == "part23" and r["division"] == "abc"[_i] and r["group"] == _lab:
                                r["inner_dh2"] = _med[0]
                ax.set_yscale("log"); ax.set_xticks(_x23n); ax.set_xticklabels([r_ for _, r_ in P23_ANNULI], fontsize=8); ax.grid(alpha=0.25, lw=0.5)
                ax.set_ylabel(_qlab); ax.legend(frameon=False, fontsize=P33_LEGFS)
                if _i == 2:
                    ax.set_xlabel("annulus [kpc]")
                if _q == 0:
                    ax.set_title(_title, fontsize=9.5, loc="left")
            # the K-S track, intrinsic apertures
            ax = axs[_i, len(P33_PROF)]
            ax.fill_between(_xs, REL23["A"] + REL23["N"] * _xs - REL23["sig"], REL23["A"] + REL23["N"] * _xs + REL23["sig"], color=REL23["color"], alpha=0.10, lw=0, zorder=0)
            ax.plot(_xs, REL23["A"] + REL23["N"] * _xs, color=REL23["color"], lw=1.4, ls=REL23.get("ls", "--"), zorder=2, label=REL23.get("label", "Bigiel+08"))
            _txt = []
            for _lab, _m, _c, _ls, _thin in _grp:
                _pts = []
                for _st in P33_KS_STG:
                    _mk = base23 & (pop23 == "Q") & (_ksg == _lab) & (stg23 == _st) & ~ul23 & np.isfinite(x23) & np.isfinite(y23)
                    if _mk.sum() < P33_NMIN:
                        continue
                    _px, _py = float(np.median(x23[_mk])), float(np.median(y23[_mk])); _pts.append((_px, _py))
                    ax.scatter(_px, _py, s=(70 if not _thin else 22), marker=_mk33.get(_st, "o"), c=_c, edgecolors="k" if not _thin else "none",
                               linewidths=0.6, alpha=1.0 if not _thin else 0.7, zorder=6 if not _thin else 4)
                    if _st == P33_KS_STG[-1]:
                        _off = float(np.median(y23[_mk] - (REL23["A"] + REL23["N"] * x23[_mk])))
                        for r in ROWS33:
                            if r["box"] == P23_BOX and r["block"] == "part23" and r["division"] == "abc"[_i] and r["group"] == _lab:
                                r.update(ks_end_n=int(_mk.sum()), ks_end_logSigmaH2=_px, ks_end_offset_B08=_off, ks_end_f_unresolved=float(np.mean(unres23[_mk])), ks_end_r50_kpc=float(np.median(r50_23[_mk])))
                        if not _thin:
                            _txt.append((f"{_lab}: end $R_{{50}}$ {np.median(r50_23[_mk]):.1f} kpc, {100 * np.mean(unres23[_mk]):.0f} % unresolved, B08 offset {_off:+.2f}", _c))
                if len(_pts) > 1:
                    ax.plot(*zip(*_pts), _ls if _thin else "-", color=_c, lw=1.1 if _thin else 2.2, alpha=0.7 if _thin else 1.0, zorder=5 if not _thin else 3, label=None if _thin else _lab)
            for _k, (_t, _c) in enumerate(_txt):
                ax.text(0.98, 0.03 + 0.045 * _k, _t, transform=ax.transAxes, fontsize=6.6, ha="right", va="bottom", color=_c, zorder=7)
            ax.set_xlim(*P23_KS_XLIM); ax.set_ylim(*P23_KS_YLIM); ax.grid(alpha=0.25, lw=0.5)
            ax.legend(frameon=False, fontsize=P33_LEGFS, loc="upper left")
            ax.set_ylabel(r"$\log\, \Sigma_{\rm SFR}$  [M$_\odot$ yr$^{-1}$ kpc$^{-2}$]")
            ax.set_title("K-S, intrinsic $R_{50}$(H$_2$): " + " $\\to$ ".join(_lb33.get(s, s) for s in P33_KS_STG) + ("  (dotted: the light half)" if _i == 2 else ""), fontsize=9.5)
            if _i == 2:
                ax.set_xlabel(r"$\log\, \Sigma_{\rm H_2}$  [M$_\odot$ pc$^{-2}$]")
        fig.suptitle(f"Part 33 — {BOX_LABEL.get(P23_BOX, P23_BOX)}, Part 23's resolved anatomy under the three divisions "
                     f"({len(_A)} quenched anchor rows; per-galaxy sightline medians, bootstrap 16-84 of the group median)", fontsize=11)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        paper_save(fig, f"agn_states_division_part23_{P23_BOX}")
        plt.show()
        print(f"Part 33 — Part 23 re-cut ({BOX_LABEL.get(P23_BOX, P23_BOX)}, {len(_A)} anchor rows): inner-kpc medians (log) and the K-S end point per group")
        for r in ROWS33:
            if r["block"] == "part23" and r["n"]:
                print(f"  ({r['division']}) {r['group']:36s} n {r['n']:3d}  dusty {100 * r['f_dusty']:3.0f} %  0-1 kpc: log M_dust/M* {np.log10(r['inner_fdust']) if np.isfinite(r.get('inner_fdust', np.nan)) and r['inner_fdust'] > 0 else np.nan:+.2f}, "
                      f"log M_dust/M_H2 {np.log10(r['inner_dh2']) if np.isfinite(r.get('inner_dh2', np.nan)) and r['inner_dh2'] > 0 else np.nan:+.2f}"
                      + (f"  |  K-S end: log Sigma_H2 {r['ks_end_logSigmaH2']:+.2f}, B08 offset {r['ks_end_offset_B08']:+.2f}, {100 * r['ks_end_f_unresolved']:.0f} % unresolved, R50 {r['ks_end_r50_kpc']:.2f} kpc" if "ks_end_n" in r else ""))
    if ROWS33:
        pd.DataFrame(ROWS33).to_csv(os.path.join(OUT, "agn_states_division_compare.csv"), index=False)
        print(f"-> {os.path.join(OUT, 'agn_states_division_compare.csv')}")

## Part 34 — the geometry axis against the particle count: is $\kappa_{\rm rot}$(gas) rotation or shot noise?

Part 22 divides the coarse boxes on the catalogue's whole-galaxy $\kappa_{\rm rot}$ of the gas and Part 32 makes it the strongest single divider of dusty against dust-poor, so the axis has to answer the obvious objection first: a quenched galaxy here carries **50–250 gas particles**, and $\kappa_{\rm rot} = K_{\rm rot}/K$ is a ratio of sums over them — a gas-poor galaxy could be called *rotating* because a handful of particles happened to line up. That worry is not idle: the m25 H$_2$-core $\kappa$ of the 2026-09-04 rework did behave that way (the whole signal sat in the $n_{\rm H_2} < 100$ galaxies and flipped sign above it), which is why Part 22 keeps the KS-track axis for m25 and the catalogue axis for the coarse boxes.

**The null.** For an isotropic velocity field $\kappa_{\rm rot}$ has expectation $1/3$ — one of three components is azimuthal — and a sampling variance ${\rm Var}(f)/N_{\rm eff}$ with $f = v_\phi^2/v^2$: ${\rm Var}(f) = 4/45$ for a uniform direction, and the $v^2$ weighting of the denominator gives $N_{\rm eff} = N\,(\langle w\rangle^2/\langle w^2\rangle) = 0.6\,N$ for a Maxwellian. So

$$s_{\rm iso}(N) = \sqrt{\frac{4/45}{0.6\,N}} = \frac{0.385}{\sqrt{N}},$$

which is $\pm 0.044$ at $N = 76$, the median of the m100 sample. Every number in this part is measured against that yardstick, and `P34_WEFF` is the knob if one wants a different effective $N$.

**Four tests, one per figure row.**

* **(a) the null** — $\kappa$ against $N$ with the isotropic $1/3 \pm s_{\rm iso}(N)$ band: the scatter the null allows against the observed one, the share above $1/3 + 2\,s_{\rm iso}$ against the 2.3 % the null allows, the dusty−dust-poor gap in units of $s_{\rm iso}$, and whether the **false positives** (dust-poor galaxies above $\kappa =$ `P34_HIFP`) are the few-particle ones.
* **(b) where the signal lives** — the AUC of raw $\kappa$ toward dusty *inside bins of $N$*. Any function of $\kappa$ alone is AUC-identical inside a bin, so this row cannot be argued away by re-weighting: shot noise dilutes, so the AUC would have to climb with $N$.
* **(c) what a weighting can buy** — the whole shrinkage family $1/3 + (\kappa - 1/3)\,N/(N+N_0)$, which *is* the empirical-Bayes posterior mean for a sampling noise inflated by $f = N_0\,\sigma_{\rm int}^2/C_{\rm iso}^2$, drawn against $f$; with **the ceiling** (the AUC a Gaussian deconvolution says *perfect* de-noising would reach at that noise level) and **the sham** (the same weighting driven by $N$ shuffled inside $\log M_\star$ deciles). A gain above the ceiling is not de-noising — it is the particle count, i.e. the gas mass, i.e. almost the dust label, entering the axis through the back door.
* **(d) every version** — the single-variable AUC ladder of raw / significance / empirical-Bayes / $N$-weighted / $N$-trend-residual $\kappa$ against the controls ($\log N_{\rm gas}$, $\kappa_\star$, $\log M_\star$) and the out-of-fold scores of Part 32's machinery, with the ceiling marked.

Printed as well, block **(e)**: Part 22's geometry split at its median against *rotating* $= \kappa > 1/3 + $ `P34_SIG` $s_{\rm iso}(N)$ and against a **fixed** cut at the same average level — the size and the dusty share of the red group (low dose + rotating) under each.

**What it says (2026-09-09, at `DUSTY_CUT` $=-4$).** The estimator's noise is real and small: $s_{\rm iso} = 0.044$ / 0.045 / 0.025 (m100 / m50 / m25) against a dusty−dust-poor gap of 0.42 / 0.40 / 0.65, i.e. **9.5 / 8.7 / 26 $\sigma$**. Under the isotropic null $\kappa$ would scatter with sd 0.043; it scatters with 0.226, and 33 % of the galaxies sit above $1/3 + 2 s_{\rm iso}$ where the null allows 2.3 %. The dust-poor galaxies with $\kappa > 0.5$ — the false positives that cap the AUC — have median $N_{\rm gas}$ 68 against 76 for the rest of their class, and raw $\kappa$'s AUC inside $N$ quintiles is flat (0.81, 0.88, 0.77, 0.88, 0.85 in m100). **This is not the m25 few-particle $\kappa$ artefact.** Deconvolved, the sampling noise is 3.6 / 4.0 / 2.0 % of the within-class variance, so perfect de-noising is worth **+0.004 / +0.005 / +0.002** in AUC — and the physically calibrated shrinkage delivers exactly that nothing ($w(N) = 0.95$–0.98, AUC 0.822 → 0.823, the same as the sham). The aggressive members of the family reach 0.835 / 0.883, *above* the ceiling, which is the proof that they are not de-noising: at those $N_0$ one is assuming a noise 250–500 times the isotropic value, the variable is essentially $(\kappa - 1/3)\times N$, and $\log N_{\rm gas}$ alone already scores 0.76 because the gas mass carries the dust. Nor does a de-noised axis make Part 32's third axis redundant: $\kappa + \log M_\star$ is 0.874 / 0.925 out-of-fold against 0.820 / 0.836 for any single de-noised $\kappa$, and $\rho(\kappa, \log N_{\rm gas}) = -0.06$. **The one thing worth taking is the threshold, not the weighting**: cutting at $1/3 + 2 s_{\rm iso}$ instead of the median 0.30 lifts the red group from 17.8 % to 23.7 % dusty in m100 (25.0 % in m50, 87.5 % in m25) while keeping ~70 % of the dusty galaxies — but a *fixed* cut at 0.42 does the same to within 24 of 1683 galaxies. Set `P22_GCUT = 0.42` (Part 22 already takes a number there) if the cleaner red group is wanted; the axis itself needs no correction.

**Knobs.** `P34_KCOL` (the axis on trial), `P34_VARF` / `P34_WEFF` (the null: change `P34_WEFF` to assume clumped, correlated gas — the printed block also quotes the ceiling at $N_{\rm eff} = N/4$ and $N/9$ through `P34_NEFF`), `P34_SIG`, `P34_HIFP`, `P34_FGRID`, `P34_NBOOT`, `P34_NBIN`. Products `agn_states_kappa_resolution.{png,pdf}` and `agn_states_kappa_resolution.csv` (per box and block: the null, the $N$ bins, the ceiling, the ladder, the sham, the out-of-fold scores, the rules).

In [ ]:
# ── Part 34 — THE GEOMETRY AXIS AGAINST THE PARTICLE COUNT: is kappa_gas rotation, or is it shot noise? ──
# Part 22 splits the coarse boxes on the catalogue's whole-galaxy kappa_rot of the gas and Part 32 makes it the strongest single
# divider, so the axis has to survive the obvious objection: a quenched galaxy here holds ~50-250 gas particles, and kappa_rot is
# a RATIO of sums over them — a gas-poor galaxy could be called rotating because a handful of particles happened to line up.
#   THE NULL.  For an isotropic velocity field kappa_rot = K_rot/K has E = 1/3 (one of three components is azimuthal) and a
#   sampling variance Var(f)/N_eff with f = v_phi^2/v^2: Var(f) = 4/45 for a uniform direction, and the v^2 weighting of the
#   denominator gives N_eff = N (E w)^2 / E w^2 = P34_WEFF x N = 0.6 N for a Maxwellian.  So s_iso(N) = sqrt(4/45 / 0.6 N) =
#   0.385 / sqrt(N): +-0.044 at N = 76, the median of the m100 sample.  Everything below is measured against that yardstick.
#   FOUR TESTS, one per figure row:
#     (a) THE NULL — kappa against N with the isotropic 1/3 +- s_iso(N) band: the sd the null allows against the observed sd, the
#         share above 1/3 + 2 s_iso against the 2.3 % the null allows, the class gap in units of s_iso, and whether the FALSE
#         POSITIVES (dust-poor galaxies with a high kappa) are the few-particle ones.
#     (b) WHERE THE SIGNAL LIVES — the AUC of raw kappa toward dusty INSIDE bins of N. Any function of kappa alone is
#         AUC-identical inside a bin, so this row is untouchable by re-weighting: shot noise would make it climb with N.
#     (c) WHAT A WEIGHTING CAN BUY — the whole shrinkage family 1/3 + (kappa - 1/3) N/(N + N0), which IS the empirical-Bayes
#         posterior mean for a noise inflated by f = N0 sd_int^2 / C_ISO^2, drawn against f; with (i) the CEILING, the AUC a
#         Gaussian deconvolution says PERFECT de-noising would reach at that noise level, and (ii) the SHAM, the same weighting
#         driven by N shuffled inside log M* deciles. A gain above the ceiling is not de-noising: it is the particle count (i.e.
#         the gas mass, i.e. almost the dust label) entering the axis through the back door.
#     (d) EVERY VERSION — the single-variable AUC ladder of raw / significance / empirical-Bayes / N-weighted / N-trend-residual
#         kappa against the controls (log N_gas, log M*) and the out-of-fold scores, with the ceiling marked.
#   PRINTED  the four blocks above plus the PRACTICAL one: Part 22's geometry split at its median against "rotating = kappa above
#            1/3 + P34_SIG s_iso(N)" and against a FIXED cut at the same level — the dusty share of the red group under each.
#   Pure reads of Part 32's frames (S32) and Part 0's catalogue rows (QM, for the particle counts). Nothing here re-measures kappa.
P34_BOXES = [b for b in globals().get("P32_BOXES", ["cis100", "cis50", "cis25"]) if b in globals().get("S32", {})]
P34_KCOL  = globals().get("P22_GEOM_CAT", "kappa_gas")   # the axis on trial: the catalogue's whole-galaxy kappa_rot
P34_VARF, P34_WEFF = 4.0 / 45.0, 0.6     # Var(v_phi^2/v^2) for a uniform direction, and N_eff / N under the v^2 weighting
P34_SIG   = 2.0                          # the significance cut of the practical block: kappa > 1/3 + P34_SIG s_iso(N)
P34_HIFP  = 0.5                          # a "false positive" is a dust-poor galaxy above this kappa (0.5 = a disc, not a ratio near 1/3)
P34_FGRID = [1, 2, 4, 9, 20, 50, 120, 300, 800, 2000]    # the noise levels of row (c), in units of the isotropic variance
P34_NEFF  = [4, 9]                       # the pessimistic ceilings: if the gas is clumped so that N_eff = N / this
P34_NBOOT = min(int(globals().get("NBOOT", 300)), 300)
P34_NBIN  = 8                            # equal-count bins of N for the variance check
P34_MS    = 3.0

if "S32" not in globals() or not P34_BOXES:
    print("Part 34: needs Part 32's frames (S32) — run it first.")
elif "QM" not in globals():
    print("Part 34: needs Part 0's catalogue rows (QM) for the particle counts.")
else:
    import math
    C_ISO34 = float(np.sqrt(P34_VARF / P34_WEFF))          # s_iso(N) = C_ISO34 / sqrt(N)
    _rng34 = np.random.default_rng(7)
    _phi34 = lambda z: 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))   # the Gaussian CDF: AUC = Phi(separation / sqrt(2) sd), no scipy on the cluster kernel

    def _auc34(x, y, nb=P34_NBOOT):
        """AUC toward dusty with a bootstrap 16-84 over the rows."""
        x = np.asarray(x, float); y = np.asarray(y, bool); ok = np.isfinite(x); x, y = x[ok], y[ok]
        if y.sum() < 3 or (~y).sum() < 3:
            return np.nan, np.nan, np.nan, int(ok.sum())
        a = mw_auc(x, y)[0]; bs = []
        for _ in range(nb):
            i = _rng34.integers(0, len(x), len(x))
            if y[i].sum() >= 2 and (~y[i]).sum() >= 2:
                bs.append(mw_auc(x[i], y[i])[0])
        return a, float(np.nanpercentile(bs, 16)), float(np.nanpercentile(bs, 84)), int(len(x))

    def _resid34(v, X):
        """v with a linear trend in the columns of X removed (the class and mass trends, so the scatter left is the galaxy-to-galaxy one)."""
        v = np.asarray(v, float); X = np.column_stack([np.ones(len(v))] + [np.asarray(c, float) for c in X])
        ok = np.isfinite(v) & np.isfinite(X).all(1); r = np.full(len(v), np.nan)
        if ok.sum() > X.shape[1] + 2:
            r[ok] = v[ok] - X[ok] @ np.linalg.lstsq(X[ok], v[ok], rcond=None)[0]
        return r

    def _varN34(r, n, nbin=P34_NBIN):
        """The robust variance of a residual in equal-count bins of N, and the weighted fit Var = A + a/N (a = the noise amplitude squared)."""
        ok = np.isfinite(r) & np.isfinite(n) & (n > 0); r, n = r[ok], n[ok]
        q = np.quantile(n, np.linspace(0, 1, nbin + 1)); rows = []
        for i in range(nbin):
            m = (n >= q[i]) & ((n <= q[i + 1]) if i == nbin - 1 else (n < q[i + 1]))
            if m.sum() < 20:
                continue
            s_mad = 1.4826 * np.median(np.abs(r[m] - np.median(r[m])))
            rows.append(dict(n_med=float(np.median(n[m])), inv_n=float(np.mean(1.0 / n[m])), var=float(s_mad ** 2), k=int(m.sum())))
        D = pd.DataFrame(rows)
        if len(D) < 3:
            return D, np.nan, np.nan
        W = np.sqrt(D["k"].to_numpy(float))                # a bin's variance carries ~ 2 var^2 / k: weight by sqrt(k)
        c = np.linalg.lstsq(np.column_stack([np.ones(len(D)), D["inv_n"]]) * W[:, None], D["var"].to_numpy(float) * W, rcond=None)[0]
        return D, float(c[0]), float(c[1])

    def _run34(v, x, nbin=P34_NBIN):
        """v minus its running median against x (equal-count bins, interpolated) — any trend with N removed, the sign kept."""
        v = np.asarray(v, float); x = np.asarray(x, float); ok = np.isfinite(v) & np.isfinite(x); out = np.full(len(v), np.nan)
        q = np.quantile(x[ok], np.linspace(0, 1, nbin + 1)); xc, mc = [], []
        for i in range(nbin):
            m = ok & (x >= q[i]) & ((x <= q[i + 1]) if i == nbin - 1 else (x < q[i + 1]))
            if m.sum() >= 8:
                xc.append(np.median(x[m])); mc.append(np.median(v[m]))
        if len(xc) >= 2:
            out[ok] = v[ok] - np.interp(x[ok], xc, mc)
        return out

    def _est34(k, n, s, sd_int):
        """Every N-aware version of the axis. sd_int = the intrinsic (deconvolved) spread, i.e. the shrinkage prior."""
        t2 = max(sd_int, 0.05) ** 2; w = t2 / (t2 + s ** 2); mu = float(np.nanmean(k))
        E = {"kappa (raw)": k, "significance (kappa-1/3)/s_iso": (k - 1 / 3) / s,
             "EB shrunk to the population mean": mu + (k - mu) * w, "EB shrunk to isotropy 1/3": 1 / 3 + (k - 1 / 3) * w,
             "N-trend residual (kappa | N)": _run34(k, np.log10(np.where(n > 0, n, np.nan)))}
        for n0 in (50, 200, 800):
            E["1/3 + (kappa-1/3) N/(N+%d)" % n0] = 1 / 3 + (k - 1 / 3) * n / (n + n0)
        return E

    FR34, ROWS34, LAD34, BIN34, OOF34 = {}, [], [], {}, {}
    print("=" * 122)
    print(f"Part 34 — {P34_KCOL} against the particle count.  The isotropic null of the Sales+12 estimator: E[kappa] = 1/3 and "
          f"s_iso(N) = {C_ISO34:.3f} / sqrt(N)")
    print(f"          (Var(v_phi^2/v^2) = {P34_VARF:.4f} for a uniform direction, N_eff = {P34_WEFF:g} N under the v^2 weighting)")
    print("=" * 122)
    for _b in P34_BOXES:
        _Q = S32[_b]["Q"].copy(); _y = np.asarray(S32[_b]["y"], bool)
        _add = QM[QM["box"] == _b][[c for c in ("snap", "gal_id", "ngas", "nstar", "mgas", "mstar") if c in QM.columns]]
        _Q = _Q.merge(_add, on=["snap", "gal_id"], how="left", suffixes=("", "_cat"))
        if len(_Q) != len(_y) or "ngas" not in _Q.columns:
            print(f"[{_b}] the particle counts do not join onto Part 32's rows — skipped"); continue
        _n = _Q["ngas"].to_numpy(float); _k = _Q[P34_KCOL].to_numpy(float)
        _s = C_ISO34 / np.sqrt(np.where(_n > 0, _n, np.nan))
        _Q["y"], _Q["n34"], _Q["s34"] = _y, _n, _s
        _ok = np.isfinite(_k) & np.isfinite(_s)
        # ── (a) the null ──
        _sd_null, _sd_obs = float(np.nanmean(_s[_ok])), float(np.nanstd(_k[_ok]))
        _f2 = float(np.nanmean(_k[_ok] > 1 / 3 + 2 * _s[_ok])); _gap = float(np.nanmedian(_k[_y]) - np.nanmedian(_k[~_y]))
        _hi = _ok & ~_y & (_k > P34_HIFP); _lo = _ok & ~_y & (_k <= P34_HIFP)
        _afp = mw_auc(np.log10(_n[_ok & ~_y]), (_k[_ok & ~_y] > P34_HIFP))[0]
        print(f"\n[{_b}] {len(_Q)} anchor rows, {int(_y.sum())} dusty ({100 * _y.mean():.0f} %);  N_gas 5/25/50/75/95 = "
              + " / ".join(f"{np.nanpercentile(_n, p):.0f}" for p in (5, 25, 50, 75, 95))
              + f"  ->  s_iso = {np.nanpercentile(_s, 95):.3f} / {np.nanmedian(_s):.3f} / {np.nanpercentile(_s, 5):.3f}")
        print(f"   (a) THE NULL: if every galaxy were isotropic the sampling noise alone would give sd {_sd_null:.3f} — observed {_sd_obs:.3f}; "
              f"it allows 2.3 % above 1/3 + 2 s_iso — observed {100 * _f2:.0f} %")
        print(f"       the dusty - pure gap in {P34_KCOL} is {_gap:+.3f} = {_gap / np.nanmedian(_s[_ok]):.1f} x the median sampling noise "
              f"(medians {np.nanmedian(_k[_y]):.2f} dusty vs {np.nanmedian(_k[~_y]):.2f} pure)")
        print(f"       the false positives (pure QGs above kappa {P34_HIFP:g}, n {int(_hi.sum())}): median N_gas {np.nanmedian(_n[_hi]):.0f} against "
              f"{np.nanmedian(_n[_lo]):.0f} for the rest of the pure class — AUC of log N toward high kappa inside it {_afp:.2f}, "
              + ("CHECK: they DO lean to few particles" if _afp < 0.40 else "they lean to MORE particles, the opposite of the artefact" if _afp > 0.60
                 else "no link: they are not the few-particle ones"))
        _D, _A0, _a1 = _varN34(_resid34(_k, [_Q["log_mstar"].to_numpy(float), _y.astype(float)]), _n)
        if len(_D) >= 3:
            print(f"       the residual scatter per N bin: " + "  ".join(f"N~{int(x.n_med)}:{np.sqrt(x.var):.3f}" for x in _D.itertuples())
                  + f"   (fit Var = {_A0:.4f} + {_a1:.2f}/N; N spans only x{np.nanpercentile(_n, 95) / np.nanpercentile(_n, 5):.1f} here, a weak 1/N lever arm)")
        # the intrinsic spread: the within-class variance with the sampling noise deconvolved — the shrinkage prior and the ceiling
        _vobs = 0.5 * (np.nanvar(_k[_ok & _y], ddof=1) + np.nanvar(_k[_ok & ~_y], ddof=1)); _vn = float(np.nanmean(_s[_ok] ** 2))
        _sd_int = float(np.sqrt(max(_vobs - _vn, 1e-4))); _d = float(np.nanmean(_k[_ok & _y]) - np.nanmean(_k[_ok & ~_y]))
        ROWS34.append(dict(box=_b, block="null", label="isotropic null", n=int(_ok.sum()), value=_sd_obs, ref=_sd_null,
                           extra=f"{100 * _f2:.0f} % above 1/3+2s (null 2.3 %); gap {_gap:+.3f} = {_gap / np.nanmedian(_s[_ok]):.1f} s_iso; FP AUC {_afp:.2f}"))
        # ── (b) where the signal lives ──
        _nb = int(np.clip(int(_y.sum()) // 8, 2, 5)); _q = np.nanquantile(_n, np.linspace(0, 1, _nb + 1)); _rows = []
        for i in range(_nb):
            _m = (_n >= _q[i]) & ((_n <= _q[i + 1]) if i == _nb - 1 else (_n < _q[i + 1]))
            _a, _l, _h, _nn = _auc34(_k[_m], _y[_m])
            _rows.append(dict(lo_n=_q[i], hi_n=_q[i + 1], auc=_a, alo=_l, ahi=_h, n=_nn, f_dusty=float(np.nanmean(_y[_m])),
                              s_iso=float(np.nanmedian(_s[_m]))))
            ROWS34.append(dict(box=_b, block="bins", label=f"N {_q[i]:.0f}-{_q[i + 1]:.0f}", n=_nn, value=_a, ref=np.nan,
                               extra=f"[{_l:.2f},{_h:.2f}], {100 * _rows[-1]['f_dusty']:.0f} % dusty, s_iso {_rows[-1]['s_iso']:.3f}"))
        BIN34[_b] = pd.DataFrame(_rows)
        print(f"   (b) THE SIGNAL INSIDE BINS OF N (no function of kappa can change these): "
              + "   ".join(f"N {int(r.lo_n)}-{int(r.hi_n)}: {r.auc:.2f} [{r.alo:.2f},{r.ahi:.2f}] (n {r.n}, {100 * r.f_dusty:.0f} % dusty)" for r in BIN34[_b].itertuples()))
        # ── (c) the ceiling and the family ──
        _c1 = _phi34(_d / np.sqrt(2 * max(_vobs - _vn, 1e-6))); _c0 = _phi34(_d / np.sqrt(2 * _vobs))
        print(f"   (c) THE CEILING: within-class variance {_vobs:.4f} = intrinsic {max(_vobs - _vn, 0):.4f} + sampling {_vn:.4f} "
              f"({100 * _vn / _vobs:.1f} % of it)  ->  perfect de-noising moves a Gaussian AUC {_c0:.3f} -> {_c1:.3f} (+{_c1 - _c0:.3f})")
        for _f in P34_NEFF:
            _vt = max(_vobs - _f * _vn, 1e-6)
            print(f"       even if the gas were clumped so that N_eff = N/{_f} ({100 * _f * _vn / _vobs:.0f} % of the spread): ceiling {_phi34(_d / np.sqrt(2 * _vt)):.3f} "
                  f"(+{_phi34(_d / np.sqrt(2 * _vt)) - _c0:.3f})")
        ROWS34.append(dict(box=_b, block="ceiling", label="perfect de-noising", n=int(_ok.sum()), value=_c1, ref=_c0,
                           extra=f"sampling {100 * _vn / _vobs:.1f} % of the within-class variance; sd_int {_sd_int:.3f}"))
        # ── (d) every version, and the sham ──
        _E = _est34(_k, _n, _s, _sd_int)
        _ctrl = {"log N_gas (control: the count itself)": np.log10(np.where(_n > 0, _n, np.nan)),
                 "kappa_star (the well-sampled twin)": _Q["kappa_star"].to_numpy(float) if "kappa_star" in _Q.columns else np.full(len(_Q), np.nan),
                 "log M* (Part 32's third axis)": _Q["log_mstar"].to_numpy(float)}
        print(f"   (d) EVERY VERSION OF THE AXIS (single-variable AUC toward dusty, bootstrap 16-84; the intrinsic spread is "
              f"{_sd_int:.3f}, so the shrinkage weight w(N) runs {np.nanpercentile(max(_sd_int, 0.05) ** 2 / (max(_sd_int, 0.05) ** 2 + _s ** 2), 5):.2f}-"
              f"{np.nanpercentile(max(_sd_int, 0.05) ** 2 / (max(_sd_int, 0.05) ** 2 + _s ** 2), 95):.2f})")
        for _nm, _v in list(_E.items()) + list(_ctrl.items()):
            _a, _l, _h, _nn = _auc34(_v, _y)
            if np.isfinite(_a):
                print(f"       {_nm:42s} {_a:.3f}  [{_l:.3f}, {_h:.3f}]")
                LAD34.append(dict(box=_b, kind="axis" if _nm in _E else "control", var=_nm, auc=_a, lo=_l, hi=_h, n=_nn))
                ROWS34.append(dict(box=_b, block="ladder", label=_nm, n=_nn, value=_a, ref=np.nan, extra=f"[{_l:.3f},{_h:.3f}]"))
        _t2 = max(_sd_int, 0.05) ** 2
        print("       what the N0 mean as a noise level: " + ", ".join(f"N0 {n0} = the isotropic noise x{n0 * _t2 / C_ISO34 ** 2:.0f}" for n0 in (50, 200, 800))
              + " (in variance; x1 is the physical value)")
        # the sham: the same shrinkage driven by N shuffled inside log M* deciles
        _lm = _Q["log_mstar"].to_numpy(float); _dec = np.searchsorted(np.nanquantile(_lm, np.linspace(0, 1, 11))[1:-1], _lm)
        _mu = float(np.nanmean(_k)); _sh = []
        for _ in range(40):
            _ns = _n.copy()
            for _d0 in np.unique(_dec):
                _m = _dec == _d0; _v = _ns[_m]; _rng34.shuffle(_v); _ns[_m] = _v
            _w = _t2 / (_t2 + C_ISO34 ** 2 / np.where(_ns > 0, _ns, np.nan))
            _sh.append(mw_auc(_mu + (_k - _mu) * _w, _y)[0])
        _atrue = mw_auc(_mu + (_k - _mu) * (_t2 / (_t2 + _s ** 2)), _y)[0]
        print(f"       SHAM: the EB shrinkage with the true N gives {_atrue:.3f}; with N shuffled at fixed M* {np.mean(_sh):.3f} +- {np.std(_sh):.3f} "
              f"(raw {mw_auc(_k, _y)[0]:.3f}) — the physical correction is doing nothing at all")
        ROWS34.append(dict(box=_b, block="sham", label="EB shrinkage, true vs shuffled N", n=len(_Q), value=_atrue, ref=float(np.mean(_sh)),
                           extra=f"raw {mw_auc(_k, _y)[0]:.3f}, sham sd {np.std(_sh):.3f}"))
        # the out-of-fold scores: does a de-noised axis make Part 32's mass axis redundant?
        OOF34[_b] = {}
        if "_cv32" in globals():
            _Q["k_eb"] = _E["EB shrunk to isotropy 1/3"]; _Q["k_z"] = _E["significance (kappa-1/3)/s_iso"]
            _Q["lngas"] = np.log10(np.where(_n > 0, _n, np.nan))
            for _nm, _cols in (("kappa", [P34_KCOL]), ("kappa + M*", [P34_KCOL, "log_mstar"]), ("EB kappa", ["k_eb"]),
                               ("EB kappa + M*", ["k_eb", "log_mstar"]), ("significance + M*", ["k_z", "log_mstar"]),
                               ("kappa + log N_gas", [P34_KCOL, "lngas"]), ("kappa + log N_gas + M*", [P34_KCOL, "lngas", "log_mstar"])):
                _okc = np.isfinite(_Q[_cols].to_numpy(float)).all(axis=1)
                if _okc.sum() < 60 or _y[_okc].sum() < 5:
                    continue
                _p, _a, _sd = _cv32(_Q.loc[_okc, _cols].to_numpy(float), _y[_okc].astype(int))
                OOF34[_b][_nm] = (_a, _sd)
                ROWS34.append(dict(box=_b, block="oof", label=_nm, n=int(_okc.sum()), value=_a, ref=_sd, extra="+".join(_cols)))
            print("       out-of-fold scores: " + "   ".join(f"{k_} {v[0]:.3f}" for k_, v in OOF34[_b].items())
                  + "   <- the mass axis is not a resolution proxy: it survives a de-noised kappa")
        # ── the practical block: Part 22's split at the median, at a significance cut, at a fixed cut of the same level ──
        _dose = _Q[P32_DOSE].to_numpy(float); _dc = CUT28.get(_b, np.nan); _kc = float(np.nanmedian(_k))
        _fix = float(np.nanmedian(1 / 3 + P34_SIG * _s))
        print(f"   (e) PART 22'S SPLIT IN PRACTICE (the red group = the high side of the geometry axis at a dose below {_dc:.2f}):")
        for _lab, _hs in ((f"kappa > {_kc:.2f} (the median — Part 22's rule)", _k > _kc),
                          (f"kappa > 1/3 + {P34_SIG:g} s_iso(N) (the significance cut)", _k > 1 / 3 + P34_SIG * _s),
                          (f"kappa > {_fix:.2f} (a FIXED cut at the same level)", _k > _fix)):
            _red = _hs & (_dose < _dc)
            print(f"       {_lab:52s} rotating {int(_hs.sum()):4d} ({100 * np.nanmean(_y[_hs]):4.1f} % dusty)  ->  red group {int(_red.sum()):4d} "
                  f"({100 * np.nanmean(_y[_red]):4.1f} % dusty, {100 * (_y & _red).sum() / max(int(_y.sum()), 1):4.1f} % of all dusty)")
            ROWS34.append(dict(box=_b, block="rules", label=_lab, n=int(_red.sum()), value=float(np.nanmean(_y[_red])),
                               ref=float(np.nanmean(_y[_hs])), extra=f"recall {100 * (_y & _red).sum() / max(int(_y.sum()), 1):.1f} %, rotating {int(_hs.sum())}"))
        print(f"       {int(((_k > _kc) != (_k > 1 / 3 + P34_SIG * _s)).sum())} of {len(_k)} galaxies change side between the median rule and the "
              f"significance rule, {int(((_k > _fix) != (_k > 1 / 3 + P34_SIG * _s)).sum())} between the FIXED cut and it: what the significance "
              f"cut buys is the physical threshold, not its N dependence")
        FR34[_b] = _Q

    # ── the figure ──
    if FR34:
        _NB = len(FR34); _BX = [b for b in P34_BOXES if b in FR34]
        fig, AX = plt.subplots(4, _NB, figsize=(5.0 * _NB, 16.4))
        AX = np.atleast_2d(np.asarray(AX).reshape(4, _NB))
        _SHOW = ["kappa (raw)", "significance (kappa-1/3)/s_iso", "EB shrunk to isotropy 1/3", "EB shrunk to the population mean",
                 "1/3 + (kappa-1/3) N/(N+200)", "N-trend residual (kappa | N)", "log N_gas (control: the count itself)",
                 "log M* (Part 32's third axis)"]
        for _j, _b in enumerate(_BX):
            _Q = FR34[_b]; _y = _Q["y"].to_numpy(bool); _k = _Q[P34_KCOL].to_numpy(float); _n = _Q["n34"].to_numpy(float); _s = _Q["s34"].to_numpy(float)
            _ok = np.isfinite(_k) & np.isfinite(_s)
            _vobs = 0.5 * (np.nanvar(_k[_ok & _y], ddof=1) + np.nanvar(_k[_ok & ~_y], ddof=1)); _vn = float(np.nanmean(_s[_ok] ** 2))
            _sd_int = float(np.sqrt(max(_vobs - _vn, 1e-4))); _d = float(np.nanmean(_k[_ok & _y]) - np.nanmean(_k[_ok & ~_y]))
            _araw = mw_auc(_k, _y)[0]; _c1 = _phi34(_d / np.sqrt(2 * max(_vobs - _vn, 1e-6)))
            # (a) kappa against the particle count
            ax = AX[0, _j]
            for _lab, _m, _c, _z, _ms in (("pure QG", ~_y, POPS["pure"]["color"], 2, P34_MS), ("dusty QG", _y, POPS["dusty"]["color"], 4, P34_MS + 1.6)):
                ax.plot(_n[_m], _k[_m], "o", ms=_ms, mfc=_c, mec="none", alpha=0.55, label=f"{_lab} ({int(_m.sum())})", zorder=_z)
            _nn = np.logspace(np.log10(max(np.nanmin(_n), 30)), np.log10(np.nanmax(_n)), 60)
            ax.fill_between(_nn, 1 / 3 - C_ISO34 / np.sqrt(_nn), 1 / 3 + C_ISO34 / np.sqrt(_nn), color="0.35", alpha=0.35, lw=0, zorder=5,
                            label=r"isotropic: $1/3 \pm s_{\rm iso}(N)$")
            ax.plot(_nn, 1 / 3 + P34_SIG * C_ISO34 / np.sqrt(_nn), "k--", lw=1.0, zorder=6, label=r"$1/3 + %g\,s_{\rm iso}(N)$" % P34_SIG)
            for _m, _c in ((~_y, POPS["pure"]["color"]), (_y, POPS["dusty"]["color"])):
                _xc, _mc, _qq = [], [], np.nanquantile(_n[_m], np.linspace(0, 1, 7))
                for _i in range(6):
                    _mm = _m & (_n >= _qq[_i]) & ((_n <= _qq[_i + 1]) if _i == 5 else (_n < _qq[_i + 1]))
                    if _mm.sum() >= 8:
                        _xc.append(np.median(_n[_mm])); _mc.append(np.median(_k[_mm]))
                ax.plot(_xc, _mc, "-", color=_c, lw=3.0, zorder=7, solid_capstyle="round")
            ax.set_xscale("log"); ax.set_xlabel(r"$N_{\rm gas}$ particles in the galaxy"); ax.set_ylabel(r"$\kappa_{\rm rot}$(gas), whole galaxy")
            ax.set_title(f"{globals().get('BOX_LABEL', {}).get(_b, _b)} — (a) the noise the estimator can have", fontsize=10)
            ax.legend(fontsize=7.8, loc="upper right", framealpha=0.92); ax.set_ylim(-0.02, 1.02)
            # (b) the signal inside bins of N
            ax = AX[1, _j]; _D = BIN34[_b]; _xx = np.arange(len(_D))
            ax.errorbar(_xx, _D["auc"], yerr=[_D["auc"] - _D["alo"], _D["ahi"] - _D["auc"]], fmt="o", color="k", ms=7, lw=1.4, capsize=3,
                        zorder=3, label=r"AUC of raw $\kappa$ inside the bin")
            ax.axhline(_araw, color="#7570b3", lw=2.0, label=f"all rows ({_araw:.2f})"); ax.axhline(0.5, color="0.6", lw=1.0, ls=":")
            ax.set_xticks(_xx); ax.set_xticklabels([f"{int(r.lo_n)}–{int(r.hi_n)}\n{int(r.n)} gal\n{100 * r.f_dusty:.0f} % dusty" for r in _D.itertuples()], fontsize=6.0)
            ax.set_xlim(-0.5, len(_D) - 0.5); ax.set_xlabel(r"$N_{\rm gas}$ bin"); ax.set_ylabel("AUC toward dusty")
            ax.set_title("(b) the signal inside bins of $N$", fontsize=10); ax.legend(fontsize=8, loc="lower right")
            # (c) what a weighting by N can buy
            ax = AX[2, _j]
            _lm = _Q["log_mstar"].to_numpy(float); _dec = np.searchsorted(np.nanquantile(_lm, np.linspace(0, 1, 11))[1:-1], _lm)
            _fam, _sham, _ceil = [], [], []
            for _f in P34_FGRID:
                _N0 = _f * C_ISO34 ** 2 / max(_sd_int, 0.05) ** 2
                _fam.append(mw_auc(1 / 3 + (_k - 1 / 3) * _n / (_n + _N0), _y)[0])
                _rr = []
                for _ in range(12):
                    _ns = _n.copy()
                    for _d0 in np.unique(_dec):
                        _m = _dec == _d0; _v = _ns[_m]; _rng34.shuffle(_v); _ns[_m] = _v
                    _rr.append(mw_auc(1 / 3 + (_k - 1 / 3) * _ns / (_ns + _N0), _y)[0])
                _sham.append(float(np.mean(_rr))); _ceil.append(_phi34(_d / np.sqrt(2 * max(_vobs - _f * _vn, 1e-6))))
            _akm = OOF34[_b].get("kappa + M*", (np.nan, np.nan))[0]
            ax.plot(P34_FGRID, _fam, "o-", color="#1b9e77", lw=2.4, ms=6, label=r"$1/3+(\kappa-1/3)\,N/(N+N_0)$")
            ax.plot(P34_FGRID, _sham, "s--", color="0.45", lw=1.6, ms=5, label=r"the same, $N$ shuffled at fixed $M_\star$")
            ax.plot(P34_FGRID, _ceil, "-", color="#d95f02", lw=2.0, label="ceiling of perfect de-noising")
            ax.axhline(_araw, color="k", ls="--", lw=1.2); ax.axvline(1.0, color="0.3", lw=1.2, ls=":")
            ax.plot([1.0], [_c1], "*", color="#d95f02", ms=16, zorder=6)
            _all = np.concatenate([_fam, _sham, [_araw, _c1] + ([_akm] if np.isfinite(_akm) else [])])
            _y0, _y1 = np.min(_all) - 0.010, np.max(_all) + 0.012
            ax.set_ylim(_y0, _y1 + 0.30 * (_y1 - _y0))
            ax.text(P34_FGRID[-1], _araw - 0.004, r"raw $\kappa$ (%.3f)" % _araw, ha="right", va="top", fontsize=7.6)
            ax.text(1.35, _c1, "  the ceiling at the physical\n  noise level (%.3f)" % _c1, ha="left", va="center", fontsize=7.6, color="#d95f02")
            if np.isfinite(_akm):
                ax.axhline(_akm, color="#e7298a", lw=2.0)
                ax.text(P34_FGRID[-1], _akm + 0.002, r"$\kappa$ + $\log M_\star$, out-of-fold (%.3f)" % _akm, ha="right", va="bottom", fontsize=7.6, color="#e7298a")
            ax.set_xscale("log"); ax.set_xlabel(r"assumed sampling noise $\div$ the isotropic value")
            ax.set_ylabel("AUC toward dusty"); ax.set_title("(c) what a weighting by $N$ can buy", fontsize=10)
            ax.legend(fontsize=7.4, loc="upper left", framealpha=0.92)
            # (d) every version of the axis
            ax = AX[3, _j]
            _L = {r["var"]: r for r in LAD34 if r["box"] == _b}
            _names = [v for v in _SHOW if v in _L and np.isfinite(_L[v]["auc"])]
            _yy = np.arange(len(_names) + len(OOF34[_b]))[::-1]; _lbl = []
            for _i, _v in enumerate(_names):
                _r = _L[_v]
                ax.barh(_yy[_i], _r["auc"] - 0.5, left=0.5, color="0.35" if _r["kind"] == "control" else "#1b9e77", height=0.66, zorder=2)
                ax.plot([_r["lo"], _r["hi"]], [_yy[_i]] * 2, "-", color="k", lw=1.2, zorder=3)
                ax.text(_r["auc"] + 0.004, _yy[_i], f"{_r['auc']:.3f}", va="center", fontsize=7.8)
                _lbl.append(_v.replace(" (kappa-1/3)/s_iso", "").replace("1/3 + (kappa-1/3) ", "").replace(" (kappa | N)", ""))
            for _i, (_nm, (_a, _sd)) in enumerate(OOF34[_b].items(), start=len(_names)):
                ax.barh(_yy[_i], _a - 0.5, left=0.5, color="#d95f02" if "M*" in _nm else "#66a61e", height=0.66, zorder=2)
                ax.plot([_a - _sd, _a + _sd], [_yy[_i]] * 2, "-", color="k", lw=1.2, zorder=3)
                ax.text(_a + 0.004, _yy[_i], f"{_a:.3f}", va="center", fontsize=7.8)
                _lbl.append(_nm + " [OOF]")
            ax.set_yticks(_yy[:len(_lbl)]); ax.set_yticklabels(_lbl, fontsize=8)
            ax.set_xlim(0.5, max([_L[v]["auc"] for v in _names] + [a for a, _ in OOF34[_b].values()] + [0.9]) + 0.06)
            ax.set_xlabel("AUC toward dusty"); ax.axvline(_araw, color="k", ls="--", lw=1.0, zorder=1)
            ax.axvline(_c1, color="#d95f02", ls=":", lw=1.6, zorder=1)
            ax.set_title("(d) every version of the axis\n(black: raw, orange dots: the de-noising ceiling)", fontsize=9.5)
        fig.suptitle(r"Part 34 — the geometry axis against the particle count: is $\kappa_{\rm rot}$(gas) rotation or shot noise?"
                     + f"   (the isotropic null: $s_{{\\rm iso}}(N) = {C_ISO34:.2f}/\\sqrt{{N}}$, $N$ = the gas particles of the galaxy)", fontsize=11.5)
        fig.tight_layout(rect=(0, 0, 1, 0.975))
        paper_save(fig, "agn_states_kappa_resolution")
        plt.show()
    if ROWS34:
        pd.DataFrame(ROWS34).to_csv(os.path.join(OUT, "agn_states_kappa_resolution.csv"), index=False)
        print(f"\n-> {os.path.join(OUT, 'agn_states_kappa_resolution.csv')}")

## Part 35 — why $\kappa_{\rm gas}$ and $M_\star$: the split is in the grains, not in the gas and not in the metals

Part 32 found the winning divider and Part 34 cleared it of being shot noise, so the question left is *why those two axes*. This part answers it with an identity that holds galaxy by galaxy, with nothing fitted:

$$\log \frac{M_{\rm dust}}{M_\star} \;=\; \underbrace{\log \frac{M_{\rm dust}}{Z\,M_{\rm gas}}}_{\textbf{the depletion}} \;+\; \underbrace{\log Z}_{\textbf{the metals}} \;+\; \underbrace{\log \frac{M_{\rm gas}}{M_\star}}_{\textbf{the gas}}$$

The label the whole notebook splits on is the sum of exactly three things — how many metals the gas carries, how much gas there is per star, and what share of those metals sits in grains — so the dusty / dust-poor difference can be *decomposed* instead of correlated. The metallicity is the one term the catalogue cache does not carry: it comes from the histories at the anchor (Part 2a's `zgas` = caesar's mass-weighted gas metallicity), so a row without a main branch drops out of the panels that need it. **Fig A** is the waterfall of the three terms per box and then each term against each candidate cause ($\kappa_{\rm gas}$, $\log M_\star$, the duty fraction), one running median per box. **Fig B** is the plane the division is made on, coloured by the median depletion in equal-count cells, and the ladder of raw against partial (rank-residualised) correlations toward that depletion. Printed: the identity residual, the gap table, the quadrant anatomy, the ladder and Part 34's resolution control repeated for **both** axes.

**What it says** (Part 0's sample, `DUSTY_CUT` $= -4$; re-run for the numbers of the knobs in force). The dusty − dust-poor gap in the label is $+2.4 / +2.6 / +2.1$ dex in m100 / m50 / m25 and **all of it is the depletion term**: the two populations have the same metallicity to $0.1$ dex (AUC of $Z$ toward dusty $\approx 0.47$, and the sign is *against* the dusty ones) and the dust-poor are if anything the more gas-rich ($-0.2$ dex, AUC $0.33$). A dust-poor quenched galaxy is not metal-poor and not gas-poor: it keeps about **one metal in a thousand** in grains ($\log M_{\rm dust}/(M_{\rm dust}+Z M_{\rm gas}) \approx -3.3$) where a dusty one keeps **one in five** ($\approx -0.7$). Both axes act on that term and on nothing else — at fixed mass, metallicity and dose $\rho(\kappa_{\rm gas},\,{\rm depletion}) = +0.47 / +0.37 / +0.31$, and at fixed $\kappa$, metallicity and dose $\rho(\log M_\star,\,{\rm depletion}) = +0.55 / +0.55 / +0.32$; the dose's own partial is $-0.17 / -0.19 / -0.02$, small but **not** mediated by the kinematics ($\rho(\text{dose},\kappa \mid M_\star) \approx 0$ in m100, so Part 32's "the dose is already written into the gas kinematics" is too strong — the dose is a weak *independent* term, not a hidden one). The two axes are nearly orthogonal ($\rho(\kappa_{\rm gas}, \log M_\star) = +0.02$) and nearly additive in the depletion, which is why the AND rule works: the quadrants of the median cut run $1\,\% / 5\,\% / 3\,\% / 21\,\%$ dusty in m100 and $0 / 32 / 0 / 33\,\%$ in m25, each axis worth $\sim 0.4$ dex of depletion on its own and $\sim 1.7$ dex together. The mass does buy metals ($\rho(\log M_\star, Z) = +0.53$, the mass–metallicity relation) — but the metals are not what the split is about.

**What the mass axis is standing in for.** In m25, the box that resolves the gas, the strongest single term is not the mass: it is the **gas surface density** $\Sigma_{\rm gas} = 0.5\,M_{\rm gas}/\pi R_{50}({\rm gas})^2$ (partial $+0.39$ against $+0.28$ for $\log M_\star$ and $+0.30$ for $\kappa$) — the density that sets the grain-growth rate in SIMBA's dust model. In m100 and m50 that same term is worth nothing ($+0.02$, $-0.12$): $R_{50}$(gas) of a 50–100-particle galaxy is not a radius. So the honest reading is that $\kappa_{\rm gas}$ and $M_\star$ are the *observable* coarse-box stand-ins for "is the gas a settled, dense, self-shielded disc" — the second one only because the boxes cannot measure the density directly.

**Caveats.** The depletion is a catalogue-level ratio, not a measured rate: it says the grains are gone, not by which channel (Parts 19–20 measure the sputtering and the growth particle by particle, m25 only). $Z$ is the mass-weighted gas metallicity of the whole galaxy, and in SIMBA the grain growth depletes it, so the depletion and the metals are not fully independent quantities — which is why the bounded form $M_{\rm dust}/(M_{\rm dust}+Z M_{\rm gas})$ is drawn as well as the ratio. The frame is anchor rows, not distinct galaxies (Part 28's convention).

In [ ]:
# ── Part 35 — WHY kappa_gas AND log M*: the split is in the GRAINS, not in the gas and not in the metals ──
# Part 32 scored every candidate divider and the pair that won was the catalogue kappa_rot of the gas and the stellar mass
# (out-of-fold AUC 0.88 / 0.93 / 0.76 in m100 / m50 / m25, the dose adding ~0.002 on top). This part asks WHAT THOSE TWO AXES
# ARE MADE OF, on the identity that holds galaxy by galaxy with no fitting at all:
#
#       log M_dust/M*   =   log [ M_dust / (Z M_gas) ]   +   log Z   +   log M_gas/M*
#          the label            THE DEPLETION                METALS        THE GAS
#                          (dust per unit metal)
#
# so the dusty / dust-poor label is the sum of exactly three things: how many metals the gas carries, how much gas there is per
# star, and what fraction of those metals sits in grains. The part measures the dusty - dust-poor gap in each of the three, then
# draws every one of them against every candidate cause (kappa_gas, log M*, the AGN duty fraction) so the reader can see which
# cause moves which factor.
#   FIG A  (row 0) THE WATERFALL, one panel per box: the mean dusty - dust-poor gap split into the three terms, which add to the
#          gap in the label itself;  (rows 1-3) THE FACTOR x CAUSE GRID: each of the three factors (rows) against each of the
#          three causes (columns), one running median + 16-84 band per box — the picture of what each axis actually does.
#   FIG B  (row 0) THE PLANE the division is made on, kappa_gas x log M*, coloured by the MEDIAN DEPLETION in equal-count cells,
#          the dusty rows ringed, the four median quadrants labelled with their dusty share;  (row 1) the correlation ladder
#          toward the depletion: the raw Spearman rho of every candidate against the rank-partial rho with the others held fixed
#          (kappa_gas, log M*, log Z, log Sigma_gas, the dose, log N_gas) — what survives when the rest is controlled.
#   PRINTED per box: the identity residual (a check that the three terms really are the label), the gap table, the quadrant
#          anatomy (n, dusty share, median depletion / gas / metals / dose / age per quadrant), the ladder, and the resolution
#          control of Part 34 repeated for BOTH axes (their AUC inside quartiles of the gas-particle count).
#   Products: agn_states_why_factors.{png,pdf}, agn_states_why_plane.{png,pdf}, agn_states_why_{gaps,ladder,quadrants}.csv
# THE METALLICITY comes from the histories (the catalogue cache carries none): Part 2a reads metallicities.mass_weighted at the
# anchor into DOSE_ALL["zgas"], so a row without a main-branch chain has no depletion and drops out of every panel that needs it.
P35_BOXES  = ["cis100", "cis50", "cis25"]
P35_ZSUN   = 0.0134            # Asplund+09 Z_sun: log Z is drawn in solar units, the depletion does not depend on it
P35_DOSE   = None              # the dose axis (None = f_x_post from Part 28's frames if they exist, else Part 2a's whole-branch f_x)
P35_KFRAC, P35_NMIN, P35_NGRID = 0.30, 12, 40    # the running median of the grid panels: the fraction of the box's rows in the window, the floor, the points
P35_NCELL  = 5                 # FIG B row 0: equal-count cells per axis
P35_CMAP   = "magma"
P35_MS     = 7
P35_PLANE_LIM = None           # (kappa lims, log M* lims); None = from the data
BOXC35 = globals().get("BOXC28", {"cis100": "#252525", "cis50": "#e6ab02", "cis25": "#1b9e77"})
BOXS35 = globals().get("BOXS28", {"cis100": "m100", "cis50": "m50", "cis25": "m25"})

if "QM" not in globals():
    print("Part 35: needs Part 0 — run it first.")
elif "DOSE_ALL" not in globals() or not len(DOSE_ALL):
    print("Part 35: needs Part 2a — the metallicity at the anchor (DOSE_ALL['zgas']) comes from its pass over the histories. Run it first.")
else:
    _dose35 = P35_DOSE or ("f_x_post" if ("FR28" in globals() and any(b in FR28 for b in P35_BOXES)) else "f_x")
    _dlab35 = globals().get("DLAB28", {}).get(_dose35, "X-ray duty fraction since SFT" if _dose35 == "f_x_post" else "X-ray duty fraction")

    def _track35(x, y, kfrac=P35_KFRAC, nmin=P35_NMIN, ngrid=P35_NGRID):
        """The running median of y along x with its 16-84 spread (Part 26's estimator, repeated here so the part stands alone)."""
        x, y = np.asarray(x, float), np.asarray(y, float); ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
        if len(x) < nmin:
            return None
        k = int(min(max(int(round(kfrac * len(x))), nmin), len(x)))
        xg = np.linspace(np.percentile(x, 5), np.percentile(x, 95), ngrid)
        med, lo, hi = (np.full(ngrid, np.nan) for _ in range(3))
        for i, xc in enumerate(xg):
            v = y[np.argsort(np.abs(x - xc), kind="stable")[:k]]
            med[i], lo[i], hi[i] = np.median(v), np.percentile(v, 16), np.percentile(v, 84)
        return dict(x=xg, med=med, lo=lo, hi=hi, k=k, n=len(x))

    def _rankres35(x, Z):
        """x rank-residualised on the block Z (also ranked): the 'at fixed Z' version of x."""
        from scipy.stats import rankdata
        x, Z = np.asarray(x, float), np.asarray(Z, float)
        if Z.ndim == 1:
            Z = Z[:, None]
        m = np.isfinite(x) & np.isfinite(Z).all(axis=1)
        out = np.full(len(x), np.nan)
        if m.sum() < 3 * Z.shape[1] + 8:
            return out, m
        rx = rankdata(x[m]); RZ = np.column_stack([rankdata(Z[m, j]) for j in range(Z.shape[1])] + [np.ones(int(m.sum()))])
        out[m] = rx - RZ @ np.linalg.lstsq(RZ, rx, rcond=None)[0]
        return out, m

    def _partial35(x, y, Z):
        """Rank partial correlation rho(x, y | Z): Spearman of the two rank residuals (Part 21's p21_partial_multi, standalone)."""
        rx, m1 = _rankres35(x, Z); ry, m2 = _rankres35(y, Z); m = m1 & m2 & np.isfinite(rx) & np.isfinite(ry)
        if m.sum() < 12:
            return np.nan, int(m.sum())
        a, b = rx[m] - rx[m].mean(), ry[m] - ry[m].mean(); d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
        return (float((a * b).sum() / d) if d > 0 else np.nan), int(m.sum())

    # ── the frame: Part 0's sample + the anchor metallicity (Part 2a) + the dose axis Parts 28-33 split on ──
    F35, GAP35, LAD35, QUAD35 = {}, [], [], []
    _dcols = ["box", "snap", "gal_id", "zgas", "e_x", "f_x", "n_x", "t_quiet", "t_anchor"]
    for _b in P35_BOXES:
        _Q = QM[QM["box"] == _b].copy()
        if not len(_Q):
            print(f"[{_b}] no rows in Part 0's sample — skipped"); continue
        _Q = _Q.merge(DOSE_ALL[[c for c in _dcols if c in DOSE_ALL.columns]], on=KEY, how="left")
        if _dose35 == "f_x_post":
            if "FR28" in globals() and _b in FR28 and "f_x_post" in FR28[_b].columns:
                _Q = _Q.merge(FR28[_b][["snap", "gal_id", "f_x_post"]].drop_duplicates(["snap", "gal_id"]), on=["snap", "gal_id"], how="left")
            else:
                _Q["f_x_post"] = np.nan
        with np.errstate(all="ignore"):
            _md, _mg, _z = (_Q[c].to_numpy(float) for c in ("mdust", "mgas", "zgas"))
            _mzz = _z * _mg                                                        # the metal mass in the gas phase
            _Q["ldtm"]  = np.log10(np.where((_md > 0) & (_mzz > 0), _md / np.where(_mzz > 0, _mzz, np.nan), np.nan))
            _Q["lfdep"] = np.log10(np.where((_md >= 0) & (_mzz > 0), np.clip(_md / (_md + _mzz), 1e-6, None), np.nan))   # the bounded version: the share of the metals in grains
            _Q["lz"]    = np.log10(np.where(_z > 0, _z / P35_ZSUN, np.nan))
            _Q["lzabs"] = np.log10(np.where(_z > 0, _z, np.nan))
            _Q["lsig_gas"] = np.log10(np.where((_mg > 0) & (_Q["r_half_gas"].to_numpy(float) > 0),
                                               0.5 * _mg / (np.pi * _Q["r_half_gas"].to_numpy(float) ** 2), np.nan))
            _Q["lngas"] = np.log10(np.where(_Q["ngas"].to_numpy(float) > 0, _Q["ngas"].to_numpy(float), np.nan))
            _Q["ident"] = _Q["ldtm"] + _Q["lzabs"] + _Q["lfgas"] - _Q["lfd"]       # the identity residual: 0 wherever the dust is not on the floor
        F35[_b] = _Q
        _y = (_Q["pop"].to_numpy() == "dusty")
        _res = _Q["ident"].to_numpy(float); _res = _res[np.isfinite(_res)]
        print(f"[{_b}] {len(_Q)} anchor rows, {int(_y.sum())} dusty; metallicity on {int(np.isfinite(_Q['zgas']).sum())} of them "
              f"(the rest have no main-branch chain).  identity log f_dust = log DTM + log Z + log f_gas: |residual| max "
              f"{np.nanmax(np.abs(_res)) if len(_res) else np.nan:.2e} over {len(_res)} rows")
        # ── the gap table: the mean (exactly additive) and the median (robust) dusty - dust-poor difference of each term ──
        print(f"     the dusty - dust-poor gap, term by term (mean | median) [dex]:")
        for _c, _lab in (("lfd", "log M_dust/M*  (the label)"), ("ldtm", "log DTM = M_dust/(Z M_gas)"), ("lzabs", "log Z"), ("lfgas", "log M_gas/M*"), ("lfdep", "log metal share in grains")):
            _v = _Q[_c].to_numpy(float); _ok = np.isfinite(_v)
            _gm = np.nanmean(_v[_ok & _y]) - np.nanmean(_v[_ok & ~_y]); _gd = np.nanmedian(_v[_ok & _y]) - np.nanmedian(_v[_ok & ~_y])
            print(f"       {_lab:32s} {_gm:+6.2f} | {_gd:+6.2f}   (dusty {np.nanmedian(_v[_ok & _y]):+.2f}, dust-poor {np.nanmedian(_v[_ok & ~_y]):+.2f})")
            GAP35.append(dict(box=_b, term=_c, label=_lab, gap_mean=_gm, gap_median=_gd,
                              med_dusty=float(np.nanmedian(_v[_ok & _y])), med_pure=float(np.nanmedian(_v[_ok & ~_y])),
                              n_dusty=int((_ok & _y).sum()), n_pure=int((_ok & ~_y).sum()), auc=mw_auc(_v, _y)[0]))
        # ── the ladder: every candidate against the depletion, raw and with the others held fixed ──
        _cands = [("kappa_gas", r"$\kappa_{\rm rot}$(gas)"), ("log_mstar", r"$\log M_\star$"), ("lz", r"$\log Z$"),
                  ("lsig_gas", r"$\log \Sigma_{\rm gas}$"), (_dose35, _dlab35), ("lngas", r"$\log N_{\rm gas}$")]
        _cands = [(c, l) for c, l in _cands if c in _Q.columns and np.isfinite(_Q[c].to_numpy(float)).sum() >= 4 * P35_NMIN]
        print("     toward log DTM: raw rho | partial rho with the other candidates held fixed (rank residuals):")
        for _c, _l in _cands:
            _others = [c for c, _ in _cands if c != _c]
            _r = spearmanr(_Q[_c].to_numpy(float), _Q["ldtm"].to_numpy(float), nan_policy="omit")[0]
            _p, _n = _partial35(_Q[_c].to_numpy(float), _Q["ldtm"].to_numpy(float), _Q[_others].to_numpy(float))
            print(f"       {_c:12s} raw {_r:+.2f}   partial {_p:+.2f}  (n {_n})")
            LAD35.append(dict(box=_b, cause=_c, rho_raw=_r, rho_partial=_p, n=_n, controls="+".join(_others),
                              auc_dusty=mw_auc(_Q[_c].to_numpy(float), _y)[0]))
        # ── the quadrants of the division, and the resolution control of Part 34 for BOTH axes ──
        _kg, _lm = _Q["kappa_gas"].to_numpy(float), _Q["log_mstar"].to_numpy(float)
        _kc, _mc = float(np.nanmedian(_kg)), float(np.nanmedian(_lm))
        print(f"     the quadrants of the division (kappa_gas {_kc:.2f}, log M* {_mc:.2f}):")
        for _kh in (True, False):
            for _mh in (True, False):
                _m = ((_kg > _kc) == _kh) & ((_lm > _mc) == _mh) & np.isfinite(_kg) & np.isfinite(_lm)
                if not _m.any():
                    continue
                _row = dict(box=_b, kappa="high" if _kh else "low", mstar="high" if _mh else "low", n=int(_m.sum()),
                            dusty_frac=float(_y[_m].mean()), **{c: float(np.nanmedian(_Q[c].to_numpy(float)[_m])) for c in ("ldtm", "lfdep", "lz", "lfgas", "lfd", "age") if c in _Q.columns})
                _row[_dose35] = float(np.nanmedian(_Q[_dose35].to_numpy(float)[_m])) if _dose35 in _Q.columns else np.nan
                QUAD35.append(_row)
                print(f"       kappa {'high' if _kh else 'low ':4s} x M* {'high' if _mh else 'low ':4s}: n {_m.sum():5d}  dusty {100 * _y[_m].mean():5.1f} %  "
                      f"log DTM {_row['ldtm']:+.2f}  log Z/Zsun {_row['lz']:+.2f}  log f_gas {_row['lfgas']:+.2f}  dose {_row[_dose35]:.2f}  age {_row.get('age', np.nan):.1f} Gyr")
        _ng = _Q["lngas"].to_numpy(float); _qn = np.nanquantile(_ng[np.isfinite(_ng)], [0, 0.25, 0.5, 0.75, 1.0])
        print("     the resolution control (Part 34's question for both axes) — AUC toward dusty inside quartiles of N_gas:")
        for _c in ("kappa_gas", "log_mstar", "lsig_gas"):
            _s = []
            for _i in range(4):
                _m = (_ng >= _qn[_i]) & (_ng <= _qn[_i + 1])
                _a = mw_auc(_Q[_c].to_numpy(float)[_m], _y[_m])[0]
                _s.append(f"N {10 ** _qn[_i]:.0f}-{10 ** _qn[_i + 1]:.0f}: {_a:.2f}" if np.isfinite(_a) else f"N {10 ** _qn[_i]:.0f}-{10 ** _qn[_i + 1]:.0f}:  --")
            print(f"       {_c:12s} " + "   ".join(_s))

    # ══ FIG A — the waterfall and the factor x cause grid ══
    _bx35 = [b for b in P35_BOXES if b in F35]
    if _bx35:
        FACT35 = [("ldtm", r"$\log\, M_{\rm dust}/(Z\,M_{\rm gas})$" + "\nthe depletion"),
                  ("lz",   r"$\log\, Z/Z_\odot$" + "\nthe metals"),
                  ("lfgas", r"$\log\, M_{\rm gas}/M_\star$" + "\nthe gas")]
        CAUSE35 = [("kappa_gas", r"$\kappa_{\rm rot}$(gas), whole galaxy"), ("log_mstar", r"$\log\, M_\star$"), (_dose35, _dlab35)]
        _ncol = max(3, len(_bx35))
        fig, axs = plt.subplots(4, _ncol, figsize=(4.6 * _ncol, 3.5 * 4), squeeze=False, constrained_layout=True)
        # ── row 0: the waterfall, one panel per box ──
        for _j in range(_ncol):
            ax = axs[0, _j]
            if _j >= len(_bx35):
                ax.axis("off"); continue
            _b = _bx35[_j]; _Q = F35[_b]; _y = (_Q["pop"].to_numpy() == "dusty")
            _terms = [("ldtm", "depletion\n" + r"$M_{\rm dust}/(Z M_{\rm gas})$", "#d7191c"), ("lzabs", "metals\n" + r"$Z$", "#7570b3"), ("lfgas", "gas\n" + r"$M_{\rm gas}/M_\star$", "#2c7bb6")]
            _g = []
            for _c, _l, _col in _terms:
                _v = _Q[_c].to_numpy(float); _ok = np.isfinite(_v) & np.isfinite(_Q["ldtm"].to_numpy(float))
                _g.append(np.nanmean(_v[_ok & _y]) - np.nanmean(_v[_ok & ~_y]))
            _vlab = _Q["lfd"].to_numpy(float); _okl = np.isfinite(_Q["ldtm"].to_numpy(float))
            _tot = np.nanmean(_vlab[_okl & _y]) - np.nanmean(_vlab[_okl & ~_y])
            _base = 0.0
            for (_c, _l, _col), _v in zip(_terms, _g):
                ax.bar([_l], [_v], bottom=[_base], color=_col, edgecolor="k", lw=0.6, width=0.62)
                if abs(_v) > 0.35:
                    ax.text(_l, _base + _v / 2, f"{_v:+.2f}", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
                else:
                    ax.text(_l, _base + _v + 0.06, f"{_v:+.2f}", ha="center", va="bottom", fontsize=8.5, color="k", fontweight="bold")
                _base += _v
            ax.bar(["sum =\nthe label"], [_tot], color="0.85", edgecolor="k", lw=0.8, width=0.62)
            ax.text("sum =\nthe label", _tot / 2, f"{_tot:+.2f}", ha="center", va="center", fontsize=9, fontweight="bold")
            ax.axhline(0, color="k", lw=0.8)
            ax.set_ylabel(r"dusty $-$ dust-poor  [dex]" if _j == 0 else "")
            ax.set_title(f"{BOX_LABEL.get(_b, _b)}\n{int(_y.sum())} dusty vs {int((~_y).sum())} dust-poor", fontsize=9.5)
            ax.grid(axis="y", alpha=0.2, lw=0.5)
        # ── rows 1-3: factor (row) x cause (column), one line per box ──
        for _i, (_f, _flab) in enumerate(FACT35):
            for _k, (_c, _clab) in enumerate(CAUSE35):
                ax = axs[1 + _i, _k]
                for _b in _bx35:
                    _Q = F35[_b]
                    if _c not in _Q.columns:
                        continue
                    _x, _v = _Q[_c].to_numpy(float), _Q[_f].to_numpy(float)
                    _t = _track35(_x, _v)
                    if _t is None:
                        continue
                    _r = spearmanr(_x, _v, nan_policy="omit")[0]
                    ax.fill_between(_t["x"], _t["lo"], _t["hi"], color=BOXC35.get(_b, "0.5"), alpha=0.12, lw=0)
                    ax.plot(_t["x"], _t["med"], color=BOXC35.get(_b, "0.5"), lw=2.3, label=f"{BOXS35.get(_b, _b)}: " + r"$\rho$" + f" = {_r:+.2f}")
                ax.grid(alpha=0.2, lw=0.5)
                ax.set_xlabel(_clab if _i == len(FACT35) - 1 else "")
                ax.set_ylabel(_flab if _k == 0 else "")
                if _i == 0:
                    ax.set_title(_clab, fontsize=10)
                ax.legend(frameon=False, fontsize=7.5, loc="lower right" if _f == "ldtm" else "best")
                if _f == "ldtm":
                    ax.axhline(0, color="0.4", ls=":", lw=1.0)
                    ax.text(0.99, 0.0, " all the metals in grains ", transform=ax.get_yaxis_transform(), ha="right", va="top", fontsize=6.5, color="0.4")
        fig.suptitle("Part 35 — why the division works: the dusty label is the depletion, not the gas and not the metals.  "
                     "Top: the mean dusty $-$ dust-poor gap split into the three exact terms.  Below: each term against each candidate cause",
                     fontsize=10.5)
        paper_save(fig, "agn_states_why_factors")
        plt.show()

        # ══ FIG B — the plane the division is made on, and the ladder toward the depletion ══
        fig, axs = plt.subplots(2, len(_bx35), figsize=(5.0 * len(_bx35), 4.1 * 2), squeeze=False, constrained_layout=True)
        for _j, _b in enumerate(_bx35):
            _Q = F35[_b]; _y = (_Q["pop"].to_numpy() == "dusty")
            _kg, _lm, _dt = _Q["kappa_gas"].to_numpy(float), _Q["log_mstar"].to_numpy(float), _Q["ldtm"].to_numpy(float)
            ax = axs[0, _j]
            _ok = np.isfinite(_kg) & np.isfinite(_lm)
            _nc = int(max(2, min(P35_NCELL, np.sqrt(max(_ok.sum(), 1) / 8.0))))        # equal-count cells, fewer when the box is thin
            _ex = np.nanquantile(_kg[_ok], np.linspace(0, 1, _nc + 1)); _ey = np.nanquantile(_lm[_ok], np.linspace(0, 1, _nc + 1))
            _Z = np.full((_nc, _nc), np.nan); _N = np.zeros_like(_Z)
            for _a in range(_nc):
                for _c2 in range(_nc):
                    _m = _ok & (_kg >= _ex[_a]) & (_kg <= _ex[_a + 1]) & (_lm >= _ey[_c2]) & (_lm <= _ey[_c2 + 1])
                    _N[_c2, _a] = _m.sum()
                    if _m.sum() >= 3 and np.isfinite(_dt[_m]).sum() >= 3:
                        _Z[_c2, _a] = np.nanmedian(_dt[_m])
            _pc = ax.pcolormesh(_ex, _ey, _Z, cmap=P35_CMAP, shading="flat", zorder=1)
            plt.colorbar(_pc, ax=ax, pad=0.01).set_label(r"median $\log\, M_{\rm dust}/(Z M_{\rm gas})$", fontsize=8)
            ax.scatter(_kg[_ok & ~_y], _lm[_ok & ~_y], s=P35_MS, color="0.75", alpha=0.5, lw=0, zorder=2, rasterized=True)
            ax.scatter(_kg[_ok & _y], _lm[_ok & _y], s=P35_MS + 6, facecolor="none", edgecolors=POPS["dusty"]["color"], lw=0.7, zorder=3, label=f"dusty ({int((_ok & _y).sum())})")
            _kc, _mc = float(np.nanmedian(_kg)), float(np.nanmedian(_lm))
            ax.axvline(_kc, color="w", ls="--", lw=1.2, zorder=4); ax.axhline(_mc, color="w", ls="--", lw=1.2, zorder=4)
            for _kh, _mh in ((True, True), (True, False), (False, True), (False, False)):
                _m = ((_kg > _kc) == _kh) & ((_lm > _mc) == _mh) & _ok
                if _m.sum():
                    _tx = 0.5 * (_kc + (_ex[-1] if _kh else _ex[0])); _ty = 0.5 * (_mc + (_ey[-1] if _mh else _ey[0]))
                    ax.text(_tx, _ty, f"{100 * _y[_m].mean():.0f} % dusty\n$n$ = {int(_m.sum())}", ha="center", va="center", fontsize=8, zorder=6,
                            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.6", lw=0.5, alpha=0.85))
            ax.set_xlim(_ex[0], _ex[-1]); ax.set_ylim(_ey[0], _ey[-1])
            ax.set_xlabel(r"$\kappa_{\rm rot}$(gas), whole galaxy"); ax.set_ylabel(r"$\log\, M_\star$" if _j == 0 else "")
            ax.set_title(f"{BOX_LABEL.get(_b, _b)}: the division, coloured by the depletion", fontsize=9.5)
            ax.legend(frameon=True, fontsize=7, loc="lower right", framealpha=0.85, edgecolor="0.7")
            # ── row 1: the ladder ──
            ax = axs[1, _j]
            _L = [r for r in LAD35 if r["box"] == _b]
            if _L:
                _nm = {"kappa_gas": r"$\kappa_{\rm rot}$(gas)", "log_mstar": r"$\log M_\star$", "lz": r"$\log Z$", "lsig_gas": r"$\log \Sigma_{\rm gas}$",
                       _dose35: "dose", "lngas": r"$\log N_{\rm gas}$"}
                _yy = np.arange(len(_L))
                ax.barh(_yy - 0.19, [r["rho_raw"] for r in _L], height=0.36, color="0.7", edgecolor="k", lw=0.5, label=r"raw $\rho$")
                ax.barh(_yy + 0.19, [r["rho_partial"] for r in _L], height=0.36, color="#d7191c", edgecolor="k", lw=0.5, label=r"partial $\rho$ (the others fixed)")
                ax.set_yticks(_yy); ax.set_yticklabels([_nm.get(r["cause"], r["cause"]) for r in _L], fontsize=9)
                ax.axvline(0, color="k", lw=0.8); ax.invert_yaxis(); ax.grid(axis="x", alpha=0.2, lw=0.5)
                ax.set_xlabel(r"Spearman $\rho$ toward $\log\, M_{\rm dust}/(Z M_{\rm gas})$")
                if _j == 0:
                    ax.legend(frameon=False, fontsize=7.5, loc="lower right")
                ax.set_title("what survives when the others are held fixed", fontsize=9.5)
        fig.suptitle("Part 35 — the plane the division is made on, coloured by the share of the metals that sits in grains, and the ladder of causes behind it", fontsize=10.5)
        paper_save(fig, "agn_states_why_plane")
        plt.show()

        pd.DataFrame(GAP35).to_csv(os.path.join(OUT, "agn_states_why_gaps.csv"), index=False)
        pd.DataFrame(LAD35).to_csv(os.path.join(OUT, "agn_states_why_ladder.csv"), index=False)
        pd.DataFrame(QUAD35).to_csv(os.path.join(OUT, "agn_states_why_quadrants.csv"), index=False)
        print(f"-> {os.path.join(OUT, 'agn_states_why_{gaps,ladder,quadrants}.csv')}")

## Part 36 — when the division is made: the two axes on the SFT clock and on the first X-ray clock

Part 35 says what the dusty / dust-poor label is made of and which two axes predict it. This part asks **when** the difference appears: is the division a condition the galaxy already had before it stopped forming stars, is it made at the truncation, or is it made when SIMBA's X-ray channel first fires? Every chain is put on two clocks — $t - t_{\rm SFT}$ (the classifier's truncation, the clock of Parts 3–4) and $t - t_{x1}$, the **first** snapshot at which the X-ray channel fires along the main branch (Part 0's rule: full-velocity jet *and* $f_{\rm gas} <$ `XRAY_FGAS_MAX`) — and followed through both, split by the label it ends with.

**The kinematics along the branch do not exist in the boxes notebook's histories** (they carry `mstar, sfr, mgas, mbh, fedd, mdust, mh2, zgas`), so this part builds them itself, once, into a cached sidecar `agn_states_kin_histories_{box}.hdf5`: it walks the *same* main branch (caesar's `tree_data/progen_galaxy_star`, most massive progenitor, the snapshot ladder the histories store) and reads $\kappa_{\rm rot}$ of the gas and of the stars, the two half-mass radii, $M_{\rm HI}$ and $N_{\rm gas}$ from each catalogue — the pattern Part 29 uses for the $r_{20}/r_{80}$ radii, one file open per snapshot (~100 per box, minutes on the cluster, cached afterwards; `P36_BUILD = False` to read only what is there). Without the catalogues the part still runs: the depletion, the gas and the sSFR come from the histories alone and the $\kappa$ panels say so. The sidecar is matched to the histories on the snapshot numbers and the galaxy ids themselves, so a partial build leaves NaNs rather than misaligning a branch.

Products: `agn_states_when_sft`, `agn_states_when_x1` (the tracks, rows $\kappa_{\rm rot}$(gas) / the depletion / the gas / the sSFR, columns box), `agn_states_when_power` (the AUC of $\kappa_{\rm gas}$, $\log M_\star$, the depletion and the gas fraction toward the *final* label at every epoch of both clocks) and `agn_states_when_plane` (the $(\kappa_{\rm gas}, \log M_\star)$ plane itself at four epochs, the class medians joined by arrows), with `agn_states_when_{tracks,power,clocks}.csv`.

**What it says.** (i) **The two clocks are nearly one clock**: $t_{x1} - t_{\rm SFT}$ has median $+0.07$ Gyr in m100 and m50 and $55$–$56\,\%$ of the chains have $|{\rm lag}| < 0.25$ Gyr — the gas-poor gate does not open before the galaxy stops forming stars, it opens as it does. (ii) **Before the event the two populations are the same galaxies**: the metal share in grains sits at $\approx -0.4$ (four metals in ten in grains) for *both*, with AUC $0.50$ at $t-t_{\rm SFT} = -1$ Gyr, and the gas fractions and the sSFR tracks lie on top of each other. (iii) **The division is made after it**: the eventually-dust-poor lose $\approx 2.5$ dex of depletion within $\sim 1.5$ Gyr of the truncation while the eventually-dusty stay flat — the same picture on the X-ray clock, where the collapse starts within a snapshot of $t_{x1}$. (iv) **$\kappa_{\rm gas}$ does the same thing, not before it**: in m25 (the only box whose sidecar the local test built) both classes are rotating discs before the truncation ($\kappa \approx 0.6$–$0.8$, the dusty ones if anything *lower*, AUC $0.41$ at $-1$ Gyr), both are disturbed through it, and only afterwards do they part — the dusty re-settle to $\kappa \approx 0.8$–$0.9$ while the dust-poor fall to $\approx 0.2$, well below isotropy (AUC $0.69$ at $t=0$, $0.80$ at $+2$ Gyr; the class gap runs from $-0.14$ before the truncation to $+0.6$ after it, and Part 34's sampling noise $s_{\rm iso} = 0.385/\sqrt{N}$ is far below either — it is the *size* that changes, not the significance). (v) **The mass is the exception**: $\log M_\star$ already separates the two classes at $t-t_{\rm SFT} = -3$ Gyr (AUC $0.6$–$0.8$) and does not change — it is the pre-existing half of the division, the geometry is the made half.

So the answer to "why does $\kappa_{\rm gas} + M_\star$ divide them" has a time in it: **the massive galaxies whose gas re-settles into a rotating disc after the truncation keep their grains; the ones whose gas is left dispersion-supported lose them within a Gyr and a half** — and the X-ray channel fires at the truncation for essentially all of them, which is why the dose (a *duration*) is such a weak divider once the geometry is known.

**Caveats.** The depletion track is not an independent test at late times — the label *is* the dust at the anchor, so the AUC there is fixed by construction; what is meaningful is *when* the divergence starts, and the median anchor sits at $t - t_{\rm SFT} \approx +2.3$ Gyr, well after it. Tracks past $\sim 2$ Gyr are the subset whose anchor is that late (the per-epoch counts are in the CSV and behind the legends). Rows are anchors, not distinct galaxies (Parts 2b–4's convention), so the effective $N$ behind an AUC is smaller than the count. The $\kappa$ conclusion rests on m25 until the sidecar is built for the coarse boxes — that build is the first thing this part does when the catalogues are reachable.

In [ ]:
# ── Part 36 — WHEN the division is made: the two axes and the depletion on the SFT clock and on the FIRST X-RAY clock ──
# Part 35 says WHAT the dusty / dust-poor label is made of (the share of the metals locked in grains, at the same metallicity and
# the same gas) and which two axes predict it. This part asks WHEN the difference appears — is the division a condition the
# galaxy already had before it stopped forming stars, or is it made at the truncation, or at the moment SIMBA's X-ray channel
# first fires?  Every galaxy is put on TWO clocks and its branch is followed through both:
#     t - t_SFT   the star-formation truncation of the boxes notebook's classifier (the clock Parts 3-4 use)
#     t - t_x1    the FIRST snapshot at which the X-ray channel fires along the main branch (Part 0's rule: full-velocity jet
#                 AND f_gas < XRAY_FGAS_MAX), i.e. the moment the gas-poor gate opens for the first time
# THE KINEMATICS ALONG THE BRANCH are not in the boxes notebook's histories (they carry mstar, sfr, mgas, mbh, fedd, mdust, mh2,
# zgas). This part builds them itself, once, into a cached sidecar next to the other products: it walks the SAME main branch
# (caesar's tree_data/progen_galaxy_star, most massive progenitor, the same snapshot ladder the histories store) and reads
# kappa_rot of the gas and of the stars, the two half-mass radii, M_HI and N_gas from each catalogue — the pattern Part 29 uses
# for the r20 / r80 radii, one file per snapshot, cached in OUT.  Without the catalogues the part still runs: it draws the
# depletion, the gas and the sSFR from the histories alone and prints which panels are missing.
#   FIG per clock (agn_states_when_{sft,x1}): rows = kappa_rot(gas) / the depletion / the gas fraction / the sSFR, columns = box.
#          The median track of the galaxies that END dusty against those that end dust-poor, bootstrap 16-84 bands, the epoch
#          counts behind, the isotropic 1/3 line on the kappa row (Part 34's yardstick) and t = 0 marked.
#   FIG agn_states_when_power: the AUC toward the final label of kappa_gas, log M*, the depletion and the gas fraction at every
#          epoch of both clocks — the divider's power as a function of time. Flat and high before 0 = the division was already
#          there; rising after 0 = it is made by what happens then.
#   FIG agn_states_when_plane: the (kappa_gas, log M*) plane itself at P36_EPOCHS, one row per box, the two populations as clouds
#          with their medians joined by an arrow — the division drawn as it forms.
#   PRINTED  the coverage, the lag t_x1 - t_SFT (are the two clocks the same clock?), the medians and the AUC at each marked
#          epoch, and the epoch at which each axis first crosses P36_AUCTHR.
#   Products: agn_states_when_{sft,x1}.{png,pdf}, agn_states_when_{power,plane}.{png,pdf}, agn_states_when_{tracks,power}.csv,
#          and the sidecar agn_states_kin_histories_{box}.hdf5 (delete it to rebuild).
# CAVEAT  a galaxy quenched at several anchors contributes one chain per anchor, exactly as in Parts 2b-4: the tracks are
#          per-anchor-row, so the counts are rows, not distinct galaxies, and the AUC's effective N is smaller than it looks.
P36_BOXES  = ["cis100", "cis50", "cis25"]
P36_TG     = np.arange(-3.0, 3.001, 0.25)      # the clock grid [Gyr]
P36_NMIN   = 8                                 # an epoch is drawn only where this many rows contribute
P36_NBOOT  = 300
P36_BUILD  = True                              # build the kinematic sidecar where it is missing (False = read what is there)
P36_CATDIR = globals().get("P29_CATDIR", {})   # {"cis100": "/path/to/Groups"} overrides of the config's catalog_dir
P36_KIN_FILE = os.path.join(OUT, "agn_states_kin_histories_{box}.hdf5")
P36_KIN_COLS = {"kappa_gas": "galaxy_data/dicts/rotation.gas_kappa_rot", "kappa_star": "galaxy_data/dicts/rotation.stellar_kappa_rot",
                "r_gas": "galaxy_data/dicts/radii.gas_half_mass", "r_star": "galaxy_data/dicts/radii.stellar_half_mass",
                "mhi": "galaxy_data/dicts/masses.HI", "ngas": "galaxy_data/ngas"}
P36_EPOCHS = [-1.0, 0.0, 1.0, 2.0]             # the epochs of the plane figure and of the printed table [Gyr on the clock]
P36_PLANE_CLOCK = "sft"                        # which clock the plane figure walks
P36_AUCTHR = 0.70                              # "the divider works" threshold of the power figure
P36_MS     = 7
P36_ISO    = 1.0 / 3.0                         # the isotropic kappa_rot (Part 34)
CLK36 = {"sft": (r"$t - t_{\rm SFT}$  [Gyr]", "the star-formation truncation"),
         "x1":  (r"$t - t_{x1}$  [Gyr]", "the first X-ray episode")}
QUANT36 = [("kappa_gas", r"$\kappa_{\rm rot}$(gas)", "the geometry axis"),
           ("ldep",      r"$\log\, M_{\rm dust}/(M_{\rm dust} + Z M_{\rm gas})$", "the metal share in grains"),
           ("lfgas",     r"$\log\, M_{\rm gas}/M_\star$", "the gas"),
           ("lssfr",     r"$\log$ sSFR [yr$^{-1}$]", "the star formation")]
AUCQ36 = ["kappa_gas", "log_mstar", "ldep", "lfgas"]
AUCC36 = {"kappa_gas": "#d7191c", "log_mstar": "#2c7bb6", "ldep": "k", "lfgas": "#e6ab02"}
AUCL36 = {"kappa_gas": r"$\kappa_{\rm rot}$(gas)", "log_mstar": r"$\log M_\star$", "ldep": "the depletion", "lfgas": r"$M_{\rm gas}/M_\star$"}

import time

if "QM" not in globals() or "BOXES_OK" not in globals():
    print("Part 36: needs Part 0 — run it first.")
else:
    # ── the catalogue walk: the same main branch as the histories, kinematics instead of masses ──
    def _catcfg36(box):
        """(catalog_dir, file_format) of a box: P36_CATDIR wins, then ~/.simbanator/config.json (Part 29's rule), then the first
        CANDIDATE that exists on disk — the share's s50 catalogues moved out of .../SIMBA_50/s50/Groups into .../SIMBA_50/Groups
        (2026-09-01) while the config kept the old path, so the data_dir's and the grandparent's Groups are tried as well.
        Returns the configured path unchanged when nothing exists, so the caller can say what it looked for."""
        import json as _json36
        try:
            cfg = _json36.load(open(os.path.expanduser("~/.simbanator/config.json"))); ent = (cfg.get("simulations", cfg)).get(box, {})
        except Exception:
            ent = {}
        _stem = {"cis100": "m100n1024", "cis50": "m50n512", "cis50nox": "m50n512", "cis25": "m25n512"}.get(box, box)
        fmt = ent.get("file_format", "%s_{snap:03d}.hdf5" % _stem)
        if box in P36_CATDIR:
            return P36_CATDIR[box], fmt
        _d = ent.get("catalog_dir")
        _cand = [_d, os.path.join(ent.get("data_dir", ""), "Groups") if ent.get("data_dir") else None,
                 os.path.join(os.path.dirname(os.path.dirname(_d)), "Groups") if _d else None, os.path.dirname(_d) if _d else None]
        for _c in _cand:
            if _c and os.path.isdir(_c):
                return _c, fmt
        return _d, fmt

    def _read36(box, snap):
        """ONE read of one catalogue: the kinematic columns, the scale factor and the most massive progenitor pointer
        (tree_data/progen_galaxy_star, first column). The pointer comes from the catalogue itself where caesar wrote it and from
        the progen_links sidecar otherwise (the m25 case — its catalogues carry no merger tree), so a snapshot costs one file
        open, not two. Anything missing comes back as None and lands as NaN in the branch."""
        cdir, fmt = _catcfg36(box); fn = fmt.format(snap=int(snap))
        _pg = lambda f: ((lambda a: (a[:, 0] if a.ndim == 2 else a).astype(np.int64))(f["tree_data/progen_galaxy_star"][:])
                         if "tree_data/progen_galaxy_star" in f else None)
        with h5py.File(os.path.join(cdir or "", fn), "r") as f:
            sf = float(f["simulation_attributes"].attrs["scale_factor"])
            cols = {k: (f[d][:].astype(float) if d in f else None) for k, d in P36_KIN_COLS.items()}
            prog = _pg(f)
        if prog is None:
            _q = os.path.join(os.getcwd(), "output", box, "progen_links", fn)
            if os.path.exists(_q):
                with h5py.File(_q, "r") as f:
                    prog = _pg(f)
        for k in ("r_gas", "r_star"):                       # caesar radii are comoving kpc
            if cols.get(k) is not None:
                cols[k] = cols[k] * sf
        return cols, prog

    def _kinbuild36(box):
        """The kinematic sidecar of one box: one group per anchor, arrays (n_snap, n_gal) built by walking the SAME main branch
        the boxes notebook's histories were built from — the anchor's galaxy ids at the anchor snapshot, then the most massive
        progenitor at every step down to the end of that anchor's ladder. One catalogue read per snapshot; the anchors already
        in the file are left alone (delete the file to rebuild). Roughly one file open per snapshot per box: ~100 opens for a
        coarse box, a few minutes on the cluster's own filesystem, and the result is cached for every later run."""
        hp, kp = HIST_FILE.format(box=box), P36_KIN_FILE.format(box=box)
        if not os.path.exists(hp):
            return
        with h5py.File(hp, "r") as f:
            groups = {int(k[4:]): (f[k]["snaps"][:].astype(int), f[k]["gal_ids"][:].astype(np.int64)) for k in f if k.startswith("snap")}
        done = []
        if os.path.exists(kp):
            with h5py.File(kp, "r") as f:
                done = [int(k[4:]) for k in f if k.startswith("snap")]
        todo = {a: v for a, v in groups.items() if a not in done}
        if not todo:
            return
        cdir, _ = _catcfg36(box)
        if not cdir or not os.path.isdir(cdir):
            print(f"  [{box}] no catalogue directory ({cdir}) — the kinematic branch cannot be built (set P36_CATDIR); the kappa panels stay empty")
            return
        smax, smin = max(todo), min(sn.min() for sn, _ in todo.values())
        print(f"  [{box}] building the kinematic branch of anchors {sorted(todo)}: {smax - smin + 1} catalogues, one read each, from {cdir}", flush=True)
        st = {a: dict(idx=ids.astype(float), rec={k: [] for k in P36_KIN_COLS}, snaps=[], ids=ids) for a, (sn, ids) in todo.items()}
        t0, miss = time.time(), []
        for s in range(smax, smin - 1, -1):
            act = [a for a, (sn, _) in todo.items() if a >= s >= sn.min()]
            if not act:
                continue
            try:
                cols, prog = _read36(box, s)
            except Exception as e:
                miss.append(f"snap {s}: {type(e).__name__}: {e}"); cols, prog = {k: None for k in P36_KIN_COLS}, None
            n = len(cols["kappa_gas"]) if cols.get("kappa_gas") is not None else 0
            for a in act:
                S = st[a]; idx = S["idx"]; ok = np.isfinite(idx) & (idx >= 0) & (idx < n); ii = idx[ok].astype(int)
                for k in P36_KIN_COLS:
                    arr = np.full(len(idx), np.nan)
                    if cols.get(k) is not None:
                        arr[ok] = cols[k][ii]
                    S["rec"][k].append(arr)
                S["snaps"].append(s)
                nxt = np.full(len(idx), np.nan)
                if prog is not None:
                    q = prog[ii]; nxt[ok] = np.where(q >= 0, q, np.nan)
                S["idx"] = nxt
            if (smax - s) % 20 == 19:
                print(f"     snap {s:3d}, {time.time() - t0:5.0f} s", flush=True)
        for m in miss[:5]:
            print(f"     catalogue not read, {m}")
        with h5py.File(kp, "a") as f:
            for a, S in st.items():
                g = f.require_group(f"snap{a:03d}")
                for k in list(g.keys()):
                    del g[k]
                g["snaps"] = np.array(S["snaps"], np.int32); g["gal_ids"] = S["ids"]
                for k in P36_KIN_COLS:
                    g[k] = np.vstack(S["rec"][k])
        _cells = [np.vstack(S["rec"]["kappa_gas"]) for S in st.values()]          # one block per anchor: the anchors have different widths
        _fin, _tot = sum(int(np.isfinite(a).sum()) for a in _cells), sum(int(a.size) for a in _cells)
        print(f"  [{box}] kinematic branch -> {kp}  ({time.time() - t0:.0f} s, {100 * _fin / max(_tot, 1):.0f} % of the cells filled — "
              "a cell is empty where the branch has already ended, exactly as in the histories)")

    def _kinread36(box):
        """{anchor: {col: (n_snap, n_gal)}} of the sidecar, keyed the same way as the histories (snaps descending, gal_ids in order)."""
        kp = P36_KIN_FILE.format(box=box)
        if not os.path.exists(kp):
            return {}
        out = {}
        with h5py.File(kp, "r") as f:
            for k in f:
                g = f[k]; out[int(k[4:])] = {c: g[c][:] for c in g.keys()}
        return out

    # ── the tracks: one row per anchor galaxy, on both clocks ──
    T36 = {c: {b: {} for b in P36_BOXES} for c in CLK36}      # clock -> box -> quantity -> {"dusty": array, "pure": array}
    LAG36, COV36 = [], {}
    for _b in [b for b in P36_BOXES if b in BOXES_OK]:
        if P36_BUILD:
            _kinbuild36(_b)
        _KIN = _kinread36(_b)
        _Q = QM[(QM["box"] == _b) & np.isfinite(QM["t_sft"])]
        _want = {(int(s), int(g)): (p == "dusty", float(ts), float(lm)) for s, g, p, ts, lm
                 in zip(_Q["snap"], _Q["gal_id"], _Q["pop"], _Q["t_sft"], _Q["log_mstar"])}
        _acc = {c: {q: {"dusty": [], "pure": []} for q in [x[0] for x in QUANT36] + ["log_mstar", "ngas"]} for c in CLK36}
        _n_chain, _n_kin, _n_nox = 0, 0, 0
        with h5py.File(HIST_FILE.format(box=_b), "r") as f:
            for _k in f:
                _a = int(_k[4:]); _g = f[_k]
                _t = _g["t_yr"][:] / 1e9; _o = np.argsort(_t); _t = _t[_o]
                _ms, _sf, _mg, _mb, _fe = (_g[c][:][_o] for c in ("mstar", "sfr", "mgas", "mbh", "fedd"))
                _md = _g["mdust"][:][_o] if "mdust" in _g else np.full_like(_ms, np.nan)
                _zg = _g["zgas"][:][_o] if "zgas" in _g else np.full_like(_ms, np.nan)
                _S, _, _X = agn_states(_ms, _mg, _mb, _fe)
                # the sidecar is matched on the snapshot numbers and the galaxy ids themselves, never on the array order: a
                # partial build (an interrupted walk, a shorter ladder) then simply leaves those cells NaN instead of dropping out
                _sh = np.asarray(_g["snaps"][:]).astype(int)[_o] if "snaps" in _g else None
                _gid_h = np.asarray(_g["gal_ids"][:]).astype(np.int64)
                _K, _kin = _KIN.get(_a), {}
                if _K is not None and _sh is not None:
                    _rs = {int(s): i for i, s in enumerate(np.asarray(_K["snaps"]).astype(int))}
                    _ri = {int(g): j for j, g in enumerate(np.asarray(_K["gal_ids"]).astype(np.int64))}
                    _row = np.array([_rs.get(int(s), -1) for s in _sh]); _col = np.array([_ri.get(int(g), -1) for g in _gid_h])
                    _mr, _mc = _row >= 0, _col >= 0
                    for _cc in ("kappa_gas", "kappa_star", "ngas"):
                        if _cc in _K and _mr.any() and _mc.any():
                            _B = np.full((len(_sh), len(_gid_h)), np.nan)
                            _B[np.ix_(_mr, _mc)] = np.asarray(_K[_cc], float)[np.ix_(_row[_mr], _col[_mc])]
                            _kin[_cc] = _B
                with np.errstate(all="ignore"):
                    _den = np.where(_X["exist"], _ms, np.nan)
                    _mz = _zg * _mg
                    _V = {"ldep": np.log10(np.clip(np.where((_md >= 0) & (_mz > 0), _md / (_md + _mz), np.nan), 1e-6, None)),
                          "lfgas": np.log10(_X["fgas"]),
                          "lssfr": np.log10(np.clip(np.where(_X["exist"], np.clip(_sf, 0.0, None) / _den, np.nan), SSFR_FLOOR, None)),
                          "log_mstar": np.log10(_den)}
                    _V["kappa_gas"] = _kin.get("kappa_gas", np.full_like(_ms, np.nan))
                    _V["ngas"] = _kin.get("ngas", np.full_like(_ms, np.nan))
                for _j, _gid in enumerate(_g["gal_ids"][:]):
                    _key = (_a, int(_gid))
                    if _key not in _want:
                        continue
                    _dusty, _tsft, _lm = _want[_key]; _e = _X["exist"][:, _j]
                    if _e.sum() < 3:
                        continue
                    _n_chain += 1
                    _n_kin += int(np.isfinite(_V["kappa_gas"][:, _j]).any())
                    _xr = (np.nan_to_num(_S["xray"][:, _j]) > 0.5) & _e
                    _tx1 = float(_t[_xr][0]) if _xr.any() else np.nan
                    _n_nox += int(not _xr.any())
                    LAG36.append(dict(box=_b, snap=_a, gal_id=int(_gid), dusty=bool(_dusty), t_sft=_tsft, t_x1=_tx1,
                                      t_anchor=float(_t[_e][-1]), lag=_tx1 - _tsft if np.isfinite(_tx1) else np.nan, log_mstar=_lm))
                    _p = "dusty" if _dusty else "pure"
                    for _c, _t0 in (("sft", _tsft), ("x1", _tx1)):
                        for _q in _acc[_c]:
                            _v = _V[_q][:, _j]; _ok = _e & np.isfinite(_v)
                            _acc[_c][_q][_p].append(np.interp(P36_TG, (_t - _t0)[_ok], _v[_ok], left=np.nan, right=np.nan)
                                                    if (_ok.sum() >= 3 and np.isfinite(_t0)) else np.full(len(P36_TG), np.nan))
        for _c in CLK36:
            T36[_c][_b] = {_q: {_p: np.asarray(_acc[_c][_q][_p], float).reshape(-1, len(P36_TG)) for _p in ("dusty", "pure")} for _q in _acc[_c]}
        COV36[_b] = (_n_chain, _n_kin, _n_nox)
        print(f"[{_b}] {_n_chain} chains on the clocks, {_n_kin} of them with kinematics ({100 * _n_kin / max(_n_chain, 1):.0f} %), "
              f"{_n_nox} never fire the X-ray channel (no t_x1)")
    LAG36 = pd.DataFrame(LAG36)

    # ── the estimators of the tracks ──
    def _med36(A, seed=36):
        """A (n_rows, n_t) -> median track, its bootstrap 16-84 band and the per-epoch count; epochs under P36_NMIN are blanked."""
        A = np.asarray(A, float)
        if A.ndim != 2 or not len(A):
            return (np.full(len(P36_TG), np.nan),) * 3 + (np.zeros(len(P36_TG), int),)
        n = np.isfinite(A).sum(axis=0)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            cent = np.nanmedian(A, axis=0)
            rng = np.random.default_rng(seed)
            bs = np.array([np.nanmedian(A[rng.integers(0, len(A), len(A))], axis=0) for _ in range(P36_NBOOT)])
            lo, hi = np.nanpercentile(bs, 16, axis=0), np.nanpercentile(bs, 84, axis=0)
        ok = n >= P36_NMIN
        return np.where(ok, cent, np.nan), np.where(ok, lo, np.nan), np.where(ok, hi, np.nan), n

    def _nm36(v):
        """The median of the finite values, nan for an empty column (np.nanmedian would warn)."""
        v = np.asarray(v, float); v = v[np.isfinite(v)]
        return float(np.median(v)) if len(v) else np.nan

    def _auc36(A, B):
        """The AUC toward 'dusty' at every epoch, from the two stacks."""
        out = np.full(len(P36_TG), np.nan)
        for i in range(len(P36_TG)):
            a, b = A[:, i], B[:, i]
            v = np.concatenate([a, b]); m1 = np.concatenate([np.ones(len(a), bool), np.zeros(len(b), bool)])
            if np.isfinite(a).sum() >= P36_NMIN and np.isfinite(b).sum() >= P36_NMIN:
                out[i] = mw_auc(v, m1)[0]
        return out

    _bx36 = [b for b in P36_BOXES if b in BOXES_OK and T36["sft"].get(b)]
    ROW36, POW36 = [], []
    if not _bx36:
        print("Part 36: no box has both a histories cache and a sample — nothing to draw.")
    else:
        if len(LAG36):
            print("\nthe two clocks against each other — t_x1 - t_SFT [Gyr] (negative = the X-ray channel fired BEFORE the truncation):")
            for _b in _bx36:
                _L = LAG36[(LAG36["box"] == _b) & np.isfinite(LAG36["lag"])]
                if not len(_L):
                    continue
                _q = np.percentile(_L["lag"], [16, 50, 84])
                print(f"  {_b}: median {_q[1]:+.2f} (16-84 {_q[0]:+.2f} to {_q[2]:+.2f}) over {len(_L)} chains; "
                      f"dusty {np.median(_L.loc[_L['dusty'], 'lag']):+.2f} vs dust-poor {np.median(_L.loc[~_L['dusty'], 'lag']):+.2f}; "
                      f"|lag| < 0.25 Gyr for {100 * np.mean(np.abs(_L['lag']) < 0.25):.0f} % of them")
        # ══ FIG 1-2 — the tracks, one figure per clock ══
        for _c, (_xlab, _what) in CLK36.items():
            fig, axs = plt.subplots(len(QUANT36), len(_bx36), figsize=(4.9 * len(_bx36), 2.9 * len(QUANT36)), squeeze=False, constrained_layout=True)
            _leg_done = False
            for _j, _b in enumerate(_bx36):
                _S = T36[_c][_b]
                for _i, (_q, _ylab, _wq) in enumerate(QUANT36):
                    ax = axs[_i, _j]
                    _any = False
                    for _p in ("dusty", "pure"):
                        _A = _S[_q][_p]
                        _m, _lo, _hi, _n = _med36(_A)
                        if not np.isfinite(_m).any():
                            continue
                        _any = True
                        _col = POPS["dusty" if _p == "dusty" else "pure"]["color"]
                        ax.fill_between(P36_TG, _lo, _hi, color=_col, alpha=0.18, lw=0)
                        ax.plot(P36_TG, _m, color=_col, lw=2.4, label=f"ends {'dusty' if _p == 'dusty' else 'dust-poor'} ({int(np.nanmax(_n))})")
                        for _t0 in P36_EPOCHS:
                            _k = int(np.argmin(np.abs(P36_TG - _t0)))
                            if np.isfinite(_m[_k]):
                                ROW36.append(dict(box=_b, clock=_c, quantity=_q, t=_t0, pop=_p, median=_m[_k], lo=_lo[_k], hi=_hi[_k], n=int(_n[_k])))
                    if not _any:
                        ax.text(0.5, 0.5, "no kinematic branch\n(the sidecar was not built)" if _q == "kappa_gas" else "no data",
                                transform=ax.transAxes, ha="center", va="center", fontsize=9, color="0.5")
                    ax.axvline(0, color="k", lw=1.0, ls="--")
                    if _q == "kappa_gas":
                        ax.axhline(P36_ISO, color="0.45", ls=":", lw=1.0)
                        ax.text(0.99, P36_ISO, " isotropic 1/3 ", transform=ax.get_yaxis_transform(), ha="right", va="bottom", fontsize=7, color="0.45")
                    ax.grid(alpha=0.2, lw=0.5)
                    ax.set_xlim(P36_TG[0], P36_TG[-1])
                    ax.set_ylabel(_ylab if _j == 0 else "")
                    ax.set_xlabel(_xlab if _i == len(QUANT36) - 1 else "")
                    if _i == 0:
                        ax.set_title(f"{BOX_LABEL.get(_b, _b)}", fontsize=9.5)
                    if _any and not _leg_done:
                        ax.legend(frameon=False, fontsize=8, loc="best"); _leg_done = True
                    if _j == len(_bx36) - 1:
                        ax.text(1.02, 0.5, _wq, transform=ax.transAxes, rotation=90, va="center", ha="left", fontsize=8, color="0.35")
            fig.suptitle(f"Part 36 — the division on the clock of {_what}: the galaxies that END dusty against those that end dust-poor, "
                         f"one median track each ($t = 0$ dashed)", fontsize=10.5)
            paper_save(fig, f"agn_states_when_{_c}")
            plt.show()
        # ══ FIG 3 — the divider's power against time ══
        fig, axs = plt.subplots(len(CLK36), len(_bx36), figsize=(4.9 * len(_bx36), 3.5 * len(CLK36)), squeeze=False, constrained_layout=True)
        for _r, (_c, (_xlab, _what)) in enumerate(CLK36.items()):
            for _j, _b in enumerate(_bx36):
                ax = axs[_r, _j]; _S = T36[_c][_b]
                for _q in AUCQ36:
                    if _q not in _S:
                        continue
                    _a = _auc36(_S[_q]["dusty"], _S[_q]["pure"])
                    if not np.isfinite(_a).any():
                        continue
                    ax.plot(P36_TG, _a, color=AUCC36.get(_q, "0.5"), lw=2.2, label=AUCL36.get(_q, _q))
                    _cross = [t for t, v in zip(P36_TG, _a) if np.isfinite(v) and (v >= P36_AUCTHR or v <= 1 - P36_AUCTHR)]
                    for _t, _v in zip(P36_TG, _a):
                        POW36.append(dict(box=_b, clock=_c, quantity=_q, t=_t, auc=_v))
                    if _cross:
                        ax.plot([_cross[0]], [np.interp(_cross[0], P36_TG, _a)], marker="o", ms=5, color=AUCC36.get(_q, "0.5"), zorder=5)
                ax.axhline(0.5, color="0.4", ls=":", lw=1.0); ax.axvline(0, color="k", ls="--", lw=1.0)
                ax.axhline(P36_AUCTHR, color="0.7", ls="-", lw=0.8); ax.axhline(1 - P36_AUCTHR, color="0.7", ls="-", lw=0.8)
                ax.set_ylim(0.1, 1.0); ax.set_xlim(P36_TG[0], P36_TG[-1]); ax.grid(alpha=0.2, lw=0.5)
                ax.set_xlabel(_xlab); ax.set_ylabel("AUC toward the final dusty label" if _j == 0 else "")
                ax.set_title(f"{BOX_LABEL.get(_b, _b)} — {_what}", fontsize=9.5)
                if _r == 0 and _j == 0:
                    ax.legend(handles=[Line2D([], [], color=AUCC36.get(_q, "0.5"), lw=2.2, label=AUCL36.get(_q, _q)) for _q in AUCQ36],
                              frameon=False, fontsize=8, loc="upper left", title="the dot = first epoch past the threshold", title_fontsize=7)
        fig.suptitle("Part 36 — when each axis becomes a divider: the AUC toward the final dusty label at every epoch of the two clocks", fontsize=10.5)
        paper_save(fig, "agn_states_when_power")
        plt.show()
        # ══ FIG 4 — the plane as it forms ══
        _c = P36_PLANE_CLOCK
        _bk36 = [b for b in _bx36 if np.isfinite(T36[_c][b]["kappa_gas"]["dusty"]).any() or np.isfinite(T36[_c][b]["kappa_gas"]["pure"]).any()]
        if not _bk36:
            print("the plane figure needs the kinematic sidecar — none of the boxes has one (set P36_BUILD = True with the catalogues reachable)")
        else:
            fig, axs = plt.subplots(len(_bk36), len(P36_EPOCHS), figsize=(3.7 * len(P36_EPOCHS), 3.5 * len(_bk36)), squeeze=False, constrained_layout=True)
            for _i, _b in enumerate(_bk36):
                _S = T36[_c][_b]; _prev = {}
                _ally = np.concatenate([_S["log_mstar"][p][:, [int(np.argmin(np.abs(P36_TG - t))) for t in P36_EPOCHS]].ravel() for p in ("dusty", "pure")])
                _ally = _ally[np.isfinite(_ally)]
                _ylim = (np.percentile(_ally, 1) - 0.05, np.percentile(_ally, 99) + 0.05) if len(_ally) else None
                for _k, _t0 in enumerate(P36_EPOCHS):
                    ax = axs[_i, _k]; _ix = int(np.argmin(np.abs(P36_TG - _t0)))
                    for _p in ("pure", "dusty"):
                        _x, _y = _S["kappa_gas"][_p][:, _ix], _S["log_mstar"][_p][:, _ix]
                        _ok = np.isfinite(_x) & np.isfinite(_y)
                        _col = POPS["dusty" if _p == "dusty" else "pure"]["color"]
                        ax.scatter(_x[_ok], _y[_ok], s=P36_MS + (4 if _p == "dusty" else 0), color=_col, alpha=0.5 if _p == "pure" else 0.85,
                                   lw=0, rasterized=True, label=f"ends {'dusty' if _p == 'dusty' else 'dust-poor'} ({int(_ok.sum())})")
                        if _ok.sum() >= P36_NMIN:
                            _mx, _my = _nm36(_x[_ok]), _nm36(_y[_ok])
                            ax.plot([_mx], [_my], marker="*", ms=15, color=_col, mec="k", mew=0.7, zorder=6)
                            if _p in _prev:
                                ax.annotate("", xy=(_mx, _my), xytext=_prev[_p], arrowprops=dict(arrowstyle="->", color=_col, lw=1.6, alpha=0.9))
                            _prev[_p] = (_mx, _my)
                    ax.axvline(P36_ISO, color="0.45", ls=":", lw=1.0)
                    ax.grid(alpha=0.2, lw=0.5); ax.set_xlim(0, 1)
                    if _ylim:
                        ax.set_ylim(*_ylim)
                    ax.set_xlabel(r"$\kappa_{\rm rot}$(gas)" if _i == len(_bk36) - 1 else "")
                    ax.set_ylabel(f"{BOX_LABEL.get(_b, _b)}\n" + r"$\log\, M_\star$" if _k == 0 else "")
                    ax.set_title(f"$t - t_{{\\rm {'SFT' if _c == 'sft' else 'x1'}}}$ = {_t0:+.1f} Gyr", fontsize=9.5)
                    if _i == 0 and _k == 0:
                        ax.legend(frameon=False, fontsize=7.5, loc="lower right")
            fig.suptitle(f"Part 36 — the division as it forms: the ($\\kappa_{{\\rm rot}}$(gas), $\\log M_\\star$) plane at four epochs of {CLK36[_c][1]}; "
                         "the stars are the class medians, the arrows their motion", fontsize=10.5)
            paper_save(fig, "agn_states_when_plane")
            plt.show()
        # ── the printed table and the products ──
        print("\nthe medians at the marked epochs (dusty | dust-poor) and the AUC toward the final label:")
        for _c in CLK36:
            print(f"  clock {_c} ({CLK36[_c][1]}):")
            for _b in _bx36:
                _S = T36[_c][_b]
                for _q, _lab, _ in QUANT36:
                    _A = _S[_q]["dusty"]; _B = _S[_q]["pure"]
                    if not (np.isfinite(_A).any() and np.isfinite(_B).any()):
                        continue
                    _au = _auc36(_A, _B)
                    _s = []
                    for _t0 in P36_EPOCHS:
                        _ix = int(np.argmin(np.abs(P36_TG - _t0)))
                        _e = ""
                        if _q == "kappa_gas":               # Part 34's yardstick: the isotropic sampling noise 0.385 / sqrt(N)
                            _ng = _nm36(np.concatenate([_S["ngas"]["dusty"][:, _ix], _S["ngas"]["pure"][:, _ix]]))
                            if np.isfinite(_ng) and _ng > 0:
                                _e = f" [{(_nm36(_A[:, _ix]) - _nm36(_B[:, _ix])) / (0.385 / np.sqrt(_ng)):+.1f} s_iso]"
                        _s.append(f"t{_t0:+.0f}: {_nm36(_A[:, _ix]):+.2f}|{_nm36(_B[:, _ix]):+.2f} (AUC {_au[_ix]:.2f}){_e}")
                    print(f"    {_b:8s} {_q:10s} " + "   ".join(_s))
        pd.DataFrame(ROW36).to_csv(os.path.join(OUT, "agn_states_when_tracks.csv"), index=False)
        pd.DataFrame(POW36).to_csv(os.path.join(OUT, "agn_states_when_power.csv"), index=False)
        if len(LAG36):
            LAG36.to_csv(os.path.join(OUT, "agn_states_when_clocks.csv"), index=False)
        print(f"-> {os.path.join(OUT, 'agn_states_when_{tracks,power,clocks}.csv')}")

## Conventions, caveats, what the pieces are

* **Pure reads.** Every input is a product of `paper_ism_prediction_boxes.ipynb`: the Part 1 catalogue cache (`ism_prediction_catalogue.fits`), the Part 6a classes (`ism_prediction_agn_classes.csv`, for `t_sft` / `t_qt` / `tau_q`) and the Part 6a per-box history caches (`ism_prediction_histories_<box>.hdf5`). Nothing here opens a caesar catalogue or a snapshot; re-running that notebook refreshes the inputs. A box whose history cache is absent is reported and skipped, not faked.
* **The selection, in full.** The m25 quenched rule of the boxes notebook — $\log M_\star > 10$, ${\rm sSFR} < 0.2/t_{\rm H}$, $N_{\rm gas} \geq 21$, $N_\star \geq 20$, $M_{\rm dust} \geq 10^{-4} M_{\rm H_2}$ — then **three cuts declared once in Part 0 and applied once**: $\log M_\star >$ `MASS_MIN` $= 10.25$, $z \leq$ `Z_MAX` $= 1$ and age $\geq$ `AGE_MIN` $= 2$ Gyr (`AGE_MAX` optional, unset). Then the single split at $\log M_{\rm dust}/M_\star =$ `DUSTY_CUT`. Zero-dust galaxies are kept at `P9_FLOOR` and land in *dust-poor* by construction. Part 0 prints each cut's own toll and the per-box totals, so the sample is auditable in one place — and because it is applied to `QM` before anything else reads it, **every part inherits it**, which is what it is for: before it existed Parts 8-12 ran on a z <= 0.5 subset inherited from the dose engine while Parts 13-15 used every anchor. **The age cut is a deliberate change of principle.** The notebook was built with none, on the grounds that if the dusty and the dust-poor populations turn out to be an old and a young one that should be MEASURED on a dust-selected pair, not defined. `AGE_MIN` trades that for a sample uniform across the notebook and comparable with the observed one (the ALMA-C11 QGs run 2.95-6.84 Gyr). It is NOT neutral: the young under-dosed galaxies it removes are one of the three causes of the *timing-explained* route, so that route shrinks — in m25 the under-dosed fall from 60 to 5 — and Part 8's dose-response weakens with them (cis100 rho over the exposed -0.35 -> -0.12). `AGE_MIN = None` restores the original behaviour.
* **The four states are exclusive and exhaustive** over the snapshots where the galaxy exists, and they are evaluated per snapshot of the main-progenitor branch, not at the anchor. The order of evaluation is dormant $\to$ X-ray on $\to$ X-ray choked $\to$ radiative, as in Part 2. Two conventions inside them are worth stating: a snapshot whose catalogue has **no BH columns at all** (the early ladder) counts as *dormant*, following the boxes notebook's own rule that a missing BH is $w_{\rm jet} = 0$ wherever the galaxy exists; and *radiative* holds both the genuine quasar mode ($f_{\rm Edd} \geq 0.2$) and the accreting BH still under the $10^{7.5}\,M_\odot$ jet threshold. Both sub-shares are printed by Part 2 per box and population and written to `agn_states_snapshot_census.csv`.
* **`XRAY_FGAS_MAX` is the one physics knob.** SIMBA's X-ray heating requires full-velocity jets *and* $M_{\rm gas}/M_\star < 0.2$ (Davé+19), which is the default here and the value `paper_ism_prediction_boxes` Part 6a uses to build `t_onset`. `noxray_analogues.ipynb` explored a looser $0.6$ gate; set the environment variable `XRAY_FGAS_MAX=0.6` before running to reproduce that reading. Only the split between *X-ray on* and *X-ray choked* moves with it — the other two states do not.
* **$\dot M_{\rm BH}$ is derived, not stored.** The catalogues carry `masses.bh` and `bh_fedd`; $\dot M_{\rm BH} = f_{\rm Edd} M_{\rm BH}/(4.5\times10^7\,{\rm yr})$ at $\eta = 0.1$. It therefore adds no information beyond $(M_{\rm BH}, f_{\rm Edd})$ — it is tracked because it is the quantity the state definition is phrased in and the one an accretion-rate argument is made in.
* **Alignment and interpolation.** Each chain is put on $t - t_{\rm SFT}$ and linearly interpolated onto `TG`; nothing is extrapolated beyond a galaxy's sampled range, and a chain needs at least three finite samples. The four state indicators are interpolated as 0/1 series, so they still sum to one at every sampled epoch and their means are honest occupancy fractions; for the state-conditioned rows of Part 4 the assignment is the argmax of those four, which is the nearest-sampled state everywhere except across a transition. Galaxies without an SFT (no history built, or `find_quenching_times` found no event) never enter Parts 2–4 — the Part 1 census prints how many that is per population.
* **The counts behind every panel.** The histogram is the number of galaxies with a finite value at that epoch, **per route**, on one shared scale whose peak is printed in the corner. Tracks are blanked below `NMIN` $= 3$ (lowered from 5 when Parts 3-4 moved from the dusty/pure split to the three routes — the firing-dusty route is a handful of chains outside m100). The count falls off on the right because each chain ends at its own anchor snapshot, and on the left because the tracked history ends (`END_SNAP` in the boxes notebook) — neither is a physical statement.
* **Chains, not galaxies.** A galaxy selected at several anchors contributes one chain per anchor, exactly as in the boxes notebook's Part 9 and `noxray_analogues`' Parts 6c–6m. The per-epoch AUCs are therefore not independent across anchors, and the printed $p$ values are indicative rather than a population test.
* **What replaces what.** This notebook supersedes the `noxray_analogues` Parts 6c–6i line of argument (the age $\times$ dust 2$\times$2 on three clocks, the coupling weight $x_{\rm coup}$ as a continuous 0–1 axis, the five-state 6h taxonomy). The difference is deliberate: one flat dust split instead of a 2$\times$2, and four **named physical states** instead of a weight. The nox run is not a column here by default — `BOXES` takes `"cis50nox"` if its would-be states are wanted, and every state there is a counterfactual since the X-ray energy is never injected in that run.
* **Part 2a integrates the history, and refuses to let a dead black hole look coupled.** (This was Part 7, drawn after the tracks; it moved to the front because the three routes are defined from it and Parts 2b-4 are split on them.) Per galaxy-anchor it builds the *time integrals* of the branch: $\int \dot M_{\rm BH}\,{\rm d}t$ (the total energy budget), the same integral restricted to the `xray` snapshots, the exposure $E_x$, the coupling $\int w_{\rm jet}\,{\rm d}t$, $t_{\rm quiet}$ and $t_{\rm since\ acc}$. Because $w_{\rm jet} = 1$ for any $f_{\rm Edd} \leq 0.02$, a black hole that has stopped accreting is *maximally coupled by construction*: the coupling integral is therefore gated at `P7_FEDD_ACT` $= 10^{-4}$, the accretion-weighted versions are computed alongside it, and the ungated integral is kept with the share of $E_x$ delivered below the floor — the gap between them is the dead-BH coupling, measured. The window is `P7_WINDOW` (the whole branch by default; a float = that many Gyr, `"sft"` = since $t_{\rm SFT}$). **The integrals are built for every anchor** (Parts 2b-4 need a route for every tracked chain); `DOSE`, what Parts 8-13 read, is the $z \leq$ `P7_ZMAX` subset — the same sample those parts have always used. `agn_states_dose_vs_dust_galaxies.csv` is the full frame, with `z`, `cat` and `grp` columns.
* **Part 8 is Part 7's answer, read as an implication.** The dose-response is a *step*: the dusty fraction falls across the first fraction of a Gyr of $E_x$ and is then flat, so $\rho$ over the exposed collapses once a floor is put under $E_x$ (the saturation scan prints it). What survives is one-way: $P(\text{ever exposed} \mid \text{pure})$ is the **necessity** — a galaxy essentially cannot end up dust-poor without the channel having fired — while $P(\text{pure} \mid \text{ever exposed})$ is well under 1, so exposure is **not sufficient**. The converse $P(\text{never exposed} \mid \text{dusty})$ is small *because the never-exposed are a few per cent of the population*: that is arithmetic on the base rates, not a weakening of the rule, and both are drawn side by side with Wilson intervals so they cannot be confused. Panel (C) asks whether the label is doing the work or just restating *young, small black hole*, and the printed census separates the never-exposed that **did** launch a jet (the gate shut them off) from those that never grew a black hole. `P8_SPAN_FRAC` re-runs the conditionals over chains long enough for a 'never' to mean something — a rule that dissolves there was a statement about short chains.
* **Part 9 is the argument on one page.** Top row: the channel is necessary and then saturates (a); the dust returns once it stops (b); and every dusty QG sorted into one of four boxes by *when* the channel fired — never armed / under-dosed / dosed-then-quiet / **dosed and still firing** (c), the first three being the timing story and the fourth being what it cannot reach. Bottom row: those residual galaxies, dusty against dust-poor, on age, exposure, coupling, accreted mass, $M_{\rm BH}$, $M_\star$, $f_{\rm gas}$ and $f_{\rm H_2}$ — every one a coin flip — and then on $M_{\rm dust}/M_{\rm gas}$. It is a MATCHED comparison, not a fit: same age, same dose, same black hole, same gas, and more than a dex apart in dust-to-gas. Two honesty notes carried in the cell: `PF_EX` / `PF_TQ` are cuts, so the decomposition is printed for every box, not just the drawn one; and the last panel's AUC is half-circular (the sample is split on $M_{\rm dust}/M_\star$, which shares a numerator with D/G) — the claim rests on the *gas* panel beside it showing nothing, not on the D/G panel alone.
* **The metallicity and dust-to-metal panels close the escape route.** A dust-to-gas difference is either a difference in how many metals a galaxy has, or in how much of them sits in grains. Part 9 draws $\log Z_{\rm gas}$ immediately before $\log$ (dust-to-metal) so the pair decides between them, and the argument rests on the DENOMINATORS: gas, molecular gas and metallicity are all independent of the `DUSTY_CUT` split axis and all three are coin flips, which leaves the grains. $Z_{\rm gas}$ comes from `zgas` at the anchor, which **Part 7 records** (a `DOSE` built before that column existed raises with the instruction to re-run Part 7). `PF_Z_HAS_DUST` carries the one assumption: SIMBA depletes grain metals out of the gas phase, so the metal budget is $Z_{\rm gas}M_{\rm gas} + M_{\rm dust}$ — set it `True` if the catalogue's `zgas` is already the total.
* **Part 11 turns the three-route comparison into a rate.** $M_{\rm dust}/M_{\rm H_2}$, D/M, $Z_{\rm gas}$, $f_{\rm dust}$, $f_{\rm H_2}$ and sSFR followed on $t - t_{\rm SFT}$ for the unexplained residual / the timing-explained dusty / the dust-poor, with the slope over `P11_FIT` bootstrapped and labelled RISES / FALLS / holds. The D/M row is what enrichment cannot fake, and the $f_{\rm H_2}$ and sSFR rows are the control. The routes are defined AT THE ANCHOR, so the curves are conditioned on the outcome and each chain ends at its own anchor — read the count histogram before reading a late excursion.
* **Part 9's panel (b) is a cross-section, not a track.** It shows that galaxies quiet for longer are more often dusty; Part 11 shows that no route's D/G rises after the SFT. Both are true: the long-quiet are dusty because they are destroying their dust more slowly, not because they rebuild it. The panel title says 'more of them are dusty', not 'the dust comes back'.
* **Part 12 is the plain one.** Fixed (log-spaced) $t_{\rm sq}$ bins, one point per route per bin: median $t_{\rm sq}$, median property, bootstrap error ON THE MEDIAN. Single-galaxy bins are kept but drawn open with no bar. Dust-to-H$_2$ and gas-to-H$_2$ divide out the notebook's 1.36 helium factor; $12+\log({\rm O/H}) = 8.69 + \log(Z/0.0134)$ is scaled from the TOTAL metal mass fraction (solar abundance ratios assumed — the catalogue has no oxygen), with solar drawn. The population's own 16-84 spread is in the CSV, not the figure.
* **Part 13 is the only part that is not a pure read of the boxes products**, and the only one that opens the galaxies up. It takes the three routes of Parts 11-12 — *dusty, dosed & firing* (the residual the timing cannot reach), *dusty, timing-explained* and *dust-poor*, under those same names — into the m25 particle-level tables: (A) the $f_{\rm dust}$ / $f_{\rm H_2}$ / sSFR histories on the SFT clock (the same cache as everywhere else, plus the molecular leg Part 11 does not carry), (B) the radial anatomy per annulus from `output/cis25/tables/{annulus_ism_truth,aperture_truth}.fits` (+ `annulus_av_allincl`, `annulus_kinematics`, `powderday_quenched_selection` where present), and (C) the K-S plane from `output/cis25/ks_tracks_pt/ks_tracks.fits` with `ks_tracks_lib`. Each block gates on its own file and reports what is missing, so the cell runs unchanged off the cluster. Two things to hold onto reading it: the profiles are **per-galaxy sightline medians** (a galaxy counts once per annulus, whatever its number of inclinations) with the band a bootstrap of the group median, and `P13_NMIN` = 3 is below the notebook's `NMIN` because the m25 residual is a handful of objects — every point carries its count, and a three-galaxy median is a three-galaxy median.
* **The three routes are the notebook's unit of comparison.** Parts 2b-4 (the SFT-clock tracks), Parts 9 and 11-13 all split on the same three: *dusty, dosed & firing* (dusty with $E_x \geq$ `PF_EX` **and** $t_{\rm quiet} <$ `PF_TQ`), *dusty, timing-explained* (dusty, but never armed / under-dosed / long quiet), *dust-poor*. `PF_EX`, `PF_TQ`, `CATS`, `route_cat` and `route_grp` all live in **Part 2a** — change them once, there. Part 9 draws the four-way version of the same rule (the timing-explained route opened into its three causes); everything else uses the three. Two of the three differ in AGN history *by construction*, so the contrast that carries information is **firing-dusty against dust-poor**: galaxies dosed alike, differing only in whether the dust survived. The per-epoch AUCs of all three contrasts are in `GPAIRS` and written to `agn_states_tracks.csv`.
* **H$_2$, not the total gas.** Every measured ISM axis in the notebook is molecular or dust: $\log 1.36\,M_{\rm H_2}/M_\star$ in the Part 1 census and on the Part 3 tracks, $\log M_{\rm dust}/M_{\rm H_2}$ in Parts 9, 11 and 12, $M_{\rm H_2}/M_\star$ and $\kappa_{\rm rot}^{\rm H_2}$ in Part 13's annuli. $M_{\rm gas}$ appears in exactly two places and both are structural: **SIMBA's own X-ray gate** $f_{\rm gas} <$ `XRAY_FGAS_MAX`, which is a criterion of the run and not a measurement of ours, and the **metal budget** $Z_{\rm gas}M_{\rm gas} + M_{\rm dust}$ behind the dust-to-metal ratio, where the metals really do live in the whole reservoir. `lfgas` is still carried in the `DOSE` frame so that budget can be formed; nothing plots it. Note that the helium factor `HE` $= 1.36$ is *in* $f_{\rm H_2}$ and *out* of $M_{\rm dust}/M_{\rm H_2}$, so the ratio is per unit bare molecular mass.
* **One sample, and how it got that way.** Part 0 now cuts on mass, redshift and age before anything reads `QM`, so every part runs on the same galaxies (cis100 1373 / cis50 123 / cis25 80 at `Z_MAX` 1, `AGE_MIN` 2). It was not always so: the dose engine used to carry its own `P7_ZMAX` $= 0.5$, which Parts 8-12 inherited through `DOSE` while Parts 13-15 read every anchor — the m25 close-up was silently running on **37 of 178** galaxies. `P7_ZMAX` survives as an EXTRA restriction on the frame Parts 8-13 read (default `None`), and `P13_ZMAX` / `P14_ZMAX` do the same for the m25 close-ups; all three are off, and Part 0 is the one place the sample is set. `DOSE_ALL` and `DOSE` are therefore now the same frame — the distinction is kept so that tightening `P7_ZMAX` alone still works.
* **THE AGE AXES ARE NOT THE SAME QUANTITY on the two sides, and every place they meet says so.** The simulation's `age` is caesar's `dicts/ages.mass_weighted` — a MASS-weighted stellar age — and so is `age_mw_gyr` in the K-S `ks_stage_kinematics` cache. The ALMA-C11 `logage` is CIGALE's `sfhdelayedbq` **`age` parameter**: the ONSET of star formation, a formation time weighted by nothing. (The table's other age columns are no better for this: `logage_obs` is the Dn4000-derived age, which is LIGHT-weighted, and `age_bq` is the age of the quenching episode.) For a delayed-$\tau$ SFH the mass-weighted age sits about $2\tau$ below the onset age — 0.6-1.8 Gyr at these ages for $\tau = 0.3$-1 Gyr, and the burst/quench episode pushes it lower still — so **the observed ages are an UPPER bound on the like-for-like value** and the observed points sit too far to the old side. The table carries no $\tau$, so the conversion cannot be done here; instead Part 14 hatches the observed age band and labels it, Part 15 marks the age column and **excludes it from the scorecard** (`G15_LIKE`; the CSV keeps every panel with an `x_like_for_like` flag). `P14_OBS_AGE_LIKE` / `G15_LIKE` are the switches to flip if a mass-weighted observed age ever lands in the table. Everything else that meets the observations IS like for like: $\Sigma_{\rm e}$, sSFR, $f_{\rm dust}$, and $f_{\rm H_2}$ with the 1.36 helium factor handled explicitly on both sides. **The headline result does not depend on the age axis** — excluding it moves the Part 15 offsets by less than 0.1 dex. Note also that `AGE_MIN` in Part 0 is a cut on the SIMULATION's mass-weighted age: internally consistent, but not calibrated on the observed floor.
* **Part 14 is noxray_analogues Part 8 re-cut on the routes** (and that was the X-ray notebook's Section 4, lean): the m25 galaxies at their CRITICAL POINTS - AGN ignition, jet-mode onset, SFT, QT, track end - in the two spherical zones of the K-S notebook's Part 3b, core ($r < 3.2$ kpc) and outskirt (3.2-10 kpc). Per stage and zone: $\kappa_{\rm rot}^{\rm H_2}$, $\kappa_{\rm rot}^\star$ and the age from the rule-free `ks_stage_kinematics` cache, the ISM from the tracks' PROJECTED apertures (core = `ap3kpc`, outskirt = `ap10kpc` - `ap3kpc`). Zero dust or H$_2$ is NaN, never a floor; zero SFR sits at `SSFR_FLOOR` because the quenching is the point. **The question it settles**: a distinct ROUTE separates already at SFT/QT, a late STATE only at `end`. Not a pure read - it gates on `ks_tracks_pt/ks_tracks.fits` and `ks_tracks/ks_stage_kinematics.fits` and says which is missing.
* **Part 15 is noxray_analogues Part 10 re-cut on the routes** (the X-ray notebook's Section 5b, lean), and it is the one place the notebook meets the OBSERVATIONS: $\log M_{\rm dust}/M_\star$ and $\log 1.36\,M_{\rm H_2}/M_\star$ against age, $\Sigma_{\rm e}$ and sSFR, one figure per box - the three route tracks with bootstrap bands, the *all quenched with a route* dotted track, the channel-off `cis50nox` dashed track, and the ALMA-C11 QGs as points with the upper limits arrowed. The H$_2$ row compares the observed $\alpha_{\rm CO}$ H$_2$ (helium included) to the simulated $1.36\,M_{\rm H_2}$ - like for like. **The scorecard is the test**: the median (measured - track) offset over the DETECTIONS that fall inside each track's own $x$ range, so the route the observations sit on is the one closest to zero. Read it knowing the routes are defined AT THE ANCHOR: a track is the locus of galaxies that ENDED in that route, not a path through the plane. noxray Part 9 (dust-to-H$_2$ against the exposure and the clock) was not ported - Parts 7-9 here already carry that comparison.
* **The Wilson interval is clamped to contain $\hat p$.** At $k = n$ the Wilson end point equals 1 algebraically but can land at $1 - 10^{-16}$ in floating point, which is a *negative* `yerr` and an outright crash in `errorbar`. That is what a dust cut at which every never-exposed galaxy is dusty ($k = n$) used to hit. The printed intervals are unchanged.
* **Parts 28-29 (2026-09-08) both work on Part 0's ANCHOR ROWS**, not Part 22's de-duplicated galaxies — a galaxy quenched at several anchors counts once per anchor, on both sides of every share and fraction, so their counts exceed Part 22's (cis100 1683 rows against 1625 galaxies). Part 28 puts the four runs' dusty / dust-poor tracks on the $f_{\rm dust}$-age plane with the ALMA-C11 points and Part 26's in-box m100 galaxies, and beside it the distribution of the exposure Parts 22-27 split on (`P28_DOSE`, the X-ray duty fraction since SFT, re-run over every anchor row with Part 22's engine; the cut is Part 22's own, `P28_DCUT`) with the four shares — dusty / dust-poor $\times$ low / high dose — written as percentages of the SELECTED SAMPLE (`P28_DENOM`; the four sum to the chain coverage, which Part 0 prints, or to 100 % of the dosed rows). **Part 29 puts the cuts themselves on trial**: along the dose and along two compactness axes — Parts 22-26's $-\log R_{50}({\rm gas})/R_{50}(\star)$ and the gas's own concentration $\log R_{80}/R_{20}$ from caesar's `radii.gas_r20` / `gas_r80` (cached in `agn_states_gas_concentration_<box>.csv`; `P29_CATDIR` overrides the config's catalogue directory) — the dusty fraction in equal-count bins, the contrast of every candidate cut scanned, and the dose $\times$ compactness plane in equal-count cells with the four median quadrants (the classes) and their own fractions. The reading rule is written on the figure: a step is a threshold, a slope a continuum, and a cut on a flat stretch is arbitrary whatever contrast it has; the printed AUC of each axis for dusty vs dust-poor is the cut-free measure of what the axis knows. Neither part reads anything new except those two caesar radii; the rest is the catalogue cache, the histories through Part 22's `_chains22` / `_dose22`, and the ALMA-C11 tables Parts 15 and 26 already read.
* **Part 30 (2026-09-08) draws the two planes a new class division would be made on**, dose $\times$ concentration and $E_x$ $\times$ concentration, with the compact / extended division on the gas concentration $\log R_{80}/R_{20}$ (`P30_CCUT`, a fixed 0.30 = a uniform disc by default; `"best"` = Part 29's per-box cut) and the quadrants' N and dusty fraction written in. It reads Part 29's frames only. The physical anchors of the concentration axis (uniform sphere 0.20, uniform disc 0.30, exponential disc 0.56) are drawn so the numbers mean something: the quenched galaxies' gas is typically a uniform blob, and the dusty ones are the minority with a core.
* **Part 31 (2026-09-08) measures the outcome as a REMOVAL DEPTH**, $\log M_{\rm dust}({\rm anchor})/M_{\rm dust}({\rm SFT})$ (and the H$_2$, the gas), and tests it against the channel (the duty fraction) and the well (the parent halo's $M_{200c}$ and the NFW escape speed at the galaxy's position, read from caesar's `parent_halo_index` and cached) with the partial correlations at fixed dose, at fixed mass and at fixed both. It exists because Parts 29-30 found the mass to be a third axis that no dose definition absorbs: the X-ray energy delivered does not separate the dust at fixed mass, only the time in the state does, and at a given time the massive galaxies are stripped less deeply.
* **Part 32 (2026-09-08) puts the division itself on trial**: every candidate divider — single properties, linear scores of two to four properties (a hand-rolled ridge logistic regression, out-of-fold AUC of a repeated stratified $k$-fold) and two-cut AND rules from a blind quantile scan (largest Youden $J$) — scored on the same anchor rows, with the score's threshold set by the class prior (`P32_THRESH`) rather than by hand. It answers the question Parts 28-31 built up to. First version: the current pair (duty + concentration) does no better than mass + concentration without any dose, the missing axis is the stellar mass (Part 31's gate), the two-cut rule *duty < 0.78-0.79 and $\log M_\star$ > 10.5-10.6* comes out the same in m100 and m50 independently. Second version (same day, `kappa_gas` added — Part 22's own coarse-box axis, which the first ladder lacked): the whole-galaxy $\kappa_{\rm gas}$ is the strongest single divider in m100 / m50, independent of $M_\star$ and stable at every mass and particle count; $\kappa_{\rm gas} + M_\star$ without any dose beats every dose-bearing pair and the duty adds nothing on top (the dose is already written into the gas kinematics; the concentration was a weak proxy of it), so Part 22's division gains a third axis rather than being replaced (Part 33). Still: no divider is clean — the outcome at fixed causes is a gradient (row 3), so the classes are quadrants of a plane, not thresholds on two histograms. In m25 the concentration is the division (the dose is saturated) and one fixed rule does not transfer; a per-box score threshold at the prior does.
* **Part 33 (2026-09-08) shows what the division changes in the pictures**: Part 22's fan and Part 23's resolved anatomy re-cut on the *same* frames under three divisions — as drawn (dose × geometry), the Part 32 cut (the drawn score at its threshold; in m25 only on the rows of Part 0's sample, Part 32's frame there) and the best cut (dose × geometry × $M_\star$, the third axis Part 32 found, eight cells with the light half dotted) — every legend entry carrying $N$ and the dusty share. Pure reads of Parts 22, 23 and 32's globals; the m25 figure needs Part 23's frames in memory. Products `agn_states_division_fans`, `agn_states_division_part23_cis25`, `agn_states_division_compare.csv`.
* **Part 34 (2026-09-09) puts the geometry axis itself on trial**: the catalogue $\kappa_{\rm rot}$(gas) of Parts 22/32 is measured from 50–250 gas particles, so it is scored against the isotropic sampling null $s_{\rm iso}(N) = \sqrt{(4/45)/0.6N} = 0.385/\sqrt{N}$ — (a) the null (observed sd 0.226 against the 0.043 it allows; 33 % above $1/3+2s_{\rm iso}$ against 2.3 %; the class gap 9.5 $\sigma$; the false positives are *not* the few-particle galaxies), (b) the AUC of raw $\kappa$ inside bins of $N$ (flat: 0.81–0.88, so no dilution), (c) the shrinkage family $1/3+(\kappa-1/3)N/(N+N_0)$ against the deconvolved ceiling and against a sham with $N$ shuffled at fixed $M_\star$, (d) the ladder of every de-noised version against $\log N_{\rm gas}$ / $\kappa_\star$ / $\log M_\star$. **The answer: the noise is 2–4 % of the within-class variance, so perfect de-noising is worth +0.002 to +0.005 AUC and the physical shrinkage delivers +0.001; the variants that "gain" more sit above the ceiling because $N$ is the gas mass and the gas carries the dust ($\log N_{\rm gas}$ alone scores 0.76). A de-noised $\kappa$ does not replace Part 32's mass axis (0.820 against 0.874 out-of-fold).** What is worth taking is the *threshold*: `P22_GCUT = 0.42` ($\approx 1/3+2s_{\rm iso}$, a real disc instead of "above the median") lifts Part 22's red group from 17.8 % to 23.7 % dusty in m100 at ~70 % recall — and a fixed 0.42 matches the $N$-dependent cut to within 24 of 1683 galaxies. Product `agn_states_kappa_resolution.{png,pdf,csv}`.

* **Part 35 (2026-09-09) asks why that pair**, and answers it by decomposing the label instead of correlating with it: $\log M_{\rm dust}/M_\star = \log[M_{\rm dust}/(Z M_{\rm gas})] + \log Z + \log M_{\rm gas}/M_\star$ holds galaxy by galaxy (the identity residual is machine zero), so the dusty $-$ dust-poor gap can be split into a depletion, a metals and a gas term. **All of the gap is the depletion**: $+2.4/+2.6/+2.1$ dex in m100 / m50 / m25 against $\pm 0.1$ dex in $Z$ (AUC $0.47$) and $-0.2$ dex in the gas (the dust-poor are the more gas-rich). A dust-poor QG keeps one metal in a thousand in grains, a dusty one one in five. Both axes act on that term alone — at fixed mass, metallicity and dose $\rho(\kappa_{\rm gas},\,{\rm depletion}) = +0.47/+0.37/+0.31$, at fixed $\kappa$, metallicity and dose $\rho(\log M_\star,\,{\rm depletion}) = +0.55/+0.55/+0.32$ — they are orthogonal ($\rho = +0.02$) and nearly additive, which is why the AND rule works (quadrants $1/5/3/21$ % dusty in m100). The dose keeps a small partial of its own ($-0.17$) and is **not** mediated by the kinematics ($\rho(\text{dose},\kappa\mid M_\star)\approx 0$), so Part 32's 'the dose is already written into the gas kinematics' was too strong. In m25 — the box that resolves the gas — the strongest term is the **gas surface density** ($+0.39$), the grain-growth rate of SIMBA's dust model: the mass axis is the coarse boxes' stand-in for a density they cannot measure. Products `agn_states_why_factors`, `agn_states_why_plane`, `agn_states_why_{gaps,ladder,quadrants}.csv`.
* **Part 36 (2026-09-09) asks when the division is made**, on two clocks: $t - t_{\rm SFT}$ and $t - t_{x1}$ (the FIRST X-ray episode along the branch). It builds the kinematics along the main branch itself — a cached sidecar `agn_states_kin_histories_{box}.hdf5` walking caesar's `progen_galaxy_star` over the histories' own snapshot ladder for $\kappa_{\rm rot}$(gas), $\kappa_{\rm rot}(\star)$, the two half-mass radii, $M_{\rm HI}$ and $N_{\rm gas}$ (one file open per snapshot, Part 29's pattern; `P36_BUILD = False` to read only what is there, and the part still draws the history-only rows without it). **The two clocks are one clock** (median $t_{x1}-t_{\rm SFT} = +0.07$ Gyr, $56$ % within a quarter Gyr: the gas-poor gate opens as the galaxy truncates). **Before it the two populations are identical** (metal share in grains $-0.4$ for both, AUC $0.50$ at $-1$ Gyr) and **the split is made after it**: the dust-poor lose $\sim 2.5$ dex of depletion within $\sim 1.5$ Gyr. $\kappa_{\rm gas}$ does the same — both classes are discs before the truncation (AUC $0.41$), both are disturbed through it, and only afterwards do the dusty re-settle to $\kappa \approx 0.8$-$0.9$ while the dust-poor fall to $\approx 0.2$ (m25, the box whose sidecar exists so far). $\log M_\star$ is the exception: it separates them already $3$ Gyr before the truncation — the pre-existing half of the division, the geometry being the made half. Caveat: the depletion track is fixed by construction at the anchor itself, so what it dates is the *start* of the divergence, not its size. Products `agn_states_when_{sft,x1}`, `agn_states_when_{power,plane}`, `agn_states_when_{tracks,power,clocks}.csv`.
